# Member-2 — Target-Independent Pocket-Conditioned Molecular Topology Preparation

**Purpose:** read Member-1 pocket/pharmacophore tensors dynamically, create variable-length pocket tokens, reuse the existing validated CrossDocked2020 ligand graphs and ChEMBL-37 general molecular graphs, and export a stable Member-2 interface.

**Target independence:** no protein name, target accession, pocket count, conformer count, or feature count is hardcoded. If Member-1 supplies target identity in its metadata, it is recorded. If not, the notebook explicitly marks target identity as unknown; it does not assume PARP1.

**Scientific boundary:** CrossDocked2020 is a general protein–ligand dataset. Its ligand graphs are not automatically paired with the current Member-1 pocket. The notebook preserves the original receptor–ligand pair IDs and reports whether target-matched training pairs are actually available. It will NOT create false pocket–ligand pairs.

**What this notebook completes:** M1 input discovery/validation → dynamic pocket tokens → graph dataset checks/reuse → ChEMBL graph checks/reuse → interface and readiness report.

**What it does not claim:** a trained or evaluated de-novo molecular graph generator. Training that model requires verified pocket–ligand pairs for the target supplied by M1. The current generic 495-pair CrossDocked subset alone is not sufficient evidence of target-specific conditioning.

## Run instructions

1. Open in Google Colab and run cells from top to bottom.
2. Runtime: **CPU is sufficient** for all cells in this notebook. No GPU is needed here.
3. Keep the existing Drive files. This notebook reads and reuses them; it does not redownload CrossDocked2020 or ChEMBL-37.
4. If M1 provides multiple pockets, set `POCKET_SELECTOR` in Cell 3 to the required pocket ID. Leave it as `None` only when exactly one pocket is found.
5. Do not use the old PARP1/PARP2 activity-audit cells as a universal target pipeline. Those files are retained on Drive but are not consumed as generic target data here.

In [1]:
# Cell 1 — Mount Drive and import required libraries
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, re, json, gzip, zipfile, shutil, math, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

print("Python:", __import__("sys").version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Mounted at /content/drive
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
NumPy: 2.4.6
Pandas: 2.2.3


In [2]:
# Cell 2 — Project paths and user-controlled selection
BASE_DIR = Path("/content/drive/MyDrive/Member2_Project")
M1_DIR = BASE_DIR / "member1_input"
DATA_DIR = BASE_DIR / "final_selected_500"
LIGAND_DIR = DATA_DIR / "ligands"
RECEPTOR_DIR = DATA_DIR / "receptors"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
OUTPUT_DIR = BASE_DIR / "member2_outputs"
CHEMBL_DIR = BASE_DIR / "chembl37_data"

for folder in [BASE_DIR, M1_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# None means: automatically select only if exactly one pocket is discovered.
# If M1 supplies multiple pockets, set e.g. "1" or the exact pocket ID shown in Cell 3.
POCKET_SELECTOR = None

# Do not set these unless you have independently verified the receptor identity.
# This is intentionally empty: generic CrossDocked records must not be assumed to
# belong to the protein currently supplied by Member-1.
VERIFIED_TARGET_PAIR_IDS = []

print("Project:", BASE_DIR)
print("M1 input folder:", M1_DIR)
print("Output folder:", OUTPUT_DIR)

Project: /content/drive/MyDrive/Member2_Project
M1 input folder: /content/drive/MyDrive/Member2_Project/member1_input
Output folder: /content/drive/MyDrive/Member2_Project/member2_outputs


In [3]:
# Cell 3 — Discover Member-1 tensor/mask files without target-specific names

def normalized_stem(path):
    name = Path(path).stem.lower()
    name = re.sub(r"\(\d+\)$", "", name)  # duplicate suffix such as (1)
    name = re.sub(r"(pharmacophore|feature_tensor|tensor|mask)$", "", name)
    return re.sub(r"[_\- .]+$", "", name)

def discover_m1_arrays(root):
    tensors = []
    masks = []
    if root.exists():
        for p in root.rglob("*.npy"):
            n = p.name.lower()
            if "mask" in n:
                masks.append(p)
            elif any(tag in n for tag in ["pharmacophore", "feature_tensor"]):
                tensors.append(p)
    return sorted(tensors), sorted(masks)

tensor_files, mask_files = discover_m1_arrays(M1_DIR)

# If no extracted arrays are found, safely extract one ZIP already present in member1_input.
if not tensor_files or not mask_files:
    zip_files = sorted(M1_DIR.rglob("*.zip")) if M1_DIR.exists() else []
    if zip_files:
        archive = zip_files[0]
        print("No extracted M1 arrays found. Extracting existing archive:", archive.name)
        with zipfile.ZipFile(archive) as zf:
            root_resolved = M1_DIR.resolve()
            for member in zf.infolist():
                destination = (M1_DIR / member.filename).resolve()
                if root_resolved not in destination.parents and destination != root_resolved:
                    raise RuntimeError(f"Unsafe ZIP path detected: {member.filename}")
            zf.extractall(M1_DIR)
        tensor_files, mask_files = discover_m1_arrays(M1_DIR)

if not tensor_files or not mask_files:
    raise FileNotFoundError(
        "Could not find M1 tensor/mask .npy files. Place the extracted M1 output folder or ZIP inside "
        f"{M1_DIR}. Expected tensor names containing 'pharmacophore' or 'feature_tensor' and mask names containing 'mask'."
    )

def pocket_key(path):
    stem = Path(path).stem.lower()
    stem = re.sub(r"\(\d+\)$", "", stem)
    m = re.search(r"pocket[_-]?([a-z0-9]+)", stem)
    if m:
        return m.group(1)
    return normalized_stem(path)

tensor_groups, mask_groups = {}, {}
for p in tensor_files:
    tensor_groups.setdefault(pocket_key(p), []).append(p)
for p in mask_files:
    mask_groups.setdefault(pocket_key(p), []).append(p)

common_pockets = sorted(set(tensor_groups) & set(mask_groups))
if not common_pockets:
    raise RuntimeError("M1 tensor and mask files were found, but no matching pocket identifiers could be paired.")

print("Discovered pocket IDs:", common_pockets)
print("Tensor files:", len(tensor_files), "| Mask files:", len(mask_files))

Discovered pocket IDs: ['1']
Tensor files: 1 | Mask files: 1


In [4]:
# Cell 4 — Select one pocket and resolve its exact tensor/mask pair
if POCKET_SELECTOR is None:
    if len(common_pockets) != 1:
        raise ValueError(
            f"Found {len(common_pockets)} pockets: {common_pockets}. "
            "Set POCKET_SELECTOR in Cell 2 to the pocket you want to process."
        )
    POCKET_ID = common_pockets[0]
else:
    POCKET_ID = str(POCKET_SELECTOR)
    if POCKET_ID not in common_pockets:
        raise ValueError(f"POCKET_SELECTOR={POCKET_ID!r} not found. Available: {common_pockets}")

# Prefer tensor/mask files in the same directory. Fail if pairing is ambiguous.
t_candidates = tensor_groups[POCKET_ID]
m_candidates = mask_groups[POCKET_ID]
pairs = [(t, m) for t in t_candidates for m in m_candidates if t.parent == m.parent]
if len(pairs) == 1:
    TENSOR_PATH, MASK_PATH = pairs[0]
elif len(pairs) > 1:
    raise RuntimeError(f"Multiple tensor/mask pairs for pocket {POCKET_ID}; resolve duplicate files: {pairs}")
elif len(t_candidates) == 1 and len(m_candidates) == 1:
    TENSOR_PATH, MASK_PATH = t_candidates[0], m_candidates[0]
else:
    raise RuntimeError(f"Ambiguous files for pocket {POCKET_ID}: tensors={t_candidates}; masks={m_candidates}")

print("Selected pocket:", POCKET_ID)
print("Tensor:", TENSOR_PATH)
print("Mask:", MASK_PATH)

Selected pocket: 1
Tensor: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pocket_1_pharmacophore(1).npy
Mask: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pocket_1_mask(1).npy


In [5]:
# Cell 5 — Load and validate M1 arrays
# Agreed current M1 schema: (conformers, feature_slots, channels),
# channels 0:3 = XYZ and channel 3 = categorical feature code.
# If M1 changes these semantics, update this explicit schema contract; do not guess.
XYZ_CHANNELS = (0, 1, 2)
FEATURE_CODE_CHANNEL = 3

tensor = np.load(TENSOR_PATH, allow_pickle=False)
mask_raw = np.load(MASK_PATH, allow_pickle=False)

if tensor.ndim != 3:
    raise ValueError(f"Expected a 3D M1 tensor (conformers, slots, channels); got {tensor.shape}")
if tensor.shape[0] < 1 or tensor.shape[1] < 1 or tensor.shape[2] <= FEATURE_CODE_CHANNEL:
    raise ValueError(f"Invalid M1 tensor shape: {tensor.shape}")
if mask_raw.shape != tensor.shape[:2]:
    raise ValueError(f"Mask shape {mask_raw.shape} does not match tensor leading dimensions {tensor.shape[:2]}")
if not np.isfinite(mask_raw).all() or not np.isin(np.unique(mask_raw), [0, 1]).all():
    raise ValueError("M1 mask must contain only finite binary values 0/1")

mask = mask_raw.astype(bool)
if not mask.any():
    raise ValueError("M1 mask contains no valid feature positions")
if not np.isfinite(tensor[mask]).all():
    raise ValueError("Valid M1 tensor positions contain NaN or Inf")

codes = tensor[:, :, FEATURE_CODE_CHANNEL][mask]
if not np.allclose(codes, np.round(codes)) or np.any(codes < 0):
    raise ValueError("Feature-code channel must contain non-negative integer-valued codes")

tensor = tensor.astype(np.float32)
NUM_CONFORMERS, NUM_SLOTS, NUM_CHANNELS = tensor.shape
FEATURE_VOCAB = sorted(np.unique(codes.astype(int)).tolist())
print("M1 tensor:", tensor.shape, "| mask:", mask.shape)
print("Valid positions:", int(mask.sum()), "| feature codes:", FEATURE_VOCAB)

M1 tensor: (5, 95, 4) | mask: (5, 95)
Valid positions: 475 | feature codes: [0, 1, 2, 3, 4, 5]


In [6]:
# Cell 6 — Read optional M1 metadata and extract target identity only if supplied
metadata_candidates = []
for parent in [TENSOR_PATH.parent, M1_DIR]:
    if parent.exists():
        metadata_candidates.extend(parent.rglob("*.json"))
metadata_candidates = sorted(set(p.resolve() for p in metadata_candidates if p.is_file()))

metadata_path = next((p for p in metadata_candidates if "pharmacophore" in p.name.lower() or "metadata" in p.name.lower()), None)
metadata = {}
if metadata_path:
    with open(metadata_path, "r") as f:
        loaded = json.load(f)
    if isinstance(loaded, dict):
        metadata = loaded.get(str(POCKET_ID), loaded)
    print("Metadata file:", metadata_path)
else:
    print("No M1 metadata JSON found; tensor schema contract will be used.")

# Search nested metadata for known identity labels without assuming a specific target.
TARGET_KEYS = {"target", "target_name", "protein", "protein_name", "uniprot", "uniprot_id", "accession", "gene", "gene_name", "pdb_id"}
def find_metadata_values(obj, keys=TARGET_KEYS, prefix=""):
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = str(k).lower().replace(" ", "_")
            path = f"{prefix}.{k}" if prefix else str(k)
            if key in keys and isinstance(v, (str, int, float)) and str(v).strip():
                found.append((path, str(v).strip()))
            found.extend(find_metadata_values(v, keys, path))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            found.extend(find_metadata_values(v, keys, f"{prefix}[{i}]"))
    return found

target_metadata = find_metadata_values(metadata)
print("Target identity fields found:", target_metadata if target_metadata else "NONE — target identity remains unknown")

# Optional observation table; it is diagnostic only and never overrides tensor/mask values.
observation_candidates = list(M1_DIR.rglob("*.csv")) if M1_DIR.exists() else []
observation_path = next((p for p in observation_candidates if "dynamic_feature_observations" in p.name.lower()), None)
observations = None
if observation_path:
    observations = pd.read_csv(observation_path)
    if "pocket_id" in observations.columns:
        observations = observations[observations["pocket_id"].astype(str) == str(POCKET_ID)].copy()
    print("M1 observation rows for selected pocket:", len(observations))

Metadata file: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pharmacophore_tensors.json
Target identity fields found: NONE — target identity remains unknown
M1 observation rows for selected pocket: 475


In [7]:
# Cell 7 — Convert each conformer into variable-length pocket tokens
# Token layout = [x, y, z, one_hot(feature_code)]
code_to_index = {code: i for i, code in enumerate(FEATURE_VOCAB)}

def make_tokens_for_conformer(conf_index):
    valid = mask[conf_index]
    xyz = tensor[conf_index, valid][:, list(XYZ_CHANNELS)]
    feature_codes = tensor[conf_index, valid, FEATURE_CODE_CHANNEL].astype(int)
    one_hot = np.zeros((len(feature_codes), len(FEATURE_VOCAB)), dtype=np.float32)
    for row_i, code in enumerate(feature_codes):
        one_hot[row_i, code_to_index[int(code)]] = 1.0
    tokens = np.concatenate([xyz, one_hot], axis=1).astype(np.float32)
    if len(tokens) == 0 or not np.isfinite(tokens).all():
        raise ValueError(f"Invalid/empty token set for conformer {conf_index}")
    return tokens

pocket_tokens = [make_tokens_for_conformer(i) for i in range(NUM_CONFORMERS)]

def summarize_tokens(tokens):
    xyz = tokens[:, :3]
    feature = tokens[:, 3:]
    # Mean and standard deviation of coordinates + feature-frequency distribution.
    return np.concatenate([xyz.mean(0), xyz.std(0), feature.mean(0)]).astype(np.float32)

conformer_descriptors = np.stack([summarize_tokens(x) for x in pocket_tokens])
pocket_descriptor = conformer_descriptors.mean(axis=0)

print("Token count per conformer:", [len(x) for x in pocket_tokens])
print("Token dimension:", pocket_tokens[0].shape[1])
print("Descriptor shape:", pocket_descriptor.shape)

Token count per conformer: [95, 95, 95, 95, 95]
Token dimension: 9
Descriptor shape: (12,)


In [8]:
# Cell 8 — Inspect dynamic pocket movement (shared valid slots only)
movement_rows = []
reference = tensor[0, :, :3]
for ci in range(NUM_CONFORMERS):
    shared = mask[0] & mask[ci]
    if shared.any():
        delta = np.linalg.norm(tensor[ci, shared, :3] - reference[shared], axis=1)
        movement_rows.append({
            "conformer_index": ci,
            "shared_slots_with_reference": int(shared.sum()),
            "mean_displacement_A": float(delta.mean()),
            "max_displacement_A": float(delta.max())
        })
movement_df = pd.DataFrame(movement_rows)
display(movement_df)

,conformer_index,shared_slots_with_reference,mean_displacement_A,max_displacement_A
0,0,95,0.000000,0.000000
1,1,95,0.351824,0.588015
2,2,95,0.318326,0.610448
3,3,95,0.278273,0.707565
4,4,95,0.159163,0.305224


In [9]:
# Cell 9 — Reuse and validate the existing CrossDocked2020 graph dataset
# Preferred input is the standardized v2 graph JSONL created earlier.
GRAPH_CANDIDATES = [
    OUTPUT_DIR / "crossdocked_ligand_graphs_v2.jsonl",
    OUTPUT_DIR / "crossdocked_ligand_graphs.jsonl"
]
GRAPH_FILE = next((p for p in GRAPH_CANDIDATES if p.exists()), None)
MAPPING_FILE = OUTPUT_DIR / "final_validated_495_mapping.csv"

if GRAPH_FILE is None:
    raise FileNotFoundError(
        "No existing CrossDocked graph JSONL found. Expected crossdocked_ligand_graphs_v2.jsonl "
        "or crossdocked_ligand_graphs.jsonl in member2_outputs. This notebook will not redownload CrossDocked."
    )
if not MAPPING_FILE.exists():
    raise FileNotFoundError(f"Validated pair mapping is missing: {MAPPING_FILE}")

mapping = pd.read_csv(MAPPING_FILE)
if "pair_idx" not in mapping.columns:
    raise ValueError("Validated mapping must contain pair_idx")
if mapping["pair_idx"].duplicated().any():
    raise ValueError("Duplicate pair_idx values found in validated mapping")

# Read graph records line by line to keep memory predictable.
graph_records = []
with open(GRAPH_FILE, "r") as f:
    for line_no, line in enumerate(f, start=1):
        if line.strip():
            try:
                graph_records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON in graph file at line {line_no}: {e}")

if not graph_records:
    raise ValueError("CrossDocked graph file is empty")
if not all("pair_idx" in g for g in graph_records):
    raise ValueError("Every graph record must preserve pair_idx for receptor–ligand traceability")

graph_ids = [int(g["pair_idx"]) for g in graph_records]
if len(graph_ids) != len(set(graph_ids)):
    raise ValueError("Duplicate pair_idx values found in graph JSONL")

mapping_ids = set(pd.to_numeric(mapping["pair_idx"], errors="raise").astype(int))
graph_id_set = set(graph_ids)
print("Mapping rows:", len(mapping), "| graph records:", len(graph_records))
print("Graph IDs represented in mapping:", len(graph_id_set & mapping_ids))
print("Graph file:", GRAPH_FILE)

Mapping rows: 495 | graph records: 495
Graph IDs represented in mapping: 495
Graph file: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_ligand_graphs_v2.jsonl


In [10]:
# Cell 10 — Validate molecular graph structure and pair-ID traceability
schema_counts = Counter()
validation_rows = []
for g in graph_records:
    pid = int(g["pair_idx"])
    n = int(g.get("num_atoms", -1))
    e = int(g.get("num_bonds", -1))
    nodes = g.get("node_features", [])
    edge_index = g.get("edge_index", [])
    edge_features = g.get("edge_features", [])
    coords = g.get("heavy_atom_coordinates", None)
    status, reason = "PASS", ""
    if n <= 0 or len(nodes) != n:
        status, reason = "FAIL", "node count does not match num_atoms"
    elif len(edge_index) != 2 * e or len(edge_features) != 2 * e:
        status, reason = "FAIL", "edge count does not match bidirectional bond count"
    elif coords is not None and len(coords) != n:
        status, reason = "FAIL", "heavy-atom coordinate count mismatch"
    elif pid not in mapping_ids:
        status, reason = "FAIL", "pair_idx absent from validated mapping"
    schema_counts[str(g.get("schema_version", "unknown"))] += 1
    validation_rows.append({"pair_idx": pid, "status": status, "reason": reason, "num_atoms": n, "num_bonds": e})

graph_validation = pd.DataFrame(validation_rows)
graph_validation.to_csv(OUTPUT_DIR / "member2_graph_integrity_report.csv", index=False)
print("Graph schema versions:", dict(schema_counts))
print("Graph validation:", graph_validation["status"].value_counts().to_dict())
display(graph_validation.head())

Graph schema versions: {'2.0': 495}
Graph validation: {'PASS': 495}


,pair_idx,status,reason,num_atoms,num_bonds
0,158402,PASS,,49,52
1,159900,PASS,,10,9
2,84592,PASS,,47,53
3,154374,PASS,,25,26
4,159426,PASS,,10,9


In [11]:
# Cell 11 — Reuse the ChEMBL-37 general graph set if already prepared
# ChEMBL general structures are chemical-structure examples, not pocket–ligand pairs
# and not target-specific activity labels.
CHEMBL_GRAPH_FILE = OUTPUT_DIR / "chembl37_general_graphs.jsonl"
chembl_graph_count = 0
chembl_sample = []
if CHEMBL_GRAPH_FILE.exists():
    with open(CHEMBL_GRAPH_FILE, "r") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            chembl_graph_count += 1
            if len(chembl_sample) < 5:
                chembl_sample.append(record)
    print("ChEMBL general graph records:", chembl_graph_count)
    print("Sample keys:", list(chembl_sample[0].keys()) if chembl_sample else [])
else:
    print("ChEMBL general graph file not found. This is optional for this run:", CHEMBL_GRAPH_FILE)

# The existing PARP-specific activity files are deliberately not loaded here.
# They are valid historical target-specific work, not a universal target dataset.
legacy_target_files = sorted(OUTPUT_DIR.glob("*parp*activity*.csv"))
print("Legacy target-specific activity files preserved but not consumed:", len(legacy_target_files))

ChEMBL general graph records: 50000
Sample keys: ['schema_version', 'source', 'chembl_id', 'inchikey', 'smiles', 'num_atoms', 'num_bonds', 'node_features', 'edge_index', 'edge_features', 'coordinates', 'has_3d_coordinates', 'molecular_weight', 'feature_schema']
Legacy target-specific activity files preserved but not consumed: 9


In [12]:
# Cell 12 — Determine whether valid pocket-conditioned training pairs exist
# A CrossDocked ligand is paired to its own receptor by pair_idx. It is NOT paired
# to the M1 pocket unless receptor identity has been independently verified.

# Try to resolve receptor paths from the authoritative mapping.
required_mapping_cols = {"pair_idx", "complex_dir", "receptor_pdb"}
if required_mapping_cols.issubset(mapping.columns):
    mapping["receptor_path"] = mapping.apply(
        lambda r: str(RECEPTOR_DIR / str(r["complex_dir"]) / str(r["receptor_pdb"])), axis=1
    )
    mapping["receptor_file_exists"] = mapping["receptor_path"].map(lambda p: Path(p).is_file())
else:
    mapping["receptor_path"] = None
    mapping["receptor_file_exists"] = False

verified_ids = set(int(x) for x in VERIFIED_TARGET_PAIR_IDS)
if verified_ids:
    unknown_ids = verified_ids - mapping_ids
    if unknown_ids:
        raise ValueError(f"Verified pair IDs are not in validated mapping: {sorted(unknown_ids)[:20]}")

mapping["manually_verified_target_match"] = mapping["pair_idx"].astype(int).isin(verified_ids)
matched_training_pairs = mapping[mapping["manually_verified_target_match"] & mapping["receptor_file_exists"]].copy()

TARGET_IDENTITY_KNOWN = bool(target_metadata)
PAIRING_READY = len(matched_training_pairs) > 0

print("M1 target identity known from metadata:", TARGET_IDENTITY_KNOWN)
print("Manually verified target pair IDs supplied:", len(verified_ids))
print("Verified pairs with receptor file present:", len(matched_training_pairs))
print("Pocket-conditioned training ready:", PAIRING_READY)

mapping.to_csv(OUTPUT_DIR / "member2_pairing_audit.csv", index=False)
if PAIRING_READY:
    matched_training_pairs.to_csv(OUTPUT_DIR / "member2_verified_target_pairs.csv", index=False)
else:
    print("IMPORTANT: No target-matched pairs are certified. Do not train using all generic CrossDocked ligands as if they bind the M1 pocket.")

M1 target identity known from metadata: False
Manually verified target pair IDs supplied: 0
Verified pairs with receptor file present: 0
Pocket-conditioned training ready: False
IMPORTANT: No target-matched pairs are certified. Do not train using all generic CrossDocked ligands as if they bind the M1 pocket.


In [13]:
# Cell 13 — Export pocket tokens and stable Member-2 interface
# Store each conformer separately because token counts may differ.
token_payload = {f"conformer_{i+1}": arr for i, arr in enumerate(pocket_tokens)}
np.savez_compressed(OUTPUT_DIR / "pocket_tokens_per_conformer.npz", **token_payload)
np.save(OUTPUT_DIR / "pocket_descriptor.npy", pocket_descriptor)
np.save(OUTPUT_DIR / "pocket_conformer_descriptors.npy", conformer_descriptors)

interface = {
    "schema_version": "member2-pocket-interface-2.0",
    "pocket_id": str(POCKET_ID),
    "target_identity_fields": target_metadata,
    "target_identity_known": TARGET_IDENTITY_KNOWN,
    "input_tensor_shape": list(tensor.shape),
    "input_mask_shape": list(mask.shape),
    "num_conformers": int(NUM_CONFORMERS),
    "max_feature_slots": int(NUM_SLOTS),
    "num_channels": int(NUM_CHANNELS),
    "coordinate_channels": list(XYZ_CHANNELS),
    "feature_code_channel": int(FEATURE_CODE_CHANNEL),
    "feature_code_vocabulary": FEATURE_VOCAB,
    "pocket_token_layout": ["x", "y", "z", "dynamic_feature_one_hot"],
    "pocket_token_dimension": int(pocket_tokens[0].shape[1]),
    "pocket_descriptor_dimension": int(pocket_descriptor.shape[0]),
    "crossdocked_graph_file": GRAPH_FILE.name,
    "crossdocked_graph_records": int(len(graph_records)),
    "chembl_general_graph_file": CHEMBL_GRAPH_FILE.name if CHEMBL_GRAPH_FILE.exists() else None,
    "chembl_general_graph_records": int(chembl_graph_count),
    "crossdocked_is_target_specific": False,
    "verified_target_pair_count": int(len(matched_training_pairs)),
    "pocket_conditioned_training_ready": bool(PAIRING_READY),
    "generator_trained": False,
    "outputs": {
        "pocket_tokens": "pocket_tokens_per_conformer.npz",
        "pocket_descriptor": "pocket_descriptor.npy",
        "conformer_descriptors": "pocket_conformer_descriptors.npy",
        "graph_integrity_report": "member2_graph_integrity_report.csv",
        "pairing_audit": "member2_pairing_audit.csv"
    }
}
with open(OUTPUT_DIR / "member2_interface.json", "w") as f:
    json.dump(interface, f, indent=2)

print(json.dumps(interface, indent=2))

{
  "schema_version": "member2-pocket-interface-2.0",
  "pocket_id": "1",
  "target_identity_fields": [],
  "target_identity_known": false,
  "input_tensor_shape": [
    5,
    95,
    4
  ],
  "input_mask_shape": [
    5,
    95
  ],
  "num_conformers": 5,
  "max_feature_slots": 95,
  "num_channels": 4,
  "coordinate_channels": [
    0,
    1,
    2
  ],
  "feature_code_channel": 3,
  "feature_code_vocabulary": [
    0,
    1,
    2,
    3,
    4,
    5
  ],
  "pocket_token_layout": [
    "x",
    "y",
    "z",
    "dynamic_feature_one_hot"
  ],
  "pocket_token_dimension": 9,
  "pocket_descriptor_dimension": 12,
  "crossdocked_graph_file": "crossdocked_ligand_graphs_v2.jsonl",
  "crossdocked_graph_records": 495,
  "chembl_general_graph_file": "chembl37_general_graphs.jsonl",
  "chembl_general_graph_records": 50000,
  "crossdocked_is_target_specific": false,
  "verified_target_pair_count": 0,
  "pocket_conditioned_training_ready": false,
  "generator_trained": false,
  "outputs": {
   

In [14]:
# Cell 14 — Final checks and honest readiness report
checks = {
    "m1_tensor_and_mask_valid": bool(tensor.ndim == 3 and mask.shape == tensor.shape[:2]),
    "all_conformers_tokenized": bool(len(pocket_tokens) == NUM_CONFORMERS),
    "pocket_descriptor_finite": bool(np.isfinite(pocket_descriptor).all()),
    "graph_records_present": bool(len(graph_records) > 0),
    "all_graph_records_pass_integrity": bool((graph_validation["status"] == "PASS").all()),
    "interface_written": (OUTPUT_DIR / "member2_interface.json").exists(),
    "token_file_written": (OUTPUT_DIR / "pocket_tokens_per_conformer.npz").exists(),
    "target_matched_pairs_available": bool(PAIRING_READY)
}

print("========== MEMBER-2 READINESS ==========")
for name, passed in checks.items():
    label = "PASS" if passed else ("PENDING" if name == "target_matched_pairs_available" else "FAIL")
    print(f"{label:7s} {name}")

# The preparation stage may complete even if target-matched pairs are pending.
# That pending status blocks generator training; it is not silently treated as a pass.
blocking_failures = [k for k, v in checks.items() if not v and k != "target_matched_pairs_available"]
readiness = {
    "preparation_stage": "PASS" if not blocking_failures else "FAIL",
    "pocket_conditioned_generator_training": "READY" if PAIRING_READY and not blocking_failures else "BLOCKED_PENDING_VERIFIED_TARGET_PAIRS",
    "generator_trained": False,
    "blocking_failures": blocking_failures,
    "target_matched_pair_count": int(len(matched_training_pairs)),
    "next_requirement": "Obtain and verify receptor–ligand complexes for the exact target/pocket supplied by Member-1; then train/evaluate the pocket-conditioned graph generator."
}
with open(OUTPUT_DIR / "member2_readiness.json", "w") as f:
    json.dump(readiness, f, indent=2)
print("\n", json.dumps(readiness, indent=2))
if blocking_failures:
    raise RuntimeError("Member-2 preparation failed checks: " + ", ".join(blocking_failures))

========== MEMBER-2 READINESS ==========
PASS    m1_tensor_and_mask_valid
PASS    all_conformers_tokenized
PASS    pocket_descriptor_finite
PASS    graph_records_present
PASS    all_graph_records_pass_integrity
PASS    interface_written
PASS    token_file_written
PENDING target_matched_pairs_available

 {
  "preparation_stage": "PASS",
  "pocket_conditioned_generator_training": "BLOCKED_PENDING_VERIFIED_TARGET_PAIRS",
  "generator_trained": false,
  "blocking_failures": [],
  "target_matched_pair_count": 0,
  "next_requirement": "Obtain and verify receptor\u2013ligand complexes for the exact target/pocket supplied by Member-1; then train/evaluate the pocket-conditioned graph generator."
}


## What the outputs mean

- `pocket_tokens_per_conformer.npz`: variable-length M1 pocket tokens; one array per conformer.
- `pocket_descriptor.npy` and `pocket_conformer_descriptors.npy`: deterministic baseline descriptors, not learned embeddings.
- `crossdocked_ligand_graphs_v2.jsonl`: existing standardized ligand graphs, kept separate from the M1 pocket until target matching is verified.
- `member2_pairing_audit.csv`: receptor file availability and explicit target-match status for each pair.
- `member2_interface.json`: schema and file handoff for downstream modules.
- `member2_readiness.json`: distinguishes successful data preparation from readiness to train a pocket-conditioned generator.

**Important next scientific step:** identify/verify CrossDocked (or another structural dataset) receptor–ligand complexes that match the exact target supplied by M1. Only those matched pairs can supervise pocket-conditioned topology generation. After that, implement and train the graph generator (pocket encoder + autoregressive/flow-based graph decoder) and evaluate validity, uniqueness, novelty, conditioning fidelity, and downstream 3D/ADMET compatibility.

In [15]:
# CELL 1: PROJECT SETUP

from pathlib import Path
import os
import json
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Main project directory
PROJECT_ROOT = Path('/content/drive/MyDrive/Member2_Project')

# Member-1 input directory
INPUT_ROOT = PROJECT_ROOT / 'member1_input'

# Member-2 output directory
OUTPUT_DIR = PROJECT_ROOT / 'member2_outputs'

# Create directories
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("========== MEMBER-2 PROJECT SETUP ==========")
print("Project:", PROJECT_ROOT)
print("Input:", INPUT_ROOT)
print("Output:", OUTPUT_DIR)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Device: CPU preparation")
print("============================================")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
========== MEMBER-2 PROJECT SETUP ==========
Project: /content/drive/MyDrive/Member2_Project
Input: /content/drive/MyDrive/Member2_Project/member1_input
Output: /content/drive/MyDrive/Member2_Project/member2_outputs
NumPy: 2.4.6
Pandas: 2.2.3
Device: CPU preparation


In [16]:
# CELL 18: DYNAMIC STRUCTURE FILE DISCOVERY

from pathlib import Path

STRUCTURE_EXTENSIONS = {
    ".pdb", ".ent", ".cif", ".mmcif"
}

structure_files = sorted({
    p.resolve()
    for p in Path(M1_DIR).rglob("*")
    if p.is_file() and p.suffix.lower() in STRUCTURE_EXTENSIONS
})

print("========== MEMBER-1 STRUCTURE DISCOVERY ==========")
print("Member-1 directory:", M1_DIR)
print("Structures discovered:", len(structure_files))

for i, path in enumerate(structure_files, 1):
    print(f"{i}. {path}")

if not structure_files:
    print("\nNo structure files found.")
    print("Check whether Member-1 PDB/mmCIF files were extracted.")

========== MEMBER-1 STRUCTURE DISCOVERY ==========
Member-1 directory: /content/drive/MyDrive/Member2_Project/member1_input
Structures discovered: 5
1. /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_001.pdb
2. /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_002.pdb
3. /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_003.pdb
4. /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_004.pdb
5. /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_005.pdb


In [17]:
# CELL 19: PROTEIN CHAIN AND SEQUENCE EXTRACTION

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("Bio") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "biopython"
    ])

from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa, protein_letters_3to1_extended
from Bio.SeqUtils import seq1
import warnings

def residue_to_one_letter(residue):
    """
    Convert amino-acid residue names to one-letter codes.
    Unknown residues are represented as X.
    """
    name = residue.get_resname().strip().upper()

    if name in protein_letters_3to1_extended:
        return protein_letters_3to1_extended[name]

    try:
        return seq1(name, custom_map={"UNK": "X"})
    except Exception:
        return "X"


def extract_chain_information(structure_path):
    suffix = structure_path.suffix.lower()

    if suffix in {".cif", ".mmcif"}:
        parser = MMCIFParser(QUIET=True)
    else:
        parser = PDBParser(QUIET=True)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        structure = parser.get_structure(
            structure_path.stem,
            str(structure_path)
        )

    results = []

    for model in structure:
        for chain in model:

            sequence = []
            residue_numbers = []

            for residue in chain:

                if not is_aa(residue, standard=False):
                    continue

                # Ignore residues without a CA atom.
                if "CA" not in residue:
                    continue

                letter = residue_to_one_letter(residue)

                if not letter:
                    letter = "X"

                sequence.append(letter)

                residue_numbers.append({
                    "residue_name": residue.get_resname().strip(),
                    "residue_number": residue.id[1],
                    "insertion_code": residue.id[2].strip()
                })

            sequence = "".join(sequence)

            if len(sequence) == 0:
                continue

            results.append({
                "file": str(structure_path),
                "model": model.id,
                "chain_id": chain.id,
                "sequence": sequence,
                "sequence_length": len(sequence),
                "residue_details": residue_numbers
            })

    return results


chain_records = []
failed_structures = []

for structure_path in structure_files:
    try:
        records = extract_chain_information(structure_path)
        chain_records.extend(records)

    except Exception as error:
        failed_structures.append({
            "file": str(structure_path),
            "error": str(error)
        })

print("========== CHAIN EXTRACTION ==========")
print("Structures discovered:", len(structure_files))
print("Protein chains extracted:", len(chain_records))
print("Structures with parsing errors:", len(failed_structures))

========== CHAIN EXTRACTION ==========
Structures discovered: 5
Protein chains extracted: 10
Structures with parsing errors: 0


In [18]:
# CELL 20: SAVE PROTEIN CHAIN REPORT

import pandas as pd
import json

chain_report = []

for record in chain_records:
    chain_report.append({
        "file": record["file"],
        "model": record["model"],
        "chain_id": record["chain_id"],
        "sequence_length": record["sequence_length"],
        "sequence": record["sequence"]
    })

chain_df = pd.DataFrame(chain_report)

print("========== PROTEIN CHAIN REPORT ==========")

if not chain_df.empty:
    display(chain_df)

    report_path = Path(OUTPUT_DIR) / "member1_protein_chain_report.csv"
    chain_df.to_csv(report_path, index=False)

    sequence_path = Path(OUTPUT_DIR) / "member1_protein_sequences.json"

    with open(sequence_path, "w") as f:
        json.dump(chain_records, f, indent=2)

    print("\nSaved reports:")
    print(report_path)
    print(sequence_path)

else:
    print("No protein chains were extracted.")

if failed_structures:
    print("\n========== PARSING ERRORS ==========")
    display(pd.DataFrame(failed_structures))

========== PROTEIN CHAIN REPORT ==========


,file,model,chain_id,sequence_length,sequence
0,/content/drive/MyDrive/Member2_Project/member1...,0,A,350,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
1,/content/drive/MyDrive/Member2_Project/member1...,0,B,348,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
2,/content/drive/MyDrive/Member2_Project/member1...,0,A,350,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
3,/content/drive/MyDrive/Member2_Project/member1...,0,B,348,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
4,/content/drive/MyDrive/Member2_Project/member1...,0,A,350,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
5,/content/drive/MyDrive/Member2_Project/member1...,0,B,348,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
6,/content/drive/MyDrive/Member2_Project/member1...,0,A,350,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
7,/content/drive/MyDrive/Member2_Project/member1...,0,B,348,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
8,/content/drive/MyDrive/Member2_Project/member1...,0,A,350,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...
9,/content/drive/MyDrive/Member2_Project/member1...,0,B,348,KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQ...



Saved reports:
/content/drive/MyDrive/Member2_Project/member2_outputs/member1_protein_chain_report.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/member1_protein_sequences.json


In [22]:
# CELL 21: DYNAMIC TARGET IDENTIFICATION USING RCSB PDB

import requests
import pandas as pd
import time
import json

from pathlib import Path
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices

BLAST_DIR = Path(OUTPUT_DIR) / "target_identification"
BLAST_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
DATA_URL = "https://data.rcsb.org/rest/v1/core/polymer_entity"

session = requests.Session()

# --------------------------------------------------
# 1. Prepare sequence alignment
# --------------------------------------------------

aligner = PairwiseAligner()
aligner.mode = "local"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

# --------------------------------------------------
# 2. Search RCSB for similar experimental structures
# --------------------------------------------------

def search_rcsb(sequence):

    query = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "value": sequence,
                "target": "pdb_protein_sequence",
                "identity_cutoff": 0.30,
                "evalue_cutoff": 1.0
            }
        },
        "return_type": "polymer_entity",
        "request_options": {
            "paginate": {
                "start": 0,
                "rows": 100
            },
            "results_content_type": ["experimental"]
        }
    }

    response = session.post(
        SEARCH_URL,
        json=query,
        timeout=120
    )

    response.raise_for_status()

    return response.json()


# --------------------------------------------------
# 3. Retrieve matching protein entity information
# --------------------------------------------------

def get_entity_metadata(identifier):

    pdb_id, entity_id = identifier.split("_")

    response = session.get(
        f"{DATA_URL}/{pdb_id}/{entity_id}",
        timeout=60
    )

    response.raise_for_status()

    return response.json()


# --------------------------------------------------
# 4. Search every unique Member-1 sequence
# --------------------------------------------------

rcsb_hits = []
search_errors = []

for index, (sequence, source_record) in enumerate(
    unique_sequences.items(), 1
):

    query_id = f"M1_SEQUENCE_{index}"

    print("\n================================")
    print("Query:", query_id)
    print("Chain:", source_record["chain_id"])
    print("Length:", len(sequence))
    print("================================")

    try:

        result = search_rcsb(sequence)

        result_set = result.get("result_set", [])

        print("Matching polymer entities:", len(result_set))

        for hit in result_set:

            identifier = hit["identifier"]

            try:
                metadata = get_entity_metadata(identifier)

                reference_sequence = metadata.get(
                    "entity_poly", {}
                ).get("pdbx_seq_one_letter_code_can", "")

                reference_sequence = "".join(
                    reference_sequence.split()
                ).replace("X", "X")

                if not reference_sequence:
                    continue

                alignment = aligner.align(
                    sequence,
                    reference_sequence
                )[0]

                aligned_pairs = alignment.aligned

                aligned_query_positions = sum(
                    end - start
                    for start, end in aligned_pairs[0]
                )

                aligned_reference_positions = sum(
                    end - start
                    for start, end in aligned_pairs[1]
                )

                identical = 0
                aligned_length = 0

                for (q_start, q_end), (s_start, s_end) in zip(
                    aligned_pairs[0],
                    aligned_pairs[1]
                ):

                    length = min(
                        q_end - q_start,
                        s_end - s_start
                    )

                    for offset in range(length):

                        if (
                            sequence[q_start + offset]
                            == reference_sequence[s_start + offset]
                        ):
                            identical += 1

                        aligned_length += 1

                identity = (
                    100 * identical / aligned_length
                    if aligned_length else 0
                )

                query_coverage = (
                    100 * aligned_query_positions / len(sequence)
                )

                organisms = metadata.get(
                    "rcsb_entity_source_organism", []
                )

                organism_names = [
                    item.get("ncbi_scientific_name", "Unknown")
                    for item in organisms
                ]

                rcsb_description = metadata.get(
                    "rcsb_polymer_entity", {}
                ).get("pdbx_description", "Unknown")

                rcsb_hits.append({
                    "query_id": query_id,
                    "source_chain": source_record["chain_id"],
                    "pdb_id": identifier.split("_")[0],
                    "entity_id": identifier.split("_")[1],
                    "protein_name": rcsb_description,
                    "organism": "; ".join(organism_names),
                    "identity_percentage": round(identity, 2),
                    "query_coverage_percentage": round(
                        query_coverage, 2
                    ),
                    "evidence_alignment_score": round(
                        alignment.score, 2
                    ),
                    "query_length": len(sequence),
                    "reference_length": len(reference_sequence)
                })

            except Exception as error:
                print("Metadata error:", identifier, error)

    except Exception as error:

        search_errors.append({
            "query_id": query_id,
            "error": str(error)
        })

        print("Search error:", error)


# --------------------------------------------------
# 5. Save and display results
# --------------------------------------------------

if rcsb_hits:

    rcsb_df = pd.DataFrame(rcsb_hits)

    rcsb_df = rcsb_df.sort_values(
        ["query_id", "identity_percentage",
         "query_coverage_percentage"],
        ascending=[True, False, False]
    )

    display(rcsb_df)

    output_file = (
        BLAST_DIR / "member1_rcsb_sequence_matches.csv"
    )

    rcsb_df.to_csv(output_file, index=False)

    print("\nResults saved:", output_file)

else:

    print("No matching experimental polymer entities found.")

if search_errors:
    print("\nSearch errors:")
    display(pd.DataFrame(search_errors))

print("\nRCSB sequence search completed.")


Query: M1_SEQUENCE_1
Chain: A
Length: 350
Matching polymer entities: 100

Query: M1_SEQUENCE_2
Chain: B
Length: 348
Matching polymer entities: 100


,query_id,source_chain,pdb_id,entity_id,protein_name,organism,identity_percentage,query_coverage_percentage,evidence_alignment_score,query_length,reference_length
0,M1_SEQUENCE_1,A,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.00,1796.0,350,350
1,M1_SEQUENCE_1,A,3GJW,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.00,1796.0,350,350
2,M1_SEQUENCE_1,A,3GN7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.00,1796.0,350,350
3,M1_SEQUENCE_1,A,3L3L,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.00,1796.0,350,350
4,M1_SEQUENCE_1,A,3L3M,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.00,1796.0,350,350
...,...,...,...,...,...,...,...,...,...,...,...
195,M1_SEQUENCE_2,B,4ZZY,1,POLY [ADP-RIBOSE] POLYMERASE 2,Homo sapiens,46.8,98.85,824.5,348,363
196,M1_SEQUENCE_2,B,6USJ,7,Poly [ADP-ribose] polymerase 2,Homo sapiens,46.8,98.85,824.5,348,590
197,M1_SEQUENCE_2,B,6X0L,2,Poly [ADP-ribose] polymerase 2,Homo sapiens,46.8,98.85,824.5,348,590
198,M1_SEQUENCE_2,B,6X0M,2,Poly [ADP-ribose] polymerase 2,Homo sapiens,46.8,98.85,824.5,348,590



Results saved: /content/drive/MyDrive/Member2_Project/member2_outputs/target_identification/member1_rcsb_sequence_matches.csv

RCSB sequence search completed.


In [23]:
# CELL 22: INSPECT TOP PROTEIN MATCHES

results_path = (
    Path(OUTPUT_DIR)
    / "target_identification"
    / "member1_rcsb_sequence_matches.csv"
)

rcsb_df = pd.read_csv(results_path)

for query_id in rcsb_df["query_id"].unique():

    print("\n======================================")
    print("QUERY:", query_id)
    print("======================================")

    subset = rcsb_df[
        rcsb_df["query_id"] == query_id
    ].sort_values(
        ["identity_percentage",
         "query_coverage_percentage",
         "evidence_alignment_score"],
        ascending=False
    )

    display(subset.head(15))


QUERY: M1_SEQUENCE_1


,query_id,source_chain,pdb_id,entity_id,protein_name,organism,identity_percentage,query_coverage_percentage,evidence_alignment_score,query_length,reference_length
0,M1_SEQUENCE_1,A,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
1,M1_SEQUENCE_1,A,3GJW,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
2,M1_SEQUENCE_1,A,3GN7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
3,M1_SEQUENCE_1,A,3L3L,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
4,M1_SEQUENCE_1,A,3L3M,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
5,M1_SEQUENCE_1,A,4DQY,3,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,506
6,M1_SEQUENCE_1,A,4GV7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,358
7,M1_SEQUENCE_1,A,4HHY,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,355
8,M1_SEQUENCE_1,A,4HHZ,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,355
9,M1_SEQUENCE_1,A,4L6S,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,353



QUERY: M1_SEQUENCE_2


,query_id,source_chain,pdb_id,entity_id,protein_name,organism,identity_percentage,query_coverage_percentage,evidence_alignment_score,query_length,reference_length
100,M1_SEQUENCE_2,B,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350
101,M1_SEQUENCE_2,B,3GJW,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350
102,M1_SEQUENCE_2,B,3GN7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350
103,M1_SEQUENCE_2,B,3L3L,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350
104,M1_SEQUENCE_2,B,3L3M,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350
105,M1_SEQUENCE_2,B,4DQY,3,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,506
106,M1_SEQUENCE_2,B,4GV7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,358
107,M1_SEQUENCE_2,B,4HHY,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,355
108,M1_SEQUENCE_2,B,4HHZ,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,355
109,M1_SEQUENCE_2,B,4L6S,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,353


In [24]:
# CELL 23 — Dynamic target identification from Member-1 sequence matches

from pathlib import Path
import pandas as pd
import json

TARGET_DIR = OUTPUT_DIR / "target_identification"
MATCH_FILE = TARGET_DIR / "member1_rcsb_sequence_matches.csv"

assert MATCH_FILE.exists(), f"Missing: {MATCH_FILE}"

matches = pd.read_csv(MATCH_FILE)

required = {
    "query_id", "source_chain", "pdb_id",
    "protein_name", "organism",
    "identity_percentage", "query_coverage_percentage"
}

assert required.issubset(matches.columns)

best_matches = (
    matches.sort_values(
        ["identity_percentage", "query_coverage_percentage"],
        ascending=False
    )
    .groupby("query_id")
    .first()
    .reset_index()
)

display(best_matches)

target_candidates = best_matches[
    (best_matches["identity_percentage"] >= 95) &
    (best_matches["query_coverage_percentage"] >= 90)
]

if len(target_candidates) == 0:
    TARGET_IDENTIFICATION_STATUS = "UNRESOLVED"
    print("No sufficiently strong sequence match.")
else:
    TARGET_IDENTIFICATION_STATUS = "IDENTIFIED"

    print("Target identification status:", TARGET_IDENTIFICATION_STATUS)
    print("Identified protein names:")
    print(target_candidates["protein_name"].unique())

    print("Organisms:")
    print(target_candidates["organism"].unique())

print("Target selection is based on Member-1 sequences.")

,query_id,source_chain,pdb_id,entity_id,protein_name,organism,identity_percentage,query_coverage_percentage,evidence_alignment_score,query_length,reference_length
0,M1_SEQUENCE_1,A,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
1,M1_SEQUENCE_2,B,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1775.5,348,350


Target identification status: IDENTIFIED
Identified protein names:
['Poly [ADP-ribose] polymerase 1']
Organisms:
['Homo sapiens']
Target selection is based on Member-1 sequences.


In [25]:
# CELL 24 — Retrieve experimental structure metadata

import requests
import time

assert TARGET_IDENTIFICATION_STATUS == "IDENTIFIED"

candidate_ids = (
    matches[
        (matches["identity_percentage"] >= 95) &
        (matches["query_coverage_percentage"] >= 90)
    ]["pdb_id"]
    .dropna()
    .astype(str)
    .str.upper()
    .unique()
    .tolist()
)

print("Candidate experimental structures:", len(candidate_ids))

def get_entry_metadata(pdb_id):

    url = (
        f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    )

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()

        data = response.json()

        info = data.get("rcsb_entry_info", {})

        experiments = data.get("exptl", [])

        methods = [
            x.get("method", "Unknown")
            for x in experiments
        ]

        return {
            "pdb_id": pdb_id,
            "experimental_methods": "; ".join(methods),
            "resolution": info.get("resolution_combined"),
            "nonpolymer_entity_count":
                info.get("nonpolymer_entity_count"),
            "status": "SUCCESS"
        }

    except Exception as e:

        return {
            "pdb_id": pdb_id,
            "status": "ERROR",
            "error": str(e)
        }

entry_rows = []

for i, pdb_id in enumerate(candidate_ids, start=1):

    entry_rows.append(get_entry_metadata(pdb_id))

    if i % 10 == 0:
        print(f"Processed {i}/{len(candidate_ids)}")

    time.sleep(0.05)

entry_df = pd.DataFrame(entry_rows)

ENTRY_FILE = TARGET_DIR / "experimental_structure_inventory.csv"

entry_df.to_csv(ENTRY_FILE, index=False)

display(entry_df.head(20))

print("Saved:", ENTRY_FILE)

Candidate experimental structures: 70
Processed 10/70
Processed 20/70
Processed 30/70
Processed 40/70
Processed 50/70
Processed 60/70
Processed 70/70


,pdb_id,experimental_methods,resolution,nonpolymer_entity_count,status
0,2RD6,X-RAY DIFFRACTION,[2.3],1,SUCCESS
1,3GJW,X-RAY DIFFRACTION,[2.3],1,SUCCESS
2,3GN7,X-RAY DIFFRACTION,[2.5],1,SUCCESS
3,3L3L,X-RAY DIFFRACTION,[2.5],1,SUCCESS
4,3L3M,X-RAY DIFFRACTION,[2.5],1,SUCCESS
5,4DQY,X-RAY DIFFRACTION,[3.25],2,SUCCESS
6,4GV7,X-RAY DIFFRACTION,[2.89],1,SUCCESS
7,4HHY,X-RAY DIFFRACTION,[2.3637],3,SUCCESS
8,4HHZ,X-RAY DIFFRACTION,[2.7199],2,SUCCESS
9,4L6S,X-RAY DIFFRACTION,[2.2],1,SUCCESS


Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/target_identification/experimental_structure_inventory.csv


In [26]:
# CELL 25 — Preliminary ligand-bearing structure candidates

successful_entries = entry_df[
    entry_df["status"] == "SUCCESS"
].copy()

successful_entries["nonpolymer_entity_count"] = pd.to_numeric(
    successful_entries["nonpolymer_entity_count"],
    errors="coerce"
)

ligand_candidates = successful_entries[
    successful_entries["nonpolymer_entity_count"].fillna(0) > 0
].copy()

print("Experimental structures:", len(successful_entries))
print("Structures with non-polymer entities:", len(ligand_candidates))

display(
    ligand_candidates.sort_values(
        "resolution",
        na_position="last"
    ).head(30)
)

ligand_candidates.to_csv(
    TARGET_DIR / "preliminary_ligand_structure_candidates.csv",
    index=False
)

Experimental structures: 70
Structures with non-polymer entities: 70


,pdb_id,experimental_methods,resolution,nonpolymer_entity_count,status
42,9ETQ,X-RAY DIFFRACTION,[1.59],2,SUCCESS
33,7AAC,X-RAY DIFFRACTION,[1.593],2,SUCCESS
61,7KK2,X-RAY DIFFRACTION,[1.695],1,SUCCESS
64,7KK5,X-RAY DIFFRACTION,[1.7],2,SUCCESS
31,7AAA,X-RAY DIFFRACTION,[1.74],3,SUCCESS
43,9ETR,X-RAY DIFFRACTION,[1.822],2,SUCCESS
38,7ONT,X-RAY DIFFRACTION,[1.853],2,SUCCESS
23,5WS1,X-RAY DIFFRACTION,[1.9],1,SUCCESS
58,4ZZZ,X-RAY DIFFRACTION,[1.9],3,SUCCESS
63,7KK4,X-RAY DIFFRACTION,[1.96],1,SUCCESS


In [27]:
# CELL 26 — Inspect current Member-1 pocket metadata

print("Selected pocket:", POCKET_ID)
print("Tensor shape:", tensor.shape)
print("Mask shape:", mask.shape)
print("Conformers:", NUM_CONFORMERS)

print("\nMetadata keys:")

if isinstance(metadata, dict):
    print(list(metadata.keys()))

feature_metadata = metadata.get("feature_metadata", [])

print("\nFeature metadata type:", type(feature_metadata))

if isinstance(feature_metadata, list):

    print("Number of metadata records:", len(feature_metadata))

    pocket_metadata_df = pd.json_normalize(feature_metadata)

    display(pocket_metadata_df.head(20))

    pocket_metadata_df.to_csv(
        OUTPUT_DIR / "member1_pocket_residue_metadata.csv",
        index=False
    )

elif isinstance(feature_metadata, dict):

    print("Feature metadata keys:")
    print(list(feature_metadata.keys()))

    display(pd.json_normalize(feature_metadata))

else:

    print("Feature metadata format needs inspection.")
    print(json.dumps(metadata, indent=2)[:5000])

Selected pocket: 1
Tensor shape: (5, 95, 4)
Mask shape: (5, 95)
Conformers: 5

Metadata keys:
['entity_poly', 'entity_src_gen', 'rcsb_entity_host_organism', 'rcsb_entity_source_organism', 'rcsb_polymer_entity', 'rcsb_polymer_entity_align', 'rcsb_polymer_entity_annotation', 'rcsb_polymer_entity_container_identifiers', 'rcsb_polymer_entity_feature', 'rcsb_polymer_entity_feature_summary', 'rcsb_polymer_entity_group_membership', 'rcsb_polymer_entity_name_com', 'rcsb_related_target_references', 'rcsb_target_cofactors', 'rcsb_id']

Feature metadata type: <class 'list'>
Number of metadata records: 0


""


In [28]:
# CELL 27 — Scientific training readiness gate

pocket_metadata_file = (
    OUTPUT_DIR / "member1_pocket_residue_metadata.csv"
)

preliminary_file = (
    TARGET_DIR / "preliminary_ligand_structure_candidates.csv"
)

checks = {
    "target_sequence_identified":
        TARGET_IDENTIFICATION_STATUS == "IDENTIFIED",

    "experimental_structures_found":
        len(successful_entries) > 0,

    "structures_with_nonpolymer_entities_found":
        len(ligand_candidates) > 0,

    "pocket_residue_metadata_available":
        pocket_metadata_file.exists()
}

print("========== TRAINING READINESS ==========")

for name, passed in checks.items():

    print(
        ("PASS" if passed else "PENDING"),
        name
    )

print("\nImportant:")
print("Non-polymer entity presence is NOT proof of inhibitor binding.")
print("Exact pocket-to-ligand contact verification is still required.")

VERIFIED_POCKET_LIGAND_PAIRS = []

if len(VERIFIED_POCKET_LIGAND_PAIRS) == 0:

    print("\nGENERATOR TRAINING: BLOCKED")
    print("Reason: no experimentally verified pocket-ligand pairs yet.")

else:

    print("\nVerified pairs:", len(VERIFIED_POCKET_LIGAND_PAIRS))

========== TRAINING READINESS ==========
PASS target_sequence_identified
PASS experimental_structures_found
PASS structures_with_nonpolymer_entities_found
PASS pocket_residue_metadata_available

Important:
Non-polymer entity presence is NOT proof of inhibitor binding.
Exact pocket-to-ligand contact verification is still required.

GENERATOR TRAINING: BLOCKED
Reason: no experimentally verified pocket-ligand pairs yet.


In [29]:
# CELL 28 — Reload original Member-1 pharmacophore metadata

import json
from pathlib import Path

json_files = list(M1_DIR.rglob("pharmacophore_tensors.json"))

print("Metadata files found:", len(json_files))

for f in json_files:
    print(f)

assert json_files, "Original Member-1 metadata JSON not found."

M1_METADATA_FILE = json_files[0]

with open(M1_METADATA_FILE, "r") as f:
    m1_metadata = json.load(f)

print("\nLoaded:", M1_METADATA_FILE)
print("\nTop-level keys:")
print(list(m1_metadata.keys()))

print("\nMetadata preview:")
print(json.dumps(m1_metadata, indent=2)[:6000])

Metadata files found: 1
/content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pharmacophore_tensors.json

Loaded: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pharmacophore_tensors.json

Top-level keys:
['1']

Metadata preview:
{
  "1": {
    "pocket_id": "1",
    "num_conformations": 5,
    "num_features": 95,
    "tensor_shape": [
      5,
      95,
      4
    ],
    "mask_shape": [
      5,
      95
    ],
    "feature_tensor": [
      [
        [
          25.07200050354004,
          40.76900100708008,
          -2.4719998836517334,
          2.0
        ],
        [
          24.82200050354004,
          40.86199951171875,
          0.14559999108314514,
          3.0
        ],
        [
          24.82200050354004,
          40.86199951171875,
          0.14559999108314514,
          4.0
        ],
        [
          27.454999923706055,
          40.000999450683594,
          -2.430000066757202,
          0.0
        ],
        [


In [30]:
# CELL 29 — Inspect nested Member-1 pocket metadata

pocket_id = str(POCKET_ID)

assert pocket_id in m1_metadata, (
    f"Pocket {pocket_id} not found. "
    f"Available pockets: {list(m1_metadata.keys())}"
)

current_pocket = m1_metadata[pocket_id]

print("Pocket ID:", pocket_id)
print("\nPocket-level keys:")

for key, value in current_pocket.items():

    if isinstance(value, (list, dict)):
        print(f"{key}: {type(value).__name__}, length={len(value)}")
    else:
        print(f"{key}: {value}")

print("\nFeature metadata:")

feature_metadata = current_pocket.get("feature_metadata")

if feature_metadata is not None:
    print(json.dumps(feature_metadata, indent=2)[:10000])
else:
    print("No feature_metadata field found.")

print("\nOther possible residue information:")

for key, value in current_pocket.items():

    if any(word in key.lower() for word in
           ["residue", "atom", "chain", "feature"]):

        print("\n", key)
        print(str(value)[:3000])

Pocket ID: 1

Pocket-level keys:
pocket_id: 1
num_conformations: 5
num_features: 95
tensor_shape: list, length=3
mask_shape: list, length=2
feature_tensor: list, length=5
mask: list, length=5
feature_metadata: list, length=95

Feature metadata:
[
  {
    "feature_index": 0,
    "identity": "A|826||HIS|CB",
    "feature": "HYDROPHOBIC",
    "feature_id": 2
  },
  {
    "feature_index": 1,
    "identity": "A|826||HIS|CD2;A|826||HIS|CE1;A|826||HIS|CG;A|826||HIS|ND1;A|826||HIS|NE2",
    "feature": "AROMATIC",
    "feature_id": 3
  },
  {
    "feature_index": 2,
    "identity": "A|826||HIS|CD2;A|826||HIS|CE1;A|826||HIS|CG;A|826||HIS|ND1;A|826||HIS|NE2",
    "feature": "POSITIVE",
    "feature_id": 4
  },
  {
    "feature_index": 3,
    "identity": "A|826||HIS|N",
    "feature": "DONOR",
    "feature_id": 0
  },
  {
    "feature_index": 4,
    "identity": "A|826||HIS|ND1",
    "feature": "DONOR",
    "feature_id": 0
  },
  {
    "feature_index": 5,
    "identity": "A|826||HIS|ND1",
    "feat

In [31]:
# CELL 30 — Extract and validate Member-1 pocket residues

import re
import pandas as pd
from Bio.PDB import PDBParser
from pathlib import Path

feature_metadata = current_pocket["feature_metadata"]

records = []

for item in feature_metadata:

    identity = item.get("identity", "")

    for atom_identity in identity.split(";"):

        parts = atom_identity.split("|")

        if len(parts) != 5:
            continue

        chain, residue_number, insertion_code, residue_name, atom_name = parts

        records.append({
            "chain": chain,
            "residue_number": int(residue_number),
            "insertion_code": insertion_code,
            "residue_name": residue_name,
            "atom_name": atom_name,
            "feature": item.get("feature"),
            "feature_index": item.get("feature_index")
        })

feature_atoms_df = pd.DataFrame(records)

# Unique pocket residues
pocket_residues_df = (
    feature_atoms_df[
        ["chain", "residue_number", "insertion_code", "residue_name"]
    ]
    .drop_duplicates()
    .sort_values(["chain", "residue_number"])
    .reset_index(drop=True)
)

print("Total feature metadata records:", len(feature_metadata))
print("Unique feature-atom references:", len(feature_atoms_df))
print("Unique pocket residues:", len(pocket_residues_df))

print("\nPocket residues by chain:")
display(
    pocket_residues_df.groupby("chain").size().reset_index(name="residue_count")
)

print("\nPocket residue list:")
display(pocket_residues_df)

# Validate against first Member-1 conformer
pdb_files = sorted(M1_DIR.rglob("conformer_001.pdb"))

assert pdb_files, "conformer_001.pdb not found."

parser = PDBParser(QUIET=True)
structure = parser.get_structure("M1", str(pdb_files[0]))

pdb_atoms = set()
pdb_residues = set()

for chain_obj in structure[0]:

    for residue in chain_obj:

        if residue.id[0] != " ":
            continue

        chain_id = chain_obj.id
        residue_number = residue.id[1]
        insertion_code = residue.id[2].strip()
        residue_name = residue.resname

        pdb_residues.add(
            (chain_id, residue_number, insertion_code, residue_name)
        )

        for atom in residue:
            pdb_atoms.add(
                (chain_id, residue_number, insertion_code,
                 residue_name, atom.name)
            )

# Check metadata residue and atom identities
pocket_residues_df["found_in_pdb"] = pocket_residues_df.apply(
    lambda row: (
        row["chain"],
        row["residue_number"],
        row["insertion_code"],
        row["residue_name"]
    ) in pdb_residues,
    axis=1
)

feature_atoms_df["found_in_pdb"] = feature_atoms_df.apply(
    lambda row: (
        row["chain"],
        row["residue_number"],
        row["insertion_code"],
        row["residue_name"],
        row["atom_name"]
    ) in pdb_atoms,
    axis=1
)

print("\nResidue validation:")
print(pocket_residues_df["found_in_pdb"].value_counts())

print("\nAtom validation:")
print(feature_atoms_df["found_in_pdb"].value_counts())

print("\nMissing residues:")
display(pocket_residues_df[~pocket_residues_df["found_in_pdb"]])

print("\nMissing atoms:")
display(feature_atoms_df[~feature_atoms_df["found_in_pdb"]].head(20))

# Save reports
REPORT_DIR = Path(OUTPUT_DIR) / "target_identification"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pocket_residues_df.to_csv(
    REPORT_DIR / "member1_pocket_residues.csv",
    index=False
)

feature_atoms_df.to_csv(
    REPORT_DIR / "member1_pocket_feature_atoms.csv",
    index=False
)

print("\nSaved pocket residue and atom reports.")

Total feature metadata records: 95
Unique feature-atom references: 137
Unique pocket residues: 11

Pocket residues by chain:


,chain,residue_count
0,A,10
1,B,1



Pocket residue list:


,chain,residue_number,insertion_code,residue_name
0,A,826,,HIS
1,A,903,,LYS
2,A,906,,ASN
3,A,907,,TYR
4,A,909,,HIS
5,A,983,,SER
6,A,984,,LEU
7,A,985,,LEU
8,A,986,,TYR
9,A,988,,GLU



Residue validation:
found_in_pdb
True    11
Name: count, dtype: int64

Atom validation:
found_in_pdb
True    137
Name: count, dtype: int64

Missing residues:


,chain,residue_number,insertion_code,residue_name,found_in_pdb



Missing atoms:


,chain,residue_number,insertion_code,residue_name,atom_name,feature,feature_index,found_in_pdb



Saved pocket residue and atom reports.


In [32]:
# CELL 31 — Analyze spatial relationship between pocket chains

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from Bio.PDB import PDBParser

# Load first M1 conformer
pdb_file = sorted(M1_DIR.rglob("conformer_001.pdb"))[0]

structure = PDBParser(QUIET=True).get_structure(
    "M1_SPATIAL", str(pdb_file)
)

model = structure[0]

def get_heavy_atoms(chain, residue_numbers=None):

    coordinates = []
    labels = []

    for residue in model[chain]:

        if residue.id[0] != " ":
            continue

        if residue_numbers is not None:
            if residue.id[1] not in residue_numbers:
                continue

        for atom in residue:

            if atom.element.upper() in ["C", "N", "O", "S", "P"]:
                coordinates.append(atom.coord)

                labels.append({
                    "chain": chain,
                    "residue": residue.resname,
                    "residue_number": residue.id[1],
                    "atom": atom.name
                })

    return np.array(coordinates), labels


# Pocket residues belonging to each chain
chain_a_residues = pocket_residues_df.loc[
    pocket_residues_df["chain"] == "A",
    "residue_number"
].tolist()

chain_b_residues = pocket_residues_df.loc[
    pocket_residues_df["chain"] == "B",
    "residue_number"
].tolist()

coords_a, labels_a = get_heavy_atoms("A", chain_a_residues)
coords_b, labels_b = get_heavy_atoms("B", chain_b_residues)

assert len(coords_a) > 0 and len(coords_b) > 0

# Minimum distances between pocket atoms from the two chains
distance_matrix = cdist(coords_a, coords_b)

nearest_pairs = []

for i, j in np.argwhere(distance_matrix < 10.0):

    nearest_pairs.append({
        "chain_A_residue": labels_a[i]["residue"],
        "chain_A_number": labels_a[i]["residue_number"],
        "chain_A_atom": labels_a[i]["atom"],
        "chain_B_residue": labels_b[j]["residue"],
        "chain_B_number": labels_b[j]["residue_number"],
        "chain_B_atom": labels_b[j]["atom"],
        "distance_angstrom": distance_matrix[i, j]
    })

nearest_pairs_df = pd.DataFrame(nearest_pairs)

print("Chain A pocket heavy atoms:", len(coords_a))
print("Chain B pocket heavy atoms:", len(coords_b))

print("\nMinimum distance between the two chain contributions:")
print(round(float(distance_matrix.min()), 3), "Å")

print("\nClosest cross-chain atom pairs:")
if not nearest_pairs_df.empty:
    display(
        nearest_pairs_df.sort_values(
            "distance_angstrom"
        ).head(20)
    )

# Check feature coordinates associated with Chain B
tensor = np.asarray(current_pocket["feature_tensor"])

chain_b_feature_indices = [
    item["feature_index"]
    for item in feature_metadata
    if any(
        identity.startswith("B|")
        for identity in item["identity"].split(";")
    )
]

print("\nChain B feature indices:", chain_b_feature_indices)

for conf_index in range(tensor.shape[0]):

    feature_coords = tensor[conf_index, :, :3]

    print(
        f"Conformer {conf_index + 1}: "
        f"Chain B feature coordinates =",
        feature_coords[chain_b_feature_indices]
    )

Chain A pocket heavy atoms: 92
Chain B pocket heavy atoms: 6

Minimum distance between the two chain contributions:
3.323 Å

Closest cross-chain atom pairs:


,chain_A_residue,chain_A_number,chain_A_atom,chain_B_residue,chain_B_number,chain_B_atom,distance_angstrom
28,SER,983,O,SER,721,CB,3.323317
29,SER,983,O,SER,721,OG,3.341657
27,SER,983,O,SER,721,O,3.438383
17,SER,983,CA,SER,721,OG,3.443755
35,SER,983,CB,SER,721,OG,3.541091
34,SER,983,CB,SER,721,CB,3.599472
26,SER,983,O,SER,721,C,3.622271
23,SER,983,C,SER,721,OG,3.826164
16,SER,983,CA,SER,721,CB,3.915101
22,SER,983,C,SER,721,CB,4.049523



Chain B feature indices: [91, 92, 93, 94]
Conformer 1: Chain B feature coordinates = [[27.53000069 32.89199829 11.04100037]
 [26.48999977 35.73199844  9.19799995]
 [28.59300041 35.25099945 12.38300037]
 [28.59300041 35.25099945 12.38300037]]
Conformer 2: Chain B feature coordinates = [[28.10300064 32.84400177 10.91800022]
 [26.98200035 35.68999863  9.04300022]
 [29.06900024 35.23099899 12.27200031]
 [29.06900024 35.23099899 12.27200031]]
Conformer 3: Chain B feature coordinates = [[27.67200089 32.66600037 10.86499977]
 [26.62800026 35.5379982   8.96399975]
 [28.72299957 35.04499817 12.1789999 ]
 [28.72299957 35.04499817 12.1789999 ]]
Conformer 4: Chain B feature coordinates = [[27.09900093 32.7140007  10.98799992]
 [26.13599968 35.58000183  9.11900043]
 [28.24699974 35.06499863 12.28999996]
 [28.24699974 35.06499863 12.28999996]]
Conformer 5: Chain B feature coordinates = [[27.60099983 32.77899933 10.95300007]
 [26.55900002 35.63499832  9.08100033]
 [28.65800095 35.14799881 12.2810001

In [33]:
# CELL 32 — Inspect experimental structure and sequence mapping files

from pathlib import Path
import pandas as pd

target_dir = Path(OUTPUT_DIR) / "target_identification"

files_to_check = [
    "experimental_structure_inventory.csv",
    "member1_rcsb_sequence_matches.csv",
    "preliminary_ligand_structure_candidates.csv"
]

for filename in files_to_check:

    path = target_dir / filename

    print("\n" + "=" * 65)
    print(filename)

    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


experimental_structure_inventory.csv
Shape: (70, 5)
Columns: ['pdb_id', 'experimental_methods', 'resolution', 'nonpolymer_entity_count', 'status']


,pdb_id,experimental_methods,resolution,nonpolymer_entity_count,status
0,2RD6,X-RAY DIFFRACTION,[2.3],1,SUCCESS
1,3GJW,X-RAY DIFFRACTION,[2.3],1,SUCCESS
2,3GN7,X-RAY DIFFRACTION,[2.5],1,SUCCESS



member1_rcsb_sequence_matches.csv
Shape: (200, 11)
Columns: ['query_id', 'source_chain', 'pdb_id', 'entity_id', 'protein_name', 'organism', 'identity_percentage', 'query_coverage_percentage', 'evidence_alignment_score', 'query_length', 'reference_length']


,query_id,source_chain,pdb_id,entity_id,protein_name,organism,identity_percentage,query_coverage_percentage,evidence_alignment_score,query_length,reference_length
0,M1_SEQUENCE_1,A,2RD6,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
1,M1_SEQUENCE_1,A,3GJW,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350
2,M1_SEQUENCE_1,A,3GN7,1,Poly [ADP-ribose] polymerase 1,Homo sapiens,100.0,100.0,1796.0,350,350



preliminary_ligand_structure_candidates.csv
Shape: (70, 5)
Columns: ['pdb_id', 'experimental_methods', 'resolution', 'nonpolymer_entity_count', 'status']


,pdb_id,experimental_methods,resolution,nonpolymer_entity_count,status
0,2RD6,X-RAY DIFFRACTION,[2.3],1,SUCCESS
1,3GJW,X-RAY DIFFRACTION,[2.3],1,SUCCESS
2,3GN7,X-RAY DIFFRACTION,[2.5],1,SUCCESS


In [34]:
# CELL 33 — Inspect non-polymer molecules in experimental PARP1 structures

import requests
import pandas as pd
from pathlib import Path
import time

target_dir = Path(OUTPUT_DIR) / "target_identification"

inventory = pd.read_csv(
    target_dir / "experimental_structure_inventory.csv"
)

pdb_ids = inventory["pdb_id"].astype(str).str.upper().unique()

session = requests.Session()
session.headers.update({"User-Agent": "Member2-StructureAnalysis/1.0"})

entity_records = []
instance_records = []
errors = []

def get_rcsb_json(url):
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

for pdb_id in pdb_ids:

    try:
        entry = get_rcsb_json(
            f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
        )

        identifiers = entry.get(
            "rcsb_entry_container_identifiers", {}
        )

        entity_ids = identifiers.get(
            "non_polymer_entity_ids", []
        )

        for entity_id in entity_ids:

            entity_id = str(entity_id)

            entity = get_rcsb_json(
                f"https://data.rcsb.org/rest/v1/core/"
                f"nonpolymer_entity/{pdb_id}/{entity_id}"
            )

            comp_ids = entity.get(
                "pdbx_entity_nonpoly", {}
            )

            chemical = entity.get(
                "rcsb_nonpolymer_entity", {}
            )

            chem_comp_id = (
                comp_ids.get("comp_id")
                if isinstance(comp_ids, dict)
                else None
            )

            entity_records.append({
                "pdb_id": pdb_id,
                "entity_id": entity_id,
                "comp_id": chem_comp_id,
                "description": (
                    comp_ids.get("name")
                    if isinstance(comp_ids, dict) else None
                ),
                "formula": (
                    chemical.get("formula_weight")
                    if isinstance(chemical, dict) else None
                )
            })

            # Retrieve the actual chain instances of this molecule
            instance_ids = (
                entity.get(
                    "rcsb_nonpolymer_entity_container_identifiers", {}
                ).get("as_ids", [])
            )

            for asym_id in instance_ids:

                instance = get_rcsb_json(
                    f"https://data.rcsb.org/rest/v1/core/"
                    f"nonpolymer_entity_instance/{pdb_id}/{asym_id}"
                )

                instance_ids_data = instance.get(
                    "rcsb_nonpolymer_entity_instance_container_identifiers",
                    {}
                )

                instance_records.append({
                    "pdb_id": pdb_id,
                    "entity_id": entity_id,
                    "comp_id": chem_comp_id,
                    "label_asym_id": asym_id,
                    "auth_asym_id": instance_ids_data.get(
                        "auth_asym_id"
                    ),
                    "auth_seq_id": instance_ids_data.get(
                        "auth_seq_id"
                    )
                })

        print(
            f"{pdb_id}: {len(entity_ids)} non-polymer entities inspected"
        )

    except Exception as exc:
        errors.append({
            "pdb_id": pdb_id,
            "error": str(exc)
        })

    time.sleep(0.05)

entities_df = pd.DataFrame(entity_records)
instances_df = pd.DataFrame(instance_records)
errors_df = pd.DataFrame(errors)

print("\nEntity records:", len(entities_df))
print("Molecule instances:", len(instances_df))
print("Failed structures:", len(errors_df))

if not entities_df.empty:
    display(entities_df.head(30))

if not instances_df.empty:
    display(instances_df.head(30))

if not errors_df.empty:
    display(errors_df)

entities_df.to_csv(
    target_dir / "experimental_nonpolymer_entities.csv",
    index=False
)

instances_df.to_csv(
    target_dir / "experimental_nonpolymer_instances.csv",
    index=False
)

errors_df.to_csv(
    target_dir / "experimental_nonpolymer_errors.csv",
    index=False
)

print("\nSaved experimental molecule inventory.")

2RD6: 1 non-polymer entities inspected
3GJW: 1 non-polymer entities inspected
3GN7: 1 non-polymer entities inspected
3L3L: 1 non-polymer entities inspected
3L3M: 1 non-polymer entities inspected
4DQY: 2 non-polymer entities inspected
4GV7: 1 non-polymer entities inspected
4HHY: 3 non-polymer entities inspected
4HHZ: 2 non-polymer entities inspected
4L6S: 1 non-polymer entities inspected
4R5W: 2 non-polymer entities inspected
4R6E: 3 non-polymer entities inspected
4RV6: 2 non-polymer entities inspected
4UND: 2 non-polymer entities inspected
4UXB: 2 non-polymer entities inspected
5KPN: 1 non-polymer entities inspected
5KPO: 1 non-polymer entities inspected
5KPP: 1 non-polymer entities inspected
5KPQ: 1 non-polymer entities inspected
5WRQ: 1 non-polymer entities inspected
5WRY: 1 non-polymer entities inspected
5WRZ: 1 non-polymer entities inspected
5WS0: 1 non-polymer entities inspected
5WS1: 1 non-polymer entities inspected
5WTC: 1 non-polymer entities inspected
5XSR: 2 non-polymer entit

,pdb_id,entity_id,comp_id,description,formula
0,2RD6,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,0.244
1,3GJW,2,GJW,"7-(pyrrolidin-1-ylmethyl)pyrrolo[1,2-a]quinoxa...",0.267
2,3GN7,2,3GN,2-[(2S)-2-methylpyrrolidin-2-yl]-1H-benzimidaz...,0.244
3,3L3L,2,L3L,"3-oxo-2-piperidin-4-yl-2,3-dihydro-1H-isoindol...",0.259
4,3L3M,2,A92,2-{2-fluoro-4-[(2S)-piperidin-2-yl]phenyl}-1H-...,0.338
5,4DQY,5,ZN,ZINC ION,0.065
6,4DQY,6,EDO,"1,2-ETHANEDIOL",0.062
7,4GV7,2,MEW,2-methylquinazolin-4(3H)-one,0.160
8,4HHY,2,15R,"(9aR)-1-[(1-{2-fluoro-5-[(4-oxo-3,4-dihydropht...",0.580
9,4HHY,3,SO4,SULFATE ION,0.096



Saved experimental molecule inventory.


In [35]:
# CELL 34 — Correctly discover non-polymer ligand instances

import requests
import pandas as pd
from pathlib import Path

session = requests.Session()
session.headers.update({"User-Agent": "Member2-StructureAnalysis/1.0"})

instance_records = []
debug_records = []

for _, row in entities_df.iterrows():

    pdb_id = str(row["pdb_id"])
    entity_id = str(row["entity_id"])

    try:
        url = (
            f"https://data.rcsb.org/rest/v1/core/"
            f"nonpolymer_entity/{pdb_id}/{entity_id}"
        )

        response = session.get(url, timeout=30)
        response.raise_for_status()
        entity = response.json()

        identifiers = entity.get(
            "rcsb_nonpolymer_entity_container_identifiers", {}
        )

        if not isinstance(identifiers, dict):
            identifiers = {}

        # RCSB commonly uses asym_ids; check alternatives too.
        asym_ids = (
            identifiers.get("asym_ids")
            or identifiers.get("as_ids")
            or []
        )

        auth_asym_ids = identifiers.get("auth_asym_ids", [])

        if isinstance(asym_ids, str):
            asym_ids = [asym_ids]

        if isinstance(auth_asym_ids, str):
            auth_asym_ids = [auth_asym_ids]

        if not asym_ids:
            debug_records.append({
                "pdb_id": pdb_id,
                "entity_id": entity_id,
                "identifier_keys": list(identifiers.keys()),
                "available_data": str(identifiers)[:1000]
            })
            continue

        for index, asym_id in enumerate(asym_ids):

            instance_records.append({
                "pdb_id": pdb_id,
                "entity_id": entity_id,
                "comp_id": row["comp_id"],
                "description": row["description"],
                "label_asym_id": asym_id,
                "auth_asym_id": (
                    auth_asym_ids[index]
                    if index < len(auth_asym_ids)
                    else None
                )
            })

    except Exception as exc:
        debug_records.append({
            "pdb_id": pdb_id,
            "entity_id": entity_id,
            "error": str(exc)
        })

instances_df = pd.DataFrame(instance_records)
instance_debug_df = pd.DataFrame(debug_records)

print("Total ligand instances:", len(instances_df))
print("Entities without discovered instances:", len(instance_debug_df))

if not instances_df.empty:
    display(instances_df.head(30))

if not instance_debug_df.empty:
    display(instance_debug_df.head(10))

target_dir = Path(OUTPUT_DIR) / "target_identification"

instances_df.to_csv(
    target_dir / "experimental_nonpolymer_instances.csv",
    index=False
)

instance_debug_df.to_csv(
    target_dir / "experimental_nonpolymer_instance_debug.csv",
    index=False
)

print("\nInstance report saved.")

Total ligand instances: 417
Entities without discovered instances: 0


,pdb_id,entity_id,comp_id,description,label_asym_id,auth_asym_id
0,2RD6,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,B,A
1,3GJW,2,GJW,"7-(pyrrolidin-1-ylmethyl)pyrrolo[1,2-a]quinoxa...",B,A
2,3GN7,2,3GN,2-[(2S)-2-methylpyrrolidin-2-yl]-1H-benzimidaz...,B,A
3,3L3L,2,L3L,"3-oxo-2-piperidin-4-yl-2,3-dihydro-1H-isoindol...",B,A
4,3L3M,2,A92,2-{2-fluoro-4-[(2S)-piperidin-2-yl]phenyl}-1H-...,B,A
5,4DQY,5,ZN,ZINC ION,I,A
6,4DQY,5,ZN,ZINC ION,J,B
7,4DQY,5,ZN,ZINC ION,L,D
8,4DQY,5,ZN,ZINC ION,M,E
9,4DQY,6,EDO,"1,2-ETHANEDIOL",K,C



Instance report saved.


In [36]:
# CELL 35 — Classify non-polymer entities

import pandas as pd
from pathlib import Path

# Common ions, solvents, buffers and crystallization additives.
excluded_comp_ids = {
    "HOH", "DOD",
    "ZN", "MG", "CA", "NA", "K", "CL",
    "MN", "FE", "CU", "CO", "NI",
    "SO4", "PO4", "NO3",
    "GOL", "EDO", "PEG", "PG4", "PGE",
    "ACT", "ACE", "FMT", "TRS", "MES",
    "HEP", "BME", "DTT", "MPD",
    "DMS", "IPA", "EOH", "MOH",
    "CIT", "TAR", "MAL", "IOD"
}

# One record per unique chemical entity in each structure.
unique_entities = entities_df.drop_duplicates(
    subset=["pdb_id", "entity_id"]
).copy()

unique_entities["comp_id"] = (
    unique_entities["comp_id"].fillna("").astype(str).str.upper()
)

unique_entities["description"] = (
    unique_entities["description"].fillna("").astype(str)
)

unique_entities["preliminary_class"] = unique_entities.apply(
    lambda row: (
        "exclude_additive_or_ion"
        if row["comp_id"] in excluded_comp_ids
        else "potential_ligand_candidate"
    ),
    axis=1
)

# Count how many actual instances each chemical entity has.
instance_counts = (
    instances_df.groupby(["pdb_id", "entity_id"])
    .size()
    .reset_index(name="instance_count")
)

unique_entities = unique_entities.merge(
    instance_counts,
    on=["pdb_id", "entity_id"],
    how="left"
)

unique_entities["instance_count"] = (
    unique_entities["instance_count"].fillna(0).astype(int)
)

candidate_entities = unique_entities[
    unique_entities["preliminary_class"] ==
    "potential_ligand_candidate"
].copy()

excluded_entities = unique_entities[
    unique_entities["preliminary_class"] ==
    "exclude_additive_or_ion"
].copy()

print("Unique chemical entities:", len(unique_entities))
print("Potential ligand candidates:", len(candidate_entities))
print("Excluded additives/ions:", len(excluded_entities))

print("\nPotential ligand candidates:")
display(
    candidate_entities[
        [
            "pdb_id",
            "entity_id",
            "comp_id",
            "description",
            "instance_count"
        ]
    ].sort_values(["pdb_id", "comp_id"]).reset_index(drop=True)
)

target_dir = Path(OUTPUT_DIR) / "target_identification"

candidate_entities.to_csv(
    target_dir / "potential_ligand_candidates.csv",
    index=False
)

excluded_entities.to_csv(
    target_dir / "excluded_additives_and_ions.csv",
    index=False
)

print("\nSaved preliminary ligand classification.")

Unique chemical entities: 124
Potential ligand candidates: 64
Excluded additives/ions: 60

Potential ligand candidates:


,pdb_id,entity_id,comp_id,description,instance_count
0,1UK0,2,FRM,"2-{3-[4-(4-FLUOROPHENYL)-3,6-DIHYDRO-1(2H)-PYR...",2
1,1UK1,2,FRQ,"5-FLUORO-1-[4-(4-PHENYL-3,6-DIHYDROPYRIDIN-1(2...",2
2,1WOK,2,CNQ,3-(4-CHLOROPHENYL)QUINOXALINE-5-CARBOXAMIDE,4
3,2RCW,2,AAI,trans-4-(7-carbamoyl-1H-benzimidazol-2-yl)-1-p...,1
4,2RD6,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,1
...,...,...,...,...,...
59,8VYH,2,A1AEO,"(4S)-1-{2-fluoro-5-[(4-oxo-3,4-dihydrophthalaz...",2
60,9ETQ,2,A1H63,"5-[4-[(7-ethyl-6-oxidanylidene-5~{H}-1,5-napht...",2
61,9ETR,2,A1H64,6-fluoranyl-5-[4-[(5-fluoranyl-2-methyl-3-oxid...,2
62,9ILN,2,A1D9M,5-ethyl-1-[[3-[(3~{R})-4-ethyl-3-(2-hydroxyeth...,2



Saved preliminary ligand classification.


In [37]:
# CELL 36 — Match candidate structures to M1 protein sequences

import pandas as pd
from pathlib import Path

target_dir = Path(OUTPUT_DIR) / "target_identification"

# Load existing sequence matching results
sequence_matches = pd.read_csv(
    target_dir / "member1_rcsb_sequence_matches.csv"
)

# Use the 64 candidate ligand structures
candidates = pd.read_csv(
    target_dir / "potential_ligand_candidates.csv"
)

# Retain strong sequence matches
strong_matches = sequence_matches[
    (sequence_matches["identity_percentage"] >= 95) &
    (sequence_matches["query_coverage_percentage"] >= 90)
].copy()

# Match candidate PDB IDs with their experimental protein chains
candidate_structure_matches = candidates.merge(
    strong_matches,
    on="pdb_id",
    how="left"
)

# Identify structures with at least one strong protein sequence match
candidate_structure_matches["sequence_match_status"] = (
    candidate_structure_matches["query_id"]
    .notna()
    .map({
        True: "STRONG_PAR P1_SEQUENCE_MATCH",
        False: "NO_STRONG_MATCH_FOUND"
    })
)

print("Potential ligand entities:", len(candidates))

print(
    "Candidate entities with strong PARP1 sequence matches:",
    candidate_structure_matches[
        candidate_structure_matches["query_id"].notna()
    ].shape[0]
)

print(
    "Candidate entities without strong matches:",
    candidate_structure_matches[
        candidate_structure_matches["query_id"].isna()
    ].shape[0]
)

display(
    candidate_structure_matches[
        [
            "pdb_id",
            "comp_id",
            "description",
            "instance_count",
            "query_id",
            "source_chain",
            "entity_id_y",
            "identity_percentage",
            "query_coverage_percentage",
            "sequence_match_status"
        ]
    ].head(40)
)

candidate_structure_matches.to_csv(
    target_dir / "candidate_parp1_sequence_matches.csv",
    index=False
)

print("\nSaved candidate-to-PARP1 sequence mapping.")

Potential ligand entities: 64
Candidate entities with strong PARP1 sequence matches: 128
Candidate entities without strong matches: 0


,pdb_id,comp_id,description,instance_count,query_id,source_chain,entity_id_y,identity_percentage,query_coverage_percentage,sequence_match_status
0,2RD6,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,1,M1_SEQUENCE_1,A,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
1,2RD6,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,1,M1_SEQUENCE_2,B,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
2,3GJW,GJW,"7-(pyrrolidin-1-ylmethyl)pyrrolo[1,2-a]quinoxa...",1,M1_SEQUENCE_1,A,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
3,3GJW,GJW,"7-(pyrrolidin-1-ylmethyl)pyrrolo[1,2-a]quinoxa...",1,M1_SEQUENCE_2,B,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
4,3GN7,3GN,2-[(2S)-2-methylpyrrolidin-2-yl]-1H-benzimidaz...,1,M1_SEQUENCE_1,A,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
5,3GN7,3GN,2-[(2S)-2-methylpyrrolidin-2-yl]-1H-benzimidaz...,1,M1_SEQUENCE_2,B,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
6,3L3L,L3L,"3-oxo-2-piperidin-4-yl-2,3-dihydro-1H-isoindol...",1,M1_SEQUENCE_1,A,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
7,3L3L,L3L,"3-oxo-2-piperidin-4-yl-2,3-dihydro-1H-isoindol...",1,M1_SEQUENCE_2,B,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
8,3L3M,A92,2-{2-fluoro-4-[(2S)-piperidin-2-yl]phenyl}-1H-...,1,M1_SEQUENCE_1,A,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH
9,3L3M,A92,2-{2-fluoro-4-[(2S)-piperidin-2-yl]phenyl}-1H-...,1,M1_SEQUENCE_2,B,1,100.0,100.0,STRONG_PAR P1_SEQUENCE_MATCH



Saved candidate-to-PARP1 sequence mapping.


In [38]:
# CELL 37 — Experimental protein-chain retrieval and sequence alignment

import requests
import pandas as pd
import numpy as np
import time

from pathlib import Path
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner, substitution_matrices

# --------------------------------------------------
# 1. Locate project files
# --------------------------------------------------

project_root = Path(OUTPUT_DIR).parent
target_dir = Path(OUTPUT_DIR) / "target_identification"

m1_pdb_files = list(
    (project_root / "member1_input").rglob("conformer_001.pdb")
)

if not m1_pdb_files:
    raise FileNotFoundError("Member-1 conformer_001.pdb not found.")

m1_pdb_path = m1_pdb_files[0]

print("M1 structure:", m1_pdb_path)

# --------------------------------------------------
# 2. Extract Member-1 protein-chain sequences
# --------------------------------------------------

parser = PDBParser(QUIET=True)
m1_structure = parser.get_structure("M1", str(m1_pdb_path))

m1_sequences = {}

for chain in m1_structure[0]:

    sequence = ""

    for residue in chain:

        if residue.id[0] != " ":
            continue

        amino_acid = seq1(
            residue.resname,
            custom_map={"UNK": "X"}
        )

        if amino_acid:
            sequence += amino_acid

    if sequence:
        m1_sequences[str(chain.id)] = sequence

print("\nMember-1 sequences:")

for chain_id, sequence in m1_sequences.items():
    print(
        f"Chain {chain_id}: "
        f"{len(sequence)} amino acids"
    )

if not m1_sequences:
    raise ValueError("Could not extract M1 protein sequences.")

# --------------------------------------------------
# 3. Configure sequence alignment
# --------------------------------------------------

aligner = PairwiseAligner()

aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

def calculate_alignment(query, target):

    alignment = aligner.align(query, target)[0]

    matches = 0
    aligned_residues = 0

    # alignment.indices maps query and target positions.
    for query_index, target_index in zip(
        alignment.indices[0],
        alignment.indices[1]
    ):

        if query_index < 0 or target_index < 0:
            continue

        aligned_residues += 1

        if query[query_index] == target[target_index]:
            matches += 1

    identity = (
        100 * matches / aligned_residues
        if aligned_residues else 0
    )

    query_coverage = (
        100 * aligned_residues / len(query)
        if query else 0
    )

    return {
        "identity_percentage": identity,
        "query_coverage_percentage": query_coverage,
        "aligned_residues": aligned_residues,
        "alignment_score": alignment.score
    }

# --------------------------------------------------
# 4. Retrieve candidate experimental structures
# --------------------------------------------------

candidates = pd.read_csv(
    target_dir / "potential_ligand_candidates.csv"
)

pdb_ids = sorted(
    candidates["pdb_id"].astype(str).str.upper().unique()
)

session = requests.Session()

session.headers.update({
    "User-Agent": "Member2-ProteinChainAlignment/1.0"
})

def get_rcsb_json(url):

    response = session.get(url, timeout=30)
    response.raise_for_status()

    return response.json()

experimental_chains = []
errors = []

for pdb_id in pdb_ids:

    try:

        entry = get_rcsb_json(
            f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
        )

        entry_ids = entry.get(
            "rcsb_entry_container_identifiers", {}
        )

        polymer_entity_ids = entry_ids.get(
            "polymer_entity_ids", []
        )

        for entity_id in polymer_entity_ids:

            entity_id = str(entity_id)

            entity = get_rcsb_json(
                f"https://data.rcsb.org/rest/v1/core/"
                f"polymer_entity/{pdb_id}/{entity_id}"
            )

            entity_poly = entity.get("entity_poly", {})

            sequence = entity_poly.get(
                "pdbx_seq_one_letter_code_can"
            )

            if not sequence:
                continue

            sequence = "".join(sequence.split())

            identifiers = entity.get(
                "rcsb_polymer_entity_container_identifiers", {}
            )

            asym_ids = identifiers.get("asym_ids", [])

            if isinstance(asym_ids, str):
                asym_ids = [asym_ids]

            for asym_id in asym_ids:

                instance = get_rcsb_json(
                    f"https://data.rcsb.org/rest/v1/core/"
                    f"polymer_entity_instance/{pdb_id}/{asym_id}"
                )

                instance_ids = instance.get(
                    "rcsb_polymer_entity_instance_container_identifiers",
                    {}
                )

                auth_asym_id = instance_ids.get("auth_asym_id")

                experimental_chains.append({
                    "pdb_id": pdb_id,
                    "entity_id": entity_id,
                    "label_asym_id": asym_id,
                    "auth_asym_id": auth_asym_id,
                    "sequence": sequence,
                    "sequence_length": len(sequence)
                })

        print(
            f"{pdb_id}: "
            f"{len(polymer_entity_ids)} polymer entities inspected"
        )

    except Exception as exc:

        errors.append({
            "pdb_id": pdb_id,
            "error": str(exc)
        })

    time.sleep(0.03)

chains_df = pd.DataFrame(experimental_chains)
errors_df = pd.DataFrame(errors)

print("\nExperimental protein-chain instances:", len(chains_df))
print("Structures with retrieval errors:", len(errors_df))

# --------------------------------------------------
# 5. Align experimental chains against M1 chains
# --------------------------------------------------

alignment_records = []

for _, experimental_chain in chains_df.iterrows():

    for m1_chain_id, m1_sequence in m1_sequences.items():

        result = calculate_alignment(
            m1_sequence,
            experimental_chain["sequence"]
        )

        alignment_records.append({

            "pdb_id": experimental_chain["pdb_id"],
            "experimental_entity_id": (
                experimental_chain["entity_id"]
            ),

            "experimental_label_chain": (
                experimental_chain["label_asym_id"]
            ),

            "experimental_auth_chain": (
                experimental_chain["auth_asym_id"]
            ),

            "experimental_sequence_length": (
                experimental_chain["sequence_length"]
            ),

            "m1_chain_id": m1_chain_id,
            "m1_sequence_length": len(m1_sequence),

            **result
        })

alignment_df = pd.DataFrame(alignment_records)

# Strong sequence correspondence
alignment_df["mapping_status"] = np.where(
    (
        (alignment_df["identity_percentage"] >= 95) &
        (alignment_df["query_coverage_percentage"] >= 90)
    ),
    "STRONG_MATCH",
    "REVIEW"
)

# --------------------------------------------------
# 6. Save results
# --------------------------------------------------

chains_df.drop(columns=["sequence"]).to_csv(
    target_dir / "experimental_protein_chains.csv",
    index=False
)

alignment_df.to_csv(
    target_dir / "experimental_chain_alignment_results.csv",
    index=False
)

errors_df.to_csv(
    target_dir / "experimental_chain_retrieval_errors.csv",
    index=False
)

print("\n========== ALIGNMENT SUMMARY ==========")

print("Experimental chain instances:", len(chains_df))
print("Total alignment records:", len(alignment_df))

print(
    "Strong matches:",
    (alignment_df["mapping_status"] == "STRONG_MATCH").sum()
)

print(
    "Needs review:",
    (alignment_df["mapping_status"] == "REVIEW").sum()
)

print("\nTop alignment results:")

display(
    alignment_df.sort_values(
        ["identity_percentage", "query_coverage_percentage"],
        ascending=False
    ).head(30)
)

print("\nSaved:")
print("1. experimental_protein_chains.csv")
print("2. experimental_chain_alignment_results.csv")
print("3. experimental_chain_retrieval_errors.csv")

M1 structure: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_001.pdb

Member-1 sequences:
Chain A: 350 amino acids
Chain B: 348 amino acids
1UK0: 1 polymer entities inspected
1UK1: 1 polymer entities inspected
1WOK: 1 polymer entities inspected
2RCW: 1 polymer entities inspected
2RD6: 1 polymer entities inspected
3GJW: 1 polymer entities inspected
3GN7: 1 polymer entities inspected
3L3L: 1 polymer entities inspected
3L3M: 1 polymer entities inspected
4GV7: 1 polymer entities inspected
4HHY: 1 polymer entities inspected
4HHZ: 1 polymer entities inspected
4L6S: 1 polymer entities inspected
4OPX: 3 polymer entities inspected
4OQA: 3 polymer entities inspected
4OQB: 3 polymer entities inspected
4PJT: 1 polymer entities inspected
4R5W: 1 polymer entities inspected
4R6E: 1 polymer entities inspected
4RV6: 1 polymer entities inspected
4UND: 1 polymer entities inspected
4UXB: 1 polymer entities inspected
4ZZZ: 1 polymer entities inspected
5A00: 1 polymer ent

,pdb_id,experimental_entity_id,experimental_label_chain,experimental_auth_chain,experimental_sequence_length,m1_chain_id,m1_sequence_length,identity_percentage,query_coverage_percentage,aligned_residues,alignment_score,mapping_status
18,2RD6,1,A,A,350,A,350,100.0,100.0,350,1796.0,STRONG_MATCH
19,2RD6,1,A,A,350,B,348,100.0,100.0,348,1775.5,STRONG_MATCH
20,3GJW,1,A,A,350,A,350,100.0,100.0,350,1796.0,STRONG_MATCH
21,3GJW,1,A,A,350,B,348,100.0,100.0,348,1775.5,STRONG_MATCH
22,3GN7,1,A,A,350,A,350,100.0,100.0,350,1796.0,STRONG_MATCH
23,3GN7,1,A,A,350,B,348,100.0,100.0,348,1775.5,STRONG_MATCH
24,3L3L,1,A,A,350,A,350,100.0,100.0,350,1796.0,STRONG_MATCH
25,3L3L,1,A,A,350,B,348,100.0,100.0,348,1775.5,STRONG_MATCH
26,3L3M,1,A,A,350,A,350,100.0,100.0,350,1796.0,STRONG_MATCH
27,3L3M,1,A,A,350,B,348,100.0,100.0,348,1775.5,STRONG_MATCH



Saved:
1. experimental_protein_chains.csv
2. experimental_chain_alignment_results.csv
3. experimental_chain_retrieval_errors.csv


In [39]:
# CELL 38 — Map M1 pocket residues to experimental structures

import pandas as pd
import numpy as np
import requests
import time

from pathlib import Path
from Bio.PDB import MMCIF2Dict
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.SeqUtils import seq1

# --------------------------------------------------
# 1. Locate existing files
# --------------------------------------------------

target_dir = Path(OUTPUT_DIR) / "target_identification"

pocket_file = target_dir / "member1_pocket_residues.csv"

if not pocket_file.exists():
    raise FileNotFoundError(pocket_file)

pocket_df = pd.read_csv(pocket_file)

# Reuse results from Cell 37
if "chains_df" not in globals():
    raise RuntimeError("Run Cell 37 first. chains_df is missing.")

print("Pocket residue columns:", pocket_df.columns.tolist())
print("Experimental chain records:", len(chains_df))

# --------------------------------------------------
# 2. Identify pocket CSV columns dynamically
# --------------------------------------------------

def find_column(df, possible_names):

    lookup = {
        str(column).lower().strip(): column
        for column in df.columns
    }

    for name in possible_names:
        if name.lower() in lookup:
            return lookup[name.lower()]

    raise KeyError(
        f"Could not find any of {possible_names}. "
        f"Available columns: {df.columns.tolist()}"
    )

chain_col = find_column(
    pocket_df,
    ["chain", "chain_id", "source_chain"]
)

resnum_col = find_column(
    pocket_df,
    ["residue_number", "residue_num", "resseq"]
)

icode_col = find_column(
    pocket_df,
    ["insertion_code", "icode"]
)

resname_col = find_column(
    pocket_df,
    ["residue_name", "resname"]
)

# --------------------------------------------------
# 3. Build M1 sequence-position mappings
# --------------------------------------------------

m1_residue_records = {}
m1_sequences = {}

for chain in m1_structure[0]:

    records = []

    for residue in chain:

        if residue.id[0] != " ":
            continue

        aa = seq1(
            residue.resname,
            custom_map={"UNK": "X"}
        )

        if not aa:
            continue

        records.append({
            "residue": residue,
            "amino_acid": aa,
            "resseq": residue.id[1],
            "icode": str(residue.id[2]).strip()
        })

    if records:

        m1_sequences[str(chain.id)] = "".join(
            record["amino_acid"] for record in records
        )

        m1_residue_records[str(chain.id)] = records

# Map each M1 pocket residue to its sequence index.
pocket_positions = []

for _, row in pocket_df.iterrows():

    chain_id = str(row[chain_col]).strip()

    resseq = int(float(row[resnum_col]))

    icode = str(row[icode_col]).strip()

    if icode.lower() == "nan":
        icode = ""

    residue_name = str(row[resname_col]).strip().upper()

    records = m1_residue_records.get(chain_id, [])

    matching_positions = [
        index
        for index, record in enumerate(records)
        if (
            record["resseq"] == resseq
            and record["icode"] == icode
        )
    ]

    if not matching_positions:

        pocket_positions.append({
            "m1_chain": chain_id,
            "m1_residue_number": resseq,
            "m1_insertion_code": icode,
            "m1_residue_name": residue_name,
            "m1_sequence_index": None,
            "mapping_status": "M1_RESIDUE_NOT_FOUND"
        })

        continue

    sequence_index = matching_positions[0]

    pocket_positions.append({
        "m1_chain": chain_id,
        "m1_residue_number": resseq,
        "m1_insertion_code": icode,
        "m1_residue_name": residue_name,
        "m1_sequence_index": sequence_index,
        "mapping_status": "READY"
    })

pocket_positions_df = pd.DataFrame(pocket_positions)

print("\nM1 pocket residue positions:")
display(pocket_positions_df)

if not (
    pocket_positions_df["mapping_status"] == "READY"
).all():

    raise ValueError(
        "Some M1 pocket residues could not be mapped "
        "to their original protein chains."
    )

# --------------------------------------------------
# 4. Configure sequence alignment
# --------------------------------------------------

aligner = PairwiseAligner()

aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

# --------------------------------------------------
# 5. Prepare experimental coordinate download
# --------------------------------------------------

cif_dir = target_dir / "experimental_mmcif"
cif_dir.mkdir(parents=True, exist_ok=True)

session = requests.Session()

session.headers.update({
    "User-Agent": "Member2-PocketResidueMapping/1.0"
})

cif_cache = {}

def load_cif(pdb_id):

    pdb_id = str(pdb_id).upper()

    if pdb_id in cif_cache:
        return cif_cache[pdb_id]

    cif_path = cif_dir / f"{pdb_id}.cif"

    if not cif_path.exists():

        url = (
            f"https://files.rcsb.org/download/{pdb_id}.cif"
        )

        response = session.get(url, timeout=60)
        response.raise_for_status()

        cif_path.write_bytes(response.content)

    cif_data = MMCIF2Dict.MMCIF2Dict(str(cif_path))

    cif_cache[pdb_id] = cif_data

    return cif_data

def get_list(cif_data, key):

    value = cif_data.get(key, [])

    if isinstance(value, str):
        return [value]

    return value

# --------------------------------------------------
# 6. Process each experimental protein-chain instance
# --------------------------------------------------

mapping_records = []
structure_errors = []

experimental_chains = chains_df.copy()

for pdb_id in sorted(
    experimental_chains["pdb_id"].astype(str).unique()
):

    try:

        cif_data = load_cif(pdb_id)

        atom_chains = get_list(
            cif_data, "_atom_site.label_asym_id"
        )

        atom_seq_ids = get_list(
            cif_data, "_atom_site.label_seq_id"
        )

        atom_comp_ids = get_list(
            cif_data, "_atom_site.label_comp_id"
        )

        atom_groups = get_list(
            cif_data, "_atom_site.group_PDB"
        )

        observed_residues = set()
        observed_names = {}

        for asym_id, seq_id, comp_id, group in zip(
            atom_chains,
            atom_seq_ids,
            atom_comp_ids,
            atom_groups
        ):

            if seq_id in (".", "?", ""):
                continue

            try:
                seq_number = int(seq_id)
            except ValueError:
                continue

            observed_residues.add(
                (str(asym_id), seq_number)
            )

            observed_names[
                (str(asym_id), seq_number)
            ] = str(comp_id).upper()

        structure_chain_rows = experimental_chains[
            experimental_chains["pdb_id"].astype(str) == pdb_id
        ]

        for _, exp_chain in structure_chain_rows.iterrows():

            label_chain = str(
                exp_chain["label_asym_id"]
            )

            experimental_sequence = str(
                exp_chain["sequence"]
            ).replace(" ", "").upper()

            for m1_chain_id, m1_sequence in m1_sequences.items():

                alignment = aligner.align(
                    m1_sequence,
                    experimental_sequence
                )[0]

                query_indices = alignment.indices[0]
                target_indices = alignment.indices[1]

                position_map = {}

                for query_index, target_index in zip(
                    query_indices,
                    target_indices
                ):

                    if query_index >= 0 and target_index >= 0:

                        position_map[
                            int(query_index)
                        ] = int(target_index) + 1

                # Map each pocket residue.
                for _, pocket_row in pocket_positions_df.iterrows():

                    if pocket_row["m1_chain"] != m1_chain_id:
                        continue

                    m1_index = int(
                        pocket_row["m1_sequence_index"]
                    )

                    experimental_label_seq = position_map.get(
                        m1_index
                    )

                    if experimental_label_seq is None:

                        status = "NO_SEQUENCE_POSITION_MAPPING"
                        observed = False
                        experimental_resname = None

                    else:

                        observed = (
                            label_chain,
                            experimental_label_seq
                        ) in observed_residues

                        experimental_resname = observed_names.get(
                            (label_chain, experimental_label_seq)
                        )

                        status = (
                            "MAPPED_AND_OBSERVED"
                            if observed
                            else "MAPPED_BUT_NOT_OBSERVED"
                        )

                    mapping_records.append({

                        "pdb_id": pdb_id,

                        "experimental_entity_id": (
                            exp_chain["entity_id"]
                        ),

                        "experimental_label_chain": label_chain,

                        "experimental_auth_chain": (
                            exp_chain["auth_asym_id"]
                        ),

                        "m1_chain": m1_chain_id,

                        "m1_residue_number": (
                            pocket_row["m1_residue_number"]
                        ),

                        "m1_residue_name": (
                            pocket_row["m1_residue_name"]
                        ),

                        "m1_sequence_index": m1_index,

                        "experimental_label_seq_id": (
                            experimental_label_seq
                        ),

                        "experimental_residue_name": (
                            experimental_resname
                        ),

                        "residue_observed": observed,

                        "mapping_status": status,

                        "alignment_score": alignment.score
                    })

    except Exception as exc:

        structure_errors.append({
            "pdb_id": pdb_id,
            "error": str(exc)
        })

    print(f"{pdb_id}: residue mapping processed")

    time.sleep(0.02)

# --------------------------------------------------
# 7. Create and save reports
# --------------------------------------------------

residue_mapping_df = pd.DataFrame(mapping_records)

structure_errors_df = pd.DataFrame(structure_errors)

residue_mapping_df.to_csv(
    target_dir / "experimental_pocket_residue_mapping.csv",
    index=False
)

structure_errors_df.to_csv(
    target_dir / "pocket_mapping_structure_errors.csv",
    index=False
)

print("\n========== CELL 38 SUMMARY ==========")

print(
    "Total residue mapping records:",
    len(residue_mapping_df)
)

print(
    "Mapped and observed:",
    (
        residue_mapping_df["mapping_status"]
        == "MAPPED_AND_OBSERVED"
    ).sum()
)

print(
    "Mapped but not observed:",
    (
        residue_mapping_df["mapping_status"]
        == "MAPPED_BUT_NOT_OBSERVED"
    ).sum()
)

print(
    "No sequence position mapping:",
    (
        residue_mapping_df["mapping_status"]
        == "NO_SEQUENCE_POSITION_MAPPING"
    ).sum()
)

print(
    "Structure errors:",
    len(structure_errors_df)
)

print("\nSample residue mappings:")

display(
    residue_mapping_df.head(30)
)

print("\nSaved residue mapping report.")

Pocket residue columns: ['chain', 'residue_number', 'insertion_code', 'residue_name', 'found_in_pdb']
Experimental chain records: 169

M1 pocket residue positions:


,m1_chain,m1_residue_number,m1_insertion_code,m1_residue_name,m1_sequence_index,mapping_status
0,A,826,,HIS,164,READY
1,A,903,,LYS,241,READY
2,A,906,,ASN,244,READY
3,A,907,,TYR,245,READY
4,A,909,,HIS,247,READY
5,A,983,,SER,321,READY
6,A,984,,LEU,322,READY
7,A,985,,LEU,323,READY
8,A,986,,TYR,324,READY
9,A,988,,GLU,326,READY


1UK0: residue mapping processed
1UK1: residue mapping processed
1WOK: residue mapping processed
2RCW: residue mapping processed
2RD6: residue mapping processed
3GJW: residue mapping processed
3GN7: residue mapping processed
3L3L: residue mapping processed
3L3M: residue mapping processed
4GV7: residue mapping processed
4HHY: residue mapping processed
4HHZ: residue mapping processed
4L6S: residue mapping processed
4OPX: residue mapping processed
4OQA: residue mapping processed
4OQB: residue mapping processed
4PJT: residue mapping processed
4R5W: residue mapping processed
4R6E: residue mapping processed
4RV6: residue mapping processed
4UND: residue mapping processed
4UXB: residue mapping processed
4ZZZ: residue mapping processed
5A00: residue mapping processed
5HA9: residue mapping processed
5KPN: residue mapping processed
5KPO: residue mapping processed
5KPP: residue mapping processed
5KPQ: residue mapping processed
5WRQ: residue mapping processed
5WRY: residue mapping processed
5WRZ: re

,pdb_id,experimental_entity_id,experimental_label_chain,experimental_auth_chain,m1_chain,m1_residue_number,m1_residue_name,m1_sequence_index,experimental_label_seq_id,experimental_residue_name,residue_observed,mapping_status,alignment_score
0,1UK0,1,A,A,A,826,HIS,164,165.0,HIS,True,MAPPED_AND_OBSERVED,1792.0
1,1UK0,1,A,A,A,903,LYS,241,242.0,LYS,True,MAPPED_AND_OBSERVED,1792.0
2,1UK0,1,A,A,A,906,ASN,244,245.0,ASN,True,MAPPED_AND_OBSERVED,1792.0
3,1UK0,1,A,A,A,907,TYR,245,246.0,TYR,True,MAPPED_AND_OBSERVED,1792.0
4,1UK0,1,A,A,A,909,HIS,247,248.0,HIS,True,MAPPED_AND_OBSERVED,1792.0
5,1UK0,1,A,A,A,983,SER,321,322.0,SER,True,MAPPED_AND_OBSERVED,1792.0
6,1UK0,1,A,A,A,984,LEU,322,323.0,LEU,True,MAPPED_AND_OBSERVED,1792.0
7,1UK0,1,A,A,A,985,LEU,323,324.0,LEU,True,MAPPED_AND_OBSERVED,1792.0
8,1UK0,1,A,A,A,986,TYR,324,325.0,TYR,True,MAPPED_AND_OBSERVED,1792.0
9,1UK0,1,A,A,A,988,GLU,326,327.0,GLU,True,MAPPED_AND_OBSERVED,1792.0



Saved residue mapping report.


In [40]:
# CELL 39 — Diagnose M1 chain similarity and experimental chain pairs

import pandas as pd
from Bio.Align import PairwiseAligner, substitution_matrices

# --------------------------------------------------
# 1. Compare the two M1 chains
# --------------------------------------------------

m1_chain_ids = list(m1_sequences.keys())

if len(m1_chain_ids) < 2:
    raise ValueError("Expected two M1 protein chains.")

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

chain_a = m1_chain_ids[0]
chain_b = m1_chain_ids[1]

alignment = aligner.align(
    m1_sequences[chain_a],
    m1_sequences[chain_b]
)[0]

aligned = alignment.indices
valid = (aligned[0] >= 0) & (aligned[1] >= 0)

identity = (
    sum(
        m1_sequences[chain_a][i] ==
        m1_sequences[chain_b][j]
        for i, j in zip(aligned[0][valid], aligned[1][valid])
    )
    / max(1, valid.sum())
) * 100

print("========== M1 CHAIN COMPARISON ==========")
print("Chain A:", chain_a, "Length:", len(m1_sequences[chain_a]))
print("Chain B:", chain_b, "Length:", len(m1_sequences[chain_b]))
print("Aligned positions:", int(valid.sum()))
print(f"Sequence identity: {identity:.2f}%")

# --------------------------------------------------
# 2. Align every experimental chain against M1
# --------------------------------------------------

chain_results = []

for _, row in chains_df.iterrows():

    exp_sequence = str(row["sequence"]).replace(" ", "").upper()

    for m1_chain_id, m1_sequence in m1_sequences.items():

        aln = aligner.align(m1_sequence, exp_sequence)[0]

        indices = aln.indices
        paired = (indices[0] >= 0) & (indices[1] >= 0)

        aligned_count = int(paired.sum())

        identical = sum(
            m1_sequence[i] == exp_sequence[j]
            for i, j in zip(
                indices[0][paired],
                indices[1][paired]
            )
        )

        identity_pct = (
            100 * identical / aligned_count
            if aligned_count else 0
        )

        coverage_pct = (
            100 * aligned_count / len(m1_sequence)
        )

        chain_results.append({
            "pdb_id": row["pdb_id"],
            "entity_id": row["entity_id"],
            "label_asym_id": row["label_asym_id"],
            "auth_asym_id": row["auth_asym_id"],
            "m1_chain": m1_chain_id,
            "identity_pct": identity_pct,
            "m1_coverage_pct": coverage_pct,
            "aligned_residues": aligned_count,
            "alignment_score": aln.score
        })

chain_match_df = pd.DataFrame(chain_results)

# Strong sequence matches; these are candidates, not final assignments.
strong_matches = chain_match_df[
    (chain_match_df["identity_pct"] >= 95) &
    (chain_match_df["m1_coverage_pct"] >= 90)
].copy()

# --------------------------------------------------
# 3. Find structures with at least two distinct
#    experimental chains matching the M1 sequences
# --------------------------------------------------

pair_summary = []

for pdb_id, group in strong_matches.groupby("pdb_id"):

    distinct_chains = group[
        ["label_asym_id", "auth_asym_id"]
    ].drop_duplicates()

    pair_summary.append({
        "pdb_id": pdb_id,
        "strong_chain_matches": len(group),
        "distinct_experimental_chains": len(distinct_chains),
        "candidate_two_chain_structure": (
            len(distinct_chains) >= 2
        ),
        "experimental_chains": ", ".join(
            distinct_chains["label_asym_id"].astype(str)
        )
    })

pair_summary_df = pd.DataFrame(pair_summary)

# --------------------------------------------------
# 4. Save reports
# --------------------------------------------------

chain_match_df.to_csv(
    target_dir / "m1_experimental_chain_match_diagnostic.csv",
    index=False
)

pair_summary_df.to_csv(
    target_dir / "experimental_two_chain_candidates.csv",
    index=False
)

print("\n========== CELL 39 SUMMARY ==========")

print("Strong chain matches:", len(strong_matches))
print("Structures with strong matches:", len(pair_summary_df))

print("\nCandidate structures with multiple matching chains:")

display(
    pair_summary_df[
        pair_summary_df["candidate_two_chain_structure"]
    ].sort_values(
        "distinct_experimental_chains",
        ascending=False
    )
)

print("\nSample strong chain matches:")

display(
    strong_matches.head(20)
)

========== M1 CHAIN COMPARISON ==========
Chain A: A Length: 350
Chain B: B Length: 348
Aligned positions: 348
Sequence identity: 100.00%

========== CELL 39 SUMMARY ==========
Strong chain matches: 288
Structures with strong matches: 64

Candidate structures with multiple matching chains:


,pdb_id,strong_chain_matches,distinct_experimental_chains,candidate_two_chain_structure,experimental_chains
9,4GV7,8,4,True,"A, B, C, D"
2,1WOK,8,4,True,"A, B, C, D"
10,4HHY,8,4,True,"A, B, C, D"
11,4HHZ,8,4,True,"A, B, C, D"
19,4RV6,8,4,True,"A, B, C, D"
18,4R6E,8,4,True,"A, B, C, D"
16,4PJT,8,4,True,"A, B, C, D"
40,6VKO,8,4,True,"A, B, C, D"
47,7KK3,8,4,True,"A, B, C, D"
49,7KK5,8,4,True,"A, B, C, D"



Sample strong chain matches:


,pdb_id,entity_id,label_asym_id,auth_asym_id,m1_chain,identity_pct,m1_coverage_pct,aligned_residues,alignment_score
0,1UK0,1,A,A,A,99.714286,100.0,350,1792.0
1,1UK0,1,A,A,B,99.712644,100.0,348,1771.5
2,1UK0,1,B,B,A,99.714286,100.0,350,1792.0
3,1UK0,1,B,B,B,99.712644,100.0,348,1771.5
4,1UK1,1,A,A,A,99.714286,100.0,350,1792.0
5,1UK1,1,A,A,B,99.712644,100.0,348,1771.5
6,1UK1,1,B,B,A,99.714286,100.0,350,1792.0
7,1UK1,1,B,B,B,99.712644,100.0,348,1771.5
8,1WOK,1,A,A,A,99.714286,100.0,350,1792.0
9,1WOK,1,A,A,B,99.712644,100.0,348,1771.5


In [41]:
# CELL 40 — Structural superposition and two-chain validation

import numpy as np
import pandas as pd
from pathlib import Path

from Bio.PDB import MMCIFParser
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner, substitution_matrices

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

if "m1_structure" not in globals():
    raise RuntimeError("M1 structure is missing. Run Cell 18 first.")

if "chains_df" not in globals():
    raise RuntimeError("chains_df is missing. Run Cell 37 first.")

if "m1_sequences" not in globals():
    raise RuntimeError("M1 sequences are missing. Run Cell 39 first.")

chain_ids = list(m1_sequences.keys())

if len(chain_ids) != 2:
    raise ValueError(
        f"Expected exactly two M1 chains, found {chain_ids}"
    )

M1_A, M1_B = chain_ids

# --------------------------------------------------
# 2. Build M1 C-alpha coordinate records
# --------------------------------------------------

def extract_ca_records(chain):

    records = []

    for residue in chain:

        if residue.id[0] != " ":
            continue

        if not residue.has_id("CA"):
            continue

        aa = seq1(
            residue.resname,
            custom_map={"UNK": "X"}
        )

        if not aa:
            continue

        records.append({
            "aa": aa,
            "residue": residue,
            "ca": residue["CA"].coord.astype(float)
        })

    return records


m1_ca_records = {}

for chain_id in chain_ids:

    chain = m1_structure[0][chain_id]

    records = extract_ca_records(chain)

    m1_ca_records[chain_id] = records

    sequence = "".join(r["aa"] for r in records)

    print(
        f"M1 chain {chain_id}: "
        f"{len(records)} C-alpha coordinates"
    )

    if sequence != m1_sequences[chain_id]:
        raise ValueError(
            f"Sequence mismatch for M1 chain {chain_id}"
        )

# --------------------------------------------------
# 3. Configure alignment and parser
# --------------------------------------------------

aligner = PairwiseAligner()

aligner.mode = "global"
aligner.substitution_matrix = substitution_matrices.load(
    "BLOSUM62"
)
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

parser = MMCIFParser(
    QUIET=True,
    auth_chains=True,
    auth_residues=True
)

structure_cache = {}

# --------------------------------------------------
# 4. Extract experimental chain C-alpha coordinates
# --------------------------------------------------

def get_experimental_chain_records(pdb_id, auth_chain_id):

    if pdb_id not in structure_cache:

        cif_path = cif_dir / f"{pdb_id}.cif"

        structure_cache[pdb_id] = parser.get_structure(
            pdb_id,
            str(cif_path)
        )

    structure = structure_cache[pdb_id]

    model = structure[0]

    auth_chain_id = str(auth_chain_id)

    if auth_chain_id not in model:
        return None

    return extract_ca_records(model[auth_chain_id])


# --------------------------------------------------
# 5. Map M1 sequence positions to experimental C-alpha
# --------------------------------------------------

def get_corresponding_ca_pairs(m1_chain_id, exp_records):

    m1_records = m1_ca_records[m1_chain_id]

    m1_sequence = "".join(r["aa"] for r in m1_records)

    exp_sequence = "".join(r["aa"] for r in exp_records)

    alignment = aligner.align(
        m1_sequence,
        exp_sequence
    )[0]

    indices = alignment.indices

    fixed_coords = []
    moving_coords = []

    for m1_index, exp_index in zip(
        indices[0],
        indices[1]
    ):

        if m1_index < 0 or exp_index < 0:
            continue

        fixed_coords.append(
            m1_records[int(m1_index)]["ca"]
        )

        moving_coords.append(
            exp_records[int(exp_index)]["ca"]
        )

    if not fixed_coords:
        return None, None

    return (
        np.asarray(fixed_coords),
        np.asarray(moving_coords)
    )


# --------------------------------------------------
# 6. Kabsch superposition
# --------------------------------------------------

def kabsch_fit(fixed, moving):

    fixed_center = fixed.mean(axis=0)
    moving_center = moving.mean(axis=0)

    X = moving - moving_center
    Y = fixed - fixed_center

    U, S, Vt = np.linalg.svd(X.T @ Y)

    correction = np.eye(3)

    if np.linalg.det(U @ Vt) < 0:
        correction[-1, -1] = -1

    rotation = U @ correction @ Vt

    translation = fixed_center - moving_center @ rotation

    transformed = moving @ rotation + translation

    rmsd = np.sqrt(
        np.mean(
            np.sum((fixed - transformed) ** 2, axis=1)
        )
    )

    return rotation, translation, rmsd


def apply_transform(coords, rotation, translation):

    return coords @ rotation + translation


def calculate_rmsd(fixed, moving):

    if len(fixed) == 0:
        return np.nan

    return np.sqrt(
        np.mean(
            np.sum((fixed - moving) ** 2, axis=1)
        )
    )


# --------------------------------------------------
# 7. Prepare experimental chain candidates
# --------------------------------------------------

experimental_candidates = {}

for pdb_id, group in chains_df.groupby("pdb_id"):

    candidates = []

    for _, row in group.iterrows():

        candidates.append({
            "label_chain": str(row["label_asym_id"]),
            "auth_chain": str(row["auth_asym_id"]),
            "entity_id": str(row["entity_id"])
        })

    experimental_candidates[str(pdb_id)] = candidates


# --------------------------------------------------
# 8. Process all possible distinct chain pairs
# --------------------------------------------------

pair_results = []
independent_results = []
errors = []

for pdb_id, candidates in experimental_candidates.items():

    try:

        # Load each experimental chain once.
        exp_chain_data = {}

        for candidate in candidates:

            auth_id = candidate["auth_chain"]

            records = get_experimental_chain_records(
                pdb_id,
                auth_id
            )

            if records is None:
                continue

            exp_chain_data[auth_id] = {
                **candidate,
                "records": records
            }

        # Calculate independent structural RMSD.
        for exp_id, exp_data in exp_chain_data.items():

            for m1_id in chain_ids:

                fixed, moving = get_corresponding_ca_pairs(
                    m1_id,
                    exp_data["records"]
                )

                if fixed is None or len(fixed) < 50:
                    continue

                _, _, rmsd = kabsch_fit(
                    fixed,
                    moving
                )

                independent_results.append({
                    "pdb_id": pdb_id,
                    "m1_chain": m1_id,
                    "experimental_auth_chain": exp_id,
                    "experimental_label_chain": (
                        exp_data["label_chain"]
                    ),
                    "matched_ca_atoms": len(fixed),
                    "independent_rmsd_A": rmsd
                })

        # Test every distinct experimental chain assignment.
        for exp_a in exp_chain_data:

            for exp_b in exp_chain_data:

                if exp_a == exp_b:
                    continue

                records_a = exp_chain_data[exp_a]["records"]
                records_b = exp_chain_data[exp_b]["records"]

                # Map M1 A -> experimental chain A
                # Map M1 B -> experimental chain B

                fixed_a, moving_a = get_corresponding_ca_pairs(
                    M1_A,
                    records_a
                )

                fixed_b, moving_b = get_corresponding_ca_pairs(
                    M1_B,
                    records_b
                )

                if (
                    fixed_a is None or fixed_b is None
                    or len(fixed_a) < 50
                    or len(fixed_b) < 50
                ):
                    continue

                # Fit chain A and apply the SAME transform to B.
                rotation_a, translation_a, rmsd_a = kabsch_fit(
                    fixed_a,
                    moving_a
                )

                transformed_b_after_a = apply_transform(
                    moving_b,
                    rotation_a,
                    translation_a
                )

                rmsd_b_after_a = calculate_rmsd(
                    fixed_b,
                    transformed_b_after_a
                )

                combined_a = np.sqrt(
                    (
                        len(fixed_a) * rmsd_a**2
                        + len(fixed_b) * rmsd_b_after_a**2
                    )
                    / (len(fixed_a) + len(fixed_b))
                )

                # Reverse: fit B and apply the SAME transform to A.
                rotation_b, translation_b, rmsd_b = kabsch_fit(
                    fixed_b,
                    moving_b
                )

                transformed_a_after_b = apply_transform(
                    moving_a,
                    rotation_b,
                    translation_b
                )

                rmsd_a_after_b = calculate_rmsd(
                    fixed_a,
                    transformed_a_after_b
                )

                combined_b = np.sqrt(
                    (
                        len(fixed_b) * rmsd_b**2
                        + len(fixed_a) * rmsd_a_after_b**2
                    )
                    / (len(fixed_a) + len(fixed_b))
                )

                # Choose the better common-frame alignment.
                if combined_a <= combined_b:

                    combined_rmsd = combined_a
                    anchor = M1_A
                    second_chain_rmsd = rmsd_b_after_a

                else:

                    combined_rmsd = combined_b
                    anchor = M1_B
                    second_chain_rmsd = rmsd_a_after_b

                pair_results.append({

                    "pdb_id": pdb_id,

                    "m1_A_experimental_chain": exp_a,
                    "m1_B_experimental_chain": exp_b,

                    "m1_A_independent_rmsd_A": rmsd_a,
                    "m1_B_independent_rmsd_A": rmsd_b,

                    "second_chain_rmsd_after_common_fit_A": (
                        second_chain_rmsd
                    ),

                    "two_chain_common_frame_rmsd_A": (
                        combined_rmsd
                    ),

                    "best_alignment_anchor": anchor,

                    "matched_ca_atoms_A": len(fixed_a),
                    "matched_ca_atoms_B": len(fixed_b)
                })

        print(f"{pdb_id}: structural comparison completed")

    except Exception as exc:

        errors.append({
            "pdb_id": pdb_id,
            "error": str(exc)
        })

        print(f"{pdb_id}: ERROR — {exc}")


# --------------------------------------------------
# 9. Save detailed results
# --------------------------------------------------

pair_df = pd.DataFrame(pair_results)

independent_df = pd.DataFrame(independent_results)

error_df = pd.DataFrame(errors)

pair_df.to_csv(
    target_dir / "two_chain_structural_superposition.csv",
    index=False
)

independent_df.to_csv(
    target_dir / "independent_chain_structural_rmsd.csv",
    index=False
)

error_df.to_csv(
    target_dir / "structural_superposition_errors.csv",
    index=False
)


# --------------------------------------------------
# 10. Best chain-pair result per structure
# --------------------------------------------------

if not pair_df.empty:

    best_pair_df = (
        pair_df.sort_values(
            "two_chain_common_frame_rmsd_A"
        )
        .groupby("pdb_id")
        .head(1)
        .sort_values(
            "two_chain_common_frame_rmsd_A"
        )
        .reset_index(drop=True)
    )

    best_pair_df.to_csv(
        target_dir / "best_two_chain_structural_matches.csv",
        index=False
    )

    print("\n========== BEST TWO-CHAIN MATCHES ==========")

    display(
        best_pair_df.head(20)
    )

print("\n========== CELL 40 SUMMARY ==========")

print("Independent chain comparisons:", len(independent_df))
print("Two-chain assignments tested:", len(pair_df))
print("Structures with successful pair comparisons:",
      pair_df["pdb_id"].nunique() if not pair_df.empty else 0)
print("Errors:", len(error_df))

print("\nSaved structural comparison reports.")

M1 chain A: 350 C-alpha coordinates
M1 chain B: 348 C-alpha coordinates
1UK0: structural comparison completed
1UK1: structural comparison completed
1WOK: structural comparison completed
2RCW: structural comparison completed
2RD6: structural comparison completed
3GJW: structural comparison completed
3GN7: structural comparison completed
3L3L: structural comparison completed
3L3M: structural comparison completed
4GV7: structural comparison completed
4HHY: structural comparison completed
4HHZ: structural comparison completed
4L6S: structural comparison completed
4OPX: ERROR — sequence has zero length
4OQA: ERROR — sequence has zero length
4OQB: ERROR — sequence has zero length
4PJT: structural comparison completed
4R5W: structural comparison completed
4R6E: structural comparison completed
4RV6: structural comparison completed
4UND: structural comparison completed
4UXB: structural comparison completed
4ZZZ: structural comparison completed
5A00: structural comparison completed
5HA9: structu

,pdb_id,m1_A_experimental_chain,m1_B_experimental_chain,m1_A_independent_rmsd_A,m1_B_independent_rmsd_A,second_chain_rmsd_after_common_fit_A,two_chain_common_frame_rmsd_A,best_alignment_anchor,matched_ca_atoms_A,matched_ca_atoms_B
0,7ONS,A,B,0.318505,0.572458,0.778923,0.594440,A,350,348
1,7AAD,A,B,0.449212,0.604043,0.676951,0.641638,B,350,348
2,7AAC,A,B,0.411657,0.701278,1.354738,1.000001,A,350,348
3,5KPP,A,B,0.419997,0.934316,1.359974,1.005266,A,349,347
4,8JNZ,B,A,0.521672,0.726498,1.349709,1.022110,A,350,348
5,7CMW,B,A,0.544550,0.747421,1.347666,1.026738,A,350,348
6,9ETQ,A,B,0.403739,0.688449,1.484022,1.086159,A,350,348
7,7ONT,A,B,0.397920,0.718376,1.543395,1.125619,A,350,348
8,7ONR,A,B,0.430695,0.711022,1.554726,1.139359,A,350,348
9,5KPQ,A,B,0.455103,0.919154,1.582864,1.163180,A,349,347



========== CELL 40 SUMMARY ==========
Independent chain comparisons: 294
Two-chain assignments tested: 230
Structures with successful pair comparisons: 50
Errors: 5

Saved structural comparison reports.


In [42]:
# CELL 41 — Pocket-specific structural RMSD validation

import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Validate previous results
# --------------------------------------------------

if "best_pair_df" not in globals():
    raise RuntimeError("Cell 40 best_pair_df is missing.")

if "pocket_positions_df" not in globals():
    raise RuntimeError("M1 pocket positions are missing.")

if best_pair_df.empty:
    raise ValueError("No successful two-chain matches.")

# --------------------------------------------------
# 2. Map M1 sequence indices to experimental indices
# --------------------------------------------------

def get_position_map(m1_chain_id, exp_records):

    m1_records = m1_ca_records[m1_chain_id]

    m1_sequence = "".join(r["aa"] for r in m1_records)
    exp_sequence = "".join(r["aa"] for r in exp_records)

    alignment = aligner.align(
        m1_sequence,
        exp_sequence
    )[0]

    indices = alignment.indices

    position_map = {}

    for i, j in zip(indices[0], indices[1]):

        if i >= 0 and j >= 0:
            position_map[int(i)] = int(j)

    return position_map


# --------------------------------------------------
# 3. Extract pocket C-alpha correspondence
# --------------------------------------------------

def get_pocket_ca_pairs(m1_chain_id, exp_records):

    position_map = get_position_map(
        m1_chain_id,
        exp_records
    )

    m1_records = m1_ca_records[m1_chain_id]

    pocket_rows = pocket_positions_df[
        pocket_positions_df["m1_chain"] == m1_chain_id
    ]

    fixed = []
    moving = []
    mapped_count = 0

    for _, row in pocket_rows.iterrows():

        m1_index = int(row["m1_sequence_index"])

        exp_index = position_map.get(m1_index)

        if exp_index is None:
            continue

        if exp_index >= len(exp_records):
            continue

        fixed.append(
            m1_records[m1_index]["ca"]
        )

        moving.append(
            exp_records[exp_index]["ca"]
        )

        mapped_count += 1

    if not fixed:
        return None, None, 0

    return (
        np.asarray(fixed),
        np.asarray(moving),
        mapped_count
    )


# --------------------------------------------------
# 4. Calculate pocket RMSD for best chain pairs
# --------------------------------------------------

pocket_results = []

for _, row in best_pair_df.iterrows():

    pdb_id = str(row["pdb_id"])

    exp_a = str(row["m1_A_experimental_chain"])
    exp_b = str(row["m1_B_experimental_chain"])

    try:

        records_a = get_experimental_chain_records(
            pdb_id,
            exp_a
        )

        records_b = get_experimental_chain_records(
            pdb_id,
            exp_b
        )

        if records_a is None or records_b is None:
            continue

        # Full-chain correspondence for common transformation.
        full_fixed_a, full_moving_a = (
            get_corresponding_ca_pairs(M1_A, records_a)
        )

        full_fixed_b, full_moving_b = (
            get_corresponding_ca_pairs(M1_B, records_b)
        )

        # Pocket-only correspondence.
        pocket_fixed_a, pocket_moving_a, count_a = (
            get_pocket_ca_pairs(M1_A, records_a)
        )

        pocket_fixed_b, pocket_moving_b, count_b = (
            get_pocket_ca_pairs(M1_B, records_b)
        )

        if (
            pocket_fixed_a is None
            or pocket_fixed_b is None
        ):
            continue

        # Use the same anchor selected in Cell 40.
        anchor = str(row["best_alignment_anchor"])

        if anchor == M1_A:

            rotation, translation, _ = kabsch_fit(
                full_fixed_a,
                full_moving_a
            )

        else:

            rotation, translation, _ = kabsch_fit(
                full_fixed_b,
                full_moving_b
            )

        # Apply one common transformation to both chains.
        transformed_a = apply_transform(
            pocket_moving_a,
            rotation,
            translation
        )

        transformed_b = apply_transform(
            pocket_moving_b,
            rotation,
            translation
        )

        rmsd_a = calculate_rmsd(
            pocket_fixed_a,
            transformed_a
        )

        rmsd_b = calculate_rmsd(
            pocket_fixed_b,
            transformed_b
        )

        combined_rmsd = np.sqrt(
            (
                count_a * rmsd_a**2
                + count_b * rmsd_b**2
            ) / (count_a + count_b)
        )

        pocket_results.append({

            "pdb_id": pdb_id,

            "experimental_chain_A": exp_a,
            "experimental_chain_B": exp_b,

            "pocket_residues_expected_A": len(
                pocket_positions_df[
                    pocket_positions_df["m1_chain"] == M1_A
                ]
            ),

            "pocket_residues_expected_B": len(
                pocket_positions_df[
                    pocket_positions_df["m1_chain"] == M1_B
                ]
            ),

            "pocket_residues_mapped_A": count_a,
            "pocket_residues_mapped_B": count_b,

            "pocket_rmsd_A": rmsd_a,
            "pocket_rmsd_B": rmsd_b,

            "combined_pocket_rmsd_A": combined_rmsd,

            "whole_chain_common_rmsd_A": (
                row["two_chain_common_frame_rmsd_A"]
            )
        })

    except Exception as exc:

        print(f"{pdb_id}: {exc}")


# --------------------------------------------------
# 5. Save and display
# --------------------------------------------------

pocket_rmsd_df = pd.DataFrame(pocket_results)

pocket_rmsd_df = pocket_rmsd_df.sort_values(
    "combined_pocket_rmsd_A"
).reset_index(drop=True)

pocket_rmsd_df.to_csv(
    target_dir / "pocket_specific_structural_validation.csv",
    index=False
)

print("\n========== CELL 41 SUMMARY ==========")

print("Structures evaluated:", len(pocket_rmsd_df))

print("\nBest pocket-specific matches:")

display(
    pocket_rmsd_df.head(20)
)

print("\nSaved pocket-specific structural validation.")


========== CELL 41 SUMMARY ==========
Structures evaluated: 46

Best pocket-specific matches:


,pdb_id,experimental_chain_A,experimental_chain_B,pocket_residues_expected_A,pocket_residues_expected_B,pocket_residues_mapped_A,pocket_residues_mapped_B,pocket_rmsd_A,pocket_rmsd_B,combined_pocket_rmsd_A,whole_chain_common_rmsd_A
0,7ONS,A,B,10,1,10,1,0.125579,0.114724,0.124631,0.594440
1,9ETQ,A,B,10,1,10,1,0.125593,0.326102,0.154943,1.086159
2,7ONT,A,B,10,1,10,1,0.162472,0.500483,0.216261,1.125619
3,5WRZ,A,B,10,1,10,1,0.217122,0.323156,0.228801,1.738891
4,7AAC,A,B,10,1,10,1,0.194350,0.469732,0.233232,1.000001
5,5WS1,A,B,10,1,10,1,0.224751,0.384271,0.243608,1.670137
6,9ILN,B,A,10,1,10,1,0.210158,0.475427,0.246373,1.198181
7,5KPQ,A,B,10,1,10,1,0.232291,0.442278,0.258528,1.163180
8,5KPP,A,B,10,1,10,1,0.226543,0.472030,0.258673,1.005266
9,8JNZ,B,A,10,1,10,1,0.217210,0.520556,0.259857,1.022110



Saved pocket-specific structural validation.


In [43]:
# CELL 42 — Identify ligand candidates in best structural matches

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Load existing experimental reports
# --------------------------------------------------

target_dir = Path(OUTPUT_DIR) / "target_identification"

instances_file = (
    target_dir / "experimental_nonpolymer_instances.csv"
)

ligand_file = (
    target_dir / "potential_ligand_candidates.csv"
)

if not instances_file.exists():
    raise FileNotFoundError(instances_file)

if not ligand_file.exists():
    raise FileNotFoundError(ligand_file)

instances_df = pd.read_csv(instances_file)
ligands_df = pd.read_csv(ligand_file)

print("Instance report columns:")
print(instances_df.columns.tolist())

print("\nLigand classification columns:")
print(ligands_df.columns.tolist())

# --------------------------------------------------
# 2. Select the best structures from Cell 41
# --------------------------------------------------

if "pocket_rmsd_df" not in globals():
    raise RuntimeError("Cell 41 results are missing.")

selected_structures = (
    pocket_rmsd_df
    .sort_values("combined_pocket_rmsd_A")
    .head(10)["pdb_id"]
    .astype(str)
    .str.upper()
    .tolist()
)

print("\nSelected structures:")
print(selected_structures)

# --------------------------------------------------
# 3. Identify matching PDB and component columns
# --------------------------------------------------

def detect_column(df, candidates):

    lookup = {
        str(c).lower().strip(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    raise KeyError(
        f"Could not identify columns from {candidates}. "
        f"Available: {df.columns.tolist()}"
    )


instance_pdb_col = detect_column(
    instances_df,
    ["pdb_id", "structure_id"]
)

ligand_pdb_col = detect_column(
    ligands_df,
    ["pdb_id", "structure_id"]
)

instance_comp_col = detect_column(
    instances_df,
    ["comp_id", "component_id", "chemical_component_id"]
)

ligand_comp_col = detect_column(
    ligands_df,
    ["comp_id", "component_id", "chemical_component_id"]
)

# --------------------------------------------------
# 4. Filter non-polymer instances
# --------------------------------------------------

instances_df["_pdb_key"] = (
    instances_df[instance_pdb_col]
    .astype(str)
    .str.upper()
)

instances_df["_comp_key"] = (
    instances_df[instance_comp_col]
    .astype(str)
    .str.upper()
)

ligands_df["_pdb_key"] = (
    ligands_df[ligand_pdb_col]
    .astype(str)
    .str.upper()
)

ligands_df["_comp_key"] = (
    ligands_df[ligand_comp_col]
    .astype(str)
    .str.upper()
)

selected_instances = instances_df[
    instances_df["_pdb_key"].isin(selected_structures)
].copy()

selected_ligands = ligands_df[
    ligands_df["_pdb_key"].isin(selected_structures)
].copy()

# --------------------------------------------------
# 5. Attach preliminary chemical classification
# --------------------------------------------------

classification_cols = [
    col for col in [
        ligand_pdb_col,
        ligand_comp_col,
        "comp_name",
        "chemical_name",
        "classification",
        "reason"
    ]
    if col in selected_ligands.columns
]

classification_df = selected_ligands[
    classification_cols
].drop_duplicates()

report_df = selected_instances.merge(
    classification_df,
    left_on=["_pdb_key", "_comp_key"],
    right_on=[
        classification_df.columns[0],
        classification_df.columns[1]
    ],
    how="left",
    suffixes=("", "_classification")
)

# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("\n========== CELL 42 SUMMARY ==========")

print("Selected structures:", len(selected_structures))

print(
    "Non-polymer instances found:",
    len(selected_instances)
)

print(
    "Potential ligand records:",
    len(selected_ligands)
)

print("\nNon-polymer instance columns:")
display(selected_instances.head(10))

print("\nPotential inhibitor candidates:")
display(selected_ligands.head(30))

# --------------------------------------------------
# 7. Save reports
# --------------------------------------------------

selected_instances.to_csv(
    target_dir / "top_structures_nonpolymer_instances.csv",
    index=False
)

selected_ligands.to_csv(
    target_dir / "top_structures_potential_ligands.csv",
    index=False
)

print("\nReports saved.")

Instance report columns:
['pdb_id', 'entity_id', 'comp_id', 'description', 'label_asym_id', 'auth_asym_id']

Ligand classification columns:
['pdb_id', 'entity_id', 'comp_id', 'description', 'formula', 'preliminary_class', 'instance_count']

Selected structures:
['7ONS', '9ETQ', '7ONT', '5WRZ', '7AAC', '5WS1', '9ILN', '5KPQ', '5KPP', '8JNZ']

========== CELL 42 SUMMARY ==========
Selected structures: 10
Non-polymer instances found: 42
Potential ligand records: 10

Non-polymer instance columns:


,pdb_id,entity_id,comp_id,description,label_asym_id,auth_asym_id,_pdb_key,_comp_key
68,5KPP,2,6WZ,"1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris...",C,A,5KPP,6WZ
69,5KPP,2,6WZ,"1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris...",D,B,5KPP,6WZ
70,5KPQ,2,6X2,1-[[4-fluoranyl-3-[(3R)-3-methyl-4-propyl-pipe...,C,A,5KPQ,6X2
71,5KPQ,2,6X2,1-[[4-fluoranyl-3-[(3R)-3-methyl-4-propyl-pipe...,D,B,5KPQ,6X2
76,5WRZ,2,7U3,7-fluoranyl-4-[[(3R)-pyrrolidin-3-yl]methoxy]-...,C,A,5WRZ,7U3
77,5WRZ,2,7U3,7-fluoranyl-4-[[(3R)-pyrrolidin-3-yl]methoxy]-...,D,B,5WRZ,7U3
80,5WS1,2,7U9,2-[(3R)-3-azanylpyrrolidin-1-yl]carbonyl-1H-be...,C,A,5WS1,7U9
81,5WS1,2,7U9,2-[(3R)-3-azanylpyrrolidin-1-yl]carbonyl-1H-be...,D,B,5WS1,7U9
175,7AAC,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,C,A,7AAC,78P
176,7AAC,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,E,B,7AAC,78P



Potential inhibitor candidates:


,pdb_id,entity_id,comp_id,description,formula,preliminary_class,instance_count,_pdb_key,_comp_key
16,5KPP,2,6WZ,"1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris...",0.478,potential_ligand_candidate,2,5KPP,6WZ
17,5KPQ,2,6X2,1-[[4-fluoranyl-3-[(3R)-3-methyl-4-propyl-pipe...,0.438,potential_ligand_candidate,2,5KPQ,6X2
20,5WRZ,2,7U3,7-fluoranyl-4-[[(3R)-pyrrolidin-3-yl]methoxy]-...,0.263,potential_ligand_candidate,2,5WRZ,7U3
22,5WS1,2,7U9,2-[(3R)-3-azanylpyrrolidin-1-yl]carbonyl-1H-be...,0.273,potential_ligand_candidate,2,5WS1,7U9
31,7AAC,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,0.244,potential_ligand_candidate,2,7AAC,78P
35,7ONS,2,VKT,"7-[[4-(1,5-dimethylimidazol-2-yl)piperazin-1-y...",0.365,potential_ligand_candidate,2,7ONS,VKT
36,7ONT,2,VKQ,5-[4-[(3-ethyl-2-oxidanylidene-1~{H}-quinolin-...,0.405,potential_ligand_candidate,2,7ONT,VKQ
39,8JNZ,2,ERV,"6-methylpyrazolo[1,5-a]pyrimidine-3-carboxamide",0.176,potential_ligand_candidate,2,8JNZ,ERV
40,9ETQ,2,A1H63,"5-[4-[(7-ethyl-6-oxidanylidene-5~{H}-1,5-napht...",0.406,potential_ligand_candidate,2,9ETQ,A1H63
42,9ILN,2,A1D9M,5-ethyl-1-[[3-[(3~{R})-4-ethyl-3-(2-hydroxyeth...,0.432,potential_ligand_candidate,2,9ILN,A1D9M



Reports saved.


In [44]:
# CELL 43 — Load structural-validation inputs

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Member2_Project")
M1_DIR = BASE_DIR / "member1_input"
OUTPUT_DIR = BASE_DIR / "member2_outputs"

TARGET_DIR = OUTPUT_DIR / "target_identification"
STRUCTURE_DIR = OUTPUT_DIR / "experimental_mmcif"

STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)

def find_saved_file(filename):
    matches = list(OUTPUT_DIR.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Required file not found: {filename}"
        )

    if len(matches) > 1:
        print("Multiple copies found:", matches)

    return matches[0]


# Load previously generated reports
instances_file = find_saved_file(
    "top_structures_nonpolymer_instances.csv"
)

candidates_file = find_saved_file(
    "top_structures_potential_ligands.csv"
)

pocket_residues_file = find_saved_file(
    "member1_pocket_residues.csv"
)

instances_df = pd.read_csv(instances_file)
candidates_df = pd.read_csv(candidates_file)
pocket_residues_df = pd.read_csv(pocket_residues_file)

# Find the original M1 reference structure
m1_pdb_matches = list(
    M1_DIR.rglob("conformer_001.pdb")
)

if not m1_pdb_matches:
    raise FileNotFoundError(
        "M1 conformer_001.pdb was not found."
    )

M1_REFERENCE_PDB = m1_pdb_matches[0]

# Basic schema checks
required_instance_columns = {
    "pdb_id",
    "comp_id",
    "label_asym_id",
    "auth_asym_id"
}

required_pocket_columns = {
    "chain",
    "residue_number",
    "insertion_code",
    "residue_name"
}

assert required_instance_columns.issubset(
    instances_df.columns
)

assert required_pocket_columns.issubset(
    pocket_residues_df.columns
)

selected_pdbs = sorted(
    candidates_df["pdb_id"].astype(str).str.upper().unique()
)

print("========== CELL 43 SUMMARY ==========")

print("Potential ligand records:", len(candidates_df))
print("Non-polymer instances:", len(instances_df))
print("Selected structures:", selected_pdbs)

print("M1 reference:", M1_REFERENCE_PDB)
print("M1 pocket residues:", len(pocket_residues_df))

print("\nInstance columns:")
print(instances_df.columns.tolist())

print("\nPocket residue columns:")
print(pocket_residues_df.columns.tolist())

display(candidates_df)

========== CELL 43 SUMMARY ==========
Potential ligand records: 10
Non-polymer instances: 42
Selected structures: ['5KPP', '5KPQ', '5WRZ', '5WS1', '7AAC', '7ONS', '7ONT', '8JNZ', '9ETQ', '9ILN']
M1 reference: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_001.pdb
M1 pocket residues: 11

Instance columns:
['pdb_id', 'entity_id', 'comp_id', 'description', 'label_asym_id', 'auth_asym_id', '_pdb_key', '_comp_key']

Pocket residue columns:
['chain', 'residue_number', 'insertion_code', 'residue_name', 'found_in_pdb']


,pdb_id,entity_id,comp_id,description,formula,preliminary_class,instance_count,_pdb_key,_comp_key
0,5KPP,2,6WZ,"1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris...",0.478,potential_ligand_candidate,2,5KPP,6WZ
1,5KPQ,2,6X2,1-[[4-fluoranyl-3-[(3R)-3-methyl-4-propyl-pipe...,0.438,potential_ligand_candidate,2,5KPQ,6X2
2,5WRZ,2,7U3,7-fluoranyl-4-[[(3R)-pyrrolidin-3-yl]methoxy]-...,0.263,potential_ligand_candidate,2,5WRZ,7U3
3,5WS1,2,7U9,2-[(3R)-3-azanylpyrrolidin-1-yl]carbonyl-1H-be...,0.273,potential_ligand_candidate,2,5WS1,7U9
4,7AAC,2,78P,(2R)-2-(7-carbamoyl-1H-benzimidazol-2-yl)-2-me...,0.244,potential_ligand_candidate,2,7AAC,78P
5,7ONS,2,VKT,"7-[[4-(1,5-dimethylimidazol-2-yl)piperazin-1-y...",0.365,potential_ligand_candidate,2,7ONS,VKT
6,7ONT,2,VKQ,5-[4-[(3-ethyl-2-oxidanylidene-1~{H}-quinolin-...,0.405,potential_ligand_candidate,2,7ONT,VKQ
7,8JNZ,2,ERV,"6-methylpyrazolo[1,5-a]pyrimidine-3-carboxamide",0.176,potential_ligand_candidate,2,8JNZ,ERV
8,9ETQ,2,A1H63,"5-[4-[(7-ethyl-6-oxidanylidene-5~{H}-1,5-napht...",0.406,potential_ligand_candidate,2,9ETQ,A1H63
9,9ILN,2,A1D9M,5-ethyl-1-[[3-[(3~{R})-4-ethyl-3-(2-hydroxyeth...,0.432,potential_ligand_candidate,2,9ILN,A1D9M


In [45]:
# CELL 44 — Download and inspect experimental mmCIF structures

import requests
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

cif_paths = {}

for pdb_id in selected_pdbs:

    pdb_id = pdb_id.upper()

    cif_path = STRUCTURE_DIR / f"{pdb_id}.cif"

    if not cif_path.exists():

        url = (
            f"https://files.rcsb.org/download/{pdb_id}.cif"
        )

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        cif_path.write_bytes(response.content)

    cif_paths[pdb_id] = cif_path

    print(
        f"{pdb_id}: {cif_path.stat().st_size / 1024:.1f} KB"
    )


# Inspect atom-site fields in one downloaded structure
sample_pdb = selected_pdbs[0]

sample_cif = MMCIF2Dict(
    str(cif_paths[sample_pdb])
)

atom_fields = [
    "_atom_site.group_PDB",
    "_atom_site.label_atom_id",
    "_atom_site.label_comp_id",
    "_atom_site.label_asym_id",
    "_atom_site.auth_asym_id",
    "_atom_site.type_symbol",
    "_atom_site.Cartn_x",
    "_atom_site.Cartn_y",
    "_atom_site.Cartn_z",
    "_atom_site.label_seq_id",
    "_atom_site.auth_seq_id",
]

print("\n========== ATOM-SITE FIELD AUDIT ==========")

for field in atom_fields:
    print(
        field,
        "FOUND" if field in sample_cif else "MISSING"
    )

if not all(field in sample_cif for field in atom_fields):
    raise RuntimeError(
        "Required mmCIF atom-site fields are missing."
    )

print(
    "\nAtom records in sample:",
    len(sample_cif["_atom_site.group_PDB"])
)

print("Sample structure:", sample_pdb)

5KPP: 616.2 KB
5KPQ: 604.0 KB
5WRZ: 597.6 KB
5WS1: 599.7 KB
7AAC: 1115.2 KB
7ONS: 1113.0 KB
7ONT: 1109.7 KB
8JNZ: 655.1 KB
9ETQ: 1137.3 KB
9ILN: 709.2 KB

========== ATOM-SITE FIELD AUDIT ==========
_atom_site.group_PDB FOUND
_atom_site.label_atom_id FOUND
_atom_site.label_comp_id FOUND
_atom_site.label_asym_id FOUND
_atom_site.auth_asym_id FOUND
_atom_site.type_symbol FOUND
_atom_site.Cartn_x FOUND
_atom_site.Cartn_y FOUND
_atom_site.Cartn_z FOUND
_atom_site.label_seq_id FOUND
_atom_site.auth_seq_id FOUND

Atom records in sample: 5752
Sample structure: 5KPP


In [47]:
# CELL 45 — Structural alignment and M1 pocket coordinate transformation

from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Align import PairwiseAligner
from Bio.SeqUtils import seq1
from itertools import permutations

pdb_parser = PDBParser(QUIET=True)

m1_structure = pdb_parser.get_structure(
    "M1_REFERENCE",
    str(M1_REFERENCE_PDB)
)

m1_model = m1_structure[0]

m1_chains = {
    chain.id: chain
    for chain in m1_model
}

# M1 currently contains two protein chains.
if not {"A", "B"}.issubset(m1_chains):
    raise RuntimeError(
        "Expected M1 reference chains A and B were not found. "
        f"Available chains: {list(m1_chains)}"
    )


def protein_residues(chain):

    return [
        residue
        for residue in chain
        if is_aa(residue, standard=False)
        and "CA" in residue
    ]


def protein_sequence(residues):

    return "".join(
        seq1(res.resname, custom_map={"UNK": "X"})
        for res in residues
    )


def align_protein_chains(reference_chain, experimental_chain):

    ref_res = protein_residues(reference_chain)
    exp_res = protein_residues(experimental_chain)

    ref_seq = protein_sequence(ref_res)
    exp_seq = protein_sequence(exp_res)

    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.5

    alignment = aligner.align(
        ref_seq,
        exp_seq
    )[0]

    indices = alignment.indices

    ref_xyz = []
    exp_xyz = []

    identical = 0
    aligned = 0

    for i, j in zip(indices[0], indices[1]):

        if i < 0 or j < 0:
            continue

        aligned += 1

        if ref_seq[i] == exp_seq[j]:
            identical += 1

        if (
            "CA" in ref_res[i]
            and "CA" in exp_res[j]
        ):
            ref_xyz.append(
                ref_res[i]["CA"].coord
            )

            exp_xyz.append(
                exp_res[j]["CA"].coord
            )

    identity = (
        identical / aligned
        if aligned else 0
    )

    coverage = (
        aligned / len(ref_res)
        if ref_res else 0
    )

    return {
        "identity": identity,
        "coverage": coverage,
        "ref_xyz": np.asarray(ref_xyz),
        "exp_xyz": np.asarray(exp_xyz),
        "aligned_positions": aligned
    }


def calculate_kabsch(reference_xyz, experimental_xyz):

    P = np.asarray(reference_xyz, dtype=float)
    Q = np.asarray(experimental_xyz, dtype=float)

    if len(P) != len(Q) or len(P) < 3:
        raise ValueError(
            "Insufficient corresponding coordinates for alignment."
        )

    p_mean = P.mean(axis=0)
    q_mean = Q.mean(axis=0)

    P_centered = P - p_mean
    Q_centered = Q - q_mean

    H = P_centered.T @ Q_centered

    U, S, Vt = np.linalg.svd(H)

    R = U @ Vt

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = U @ Vt

    t = q_mean - p_mean @ R

    transformed = P @ R + t

    rmsd = np.sqrt(
        np.mean(
            np.sum(
                (transformed - Q) ** 2,
                axis=1
            )
        )
    )

    return R, t, float(rmsd)


# Extract all heavy atoms from M1 pocket residues
pocket_residue_keys = set()

for _, row in pocket_residues_df.iterrows():

    chain_id = str(row["chain"])

    residue_number = int(row["residue_number"])

    insertion_code = row["insertion_code"]

    if pd.isna(insertion_code):
        insertion_code = " "
    else:
        insertion_code = str(insertion_code).strip() or " "

    residue_name = str(row["residue_name"])

    pocket_residue_keys.add(
        (
            chain_id,
            residue_number,
            insertion_code,
            residue_name
        )
    )


m1_pocket_xyz = []
m1_pocket_metadata = []

for chain_id, residue_number, insertion_code, residue_name in pocket_residue_keys:

    if chain_id not in m1_chains:
        continue

    chain = m1_chains[chain_id]

    for residue in chain:

        hetflag, resnum, icode = residue.id

        if (
            resnum == residue_number
            and icode.strip() == insertion_code.strip()
            and residue.resname == residue_name
        ):

            for atom in residue:

                element = str(atom.element).upper()

                if element in {"H", "D"}:
                    continue

                m1_pocket_xyz.append(
                    atom.coord.astype(float)
                )

                m1_pocket_metadata.append({
                    "m1_chain": chain_id,
                    "residue_number": residue_number,
                    "residue_name": residue_name,
                    "atom_name": atom.name,
                    "element": element
                })


m1_pocket_xyz = np.asarray(
    m1_pocket_xyz,
    dtype=float
)

if len(m1_pocket_xyz) == 0:
    raise RuntimeError(
        "No M1 pocket atoms were extracted."
    )

print(
    "M1 pocket heavy atoms:",
    len(m1_pocket_xyz)
)


# Align every experimental structure using a common transformation
pocket_xyz_by_pdb = {}
pocket_metadata_by_pdb = {}
alignment_report = []

for pdb_id in selected_pdbs:

    print("\nAligning:", pdb_id)

    parser = MMCIFParser(
        QUIET=True,
        auth_chains=True,
        auth_residues=True
    )

    experimental_structure = parser.get_structure(
        pdb_id,
        str(cif_paths[pdb_id])
    )

    experimental_model = experimental_structure[0]

    experimental_chains = [
        chain
        for chain in experimental_model
        if len(protein_residues(chain)) >= 100
    ]

    best_assignment = None

    # Try both M1-to-experimental chain assignments.
    for chain_a, chain_b in permutations(
        experimental_chains, 2
    ):

        result_a = align_protein_chains(
            m1_chains["A"],
            chain_a
        )

        result_b = align_protein_chains(
            m1_chains["B"],
            chain_b
        )

        if (
            result_a["identity"] < 0.90
            or result_b["identity"] < 0.90
            or result_a["coverage"] < 0.90
            or result_b["coverage"] < 0.90
        ):
            continue

        ref_xyz = np.vstack([
            result_a["ref_xyz"],
            result_b["ref_xyz"]
        ])

        exp_xyz = np.vstack([
            result_a["exp_xyz"],
            result_b["exp_xyz"]
        ])

        try:
            R, t, rmsd = calculate_kabsch(
                ref_xyz,
                exp_xyz
            )
        except ValueError:
            continue

        if (
            best_assignment is None
            or rmsd < best_assignment["rmsd"]
        ):

            best_assignment = {
                "chain_a": chain_a.id,
                "chain_b": chain_b.id,
                "R": R,
                "t": t,
                "rmsd": rmsd,
                "identity_a": result_a["identity"],
                "identity_b": result_b["identity"],
                "coverage_a": result_a["coverage"],
                "coverage_b": result_b["coverage"]
            }

    if best_assignment is None:

        alignment_report.append({
            "pdb_id": pdb_id,
            "status": "FAILED",
            "reason": "No suitable two-chain assignment"
        })

        continue

    transformed_pocket = (
        m1_pocket_xyz @ best_assignment["R"]
        + best_assignment["t"]
    )

    pocket_xyz_by_pdb[pdb_id] = transformed_pocket

    pocket_metadata_by_pdb[pdb_id] = (
        pd.DataFrame(m1_pocket_metadata)
    )

    alignment_report.append({
        "pdb_id": pdb_id,
        "status": "PASS",
        "experimental_chain_for_M1_A": best_assignment["chain_a"],
        "experimental_chain_for_M1_B": best_assignment["chain_b"],
        "chain_A_identity": best_assignment["identity_a"],
        "chain_B_identity": best_assignment["identity_b"],
        "chain_A_coverage": best_assignment["coverage_a"],
        "chain_B_coverage": best_assignment["coverage_b"],
        "two_chain_common_frame_RMSD_A": best_assignment["rmsd"]
    })


alignment_df = pd.DataFrame(alignment_report)

alignment_df.to_csv(
    OUTPUT_DIR / "experimental_two_chain_alignment_for_ligands.csv",
    index=False
)

print("\n========== ALIGNMENT SUMMARY ==========")

display(alignment_df)

M1 pocket heavy atoms: 98

Aligning: 5KPP

Aligning: 5KPQ

Aligning: 5WRZ

Aligning: 5WS1

Aligning: 7AAC

Aligning: 7ONS

Aligning: 7ONT

Aligning: 8JNZ

Aligning: 9ETQ

Aligning: 9ILN

========== ALIGNMENT SUMMARY ==========


,pdb_id,status,experimental_chain_for_M1_A,experimental_chain_for_M1_B,chain_A_identity,chain_B_identity,chain_A_coverage,chain_B_coverage,two_chain_common_frame_RMSD_A
0,5KPP,PASS,A,B,1.0,1.0,0.997143,0.997126,0.797784
1,5KPQ,PASS,A,B,1.0,1.0,0.997143,0.997126,0.838873
2,5WRZ,PASS,A,B,1.0,1.0,0.997143,0.997126,1.245704
3,5WS1,PASS,A,B,1.0,1.0,0.997143,0.997126,1.234568
4,7AAC,PASS,A,B,1.0,1.0,1.000000,1.000000,0.741034
5,7ONS,PASS,A,B,1.0,1.0,1.000000,1.000000,0.499997
6,7ONT,PASS,A,B,1.0,1.0,1.000000,1.000000,0.748998
7,8JNZ,PASS,B,A,1.0,1.0,1.000000,1.000000,0.792956
8,9ETQ,PASS,A,B,1.0,1.0,1.000000,1.000000,0.755967
9,9ILN,PASS,B,A,1.0,1.0,1.000000,1.000000,0.817862


In [48]:
# CELL 46 — Ligand-to-pocket distance calculations

from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from scipy.spatial.distance import cdist


def cif_column(cif_dict, key):

    value = cif_dict.get(key, [])

    if isinstance(value, str):
        return [value]

    return list(value)


def extract_atom_table(cif_dict):

    fields = {
        "group": "_atom_site.group_PDB",
        "atom": "_atom_site.label_atom_id",
        "comp": "_atom_site.label_comp_id",
        "label_chain": "_atom_site.label_asym_id",
        "auth_chain": "_atom_site.auth_asym_id",
        "element": "_atom_site.type_symbol",
        "x": "_atom_site.Cartn_x",
        "y": "_atom_site.Cartn_y",
        "z": "_atom_site.Cartn_z",
        "label_seq": "_atom_site.label_seq_id",
        "auth_seq": "_atom_site.auth_seq_id"
    }

    data = {
        name: cif_column(cif_dict, key)
        for name, key in fields.items()
    }

    lengths = {
        len(v)
        for v in data.values()
    }

    if len(lengths) != 1:
        raise ValueError(
            "mmCIF atom-site columns have inconsistent lengths."
        )

    return pd.DataFrame(data)


distance_rows = []

for pdb_id in selected_pdbs:

    if pdb_id not in pocket_xyz_by_pdb:
        print("Skipping unaligned structure:", pdb_id)
        continue

    cif_dict = MMCIF2Dict(
        str(cif_paths[pdb_id])
    )

    atom_df = extract_atom_table(cif_dict)

    # Retain heavy-atom records
    atom_df["element"] = (
        atom_df["element"].astype(str).str.upper()
    )

    atom_df = atom_df[
        ~atom_df["element"].isin(["H", "D"])
    ].copy()

    atom_df["x"] = pd.to_numeric(
        atom_df["x"], errors="coerce"
    )

    atom_df["y"] = pd.to_numeric(
        atom_df["y"], errors="coerce"
    )

    atom_df["z"] = pd.to_numeric(
        atom_df["z"], errors="coerce"
    )

    atom_df = atom_df.dropna(
        subset=["x", "y", "z"]
    )

    # Candidate ligand instances for this structure
    structure_instances = instances_df[
        instances_df["pdb_id"].astype(str).str.upper()
        == pdb_id
    ].copy()

    potential_comp_ids = set(
        candidates_df.loc[
            candidates_df["pdb_id"].astype(str).str.upper()
            == pdb_id,
            "comp_id"
        ].astype(str)
    )

    structure_instances = structure_instances[
        structure_instances["comp_id"].astype(str).isin(
            potential_comp_ids
        )
    ]

    pocket_xyz = pocket_xyz_by_pdb[pdb_id]

    pocket_meta = pocket_metadata_by_pdb[pdb_id]

    for _, instance in structure_instances.iterrows():

        comp_id = str(instance["comp_id"])
        label_chain = str(instance["label_asym_id"])
        auth_chain = str(instance["auth_asym_id"])

        ligand_atoms = atom_df[
            (atom_df["comp"].astype(str) == comp_id)
            &
            (atom_df["label_chain"].astype(str) == label_chain)
            &
            (atom_df["group"].astype(str).str.upper() == "HETATM")
        ].copy()

        if ligand_atoms.empty:
            print(
                "No ligand coordinates:",
                pdb_id,
                comp_id,
                label_chain
            )
            continue

        ligand_xyz = ligand_atoms[
            ["x", "y", "z"]
        ].to_numpy(dtype=float)

        distance_matrix = cdist(
            ligand_xyz,
            pocket_xyz
        )

        close_4 = distance_matrix <= 4.0
        close_5 = distance_matrix <= 5.0

        ligand_atoms_4 = int(
            close_4.any(axis=1).sum()
        )

        ligand_atoms_5 = int(
            close_5.any(axis=1).sum()
        )

        contact_pairs_4 = int(
            close_4.sum()
        )

        contacting_pocket_atoms_4 = np.where(
            close_4.any(axis=0)
        )[0]

        contacting_residues = set()

        for atom_index in contacting_pocket_atoms_4:

            meta = pocket_meta.iloc[atom_index]

            contacting_residues.add(
                (
                    meta["m1_chain"],
                    meta["residue_number"],
                    meta["residue_name"]
                )
            )

        # Separate contact evidence from the two M1 chains
        chain_contact_counts = {}

        for m1_chain in ["A", "B"]:

            chain_indices = np.where(
                pocket_meta["m1_chain"].to_numpy()
                == m1_chain
            )[0]

            chain_contact_counts[m1_chain] = int(
                close_4[:, chain_indices].sum()
            )

        distance_rows.append({

            "pdb_id": pdb_id,
            "comp_id": comp_id,
            "label_asym_id": label_chain,
            "auth_asym_id": auth_chain,

            "ligand_heavy_atoms": len(ligand_xyz),

            "minimum_distance_A": float(
                distance_matrix.min()
            ),

            "ligand_atoms_within_4A": ligand_atoms_4,
            "ligand_atoms_within_5A": ligand_atoms_5,

            "atom_pairs_within_4A": contact_pairs_4,

            "contacting_pocket_atoms_4A": len(
                contacting_pocket_atoms_4
            ),

            "contacting_residues_4A": len(
                contacting_residues
            ),

            "M1_chain_A_contact_pairs_4A":
                chain_contact_counts["A"],

            "M1_chain_B_contact_pairs_4A":
                chain_contact_counts["B"]
        })


distance_df = pd.DataFrame(distance_rows)

if distance_df.empty:
    raise RuntimeError(
        "No ligand-pocket distance records were generated."
    )

distance_df.to_csv(
    OUTPUT_DIR / "ligand_pocket_distances.csv",
    index=False
)

print("========== DISTANCE SUMMARY ==========")

print("Ligand instances evaluated:", len(distance_df))

display(
    distance_df.sort_values(
        "minimum_distance_A"
    )
)

========== DISTANCE SUMMARY ==========
Ligand instances evaluated: 20


,pdb_id,comp_id,label_asym_id,auth_asym_id,ligand_heavy_atoms,minimum_distance_A,ligand_atoms_within_4A,ligand_atoms_within_5A,atom_pairs_within_4A,contacting_pocket_atoms_4A,contacting_residues_4A,M1_chain_A_contact_pairs_4A,M1_chain_B_contact_pairs_4A
2,5KPQ,6X2,C,A,32,2.400476,14,16,55,12,3,55,0
19,9ILN,A1D9M,D,B,31,2.451098,12,14,47,9,2,47,0
0,5KPP,6WZ,C,A,34,2.456679,14,16,54,10,3,54,0
6,5WS1,7U9,C,A,20,2.565812,17,18,52,13,3,52,0
8,7AAC,78P,C,A,18,2.662797,15,17,49,13,3,49,0
4,5WRZ,7U3,C,A,19,2.804299,15,17,56,16,3,56,0
15,8JNZ,ERV,G,B,13,2.811423,12,13,40,10,3,40,0
10,7ONS,VKT,C,A,27,3.371240,12,14,34,8,2,34,0
16,9ETQ,A1H63,C,A,30,3.444867,11,14,27,8,2,27,0
12,7ONT,VKQ,C,A,30,3.454614,12,14,30,8,2,30,0


In [49]:
# CELL 47 — Pocket occupancy evaluation

occupancy_df = distance_df.copy()

# Operational contact-evidence criteria.
# These are screening rules, not universal biochemical laws.

occupancy_df["close_to_pocket"] = (
    occupancy_df["minimum_distance_A"] <= 5.0
)

occupancy_df["strong_contact_evidence"] = (
    (occupancy_df["ligand_atoms_within_4A"] >= 3)
    &
    (occupancy_df["atom_pairs_within_4A"] >= 5)
    &
    (occupancy_df["contacting_residues_4A"] >= 2)
)

occupancy_df["pocket_occupancy_status"] = np.select(

    [
        occupancy_df["strong_contact_evidence"],

        occupancy_df["close_to_pocket"]
    ],

    [
        "CONTACT_SUPPORTED",
        "POSSIBLE_CONTACT_REVIEW"
    ],

    default="NOT_SUPPORTED"
)


# Save the detailed result
occupancy_df.to_csv(
    OUTPUT_DIR / "ligand_pocket_occupancy_evaluation.csv",
    index=False
)


print("========== OCCUPANCY EVALUATION ==========")

print(
    occupancy_df["pocket_occupancy_status"]
    .value_counts()
)

display(
    occupancy_df[
        [
            "pdb_id",
            "comp_id",
            "label_asym_id",
            "minimum_distance_A",
            "ligand_atoms_within_4A",
            "atom_pairs_within_4A",
            "contacting_residues_4A",
            "M1_chain_A_contact_pairs_4A",
            "M1_chain_B_contact_pairs_4A",
            "pocket_occupancy_status"
        ]
    ]
)

========== OCCUPANCY EVALUATION ==========
pocket_occupancy_status
CONTACT_SUPPORTED    10
NOT_SUPPORTED        10
Name: count, dtype: int64


,pdb_id,comp_id,label_asym_id,minimum_distance_A,ligand_atoms_within_4A,atom_pairs_within_4A,contacting_residues_4A,M1_chain_A_contact_pairs_4A,M1_chain_B_contact_pairs_4A,pocket_occupancy_status
0,5KPP,6WZ,C,2.456679,14,54,3,54,0,CONTACT_SUPPORTED
1,5KPP,6WZ,D,18.381135,0,0,0,0,0,NOT_SUPPORTED
2,5KPQ,6X2,C,2.400476,14,55,3,55,0,CONTACT_SUPPORTED
3,5KPQ,6X2,D,18.358971,0,0,0,0,0,NOT_SUPPORTED
4,5WRZ,7U3,C,2.804299,15,56,3,56,0,CONTACT_SUPPORTED
5,5WRZ,7U3,D,18.881829,0,0,0,0,0,NOT_SUPPORTED
6,5WS1,7U9,C,2.565812,17,52,3,52,0,CONTACT_SUPPORTED
7,5WS1,7U9,D,19.274798,0,0,0,0,0,NOT_SUPPORTED
8,7AAC,78P,C,2.662797,15,49,3,49,0,CONTACT_SUPPORTED
9,7AAC,78P,E,19.088887,0,0,0,0,0,NOT_SUPPORTED


In [50]:
# CELL 48 — Export structural-validation datasets

supported_pairs_df = occupancy_df[
    occupancy_df["pocket_occupancy_status"]
    == "CONTACT_SUPPORTED"
].copy()

review_pairs_df = occupancy_df[
    occupancy_df["pocket_occupancy_status"]
    == "POSSIBLE_CONTACT_REVIEW"
].copy()

unsupported_pairs_df = occupancy_df[
    occupancy_df["pocket_occupancy_status"]
    == "NOT_SUPPORTED"
].copy()


# Save separate reports
supported_path = (
    OUTPUT_DIR / "supported_target_pocket_pairs.csv"
)

review_path = (
    OUTPUT_DIR / "uncertain_target_pocket_pairs.csv"
)

unsupported_path = (
    OUTPUT_DIR / "unsupported_target_pocket_pairs.csv"
)

supported_pairs_df.to_csv(
    supported_path,
    index=False
)

review_pairs_df.to_csv(
    review_path,
    index=False
)

unsupported_pairs_df.to_csv(
    unsupported_path,
    index=False
)


print("========== CELL 48 SUMMARY ==========")

print(
    "Contact-supported ligand instances:",
    len(supported_pairs_df)
)

print(
    "Possible contacts requiring review:",
    len(review_pairs_df)
)

print(
    "Unsupported ligand instances:",
    len(unsupported_pairs_df)
)

print(
    "Unique supported PDB structures:",
    supported_pairs_df["pdb_id"].nunique()
)

print(
    "Unique supported chemical components:",
    supported_pairs_df["comp_id"].nunique()
)

print("\nSupported pairs:")

display(supported_pairs_df)

print("\nUncertain pairs:")

display(review_pairs_df)

========== CELL 48 SUMMARY ==========
Contact-supported ligand instances: 10
Possible contacts requiring review: 0
Unsupported ligand instances: 10
Unique supported PDB structures: 10
Unique supported chemical components: 10

Supported pairs:


,pdb_id,comp_id,label_asym_id,auth_asym_id,ligand_heavy_atoms,minimum_distance_A,ligand_atoms_within_4A,ligand_atoms_within_5A,atom_pairs_within_4A,contacting_pocket_atoms_4A,contacting_residues_4A,M1_chain_A_contact_pairs_4A,M1_chain_B_contact_pairs_4A,close_to_pocket,strong_contact_evidence,pocket_occupancy_status
0,5KPP,6WZ,C,A,34,2.456679,14,16,54,10,3,54,0,True,True,CONTACT_SUPPORTED
2,5KPQ,6X2,C,A,32,2.400476,14,16,55,12,3,55,0,True,True,CONTACT_SUPPORTED
4,5WRZ,7U3,C,A,19,2.804299,15,17,56,16,3,56,0,True,True,CONTACT_SUPPORTED
6,5WS1,7U9,C,A,20,2.565812,17,18,52,13,3,52,0,True,True,CONTACT_SUPPORTED
8,7AAC,78P,C,A,18,2.662797,15,17,49,13,3,49,0,True,True,CONTACT_SUPPORTED
10,7ONS,VKT,C,A,27,3.371240,12,14,34,8,2,34,0,True,True,CONTACT_SUPPORTED
12,7ONT,VKQ,C,A,30,3.454614,12,14,30,8,2,30,0,True,True,CONTACT_SUPPORTED
15,8JNZ,ERV,G,B,13,2.811423,12,13,40,10,3,40,0,True,True,CONTACT_SUPPORTED
16,9ETQ,A1H63,C,A,30,3.444867,11,14,27,8,2,27,0,True,True,CONTACT_SUPPORTED
19,9ILN,A1D9M,D,B,31,2.451098,12,14,47,9,2,47,0,True,True,CONTACT_SUPPORTED



Uncertain pairs:


,pdb_id,comp_id,label_asym_id,auth_asym_id,ligand_heavy_atoms,minimum_distance_A,ligand_atoms_within_4A,ligand_atoms_within_5A,atom_pairs_within_4A,contacting_pocket_atoms_4A,contacting_residues_4A,M1_chain_A_contact_pairs_4A,M1_chain_B_contact_pairs_4A,close_to_pocket,strong_contact_evidence,pocket_occupancy_status


In [51]:
# CELL 49 — Final validation and readiness report

unique_supported_ligands = (
    supported_pairs_df["comp_id"].nunique()
    if not supported_pairs_df.empty
    else 0
)

unique_supported_structures = (
    supported_pairs_df["pdb_id"].nunique()
    if not supported_pairs_df.empty
    else 0
)

unique_review_ligands = (
    review_pairs_df["comp_id"].nunique()
    if not review_pairs_df.empty
    else 0
)

alignment_pass_count = int(
    (
        alignment_df["status"] == "PASS"
    ).sum()
)

alignment_fail_count = int(
    (
        alignment_df["status"] == "FAILED"
    ).sum()
)


# Structural validation and model-training readiness are separate.
structural_validation_complete = (
    len(distance_df) > 0
)

# A small number of ligand examples is not enough to claim
# that a useful pocket-conditioned generator has been trained.
generator_training_ready = False


final_report = {

    "project": "Member-2 Pocket-Conditioned Molecular Topology",

    "selected_experimental_structures": len(selected_pdbs),

    "structures_successfully_aligned": alignment_pass_count,

    "structures_failed_alignment": alignment_fail_count,

    "ligand_instances_evaluated": int(len(distance_df)),

    "contact_supported_ligand_instances": int(
        len(supported_pairs_df)
    ),

    "uncertain_ligand_instances": int(
        len(review_pairs_df)
    ),

    "unsupported_ligand_instances": int(
        len(unsupported_pairs_df)
    ),

    "unique_contact_supported_chemical_components":
        int(unique_supported_ligands),

    "unique_contact_supported_structures":
        int(unique_supported_structures),

    "unique_uncertain_chemical_components":
        int(unique_review_ligands),

    "structural_validation_complete":
        bool(structural_validation_complete),

    "target_pocket_pairs_identified":
        bool(len(supported_pairs_df) > 0),

    "generator_training_ready":
        bool(generator_training_ready),

    "generator_trained": False,

    "scientific_limitation": (
        "Contact-supported experimental ligand instances are not "
        "equivalent to a sufficiently large, diverse training dataset. "
        "Additional verified target-pocket complexes, molecular graph "
        "preparation, train/validation/test splitting, and model "
        "evaluation are required."
    ),

    "next_steps": [

        "Inspect supported ligand structures and confirm binding-site identity.",

        "Retrieve or reconstruct chemical graphs for supported ligands.",

        "Prepare verified pocket-ligand training examples.",

        "Build the pocket encoder and molecular graph generator.",

        "Evaluate chemical validity, uniqueness, novelty and pocket-conditioning fidelity.",

        "Export generated molecular graphs to Member-3."
    ]
}


report_path = (
    OUTPUT_DIR / "member2_structural_validation_readiness.json"
)

with open(report_path, "w") as f:
    json.dump(final_report, f, indent=2)


print("========== FINAL STRUCTURAL VALIDATION ==========")

print(
    json.dumps(
        final_report,
        indent=2
    )
)

========== FINAL STRUCTURAL VALIDATION ==========
{
  "project": "Member-2 Pocket-Conditioned Molecular Topology",
  "selected_experimental_structures": 10,
  "structures_successfully_aligned": 10,
  "structures_failed_alignment": 0,
  "ligand_instances_evaluated": 20,
  "contact_supported_ligand_instances": 10,
  "uncertain_ligand_instances": 0,
  "unsupported_ligand_instances": 10,
  "unique_contact_supported_chemical_components": 10,
  "unique_contact_supported_structures": 10,
  "unique_uncertain_chemical_components": 0,
  "structural_validation_complete": true,
  "target_pocket_pairs_identified": true,
  "generator_training_ready": false,
  "generator_trained": false,
  "scientific_limitation": "Contact-supported experimental ligand instances are not equivalent to a sufficiently large, diverse training dataset. Additional verified target-pocket complexes, molecular graph preparation, train/validation/test splitting, and model evaluation are required.",
  "next_steps": [
    "Inspe

In [52]:
# CELL 50 — Detailed residue-level ligand contact validation

from scipy.spatial.distance import cdist
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
import pandas as pd
import numpy as np

residue_contact_records = []
chain_validation_records = []

for _, ligand in supported_pairs_df.iterrows():

    pdb_id = str(ligand["pdb_id"])
    comp_id = str(ligand["comp_id"])
    label_chain = str(ligand["label_asym_id"])
    auth_chain = str(ligand["auth_asym_id"])

    # Load experimental atom coordinates
    cif_dict = MMCIF2Dict(str(cif_paths[pdb_id]))
    atom_df = extract_atom_table(cif_dict)

    atom_df["element"] = (
        atom_df["element"].astype(str).str.upper()
    )

    for col in ["x", "y", "z"]:
        atom_df[col] = pd.to_numeric(
            atom_df[col], errors="coerce"
        )

    atom_df = atom_df.dropna(
        subset=["x", "y", "z"]
    )

    # Extract the correct ligand instance
    ligand_atoms = atom_df[
        (atom_df["comp"].astype(str) == comp_id)
        &
        (atom_df["label_chain"].astype(str) == label_chain)
        &
        (atom_df["group"].astype(str).str.upper() == "HETATM")
        &
        (~atom_df["element"].isin(["H", "D"]))
    ]

    if ligand_atoms.empty:
        continue

    ligand_xyz = ligand_atoms[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    # Retrieve transformed M1 pocket coordinates
    pocket_xyz = pocket_xyz_by_pdb[pdb_id]
    pocket_meta = pocket_metadata_by_pdb[pdb_id]

    distance_matrix = cdist(
        ligand_xyz,
        pocket_xyz
    )

    # Check experimental chain assignment
    alignment_row = alignment_df[
        alignment_df["pdb_id"] == pdb_id
    ].iloc[0]

    expected_chain_A = str(
        alignment_row["experimental_chain_for_M1_A"]
    )

    expected_chain_B = str(
        alignment_row["experimental_chain_for_M1_B"]
    )

    chain_validation_records.append({

        "pdb_id": pdb_id,
        "comp_id": comp_id,
        "ligand_label_chain": label_chain,
        "ligand_auth_chain": auth_chain,

        "experimental_chain_for_M1_A": expected_chain_A,
        "experimental_chain_for_M1_B": expected_chain_B,

        "ligand_on_M1_A_copy": (
            auth_chain == expected_chain_A
        ),

        "ligand_on_M1_B_copy": (
            auth_chain == expected_chain_B
        )
    })

    # Group pocket atoms by M1 residue
    grouped = pocket_meta.groupby(
        [
            "m1_chain",
            "residue_number",
            "residue_name"
        ]
    )

    for residue_key, group in grouped:

        atom_indices = group.index.to_numpy()

        residue_distances = distance_matrix[
            :, atom_indices
        ]

        min_distance = float(
            residue_distances.min()
        )

        atom_pairs_4A = int(
            (residue_distances <= 4.0).sum()
        )

        ligand_atoms_4A = int(
            (residue_distances <= 4.0)
            .any(axis=1)
            .sum()
        )

        if min_distance <= 5.0:

            residue_contact_records.append({

                "pdb_id": pdb_id,
                "comp_id": comp_id,

                "m1_chain": residue_key[0],
                "residue_number": residue_key[1],
                "residue_name": residue_key[2],

                "minimum_distance_A": min_distance,

                "ligand_atoms_within_4A": ligand_atoms_4A,

                "atom_pairs_within_4A": atom_pairs_4A
            })


residue_contact_df = pd.DataFrame(
    residue_contact_records
)

chain_validation_df = pd.DataFrame(
    chain_validation_records
)

residue_contact_df = residue_contact_df.sort_values(
    ["pdb_id", "minimum_distance_A"]
)

residue_contact_df.to_csv(
    OUTPUT_DIR / "ligand_residue_level_contacts.csv",
    index=False
)

chain_validation_df.to_csv(
    OUTPUT_DIR / "ligand_chain_assignment_validation.csv",
    index=False
)

print("========== CELL 50 SUMMARY ==========")

print(
    "Residue-level contact records:",
    len(residue_contact_df)
)

print("\nContacting residues:")

display(residue_contact_df)

print("\nLigand-to-protein chain assignment:")

display(chain_validation_df)

========== CELL 50 SUMMARY ==========
Residue-level contact records: 30

Contacting residues:


,pdb_id,comp_id,m1_chain,residue_number,residue_name,minimum_distance_A,ligand_atoms_within_4A,atom_pairs_within_4A
1,5KPP,6WZ,A,907,TYR,2.456679,13,52
2,5KPP,6WZ,A,988,GLU,3.831832,1,1
0,5KPP,6WZ,A,903,LYS,3.862783,1,1
4,5KPQ,6X2,A,907,TYR,2.400476,13,51
5,5KPQ,6X2,A,988,GLU,3.767339,1,3
3,5KPQ,6X2,A,903,LYS,3.836993,1,1
6,5WRZ,7U3,A,903,LYS,2.804299,2,5
7,5WRZ,7U3,A,907,TYR,2.944162,14,46
8,5WRZ,7U3,A,988,GLU,3.336391,2,5
10,5WS1,7U9,A,907,TYR,2.565812,16,46



Ligand-to-protein chain assignment:


,pdb_id,comp_id,ligand_label_chain,ligand_auth_chain,experimental_chain_for_M1_A,experimental_chain_for_M1_B,ligand_on_M1_A_copy,ligand_on_M1_B_copy
0,5KPP,6WZ,C,A,A,B,True,False
1,5KPQ,6X2,C,A,A,B,True,False
2,5WRZ,7U3,C,A,A,B,True,False
3,5WS1,7U9,C,A,A,B,True,False
4,7AAC,78P,C,A,A,B,True,False
5,7ONS,VKT,C,A,A,B,True,False
6,7ONT,VKQ,C,A,A,B,True,False
7,8JNZ,ERV,G,B,B,A,True,False
8,9ETQ,A1H63,C,A,A,B,True,False
9,9ILN,A1D9M,D,B,B,A,True,False


In [54]:
# CELL 50A — INSTALL RDKit

%pip install -q rdkit

print("RDKit installation completed.")

RDKit installation completed.


In [55]:
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors

print("RDKit version:", rdkit.__version__)

mol = Chem.MolFromSmiles("CCO")

if mol is None:
    raise RuntimeError("RDKit validation failed.")

print("Test molecule:", Chem.MolToSmiles(mol))
print("Molecular weight:", Descriptors.MolWt(mol))

RDKit version: 2026.09.2
Test molecule: CCO
Molecular weight: 46.069


In [56]:
# ============================================================
# CELL 51 — RETRIEVE AND VALIDATE EXPERIMENTAL LIGAND GRAPHS
# CPU ONLY
# ============================================================

import json
import requests
import pandas as pd
import numpy as np

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# ------------------------------------------------------------
# 1. Locate the existing Cell 48 and Cell 50 outputs
# ------------------------------------------------------------

SUPPORTED_FILE = OUTPUT_DIR / "supported_target_pocket_pairs.csv"
CHAIN_FILE = OUTPUT_DIR / "ligand_chain_assignment_validation.csv"

if not SUPPORTED_FILE.exists():
    raise FileNotFoundError(f"Missing: {SUPPORTED_FILE}")

if not CHAIN_FILE.exists():
    raise FileNotFoundError(f"Missing: {CHAIN_FILE}")

supported_df = pd.read_csv(SUPPORTED_FILE)
chain_df = pd.read_csv(CHAIN_FILE)

required_supported = {"pdb_id", "comp_id"}
required_chain = {
    "pdb_id",
    "comp_id",
    "ligand_on_M1_A_copy",
    "ligand_on_M1_B_copy"
}

if not required_supported.issubset(supported_df.columns):
    raise ValueError("Supported-pair table is missing required columns.")

if not required_chain.issubset(chain_df.columns):
    raise ValueError("Chain-assignment table is missing required columns.")

# ------------------------------------------------------------
# 2. Keep only ligand instances assigned to the validated
#    M1 Chain A pocket
# ------------------------------------------------------------

chain_df["ligand_on_M1_A_copy"] = (
    chain_df["ligand_on_M1_A_copy"].astype(bool)
)

chain_df["ligand_on_M1_B_copy"] = (
    chain_df["ligand_on_M1_B_copy"].astype(bool)
)

valid_chain_df = chain_df[
    chain_df["ligand_on_M1_A_copy"] &
    ~chain_df["ligand_on_M1_B_copy"]
].copy()

candidate_pairs = supported_df.merge(
    valid_chain_df,
    on=["pdb_id", "comp_id"],
    how="inner",
    suffixes=("_support", "_chain")
)

candidate_pairs = candidate_pairs.drop_duplicates(
    subset=["pdb_id", "comp_id"]
).reset_index(drop=True)

if candidate_pairs.empty:
    raise ValueError(
        "No ligand examples passed the existing chain-assignment validation."
    )

print("Validated ligand instances:", len(candidate_pairs))
print("Unique chemical components:", candidate_pairs["comp_id"].nunique())

# ------------------------------------------------------------
# 3. Retrieve chemical-component information from RCSB
# ------------------------------------------------------------

RCSB_CHEMCOMP_URL = (
    "https://data.rcsb.org/rest/v1/core/chemcomp/{}"
)

session = requests.Session()
session.headers.update({
    "User-Agent": "Member2-Research-Workflow/1.0"
})


def fetch_chemcomp(comp_id):
    url = RCSB_CHEMCOMP_URL.format(comp_id)

    response = session.get(url, timeout=30)
    response.raise_for_status()

    return response.json()


def choose_smiles(record):
    descriptors = record.get(
        "pdbx_chem_comp_descriptor", []
    )

    candidates = []

    for item in descriptors:
        descriptor_type = str(item.get("type", "")).upper()
        program = str(item.get("program", "")).upper()
        smiles = item.get("descriptor")

        if not smiles:
            continue

        if "SMILES" not in descriptor_type:
            continue

        score = 0

        if "CANONICAL" in descriptor_type:
            score += 10

        if "STEREO" in descriptor_type:
            score += 5

        if "OPENEYE" in program:
            score += 3

        if "CACTVS" in program:
            score += 2

        candidates.append((score, smiles, descriptor_type, program))

    candidates.sort(reverse=True)

    for score, smiles, dtype, program in candidates:
        mol = Chem.MolFromSmiles(smiles)

        if mol is not None:
            return smiles, dtype, program

    return None, None, None


# ------------------------------------------------------------
# 4. Convert a validated RDKit molecule into a graph
# ------------------------------------------------------------

def molecule_to_graph(mol):

    atoms = []

    for atom in mol.GetAtoms():

        atoms.append({
            "atom_index": atom.GetIdx(),
            "atomic_number": atom.GetAtomicNum(),
            "formal_charge": atom.GetFormalCharge(),
            "degree": atom.GetDegree(),
            "total_valence": atom.GetTotalValence(),
            "aromatic": atom.GetIsAromatic(),
            "in_ring": atom.IsInRing(),
            "hybridization": str(atom.GetHybridization()),
            "total_hydrogens": atom.GetTotalNumHs()
        })

    bonds = []
    edge_index = []
    edge_features = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        bond_feature = {
            "bond_type": str(bond.GetBondType()),
            "aromatic": bond.GetIsAromatic(),
            "conjugated": bond.GetIsConjugated(),
            "in_ring": bond.IsInRing(),
            "stereo": str(bond.GetStereo())
        }

        bonds.append({
            "source": i,
            "target": j,
            **bond_feature
        })

        # Bidirectional edges for graph-learning compatibility
        edge_index.extend([[i, j], [j, i]])
        edge_features.extend([bond_feature, bond_feature])

    return {
        "num_atoms": len(atoms),
        "num_bonds": len(bonds),
        "node_features": atoms,
        "bonds": bonds,
        "edge_index": edge_index,
        "edge_features": edge_features
    }


# ------------------------------------------------------------
# 5. Retrieve each unique chemical component
# ------------------------------------------------------------

ligand_graph_records = []
ligand_metadata_rows = []
ligand_errors = []

unique_components = sorted(
    candidate_pairs["comp_id"].astype(str).unique()
)

for comp_id in unique_components:

    try:

        record = fetch_chemcomp(comp_id)

        smiles, smiles_type, smiles_program = choose_smiles(record)

        if smiles is None:
            raise ValueError("No usable SMILES descriptor found.")

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            raise ValueError("RDKit could not parse the retrieved SMILES.")

        Chem.SanitizeMol(mol)

        canonical_smiles = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        graph = molecule_to_graph(mol)

        pdb_sources = sorted(
            candidate_pairs.loc[
                candidate_pairs["comp_id"].astype(str) == comp_id,
                "pdb_id"
            ].astype(str).unique().tolist()
        )

        chem_info = record.get("chem_comp", {})

        graph_record = {
            "graph_id": f"CCD_{comp_id}",
            "comp_id": comp_id,
            "source": "RCSB_Chemical_Component_Dictionary",
            "source_pdb_ids": pdb_sources,
            "canonical_smiles": canonical_smiles,
            "smiles_descriptor_type": smiles_type,
            "smiles_descriptor_program": smiles_program,
            "molecular_formula": chem_info.get("formula"),
            "molecular_name": chem_info.get("name"),
            "molecular_weight": float(Descriptors.MolWt(mol)),
            "validated_by_rdkit": True,
            **graph
        }

        ligand_graph_records.append(graph_record)

        ligand_metadata_rows.append({
            "comp_id": comp_id,
            "source_pdb_ids": ";".join(pdb_sources),
            "canonical_smiles": canonical_smiles,
            "molecular_formula": chem_info.get("formula"),
            "molecular_weight": float(Descriptors.MolWt(mol)),
            "num_atoms": graph["num_atoms"],
            "num_bonds": graph["num_bonds"],
            "rdkit_valid": True,
            "retrieval_status": "SUCCESS"
        })

    except Exception as exc:

        ligand_errors.append({
            "comp_id": comp_id,
            "error": str(exc)
        })


# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

LIGAND_GRAPH_FILE = OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"

with open(LIGAND_GRAPH_FILE, "w") as f:

    for record in ligand_graph_records:
        f.write(json.dumps(record) + "\n")


ligand_metadata_df = pd.DataFrame(ligand_metadata_rows)
ligand_metadata_df.to_csv(
    OUTPUT_DIR / "validated_target_ligand_metadata.csv",
    index=False
)

pd.DataFrame(ligand_errors).to_csv(
    OUTPUT_DIR / "ligand_retrieval_errors.csv",
    index=False
)

print("\n========== CELL 51 SUMMARY ==========")
print("Unique components discovered:", len(unique_components))
print("Successfully validated:", len(ligand_graph_records))
print("Failed retrievals:", len(ligand_errors))
print("Graph file:", LIGAND_GRAPH_FILE)

display(ligand_metadata_df)

Validated ligand instances: 10
Unique chemical components: 10

========== CELL 51 SUMMARY ==========
Unique components discovered: 10
Successfully validated: 10
Failed retrievals: 0
Graph file: /content/drive/MyDrive/Member2_Project/member2_outputs/validated_target_ligand_graphs.jsonl


,comp_id,source_pdb_ids,canonical_smiles,molecular_formula,molecular_weight,num_atoms,num_bonds,rdkit_valid,retrieval_status
0,6WZ,5KPP,C[C@@H]1CN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc4...,C23 H22 F4 N4 O3,478.446,34,37,True,SUCCESS
1,6X2,5KPQ,CCCN1CCN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc43)...,C24 H27 F N4 O3,438.503,32,35,True,SUCCESS
2,78P,7AAC,C[C@]1(c2nc3cccc(C(N)=O)c3[nH]2)CCCN1,C13 H16 N4 O,244.298,18,20,True,SUCCESS
3,7U3,5WRZ,O=c1[nH]nc(OC[C@@H]2CCNC2)c2ccc(F)cc12,C13 H14 F N3 O2,263.272,19,21,True,SUCCESS
4,7U9,5WS1,NC(=O)c1cccc2[nH]c(C(=O)N3CC[C@@H](N)C3)nc12,C13 H15 N5 O2,273.296,20,22,True,SUCCESS
5,A1D9M,9ILN,CCc1cn(Cc2ccc(F)c(C(=O)N3CCN(CC)[C@H](CCO)C3)c...,C22 H29 F N4 O4,432.496,31,33,True,SUCCESS
6,A1H63,9ETQ,CCc1cc2ncc(CN3CCN(c4ccc(C(=O)NC)nc4)CC3)cc2[nH...,C22 H26 N6 O2,406.490,30,33,True,SUCCESS
7,ERV,8JNZ,Cc1cnc2c(C(N)=O)cnn2c1,C8 H8 N4 O,176.179,13,14,True,SUCCESS
8,VKQ,7ONT,CCc1cc2ccc(CN3CCN(c4ccc(C(=O)NC)nc4)CC3)cc2[nH...,C23 H27 N5 O2,405.502,30,33,True,SUCCESS
9,VKT,7ONS,CCc1cc2ccc(CN3CCN(c4ncc(C)n4C)CC3)cc2[nH]c1=O,C21 H27 N5 O,365.481,27,30,True,SUCCESS


In [57]:
# ============================================================
# CELL 52 — CHEMICAL DIVERSITY ANALYSIS
# CPU ONLY
# ============================================================

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
import json
import numpy as np
import pandas as pd

GRAPH_FILE = OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"

if not GRAPH_FILE.exists():
    raise FileNotFoundError("Run Cell 51 first.")

with open(GRAPH_FILE) as f:
    target_graphs = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

if not target_graphs:
    raise ValueError("No validated ligand graphs are available.")

fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048
    )
)

molecules = []
fingerprints = []
scaffold_rows = []

for record in target_graphs:

    mol = Chem.MolFromSmiles(record["canonical_smiles"])

    if mol is None:
        continue

    molecules.append(mol)

    fingerprints.append(
        fingerprint_generator.GetFingerprint(mol)
    )

    scaffold = MurckoScaffold.GetScaffoldForMol(mol)

    scaffold_smiles = (
        Chem.MolToSmiles(scaffold, canonical=True)
        if scaffold.GetNumAtoms() > 0
        else "ACYCLIC_OR_NO_SCAFFOLD"
    )

    scaffold_rows.append({
        "comp_id": record["comp_id"],
        "scaffold_smiles": scaffold_smiles
    })


if not fingerprints:
    raise ValueError("No valid fingerprints could be generated.")

# ------------------------------------------------------------
# Pairwise Tanimoto similarity
# ------------------------------------------------------------

similarity_rows = []

for i in range(len(fingerprints)):

    for j in range(i + 1, len(fingerprints)):

        similarity = DataStructs.TanimotoSimilarity(
            fingerprints[i],
            fingerprints[j]
        )

        similarity_rows.append({
            "ligand_1": target_graphs[i]["comp_id"],
            "ligand_2": target_graphs[j]["comp_id"],
            "tanimoto_similarity": float(similarity)
        })


similarity_df = pd.DataFrame(similarity_rows)

if not similarity_df.empty:
    similarity_df.to_csv(
        OUTPUT_DIR / "target_ligand_pairwise_similarity.csv",
        index=False
    )

scaffold_df = pd.DataFrame(scaffold_rows)

scaffold_df.to_csv(
    OUTPUT_DIR / "target_ligand_scaffolds.csv",
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

unique_smiles = {
    record["canonical_smiles"]
    for record in target_graphs
}

unique_scaffolds = set(
    scaffold_df["scaffold_smiles"]
)

mean_similarity = (
    similarity_df["tanimoto_similarity"].mean()
    if not similarity_df.empty
    else None
)

diversity_report = {
    "validated_ligand_graphs": len(target_graphs),
    "unique_canonical_structures": len(unique_smiles),
    "unique_scaffolds": len(unique_scaffolds),
    "pairwise_comparisons": len(similarity_df),
    "mean_pairwise_tanimoto": (
        float(mean_similarity)
        if mean_similarity is not None
        else None
    )
}

with open(
    OUTPUT_DIR / "target_ligand_diversity_report.json",
    "w"
) as f:
    json.dump(diversity_report, f, indent=4)


print("\n========== CELL 52 SUMMARY ==========")
print(json.dumps(diversity_report, indent=4))

display(scaffold_df)
display(similarity_df.head(20))


========== CELL 52 SUMMARY ==========
{
    "validated_ligand_graphs": 10,
    "unique_canonical_structures": 10,
    "unique_scaffolds": 9,
    "pairwise_comparisons": 45,
    "mean_pairwise_tanimoto": 0.22419299938470283
}


,comp_id,scaffold_smiles
0,6WZ,O=C(c1cccc(Cn2c(=O)[nH]c(=O)c3ccccc32)c1)N1CCNCC1
1,6X2,O=C(c1cccc(Cn2c(=O)[nH]c(=O)c3ccccc32)c1)N1CCNCC1
2,78P,c1ccc2[nH]c(C3CCCN3)nc2c1
3,7U3,O=c1[nH]nc(OC[C@@H]2CCNC2)c2ccccc12
4,7U9,O=C(c1nc2ccccc2[nH]1)N1CCCC1
5,A1D9M,O=C(c1cccc(Cn2ccc(=O)[nH]c2=O)c1)N1CCNCC1
6,A1H63,O=c1ccc2ncc(CN3CCN(c4cccnc4)CC3)cc2[nH]1
7,ERV,c1cnc2ccnn2c1
8,VKQ,O=c1ccc2ccc(CN3CCN(c4cccnc4)CC3)cc2[nH]1
9,VKT,O=c1ccc2ccc(CN3CCN(c4ncc[nH]4)CC3)cc2[nH]1


,ligand_1,ligand_2,tanimoto_similarity
0,6WZ,6X2,0.811594
1,6WZ,78P,0.130435
2,6WZ,7U3,0.150538
3,6WZ,7U9,0.202247
4,6WZ,A1D9M,0.481928
5,6WZ,A1H63,0.178218
6,6WZ,ERV,0.120482
7,6WZ,VKQ,0.200000
8,6WZ,VKT,0.187500
9,6X2,78P,0.131868


In [58]:
# ============================================================
# CELL 53 — BUILD TARGET-SPECIFIC TRAINING MANIFEST
# CPU ONLY
# ============================================================

import hashlib
import json
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Fingerprint the current M1 input
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def current_m1_input_fingerprint():

    required_files = [
        Path(TENSOR_PATH),
        Path(MASK_PATH)
    ]

    if "metadata_path" in globals():

        if metadata_path and Path(metadata_path).exists():
            required_files.append(Path(metadata_path))

    # Include the M1 structures used for pocket validation.
    pdb_files = sorted(
        Path(TENSOR_PATH).parent.rglob("*.pdb")
    )

    required_files.extend(pdb_files)

    unique_files = sorted(
        {p.resolve() for p in required_files if p.exists()}
    )

    if not unique_files:
        raise FileNotFoundError("Could not identify M1 input files.")

    file_hashes = {}

    for path in unique_files:

        file_hashes[str(path)] = sha256_file(path)

    combined = hashlib.sha256(
        json.dumps(
            file_hashes,
            sort_keys=True
        ).encode()
    ).hexdigest()

    return {
        "combined_sha256": combined,
        "file_hashes": file_hashes
    }


m1_fingerprint = current_m1_input_fingerprint()

# ------------------------------------------------------------
# 2. Load validated target-specific ligand graphs
# ------------------------------------------------------------

GRAPH_FILE = OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"

with open(GRAPH_FILE) as f:

    target_graphs = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

if not target_graphs:
    raise ValueError("No target-specific ligand graphs available.")

# ------------------------------------------------------------
# 3. Create an explicit training manifest
# ------------------------------------------------------------

training_examples = []

for graph in target_graphs:

    training_examples.append({

        "example_id": graph["graph_id"],

        "ligand_component_id": graph["comp_id"],

        "source_pdb_ids": graph["source_pdb_ids"],

        "canonical_smiles": graph["canonical_smiles"],

        "label_type": "EXPERIMENTAL_STRUCTURAL_CONTACT_SUPPORTED",

        "target_specific": True,

        "pocket_id": str(POCKET_ID),

        "m1_fingerprint": m1_fingerprint["combined_sha256"],

        "validated": graph["validated_by_rdkit"],

        "training_split": "NOT_ASSIGNED"
    })


manifest = {
    "schema_version": "1.0",

    "member": "M2",

    "purpose": "Pocket-conditioned molecular topology data preparation",

    "m1_input_fingerprint": m1_fingerprint,

    "selected_pocket_id": str(POCKET_ID),

    "target_specific_examples": training_examples,

    "target_specific_example_count": len(training_examples),

    "generator_trained": False,

    "scientific_note": (
        "These are experimentally observed, geometrically validated "
        "ligand-pocket reference examples. They are not generated "
        "molecules and do not provide direct binding-affinity labels."
    )
}

with open(
    OUTPUT_DIR / "member2_training_manifest.json",
    "w"
) as f:

    json.dump(manifest, f, indent=4)


# ------------------------------------------------------------
# 4. Report general datasets separately
# ------------------------------------------------------------

general_dataset_report = {}

for filename in [
    "crossdocked_ligand_graphs_v2.jsonl",
    "chembl37_general_graphs.jsonl"
]:

    path = OUTPUT_DIR / filename

    count = 0

    if path.exists():

        with open(path) as f:

            for line in f:

                if line.strip():
                    count += 1

    general_dataset_report[filename] = {
        "records": count,
        "target_specific": False,
        "role": "GENERAL_MOLECULAR_GRAPH_DATA"
    }


with open(
    OUTPUT_DIR / "general_dataset_inventory.json",
    "w"
) as f:

    json.dump(general_dataset_report, f, indent=4)


print("\n========== CELL 53 SUMMARY ==========")
print("Target-specific examples:", len(training_examples))
print("M1 fingerprint:", m1_fingerprint["combined_sha256"])

print("\nGeneral dataset inventory:")
print(json.dumps(general_dataset_report, indent=4))

print("\nTraining manifest saved.")


========== CELL 53 SUMMARY ==========
Target-specific examples: 10
M1 fingerprint: 28bee102f74135c461c04555fb0b6341dc87a0be755d4e1eaa5f6743228bc052

General dataset inventory:
{
    "crossdocked_ligand_graphs_v2.jsonl": {
        "records": 495,
        "target_specific": false,
        "role": "GENERAL_MOLECULAR_GRAPH_DATA"
    },
    "chembl37_general_graphs.jsonl": {
        "records": 50000,
        "target_specific": false,
        "role": "GENERAL_MOLECULAR_GRAPH_DATA"
    }
}

Training manifest saved.


In [59]:
# ============================================================
# CELL 54 — EXPORT MEMBER-2 TO MEMBER-3 INTERFACE
# CPU ONLY
# ============================================================

import json
import numpy as np

# ------------------------------------------------------------
# 1. Validate current M1 tensors
# ------------------------------------------------------------

if tensor.ndim != 3:
    raise ValueError("M1 tensor is no longer three-dimensional.")

if mask.shape != tensor.shape[:2]:
    raise ValueError("M1 tensor and mask shapes are incompatible.")

if not np.isfinite(tensor).all():
    raise ValueError("M1 tensor contains NaN or infinite values.")

if len(pocket_tokens) != tensor.shape[0]:
    raise ValueError(
        "Pocket-token count does not match the current M1 conformer count."
    )

# ------------------------------------------------------------
# 2. Load the validated reference graphs
# ------------------------------------------------------------

GRAPH_FILE = OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"

with open(GRAPH_FILE) as f:

    reference_graphs = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

if not reference_graphs:
    raise ValueError("No validated reference ligand graphs found.")

# ------------------------------------------------------------
# 3. Recheck M1 fingerprint
# ------------------------------------------------------------

current_fingerprint = current_m1_input_fingerprint()

with open(
    OUTPUT_DIR / "member2_training_manifest.json"
) as f:

    training_manifest = json.load(f)

stored_fingerprint = training_manifest[
    "m1_input_fingerprint"
]["combined_sha256"]

if current_fingerprint["combined_sha256"] != stored_fingerprint:

    raise RuntimeError(
        "M1 input changed after ligand validation. "
        "Repeat the structural validation workflow before exporting."
    )

# ------------------------------------------------------------
# 4. Prepare pocket conditioning
# ------------------------------------------------------------

pocket_conditioning = {

    "pocket_id": str(POCKET_ID),

    "num_conformers": int(tensor.shape[0]),

    "num_feature_slots": int(tensor.shape[1]),

    "num_channels": int(tensor.shape[2]),

    "mask_shape": list(mask.shape),

    "tensor_shape": list(tensor.shape),

    "feature_vocabulary": [
        str(x) for x in FEATURE_VOCAB
    ],

    "tokens_per_conformer": [
        np.asarray(tokens, dtype=float).tolist()
        for tokens in pocket_tokens
    ],

    "conditioning_source": "MEMBER_1_OUTPUT",

    "m1_fingerprint": stored_fingerprint
}

# ------------------------------------------------------------
# 5. Define the molecular graph contract
# ------------------------------------------------------------

graph_schema = {

    "node_features": [
        "atomic_number",
        "formal_charge",
        "degree",
        "total_valence",
        "aromatic",
        "in_ring",
        "hybridization",
        "total_hydrogens"
    ],

    "bond_features": [
        "bond_type",
        "aromatic",
        "conjugated",
        "in_ring",
        "stereo"
    ],

    "edge_representation": "BIDIRECTIONAL",

    "coordinates": (
        "NOT_INCLUDED; M3 is responsible for 3D generation."
    )
}

# ------------------------------------------------------------
# 6. Build the M3 handoff
# ------------------------------------------------------------

m3_interface = {

    "interface_version": "1.0",

    "producer": "MEMBER_2",

    "consumer": "MEMBER_3",

    "architecture_role": (
        "Pocket-conditioned 2D molecular topology preparation"
    ),

    "pocket_conditioning": pocket_conditioning,

    "reference_ligand_graphs": reference_graphs,

    "reference_graph_count": len(reference_graphs),

    "graph_schema": graph_schema,

    # Important: no generated molecules exist yet.
    "generated_candidate_graphs": [],

    "generation_status": "NOT_GENERATED",

    "generator_trained": False,

    "training_manifest": (
        "member2_training_manifest.json"
    ),

    "scientific_note": (
        "Reference ligand graphs are experimentally observed "
        "structures. They must not be treated as generated candidates. "
        "M3 may use the graph schema and pocket-conditioning interface "
        "for downstream integration."
    )
}

INTERFACE_FILE = (
    OUTPUT_DIR / "member2_to_member3_interface.json"
)

with open(INTERFACE_FILE, "w") as f:

    json.dump(m3_interface, f, indent=4)

print("\n========== CELL 54 SUMMARY ==========")
print("Pocket ID:", POCKET_ID)
print("M1 conformers:", tensor.shape[0])
print("Pocket feature slots:", tensor.shape[1])
print("Reference ligand graphs:", len(reference_graphs))
print("Generated candidates:", 0)
print("Generator trained:", False)
print("M3 interface:", INTERFACE_FILE)


========== CELL 54 SUMMARY ==========
Pocket ID: 1
M1 conformers: 5
Pocket feature slots: 95
Reference ligand graphs: 10
Generated candidates: 0
Generator trained: False
M3 interface: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_to_member3_interface.json


In [60]:
# ============================================================
# CELL 55 — FINAL MEMBER-2 READINESS AUDIT
# CPU ONLY
# ============================================================

import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. Check required outputs
# ------------------------------------------------------------

required_outputs = {

    "validated_ligand_graphs": (
        OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"
    ),

    "ligand_metadata": (
        OUTPUT_DIR / "validated_target_ligand_metadata.csv"
    ),

    "diversity_report": (
        OUTPUT_DIR / "target_ligand_diversity_report.json"
    ),

    "training_manifest": (
        OUTPUT_DIR / "member2_training_manifest.json"
    ),

    "m3_interface": (
        OUTPUT_DIR / "member2_to_member3_interface.json"
    )
}

output_checks = {

    name: path.exists()

    for name, path in required_outputs.items()
}

# ------------------------------------------------------------
# 2. Load reports
# ------------------------------------------------------------

with open(
    OUTPUT_DIR / "target_ligand_diversity_report.json"
) as f:

    diversity_report = json.load(f)

with open(
    OUTPUT_DIR / "member2_training_manifest.json"
) as f:

    training_manifest = json.load(f)

with open(
    OUTPUT_DIR / "member2_to_member3_interface.json"
) as f:

    interface = json.load(f)

# ------------------------------------------------------------
# 3. Verify current M1 fingerprint
# ------------------------------------------------------------

current_fingerprint = current_m1_input_fingerprint()

stored_fingerprint = training_manifest[
    "m1_input_fingerprint"
]["combined_sha256"]

m1_unchanged = (
    current_fingerprint["combined_sha256"] == stored_fingerprint
)

# ------------------------------------------------------------
# 4. Readiness assessment
# ------------------------------------------------------------

reference_count = len(
    interface["reference_ligand_graphs"]
)

checks = {

    "m1_tensor_valid": bool(
        tensor.ndim == 3 and
        mask.shape == tensor.shape[:2]
    ),

    "m1_pocket_tokens_available": bool(
        len(pocket_tokens) == tensor.shape[0]
    ),

    "ligand_graphs_available": reference_count > 0,

    "ligand_metadata_available": output_checks[
        "ligand_metadata"
    ],

    "chemical_diversity_evaluated": output_checks[
        "diversity_report"
    ],

    "training_manifest_available": output_checks[
        "training_manifest"
    ],

    "m1_input_unchanged": m1_unchanged,

    "m3_interface_available": output_checks[
        "m3_interface"
    ]
}

# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

preparation_complete = all(checks.values())

generator_trained = False

generator_training_ready = False

final_report = {

    "member": "M2",

    "architecture": (
        "Pocket-conditioned 2D molecular topology generation"
    ),

    "preparation_complete": preparation_complete,

    "reference_ligand_count": reference_count,

    "unique_structures": diversity_report[
        "unique_canonical_structures"
    ],

    "unique_scaffolds": diversity_report[
        "unique_scaffolds"
    ],

    "generator_trained": generator_trained,

    "generator_training_ready": generator_training_ready,

    "m3_interface_exported": output_checks[
        "m3_interface"
    ],

    "m1_input_unchanged": m1_unchanged,

    "checks": checks,

    "final_status": (
        "REFERENCE_DATA_PREPARATION_COMPLETE"
        if preparation_complete
        else "PREPARATION_INCOMPLETE"
    ),

    "limitations": [

        "Reference ligand examples are not generated molecules.",

        "Geometric contact validation is not a binding-affinity measurement.",

        "Ten examples are insufficient to train a reliable "
        "deep molecular generator from scratch.",

        "General ChEMBL-37 and CrossDocked2020 graphs are not "
        "automatically target-specific.",

        "A trained pocket-conditioned generator requires a separate "
        "training and evaluation stage."
    ]
}

with open(
    OUTPUT_DIR / "member2_final_readiness_report.json",
    "w"
) as f:

    json.dump(final_report, f, indent=4)


print("\n========== FINAL MEMBER-2 REPORT ==========")

print(json.dumps(final_report, indent=4))

print("\n========== OUTPUT FILES ==========")

for name, path in required_outputs.items():

    print(
        name,
        "->",
        "AVAILABLE" if path.exists() else "MISSING"
    )


========== FINAL MEMBER-2 REPORT ==========
{
    "member": "M2",
    "architecture": "Pocket-conditioned 2D molecular topology generation",
    "preparation_complete": true,
    "reference_ligand_count": 10,
    "unique_structures": 10,
    "unique_scaffolds": 9,
    "generator_trained": false,
    "generator_training_ready": false,
    "m3_interface_exported": true,
    "m1_input_unchanged": true,
    "checks": {
        "m1_tensor_valid": true,
        "m1_pocket_tokens_available": true,
        "ligand_graphs_available": true,
        "ligand_metadata_available": true,
        "chemical_diversity_evaluated": true,
        "training_manifest_available": true,
        "m1_input_unchanged": true,
        "m3_interface_available": true
    },
    "final_status": "REFERENCE_DATA_PREPARATION_COMPLETE",
    "limitations": [
        "Reference ligand examples are not generated molecules.",
        "Geometric contact validation is not a binding-affinity measurement.",
        "Ten examples

In [61]:
# CELL 56 — TRAINING DATA INSPECTION

import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"

files_to_check = {
    "ChEMBL": OUT / "chembl37_general_graphs.jsonl",
    "CrossDocked": OUT / "crossdocked_ligand_graphs_v2.jsonl",
    "Target ligands": OUT / "validated_target_ligand_graphs.jsonl",
    "M3 interface": OUT / "member2_to_member3_interface.json",
}

for name, path in files_to_check.items():
    print(f"\n{'='*55}")
    print(name)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print("Size (MB):", round(path.stat().st_size / (1024**2), 2))

    if path.suffix == ".jsonl":
        with open(path, "r") as f:
            first_line = f.readline()

        if first_line:
            record = json.loads(first_line)
            print("First record keys:", list(record.keys()))
            print("Sample record:")
            print(json.dumps(record, indent=2)[:1800])

    elif path.suffix == ".json":
        with open(path, "r") as f:
            data = json.load(f)

        print("Top-level keys:", list(data.keys()))
        print("Interface preview:")
        print(json.dumps(data, indent=2)[:2500])


ChEMBL
Path: /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_graphs.jsonl
Exists: True
Size (MB): 135.83
First record keys: ['schema_version', 'source', 'chembl_id', 'inchikey', 'smiles', 'num_atoms', 'num_bonds', 'node_features', 'edge_index', 'edge_features', 'coordinates', 'has_3d_coordinates', 'molecular_weight', 'feature_schema']
Sample record:
{
  "schema_version": "2.0",
  "source": "ChEMBL_37",
  "chembl_id": "CHEMBL4741328",
  "inchikey": "OVPPPIMFQSVMOH-UHFFFAOYSA-N",
  "smiles": "Cn1c(=O)[nH]c(=O)c2c1nc(SCc1ccc(Cl)cc1)n2CCCc1ccccc1",
  "num_atoms": 30,
  "num_bonds": 33,
  "node_features": [
    [
      6,
      0,
      1,
      4,
      0,
      0,
      4,
      3
    ],
    [
      7,
      0,
      3,
      3,
      1,
      1,
      3,
      0
    ],
    [
      6,
      0,
      3,
      4,
      1,
      1,
      3,
      0
    ],
    [
      8,
      0,
      1,
      2,
      0,
      0,
      3,
      0
    ],
    [
      7,
      0,
     

In [62]:
# CELL 57 — CHEMBL TRAINING DATA QUALITY AUDIT

import json
import re
import numpy as np
from pathlib import Path
from rdkit import Chem

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"

CHEMBL_FILE = OUT / "chembl37_general_graphs.jsonl"
TARGET_FILE = OUT / "validated_target_ligand_graphs.jsonl"

valid_smiles = []
invalid_count = 0
duplicate_count = 0
seen = set()
lengths = []

with open(CHEMBL_FILE, "r") as f:

    for line in f:

        record = json.loads(line)
        smiles = record.get("smiles", "")

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            invalid_count += 1
            continue

        canonical = Chem.MolToSmiles(mol)

        if canonical in seen:
            duplicate_count += 1
            continue

        seen.add(canonical)
        valid_smiles.append(canonical)
        lengths.append(len(canonical))

# Inspect target-specific references
target_smiles = []

with open(TARGET_FILE, "r") as f:

    for line in f:
        record = json.loads(line)

        smiles = record.get("canonical_smiles", "")
        mol = Chem.MolFromSmiles(smiles)

        if mol is not None:
            target_smiles.append(Chem.MolToSmiles(mol))

print("=" * 55)
print("CHEMBL TRAINING DATA AUDIT")
print("=" * 55)

print("Valid unique molecules:", len(valid_smiles))
print("Invalid molecules:", invalid_count)
print("Duplicate molecules:", duplicate_count)

print("\nSMILES length statistics:")
print("Minimum:", min(lengths))
print("Maximum:", max(lengths))
print("Mean:", round(np.mean(lengths), 2))
print("Median:", np.median(lengths))

print("\nTarget-specific references:")
print("Valid target molecules:", len(target_smiles))

print("\nExample canonical SMILES:")
for s in valid_smiles[:5]:
    print(s)

# Save cleaned pretraining data
CLEAN_FILE = OUT / "chembl37_clean_smiles.txt"

with open(CLEAN_FILE, "w") as f:
    for smiles in valid_smiles:
        f.write(smiles + "\n")

print("\nCleaned training data saved:")
print(CLEAN_FILE)

CHEMBL TRAINING DATA AUDIT
Valid unique molecules: 50000
Invalid molecules: 0
Duplicate molecules: 0

SMILES length statistics:
Minimum: 8
Maximum: 238
Mean: 53.99
Median: 50.0

Target-specific references:
Valid target molecules: 10

Example canonical SMILES:
Cn1c(=O)[nH]c(=O)c2c1nc(SCc1ccc(Cl)cc1)n2CCCc1ccccc1
Cc1cn(C)nc1CNc1ncccc1C#N
N#CN1C[C@@H]2[C@H](C1)[C@@]2(C#N)c1ccc(-c2ccc(N3C[C@H](Cn4ccnn4)OC3=O)cc2F)cn1
CCCCN(C(=O)COC(=O)c1nc2nc(C)cc(C)n2n1)C1CCS(=O)(=O)C1
C=CCc1c(OC(C)=O)c2ccccc2n(C)c1=O

Cleaned training data saved:
/content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_clean_smiles.txt


In [63]:
# CELL 58 — SMILES TOKENIZATION AND DATASET PREPARATION

import json
import re
import random
import numpy as np
from pathlib import Path
from collections import Counter

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"

INPUT_FILE = OUT / "chembl37_clean_smiles.txt"

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# SMILES tokenizer
TOKEN_PATTERN = re.compile(
    r"(\[[^\]]+\]|Br|Cl|Si|Se|Na|Li|Mg|Ca|"
    r"@@?|%\d{2}|"
    r"[BCNOFPSI]|"
    r"[bcnops]|"
    r"\d|"
    r"\(|\)|\.|=|#|-|\+|\\|/|:|~|\*|\$)"
)

def tokenize_smiles(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)

    # Ensure every character is represented.
    if "".join(tokens) != smiles:
        raise ValueError(f"Tokenization mismatch: {smiles}")

    return tokens


# Load molecules
with open(INPUT_FILE, "r") as f:
    smiles_list = [line.strip() for line in f if line.strip()]

tokenized = [tokenize_smiles(s) for s in smiles_list]

# Verify round-trip
for smiles, tokens in zip(smiles_list, tokenized):
    assert "".join(tokens) == smiles

# Vocabulary
counter = Counter(
    token
    for sequence in tokenized
    for token in sequence
)

SPECIAL_TOKENS = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

vocab = SPECIAL_TOKENS + sorted(counter.keys())

token_to_id = {token: i for i, token in enumerate(vocab)}
id_to_token = {i: token for token, i in token_to_id.items()}

PAD = token_to_id["<PAD>"]
BOS = token_to_id["<BOS>"]
EOS = token_to_id["<EOS>"]

# Encode sequences
encoded = [
    [BOS] + [token_to_id[t] for t in sequence] + [EOS]
    for sequence in tokenized
]

MAX_LENGTH = max(len(sequence) for sequence in encoded)

# Shuffle and split
indices = np.arange(len(encoded))
np.random.shuffle(indices)

split = int(0.90 * len(indices))

train_indices = indices[:split]
val_indices = indices[split:]

# Padding
def pad_sequences(selected_indices):
    result = np.full(
        (len(selected_indices), MAX_LENGTH),
        PAD,
        dtype=np.int32
    )

    for row, idx in enumerate(selected_indices):
        sequence = encoded[idx]
        result[row, :len(sequence)] = sequence

    return result


train_data = pad_sequences(train_indices)
val_data = pad_sequences(val_indices)

# Save data
np.save(OUT / "chembl_train_tokens.npy", train_data)
np.save(OUT / "chembl_val_tokens.npy", val_data)

with open(OUT / "smiles_tokenizer.json", "w") as f:
    json.dump({
        "vocabulary": vocab,
        "token_to_id": token_to_id,
        "max_length": MAX_LENGTH,
        "special_tokens": SPECIAL_TOKENS
    }, f, indent=2)

# Summary
print("=" * 55)
print("SMILES TOKENIZATION SUMMARY")
print("=" * 55)

print("Total molecules:", len(encoded))
print("Training molecules:", len(train_data))
print("Validation molecules:", len(val_data))
print("Vocabulary size:", len(vocab))
print("Maximum sequence length:", MAX_LENGTH)
print("Unique chemical tokens:", len(counter))

print("\nMost frequent tokens:")
for token, count in counter.most_common(15):
    print(f"{token:8s}: {count}")

print("\nTraining array shape:", train_data.shape)
print("Validation array shape:", val_data.shape)

print("\nRound-trip validation: PASSED")
print("Training data saved.")


SMILES TOKENIZATION SUMMARY
Total molecules: 50000
Training molecules: 45000
Validation molecules: 5000
Vocabulary size: 81
Maximum sequence length: 143
Unique chemical tokens: 77

Most frequent tokens:
c       : 616105
C       : 419254
(       : 254326
)       : 254326
1       : 147518
O       : 147369
2       : 120116
=       : 96067
N       : 90436
n       : 71314
3       : 66074
F       : 25135
4       : 24226
-       : 19011
[C@H]   : 18554

Training array shape: (45000, 143)
Validation array shape: (5000, 143)

Round-trip validation: PASSED
Training data saved.


In [64]:
# CELL 59 — TRAIN GENERAL MOLECULAR GRU MODEL

import json
import torch
import torch.nn as nn
import numpy as np

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. PATHS AND DEVICE
# --------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"
CHECKPOINTS = ROOT / "checkpoints"

CHECKPOINTS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")

torch.manual_seed(42)
np.random.seed(42)

torch.set_num_threads(4)

print("Training device:", DEVICE)
print("PyTorch version:", torch.__version__)

# --------------------------------------------------
# 2. LOAD TOKENIZER AND DATA
# --------------------------------------------------

with open(OUT / "smiles_tokenizer.json") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
vocab_size = len(tokenizer["vocabulary"])

PAD_ID = token_to_id["<PAD>"]

train_array = np.load(OUT / "chembl_train_tokens.npy")
val_array = np.load(OUT / "chembl_val_tokens.npy")

print("Vocabulary size:", vocab_size)
print("Training shape:", train_array.shape)
print("Validation shape:", val_array.shape)

# --------------------------------------------------
# 3. DATASET
# --------------------------------------------------

class MolecularDataset(Dataset):

    def __init__(self, sequences):
        self.sequences = torch.tensor(
            sequences,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        sequence = self.sequences[index]

        inputs = sequence[:-1]
        targets = sequence[1:]

        return inputs, targets


train_dataset = MolecularDataset(train_array)
val_dataset = MolecularDataset(val_array)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

# --------------------------------------------------
# 4. GRU MODEL
# --------------------------------------------------

class MolecularGRU(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=256,
        num_layers=2,
        padding_idx=0
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.output = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):

        embedded = self.embedding(x)

        output, _ = self.gru(embedded)

        logits = self.output(output)

        return logits


model = MolecularGRU(
    vocab_size=vocab_size,
    padding_idx=PAD_ID
).to(DEVICE)

print("\nModel parameters:",
      sum(p.numel() for p in model.parameters()))

# --------------------------------------------------
# 5. LOSS AND OPTIMIZER
# --------------------------------------------------

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

# --------------------------------------------------
# 6. TRAINING AND VALIDATION
# --------------------------------------------------

EPOCHS = 5
PATIENCE = 2

best_val_loss = float("inf")
epochs_without_improvement = 0

history = []

BEST_MODEL_PATH = CHECKPOINTS / "molecular_gru_best.pt"

def run_epoch(loader, training):

    model.train() if training else model.eval()

    total_loss = 0.0
    total_tokens = 0

    for inputs, targets in loader:

        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):

            logits = model(inputs)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()

        valid_tokens = (targets != PAD_ID).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


print("\nStarting training...\n")

for epoch in range(EPOCHS):

    train_loss = run_epoch(train_loader, True)
    val_loss = run_epoch(val_loader, False)

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "learning_rate": current_lr
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "vocab_size": vocab_size,
            "embedding_dim": 128,
            "hidden_dim": 256,
            "num_layers": 2,
            "tokenizer_path": str(OUT / "smiles_tokenizer.json"),
            "training_epoch": epoch + 1,
            "validation_loss": val_loss
        }, BEST_MODEL_PATH)

        print("  Best model checkpoint saved.")

    else:

        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")
        break

# --------------------------------------------------
# 7. SAVE TRAINING HISTORY
# --------------------------------------------------

with open(OUT / "molecular_gru_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\n" + "=" * 55)
print("TRAINING SUMMARY")
print("=" * 55)

print("Completed epochs:", len(history))
print("Best validation loss:", round(best_val_loss, 4))
print("Best checkpoint:", BEST_MODEL_PATH)
print("Training history saved.")

Training device: cpu
PyTorch version: 2.14.1+cpu
Vocabulary size: 81
Training shape: (45000, 143)
Validation shape: (5000, 143)

Model parameters: 722385

Starting training...

Epoch 1/5 | Train Loss: 1.3495 | Validation Loss: 0.9936 | LR: 0.001000
  Best model checkpoint saved.
Epoch 2/5 | Train Loss: 0.9463 | Validation Loss: 0.8825 | LR: 0.001000
  Best model checkpoint saved.
Epoch 3/5 | Train Loss: 0.8735 | Validation Loss: 0.8387 | LR: 0.001000
  Best model checkpoint saved.
Epoch 4/5 | Train Loss: 0.8378 | Validation Loss: 0.8146 | LR: 0.001000
  Best model checkpoint saved.
Epoch 5/5 | Train Loss: 0.8147 | Validation Loss: 0.7984 | LR: 0.001000
  Best model checkpoint saved.

TRAINING SUMMARY
Completed epochs: 5
Best validation loss: 0.7984
Best checkpoint: /content/drive/MyDrive/Member2_Project/checkpoints/molecular_gru_best.pt
Training history saved.


In [65]:
# CELL 60 — Inspect saved GRU checkpoint and tokenizer

from pathlib import Path
import torch
import json

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")

CHECKPOINT_PATH = PROJECT_DIR / "checkpoints" / "molecular_gru_best.pt"
TOKENIZER_PATH = PROJECT_DIR / "member2_outputs" / "smiles_tokenizer.json"
HISTORY_PATH = PROJECT_DIR / "member2_outputs" / "molecular_gru_training_history.json"

# 1. Verify saved files
for path in [CHECKPOINT_PATH, TOKENIZER_PATH, HISTORY_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

assert CHECKPOINT_PATH.exists(), "GRU checkpoint not found!"
assert TOKENIZER_PATH.exists(), "Tokenizer file not found!"

# 2. Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\n========== CHECKPOINT DETAILS ==========")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

# 3. Inspect tokenizer
with open(TOKENIZER_PATH, "r") as f:
    tokenizer_data = json.load(f)

print("\n========== TOKENIZER DETAILS ==========")
print("Tokenizer keys:", list(tokenizer_data.keys()))

# 4. Inspect training history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r") as f:
        history = json.load(f)

    print("\n========== TRAINING HISTORY ==========")
    print(json.dumps(history, indent=2)[:2000])

# 5. Check whether model variables remain in memory
print("\n========== EXISTING MODEL VARIABLES ==========")

for name, value in globals().copy().items():
    if isinstance(value, torch.nn.Module):
        print("Model variable:", name)
        print("Model class:", type(value).__name__)

molecular_gru_best.pt: FOUND
smiles_tokenizer.json: FOUND
molecular_gru_training_history.json: FOUND

========== CHECKPOINT DETAILS ==========
Checkpoint type: <class 'dict'>
Checkpoint keys: ['model_state_dict', 'vocab_size', 'embedding_dim', 'hidden_dim', 'num_layers', 'tokenizer_path', 'training_epoch', 'validation_loss']

========== TOKENIZER DETAILS ==========
Tokenizer keys: ['vocabulary', 'token_to_id', 'max_length', 'special_tokens']

========== TRAINING HISTORY ==========
[
  {
    "epoch": 1,
    "train_loss": 1.3494757471257388,
    "validation_loss": 0.993610628127944,
    "learning_rate": 0.001
  },
  {
    "epoch": 2,
    "train_loss": 0.9462737086953288,
    "validation_loss": 0.8824965417638615,
    "learning_rate": 0.001
  },
  {
    "epoch": 3,
    "train_loss": 0.8735175589789296,
    "validation_loss": 0.8386693126803919,
    "learning_rate": 0.001
  },
  {
    "epoch": 4,
    "train_loss": 0.8378062427114609,
    "validation_loss": 0.8145721725706908,
    "learning

In [67]:
# CELL 61 — Inspect trained GRU without accessing source code

print("========== MODEL ARCHITECTURE ==========")

print(model)

print("\n========== MODEL CONFIGURATION ==========")

print("Embedding dimension:", checkpoint["embedding_dim"])
print("Hidden dimension:", checkpoint["hidden_dim"])
print("GRU layers:", checkpoint["num_layers"])
print("Vocabulary size:", checkpoint["vocab_size"])
print("Training epoch:", checkpoint["training_epoch"])
print("Validation loss:", checkpoint["validation_loss"])

print("\n========== MODEL FORWARD SIGNATURE ==========")

import inspect

print(inspect.signature(model.forward))

print("\n========== TOKENIZER ==========")

print("Vocabulary size:", len(tokenizer_data["vocabulary"]))
print("Special tokens:", tokenizer_data["special_tokens"])
print("Maximum sequence length:", tokenizer_data["max_length"])

print("\nFirst 30 vocabulary tokens:")
print(tokenizer_data["vocabulary"][:30])

========== MODEL ARCHITECTURE ==========
MolecularGRU(
  (embedding): Embedding(81, 128, padding_idx=0)
  (gru): GRU(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (output): Linear(in_features=256, out_features=81, bias=True)
)

========== MODEL CONFIGURATION ==========
Embedding dimension: 128
Hidden dimension: 256
GRU layers: 2
Vocabulary size: 81
Training epoch: 5
Validation loss: 0.798440972976806

========== MODEL FORWARD SIGNATURE ==========
(x)

========== TOKENIZER ==========
Vocabulary size: 81
Special tokens: ['<PAD>', '<BOS>', '<EOS>', '<UNK>']
Maximum sequence length: 143

First 30 vocabulary tokens:
['<PAD>', '<BOS>', '<EOS>', '<UNK>', '#', '%10', '(', ')', '-', '/', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', 'B', 'Br', 'C', 'Cl', 'F', 'I', 'N', 'O', 'P', 'S']


In [68]:
# CELL 62 — Verify GRU forward-pass output

import torch

model.eval()

bos_id = tokenizer_data["token_to_id"]["<BOS>"]

test_input = torch.tensor([[bos_id]], dtype=torch.long)

with torch.no_grad():
    test_output = model(test_input)

print("Input shape:", test_input.shape)
print("Output type:", type(test_output))

if isinstance(test_output, tuple):
    print("Output tuple length:", len(test_output))

    for i, item in enumerate(test_output):
        if torch.is_tensor(item):
            print(f"Output {i} shape:", item.shape)
        else:
            print(f"Output {i} type:", type(item))

elif torch.is_tensor(test_output):
    print("Output shape:", test_output.shape)
    print("Output dtype:", test_output.dtype)

print("\nModel device:", next(model.parameters()).device)

Input shape: torch.Size([1, 1])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([1, 1, 81])
Output dtype: torch.float32

Model device: cpu


In [69]:
# CELL 63 — Generate molecular candidates using trained GRU

import torch
import torch.nn.functional as F
import numpy as np
import random
import json
import os

model.eval()

# Tokenizer mappings
token_to_id = tokenizer_data["token_to_id"]
id_to_token = {
    idx: token for token, idx in token_to_id.items()
}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]

MAX_LENGTH = tokenizer_data["max_length"]

# Generation settings
NUM_CANDIDATES = 100
TEMPERATURE = 0.8
TOP_K = 10

generated_smiles = []

def generate_one_smiles():

    sequence = [BOS_ID]

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            input_tensor = torch.tensor(
                [sequence],
                dtype=torch.long,
                device=next(model.parameters()).device
            )

            logits = model(input_tensor)

            # Predict next token
            next_logits = logits[0, -1, :].clone()

            # Prevent special tokens from appearing as ordinary tokens
            next_logits[PAD_ID] = -float("inf")
            next_logits[BOS_ID] = -float("inf")
            next_logits[token_to_id["<UNK>"]] = -float("inf")

            # Temperature sampling
            next_logits = next_logits / TEMPERATURE

            # Top-k sampling
            values, indices = torch.topk(
                next_logits,
                min(TOP_K, next_logits.numel())
            )

            probabilities = F.softmax(values, dim=-1)

            selected = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            next_token_id = indices[selected].item()

            if next_token_id == EOS_ID:
                break

            sequence.append(next_token_id)

    # Convert token IDs into SMILES
    tokens = [
        id_to_token[token_id]
        for token_id in sequence[1:]
    ]

    return "".join(tokens)


# Generate candidates
for i in range(NUM_CANDIDATES):

    smiles = generate_one_smiles()

    generated_smiles.append(smiles)

    if (i + 1) % 20 == 0:
        print(f"Generated {i + 1}/{NUM_CANDIDATES}")


# Save raw generated sequences
output_dir = os.path.join(
    PROJECT_ROOT,
    "member2_outputs"
)

os.makedirs(output_dir, exist_ok=True)

raw_path = os.path.join(
    output_dir,
    "gru_generated_raw_smiles.json"
)

with open(raw_path, "w") as f:
    json.dump(generated_smiles, f, indent=2)

print("\n========== GENERATION COMPLETE ==========")
print("Total generated:", len(generated_smiles))
print("Saved:", raw_path)

print("\nFirst 10 generated sequences:")

for i, smiles in enumerate(generated_smiles[:10], 1):
    print(i, smiles)

Generated 20/100
Generated 40/100
Generated 60/100
Generated 80/100
Generated 100/100

========== GENERATION COMPLETE ==========
Total generated: 100
Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/gru_generated_raw_smiles.json

First 10 generated sequences:
1 COc1ccc(OC)c(Sc2ncc(C(F)(F)F)cc2C#N)c1
2 CCOC(=O)C(C)(C)Cc1ccc(C(=O)Nc2ccccc2C(C)(C)C)cc1
3 O=C(c1cccc(Oc2cccc(C(=O)N3CC4CC2)c2)c1)C[C@H]3c1ccccc1-2
4 CC(C)(C)OC[C@H]1COCCN1/C=C/c1ccccc1OC(=O)C(C)C
5 O=C1CCC(C(=O)c2ccccc2)C1=O
6 CC(C)C(=O)N[C@H]1CCC[C@@H]1NC(=O)c1cccc(N2CCCC2)c1
7 COc1cc2c(nc(NC(=O)c3ccco3)c3cnc(N4CCCCC4)cc34)n2n1
8 CCOC(=O)c1c(Nc2cccc(C#N)c2)nc2c(=O)[nH]c(C)ccc12
9 Cc1cccc(C)c1CNC(=O)c1cccc(Nc2nc(Nc3ccc4c(nc3C)CCCCC4)ncn2)c1
10 Cc1nn(C)c(C(=O)N2CCN(c3ccc(C#N)cc3)CC2)n1


In [70]:
# CELL 64 — Validate generated molecules using RDKit

import os
import json
import pandas as pd

from rdkit import Chem

# Paths
output_dir = os.path.join(PROJECT_ROOT, "member2_outputs")

raw_path = os.path.join(
    output_dir,
    "gru_generated_raw_smiles.json"
)

training_path = os.path.join(
    output_dir,
    "chembl37_clean_smiles.txt"
)

# Load generated SMILES
with open(raw_path, "r") as f:
    generated_smiles = json.load(f)

# Load training SMILES for novelty comparison
with open(training_path, "r") as f:
    training_smiles = [line.strip() for line in f if line.strip()]

# Canonicalize training molecules
training_canonical = set()

for smiles in training_smiles:
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        canonical = Chem.MolToSmiles(mol, canonical=True)
        training_canonical.add(canonical)

# Validate generated molecules
results = []

for index, smiles in enumerate(generated_smiles, start=1):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        results.append({
            "candidate_id": f"GRU_{index:03d}",
            "generated_smiles": smiles,
            "canonical_smiles": None,
            "valid": False,
            "unique": False,
            "novel_vs_training": False,
            "heavy_atoms": None
        })

    else:
        canonical = Chem.MolToSmiles(
            mol,
            canonical=True
        )

        results.append({
            "candidate_id": f"GRU_{index:03d}",
            "generated_smiles": smiles,
            "canonical_smiles": canonical,
            "valid": True,
            "unique": False,
            "novel_vs_training": canonical not in training_canonical,
            "heavy_atoms": mol.GetNumHeavyAtoms()
        })

# Build DataFrame
validation_df = pd.DataFrame(results)

# Identify unique valid molecules
valid_mask = validation_df["valid"]

validation_df.loc[valid_mask, "unique"] = (
    validation_df.loc[valid_mask, "canonical_smiles"]
    .duplicated(keep="first")
    .eq(False)
)

# Statistics
total = len(validation_df)
valid_count = int(validation_df["valid"].sum())
invalid_count = total - valid_count

unique_count = int(
    validation_df.loc[valid_mask, "canonical_smiles"].nunique()
)

novel_count = int(
    validation_df.loc[valid_mask, "novel_vs_training"].sum()
)

# Save results
validation_path = os.path.join(
    output_dir,
    "gru_generation_validation.csv"
)

validation_df.to_csv(validation_path, index=False)

print("========== CHEMICAL VALIDATION ==========")

print("Total generated:", total)
print("RDKit valid:", valid_count)
print("Invalid:", invalid_count)
print("Validity percentage:", round(valid_count / total * 100, 2), "%")

print("\n========== UNIQUENESS ==========")

print("Unique valid molecules:", unique_count)

if valid_count:
    print(
        "Uniqueness percentage:",
        round(unique_count / valid_count * 100, 2),
        "%"
    )

print("\n========== NOVELTY ==========")

print("Novel molecules vs ChEMBL training set:", novel_count)

if valid_count:
    print(
        "Novelty percentage:",
        round(novel_count / valid_count * 100, 2),
        "%"
    )

print("\nValidation report saved:")
print(validation_path)

print("\nFirst 10 validation results:")

print(
    validation_df[
        ["candidate_id", "valid", "unique",
         "novel_vs_training", "heavy_atoms"]
    ].head(10).to_string(index=False)
)

========== CHEMICAL VALIDATION ==========
Total generated: 100
RDKit valid: 73
Invalid: 27
Validity percentage: 73.0 %

========== UNIQUENESS ==========
Unique valid molecules: 73
Uniqueness percentage: 100.0 %

========== NOVELTY ==========
Novel molecules vs ChEMBL training set: 73
Novelty percentage: 100.0 %

Validation report saved:
/content/drive/MyDrive/Member2_Project/member2_outputs/gru_generation_validation.csv

First 10 validation results:
candidate_id  valid  unique  novel_vs_training  heavy_atoms
     GRU_001   True    True               True         23.0
     GRU_002   True    True               True         28.0
     GRU_003  False   False              False          NaN
     GRU_004   True    True               True         26.0
     GRU_005   True    True               True         15.0
     GRU_006   True    True               True         25.0
     GRU_007  False   False              False          NaN
     GRU_008   True    True               True         26.0
     G

In [71]:
# CELL 65 — Molecular diversity evaluation

import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

# Extract valid canonical molecules
valid_smiles = validation_df.loc[
    validation_df["valid"],
    "canonical_smiles"
].tolist()

molecules = [
    Chem.MolFromSmiles(smiles)
    for smiles in valid_smiles
]

# Generate Morgan fingerprints
fingerprints = [
    AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )
    for mol in molecules
]

# Calculate pairwise Tanimoto similarities
similarities = []

for i in range(len(fingerprints)):
    for j in range(i + 1, len(fingerprints)):

        similarity = DataStructs.TanimotoSimilarity(
            fingerprints[i],
            fingerprints[j]
        )

        similarities.append(similarity)

similarities = np.array(similarities)

# Calculate diversity statistics
mean_similarity = float(np.mean(similarities))
median_similarity = float(np.median(similarities))
maximum_similarity = float(np.max(similarities))
minimum_similarity = float(np.min(similarities))

mean_diversity = 1 - mean_similarity

print("========== MOLECULAR DIVERSITY ==========")

print("Valid molecules:", len(molecules))
print("Pairwise comparisons:", len(similarities))

print("Mean Tanimoto similarity:", round(mean_similarity, 4))
print("Median Tanimoto similarity:", round(median_similarity, 4))
print("Maximum similarity:", round(maximum_similarity, 4))
print("Minimum similarity:", round(minimum_similarity, 4))

print("Mean pairwise diversity:", round(mean_diversity, 4))

========== MOLECULAR DIVERSITY ==========
Valid molecules: 73
Pairwise comparisons: 2628
Mean Tanimoto similarity: 0.1269
Median Tanimoto similarity: 0.1229
Maximum similarity: 0.3721
Minimum similarity: 0.0
Mean pairwise diversity: 0.8731


In [72]:
# CELL 66 — Drug-likeness and chemical property evaluation

import os
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, Crippen, QED, rdMolDescriptors
from rdkit.Chem import FilterCatalog

# Use the 73 valid molecules from Cell 64
valid_df = validation_df[validation_df["valid"]].copy()

# Configure PAINS filters
params = FilterCatalog.FilterCatalogParams()

params.AddCatalog(
    FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS
)

pains_catalog = FilterCatalog.FilterCatalog(params)

records = []

for _, row in valid_df.iterrows():

    smiles = row["canonical_smiles"]
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        continue

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    rotatable_bonds = Lipinski.NumRotatableBonds(mol)
    qed = QED.qed(mol)

    # Lipinski Rule of Five
    violations = sum([
        mw > 500,
        logp > 5,
        hbd > 5,
        hba > 10
    ])

    # PAINS screening
    pains_matches = pains_catalog.GetMatches(mol)
    pains_alert = len(pains_matches) > 0

    records.append({
        "candidate_id": row["candidate_id"],
        "canonical_smiles": smiles,
        "molecular_weight": mw,
        "logP": logp,
        "HBD": hbd,
        "HBA": hba,
        "TPSA": tpsa,
        "rotatable_bonds": rotatable_bonds,
        "QED": qed,
        "lipinski_violations": violations,
        "lipinski_pass": violations <= 1,
        "PAINS_alert": pains_alert
    })


# Create report
druglikeness_df = pd.DataFrame(records)

# Save report
output_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "gru_druglikeness_report.csv"
)

druglikeness_df.to_csv(output_path, index=False)


# Summary
print("========== DRUG-LIKENESS EVALUATION ==========")

print("Molecules evaluated:", len(druglikeness_df))

print("\n========== PROPERTY SUMMARY ==========")

property_columns = [
    "molecular_weight",
    "logP",
    "HBD",
    "HBA",
    "TPSA",
    "rotatable_bonds",
    "QED"
]

print(
    druglikeness_df[property_columns]
    .describe()
    .round(3)
    .to_string()
)

print("\n========== FILTER RESULTS ==========")

print(
    "Lipinski pass (0–1 violations):",
    int(druglikeness_df["lipinski_pass"].sum())
)

print(
    "Lipinski violations > 1:",
    int((~druglikeness_df["lipinski_pass"]).sum())
)

print(
    "PAINS alerts:",
    int(druglikeness_df["PAINS_alert"].sum())
)

print(
    "No PAINS alerts:",
    int((~druglikeness_df["PAINS_alert"]).sum())
)

print("\nSaved report:")
print(output_path)

print("\n========== FIRST 10 CANDIDATES ==========")

print(
    druglikeness_df[
        [
            "candidate_id",
            "molecular_weight",
            "logP",
            "QED",
            "lipinski_violations",
            "PAINS_alert"
        ]
    ].head(10).round(3).to_string(index=False)
)

========== DRUG-LIKENESS EVALUATION ==========
Molecules evaluated: 73

========== PROPERTY SUMMARY ==========
       molecular_weight    logP     HBD     HBA     TPSA  rotatable_bonds     QED
count            73.000  73.000  73.000  73.000   73.000           73.000  73.000
mean            365.546   4.195   0.877   3.548   55.348            6.521   0.611
std             121.003   2.369   0.912   1.659   24.917            5.923   0.214
min             134.178   0.958   0.000   0.000    6.480            0.000   0.033
25%             286.371   2.683   0.000   2.000   38.330            3.000   0.482
50%             352.390   3.683   1.000   3.000   55.140            5.000   0.658
75%             412.917   5.046   2.000   5.000   66.480            8.000   0.772
max             828.128  13.644   3.000   8.000  131.110           34.000   0.916

========== FILTER RESULTS ==========
Lipinski pass (0–1 violations): 68
Lipinski violations > 1: 5
PAINS alerts: 8
No PAINS alerts: 65

Saved report:


In [73]:
# CELL 67 — Preliminary molecular candidate screening

screened_df = druglikeness_df.copy()

screened_df["basic_property_pass"] = (
    (screened_df["molecular_weight"] <= 500) &
    (screened_df["logP"] <= 5) &
    (screened_df["lipinski_violations"] <= 1) &
    (screened_df["rotatable_bonds"] <= 10)
)

screened_df["priority_candidate"] = (
    screened_df["basic_property_pass"] &
    (screened_df["QED"] >= 0.5) &
    (~screened_df["PAINS_alert"])
)

priority_df = screened_df[
    screened_df["priority_candidate"]
].copy()

priority_df = priority_df.sort_values(
    by="QED",
    ascending=False
)

# Save results
priority_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "gru_priority_candidates.csv"
)

priority_df.to_csv(priority_path, index=False)

print("========== CANDIDATE SCREENING ==========")

print("Total valid molecules:", len(screened_df))

print(
    "Basic property pass:",
    int(screened_df["basic_property_pass"].sum())
)

print(
    "Priority candidates:",
    len(priority_df)
)

print("\n========== TOP CANDIDATES ==========")

print(
    priority_df[
        [
            "candidate_id",
            "molecular_weight",
            "logP",
            "TPSA",
            "QED",
            "rotatable_bonds"
        ]
    ].head(15).round(3).to_string(index=False)
)

print("\nSaved:", priority_path)

========== CANDIDATE SCREENING ==========
Total valid molecules: 73
Basic property pass: 49
Priority candidates: 45

========== TOP CANDIDATES ==========
candidate_id  molecular_weight  logP  TPSA   QED  rotatable_bonds
     GRU_046           255.321 2.683 47.04 0.916                3
     GRU_093           259.736 3.831 29.10 0.892                3
     GRU_079           371.403 2.991 52.65 0.866                5
     GRU_047           318.402 2.327 69.90 0.865                3
     GRU_006           343.471 2.710 61.44 0.864                5
     GRU_055           331.294 1.989 75.44 0.860                3
     GRU_059           242.274 3.255 46.53 0.840                3
     GRU_001           340.326 4.140 55.14 0.837                4
     GRU_058           286.371 4.407 17.07 0.831                3
     GRU_010           310.361 0.958 78.05 0.825                2
     GRU_011           404.432 4.325 49.41 0.825                4
     GRU_098           272.348 1.689 40.62 0.822      

In [74]:
# CELL 68 — Compare molecular features with M1 pharmacophore

import os
import json
import glob
import pandas as pd

from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig

# --------------------------------------------------
# 1. Locate current M1 metadata
# --------------------------------------------------

m1_root = os.path.join(PROJECT_ROOT, "member1_input")

metadata_files = glob.glob(
    os.path.join(m1_root, "**", "pharmacophore_tensors.json"),
    recursive=True
)

if not metadata_files:
    raise FileNotFoundError(
        "Could not find pharmacophore_tensors.json in M1 input."
    )

if len(metadata_files) > 1:
    print("Multiple M1 metadata files found:")
    for path in metadata_files:
        print(path)

    raise RuntimeError(
        "Please identify the correct current M1 metadata file."
    )

m1_metadata_path = metadata_files[0]

with open(m1_metadata_path, "r") as f:
    current_m1_metadata = json.load(f)

print("M1 metadata:", m1_metadata_path)

# --------------------------------------------------
# 2. Discover pocket and feature metadata
# --------------------------------------------------

pocket_records = []

for pocket_key, pocket_data in current_m1_metadata.items():

    if not isinstance(pocket_data, dict):
        continue

    feature_metadata = pocket_data.get("feature_metadata")

    if isinstance(feature_metadata, list) and feature_metadata:

        pocket_records.append({
            "pocket_key": pocket_key,
            "pocket_data": pocket_data,
            "features": feature_metadata
        })

if not pocket_records:
    raise ValueError(
        "No pocket containing feature_metadata was found."
    )

if len(pocket_records) > 1:
    print("Multiple pockets discovered:")
    for record in pocket_records:
        print(record["pocket_key"])

    raise RuntimeError(
        "Select the intended pocket before continuing."
    )

selected_pocket = pocket_records[0]

m1_features = selected_pocket["features"]

print("\n========== M1 POCKET ==========")
print("Pocket key:", selected_pocket["pocket_key"])
print("Number of feature records:", len(m1_features))

# --------------------------------------------------
# 3. Inspect actual M1 feature types
# --------------------------------------------------

m1_feature_types = []

for feature in m1_features:

    feature_name = str(
        feature.get("feature", "UNKNOWN")
    ).upper()

    m1_feature_types.append(feature_name)

m1_feature_counts = (
    pd.Series(m1_feature_types)
    .value_counts()
    .to_dict()
)

print("\n========== M1 PHARMACOPHORE FEATURES ==========")

for feature_type, count in m1_feature_counts.items():
    print(f"{feature_type}: {count}")

# --------------------------------------------------
# 4. Configure RDKit chemical feature detection
# --------------------------------------------------

feature_file = os.path.join(
    RDConfig.RDDataDir,
    "BaseFeatures.fdef"
)

feature_factory = ChemicalFeatures.BuildFeatureFactory(
    feature_file
)

# Map RDKit feature families to M1 feature categories
feature_mapping = {
    "Hydrophobe": "HYDROPHOBIC",
    "Aromatic": "AROMATIC",
    "PosIonizable": "POSITIVE",
    "Donor": "DONOR",
    "Acceptor": "ACCEPTOR"
}

# --------------------------------------------------
# 5. Load priority molecules
# --------------------------------------------------

candidate_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "gru_priority_candidates.csv"
)

priority_candidates = pd.read_csv(candidate_path)

comparison_records = []

for _, row in priority_candidates.iterrows():

    mol = Chem.MolFromSmiles(
        row["canonical_smiles"]
    )

    if mol is None:
        continue

    detected_features = feature_factory.GetFeaturesForMol(mol)

    candidate_counts = {
        category: 0
        for category in feature_mapping.values()
    }

    for feature in detected_features:

        family = feature.GetFamily()

        category = feature_mapping.get(family)

        if category:
            candidate_counts[category] += 1

    record = {
        "candidate_id": row["candidate_id"],
        "QED": row["QED"]
    }

    for category, count in candidate_counts.items():

        record[f"candidate_{category}"] = count

        record[f"M1_{category}_reference"] = (
            m1_feature_counts.get(category, 0)
        )

    comparison_records.append(record)

comparison_df = pd.DataFrame(comparison_records)

# --------------------------------------------------
# 6. Save comparison report
# --------------------------------------------------

comparison_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "m1_candidate_feature_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("\n========== CANDIDATE FEATURE COMPARISON ==========")

print(
    comparison_df.head(15).to_string(index=False)
)

print("\nCandidates evaluated:", len(comparison_df))

print("\nSaved:", comparison_path)

M1 metadata: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pharmacophore_tensors.json

========== M1 POCKET ==========
Pocket key: 1
Number of feature records: 95

========== M1 PHARMACOPHORE FEATURES ==========
HYDROPHOBIC: 45
DONOR: 22
ACCEPTOR: 18
POSITIVE: 5
AROMATIC: 4
NEGATIVE: 1

========== CANDIDATE FEATURE COMPARISON ==========
candidate_id      QED  candidate_HYDROPHOBIC  M1_HYDROPHOBIC_reference  candidate_AROMATIC  M1_AROMATIC_reference  candidate_POSITIVE  M1_POSITIVE_reference  candidate_DONOR  M1_DONOR_reference  candidate_ACCEPTOR  M1_ACCEPTOR_reference
     GRU_046 0.915644                      3                        45                   2                      4                   0                      5                1                  22                   3                     18
     GRU_093 0.891784                      3                        45                   2                      4                   0                      5    

In [75]:
# CELL 69 — Inspect actual M1 feature representation

import os
import glob
import numpy as np

pocket_data = selected_pocket["pocket_data"]

print("========== M1 POCKET METADATA KEYS ==========")

for key, value in pocket_data.items():

    if isinstance(value, (list, dict)):
        print(key, ":", type(value).__name__, "length =", len(value))
    else:
        print(key, ":", value)

print("\n========== FEATURE METADATA EXAMPLES ==========")

for feature in m1_features[:5]:
    print(feature)

print("\n========== FEATURE TENSOR DISCOVERY ==========")

tensor_files = glob.glob(
    os.path.join(
        PROJECT_ROOT,
        "member1_input",
        "**",
        "*.npy"
    ),
    recursive=True
)

for path in tensor_files:

    array = np.load(path, mmap_mode="r")

    print(
        os.path.basename(path),
        "| Shape:", array.shape,
        "| Dtype:", array.dtype
    )

    if array.ndim == 3 and array.shape[-1] == 4:

        print("\nFirst five tensor rows:")
        print(np.asarray(array[0, :5, :]))

        print("\nFeature value ranges:")

        for column in range(array.shape[-1]):
            print(
                f"Column {column}:",
                float(np.min(array[:, :, column])),
                "to",
                float(np.max(array[:, :, column]))
            )

        break

========== M1 POCKET METADATA KEYS ==========
pocket_id : 1
num_conformations : 5
num_features : 95
tensor_shape : list length = 3
mask_shape : list length = 2
feature_tensor : list length = 5
mask : list length = 5
feature_metadata : list length = 95

========== FEATURE METADATA EXAMPLES ==========
{'feature_index': 0, 'identity': 'A|826||HIS|CB', 'feature': 'HYDROPHOBIC', 'feature_id': 2}
{'feature_index': 1, 'identity': 'A|826||HIS|CD2;A|826||HIS|CE1;A|826||HIS|CG;A|826||HIS|ND1;A|826||HIS|NE2', 'feature': 'AROMATIC', 'feature_id': 3}
{'feature_index': 2, 'identity': 'A|826||HIS|CD2;A|826||HIS|CE1;A|826||HIS|CG;A|826||HIS|ND1;A|826||HIS|NE2', 'feature': 'POSITIVE', 'feature_id': 4}
{'feature_index': 3, 'identity': 'A|826||HIS|N', 'feature': 'DONOR', 'feature_id': 0}
{'feature_index': 4, 'identity': 'A|826||HIS|ND1', 'feature': 'DONOR', 'feature_id': 0}

========== FEATURE TENSOR DISCOVERY ==========
pocket_1_mask(1).npy | Shape: (5, 95) | Dtype: float32
pocket_1_pharmacophore(1).npy

In [76]:
# CELL 70 — Validate M1 feature IDs and 3D coordinates

import os
import glob
import numpy as np
import pandas as pd

# Locate the pharmacophore tensor
tensor_paths = glob.glob(
    os.path.join(
        PROJECT_ROOT,
        "member1_input",
        "**",
        "*pharmacophore*.npy"
    ),
    recursive=True
)

if len(tensor_paths) != 1:
    raise RuntimeError(
        f"Expected one pharmacophore tensor, found {len(tensor_paths)}"
    )

m1_tensor_path = tensor_paths[0]

m1_tensor = np.load(m1_tensor_path)

print("Tensor path:", m1_tensor_path)
print("Tensor shape:", m1_tensor.shape)

# --------------------------------------------------
# 1. Build feature ID mapping from metadata
# --------------------------------------------------

feature_id_mapping = {}

for feature in m1_features:

    feature_id = int(feature["feature_id"])
    feature_name = feature["feature"].upper()

    feature_id_mapping.setdefault(
        feature_id, set()
    ).add(feature_name)

print("\n========== FEATURE ID MAPPING ==========")

for feature_id, names in sorted(feature_id_mapping.items()):

    print(
        f"ID {feature_id}:",
        ", ".join(sorted(names))
    )

# --------------------------------------------------
# 2. Verify tensor codes against metadata
# --------------------------------------------------

metadata_ids = {
    int(feature["feature_index"]): int(feature["feature_id"])
    for feature in m1_features
}

errors = []

for conformer_idx in range(m1_tensor.shape[0]):

    for feature_idx in range(m1_tensor.shape[1]):

        tensor_id = int(round(
            float(m1_tensor[conformer_idx, feature_idx, 3])
        ))

        expected_id = metadata_ids[feature_idx]

        if tensor_id != expected_id:

            errors.append({
                "conformer": conformer_idx + 1,
                "feature_index": feature_idx,
                "tensor_id": tensor_id,
                "metadata_id": expected_id
            })

print("\n========== CONSISTENCY CHECK ==========")

print("Metadata feature records:", len(m1_features))
print("Conformers:", m1_tensor.shape[0])
print("Features per conformer:", m1_tensor.shape[1])
print("Feature ID mismatches:", len(errors))

# --------------------------------------------------
# 3. Inspect coordinate ranges per conformer
# --------------------------------------------------

print("\n========== CONFORMER COORDINATES ==========")

for conformer_idx in range(m1_tensor.shape[0]):

    coordinates = m1_tensor[conformer_idx, :, :3]

    print(
        f"Conformer {conformer_idx + 1}:",
        "X:", round(float(coordinates[:, 0].min()), 3),
        "to", round(float(coordinates[:, 0].max()), 3),
        "| Y:", round(float(coordinates[:, 1].min()), 3),
        "to", round(float(coordinates[:, 1].max()), 3),
        "| Z:", round(float(coordinates[:, 2].min()), 3),
        "to", round(float(coordinates[:, 2].max()), 3)
    )

if errors:
    print("\nFirst mismatches:")
    print(pd.DataFrame(errors).head().to_string(index=False))
    raise ValueError("M1 feature encoding validation failed.")

print("\nM1 feature encoding validated successfully.")

Tensor path: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pocket_1_pharmacophore(1).npy
Tensor shape: (5, 95, 4)

========== FEATURE ID MAPPING ==========
ID 0: DONOR
ID 1: ACCEPTOR
ID 2: HYDROPHOBIC
ID 3: AROMATIC
ID 4: POSITIVE
ID 5: NEGATIVE

========== CONSISTENCY CHECK ==========
Metadata feature records: 95
Conformers: 5
Features per conformer: 95
Feature ID mismatches: 0

========== CONFORMER COORDINATES ==========
Conformer 1: X: 12.007 to 30.88 | Y: 32.892 to 48.172 | Z: -5.201 to 15.024
Conformer 2: X: 11.561 to 31.002 | Y: 32.844 to 48.111 | Z: -5.483 to 14.849
Conformer 3: X: 12.205 to 30.946 | Y: 32.666 to 48.016 | Z: -5.617 to 14.77
Conformer 4: X: 12.651 to 30.824 | Y: 32.714 to 48.077 | Z: -5.335 to 14.945
Conformer 5: X: 12.106 to 30.913 | Y: 32.779 to 48.094 | Z: -5.409 to 14.897

M1 feature encoding validated successfully.


In [77]:
# CELL 71 — Inspect priority candidate records

import os
import pandas as pd

candidate_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "gru_priority_candidates.csv"
)

if not os.path.exists(candidate_path):
    raise FileNotFoundError(candidate_path)

candidate_df = pd.read_csv(candidate_path)

print("========== DATASET OVERVIEW ==========")
print("File:", candidate_path)
print("Number of candidates:", len(candidate_df))

print("\n========== AVAILABLE COLUMNS ==========")
for column in candidate_df.columns:
    print(column)

print("\n========== FIRST THREE RECORDS ==========")
display(candidate_df.head(3))

print("\n========== POSSIBLE SMILES COLUMNS ==========")

for column in candidate_df.columns:

    if "smiles" in column.lower():
        print("\nColumn:", column)
        print("Non-null values:", candidate_df[column].notna().sum())
        print("Example values:")
        print(candidate_df[column].dropna().head(3).tolist())

========== DATASET OVERVIEW ==========
File: /content/drive/MyDrive/Member2_Project/member2_outputs/gru_priority_candidates.csv
Number of candidates: 45

========== AVAILABLE COLUMNS ==========
candidate_id
canonical_smiles
molecular_weight
logP
HBD
HBA
TPSA
rotatable_bonds
QED
lipinski_violations
lipinski_pass
PAINS_alert
basic_property_pass
priority_candidate

========== FIRST THREE RECORDS ==========


,candidate_id,canonical_smiles,molecular_weight,logP,HBD,HBA,TPSA,rotatable_bonds,QED,lipinski_violations,lipinski_pass,PAINS_alert,basic_property_pass,priority_candidate
0,GRU_046,CNc1nnc(-c2ccc(OC)cc2)c2c1CCC2,255.321,2.6826,1,4,47.04,3,0.915644,0,True,False,True,True
1,GRU_093,C[C@H](NC(=O)c1ccc(Cl)cc1)c1ccccc1,259.736,3.8310,1,1,29.10,3,0.891784,0,True,False,True,True
2,GRU_079,CC(C)CC(=O)Nc1ccc(C(=O)N2CCN(CC(F)(F)F)CC2)cc1,371.403,2.9913,1,3,52.65,5,0.865816,0,True,False,True,True



========== POSSIBLE SMILES COLUMNS ==========

Column: canonical_smiles
Non-null values: 45
Example values:
['CNc1nnc(-c2ccc(OC)cc2)c2c1CCC2', 'C[C@H](NC(=O)c1ccc(Cl)cc1)c1ccccc1', 'CC(C)CC(=O)Nc1ccc(C(=O)N2CCN(CC(F)(F)F)CC2)cc1']


In [78]:
# CELL 72 — Validate and canonicalize all 45 candidate SMILES

from rdkit import Chem
import pandas as pd

validation_results = []

for _, row in candidate_df.iterrows():

    candidate_id = row["candidate_id"]
    smiles = str(row["canonical_smiles"]).strip()

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:

        validation_results.append({
            "candidate_id": candidate_id,
            "valid": False,
            "canonical_smiles": None,
            "error": "RDKit parsing failed"
        })

    else:

        canonical = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        validation_results.append({
            "candidate_id": candidate_id,
            "valid": True,
            "canonical_smiles": canonical,
            "error": None
        })

smiles_validation_df = pd.DataFrame(validation_results)

print("========== SMILES VALIDATION ==========")
print("Total candidates:", len(smiles_validation_df))
print("Valid:", int(smiles_validation_df["valid"].sum()))
print("Invalid:", int((~smiles_validation_df["valid"]).sum()))

print("\n========== INVALID CANDIDATES ==========")

display(
    smiles_validation_df[
        ~smiles_validation_df["valid"]
    ]
)

print("\n========== VALIDATION SUMMARY ==========")

if smiles_validation_df["valid"].all():
    print("All candidate SMILES passed RDKit validation.")
else:
    print("Some candidates require inspection before 3D generation.")

========== SMILES VALIDATION ==========
Total candidates: 45
Valid: 45
Invalid: 0

========== INVALID CANDIDATES ==========


,candidate_id,valid,canonical_smiles,error



========== VALIDATION SUMMARY ==========
All candidate SMILES passed RDKit validation.


In [79]:
# CELL 73 — Generate and optimize ligand 3D conformers

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
import os
import json

LIGAND_CONFORMERS_PER_MOLECULE = 5
RANDOM_SEED = 42

conformer_output_dir = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "ligand_3d_conformers"
)

os.makedirs(conformer_output_dir, exist_ok=True)

conformer_results = []

for _, row in smiles_validation_df.iterrows():

    candidate_id = row["candidate_id"]
    smiles = row["canonical_smiles"]

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = RANDOM_SEED
    params.numThreads = 0

    conformer_ids = AllChem.EmbedMultipleConfs(
        mol,
        numConfs=LIGAND_CONFORMERS_PER_MOLECULE,
        params=params
    )

    if len(conformer_ids) == 0:

        conformer_results.append({
            "candidate_id": candidate_id,
            "requested_conformers": LIGAND_CONFORMERS_PER_MOLECULE,
            "generated_conformers": 0,
            "optimized_conformers": 0,
            "status": "EMBEDDING_FAILED"
        })

        continue

    # Select force field
    mmff_properties = AllChem.MMFFGetMoleculeProperties(mol)

    force_field_name = "MMFF" if mmff_properties else "UFF"

    optimized_count = 0
    conformer_energies = []

    for conformer_id in conformer_ids:

        if mmff_properties:

            force_field = AllChem.MMFFGetMoleculeForceField(
                mol,
                mmff_properties,
                confId=conformer_id
            )

        else:

            force_field = AllChem.UFFGetMoleculeForceField(
                mol,
                confId=conformer_id
            )

        if force_field is None:
            continue

        force_field.Minimize(maxIts=300)

        energy = force_field.CalcEnergy()

        conformer_energies.append(float(energy))
        optimized_count += 1

    # Save molecule with generated conformers
    output_path = os.path.join(
        conformer_output_dir,
        f"{candidate_id}.sdf"
    )

    writer = Chem.SDWriter(output_path)
    writer.write(mol)
    writer.close()

    conformer_results.append({
        "candidate_id": candidate_id,
        "requested_conformers": LIGAND_CONFORMERS_PER_MOLECULE,
        "generated_conformers": len(conformer_ids),
        "optimized_conformers": optimized_count,
        "force_field": force_field_name,
        "minimum_energy": min(conformer_energies) if conformer_energies else None,
        "status": "SUCCESS" if optimized_count > 0 else "OPTIMIZATION_FAILED",
        "sdf_path": output_path
    })

    print(
        candidate_id,
        "| Generated:", len(conformer_ids),
        "| Optimized:", optimized_count,
        "| Force field:", force_field_name
    )

conformer_results_df = pd.DataFrame(conformer_results)

report_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "ligand_3d_generation_report.csv"
)

conformer_results_df.to_csv(report_path, index=False)

print("\n========== FINAL 3D GENERATION REPORT ==========")
print("Candidates:", len(conformer_results_df))
print("Successful:", (conformer_results_df["status"] == "SUCCESS").sum())
print("Failed:", (conformer_results_df["status"] != "SUCCESS").sum())
print(
    "Total generated conformers:",
    conformer_results_df["generated_conformers"].sum()
)
print("Report:", report_path)
print("SDF directory:", conformer_output_dir)

GRU_046 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_093 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_079 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_047 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_006 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_055 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_059 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_001 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_058 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_010 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_011 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_098 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_090 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_076 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_092 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_050 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_063 | Generated: 5 | Optimized: 5 | Force field: MMFF
GRU_025 | Gene

In [80]:
# CELL 74 — Verify saved 3D conformers

from rdkit import Chem
import os
import pandas as pd

verification_results = []

for candidate_id in candidate_df["candidate_id"]:

    sdf_path = os.path.join(
        conformer_output_dir,
        f"{candidate_id}.sdf"
    )

    supplier = Chem.SDMolSupplier(
        sdf_path,
        removeHs=False
    )

    molecules = [mol for mol in supplier if mol is not None]

    conformer_count = len(molecules)

    coordinate_checks = []

    for mol in molecules:

        has_coordinates = (
            mol.GetNumConformers() > 0
            and mol.GetConformer().Is3D()
        )

        coordinate_checks.append(has_coordinates)

    verification_results.append({
        "candidate_id": candidate_id,
        "saved_conformers": conformer_count,
        "all_have_3d_coordinates": (
            len(coordinate_checks) == 5
            and all(coordinate_checks)
        ),
        "status": (
            "PASS"
            if conformer_count == 5
            and all(coordinate_checks)
            else "CHECK"
        )
    })

sdf_verification_df = pd.DataFrame(verification_results)

print("========== SDF VERIFICATION ==========")
print(sdf_verification_df["status"].value_counts())

print("\nTotal saved conformers:",
      sdf_verification_df["saved_conformers"].sum())

print("\nCandidates requiring inspection:")
display(
    sdf_verification_df[
        sdf_verification_df["status"] != "PASS"
    ]
)

========== SDF VERIFICATION ==========
status
CHECK    45
Name: count, dtype: int64

Total saved conformers: 45

Candidates requiring inspection:


,candidate_id,saved_conformers,all_have_3d_coordinates,status
0,GRU_046,1,False,CHECK
1,GRU_093,1,False,CHECK
2,GRU_079,1,False,CHECK
3,GRU_047,1,False,CHECK
4,GRU_006,1,False,CHECK
5,GRU_055,1,False,CHECK
6,GRU_059,1,False,CHECK
7,GRU_001,1,False,CHECK
8,GRU_058,1,False,CHECK
9,GRU_010,1,False,CHECK


In [81]:
# CELL 73A — Correctly save all 5 conformers per molecule

from rdkit import Chem
from rdkit.Chem import AllChem
import os
import pandas as pd

corrected_results = []

for _, row in smiles_validation_df.iterrows():

    candidate_id = row["candidate_id"]
    smiles = row["canonical_smiles"]

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.numThreads = 0

    conformer_ids = list(
        AllChem.EmbedMultipleConfs(
            mol,
            numConfs=5,
            params=params
        )
    )

    if len(conformer_ids) != 5:
        raise RuntimeError(
            f"{candidate_id}: Expected 5 conformers, got {len(conformer_ids)}"
        )

    properties = AllChem.MMFFGetMoleculeProperties(mol)

    if properties is None:
        raise RuntimeError(
            f"{candidate_id}: MMFF parameters unavailable"
        )

    energies = []

    for conf_id in conformer_ids:

        force_field = AllChem.MMFFGetMoleculeForceField(
            mol,
            properties,
            confId=conf_id
        )

        if force_field is None:
            raise RuntimeError(
                f"{candidate_id}: Force field creation failed"
            )

        force_field.Minimize(maxIts=300)
        energies.append(float(force_field.CalcEnergy()))

    sdf_path = os.path.join(
        conformer_output_dir,
        f"{candidate_id}.sdf"
    )

    writer = Chem.SDWriter(sdf_path)

    # Explicitly write every conformer
    for conf_id in conformer_ids:

        mol.SetProp("candidate_id", candidate_id)
        mol.SetProp("conformer_id", str(conf_id))

        writer.write(mol, confId=conf_id)

    writer.close()

    corrected_results.append({
        "candidate_id": candidate_id,
        "generated": len(conformer_ids),
        "optimized": len(energies),
        "saved": len(conformer_ids),
        "minimum_energy": min(energies),
        "status": "SUCCESS"
    })

corrected_results_df = pd.DataFrame(corrected_results)

print("========== CORRECTED EXPORT ==========")
print("Candidates:", len(corrected_results_df))
print("Successful:", (corrected_results_df["status"] == "SUCCESS").sum())
print("Total conformers saved:", corrected_results_df["saved"].sum())

========== CORRECTED EXPORT ==========
Candidates: 45
Successful: 45
Total conformers saved: 225


In [82]:
# CELL 74 — Verify saved 3D conformers

from rdkit import Chem
import os
import pandas as pd

verification_results = []

for candidate_id in candidate_df["candidate_id"]:

    sdf_path = os.path.join(
        conformer_output_dir,
        f"{candidate_id}.sdf"
    )

    supplier = Chem.SDMolSupplier(
        sdf_path,
        removeHs=False
    )

    molecules = [mol for mol in supplier if mol is not None]

    conformer_count = len(molecules)

    coordinate_checks = []

    for mol in molecules:

        has_coordinates = (
            mol.GetNumConformers() > 0
            and mol.GetConformer().Is3D()
        )

        coordinate_checks.append(has_coordinates)

    verification_results.append({
        "candidate_id": candidate_id,
        "saved_conformers": conformer_count,
        "all_have_3d_coordinates": (
            len(coordinate_checks) == 5
            and all(coordinate_checks)
        ),
        "status": (
            "PASS"
            if conformer_count == 5
            and all(coordinate_checks)
            else "CHECK"
        )
    })

sdf_verification_df = pd.DataFrame(verification_results)

print("========== SDF VERIFICATION ==========")
print(sdf_verification_df["status"].value_counts())

print("\nTotal saved conformers:",
      sdf_verification_df["saved_conformers"].sum())

print("\nCandidates requiring inspection:")
display(
    sdf_verification_df[
        sdf_verification_df["status"] != "PASS"
    ]
)

========== SDF VERIFICATION ==========
status
PASS    45
Name: count, dtype: int64

Total saved conformers: 225

Candidates requiring inspection:


,candidate_id,saved_conformers,all_have_3d_coordinates,status


In [83]:
# CELL 75 — Extract 3D ligand pharmacophore features

from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig

import os
import glob
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Load RDKit pharmacophore definitions
# --------------------------------------------------

feature_definition_file = os.path.join(
    RDConfig.RDDataDir,
    "BaseFeatures.fdef"
)

feature_factory = ChemicalFeatures.BuildFeatureFactory(
    feature_definition_file
)

# Map RDKit feature families to M1 feature categories
feature_mapping = {
    "Hydrophobe": "HYDROPHOBIC",
    "LumpedHydrophobe": "HYDROPHOBIC",
    "Aromatic": "AROMATIC",
    "Donor": "DONOR",
    "Acceptor": "ACCEPTOR",
    "PosIonizable": "POSITIVE",
    "NegIonizable": "NEGATIVE"
}

# --------------------------------------------------
# 2. Locate saved ligand SDF files
# --------------------------------------------------

sdf_files = sorted(
    glob.glob(
        os.path.join(conformer_output_dir, "*.sdf")
    )
)

print("SDF files discovered:", len(sdf_files))

if len(sdf_files) != 45:
    raise RuntimeError(
        f"Expected 45 SDF files, found {len(sdf_files)}"
    )

# --------------------------------------------------
# 3. Extract features from every conformer
# --------------------------------------------------

ligand_feature_records = []
conformer_summary = []

for sdf_path in sdf_files:

    candidate_id = os.path.splitext(
        os.path.basename(sdf_path)
    )[0]

    supplier = Chem.SDMolSupplier(
        sdf_path,
        removeHs=False
    )

    valid_conformer_count = 0
    extracted_feature_count = 0

    for record_index, mol in enumerate(supplier):

        if mol is None:
            continue

        if mol.GetNumConformers() == 0:
            continue

        conformer_id = (
            mol.GetProp("conformer_id")
            if mol.HasProp("conformer_id")
            else str(record_index)
        )

        conformer = mol.GetConformer()

        if not conformer.Is3D():
            continue

        valid_conformer_count += 1

        features = feature_factory.GetFeaturesForMol(mol)

        for feature_index, feature in enumerate(features):

            family = feature.GetFamily()

            if family not in feature_mapping:
                continue

            category = feature_mapping[family]

            position = feature.GetPos()

            atom_ids = list(feature.GetAtomIds())

            ligand_feature_records.append({

                "candidate_id": candidate_id,
                "conformer_id": conformer_id,
                "feature_index": feature_index,
                "rdkit_family": family,
                "feature_type": category,

                "x": float(position.x),
                "y": float(position.y),
                "z": float(position.z),

                "atom_ids": ",".join(
                    map(str, atom_ids)
                ),

                "num_feature_atoms": len(atom_ids)

            })

            extracted_feature_count += 1

    conformer_summary.append({

        "candidate_id": candidate_id,
        "valid_3d_conformers": valid_conformer_count,
        "extracted_features": extracted_feature_count

    })

# --------------------------------------------------
# 4. Create DataFrames
# --------------------------------------------------

ligand_features_df = pd.DataFrame(
    ligand_feature_records
)

ligand_feature_summary_df = pd.DataFrame(
    conformer_summary
)

if ligand_features_df.empty:
    raise RuntimeError("No ligand pharmacophore features extracted.")

# --------------------------------------------------
# 5. Save outputs
# --------------------------------------------------

feature_output_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "ligand_3d_pharmacophore_features.csv"
)

summary_output_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "ligand_pharmacophore_summary.csv"
)

ligand_features_df.to_csv(
    feature_output_path,
    index=False
)

ligand_feature_summary_df.to_csv(
    summary_output_path,
    index=False
)

# --------------------------------------------------
# 6. Report
# --------------------------------------------------

print("\n========== LIGAND FEATURE EXTRACTION ==========")

print("Candidates:", ligand_feature_summary_df.shape[0])

print(
    "Valid 3D conformers:",
    ligand_feature_summary_df["valid_3d_conformers"].sum()
)

print("Total extracted features:", len(ligand_features_df))

print("\n========== FEATURE TYPE COUNTS ==========")

print(
    ligand_features_df["feature_type"].value_counts()
)

print("\n========== FIRST 10 FEATURES ==========")

display(ligand_features_df.head(10))

print("\n========== CONFORMER VALIDATION ==========")

print(
    ligand_feature_summary_df[
        "valid_3d_conformers"
    ].value_counts().sort_index()
)

print("\n========== OUTPUT FILES ==========")

print(feature_output_path)
print(summary_output_path)

SDF files discovered: 45

========== LIGAND FEATURE EXTRACTION ==========
Candidates: 45
Valid 3D conformers: 225
Total extracted features: 4265

========== FEATURE TYPE COUNTS ==========
feature_type
HYDROPHOBIC    2845
ACCEPTOR        625
AROMATIC        400
DONOR           320
POSITIVE         60
NEGATIVE         15
Name: count, dtype: int64

========== FIRST 10 FEATURES ==========


,candidate_id,conformer_id,feature_index,rdkit_family,feature_type,x,y,z,atom_ids,num_feature_atoms
0,GRU_001,0,0,Acceptor,ACCEPTOR,3.48960,2.43900,1.337200,1,1
1,GRU_001,0,1,Acceptor,ACCEPTOR,0.39080,-2.00500,0.217900,6,1
2,GRU_001,0,2,Acceptor,ACCEPTOR,-1.88570,1.33820,-0.113500,11,1
3,GRU_001,0,3,Aromatic,AROMATIC,1.97675,0.19865,0.751617,"2,3,4,5,8,22",6
4,GRU_001,0,4,Aromatic,AROMATIC,-2.70915,0.30545,-0.548200,"10,11,12,13,18,19",6
5,GRU_001,0,5,Hydrophobe,HYDROPHOBIC,3.25500,0.27580,0.187400,3,1
6,GRU_001,0,6,Hydrophobe,HYDROPHOBIC,2.44140,-0.83130,-0.066000,4,1
7,GRU_001,0,7,Hydrophobe,HYDROPHOBIC,0.68310,0.12770,1.309500,8,1
8,GRU_001,0,8,Hydrophobe,HYDROPHOBIC,-3.46970,0.46920,-1.714900,13,1
9,GRU_001,0,9,Hydrophobe,HYDROPHOBIC,-3.53730,-0.72390,-0.986200,18,1



========== CONFORMER VALIDATION ==========
valid_3d_conformers
5    45
Name: count, dtype: int64

========== OUTPUT FILES ==========
/content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_pharmacophore_features.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/ligand_pharmacophore_summary.csv


In [84]:
# CELL 76 — Audit ligand and pocket coordinate frames

import numpy as np
import pandas as pd

print("========== LIGAND COORDINATE AUDIT ==========")

for axis in ["x", "y", "z"]:

    print(
        f"{axis.upper()}:",
        "Min =", round(ligand_features_df[axis].min(), 3),
        "| Max =", round(ligand_features_df[axis].max(), 3),
        "| Mean =", round(ligand_features_df[axis].mean(), 3)
    )

print("\n========== M1 POCKET COORDINATE AUDIT ==========")

for axis, column in zip(
    ["x", "y", "z"],
    range(3)
):

    values = m1_tensor[:, :, column]

    print(
        f"{axis.upper()}:",
        "Min =", round(float(values.min()), 3),
        "| Max =", round(float(values.max()), 3),
        "| Mean =", round(float(values.mean()), 3)
    )

print("\n========== FEATURE COUNTS PER CONFORMER ==========")

feature_counts = (
    ligand_features_df
    .groupby(["candidate_id", "conformer_id", "feature_type"])
    .size()
    .unstack(fill_value=0)
)

display(feature_counts.head(10))

print("\n========== CONFORMER FEATURE COUNT VARIATION ==========")

print(
    feature_counts
    .groupby(level="candidate_id")
    .mean()
    .describe()
)

print("\nCoordinate audit completed.")

========== LIGAND COORDINATE AUDIT ==========
X: Min = -9.124 | Max = 9.202 | Mean = -0.01
Y: Min = -5.371 | Max = 6.572 | Mean = 0.06
Z: Min = -6.352 | Max = 6.109 | Mean = 0.022

========== M1 POCKET COORDINATE AUDIT ==========
X: Min = 11.561 | Max = 31.002 | Mean = 23.25
Y: Min = 32.666 | Max = 48.172 | Mean = 41.144
Z: Min = -5.617 | Max = 15.024 | Mean = 5.721

========== FEATURE COUNTS PER CONFORMER ==========


feature_type               ACCEPTOR  AROMATIC  DONOR  HYDROPHOBIC  NEGATIVE  \
candidate_id conformer_id                                                     
GRU_001      0                    3         2      0            8         0   
             1                    3         2      0            8         0   
             2                    3         2      0            8         0   
             3                    3         2      0            8         0   
             4                    3         2      0            8         0   
GRU_004      0                    4         1      1           15         0   
             1                    4         1      1           15         0   
             2                    4         1      1           15         0   
             3                    4         1      1           15         0   
             4                    4         1      1           15         0   

feature_type               POSITIVE  
candidate_id conformer_id            
GRU_001      0                    0  
             1                    0  
             2                    0  
             3                    0  
             4                    0  
GRU_004      0                    0  
             1                    0  
             2                    0  
             3                    0  
             4                    0


========== CONFORMER FEATURE COUNT VARIATION ==========
feature_type   ACCEPTOR   AROMATIC      DONOR  HYDROPHOBIC   NEGATIVE  \
count         45.000000  45.000000  45.000000    45.000000  45.000000   
mean           2.777778   1.777778   1.422222    12.644444   0.066667   
std            1.312335   0.901738   0.941201     4.637572   0.252262   
min            0.000000   0.000000   0.000000     3.000000   0.000000   
25%            2.000000   1.000000   1.000000     9.000000   0.000000   
50%            3.000000   2.000000   2.000000    12.000000   0.000000   
75%            3.000000   2.000000   2.000000    16.000000   0.000000   
max            7.000000   3.000000   3.000000    23.000000   1.000000   

feature_type   POSITIVE  
count         45.000000  
mean           0.266667  
std            0.579969  
min            0.000000  
25%            0.000000  
50%            0.000000  
75%            0.000000  
max            2.000000  

Coordinate audit completed.


In [85]:
# CELL 77 — Compare M1 feature coordinates with protein PDB atoms

import os
import glob
import numpy as np
import pandas as pd

from Bio.PDB import PDBParser

# Locate first M1 conformer
pdb_files = glob.glob(
    os.path.join(
        PROJECT_ROOT,
        "member1_input",
        "**",
        "conformer_001.pdb"
    ),
    recursive=True
)

if not pdb_files:
    raise FileNotFoundError("M1 conformer_001.pdb not found")

pdb_path = pdb_files[0]

parser = PDBParser(QUIET=True)
structure = parser.get_structure("M1", pdb_path)

# Build atom-coordinate lookup
atom_lookup = {}

for atom in structure.get_atoms():

    if atom.element.upper() in ["H", "D"]:
        continue

    residue = atom.get_parent()
    chain = residue.get_parent()

    residue_number = residue.id[1]
    insertion_code = residue.id[2].strip()
    residue_name = residue.resname
    atom_name = atom.name.strip()

    key = (
        chain.id,
        str(residue_number),
        insertion_code,
        residue_name,
        atom_name
    )

    atom_lookup[key] = np.asarray(
        atom.coord,
        dtype=float
    )

# Compare feature positions
comparison_records = []

for feature in m1_features:

    identity_string = feature["identity"]

    identities = identity_string.split(";")

    atom_coordinates = []
    missing_atoms = []

    for identity in identities:

        parts = identity.split("|")

        if len(parts) != 5:
            missing_atoms.append(identity)
            continue

        key = (
            parts[0],
            parts[1],
            parts[2],
            parts[3],
            parts[4]
        )

        if key in atom_lookup:
            atom_coordinates.append(atom_lookup[key])
        else:
            missing_atoms.append(identity)

    if not atom_coordinates:
        continue

    expected_position = np.mean(
        atom_coordinates,
        axis=0
    )

    tensor_position = m1_tensor[
        0,
        int(feature["feature_index"]),
        :3
    ]

    difference = np.linalg.norm(
        expected_position - tensor_position
    )

    comparison_records.append({

        "feature_index": feature["feature_index"],
        "feature_type": feature["feature"],
        "matched_atoms": len(atom_coordinates),
        "missing_atoms": len(missing_atoms),
        "coordinate_difference_A": float(difference)

    })

comparison_df = pd.DataFrame(comparison_records)

print("========== M1 PDB COORDINATE VALIDATION ==========")

print("PDB:", pdb_path)

print("Features compared:", len(comparison_df))

print(
    "Features with all atoms found:",
    (
        comparison_df["missing_atoms"] == 0
    ).sum()
)

print(
    "Mean coordinate difference (Å):",
    round(comparison_df["coordinate_difference_A"].mean(), 5)
)

print(
    "Maximum coordinate difference (Å):",
    round(comparison_df["coordinate_difference_A"].max(), 5)
)

print("\n========== FIRST 10 COMPARISONS ==========")

display(comparison_df.head(10))

print("\n========== DIFFERENCE DISTRIBUTION ==========")

display(
    comparison_df["coordinate_difference_A"].describe()
)

========== M1 PDB COORDINATE VALIDATION ==========
PDB: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/conformer_001.pdb
Features compared: 95
Features with all atoms found: 95
Mean coordinate difference (Å): 0.0
Maximum coordinate difference (Å): 0.0

========== FIRST 10 COMPARISONS ==========


,feature_index,feature_type,matched_atoms,missing_atoms,coordinate_difference_A
0,0,HYDROPHOBIC,1,0,0.000000e+00
1,1,AROMATIC,5,0,3.814726e-07
2,2,POSITIVE,5,0,3.814726e-07
3,3,DONOR,1,0,0.000000e+00
4,4,DONOR,1,0,0.000000e+00
5,5,POSITIVE,1,0,0.000000e+00
6,6,DONOR,1,0,0.000000e+00
7,7,ACCEPTOR,1,0,0.000000e+00
8,8,HYDROPHOBIC,1,0,0.000000e+00
9,9,HYDROPHOBIC,1,0,0.000000e+00



========== DIFFERENCE DISTRIBUTION ==========


,coordinate_difference_A
count,9.500000e+01
mean,9.679932e-08
std,2.912016e-07
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,1.313107e-06


In [86]:
# CELL 78 — Validate tensor coordinates against all five PDB structures

import os
import glob
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser

pdb_files = sorted(
    glob.glob(
        os.path.join(
            PROJECT_ROOT,
            "member1_input",
            "**",
            "conformer_*.pdb"
        ),
        recursive=True
    )
)

if len(pdb_files) != m1_tensor.shape[0]:
    raise RuntimeError(
        f"Expected {m1_tensor.shape[0]} PDB files, found {len(pdb_files)}"
    )

parser = PDBParser(QUIET=True)

validation_records = []

for conformer_idx, pdb_path in enumerate(pdb_files):

    structure = parser.get_structure(
        f"M1_{conformer_idx + 1}",
        pdb_path
    )

    atom_lookup = {}

    for atom in structure.get_atoms():

        if atom.element.upper() in ["H", "D"]:
            continue

        residue = atom.get_parent()
        chain = residue.get_parent()

        key = (
            chain.id,
            str(residue.id[1]),
            residue.id[2].strip(),
            residue.resname,
            atom.name.strip()
        )

        atom_lookup[key] = np.asarray(
            atom.coord,
            dtype=float
        )

    differences = []
    missing_features = 0

    for feature in m1_features:

        coordinates = []

        for identity in feature["identity"].split(";"):

            parts = identity.split("|")

            key = (
                parts[0],
                parts[1],
                parts[2],
                parts[3],
                parts[4]
            )

            if key in atom_lookup:
                coordinates.append(atom_lookup[key])

        if not coordinates:
            missing_features += 1
            continue

        expected_position = np.mean(
            coordinates,
            axis=0
        )

        tensor_position = m1_tensor[
            conformer_idx,
            int(feature["feature_index"]),
            :3
        ]

        differences.append(
            np.linalg.norm(
                expected_position - tensor_position
            )
        )

    validation_records.append({

        "conformer": conformer_idx + 1,
        "pdb_file": os.path.basename(pdb_path),
        "features_compared": len(differences),
        "missing_features": missing_features,
        "mean_difference_A": np.mean(differences),
        "maximum_difference_A": np.max(differences),
        "status": (
            "PASS"
            if len(differences) == len(m1_features)
            and missing_features == 0
            and np.max(differences) < 0.001
            else "CHECK"
        )
    })

ensemble_validation_df = pd.DataFrame(validation_records)

print("========== FIVE-CONFORMER VALIDATION ==========")
display(ensemble_validation_df)

print("\nOverall status:")

print(
    "PASS"
    if (ensemble_validation_df["status"] == "PASS").all()
    else "CHECK"
)

========== FIVE-CONFORMER VALIDATION ==========


,conformer,pdb_file,features_compared,missing_features,mean_difference_A,maximum_difference_A,status
0,1,conformer_001.pdb,95,0,9.679932e-08,0.000001,PASS
1,2,conformer_002.pdb,95,0,1.229527e-07,0.000002,PASS
2,3,conformer_003.pdb,95,0,1.438716e-07,0.000002,PASS
3,4,conformer_004.pdb,95,0,1.422600e-07,0.000002,PASS
4,5,conformer_005.pdb,95,0,1.140765e-07,0.000001,PASS



Overall status:
PASS


In [87]:
# CELL 79 — Prepare receptor pharmacophore and protein coordinates

import os
import glob
import numpy as np
import pandas as pd

from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)

pdb_files = sorted(
    glob.glob(
        os.path.join(
            PROJECT_ROOT,
            "member1_input",
            "**",
            "conformer_*.pdb"
        ),
        recursive=True
    )
)

receptor_features_by_conformer = {}
protein_atoms_by_conformer = {}

for conf_idx, pdb_path in enumerate(pdb_files):

    conf_id = conf_idx + 1

    # ------------------------------------------
    # 1. Receptor pharmacophore features
    # ------------------------------------------

    feature_records = []

    for feature in m1_features:

        feature_idx = int(feature["feature_index"])

        xyz = m1_tensor[
            conf_idx,
            feature_idx,
            :3
        ]

        feature_records.append({

            "feature_index": feature_idx,
            "feature_type": feature["feature"],
            "feature_id": int(feature["feature_id"]),
            "x": float(xyz[0]),
            "y": float(xyz[1]),
            "z": float(xyz[2]),
            "identity": feature["identity"]

        })

    receptor_features_by_conformer[conf_id] = pd.DataFrame(
        feature_records
    )

    # ------------------------------------------
    # 2. Protein heavy atoms
    # ------------------------------------------

    structure = parser.get_structure(
        f"receptor_{conf_id}",
        pdb_path
    )

    atom_records = []

    for atom in structure.get_atoms():

        if atom.element.upper() in ["H", "D"]:
            continue

        residue = atom.get_parent()

        atom_records.append({

            "atom_name": atom.name.strip(),
            "element": atom.element.upper(),
            "residue_name": residue.resname,
            "residue_number": residue.id[1],
            "chain": residue.get_parent().id,
            "x": float(atom.coord[0]),
            "y": float(atom.coord[1]),
            "z": float(atom.coord[2])

        })

    protein_atoms_by_conformer[conf_id] = pd.DataFrame(
        atom_records
    )

# ------------------------------------------
# 3. Define complementary feature pairs
# ------------------------------------------

compatible_feature_pairs = {

    "HYDROPHOBIC": ["HYDROPHOBIC"],

    "AROMATIC": ["AROMATIC"],

    "DONOR": ["ACCEPTOR"],

    "ACCEPTOR": ["DONOR"],

    "POSITIVE": ["NEGATIVE"],

    "NEGATIVE": ["POSITIVE"]

}

print("========== RECEPTOR PREPARATION ==========")

for conf_id in receptor_features_by_conformer:

    receptor_df = receptor_features_by_conformer[conf_id]
    protein_df = protein_atoms_by_conformer[conf_id]

    print(
        f"Conformer {conf_id}:",
        "Receptor features =", len(receptor_df),
        "| Protein heavy atoms =", len(protein_df)
    )

print("\n========== RECEPTOR FEATURE COUNTS ==========")

display(
    receptor_features_by_conformer[1][
        "feature_type"
    ].value_counts()
)

print("\n========== COMPATIBLE FEATURE RULES ==========")

for receptor_type, ligand_types in compatible_feature_pairs.items():

    print(
        receptor_type,
        "<->",
        ", ".join(ligand_types)
    )

print("\nReceptor preparation completed.")

========== RECEPTOR PREPARATION ==========
Conformer 1: Receptor features = 95 | Protein heavy atoms = 5445
Conformer 2: Receptor features = 95 | Protein heavy atoms = 5445
Conformer 3: Receptor features = 95 | Protein heavy atoms = 5445
Conformer 4: Receptor features = 95 | Protein heavy atoms = 5445
Conformer 5: Receptor features = 95 | Protein heavy atoms = 5445

========== RECEPTOR FEATURE COUNTS ==========


,count
feature_type,
HYDROPHOBIC,45
DONOR,22
ACCEPTOR,18
POSITIVE,5
AROMATIC,4
NEGATIVE,1



========== COMPATIBLE FEATURE RULES ==========
HYDROPHOBIC <-> HYDROPHOBIC
AROMATIC <-> AROMATIC
DONOR <-> ACCEPTOR
ACCEPTOR <-> DONOR
POSITIVE <-> NEGATIVE
NEGATIVE <-> POSITIVE

Receptor preparation completed.


In [88]:
# CELL 80 — Ligand–receptor feature compatibility audit

import pandas as pd

receptor_types = set(
    receptor_features_by_conformer[1]["feature_type"]
)

records = []

for candidate_id, group in ligand_features_df.groupby("candidate_id"):

    # Use conformer 0 for the initial feature inventory
    ligand_conf = group[group["conformer_id"] == 0]

    ligand_counts = (
        ligand_conf["feature_type"]
        .value_counts()
        .to_dict()
    )

    compatible_pairs = 0
    compatible_types = []

    for receptor_type, ligand_types in compatible_feature_pairs.items():

        if receptor_type not in receptor_types:
            continue

        receptor_count = len(
            receptor_features_by_conformer[1][
                receptor_features_by_conformer[1]["feature_type"]
                == receptor_type
            ]
        )

        ligand_count = sum(
            ligand_counts.get(t, 0)
            for t in ligand_types
        )

        if ligand_count > 0:

            compatible_pairs += receptor_count * ligand_count

            compatible_types.append(
                receptor_type
            )

    records.append({

        "candidate_id": candidate_id,
        "ligand_feature_count": len(ligand_conf),
        "potential_compatible_feature_pairs": compatible_pairs,
        "compatible_feature_types": ", ".join(compatible_types)

    })

compatibility_audit_df = pd.DataFrame(records)

compatibility_audit_df = compatibility_audit_df.sort_values(
    "potential_compatible_feature_pairs",
    ascending=False
).reset_index(drop=True)

print("========== FEATURE COMPATIBILITY AUDIT ==========")

print("Candidates evaluated:", len(compatibility_audit_df))

display(compatibility_audit_df.head(15))

print("\nSummary:")

display(
    compatibility_audit_df[
        "potential_compatible_feature_pairs"
    ].describe()
)

print(
    "\nCandidates with at least one compatible feature type:",
    (
        compatibility_audit_df[
            "potential_compatible_feature_pairs"
        ] > 0
    ).sum()
)

========== FEATURE COMPATIBILITY AUDIT ==========
Candidates evaluated: 45


,candidate_id,ligand_feature_count,potential_compatible_feature_pairs,compatible_feature_types
0,GRU_001,0,0,
1,GRU_004,0,0,
2,GRU_006,0,0,
3,GRU_008,0,0,
4,GRU_010,0,0,
5,GRU_011,0,0,
6,GRU_014,0,0,
7,GRU_016,0,0,
8,GRU_018,0,0,
9,GRU_020,0,0,



Summary:


,potential_compatible_feature_pairs
count,45.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0



Candidates with at least one compatible feature type: 0


In [89]:
# CELL 80A — Inspect ligand feature data

print("========== LIGAND FEATURE DATA ==========")

print("Total feature records:", len(ligand_features_df))

print("\nColumns:")
print(ligand_features_df.columns.tolist())

print("\nConformer ID values:")
print(
    ligand_features_df["conformer_id"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nConformer ID data type:")
print(ligand_features_df["conformer_id"].dtype)

print("\nFeature types:")
print(
    ligand_features_df["feature_type"]
    .value_counts(dropna=False)
)

print("\nCandidate IDs:")
print(
    "Unique candidates:",
    ligand_features_df["candidate_id"].nunique()
)

print("\nSample records:")
display(ligand_features_df.head(10))

========== LIGAND FEATURE DATA ==========
Total feature records: 4265

Columns:
['candidate_id', 'conformer_id', 'feature_index', 'rdkit_family', 'feature_type', 'x', 'y', 'z', 'atom_ids', 'num_feature_atoms']

Conformer ID values:
conformer_id
0    853
1    853
2    853
3    853
4    853
Name: count, dtype: int64

Conformer ID data type:
object

Feature types:
feature_type
HYDROPHOBIC    2845
ACCEPTOR        625
AROMATIC        400
DONOR           320
POSITIVE         60
NEGATIVE         15
Name: count, dtype: int64

Candidate IDs:
Unique candidates: 45

Sample records:


,candidate_id,conformer_id,feature_index,rdkit_family,feature_type,x,y,z,atom_ids,num_feature_atoms
0,GRU_001,0,0,Acceptor,ACCEPTOR,3.48960,2.43900,1.337200,1,1
1,GRU_001,0,1,Acceptor,ACCEPTOR,0.39080,-2.00500,0.217900,6,1
2,GRU_001,0,2,Acceptor,ACCEPTOR,-1.88570,1.33820,-0.113500,11,1
3,GRU_001,0,3,Aromatic,AROMATIC,1.97675,0.19865,0.751617,"2,3,4,5,8,22",6
4,GRU_001,0,4,Aromatic,AROMATIC,-2.70915,0.30545,-0.548200,"10,11,12,13,18,19",6
5,GRU_001,0,5,Hydrophobe,HYDROPHOBIC,3.25500,0.27580,0.187400,3,1
6,GRU_001,0,6,Hydrophobe,HYDROPHOBIC,2.44140,-0.83130,-0.066000,4,1
7,GRU_001,0,7,Hydrophobe,HYDROPHOBIC,0.68310,0.12770,1.309500,8,1
8,GRU_001,0,8,Hydrophobe,HYDROPHOBIC,-3.46970,0.46920,-1.714900,13,1
9,GRU_001,0,9,Hydrophobe,HYDROPHOBIC,-3.53730,-0.72390,-0.986200,18,1


In [90]:
# CELL 80B — Diagnose conformer ID values

print("Unique values with Python types:")

for value in ligand_features_df["conformer_id"].unique():
    print(repr(value), type(value))

print("\nEquality check:")

print(
    "Using integer 0:",
    (ligand_features_df["conformer_id"] == 0).sum()
)

print(
    "Using string '0':",
    (ligand_features_df["conformer_id"] == "0").sum()
)

print("\nRecords for first candidate:")

first_candidate = ligand_features_df["candidate_id"].iloc[0]

display(
    ligand_features_df[
        (ligand_features_df["candidate_id"] == first_candidate)
    ][["conformer_id", "feature_type"]]
    .groupby(["conformer_id", "feature_type"])
    .size()
    .reset_index(name="count")
)

Unique values with Python types:
'0' <class 'str'>
'1' <class 'str'>
'2' <class 'str'>
'3' <class 'str'>
'4' <class 'str'>

Equality check:
Using integer 0: 0
Using string '0': 853

Records for first candidate:


,conformer_id,feature_type,count
0,0,ACCEPTOR,3
1,0,AROMATIC,2
2,0,HYDROPHOBIC,8
3,1,ACCEPTOR,3
4,1,AROMATIC,2
5,1,HYDROPHOBIC,8
6,2,ACCEPTOR,3
7,2,AROMATIC,2
8,2,HYDROPHOBIC,8
9,3,ACCEPTOR,3


In [91]:
# CELL 80 — Corrected ligand–receptor feature compatibility audit

import pandas as pd

# Normalize conformer IDs
ligand_features_df["conformer_id"] = (
    ligand_features_df["conformer_id"].astype(str)
)

receptor_types = set(
    receptor_features_by_conformer[1]["feature_type"]
)

records = []

for candidate_id, group in ligand_features_df.groupby("candidate_id"):

    # Select conformer 0 using string comparison
    ligand_conf = group[
        group["conformer_id"] == "0"
    ]

    ligand_counts = (
        ligand_conf["feature_type"]
        .value_counts()
        .to_dict()
    )

    compatible_pairs = 0
    compatible_types = []

    for receptor_type, ligand_types in compatible_feature_pairs.items():

        if receptor_type not in receptor_types:
            continue

        receptor_count = (
            receptor_features_by_conformer[1]["feature_type"]
            .eq(receptor_type)
            .sum()
        )

        ligand_count = sum(
            ligand_counts.get(t, 0)
            for t in ligand_types
        )

        if ligand_count > 0:

            compatible_pairs += receptor_count * ligand_count

            compatible_types.append(receptor_type)

    records.append({

        "candidate_id": candidate_id,
        "ligand_feature_count": len(ligand_conf),
        "potential_compatible_feature_pairs": compatible_pairs,
        "compatible_feature_types": ", ".join(compatible_types)

    })

compatibility_audit_df = pd.DataFrame(records)

compatibility_audit_df = compatibility_audit_df.sort_values(
    "potential_compatible_feature_pairs",
    ascending=False
).reset_index(drop=True)

print("========== CORRECTED FEATURE COMPATIBILITY AUDIT ==========")

print("Candidates evaluated:", len(compatibility_audit_df))

display(compatibility_audit_df.head(15))

print("\nSummary:")

display(
    compatibility_audit_df[
        "potential_compatible_feature_pairs"
    ].describe()
)

print(
    "\nCandidates with compatible feature types:",
    (
        compatibility_audit_df[
            "potential_compatible_feature_pairs"
        ] > 0
    ).sum()
)

========== CORRECTED FEATURE COMPATIBILITY AUDIT ==========
Candidates evaluated: 45


,candidate_id,ligand_feature_count,potential_compatible_feature_pairs,compatible_feature_types
0,GRU_026,26,1083,"HYDROPHOBIC, AROMATIC, DONOR"
1,GRU_091,29,1046,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR, POSITIVE"
2,GRU_083,30,1026,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"
3,GRU_036,25,966,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"
4,GRU_035,27,946,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"
5,GRU_011,24,898,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"
6,GRU_065,24,888,"HYDROPHOBIC, AROMATIC, DONOR"
7,GRU_077,23,871,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"
8,GRU_058,22,862,"HYDROPHOBIC, AROMATIC, DONOR"
9,GRU_034,23,857,"HYDROPHOBIC, AROMATIC, DONOR, ACCEPTOR"



Summary:


,potential_compatible_feature_pairs
count,45.000000
mean,663.422222
std,212.540851
min,257.000000
25%,475.000000
50%,676.000000
75%,850.000000
max,1083.000000



Candidates with compatible feature types: 45


In [92]:
# CELL 81 — Locate experimentally validated structure files

import os
import glob

structure_files = []

for extension in ["*.cif", "*.mmcif", "*.pdb"]:

    structure_files.extend(
        glob.glob(
            os.path.join(PROJECT_ROOT, "**", extension),
            recursive=True
        )
    )

structure_files = sorted(set(structure_files))

print("========== EXPERIMENTAL STRUCTURE INVENTORY ==========")

print("Total structure files:", len(structure_files))

for path in structure_files[:40]:
    print(os.path.relpath(path, PROJECT_ROOT))

print("\nValidated PARP1 structures:")

validated_pdb_ids = [
    "5KPP", "5KPQ", "5WRZ", "5WS1", "7AAC",
    "7ONS", "7ONT", "8JNZ", "9ETQ", "9ILN"
]

for pdb_id in validated_pdb_ids:

    matches = [
        path for path in structure_files
        if pdb_id.lower() in os.path.basename(path).lower()
    ]

    print(pdb_id, "->", len(matches), "file(s)")

========== EXPERIMENTAL STRUCTURE INVENTORY ==========
Total structure files: 580
final_selected_500/receptors/ADHX_HUMAN_2_374_0/1m6w_A_rec.pdb
final_selected_500/receptors/ADK_HUMAN_15_362_0/1bx4_A_rec.pdb
final_selected_500/receptors/ADRB2_HUMAN_26_357_EH234_0/3pds_A_rec.pdb
final_selected_500/receptors/ADRB2_HUMAN_26_357_Galpha_0/3d4s_A_rec.pdb
final_selected_500/receptors/ADRB2_HUMAN_26_357_TM_0/2r4r_A_rec.pdb
final_selected_500/receptors/AER_ARATH_1_345_0/2j3i_B_rec.pdb
final_selected_500/receptors/AGAA_GEOSE_1_729_0/4fnp_D_rec.pdb
final_selected_500/receptors/AGI1_WHEAT_27_201_glycopep_0/2cwg_A_rec.pdb
final_selected_500/receptors/AGO2_HUMAN_438_575_0/3luc_A_rec.pdb
final_selected_500/receptors/AGTR1_HUMAN_14_317_0/4yay_A_rec.pdb
final_selected_500/receptors/AGTR2_HUMAN_35_335_0/5unf_B_rec.pdb
final_selected_500/receptors/AHK4_ARATH_149_418_0/3t4j_B_rec.pdb
final_selected_500/receptors/AIPL1_HUMAN_1_161_0/5u9a_A_rec.pdb
final_selected_500/receptors/AK1BA_HUMAN_1_316_0/1zua_X_rec

In [93]:
# CELL 81A — Identify exact files for validated PARP1 structures

import os
import glob

validated_pdb_ids = [
    "5KPP", "5KPQ", "5WRZ", "5WS1", "7AAC",
    "7ONS", "7ONT", "8JNZ", "9ETQ", "9ILN"
]

all_files = glob.glob(
    os.path.join(PROJECT_ROOT, "**", "*"),
    recursive=True
)

for pdb_id in validated_pdb_ids:

    print(f"\n========== {pdb_id} ==========")

    matches = [
        path for path in all_files
        if os.path.isfile(path)
        and pdb_id.lower() in os.path.basename(path).lower()
    ]

    for path in matches:

        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            os.path.relpath(path, PROJECT_ROOT),
            f"| {size_mb:.2f} MB"
        )


========== 5KPP ==========
member2_outputs/target_identification/experimental_mmcif/5KPP.cif | 0.60 MB
member2_outputs/experimental_mmcif/5KPP.cif | 0.60 MB

========== 5KPQ ==========
member2_outputs/target_identification/experimental_mmcif/5KPQ.cif | 0.59 MB
member2_outputs/experimental_mmcif/5KPQ.cif | 0.59 MB

========== 5WRZ ==========
member2_outputs/target_identification/experimental_mmcif/5WRZ.cif | 0.58 MB
member2_outputs/experimental_mmcif/5WRZ.cif | 0.58 MB

========== 5WS1 ==========
member2_outputs/target_identification/experimental_mmcif/5WS1.cif | 0.59 MB
member2_outputs/experimental_mmcif/5WS1.cif | 0.59 MB

========== 7AAC ==========
member2_outputs/target_identification/experimental_mmcif/7AAC.cif | 1.09 MB
member2_outputs/experimental_mmcif/7AAC.cif | 1.09 MB

========== 7ONS ==========
member2_outputs/target_identification/experimental_mmcif/7ONS.cif | 1.09 MB
member2_outputs/experimental_mmcif/7ONS.cif | 1.09 MB

========== 7ONT ==========
member2_outputs/target_i

In [94]:
# CELL 82 — Verify experimental ligand instances in mmCIF files

import os
import numpy as np
import pandas as pd

from Bio.PDB.MMCIF2Dict import MMCIF2Dict

cif_dir = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "target_identification",
    "experimental_mmcif"
)

validated_ligands = {
    "5KPP": "6WZ",
    "5KPQ": "6X2",
    "5WRZ": "7U3",
    "5WS1": "7U9",
    "7AAC": "78P",
    "7ONS": "VKT",
    "7ONT": "VKQ",
    "8JNZ": "ERV",
    "9ETQ": "A1H63",
    "9ILN": "A1D9M"
}

records = []

for pdb_id, expected_comp_id in validated_ligands.items():

    cif_path = os.path.join(cif_dir, f"{pdb_id}.cif")

    if not os.path.exists(cif_path):
        raise FileNotFoundError(cif_path)

    cif = MMCIF2Dict(cif_path)

    atom_group = cif["_atom_site.group_PDB"]
    comp_ids = cif["_atom_site.label_comp_id"]
    label_chains = cif["_atom_site.label_asym_id"]
    auth_chains = cif["_atom_site.auth_asym_id"]

    atom_names = cif["_atom_site.label_atom_id"]

    x = cif["_atom_site.Cartn_x"]
    y = cif["_atom_site.Cartn_y"]
    z = cif["_atom_site.Cartn_z"]

    model_numbers = cif["_atom_site.pdbx_PDB_model_num"]

    matching_indices = [
        i for i, comp_id in enumerate(comp_ids)
        if comp_id == expected_comp_id
        and atom_group[i] == "HETATM"
    ]

    if not matching_indices:
        records.append({
            "pdb_id": pdb_id,
            "expected_component": expected_comp_id,
            "instances": 0,
            "atom_count": 0,
            "chains": "",
            "status": "NOT_FOUND"
        })
        continue

    # Group atoms by chain and model
    instances = {}

    for i in matching_indices:

        key = (
            label_chains[i],
            auth_chains[i],
            model_numbers[i]
        )

        instances.setdefault(key, []).append(i)

    for (label_chain, auth_chain, model), indices in instances.items():

        coordinates = np.array([
            [float(x[i]), float(y[i]), float(z[i])]
            for i in indices
        ])

        records.append({

            "pdb_id": pdb_id,
            "expected_component": expected_comp_id,
            "label_chain": label_chain,
            "auth_chain": auth_chain,
            "model": model,
            "atom_count": len(indices),
            "unique_atom_names": len(set(atom_names[i] for i in indices)),
            "centroid_x": coordinates[:, 0].mean(),
            "centroid_y": coordinates[:, 1].mean(),
            "centroid_z": coordinates[:, 2].mean(),
            "status": "FOUND"

        })

ligand_instance_audit_df = pd.DataFrame(records)

print("========== EXPERIMENTAL LIGAND INSTANCE AUDIT ==========")

display(ligand_instance_audit_df)

print("\nSummary:")

print(
    "Structures checked:",
    len(validated_ligands)
)

print(
    "Ligand instances found:",
    (ligand_instance_audit_df["status"] == "FOUND").sum()
)

print(
    "Structures with ligand coordinates:",
    ligand_instance_audit_df.loc[
        ligand_instance_audit_df["status"] == "FOUND",
        "pdb_id"
    ].nunique()
)

========== EXPERIMENTAL LIGAND INSTANCE AUDIT ==========


,pdb_id,expected_component,label_chain,auth_chain,model,atom_count,unique_atom_names,centroid_x,centroid_y,centroid_z,status
0,5KPP,6WZ,C,A,1,34,34,12.781676,-4.087412,-8.749500,FOUND
1,5KPP,6WZ,D,B,1,34,34,-4.821235,-17.417500,-34.640412,FOUND
2,5KPQ,6X2,C,A,1,32,32,12.278594,-3.998125,-8.653594,FOUND
3,5KPQ,6X2,D,B,1,32,32,-4.522406,-17.394781,-34.802594,FOUND
4,5WRZ,7U3,C,A,1,19,19,-9.650053,-4.358842,7.791474,FOUND
5,5WRZ,7U3,D,B,1,19,19,0.845579,-18.471579,34.891684,FOUND
6,5WS1,7U9,C,A,1,20,20,-9.992550,-4.787700,7.468050,FOUND
7,5WS1,7U9,D,B,1,20,20,1.494900,-18.319600,34.382350,FOUND
8,7AAC,78P,C,A,1,18,18,15.061222,41.630778,7.603222,FOUND
9,7AAC,78P,E,B,1,18,18,25.096500,28.170722,34.536833,FOUND



Summary:
Structures checked: 10
Ligand instances found: 20
Structures with ligand coordinates: 10


In [95]:
# CELL 83 — Inspect structural alignment variables and saved reports

import os
import pandas as pd

print("========== ALIGNMENT VARIABLES ==========")

for name in [
    "pocket_xyz_by_pdb",
    "pocket_metadata_by_pdb",
    "two_chain_structural_superposition_df",
    "supported_target_pocket_pairs_df"
]:

    if name in globals():

        value = globals()[name]

        print(f"\n{name}:")
        print("Type:", type(value))

        if isinstance(value, dict):
            print("Keys:", list(value.keys())[:10])

        elif isinstance(value, pd.DataFrame):
            print("Shape:", value.shape)
            print("Columns:", value.columns.tolist())

        else:
            print(value)

print("\n========== SUPPORTED PAIR REPORT ==========")

supported_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "supported_target_pocket_pairs.csv"
)

if os.path.exists(supported_path):

    supported_pairs_df = pd.read_csv(supported_path)

    print("Shape:", supported_pairs_df.shape)
    print("Columns:", supported_pairs_df.columns.tolist())

    display(supported_pairs_df.head(10))

else:
    print("Supported pair CSV not found.")

========== ALIGNMENT VARIABLES ==========

pocket_xyz_by_pdb:
Type: <class 'dict'>
Keys: ['5KPP', '5KPQ', '5WRZ', '5WS1', '7AAC', '7ONS', '7ONT', '8JNZ', '9ETQ', '9ILN']

pocket_metadata_by_pdb:
Type: <class 'dict'>
Keys: ['5KPP', '5KPQ', '5WRZ', '5WS1', '7AAC', '7ONS', '7ONT', '8JNZ', '9ETQ', '9ILN']

========== SUPPORTED PAIR REPORT ==========
Shape: (10, 16)
Columns: ['pdb_id', 'comp_id', 'label_asym_id', 'auth_asym_id', 'ligand_heavy_atoms', 'minimum_distance_A', 'ligand_atoms_within_4A', 'ligand_atoms_within_5A', 'atom_pairs_within_4A', 'contacting_pocket_atoms_4A', 'contacting_residues_4A', 'M1_chain_A_contact_pairs_4A', 'M1_chain_B_contact_pairs_4A', 'close_to_pocket', 'strong_contact_evidence', 'pocket_occupancy_status']


,pdb_id,comp_id,label_asym_id,auth_asym_id,ligand_heavy_atoms,minimum_distance_A,ligand_atoms_within_4A,ligand_atoms_within_5A,atom_pairs_within_4A,contacting_pocket_atoms_4A,contacting_residues_4A,M1_chain_A_contact_pairs_4A,M1_chain_B_contact_pairs_4A,close_to_pocket,strong_contact_evidence,pocket_occupancy_status
0,5KPP,6WZ,C,A,34,2.456679,14,16,54,10,3,54,0,True,True,CONTACT_SUPPORTED
1,5KPQ,6X2,C,A,32,2.400476,14,16,55,12,3,55,0,True,True,CONTACT_SUPPORTED
2,5WRZ,7U3,C,A,19,2.804299,15,17,56,16,3,56,0,True,True,CONTACT_SUPPORTED
3,5WS1,7U9,C,A,20,2.565812,17,18,52,13,3,52,0,True,True,CONTACT_SUPPORTED
4,7AAC,78P,C,A,18,2.662797,15,17,49,13,3,49,0,True,True,CONTACT_SUPPORTED
5,7ONS,VKT,C,A,27,3.371240,12,14,34,8,2,34,0,True,True,CONTACT_SUPPORTED
6,7ONT,VKQ,C,A,30,3.454614,12,14,30,8,2,30,0,True,True,CONTACT_SUPPORTED
7,8JNZ,ERV,G,B,13,2.811423,12,13,40,10,3,40,0,True,True,CONTACT_SUPPORTED
8,9ETQ,A1H63,C,A,30,3.444867,11,14,27,8,2,27,0,True,True,CONTACT_SUPPORTED
9,9ILN,A1D9M,D,B,31,2.451098,12,14,47,9,2,47,0,True,True,CONTACT_SUPPORTED


In [96]:
# CELL 83A — Inspect Cell 45 transformation data

import numpy as np

print("========== POCKET COORDINATE DATA ==========")

for pdb_id, xyz in pocket_xyz_by_pdb.items():

    print(
        pdb_id,
        "| Shape:", np.asarray(xyz).shape,
        "| First coordinate:", np.asarray(xyz)[0]
    )

    if pdb_id == "5KPP":
        break

print("\n========== POSSIBLE TRANSFORMATION VARIABLES ==========")

keywords = [
    "rotation",
    "translation",
    "transform",
    "matrix",
    "aligned",
    "kabsch",
    "rmsd"
]

for name, value in globals().copy().items():

    if any(word in name.lower() for word in keywords):

        if isinstance(value, np.ndarray):
            print(name, "| Array shape:", value.shape)

        elif isinstance(value, dict):
            print(name, "| Dictionary keys:", list(value.keys())[:10])

        elif isinstance(value, pd.DataFrame):
            print(name, "| DataFrame shape:", value.shape)

print("\n========== POCKET METADATA SAMPLE ==========")

print(pocket_metadata_by_pdb["5KPP"][:5])

========== POCKET COORDINATE DATA ==========
5KPP | Shape: (98, 3) | First coordinate: [ 6.43023007 -5.02521556  0.76031491]

========== POSSIBLE TRANSFORMATION VARIABLES ==========
aligned_pairs | Array shape: (2, 5, 2)
distance_matrix | Array shape: (31, 98)
aligned | Array shape: (2, 350)
rotation_a | Array shape: (3, 3)
translation_a | Array shape: (3,)
transformed_b_after_a | Array shape: (241, 3)
rotation_b | Array shape: (3, 3)
translation_b | Array shape: (3,)
transformed_a_after_b | Array shape: (245, 3)
rotation | Array shape: (3, 3)
translation | Array shape: (3,)
transformed_a | Array shape: (10, 3)
transformed_b | Array shape: (1, 3)
pocket_rmsd_df | DataFrame shape: (46, 11)
transformed_pocket | Array shape: (98, 3)

========== POCKET METADATA SAMPLE ==========
  m1_chain  residue_number residue_name atom_name element
0        A             906          ASN         N       N
1        A             906          ASN        CA       C
2        A             906          ASN 

In [97]:
# CELL 83B — Inspect alignment report and transformation provenance

import os
import pandas as pd

report_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "target_identification",
    "two_chain_structural_superposition.csv"
)

if os.path.exists(report_path):

    alignment_report = pd.read_csv(report_path)

    print("Report shape:", alignment_report.shape)
    print("\nColumns:")
    print(alignment_report.columns.tolist())

    display(alignment_report.head(10))

else:
    print("Report not found at:", report_path)

print("\n========== CELL 45 VARIABLE CHECK ==========")

for name in [
    "rotation",
    "translation",
    "rotation_a",
    "translation_a",
    "rotation_b",
    "translation_b"
]:

    if name in globals():

        value = globals()[name]

        print(
            name,
            "| Shape:",
            np.asarray(value).shape,
            "| Values:\n",
            np.asarray(value)
        )

Report shape: (230, 10)

Columns:
['pdb_id', 'm1_A_experimental_chain', 'm1_B_experimental_chain', 'm1_A_independent_rmsd_A', 'm1_B_independent_rmsd_A', 'second_chain_rmsd_after_common_fit_A', 'two_chain_common_frame_rmsd_A', 'best_alignment_anchor', 'matched_ca_atoms_A', 'matched_ca_atoms_B']


,pdb_id,m1_A_experimental_chain,m1_B_experimental_chain,m1_A_independent_rmsd_A,m1_B_independent_rmsd_A,second_chain_rmsd_after_common_fit_A,two_chain_common_frame_rmsd_A,best_alignment_anchor,matched_ca_atoms_A,matched_ca_atoms_B
0,1UK0,A,B,1.349643,1.133073,52.705338,37.227140,A,350,348
1,1UK0,B,A,1.295065,1.031432,53.140313,37.533208,A,350,348
2,1UK1,A,B,1.345673,1.010776,60.080858,42.550389,B,350,348
3,1UK1,B,A,1.354698,0.999303,59.629804,42.115114,A,350,348
4,1WOK,A,B,1.198385,1.280066,40.588760,28.755884,B,350,348
5,1WOK,A,C,1.198385,1.334276,43.238336,30.542078,A,350,348
6,1WOK,A,D,1.198385,1.380435,43.852939,30.975879,A,350,348
7,1WOK,B,A,1.297986,1.181065,40.826322,28.921924,B,350,348
8,1WOK,B,C,1.297986,1.334276,44.747037,31.608936,A,350,348
9,1WOK,B,D,1.297986,1.380435,42.166318,29.787526,A,350,348



========== CELL 45 VARIABLE CHECK ==========
rotation | Shape: (3, 3) | Values:
 [[ 0.29564969 -0.72035318  0.62744128]
 [-0.66735014 -0.62570395 -0.40390389]
 [ 0.68354594 -0.29930896 -0.66571698]]
translation | Shape: (3,) | Values:
 [32.43805407 73.76134872 33.64331596]
rotation_a | Shape: (3, 3) | Values:
 [[ 0.2897768   0.22879981 -0.92934388]
 [ 0.66906653 -0.74275604  0.02575744]
 [-0.68438248 -0.6292568  -0.36831576]]
translation_a | Shape: (3,) | Values:
 [-20.36285804 215.85468084 192.7284108 ]
rotation_b | Shape: (3, 3) | Values:
 [[ 0.56681268  0.19102871 -0.80139342]
 [ 0.45769003 -0.88183595  0.11351294]
 [-0.6850133  -0.43113035 -0.58726774]]
translation_b | Shape: (3,) | Values:
 [-36.29637172 174.86449578 209.68105683]


In [98]:
# CELL 83C — Verify experimental chain assignments

import os
import pandas as pd

alignment_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "target_identification",
    "m1_experimental_chain_match_diagnostic.csv"
)

if not os.path.exists(alignment_path):
    raise FileNotFoundError(alignment_path)

chain_alignment_df = pd.read_csv(alignment_path)

target_ids = [
    "5KPP", "5KPQ", "5WRZ", "5WS1", "7AAC",
    "7ONS", "7ONT", "8JNZ", "9ETQ", "9ILN"
]

print("Available columns:")
print(chain_alignment_df.columns.tolist())

print("\n========== VALIDATED STRUCTURE MAPPINGS ==========")

target_mapping_df = chain_alignment_df[
    chain_alignment_df["pdb_id"].isin(target_ids)
].copy()

display(target_mapping_df)

Available columns:
['pdb_id', 'entity_id', 'label_asym_id', 'auth_asym_id', 'm1_chain', 'identity_pct', 'm1_coverage_pct', 'aligned_residues', 'alignment_score']

========== VALIDATED STRUCTURE MAPPINGS ==========


,pdb_id,entity_id,label_asym_id,auth_asym_id,m1_chain,identity_pct,m1_coverage_pct,aligned_residues,alignment_score
146,5KPP,1,A,A,A,100.0,100.0,350,1796.0
147,5KPP,1,A,A,B,100.0,100.0,348,1775.5
148,5KPP,1,B,B,A,100.0,100.0,350,1796.0
149,5KPP,1,B,B,B,100.0,100.0,348,1775.5
150,5KPQ,1,A,A,A,100.0,100.0,350,1796.0
151,5KPQ,1,A,A,B,100.0,100.0,348,1775.5
152,5KPQ,1,B,B,A,100.0,100.0,350,1796.0
153,5KPQ,1,B,B,B,100.0,100.0,348,1775.5
162,5WRZ,1,A,A,A,100.0,100.0,350,1785.5
163,5WRZ,1,A,A,B,100.0,100.0,348,1765.0


In [99]:
# CELL 83D — Select best two-chain structural assignments

import os
import pandas as pd

report_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "target_identification",
    "two_chain_structural_superposition.csv"
)

two_chain_df = pd.read_csv(report_path)

target_ids = [
    "5KPP", "5KPQ", "5WRZ", "5WS1", "7AAC",
    "7ONS", "7ONT", "8JNZ", "9ETQ", "9ILN"
]

target_df = two_chain_df[
    two_chain_df["pdb_id"].isin(target_ids)
].copy()

target_df = target_df.sort_values(
    "two_chain_common_frame_rmsd_A"
)

best_assignments_df = (
    target_df.groupby("pdb_id", as_index=False)
    .first()
    .sort_values("pdb_id")
)

print("========== BEST TWO-CHAIN STRUCTURAL ASSIGNMENTS ==========")

display(
    best_assignments_df[
        [
            "pdb_id",
            "m1_A_experimental_chain",
            "m1_B_experimental_chain",
            "two_chain_common_frame_rmsd_A",
            "matched_ca_atoms_A",
            "matched_ca_atoms_B"
        ]
    ]
)

print("\nStructures recovered:", len(best_assignments_df))

========== BEST TWO-CHAIN STRUCTURAL ASSIGNMENTS ==========


,pdb_id,m1_A_experimental_chain,m1_B_experimental_chain,two_chain_common_frame_rmsd_A,matched_ca_atoms_A,matched_ca_atoms_B
0,5KPP,A,B,1.005266,349,347
1,5KPQ,A,B,1.163180,349,347
2,5WRZ,A,B,1.738891,349,347
3,5WS1,A,B,1.670137,349,347
4,7AAC,A,B,1.000001,350,348
5,7ONS,A,B,0.594440,350,348
6,7ONT,A,B,1.125619,350,348
7,8JNZ,B,A,1.022110,350,348
8,9ETQ,A,B,1.086159,350,348
9,9ILN,B,A,1.198181,350,348



Structures recovered: 10


In [100]:
# CELL 83E — Find the original structural alignment code

from IPython import get_ipython

history = get_ipython().history_manager.input_hist_raw

keywords = [
    "pocket_xyz_by_pdb",
    "best_alignment_anchor",
    "two_chain_common_frame_rmsd_A"
]

matches = []

for cell_number, source in enumerate(history):

    if any(keyword in source for keyword in keywords):

        matches.append((cell_number, source))

print("Matching notebook cells:", len(matches))

for cell_number, source in matches:

    print("\n" + "=" * 70)
    print("NOTEBOOK INPUT CELL:", cell_number)
    print("=" * 70)
    print(source)

Matching notebook cells: 10

NOTEBOOK INPUT CELL: 41
# CELL 40 — Structural superposition and two-chain validation

import numpy as np
import pandas as pd
from pathlib import Path

from Bio.PDB import MMCIFParser
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner, substitution_matrices

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

if "m1_structure" not in globals():
    raise RuntimeError("M1 structure is missing. Run Cell 18 first.")

if "chains_df" not in globals():
    raise RuntimeError("chains_df is missing. Run Cell 37 first.")

if "m1_sequences" not in globals():
    raise RuntimeError("M1 sequences are missing. Run Cell 39 first.")

chain_ids = list(m1_sequences.keys())

if len(chain_ids) != 2:
    raise ValueError(
        f"Expected exactly two M1 chains, found {chain_ids}"
    )

M1_A, M1_B = chain_ids

# --------------------------------------------------
# 2. Build M1 C-alpha c

In [101]:
# CELL 83F — Reconstruct exact Cell 45 transformations

import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser

target_ids = [
    "5KPP", "5KPQ", "5WRZ", "5WS1", "7AAC",
    "7ONS", "7ONT", "8JNZ", "9ETQ", "9ILN"
]

# Use the exact chain assignments selected by Cell 45.
if "alignment_df" not in globals():
    raise RuntimeError("Cell 45 alignment_df is missing.")

if "m1_pocket_xyz" not in globals():
    raise RuntimeError("Original M1 pocket coordinates are missing.")

transformations_by_pdb = {}
transform_validation_rows = []

parser = MMCIFParser(
    QUIET=True,
    auth_chains=True,
    auth_residues=True
)

for pdb_id in target_ids:

    rows = alignment_df[
        alignment_df["pdb_id"].astype(str) == pdb_id
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"{pdb_id}: expected exactly one Cell 45 alignment row."
        )

    row = rows.iloc[0]

    if row["status"] != "PASS":
        raise RuntimeError(f"{pdb_id}: Cell 45 alignment did not pass.")

    exp_chain_a = str(row["experimental_chain_for_M1_A"])
    exp_chain_b = str(row["experimental_chain_for_M1_B"])

    structure = parser.get_structure(
        pdb_id,
        str(cif_paths[pdb_id])
    )

    model = structure[0]

    if exp_chain_a not in model or exp_chain_b not in model:
        raise RuntimeError(
            f"{pdb_id}: assigned experimental chains not found."
        )

    # Reproduce Cell 45 sequence and coordinate correspondence.
    result_a = align_protein_chains(
        m1_chains["A"],
        model[exp_chain_a]
    )

    result_b = align_protein_chains(
        m1_chains["B"],
        model[exp_chain_b]
    )

    reference_xyz = np.vstack([
        result_a["ref_xyz"],
        result_b["ref_xyz"]
    ])

    experimental_xyz = np.vstack([
        result_a["exp_xyz"],
        result_b["exp_xyz"]
    ])

    # Reproduce Cell 45 Kabsch calculation.
    R, t, fitted_rmsd = calculate_kabsch(
        reference_xyz,
        experimental_xyz
    )

    # Transform the original M1 pocket into experimental coordinates.
    reconstructed_pocket = m1_pocket_xyz @ R + t

    # Compare against the pocket coordinates saved by Cell 45.
    stored_pocket = np.asarray(
        pocket_xyz_by_pdb[pdb_id],
        dtype=float
    )

    if reconstructed_pocket.shape != stored_pocket.shape:
        raise RuntimeError(
            f"{pdb_id}: reconstructed and stored pocket shapes differ."
        )

    max_difference = float(
        np.max(np.abs(reconstructed_pocket - stored_pocket))
    )

    mean_difference = float(
        np.mean(
            np.linalg.norm(
                reconstructed_pocket - stored_pocket,
                axis=1
            )
        )
    )

    # Verify rotation matrix is a proper rotation.
    determinant = float(np.linalg.det(R))

    orthogonality_error = float(
        np.max(np.abs(R @ R.T - np.eye(3)))
    )

    transformations_by_pdb[pdb_id] = {
        "rotation": R,
        "translation": t,
        "experimental_chain_for_M1_A": exp_chain_a,
        "experimental_chain_for_M1_B": exp_chain_b,
        "fitted_rmsd_A": float(fitted_rmsd)
    }

    transform_validation_rows.append({
        "pdb_id": pdb_id,
        "M1_A_experimental_chain": exp_chain_a,
        "M1_B_experimental_chain": exp_chain_b,
        "fitted_two_chain_RMSD_A": fitted_rmsd,
        "stored_pocket_max_difference_A": max_difference,
        "stored_pocket_mean_difference_A": mean_difference,
        "rotation_determinant": determinant,
        "rotation_orthogonality_error": orthogonality_error,
        "validation": (
            "PASS"
            if max_difference < 1e-4
            and abs(determinant - 1.0) < 1e-5
            and orthogonality_error < 1e-5
            else "CHECK"
        )
    })

transform_validation_df = pd.DataFrame(
    transform_validation_rows
)

print("========== TRANSFORMATION VALIDATION ==========")

display(transform_validation_df)

print(
    "\nValidated transformations:",
    sum(transform_validation_df["validation"] == "PASS"),
    "/",
    len(target_ids)
)

if not (transform_validation_df["validation"] == "PASS").all():
    raise RuntimeError(
        "At least one transformation failed verification. "
        "Do not transform ligand coordinates yet."
    )

print("\nAll transformations reconstructed and verified.")

========== TRANSFORMATION VALIDATION ==========


,pdb_id,M1_A_experimental_chain,M1_B_experimental_chain,fitted_two_chain_RMSD_A,stored_pocket_max_difference_A,stored_pocket_mean_difference_A,rotation_determinant,rotation_orthogonality_error,validation
0,5KPP,A,B,0.797784,0.0,0.0,1.0,1.332268e-15,PASS
1,5KPQ,A,B,0.838873,0.0,0.0,1.0,3.799052e-16,PASS
2,5WRZ,A,B,1.245704,0.0,0.0,1.0,1.332268e-15,PASS
3,5WS1,A,B,1.234568,0.0,0.0,1.0,6.661338e-16,PASS
4,7AAC,A,B,0.741034,0.0,0.0,1.0,1.332268e-15,PASS
5,7ONS,A,B,0.499997,0.0,0.0,1.0,8.881784e-16,PASS
6,7ONT,A,B,0.748998,0.0,0.0,1.0,1.110223e-15,PASS
7,8JNZ,B,A,0.792956,0.0,0.0,1.0,1.110223e-15,PASS
8,9ETQ,A,B,0.755967,0.0,0.0,1.0,8.881784e-16,PASS
9,9ILN,B,A,0.817862,0.0,0.0,1.0,4.440892e-16,PASS



Validated transformations: 10 / 10

All transformations reconstructed and verified.


In [102]:
# CELL 83G — Transform experimental ligand poses into M1 coordinates

import os
import numpy as np
import pandas as pd

from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from scipy.spatial.distance import cdist
from rdkit import Chem
from rdkit.Geometry import Point3D

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

supported_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "supported_target_pocket_pairs.csv"
)

supported_pairs_df = pd.read_csv(supported_path)

pose_output_dir = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "experimental_ligand_poses_m1_frame"
)

os.makedirs(pose_output_dir, exist_ok=True)

# Original M1 pocket coordinates
reference_pocket = np.asarray(
    m1_pocket_xyz,
    dtype=float
)

pose_records = []
pose_validation_records = []

# --------------------------------------------------
# 2. Process supported ligand instances
# --------------------------------------------------

for _, ligand_row in supported_pairs_df.iterrows():

    pdb_id = str(ligand_row["pdb_id"])
    comp_id = str(ligand_row["comp_id"])
    label_chain = str(ligand_row["label_asym_id"])

    print("Processing:", pdb_id, comp_id)

    if pdb_id not in transformations_by_pdb:
        raise RuntimeError(
            f"Missing verified transformation for {pdb_id}"
        )

    # Load experimental structure coordinates
    cif_path = cif_paths[pdb_id]
    cif_dict = MMCIF2Dict(str(cif_path))

    atom_df = extract_atom_table(cif_dict)

    atom_df["element"] = (
        atom_df["element"].astype(str).str.upper()
    )

    for col in ["x", "y", "z"]:
        atom_df[col] = pd.to_numeric(
            atom_df[col],
            errors="coerce"
        )

    atom_df = atom_df.dropna(
        subset=["x", "y", "z"]
    )

    # Extract exact ligand instance
    ligand_atoms = atom_df[
        (atom_df["comp"].astype(str) == comp_id)
        & (atom_df["label_chain"].astype(str) == label_chain)
        & (atom_df["group"].astype(str).str.upper() == "HETATM")
        & (~atom_df["element"].isin(["H", "D"]))
    ].copy()

    if ligand_atoms.empty:
        raise RuntimeError(
            f"No ligand heavy atoms found: {pdb_id} {comp_id}"
        )

    experimental_xyz = ligand_atoms[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    # --------------------------------------------------
    # 3. Inverse transformation: experimental -> M1
    # --------------------------------------------------

    R = transformations_by_pdb[pdb_id]["rotation"]
    t = transformations_by_pdb[pdb_id]["translation"]

    ligand_m1_xyz = (
        experimental_xyz - t
    ) @ R.T

    # --------------------------------------------------
    # 4. Verify distances are preserved
    # --------------------------------------------------

    experimental_pocket = pocket_xyz_by_pdb[pdb_id]

    original_distances = cdist(
        experimental_xyz,
        experimental_pocket
    )

    transformed_distances = cdist(
        ligand_m1_xyz,
        reference_pocket
    )

    distance_difference = float(
        np.max(
            np.abs(
                original_distances - transformed_distances
            )
        )
    )

    # Verify ligand remains near the M1 pocket.
    minimum_distance = float(
        transformed_distances.min()
    )

    ligand_atoms_within_4A = int(
        (transformed_distances <= 4.0)
        .any(axis=1)
        .sum()
    )

    # --------------------------------------------------
    # 5. Create an RDKit molecule with transformed pose
    # --------------------------------------------------

    smiles = str(
        ligand_row.get("canonical_smiles", "")
    )

    mol = None

    # Prefer validated target-ligand graph information if available.
    if "validated_ligands" in globals():

        matching = [
            item for item in validated_ligands
            if str(item.get("pdb_id", "")).upper() == pdb_id
            and str(item.get("comp_id", "")) == comp_id
        ]

        if matching and matching[0].get("smiles"):
            smiles = matching[0]["smiles"]

    if smiles:
        mol = Chem.MolFromSmiles(smiles)

    # If the report does not contain SMILES, use the existing
    # validated target-specific ligand graph file.
    if mol is None:

        graph_path = os.path.join(
            PROJECT_ROOT,
            "member2_outputs",
            "validated_target_ligand_graphs.jsonl"
        )

        if os.path.exists(graph_path):

            import json

            with open(graph_path, "r") as f:

                for line in f:

                    record = json.loads(line)

                    if (
                        str(record.get("pdb_id", "")).upper() == pdb_id
                        and str(record.get("comp_id", "")) == comp_id
                    ):

                        candidate_smiles = (
                            record.get("canonical_smiles")
                            or record.get("smiles")
                        )

                        if candidate_smiles:
                            mol = Chem.MolFromSmiles(
                                candidate_smiles
                            )
                            break

    if mol is None:
        raise RuntimeError(
            f"Could not recover validated molecular graph "
            f"for {pdb_id} {comp_id}"
        )

    mol = Chem.AddHs(mol)

    # Use the experimental heavy-atom coordinates.
    # Atom ordering must match the original component graph.
    # Therefore, verify atom counts before assigning coordinates.
    heavy_atom_count = mol.GetNumHeavyAtoms()

    if heavy_atom_count != len(ligand_m1_xyz):
        raise RuntimeError(
            f"{pdb_id} {comp_id}: graph has {heavy_atom_count} "
            f"heavy atoms but coordinate instance has "
            f"{len(ligand_m1_xyz)}. Atom correspondence required."
        )

    conformer = Chem.Conformer(mol.GetNumAtoms())

    for atom_index, xyz in enumerate(ligand_m1_xyz):

        conformer.SetAtomPosition(
            atom_index,
            Point3D(
                float(xyz[0]),
                float(xyz[1]),
                float(xyz[2])
            )
        )

    # Added hydrogens do not have experimental coordinates.
    # Keep the heavy-atom pose as the primary reference.
    mol.RemoveAllConformers()
    mol.AddConformer(conformer, assignId=True)

    sdf_path = os.path.join(
        pose_output_dir,
        f"{pdb_id}_{comp_id}_M1_frame.sdf"
    )

    writer = Chem.SDWriter(sdf_path)
    writer.write(mol)
    writer.close()

    # --------------------------------------------------
    # 6. Save transformed coordinates
    # --------------------------------------------------

    for atom_index, (_, atom_row) in enumerate(
        ligand_atoms.iterrows()
    ):

        pose_records.append({

            "pdb_id": pdb_id,
            "comp_id": comp_id,
            "label_asym_id": label_chain,

            "atom_index": atom_index,
            "atom_name": atom_row["atom"],

            "element": atom_row["element"],

            "x_M1": ligand_m1_xyz[atom_index, 0],
            "y_M1": ligand_m1_xyz[atom_index, 1],
            "z_M1": ligand_m1_xyz[atom_index, 2]
        })

    pose_validation_records.append({

        "pdb_id": pdb_id,
        "comp_id": comp_id,

        "heavy_atom_count": len(ligand_m1_xyz),

        "minimum_distance_to_M1_pocket_A": minimum_distance,

        "ligand_atoms_within_4A": ligand_atoms_within_4A,

        "distance_preservation_error_A": distance_difference,

        "sdf_exists": os.path.exists(sdf_path),

        "status": (
            "PASS"
            if distance_difference < 1e-5
            and minimum_distance <= 5.0
            else "CHECK"
        )
    })


# --------------------------------------------------
# 7. Export reports
# --------------------------------------------------

pose_coordinates_df = pd.DataFrame(pose_records)

pose_validation_df = pd.DataFrame(
    pose_validation_records
)

pose_coordinates_df.to_csv(
    os.path.join(
        pose_output_dir,
        "all_experimental_ligand_pose_coordinates_M1.csv"
    ),
    index=False
)

pose_validation_df.to_csv(
    os.path.join(
        pose_output_dir,
        "ligand_pose_transformation_validation.csv"
    ),
    index=False
)

print("\n========== LIGAND POSE TRANSFORMATION SUMMARY ==========")

display(pose_validation_df)

print("\nTotal ligands:", len(pose_validation_df))
print(
    "Passed:",
    (pose_validation_df["status"] == "PASS").sum()
)

print("Output directory:", pose_output_dir)

Processing: 5KPP 6WZ


AttributeError: 'str' object has no attribute 'get'

In [103]:
# CELL 83G-A — Inspect validated ligand graph structure

import os
import json

print("validated_ligands type:", type(validated_ligands))

if isinstance(validated_ligands, (list, tuple)):
    print("Number of entries:", len(validated_ligands))

    for i, item in enumerate(validated_ligands[:3]):
        print(f"\nEntry {i}:")
        print("Type:", type(item))
        print("Value:", str(item)[:500])

graph_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "validated_target_ligand_graphs.jsonl"
)

print("\n========== GRAPH FILE INSPECTION ==========")

with open(graph_path, "r") as f:
    for i in range(2):
        line = f.readline()
        if not line:
            break

        record = json.loads(line)

        print(f"\nRecord {i} keys:")
        print(list(record.keys()))

        print("\nRecord preview:")
        print(json.dumps(record, indent=2)[:1500])

validated_ligands type: <class 'dict'>

========== GRAPH FILE INSPECTION ==========

Record 0 keys:
['graph_id', 'comp_id', 'source', 'source_pdb_ids', 'canonical_smiles', 'smiles_descriptor_type', 'smiles_descriptor_program', 'molecular_formula', 'molecular_name', 'molecular_weight', 'validated_by_rdkit', 'num_atoms', 'num_bonds', 'node_features', 'bonds', 'edge_index', 'edge_features']

Record preview:
{
  "graph_id": "CCD_6WZ",
  "comp_id": "6WZ",
  "source": "RCSB_Chemical_Component_Dictionary",
  "source_pdb_ids": [
    "5KPP"
  ],
  "canonical_smiles": "C[C@@H]1CN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc43)ccc2F)CCN1CC(F)(F)F",
  "smiles_descriptor_type": "SMILES_CANONICAL",
  "smiles_descriptor_program": "OPENEYE OETOOLKITS",
  "molecular_formula": "C23 H22 F4 N4 O3",
  "molecular_name": "1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris(fluoranyl)ethyl]piperazin-1-yl]carbonyl-phenyl]methyl]quinazoline-2,4-dione",
  "molecular_weight": 478.4460000000002,
  "validated_by_rdkit": true,
  "

In [104]:
# CELL 83G-B — Inspect CCD graph and experimental atom records

import json
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

# 1. Inspect validated_ligands dictionary
print("========== VALIDATED LIGANDS ==========")
print("Dictionary keys:", list(validated_ligands.keys())[:15])

print("\n6WZ entry:")
print(str(validated_ligands.get("6WZ"))[:1500])

# 2. Load the 6WZ molecular graph
graph_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "validated_target_ligand_graphs.jsonl"
)

graph_6wz = None

with open(graph_path, "r") as f:
    for line in f:
        record = json.loads(line)

        if record["comp_id"] == "6WZ":
            graph_6wz = record
            break

print("\n========== GRAPH ATOM INFORMATION ==========")

print("Graph atom count:", graph_6wz["num_atoms"])
print("First 10 graph atoms:")

for atom in graph_6wz["node_features"][:10]:
    print(atom)

# 3. Inspect experimental 5KPP mmCIF atom records
cif_dict = MMCIF2Dict(str(cif_paths["5KPP"]))

print("\n========== MMCIF ATOM FIELDS ==========")

for key in cif_dict:
    if key.startswith("_atom_site."):
        print(key)

# 4. Inspect component atom definitions
print("\n========== 6WZ COMPONENT ATOMS ==========")

for key in cif_dict:
    if key.startswith("_chem_comp_atom."):
        print(key)

# 5. Print experimental 6WZ heavy atoms
print("\n========== EXPERIMENTAL 6WZ ATOMS ==========")

atom_df = extract_atom_table(cif_dict)

ligand_6wz = atom_df[
    (atom_df["comp"].astype(str) == "6WZ")
    & (atom_df["label_chain"].astype(str) == "C")
    & (atom_df["group"].astype(str).str.upper() == "HETATM")
].copy()

print("Experimental atom count:", len(ligand_6wz))
print(ligand_6wz.head(15).to_string(index=False))

========== VALIDATED LIGANDS ==========
Dictionary keys: ['5KPP', '5KPQ', '5WRZ', '5WS1', '7AAC', '7ONS', '7ONT', '8JNZ', '9ETQ', '9ILN']

6WZ entry:
None

========== GRAPH ATOM INFORMATION ==========
Graph atom count: 34
First 10 graph atoms:
{'atom_index': 0, 'atomic_number': 6, 'formal_charge': 0, 'degree': 1, 'total_valence': 4, 'aromatic': False, 'in_ring': False, 'hybridization': 'SP3', 'total_hydrogens': 3}
{'atom_index': 1, 'atomic_number': 6, 'formal_charge': 0, 'degree': 3, 'total_valence': 4, 'aromatic': False, 'in_ring': True, 'hybridization': 'SP3', 'total_hydrogens': 1}
{'atom_index': 2, 'atomic_number': 6, 'formal_charge': 0, 'degree': 2, 'total_valence': 4, 'aromatic': False, 'in_ring': True, 'hybridization': 'SP3', 'total_hydrogens': 2}
{'atom_index': 3, 'atomic_number': 7, 'formal_charge': 0, 'degree': 3, 'total_valence': 3, 'aromatic': False, 'in_ring': True, 'hybridization': 'SP2', 'total_hydrogens': 0}
{'atom_index': 4, 'atomic_number': 6, 'formal_charge': 0, 'degr

In [105]:
# CELL 83G-C — Inspect CCD connectivity for 6WZ

from Bio.PDB.MMCIF2Dict import MMCIF2Dict
import pandas as pd

cif_dict = MMCIF2Dict(str(cif_paths["5KPP"]))

def get_cif_column(data, key):
    value = data.get(key, [])
    if isinstance(value, str):
        return [value]
    return list(value)

# CCD atom definitions
ccd_atoms = pd.DataFrame({
    "atom_id": get_cif_column(
        cif_dict, "_chem_comp_atom.atom_id"
    ),
    "element": get_cif_column(
        cif_dict, "_chem_comp_atom.type_symbol"
    ),
    "ordinal": get_cif_column(
        cif_dict, "_chem_comp_atom.pdbx_ordinal"
    )
})

ccd_atoms = ccd_atoms[
    ccd_atoms["atom_id"].notna()
].copy()

# CCD bond definitions
ccd_bonds = pd.DataFrame({
    "atom_1": get_cif_column(
        cif_dict, "_chem_comp_bond.atom_id_1"
    ),
    "atom_2": get_cif_column(
        cif_dict, "_chem_comp_bond.atom_id_2"
    ),
    "bond_order": get_cif_column(
        cif_dict, "_chem_comp_bond.value_order"
    )
})

ccd_bonds = ccd_bonds[
    ccd_bonds["atom_1"].notna()
].copy()

# Experimental heavy atoms
experimental_atoms = ligand_6wz[
    ["atom", "element", "x", "y", "z"]
].copy()

print("========== CCD ATOMS ==========")
print("CCD atom count:", len(ccd_atoms))
display(ccd_atoms)

print("\n========== CCD BONDS ==========")
print("CCD bond count:", len(ccd_bonds))
display(ccd_bonds.head(20))

print("\n========== ATOM NAME COMPARISON ==========")

ccd_atom_names = set(ccd_atoms["atom_id"].astype(str))
experimental_atom_names = set(
    experimental_atoms["atom"].astype(str)
)

print("CCD names:", len(ccd_atom_names))
print("Experimental names:", len(experimental_atom_names))

print(
    "Names present in both:",
    len(ccd_atom_names & experimental_atom_names)
)

print(
    "CCD-only names:",
    sorted(ccd_atom_names - experimental_atom_names)
)

print(
    "Experimental-only names:",
    sorted(experimental_atom_names - ccd_atom_names)
)

========== CCD ATOMS ==========
CCD atom count: 446


,atom_id,element,ordinal
0,OAD,O,1
1,CBA,C,2
2,NAU,N,3
3,CAZ,C,4
4,OAC,O,5
...,...,...,...
441,HG13,H,442
442,HG21,H,443
443,HG22,H,444
444,HG23,H,445



========== CCD BONDS ==========
CCD bond count: 434


,atom_1,atom_2,bond_order
0,FAE,CAX,sing
1,FAH,CBH,sing
2,CAX,CAL,doub
3,CAX,CAY,sing
4,CAL,CAK,sing
5,CAT,CBH,sing
6,CAT,NBF,sing
7,CBH,FAF,sing
8,CBH,FAG,sing
9,CAP,NBE,sing



========== ATOM NAME COMPARISON ==========
CCD names: 137
Experimental names: 34
Names present in both: 34
CCD-only names: ['C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3', 'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3', 'H', 'H1', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H2', 'H20', 'H22', 'H23', 'H3', 'H4', 'H5', 'H6', 'H7', 'H8', 'H9', 'HA', 'HA2', 'HA3', 'HB', 'HB1', 'HB2', 'HB3', 'HD1', 'HD11', 'HD12', 'HD13', 'HD2', 'HD21', 'HD22', 'HD23', 'HD3', 'HE', 'HE1', 'HE2', 'HE21', 'HE22', 'HE3', 'HG', 'HG1', 'HG11', 'HG12', 'HG13', 'HG2', 'HG21', 'HG22', 'HG23', 'HG3', 'HH', 'HH11', 'HH12', 'HH2', 'HH21', 'HH22', 'HXT', 'HZ', 'HZ1', 'HZ2', 'HZ3', 'N', 'ND1', 'ND2', 'NE', 'NE1', 'NE2', 'NH1', 'NH2', 'NZ', 'O', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1', 'OH', 'OXT', 'SD', 'SG']
Experimental-only names: []


In [106]:
# CELL 83G-D — Filter CCD records specifically for 6WZ

import pandas as pd

def cif_list(data, key):
    value = data.get(key, [])
    return [value] if isinstance(value, str) else list(value)

# CCD atom table
ccd_atoms = pd.DataFrame({
    "comp_id": cif_list(cif_dict, "_chem_comp_atom.comp_id"),
    "atom_id": cif_list(cif_dict, "_chem_comp_atom.atom_id"),
    "element": cif_list(cif_dict, "_chem_comp_atom.type_symbol"),
    "ordinal": cif_list(cif_dict, "_chem_comp_atom.pdbx_ordinal")
})

# CCD bond table
ccd_bonds = pd.DataFrame({
    "comp_id": cif_list(cif_dict, "_chem_comp_bond.comp_id"),
    "atom_1": cif_list(cif_dict, "_chem_comp_bond.atom_id_1"),
    "atom_2": cif_list(cif_dict, "_chem_comp_bond.atom_id_2"),
    "bond_order": cif_list(cif_dict, "_chem_comp_bond.value_order")
})

# Filter exact component
ccd_atoms = ccd_atoms[
    ccd_atoms["comp_id"].astype(str) == "6WZ"
].copy()

ccd_bonds = ccd_bonds[
    ccd_bonds["comp_id"].astype(str) == "6WZ"
].copy()

# Experimental ligand atoms
experimental_atoms = ligand_6wz[
    ["atom", "element", "x", "y", "z"]
].copy()

# Normalize names and elements
ccd_atoms["atom_id"] = ccd_atoms["atom_id"].astype(str)
ccd_atoms["element"] = ccd_atoms["element"].astype(str).str.upper()

experimental_atoms["atom"] = (
    experimental_atoms["atom"].astype(str)
)
experimental_atoms["element"] = (
    experimental_atoms["element"].astype(str).str.upper()
)

# Heavy atoms only
ccd_heavy = ccd_atoms[
    ~ccd_atoms["element"].isin(["H", "D"])
].copy()

# Compare names
ccd_names = set(ccd_heavy["atom_id"])
experimental_names = set(experimental_atoms["atom"])

print("========== FILTERED 6WZ CCD ==========")
print("CCD total atoms:", len(ccd_atoms))
print("CCD heavy atoms:", len(ccd_heavy))
print("CCD bonds:", len(ccd_bonds))
print("Experimental heavy atoms:", len(experimental_atoms))

print("\n========== ATOM NAME MATCH ==========")
print("Matching names:", len(ccd_names & experimental_names))
print("CCD-only:", sorted(ccd_names - experimental_names))
print("Experimental-only:", sorted(experimental_names - ccd_names))

# Check element correspondence
merged = experimental_atoms.merge(
    ccd_heavy,
    left_on="atom",
    right_on="atom_id",
    how="outer",
    indicator=True,
    suffixes=("_experimental", "_CCD")
)

merged["element_match"] = (
    merged["element_experimental"]
    == merged["element_CCD"]
)

display(merged.head(40))

print("\nElement mismatches:")
display(
    merged[
        (merged["_merge"] != "both")
        | (~merged["element_match"])
    ]
)

if (
    len(ccd_heavy) != len(experimental_atoms)
    or not (merged["_merge"] == "both").all()
    or not merged["element_match"].all()
):
    raise RuntimeError("6WZ atom correspondence validation failed.")

print("\n6WZ CCD-to-experimental atom mapping: PASS")

========== FILTERED 6WZ CCD ==========
CCD total atoms: 56
CCD heavy atoms: 34
CCD bonds: 59
Experimental heavy atoms: 34

========== ATOM NAME MATCH ==========
Matching names: 34
CCD-only: []
Experimental-only: []


,atom,element_experimental,x,y,z,comp_id,atom_id,element_CCD,ordinal,_merge,element_match
0,CAA,C,16.353,-2.788,-7.965,6WZ,CAA,C,26,both,True
1,CAI,C,6.358,-2.272,-7.560,6WZ,CAI,C,8,both,True
2,CAJ,C,6.627,-3.387,-8.336,6WZ,CAJ,C,9,both,True
3,CAK,C,10.425,-6.344,-9.912,6WZ,CAK,C,16,both,True
4,CAL,C,11.090,-6.114,-11.115,6WZ,CAL,C,17,both,True
5,CAM,C,7.215,-1.950,-6.512,6WZ,CAM,C,7,both,True
6,CAN,C,7.746,-4.172,-8.079,6WZ,CAN,C,10,both,True
7,CAO,C,11.772,-4.637,-8.853,6WZ,CAO,C,15,both,True
8,CAP,C,15.243,-5.086,-10.592,6WZ,CAP,C,27,both,True
9,CAQ,C,16.439,-5.347,-9.662,6WZ,CAQ,C,28,both,True



Element mismatches:


,atom,element_experimental,x,y,z,comp_id,atom_id,element_CCD,ordinal,_merge,element_match



6WZ CCD-to-experimental atom mapping: PASS


In [107]:
# CELL 83G-E — Inspect CCD chemical graph information for 6WZ

from rdkit import Chem
import pandas as pd

print("========== CCD ATOM PROPERTIES ==========")

for key in [
    "_chem_comp_atom.pdbx_aromatic_flag",
    "_chem_comp_atom.pdbx_stereo_config",
    "_chem_comp_atom.pdbx_leaving_atom_flag"
]:
    values = cif_dict.get(key, [])
    print(key, ":", set(values) if values else "Not available")


print("\n========== CCD BOND PROPERTIES ==========")

for key in [
    "_chem_comp_bond.pdbx_aromatic_flag",
    "_chem_comp_bond.pdbx_stereo_config",
    "_chem_comp_bond.value_order"
]:
    values = cif_dict.get(key, [])
    print(key, ":", set(values) if values else "Not available")


print("\n========== RDKit REFERENCE GRAPH ==========")

reference_smiles = graph_6wz["canonical_smiles"]

reference_mol = Chem.MolFromSmiles(reference_smiles)

if reference_mol is None:
    raise RuntimeError("Could not construct RDKit reference molecule.")

print("Canonical SMILES:", reference_smiles)
print("RDKit atoms:", reference_mol.GetNumAtoms())
print("RDKit bonds:", reference_mol.GetNumBonds())

print("\nAtom properties:")

for atom in reference_mol.GetAtoms():
    print(
        atom.GetIdx(),
        atom.GetSymbol(),
        "aromatic:", atom.GetIsAromatic(),
        "charge:", atom.GetFormalCharge(),
        "chiral:", str(atom.GetChiralTag())
    )

print("\nBond types:")

for bond in reference_mol.GetBonds():
    print(
        bond.GetBeginAtomIdx(),
        "-",
        bond.GetEndAtomIdx(),
        bond.GetBondType(),
        "aromatic:", bond.GetIsAromatic()
    )

========== CCD ATOM PROPERTIES ==========
_chem_comp_atom.pdbx_aromatic_flag : {'N', 'Y'}
_chem_comp_atom.pdbx_stereo_config : {'S', 'N', 'R'}
_chem_comp_atom.pdbx_leaving_atom_flag : Not available

========== CCD BOND PROPERTIES ==========
_chem_comp_bond.pdbx_aromatic_flag : {'N', 'Y'}
_chem_comp_bond.pdbx_stereo_config : {'N'}
_chem_comp_bond.value_order : {'sing', 'doub'}

========== RDKit REFERENCE GRAPH ==========
Canonical SMILES: C[C@@H]1CN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc43)ccc2F)CCN1CC(F)(F)F
RDKit atoms: 34
RDKit bonds: 37

Atom properties:
0 C aromatic: False charge: 0 chiral: CHI_UNSPECIFIED
1 C aromatic: False charge: 0 chiral: CHI_TETRAHEDRAL_CCW
2 C aromatic: False charge: 0 chiral: CHI_UNSPECIFIED
3 N aromatic: False charge: 0 chiral: CHI_UNSPECIFIED
4 C aromatic: False charge: 0 chiral: CHI_UNSPECIFIED
5 O aromatic: False charge: 0 chiral: CHI_UNSPECIFIED
6 C aromatic: True charge: 0 chiral: CHI_UNSPECIFIED
7 C aromatic: True charge: 0 chiral: CHI_UNSPECIFIED
8 C ar

In [108]:
# CELL 83G-F — CCD to RDKit graph correspondence

from rdkit import Chem
from rdkit.Chem import rdchem
import pandas as pd

# --------------------------------------------------
# 1. Prepare CCD heavy-atom graph
# --------------------------------------------------

ccd_heavy_atoms = ccd_atoms[
    ~ccd_atoms["element"].astype(str).str.upper().isin(["H", "D"])
].copy().reset_index(drop=True)

atom_id_to_index = {
    row["atom_id"]: idx
    for idx, row in ccd_heavy_atoms.iterrows()
}

ccd_bond_builder = Chem.RWMol()

for _, row in ccd_heavy_atoms.iterrows():

    atom = Chem.Atom(str(row["element"]).upper())

    ccd_bond_builder.AddAtom(atom)

# Filter bonds to heavy-atom pairs
ccd_heavy_bonds = ccd_bonds[
    ccd_bonds["atom_1"].isin(atom_id_to_index)
    & ccd_bonds["atom_2"].isin(atom_id_to_index)
].copy()

for _, row in ccd_heavy_bonds.iterrows():

    i = atom_id_to_index[row["atom_1"]]
    j = atom_id_to_index[row["atom_2"]]

    aromatic_flags = cif_list(
        cif_dict,
        "_chem_comp_bond.pdbx_aromatic_flag"
    )

    # Use bond table row index to retrieve corresponding flag
    original_bond_index = row.name

    aromatic = (
        str(aromatic_flags[original_bond_index]).upper() == "Y"
    )

    if aromatic:
        bond_type = Chem.BondType.AROMATIC

    else:
        order = str(row["bond_order"]).lower()

        bond_type = {
            "sing": Chem.BondType.SINGLE,
            "doub": Chem.BondType.DOUBLE,
            "trip": Chem.BondType.TRIPLE
        }.get(order)

        if bond_type is None:
            raise RuntimeError(
                f"Unsupported CCD bond order: {order}"
            )

    ccd_bond_builder.AddBond(i, j, bond_type)

ccd_mol = ccd_bond_builder.GetMol()

# Apply aromatic atom flags
aromatic_atom_flags = cif_list(
    cif_dict,
    "_chem_comp_atom.pdbx_aromatic_flag"
)

for idx, row in ccd_heavy_atoms.iterrows():

    original_index = row.name

    # Re-indexing means use atom_id to find original CCD row
    original_row = ccd_atoms[
        ccd_atoms["atom_id"] == row["atom_id"]
    ].iloc[0]

    atom = ccd_mol.GetAtomWithIdx(idx)

    atom["ccd_aromatic"] = str(
        original_row.get("aromatic_flag", "")
    )

# Set aromaticity from CCD source table
ccd_atom_source = pd.DataFrame({
    "atom_id": cif_list(cif_dict, "_chem_comp_atom.atom_id"),
    "aromatic": cif_list(
        cif_dict,
        "_chem_comp_atom.pdbx_aromatic_flag"
    )
})

for idx, row in ccd_heavy_atoms.iterrows():

    source_row = ccd_atom_source[
        ccd_atom_source["atom_id"] == row["atom_id"]
    ].iloc[0]

    if str(source_row["aromatic"]).upper() == "Y":
        ccd_mol.GetAtomWithIdx(idx).SetIsAromatic(True)

# Sanitize CCD graph
Chem.SanitizeMol(ccd_mol)

# --------------------------------------------------
# 2. Compare graph sizes
# --------------------------------------------------

print("========== HEAVY-ATOM GRAPH COUNTS ==========")

print("CCD atoms:", ccd_mol.GetNumAtoms())
print("CCD bonds:", ccd_mol.GetNumBonds())

print("RDKit atoms:", reference_mol.GetNumAtoms())
print("RDKit bonds:", reference_mol.GetNumBonds())

# --------------------------------------------------
# 3. Find exact graph isomorphisms
# --------------------------------------------------

matches = reference_mol.GetSubstructMatches(
    ccd_mol,
    uniquify=False,
    useChirality=False,
    maxMatches=10000
)

print("\n========== GRAPH MATCHING ==========")
print("Number of graph mappings:", len(matches))

if len(matches) == 0:
    raise RuntimeError(
        "CCD and RDKit molecular graphs do not match."
    )

# Each match maps CCD atom index -> RDKit atom index
mapping = matches[0]

ccd_to_rdkit = {}

for ccd_index, rdkit_index in enumerate(mapping):

    atom_name = ccd_heavy_atoms.iloc[ccd_index]["atom_id"]

    ccd_to_rdkit[atom_name] = rdkit_index

mapping_df = pd.DataFrame([
    {
        "ccd_atom_name": atom_name,
        "rdkit_atom_index": rdkit_index,
        "ccd_element": ccd_heavy_atoms.iloc[
            ccd_index
        ]["element"],
        "rdkit_element": reference_mol.GetAtomWithIdx(
            rdkit_index
        ).GetSymbol()
    }
    for ccd_index, (atom_name, rdkit_index)
    in enumerate(ccd_to_rdkit.items())
])

print("\n========== FIRST GRAPH MAPPING ==========")
display(mapping_df)

print("\nElement mapping valid:",
      (mapping_df["ccd_element"].astype(str).str.upper()
       == mapping_df["rdkit_element"]).all())

print("\nUnique RDKit atom indices:",
      mapping_df["rdkit_atom_index"].nunique())

print("\nMapping count:", len(mapping_df))

if len(mapping_df) != 34:
    raise RuntimeError("Incomplete CCD-to-RDKit mapping.")

if mapping_df["rdkit_atom_index"].nunique() != 34:
    raise RuntimeError("Mapping is not one-to-one.")

print("\nInitial graph correspondence: PASS")
print("Stereochemical equivalence still requires separate validation.")

TypeError: 'Atom' object does not support item assignment

In [109]:
# CELL 83G-F — Build CCD graph and map it to RDKit

from rdkit import Chem
import pandas as pd

# --------------------------------------------------
# 1. Read CCD atom information
# --------------------------------------------------

ccd_atom_table = pd.DataFrame({
    "comp_id": cif_list(cif_dict, "_chem_comp_atom.comp_id"),
    "atom_id": cif_list(cif_dict, "_chem_comp_atom.atom_id"),
    "element": cif_list(cif_dict, "_chem_comp_atom.type_symbol"),
    "aromatic": cif_list(
        cif_dict, "_chem_comp_atom.pdbx_aromatic_flag"
    )
})

ccd_atom_table = ccd_atom_table[
    ccd_atom_table["comp_id"] == "6WZ"
].copy()

ccd_atom_table = ccd_atom_table[
    ~ccd_atom_table["element"].str.upper().isin(["H", "D"])
].reset_index(drop=True)

# --------------------------------------------------
# 2. Read CCD bond information
# --------------------------------------------------

ccd_bond_table = pd.DataFrame({
    "comp_id": cif_list(cif_dict, "_chem_comp_bond.comp_id"),
    "atom_1": cif_list(cif_dict, "_chem_comp_bond.atom_id_1"),
    "atom_2": cif_list(cif_dict, "_chem_comp_bond.atom_id_2"),
    "bond_order": cif_list(cif_dict, "_chem_comp_bond.value_order"),
    "aromatic": cif_list(
        cif_dict, "_chem_comp_bond.pdbx_aromatic_flag"
    )
})

ccd_bond_table = ccd_bond_table[
    ccd_bond_table["comp_id"] == "6WZ"
].copy()

atom_names = set(ccd_atom_table["atom_id"])

ccd_bond_table = ccd_bond_table[
    ccd_bond_table["atom_1"].isin(atom_names)
    & ccd_bond_table["atom_2"].isin(atom_names)
].reset_index(drop=True)

# --------------------------------------------------
# 3. Build CCD molecular graph
# --------------------------------------------------

atom_name_to_index = {
    name: idx
    for idx, name in enumerate(ccd_atom_table["atom_id"])
}

builder = Chem.RWMol()

for _, row in ccd_atom_table.iterrows():

    atom = Chem.Atom(str(row["element"]).upper())

    atom.SetIsAromatic(
        str(row["aromatic"]).upper() == "Y"
    )

    builder.AddAtom(atom)

bond_type_map = {
    "sing": Chem.BondType.SINGLE,
    "doub": Chem.BondType.DOUBLE,
    "trip": Chem.BondType.TRIPLE
}

for _, row in ccd_bond_table.iterrows():

    i = atom_name_to_index[row["atom_1"]]
    j = atom_name_to_index[row["atom_2"]]

    if str(row["aromatic"]).upper() == "Y":
        bond_type = Chem.BondType.AROMATIC
    else:
        bond_type = bond_type_map.get(
            str(row["bond_order"]).lower()
        )

    if bond_type is None:
        raise RuntimeError(
            f"Unsupported bond type: {row['bond_order']}"
        )

    builder.AddBond(i, j, bond_type)

ccd_mol = builder.GetMol()

Chem.SanitizeMol(ccd_mol)

# --------------------------------------------------
# 4. Compare graph sizes
# --------------------------------------------------

print("CCD atoms:", ccd_mol.GetNumAtoms())
print("CCD bonds:", ccd_mol.GetNumBonds())

print("RDKit atoms:", reference_mol.GetNumAtoms())
print("RDKit bonds:", reference_mol.GetNumBonds())

# --------------------------------------------------
# 5. Find graph correspondence
# --------------------------------------------------

matches = reference_mol.GetSubstructMatches(
    ccd_mol,
    uniquify=False,
    useChirality=False,
    maxMatches=10000
)

print("\nNumber of graph mappings:", len(matches))

if not matches:
    raise RuntimeError(
        "CCD and RDKit molecular graphs do not match."
    )

# Mapping: CCD atom index -> RDKit atom index
mapping = matches[0]

ccd_to_rdkit = {
    ccd_atom_table.iloc[ccd_idx]["atom_id"]: rdkit_idx
    for ccd_idx, rdkit_idx in enumerate(mapping)
}

mapping_df = pd.DataFrame([
    {
        "ccd_atom_name": atom_name,
        "rdkit_atom_index": rdkit_idx,
        "ccd_element": ccd_atom_table.iloc[idx]["element"],
        "rdkit_element": reference_mol.GetAtomWithIdx(
            rdkit_idx
        ).GetSymbol()
    }
    for idx, (atom_name, rdkit_idx)
    in enumerate(ccd_to_rdkit.items())
])

display(mapping_df)

# --------------------------------------------------
# 6. Final checks
# --------------------------------------------------

element_check = (
    mapping_df["ccd_element"].str.upper()
    == mapping_df["rdkit_element"]
).all()

unique_indices = (
    mapping_df["rdkit_atom_index"].nunique()
    == len(mapping_df)
)

print("\nElement correspondence:", element_check)
print("One-to-one mapping:", unique_indices)
print("Mapped atoms:", len(mapping_df))

if not element_check or not unique_indices:
    raise RuntimeError("Atom mapping validation failed.")

print("\nCCD-to-RDKit connectivity mapping: PASS")
print("Stereochemistry: NOT YET VALIDATED")

CCD atoms: 34
CCD bonds: 37
RDKit atoms: 34
RDKit bonds: 37

Number of graph mappings: 6


,ccd_atom_name,rdkit_atom_index,ccd_element,rdkit_element
0,OAD,12,O,O
1,CBA,11,C,C
2,NAU,13,N,N
3,CAZ,14,C,C
4,OAC,15,O,O
5,CBB,16,C,C
6,CAM,17,C,C
7,CAI,18,C,C
8,CAJ,19,C,C
9,CAN,20,C,C



Element correspondence: True
One-to-one mapping: True
Mapped atoms: 34

CCD-to-RDKit connectivity mapping: PASS
Stereochemistry: NOT YET VALIDATED


In [110]:
# CELL 83G-G — Stereochemical validation of 6WZ

from rdkit import Chem
import pandas as pd

# 1. Read CCD atom stereochemistry
ccd_stereo = pd.DataFrame({
    "atom_id": cif_list(
        cif_dict, "_chem_comp_atom.atom_id"
    ),
    "stereo": cif_list(
        cif_dict, "_chem_comp_atom.pdbx_stereo_config"
    )
})

ccd_stereo = ccd_stereo[
    ccd_stereo["atom_id"].isin(atom_name_to_index)
].copy()

ccd_stereo["stereo"] = ccd_stereo["stereo"].astype(str).str.upper()

ccd_stereo = ccd_stereo[
    ccd_stereo["stereo"].isin(["R", "S"])
]

print("CCD stereocentres:")
display(ccd_stereo)

# 2. Assign CIP labels to RDKit molecule
Chem.AssignStereochemistry(
    reference_mol,
    cleanIt=True,
    force=True
)

rdkit_cip = {}

for atom in reference_mol.GetAtoms():

    if atom.HasProp("_CIPCode"):
        rdkit_cip[atom.GetIdx()] = atom.GetProp("_CIPCode")

print("\nRDKit stereocentres:", rdkit_cip)

# 3. Check each graph mapping
stereo_mapping_results = []

for mapping_number, mapping in enumerate(matches):

    mismatches = []
    comparisons = []

    for _, row in ccd_stereo.iterrows():

        ccd_atom_name = row["atom_id"]
        expected_cip = row["stereo"]

        ccd_index = atom_name_to_index[ccd_atom_name]
        rdkit_index = mapping[ccd_index]

        actual_cip = rdkit_cip.get(
            rdkit_index,
            "NOT_ASSIGNED"
        )

        comparisons.append({
            "ccd_atom": ccd_atom_name,
            "ccd_CIP": expected_cip,
            "rdkit_atom_index": rdkit_index,
            "rdkit_CIP": actual_cip
        })

        if expected_cip != actual_cip:
            mismatches.append(ccd_atom_name)

    stereo_mapping_results.append({
        "mapping_number": mapping_number,
        "stereocentres_checked": len(comparisons),
        "mismatches": len(mismatches),
        "mismatched_atoms": mismatches,
        "stereo_status": (
            "PASS" if not mismatches else "FAIL"
        )
    })

stereo_mapping_df = pd.DataFrame(
    stereo_mapping_results
)

print("\n========== MAPPING STEREOCHEMISTRY ==========")
display(stereo_mapping_df)

valid_stereo_mappings = [
    matches[row["mapping_number"]]
    for row in stereo_mapping_results
    if row["stereo_status"] == "PASS"
]

print(
    "\nStereochemistry-compatible mappings:",
    len(valid_stereo_mappings)
)

if not valid_stereo_mappings:
    raise RuntimeError(
        "No graph mapping preserves CCD stereochemistry."
    )

# 4. Select a valid mapping provisionally
selected_mapping = valid_stereo_mappings[0]

ccd_to_rdkit = {
    ccd_atom_table.iloc[ccd_idx]["atom_id"]: rdkit_idx
    for ccd_idx, rdkit_idx in enumerate(selected_mapping)
}

print("\nStereochemistry validation: PASS")
print("Selected mapping is provisional.")

CCD stereocentres:


,atom_id,stereo
24,CBD,R



RDKit stereocentres: {1: 'R'}

========== MAPPING STEREOCHEMISTRY ==========


,mapping_number,stereocentres_checked,mismatches,mismatched_atoms,stereo_status
0,0,1,0,[],PASS
1,1,1,0,[],PASS
2,2,1,0,[],PASS
3,3,1,0,[],PASS
4,4,1,0,[],PASS
5,5,1,0,[],PASS



Stereochemistry-compatible mappings: 6

Stereochemistry validation: PASS
Selected mapping is provisional.


In [111]:
# CELL 83G-H — Compare experimental bond geometry across mappings

import numpy as np
import pandas as pd

# Experimental coordinates indexed by CCD atom name
experimental_xyz_by_name = {
    row["atom"]: np.array(
        [row["x"], row["y"], row["z"]],
        dtype=float
    )
    for _, row in experimental_atoms.iterrows()
}

mapping_geometry_results = []

for mapping_number, mapping in enumerate(matches):

    bond_lengths = []

    for _, bond in ccd_bond_table.iterrows():

        atom_1 = bond["atom_1"]
        atom_2 = bond["atom_2"]

        if atom_1 not in experimental_xyz_by_name:
            continue

        if atom_2 not in experimental_xyz_by_name:
            continue

        idx_1 = mapping[atom_name_to_index[atom_1]]
        idx_2 = mapping[atom_name_to_index[atom_2]]

        # Coordinates assigned to the corresponding RDKit atoms
        xyz_1 = experimental_xyz_by_name[atom_1]
        xyz_2 = experimental_xyz_by_name[atom_2]

        bond_length = np.linalg.norm(xyz_1 - xyz_2)

        bond_lengths.append(bond_length)

    mapping_geometry_results.append({
        "mapping_number": mapping_number,
        "bonds_checked": len(bond_lengths),
        "mean_bond_length_A": np.mean(bond_lengths),
        "minimum_bond_length_A": np.min(bond_lengths),
        "maximum_bond_length_A": np.max(bond_lengths),
        "geometry_status": (
            "PASS"
            if len(bond_lengths) == len(ccd_bond_table)
            else "CHECK"
        )
    })

mapping_geometry_df = pd.DataFrame(
    mapping_geometry_results
)

display(mapping_geometry_df)

if not (
    mapping_geometry_df["geometry_status"] == "PASS"
).all():
    raise RuntimeError("Bond geometry validation failed.")

print("\nAll six mappings cover the complete CCD heavy-atom bond set.")
print("Use the first stereochemistry-compatible mapping provisionally.")
print("Selected mapping:", 0)

,mapping_number,bonds_checked,mean_bond_length_A,minimum_bond_length_A,maximum_bond_length_A,geometry_status
0,0,37,1.403316,1.231769,1.546075,PASS
1,1,37,1.403316,1.231769,1.546075,PASS
2,2,37,1.403316,1.231769,1.546075,PASS
3,3,37,1.403316,1.231769,1.546075,PASS
4,4,37,1.403316,1.231769,1.546075,PASS
5,5,37,1.403316,1.231769,1.546075,PASS



All six mappings cover the complete CCD heavy-atom bond set.
Use the first stereochemistry-compatible mapping provisionally.
Selected mapping: 0


In [112]:
# CELL 83G-I — Verify 6WZ ligand transformation into M1 reference frame

import numpy as np
import pandas as pd

pdb_id = "5KPP"
comp_id = "6WZ"

# 1. Retrieve the verified transformation
transform = transformations_by_pdb[pdb_id]

R = np.asarray(transform["rotation"], dtype=float)
t = np.asarray(transform["translation"], dtype=float)

# 2. Prepare ligand coordinates in RDKit atom order
num_atoms = reference_mol.GetNumAtoms()

ligand_xyz_exp = np.zeros((num_atoms, 3), dtype=float)
assigned_atoms = set()

for ccd_idx, rdkit_idx in enumerate(selected_mapping):

    atom_name = str(ccd_atom_table.iloc[ccd_idx]["atom_id"])

    if atom_name not in experimental_xyz_by_name:
        raise ValueError(f"Missing experimental atom: {atom_name}")

    if rdkit_idx in assigned_atoms:
        raise ValueError("Duplicate RDKit atom assignment detected")

    ligand_xyz_exp[rdkit_idx] = experimental_xyz_by_name[atom_name]
    assigned_atoms.add(rdkit_idx)

if len(assigned_atoms) != num_atoms:
    raise ValueError("Not all RDKit atoms received coordinates")

# 3. Inverse-transform experimental ligand coordinates
# Cell 45 used: experimental = M1 @ R + t
# Therefore: M1 = (experimental - t) @ R.T

ligand_xyz_m1 = (ligand_xyz_exp - t) @ R.T

# 4. Retrieve pocket coordinates in experimental and M1 frames
pocket_exp = np.asarray(pocket_xyz_by_pdb[pdb_id], dtype=float)
pocket_m1 = np.asarray(m1_pocket_xyz, dtype=float)

# 5. Calculate ligand-pocket distances in both coordinate frames
dist_exp = np.linalg.norm(
    ligand_xyz_exp[:, None, :] - pocket_exp[None, :, :],
    axis=2
)

dist_m1 = np.linalg.norm(
    ligand_xyz_m1[:, None, :] - pocket_m1[None, :, :],
    axis=2
)

# 6. Compare distances
minimum_exp = float(dist_exp.min())
minimum_m1 = float(dist_m1.min())

maximum_distance_difference = float(
    np.max(np.abs(dist_exp - dist_m1))
)

# 7. Compare against Cell 46 contact result
expected_minimum = 2.456679

verification = {
    "pdb_id": pdb_id,
    "component_id": comp_id,
    "ligand_atoms": num_atoms,
    "pocket_atoms": len(pocket_m1),
    "minimum_distance_experimental_frame_A": minimum_exp,
    "minimum_distance_M1_frame_A": minimum_m1,
    "cell46_reference_distance_A": expected_minimum,
    "maximum_distance_difference_A": maximum_distance_difference,
    "coordinate_assignment_complete": len(assigned_atoms) == num_atoms,
    "distance_invariance": maximum_distance_difference < 1e-5,
    "cell46_distance_match": abs(minimum_exp - expected_minimum) < 1e-3
}

print(pd.Series(verification).to_string())

if not verification["distance_invariance"]:
    raise RuntimeError("Distance invariance check failed")

if not verification["cell46_distance_match"]:
    raise RuntimeError("Cell 46 distance comparison failed")

print("\nCELL 83G-I: PASS")

pdb_id                                       5KPP
component_id                                  6WZ
ligand_atoms                                   34
pocket_atoms                                   98
minimum_distance_experimental_frame_A    2.456679
minimum_distance_M1_frame_A              2.456679
cell46_reference_distance_A              2.456679
maximum_distance_difference_A                 0.0
coordinate_assignment_complete               True
distance_invariance                          True
cell46_distance_match                        True

CELL 83G-I: PASS


In [113]:
# CELL 83G-J — Reusable CCD component extractor

import numpy as np
import pandas as pd

def extract_ccd_component_tables(cif_dict, comp_id):

    # Required CCD atom fields
    atom_fields = {
        "atom_id": "_chem_comp_atom.atom_id",
        "element": "_chem_comp_atom.type_symbol",
        "aromatic_flag": "_chem_comp_atom.pdbx_aromatic_flag",
        "stereo_config": "_chem_comp_atom.pdbx_stereo_config",
        "ordinal": "_chem_comp_atom.pdbx_ordinal",
        "formal_charge": "_chem_comp_atom.charge"
    }

    # Required CCD bond fields
    bond_fields = {
        "atom_1": "_chem_comp_bond.atom_id_1",
        "atom_2": "_chem_comp_bond.atom_id_2",
        "value_order": "_chem_comp_bond.value_order",
        "aromatic_flag": "_chem_comp_bond.pdbx_aromatic_flag",
        "stereo_config": "_chem_comp_bond.pdbx_stereo_config"
    }

    # Check required fields
    for field in [
        "_chem_comp_atom.comp_id",
        "_chem_comp_bond.comp_id"
    ]:
        if field not in cif_dict:
            raise KeyError(f"Missing mmCIF field: {field}")

    # Extract atom records
    atom_comp_ids = cif_dict["_chem_comp_atom.comp_id"]

    atom_data = {}

    for name, field in atom_fields.items():
        values = cif_dict.get(field)

        if values is None:
            if name == "formal_charge":
                values = ["0"] * len(atom_comp_ids)
            else:
                raise KeyError(f"Missing CCD atom field: {field}")

        atom_data[name] = values

    atom_data["comp_id"] = atom_comp_ids

    atom_df = pd.DataFrame(atom_data)

    atom_df = atom_df[
        atom_df["comp_id"].astype(str) == str(comp_id)
    ].copy()

    # Extract bond records
    bond_comp_ids = cif_dict["_chem_comp_bond.comp_id"]

    bond_data = {}

    for name, field in bond_fields.items():
        values = cif_dict.get(field)

        if values is None:
            if name in ["aromatic_flag", "stereo_config"]:
                values = ["N"] * len(bond_comp_ids)
            else:
                raise KeyError(f"Missing CCD bond field: {field}")

        bond_data[name] = values

    bond_data["comp_id"] = bond_comp_ids

    bond_df = pd.DataFrame(bond_data)

    bond_df = bond_df[
        bond_df["comp_id"].astype(str) == str(comp_id)
    ].copy()

    # Keep heavy atoms only
    atom_df["element"] = atom_df["element"].astype(str).str.upper()

    atom_df = atom_df[
        atom_df["element"] != "H"
    ].copy()

    # Keep bonds connecting heavy atoms only
    heavy_atom_names = set(atom_df["atom_id"].astype(str))

    bond_df = bond_df[
        bond_df["atom_1"].astype(str).isin(heavy_atom_names)
        & bond_df["atom_2"].astype(str).isin(heavy_atom_names)
    ].copy()

    # Convert ordinal and charge fields
    atom_df["ordinal"] = pd.to_numeric(
        atom_df["ordinal"], errors="raise"
    )

    atom_df["formal_charge"] = pd.to_numeric(
        atom_df["formal_charge"], errors="coerce"
    ).fillna(0).astype(int)

    atom_df = atom_df.sort_values("ordinal").reset_index(drop=True)

    bond_df = bond_df.reset_index(drop=True)

    # Basic integrity checks
    if atom_df.empty:
        raise ValueError(f"No heavy atoms found for {comp_id}")

    if atom_df["atom_id"].duplicated().any():
        raise ValueError(f"Duplicate CCD atom names in {comp_id}")

    if len(bond_df) == 0:
        raise ValueError(f"No heavy-atom bonds found for {comp_id}")

    print(f"CCD component: {comp_id}")
    print("Heavy atoms:", len(atom_df))
    print("Heavy-atom bonds:", len(bond_df))
    print("Unique elements:", sorted(atom_df["element"].unique()))

    return atom_df, bond_df


# Test using the already inspected 6WZ mmCIF dictionary
ccd_atom_test, ccd_bond_test = extract_ccd_component_tables(
    cif_dict,
    "6WZ"
)

# Compare against our previous validated tables
print("\nComparison with previous extraction:")

print("Atom count matches:",
      len(ccd_atom_test) == len(ccd_atom_table))

print("Bond count matches:",
      len(ccd_bond_test) == len(ccd_bond_table))

if len(ccd_atom_test) != 34:
    raise RuntimeError("6WZ heavy atom count mismatch")

if len(ccd_bond_test) != 37:
    raise RuntimeError("6WZ heavy bond count mismatch")

print("\nCELL 83G-J: PASS")

CCD component: 6WZ
Heavy atoms: 34
Heavy-atom bonds: 37
Unique elements: ['C', 'F', 'N', 'O']

Comparison with previous extraction:
Atom count matches: True
Bond count matches: True

CELL 83G-J: PASS


In [114]:
# CELL 83G-K — CCD-to-RDKit graph and stereochemistry validation

from rdkit import Chem
from rdkit.Chem import rdchem

def validate_ccd_rdkit_mapping(atom_df, bond_df, reference_mol):

    atom_df = atom_df.reset_index(drop=True)

    ccd_mol = Chem.RWMol()

    # 1. Build CCD atoms
    for _, row in atom_df.iterrows():

        atom = Chem.Atom(str(row["element"]))
        atom.SetFormalCharge(int(row["formal_charge"]))

        if str(row["aromatic_flag"]).upper() == "Y":
            atom.SetIsAromatic(True)

        ccd_mol.AddAtom(atom)

    atom_name_to_index = {
        str(row["atom_id"]): idx
        for idx, row in atom_df.iterrows()
    }

    # 2. Build CCD bonds
    for _, row in bond_df.iterrows():

        atom_1 = str(row["atom_1"])
        atom_2 = str(row["atom_2"])

        if atom_1 not in atom_name_to_index:
            raise ValueError(f"Unknown CCD atom: {atom_1}")

        if atom_2 not in atom_name_to_index:
            raise ValueError(f"Unknown CCD atom: {atom_2}")

        if str(row["aromatic_flag"]).upper() == "Y":
            bond_type = rdchem.BondType.AROMATIC

        else:
            order = str(row["value_order"]).lower()

            bond_type = {
                "sing": rdchem.BondType.SINGLE,
                "doub": rdchem.BondType.DOUBLE,
                "trip": rdchem.BondType.TRIPLE
            }.get(order)

            if bond_type is None:
                raise ValueError(
                    f"Unsupported CCD bond order: {order}"
                )

        idx_1 = atom_name_to_index[atom_1]
        idx_2 = atom_name_to_index[atom_2]

        ccd_mol.AddBond(idx_1, idx_2, bond_type)

        if str(row["aromatic_flag"]).upper() == "Y":
            bond = ccd_mol.GetBondBetweenAtoms(idx_1, idx_2)
            bond.SetIsAromatic(True)

    ccd_mol = ccd_mol.GetMol()

    # 3. Sanitize CCD graph
    Chem.SanitizeMol(ccd_mol)

    # 4. Prepare reference graph
    reference_mol = Chem.Mol(reference_mol)

    Chem.SanitizeMol(reference_mol)

    Chem.AssignStereochemistry(
        reference_mol,
        cleanIt=True,
        force=True
    )

    # 5. Basic graph checks
    atom_count_match = (
        ccd_mol.GetNumAtoms() == reference_mol.GetNumAtoms()
    )

    bond_count_match = (
        ccd_mol.GetNumBonds() == reference_mol.GetNumBonds()
    )

    if not atom_count_match or not bond_count_match:
        raise ValueError("CCD and reference graph sizes differ")

    # 6. Find graph-isomorphic mappings
    matches = reference_mol.GetSubstructMatches(
        ccd_mol,
        uniquify=False,
        useChirality=False,
        maxMatches=10000
    )

    valid_mappings = []

    # 7. Check element, charge and stereochemistry
    stereo_rows = []

    for mapping_number, mapping in enumerate(matches):

        element_ok = True
        charge_ok = True
        stereo_ok = True

        for ccd_idx, rdkit_idx in enumerate(mapping):

            ccd_atom = ccd_mol.GetAtomWithIdx(ccd_idx)
            rdkit_atom = reference_mol.GetAtomWithIdx(rdkit_idx)

            if ccd_atom.GetSymbol() != rdkit_atom.GetSymbol():
                element_ok = False

            if ccd_atom.GetFormalCharge() != rdkit_atom.GetFormalCharge():
                charge_ok = False

            ccd_stereo = str(
                atom_df.iloc[ccd_idx]["stereo_config"]
            ).upper()

            if ccd_stereo in ["R", "S"]:

                rdkit_stereo = rdkit_atom.GetProp("_CIPCode") \
                    if rdkit_atom.HasProp("_CIPCode") else None

                if rdkit_stereo != ccd_stereo:
                    stereo_ok = False

        status = (
            element_ok
            and charge_ok
            and stereo_ok
        )

        stereo_rows.append({
            "mapping_number": mapping_number,
            "element_match": element_ok,
            "formal_charge_match": charge_ok,
            "stereochemistry_match": stereo_ok,
            "status": "PASS" if status else "FAIL"
        })

        if status:
            valid_mappings.append(tuple(mapping))

    result_df = pd.DataFrame(stereo_rows)

    # 8. Final checks
    if not matches:
        raise RuntimeError("No graph-isomorphic mapping found")

    if not valid_mappings:
        raise RuntimeError(
            "No mapping passed element, charge and stereochemistry checks"
        )

    selected_mapping = valid_mappings[0]

    if len(set(selected_mapping)) != reference_mol.GetNumAtoms():
        raise RuntimeError("Selected mapping is not one-to-one")

    return {
        "ccd_mol": ccd_mol,
        "matches": matches,
        "valid_mappings": valid_mappings,
        "selected_mapping": selected_mapping,
        "mapping_report": result_df,
        "atom_name_to_index": atom_name_to_index
    }


# Test on 6WZ
mapping_result_6wz = validate_ccd_rdkit_mapping(
    ccd_atom_test,
    ccd_bond_test,
    reference_mol
)

print("CCD atoms:", mapping_result_6wz["ccd_mol"].GetNumAtoms())
print("CCD bonds:", mapping_result_6wz["ccd_mol"].GetNumBonds())

print("Graph mappings:", len(mapping_result_6wz["matches"]))
print("Valid mappings:", len(mapping_result_6wz["valid_mappings"]))

print("\nMapping validation:")
display(mapping_result_6wz["mapping_report"])

print("\nSelected mapping:")
print(mapping_result_6wz["selected_mapping"])

print("\nCELL 83G-K: PASS")

CCD atoms: 34
CCD bonds: 37
Graph mappings: 6
Valid mappings: 6

Mapping validation:


,mapping_number,element_match,formal_charge_match,stereochemistry_match,status
0,0,True,True,True,PASS
1,1,True,True,True,PASS
2,2,True,True,True,PASS
3,3,True,True,True,PASS
4,4,True,True,True,PASS
5,5,True,True,True,PASS



Selected mapping:
(12, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 10, 9, 8, 7, 22, 23, 24, 25, 6, 4, 5, 3, 2, 1, 0, 26, 27, 28, 29, 30, 31, 32, 33)

CELL 83G-K: PASS


In [115]:
# CELL 83G-L — Extract all 10 validated experimental ligand poses

import json
import numpy as np
import pandas as pd

from pathlib import Path
from rdkit import Chem
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

# --------------------------------------------------
# 1. Paths and input files
# --------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/Member2_Project")

OUTPUT_DIR = BASE_DIR / "member2_outputs"

POSE_DIR = OUTPUT_DIR / "experimental_ligand_poses_m1_frame"
POSE_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_FILE = OUTPUT_DIR / "validated_target_ligand_graphs.jsonl"

# --------------------------------------------------
# 2. Load validated target-specific ligand graphs
# --------------------------------------------------

graph_records_by_comp = {}

with open(GRAPH_FILE, "r") as f:

    for line in f:

        if not line.strip():
            continue

        record = json.loads(line)

        graph_records_by_comp[str(record["comp_id"])] = record

# --------------------------------------------------
# 3. Load supported experimental ligand instances
# --------------------------------------------------

supported_df = pd.read_csv(
    OUTPUT_DIR / "supported_target_pocket_pairs.csv"
)

if len(supported_df) != 10:
    raise ValueError("Expected 10 supported ligand instances")

# --------------------------------------------------
# 4. Helper for mmCIF atom-site extraction
# --------------------------------------------------

def cif_values(cif_dict, field):

    values = cif_dict.get(field)

    if values is None:
        raise KeyError(f"Missing mmCIF field: {field}")

    if isinstance(values, str):
        return [values]

    return list(values)


def extract_experimental_ligand_atoms(
    cif_dict,
    comp_id,
    label_asym_id
):

    fields = {
        "group": "_atom_site.group_PDB",
        "atom": "_atom_site.label_atom_id",
        "comp": "_atom_site.label_comp_id",
        "label_chain": "_atom_site.label_asym_id",
        "element": "_atom_site.type_symbol",
        "x": "_atom_site.Cartn_x",
        "y": "_atom_site.Cartn_y",
        "z": "_atom_site.Cartn_z",
        "alt": "_atom_site.label_alt_id",
        "model": "_atom_site.pdbx_PDB_model_num"
    }

    data = {
        key: cif_values(cif_dict, field)
        for key, field in fields.items()
    }

    lengths = {len(v) for v in data.values()}

    if len(lengths) != 1:
        raise ValueError("mmCIF atom-site column lengths differ")

    df = pd.DataFrame(data)

    # Select exact ligand component and label chain
    df = df[
        (df["comp"].astype(str) == str(comp_id))
        & (df["label_chain"].astype(str) == str(label_asym_id))
        & (df["group"].astype(str) == "HETATM")
        & (df["model"].astype(str) == "1")
    ].copy()

    # Keep blank or primary alternate conformations
    df = df[
        df["alt"].astype(str).isin([".", "?", "A"])
    ].copy()

    # Heavy atoms only
    df["element"] = df["element"].astype(str).str.upper()

    df = df[df["element"] != "H"].copy()

    for col in ["x", "y", "z"]:
        df[col] = pd.to_numeric(df[col], errors="raise")

    df = df.drop_duplicates(subset=["atom"])

    if df["atom"].duplicated().any():
        raise ValueError("Duplicate experimental ligand atom names")

    return df.reset_index(drop=True)


# --------------------------------------------------
# 5. Process all 10 ligands
# --------------------------------------------------

all_pose_records = []
extraction_summary = []

for _, row in supported_df.iterrows():

    pdb_id = str(row["pdb_id"]).upper()
    comp_id = str(row["comp_id"])
    label_chain = str(row["label_asym_id"])

    print(f"\nProcessing {pdb_id} — {comp_id}")

    # A. Locate experimental mmCIF
    cif_path = Path(cif_paths[pdb_id])

    if not cif_path.exists():
        raise FileNotFoundError(cif_path)

    cif_dict_current = MMCIF2Dict(str(cif_path))

    # B. Extract CCD atom and bond definitions
    atom_df, bond_df = extract_ccd_component_tables(
        cif_dict_current,
        comp_id
    )

    # C. Retrieve validated molecular graph
    if comp_id not in graph_records_by_comp:
        raise KeyError(f"Missing validated graph: {comp_id}")

    graph_record = graph_records_by_comp[comp_id]

    if not graph_record.get("validated_by_rdkit", False):
        raise ValueError(f"Graph not RDKit validated: {comp_id}")

    reference_mol = Chem.MolFromSmiles(
        graph_record["canonical_smiles"]
    )

    if reference_mol is None:
        raise ValueError(f"Invalid reference SMILES: {comp_id}")

    # D. Validate CCD-to-RDKit mapping
    mapping_result = validate_ccd_rdkit_mapping(
        atom_df,
        bond_df,
        reference_mol
    )

    selected_mapping = mapping_result["selected_mapping"]

    # E. Extract experimental ligand coordinates
    experimental_df = extract_experimental_ligand_atoms(
        cif_dict_current,
        comp_id,
        label_chain
    )

    # F. Verify exact atom-name and element correspondence
    ccd_names = set(atom_df["atom_id"].astype(str))
    experimental_names = set(experimental_df["atom"].astype(str))

    if ccd_names != experimental_names:

        raise ValueError(
            f"Atom-name mismatch for {pdb_id}/{comp_id}: "
            f"CCD-only={ccd_names-experimental_names}, "
            f"experimental-only={experimental_names-ccd_names}"
        )

    experimental_elements = dict(
        zip(
            experimental_df["atom"].astype(str),
            experimental_df["element"].astype(str)
        )
    )

    for _, atom_row in atom_df.iterrows():

        atom_name = str(atom_row["atom_id"])

        if experimental_elements[atom_name] != str(atom_row["element"]):
            raise ValueError(
                f"Element mismatch: {pdb_id}/{comp_id}/{atom_name}"
            )

    # G. Obtain verified transformation
    transform = transformations_by_pdb[pdb_id]

    R = np.asarray(transform["rotation"], dtype=float)
    t = np.asarray(transform["translation"], dtype=float)

    # H. Assign coordinates in RDKit atom order
    n_atoms = reference_mol.GetNumAtoms()

    ligand_xyz_exp = np.zeros((n_atoms, 3), dtype=float)

    xyz_by_name = {
        str(r["atom"]): np.array(
            [r["x"], r["y"], r["z"]],
            dtype=float
        )
        for _, r in experimental_df.iterrows()
    }

    for ccd_idx, rdkit_idx in enumerate(selected_mapping):

        atom_name = str(atom_df.iloc[ccd_idx]["atom_id"])

        ligand_xyz_exp[rdkit_idx] = xyz_by_name[atom_name]

    # I. Transform experimental coordinates into M1 frame
    ligand_xyz_m1 = (ligand_xyz_exp - t) @ R.T

    # J. Store coordinates and metadata
    for atom_idx in range(n_atoms):

        atom = reference_mol.GetAtomWithIdx(atom_idx)

        all_pose_records.append({

            "pdb_id": pdb_id,
            "comp_id": comp_id,
            "label_asym_id": label_chain,
            "atom_index": atom_idx,
            "element": atom.GetSymbol(),
            "x_experimental": ligand_xyz_exp[atom_idx, 0],
            "y_experimental": ligand_xyz_exp[atom_idx, 1],
            "z_experimental": ligand_xyz_exp[atom_idx, 2],
            "x_m1": ligand_xyz_m1[atom_idx, 0],
            "y_m1": ligand_xyz_m1[atom_idx, 1],
            "z_m1": ligand_xyz_m1[atom_idx, 2]

        })

    extraction_summary.append({

        "pdb_id": pdb_id,
        "comp_id": comp_id,
        "experimental_atoms": len(experimental_df),
        "ccd_atoms": len(atom_df),
        "ccd_bonds": len(bond_df),
        "rdkit_atoms": n_atoms,
        "valid_graph_mappings": len(mapping_result["valid_mappings"]),
        "atom_names_match": True,
        "elements_match": True,
        "stereochemistry_valid": True,
        "status": "PASS"

    })

    print("Atoms:", n_atoms)
    print("Valid mappings:", len(mapping_result["valid_mappings"]))
    print("Extraction: PASS")


# --------------------------------------------------
# 6. Export results
# --------------------------------------------------

pose_coordinates_df = pd.DataFrame(all_pose_records)

extraction_summary_df = pd.DataFrame(extraction_summary)

coordinates_path = POSE_DIR / "all_experimental_ligand_coordinates.csv"

summary_path = POSE_DIR / "ligand_extraction_summary.csv"

pose_coordinates_df.to_csv(
    coordinates_path,
    index=False
)

extraction_summary_df.to_csv(
    summary_path,
    index=False
)

# --------------------------------------------------
# 7. Final checks
# --------------------------------------------------

print("\n========== EXTRACTION SUMMARY ==========")

display(extraction_summary_df)

print("\nTotal ligand structures:", len(extraction_summary_df))
print("Total heavy atoms:", len(pose_coordinates_df))
print(
    "Successful structures:",
    (extraction_summary_df["status"] == "PASS").sum()
)

print("\nCoordinate file:", coordinates_path)
print("Summary file:", summary_path)

if len(extraction_summary_df) != 10:
    raise RuntimeError("Not all 10 ligand structures were processed")

if not (extraction_summary_df["status"] == "PASS").all():
    raise RuntimeError("One or more ligand extractions failed")

print("\nCELL 83G-L: PASS")


Processing 5KPP — 6WZ
CCD component: 6WZ
Heavy atoms: 34
Heavy-atom bonds: 37
Unique elements: ['C', 'F', 'N', 'O']
Atoms: 34
Valid mappings: 6
Extraction: PASS

Processing 5KPQ — 6X2
CCD component: 6X2
Heavy atoms: 32
Heavy-atom bonds: 35
Unique elements: ['C', 'F', 'N', 'O']
Atoms: 32
Valid mappings: 1
Extraction: PASS

Processing 5WRZ — 7U3
CCD component: 7U3
Heavy atoms: 19
Heavy-atom bonds: 21
Unique elements: ['C', 'F', 'N', 'O']
Atoms: 19
Valid mappings: 1
Extraction: PASS

Processing 5WS1 — 7U9
CCD component: 7U9
Heavy atoms: 20
Heavy-atom bonds: 22
Unique elements: ['C', 'N', 'O']


KekulizeException: Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 12 13 14 15 16

In [116]:
# CELL 83G-L1 — Diagnose 7U9 CCD aromaticity

print("ATOM AROMATICITY COUNTS")
display(
    atom_df["aromatic_flag"]
    .astype(str)
    .str.upper()
    .value_counts()
)

print("\nBOND ORDER COUNTS")
display(
    bond_df["value_order"]
    .astype(str)
    .str.lower()
    .value_counts()
)

print("\nBOND AROMATICITY COUNTS")
display(
    bond_df["aromatic_flag"]
    .astype(str)
    .str.upper()
    .value_counts()
)

print("\nAROMATIC ATOMS")
display(
    atom_df[
        atom_df["aromatic_flag"].astype(str).str.upper() == "Y"
    ][["atom_id", "element", "aromatic_flag", "stereo_config"]]
)

print("\nAROMATIC BONDS")
display(
    bond_df[
        bond_df["aromatic_flag"].astype(str).str.upper() == "Y"
    ][["atom_1", "atom_2", "value_order", "aromatic_flag"]]
)

print("\nALL BONDS")
display(bond_df[
    ["atom_1", "atom_2", "value_order", "aromatic_flag"]
])

print("\nCELL 83G-L1: DIAGNOSTIC COMPLETE")

ATOM AROMATICITY COUNTS


,count
aromatic_flag,
N,11
Y,9



BOND ORDER COUNTS


,count
value_order,
sing,16
doub,6



BOND AROMATICITY COUNTS


,count
aromatic_flag,
N,12
Y,10


... (3 outputs left out)

,atom_1,atom_2,value_order,aromatic_flag
3,CAO,CAF,sing,Y
4,CAO,CAR,doub,Y
5,CAF,CAE,doub,Y
9,CAR,NAK,sing,Y
10,CAR,CAQ,sing,Y
11,NAK,CAP,doub,Y
12,CAE,CAG,sing,Y
17,CAQ,CAG,doub,Y
18,CAQ,NAL,sing,Y
20,CAP,NAL,sing,Y



ALL BONDS


,atom_1,atom_2,value_order,aromatic_flag
0,OAC,CAM,doub,N
1,NAA,CAM,sing,N
2,CAM,CAO,sing,N
3,CAO,CAF,sing,Y
4,CAO,CAR,doub,Y
5,CAF,CAE,doub,Y
6,CAH,CAI,sing,N
7,CAH,CAS,sing,N
8,CAI,NAT,sing,N
9,CAR,NAK,sing,Y



CELL 83G-L1: DIAGNOSTIC COMPLETE


In [117]:
# CELL 83G-L2 — Corrected CCD aromaticity handling

from rdkit import Chem
from rdkit.Chem import rdchem

def validate_ccd_rdkit_mapping_v2(atom_df, bond_df, reference_mol):

    atom_df = atom_df.reset_index(drop=True)

    ccd_mol = Chem.RWMol()

    # 1. Construct atoms without forcing aromaticity
    for _, row in atom_df.iterrows():

        atom = Chem.Atom(str(row["element"]))
        atom.SetFormalCharge(int(row["formal_charge"]))

        ccd_mol.AddAtom(atom)

    atom_name_to_index = {
        str(row["atom_id"]): idx
        for idx, row in atom_df.iterrows()
    }

    # 2. Preserve original CCD bond orders
    bond_types = {
        "sing": rdchem.BondType.SINGLE,
        "doub": rdchem.BondType.DOUBLE,
        "trip": rdchem.BondType.TRIPLE
    }

    for _, row in bond_df.iterrows():

        atom_1 = str(row["atom_1"])
        atom_2 = str(row["atom_2"])

        order = str(row["value_order"]).lower()

        if order not in bond_types:
            raise ValueError(f"Unsupported bond order: {order}")

        ccd_mol.AddBond(
            atom_name_to_index[atom_1],
            atom_name_to_index[atom_2],
            bond_types[order]
        )

    ccd_mol = ccd_mol.GetMol()

    # 3. Let RDKit perceive aromaticity
    Chem.SanitizeMol(ccd_mol)

    # 4. Prepare reference molecule
    reference_mol = Chem.Mol(reference_mol)

    Chem.SanitizeMol(reference_mol)

    Chem.AssignStereochemistry(
        reference_mol,
        cleanIt=True,
        force=True
    )

    # 5. Verify graph sizes
    if ccd_mol.GetNumAtoms() != reference_mol.GetNumAtoms():
        raise ValueError("Atom counts differ")

    if ccd_mol.GetNumBonds() != reference_mol.GetNumBonds():
        raise ValueError("Bond counts differ")

    # 6. Find graph mappings
    matches = reference_mol.GetSubstructMatches(
        ccd_mol,
        uniquify=False,
        useChirality=False,
        maxMatches=10000
    )

    if not matches:
        raise RuntimeError("No graph-isomorphic mapping found")

    valid_mappings = []
    reports = []

    for mapping_number, mapping in enumerate(matches):

        element_ok = True
        charge_ok = True
        stereo_ok = True

        for ccd_idx, rdkit_idx in enumerate(mapping):

            ccd_atom = ccd_mol.GetAtomWithIdx(ccd_idx)
            rdkit_atom = reference_mol.GetAtomWithIdx(rdkit_idx)

            if ccd_atom.GetSymbol() != rdkit_atom.GetSymbol():
                element_ok = False

            if ccd_atom.GetFormalCharge() != rdkit_atom.GetFormalCharge():
                charge_ok = False

            stereo = str(
                atom_df.iloc[ccd_idx]["stereo_config"]
            ).upper()

            if stereo in ["R", "S"]:

                cip = (
                    rdkit_atom.GetProp("_CIPCode")
                    if rdkit_atom.HasProp("_CIPCode")
                    else None
                )

                if cip != stereo:
                    stereo_ok = False

        passed = element_ok and charge_ok and stereo_ok

        reports.append({
            "mapping_number": mapping_number,
            "element_match": element_ok,
            "formal_charge_match": charge_ok,
            "stereochemistry_match": stereo_ok,
            "status": "PASS" if passed else "FAIL"
        })

        if passed:
            valid_mappings.append(tuple(mapping))

    if not valid_mappings:
        raise RuntimeError("No mapping passed validation")

    return {
        "ccd_mol": ccd_mol,
        "matches": matches,
        "valid_mappings": valid_mappings,
        "selected_mapping": valid_mappings[0],
        "mapping_report": pd.DataFrame(reports),
        "atom_name_to_index": atom_name_to_index
    }


# Test 7U9
mapping_result_7u9 = validate_ccd_rdkit_mapping_v2(
    atom_df,
    bond_df,
    Chem.MolFromSmiles(graph_record["canonical_smiles"])
)

print("CCD atoms:", mapping_result_7u9["ccd_mol"].GetNumAtoms())
print("CCD bonds:", mapping_result_7u9["ccd_mol"].GetNumBonds())

print("Graph mappings:", len(mapping_result_7u9["matches"]))
print("Valid mappings:", len(mapping_result_7u9["valid_mappings"]))

display(mapping_result_7u9["mapping_report"])

print("\nCELL 83G-L2: PASS")

CCD atoms: 20
CCD bonds: 22
Graph mappings: 1
Valid mappings: 1


,mapping_number,element_match,formal_charge_match,stereochemistry_match,status
0,0,True,True,True,PASS



CELL 83G-L2: PASS


In [118]:
# CELL 83G-L3 — Resume extraction from 5WS1

import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from Bio.PDB.MMCIF2Dict import MMCIF2Dict

remaining_pdbs = [
    "5WS1", "7AAC", "7ONS", "7ONT",
    "8JNZ", "9ETQ", "9ILN"
]

remaining_results = []

for pdb_id in remaining_pdbs:

    row = supported_df[
        supported_df["pdb_id"].astype(str).str.upper() == pdb_id
    ].iloc[0]

    comp_id = str(row["comp_id"])
    label_chain = str(row["label_asym_id"])

    print(f"\nProcessing {pdb_id} — {comp_id}")

    cif_dict_current = MMCIF2Dict(str(Path(cif_paths[pdb_id])))

    atom_df, bond_df = extract_ccd_component_tables(
        cif_dict_current, comp_id
    )

    graph_record = graph_records_by_comp[comp_id]

    reference_mol = Chem.MolFromSmiles(
        graph_record["canonical_smiles"]
    )

    if reference_mol is None:
        raise ValueError(f"Invalid reference graph: {comp_id}")

    mapping_result = validate_ccd_rdkit_mapping_v2(
        atom_df, bond_df, reference_mol
    )

    experimental_df = extract_experimental_ligand_atoms(
        cif_dict_current, comp_id, label_chain
    )

    ccd_names = set(atom_df["atom_id"].astype(str))
    experimental_names = set(experimental_df["atom"].astype(str))

    if ccd_names != experimental_names:
        raise ValueError(
            f"Atom-name mismatch for {pdb_id}/{comp_id}"
        )

    experimental_elements = dict(
        zip(experimental_df["atom"], experimental_df["element"])
    )

    for _, atom_row in atom_df.iterrows():

        name = str(atom_row["atom_id"])

        if experimental_elements[name] != str(atom_row["element"]):
            raise ValueError(f"Element mismatch: {name}")

    transform = transformations_by_pdb[pdb_id]

    R = np.asarray(transform["rotation"], dtype=float)
    t = np.asarray(transform["translation"], dtype=float)

    n_atoms = reference_mol.GetNumAtoms()

    ligand_xyz_exp = np.zeros((n_atoms, 3), dtype=float)

    xyz_by_name = {
        str(r["atom"]): np.array(
            [r["x"], r["y"], r["z"]], dtype=float
        )
        for _, r in experimental_df.iterrows()
    }

    selected_mapping = mapping_result["selected_mapping"]

    for ccd_idx, rdkit_idx in enumerate(selected_mapping):

        atom_name = str(atom_df.iloc[ccd_idx]["atom_id"])

        ligand_xyz_exp[rdkit_idx] = xyz_by_name[atom_name]

    ligand_xyz_m1 = (ligand_xyz_exp - t) @ R.T

    for atom_idx in range(n_atoms):

        atom = reference_mol.GetAtomWithIdx(atom_idx)

        all_pose_records.append({

            "pdb_id": pdb_id,
            "comp_id": comp_id,
            "label_asym_id": label_chain,
            "atom_index": atom_idx,
            "element": atom.GetSymbol(),
            "x_experimental": ligand_xyz_exp[atom_idx, 0],
            "y_experimental": ligand_xyz_exp[atom_idx, 1],
            "z_experimental": ligand_xyz_exp[atom_idx, 2],
            "x_m1": ligand_xyz_m1[atom_idx, 0],
            "y_m1": ligand_xyz_m1[atom_idx, 1],
            "z_m1": ligand_xyz_m1[atom_idx, 2]

        })

    remaining_results.append({

        "pdb_id": pdb_id,
        "comp_id": comp_id,
        "atoms": n_atoms,
        "bonds": reference_mol.GetNumBonds(),
        "valid_mappings": len(mapping_result["valid_mappings"]),
        "status": "PASS"

    })

    print(
        "Atoms:", n_atoms,
        "| Valid mappings:", len(mapping_result["valid_mappings"]),
        "| PASS"
    )


# Export complete results
pose_coordinates_df = pd.DataFrame(all_pose_records)

extraction_summary_df = pd.DataFrame(
    extraction_summary + remaining_results
)

pose_coordinates_df.to_csv(
    POSE_DIR / "all_experimental_ligand_coordinates.csv",
    index=False
)

extraction_summary_df.to_csv(
    POSE_DIR / "ligand_extraction_summary.csv",
    index=False
)

print("\n========== FINAL EXTRACTION ==========")

display(extraction_summary_df)

print("Total structures:", len(extraction_summary_df))
print("Total heavy atoms:", len(pose_coordinates_df))

assert len(extraction_summary_df) == 10
assert extraction_summary_df["status"].eq("PASS").all()

print("\nCELL 83G-L3: PASS")


Processing 5WS1 — 7U9
CCD component: 7U9
Heavy atoms: 20
Heavy-atom bonds: 22
Unique elements: ['C', 'N', 'O']
Atoms: 20 | Valid mappings: 1 | PASS

Processing 7AAC — 78P
CCD component: 78P
Heavy atoms: 18
Heavy-atom bonds: 20
Unique elements: ['C', 'N', 'O']
Atoms: 18 | Valid mappings: 1 | PASS

Processing 7ONS — VKT
CCD component: VKT
Heavy atoms: 27
Heavy-atom bonds: 30
Unique elements: ['C', 'N', 'O']
Atoms: 27 | Valid mappings: 2 | PASS

Processing 7ONT — VKQ
CCD component: VKQ
Heavy atoms: 30
Heavy-atom bonds: 33
Unique elements: ['C', 'N', 'O']
Atoms: 30 | Valid mappings: 2 | PASS

Processing 8JNZ — ERV
CCD component: ERV
Heavy atoms: 13
Heavy-atom bonds: 14
Unique elements: ['C', 'N', 'O']
Atoms: 13 | Valid mappings: 1 | PASS

Processing 9ETQ — A1H63
CCD component: A1H63
Heavy atoms: 30
Heavy-atom bonds: 33
Unique elements: ['C', 'N', 'O']
Atoms: 30 | Valid mappings: 2 | PASS

Processing 9ILN — A1D9M
CCD component: A1D9M
Heavy atoms: 31
Heavy-atom bonds: 33
Unique elements: ['

,pdb_id,comp_id,experimental_atoms,ccd_atoms,ccd_bonds,rdkit_atoms,valid_graph_mappings,atom_names_match,elements_match,stereochemistry_valid,status,atoms,bonds,valid_mappings
0,5KPP,6WZ,34.0,34.0,37.0,34.0,6.0,True,True,True,PASS,NaN,NaN,NaN
1,5KPQ,6X2,32.0,32.0,35.0,32.0,1.0,True,True,True,PASS,NaN,NaN,NaN
2,5WRZ,7U3,19.0,19.0,21.0,19.0,1.0,True,True,True,PASS,NaN,NaN,NaN
3,5WS1,7U9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,20.0,22.0,1.0
4,7AAC,78P,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,18.0,20.0,1.0
5,7ONS,VKT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,27.0,30.0,2.0
6,7ONT,VKQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,30.0,33.0,2.0
7,8JNZ,ERV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,13.0,14.0,1.0
8,9ETQ,A1H63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,30.0,33.0,2.0
9,9ILN,A1D9M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PASS,31.0,33.0,1.0


Total structures: 10
Total heavy atoms: 254

CELL 83G-L3: PASS


In [119]:
# CELL 83G-M — Validate all 10 ligand coordinate transformations

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

validation_results = []

for pdb_id in sorted(pose_coordinates_df["pdb_id"].unique()):

    ligand_df = pose_coordinates_df[
        pose_coordinates_df["pdb_id"] == pdb_id
    ].sort_values("atom_index")

    ligand_exp = ligand_df[
        ["x_experimental", "y_experimental", "z_experimental"]
    ].to_numpy(dtype=float)

    ligand_m1 = ligand_df[
        ["x_m1", "y_m1", "z_m1"]
    ].to_numpy(dtype=float)

    # Retrieve transformation
    transform = transformations_by_pdb[pdb_id]

    R = np.asarray(transform["rotation"], dtype=float)
    t = np.asarray(transform["translation"], dtype=float)

    # Recalculate transformed coordinates independently
    recalculated_m1 = (ligand_exp - t) @ R.T

    coordinate_error = np.max(
        np.abs(recalculated_m1 - ligand_m1)
    )

    # Pocket coordinates in experimental frame
    pocket_exp = np.asarray(
        pocket_xyz_by_pdb[pdb_id],
        dtype=float
    )

    # Pocket coordinates in M1 reference frame
    pocket_reference = np.asarray(
        m1_pocket_xyz,
        dtype=float
    )

    # Calculate ligand-pocket distances in both frames
    distances_exp = cdist(ligand_exp, pocket_exp)

    distances_m1 = cdist(ligand_m1, pocket_reference)

    min_distance_exp = float(distances_exp.min())
    min_distance_m1 = float(distances_m1.min())

    distance_error = abs(
        min_distance_exp - min_distance_m1
    )

    # Compare with Cell 46 reference
    reference_row = supported_df[
        supported_df["pdb_id"].astype(str).str.upper() == pdb_id
    ].iloc[0]

    cell46_distance = float(
        reference_row["minimum_distance_A"]
    )

    cell46_error = abs(
        min_distance_exp - cell46_distance
    )

    passed = (
        coordinate_error < 1e-5
        and distance_error < 1e-5
        and cell46_error < 1e-4
    )

    validation_results.append({

        "pdb_id": pdb_id,
        "ligand": str(reference_row["comp_id"]),
        "ligand_atoms": len(ligand_exp),
        "coordinate_max_error_A": coordinate_error,
        "experimental_min_distance_A": min_distance_exp,
        "M1_min_distance_A": min_distance_m1,
        "distance_invariance_error_A": distance_error,
        "Cell46_reference_distance_A": cell46_distance,
        "Cell46_difference_A": cell46_error,
        "status": "PASS" if passed else "FAIL"

    })


# --------------------------------------------------
# Final validation report
# --------------------------------------------------

coordinate_validation_df = pd.DataFrame(validation_results)

display(coordinate_validation_df)

print("\n========== VALIDATION SUMMARY ==========")

print("Structures:", len(coordinate_validation_df))

print(
    "Passed:",
    (coordinate_validation_df["status"] == "PASS").sum()
)

print(
    "Failed:",
    (coordinate_validation_df["status"] == "FAIL").sum()
)

validation_path = (
    POSE_DIR / "coordinate_transformation_validation.csv"
)

coordinate_validation_df.to_csv(
    validation_path,
    index=False
)

assert len(coordinate_validation_df) == 10
assert coordinate_validation_df["status"].eq("PASS").all()

print("\nSaved:", validation_path)
print("\nCELL 83G-M: PASS")

,pdb_id,ligand,ligand_atoms,coordinate_max_error_A,experimental_min_distance_A,M1_min_distance_A,distance_invariance_error_A,Cell46_reference_distance_A,Cell46_difference_A,status
0,5KPP,6WZ,34,0.0,2.456679,2.456679,1.820766e-14,2.456679,0.000000e+00,PASS
1,5KPQ,6X2,32,0.0,2.400476,2.400476,5.773160e-15,2.400476,4.440892e-16,PASS
2,5WRZ,7U3,19,0.0,2.804299,2.804299,1.154632e-14,2.804299,0.000000e+00,PASS
3,5WS1,7U9,20,0.0,2.565812,2.565812,9.325873e-15,2.565812,4.440892e-16,PASS
4,7AAC,78P,18,0.0,2.662797,2.662797,2.442491e-14,2.662797,0.000000e+00,PASS
5,7ONS,VKT,27,0.0,3.371240,3.371240,4.440892e-15,3.371240,0.000000e+00,PASS
6,7ONT,VKQ,30,0.0,3.454614,3.454614,3.463896e-14,3.454614,0.000000e+00,PASS
7,8JNZ,ERV,13,0.0,2.811423,2.811423,3.375078e-14,2.811423,0.000000e+00,PASS
8,9ETQ,A1H63,30,0.0,3.444867,3.444867,1.065814e-14,3.444867,0.000000e+00,PASS
9,9ILN,A1D9M,31,0.0,2.451098,2.451098,1.332268e-14,2.451098,0.000000e+00,PASS



========== VALIDATION SUMMARY ==========
Structures: 10
Passed: 10
Failed: 0

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/experimental_ligand_poses_m1_frame/coordinate_transformation_validation.csv

CELL 83G-M: PASS


In [120]:
# CELL 83G-N — Export 10 experimental ligand poses as SDF

from rdkit import Chem
from rdkit.Geometry import Point3D
from pathlib import Path
import pandas as pd
import numpy as np

SDF_DIR = POSE_DIR / "validated_sdf"
SDF_DIR.mkdir(parents=True, exist_ok=True)

sdf_validation = []

for pdb_id in sorted(pose_coordinates_df["pdb_id"].unique()):

    ligand_df = pose_coordinates_df[
        pose_coordinates_df["pdb_id"] == pdb_id
    ].sort_values("atom_index")

    comp_id = str(ligand_df["comp_id"].iloc[0])

    record = graph_records_by_comp[comp_id]

    mol = Chem.MolFromSmiles(record["canonical_smiles"])

    if mol is None:
        raise ValueError(f"Invalid molecular graph: {comp_id}")

    xyz = ligand_df[
        ["x_m1", "y_m1", "z_m1"]
    ].to_numpy(dtype=float)

    if len(xyz) != mol.GetNumAtoms():
        raise ValueError(f"Atom count mismatch: {comp_id}")

    conf = Chem.Conformer(mol.GetNumAtoms())

    for idx, coord in enumerate(xyz):

        conf.SetAtomPosition(
            idx,
            Point3D(
                float(coord[0]),
                float(coord[1]),
                float(coord[2])
            )
        )

    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)

    mol.SetProp("_Name", f"{pdb_id}_{comp_id}")
    mol.SetProp("PDB_ID", pdb_id)
    mol.SetProp("CCD_COMPONENT", comp_id)
    mol.SetProp("COORDINATE_FRAME", "M1_REFERENCE")

    sdf_path = SDF_DIR / f"{pdb_id}_{comp_id}.sdf"

    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol)
    writer.close()

    # Reopen exported SDF
    reopened = Chem.SDMolSupplier(
        str(sdf_path),
        removeHs=False,
        sanitize=True
    )

    check_mol = reopened[0]

    if check_mol is None:
        raise ValueError(f"SDF reopening failed: {sdf_path}")

    if check_mol.GetNumAtoms() != mol.GetNumAtoms():
        raise ValueError(f"SDF atom count mismatch: {comp_id}")

    if check_mol.GetNumBonds() != mol.GetNumBonds():
        raise ValueError(f"SDF bond count mismatch: {comp_id}")

    if check_mol.GetNumConformers() != 1:
        raise ValueError(f"Missing 3D conformer: {comp_id}")

    sdf_validation.append({

        "pdb_id": pdb_id,
        "comp_id": comp_id,
        "atoms": check_mol.GetNumAtoms(),
        "bonds": check_mol.GetNumBonds(),
        "has_3d_coordinates": check_mol.GetConformer().Is3D(),
        "sdf_file": str(sdf_path),
        "status": "PASS"

    })


sdf_validation_df = pd.DataFrame(sdf_validation)

display(sdf_validation_df)

sdf_validation_df.to_csv(
    SDF_DIR / "sdf_export_validation.csv",
    index=False
)

assert len(sdf_validation_df) == 10
assert sdf_validation_df["status"].eq("PASS").all()
assert sdf_validation_df["has_3d_coordinates"].all()

print("\nSDF files exported:", len(sdf_validation_df))
print("Output directory:", SDF_DIR)
print("\nCELL 83G-N: PASS")

,pdb_id,comp_id,atoms,bonds,has_3d_coordinates,sdf_file,status
0,5KPP,6WZ,34,37,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
1,5KPQ,6X2,32,35,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
2,5WRZ,7U3,19,21,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
3,5WS1,7U9,20,22,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
4,7AAC,78P,18,20,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
5,7ONS,VKT,27,30,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
6,7ONT,VKQ,30,33,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
7,8JNZ,ERV,13,14,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
8,9ETQ,A1H63,30,33,True,/content/drive/MyDrive/Member2_Project/member2...,PASS
9,9ILN,A1D9M,31,33,True,/content/drive/MyDrive/Member2_Project/member2...,PASS



SDF files exported: 10
Output directory: /content/drive/MyDrive/Member2_Project/member2_outputs/experimental_ligand_poses_m1_frame/validated_sdf

CELL 83G-N: PASS


In [121]:
# ============================================================
# CELL 83G-O
# EXPERIMENTAL LIGAND–RECEPTOR SPATIAL PHARMACOPHORE ANALYSIS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem, RDConfig
from rdkit.Chem import ChemicalFeatures
from scipy.spatial.distance import cdist
import json

# ------------------------------------------------------------
# 1. CHECK REQUIRED INPUTS
# ------------------------------------------------------------

required_variables = [
    "receptor_features_by_conformer",
]

for var in required_variables:
    if var not in globals():
        raise NameError(f"Required variable '{var}' is missing. Run its preparation cell first.")

if 1 not in receptor_features_by_conformer:
    raise ValueError("M1 conformer 1 receptor features are unavailable.")

# ------------------------------------------------------------
# 2. LOCATE VALIDATED EXPERIMENTAL LIGAND SDFS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/Member2_Project")

SDF_DIR = (
    PROJECT_ROOT
    / "member2_outputs"
    / "experimental_ligand_poses_m1_frame"
    / "validated_sdf"
)

if not SDF_DIR.exists():
    raise FileNotFoundError(f"Validated SDF directory not found: {SDF_DIR}")

sdf_files = sorted(SDF_DIR.glob("*.sdf"))

if len(sdf_files) != 10:
    raise ValueError(
        f"Expected 10 experimental ligand SDF files, found {len(sdf_files)}."
    )

# ------------------------------------------------------------
# 3. LOAD M1 RECEPTOR FEATURES
# ------------------------------------------------------------

receptor_df = receptor_features_by_conformer[1].copy()

required_columns = {"feature_index", "x", "y", "z", "identity"}

missing_columns = required_columns - set(receptor_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing receptor feature columns: {missing_columns}"
    )

type_column = next(
    (c for c in ["type", "feature_type", "feature"] if c in receptor_df.columns),
    None
)

if type_column is None:
    raise ValueError("Could not identify the receptor feature-type column.")

# Support both textual feature names and M1 numeric feature codes.
feature_code_map = {
    0: "DONOR",
    1: "ACCEPTOR",
    2: "HYDROPHOBIC",
    3: "AROMATIC",
    4: "POSITIVE",
    5: "NEGATIVE"
}

def normalize_feature_type(value):
    if isinstance(value, (int, np.integer)):
        return feature_code_map.get(int(value), str(value).upper())

    text_value = str(value).strip().upper()

    if text_value.isdigit():
        return feature_code_map.get(int(text_value), text_value)

    aliases = {
        "HYDROPHOBE": "HYDROPHOBIC",
        "LUMPHOBIC": "HYDROPHOBIC",
        "LUMPHOBE": "HYDROPHOBIC",
        "POSIONIZABLE": "POSITIVE",
        "NEGIONIZABLE": "NEGATIVE"
    }

    return aliases.get(text_value, text_value)

receptor_df["normalized_type"] = (
    receptor_df[type_column].apply(normalize_feature_type)
)

receptor_xyz = receptor_df[["x", "y", "z"]].to_numpy(dtype=float)

if not np.isfinite(receptor_xyz).all():
    raise ValueError("M1 receptor feature coordinates contain NaN or infinity.")

print("M1 receptor features loaded:", len(receptor_df))
print("Conformer used: 1")
print("Feature types:")
print(receptor_df["normalized_type"].value_counts())

# ------------------------------------------------------------
# 4. CREATE RDKit 3D PHARMACOPHORE FEATURE EXTRACTOR
# ------------------------------------------------------------

feature_definition = str(
    Path(RDConfig.RDDataDir) / "BaseFeatures.fdef"
)

factory = ChemicalFeatures.BuildFeatureFactory(feature_definition)

ligand_family_map = {
    "Hydrophobe": "HYDROPHOBIC",
    "LumpedHydrophobe": "HYDROPHOBIC",
    "Aromatic": "AROMATIC",
    "Donor": "DONOR",
    "Acceptor": "ACCEPTOR",
    "PosIonizable": "POSITIVE",
    "NegIonizable": "NEGATIVE"
}

# Chemically compatible feature pairs.
compatible_pairs = {
    ("HYDROPHOBIC", "HYDROPHOBIC"),
    ("AROMATIC", "AROMATIC"),
    ("DONOR", "ACCEPTOR"),
    ("ACCEPTOR", "DONOR"),
    ("POSITIVE", "NEGATIVE"),
    ("NEGATIVE", "POSITIVE")
}

# Screening cutoffs in Angstroms.
# These are proximity heuristics, not universal interaction laws.
screening_cutoffs = {
    "HYDROPHOBIC": 4.5,
    "AROMATIC": 5.0,
    "DONOR_ACCEPTOR": 3.5,
    "IONIC": 4.0
}

def get_pair_cutoff(receptor_type, ligand_type):

    if receptor_type == "HYDROPHOBIC":
        return screening_cutoffs["HYDROPHOBIC"]

    if receptor_type == "AROMATIC":
        return screening_cutoffs["AROMATIC"]

    if {receptor_type, ligand_type} == {"DONOR", "ACCEPTOR"}:
        return screening_cutoffs["DONOR_ACCEPTOR"]

    if {receptor_type, ligand_type} == {"POSITIVE", "NEGATIVE"}:
        return screening_cutoffs["IONIC"]

    return None

# ------------------------------------------------------------
# 5. EXTRACT LIGAND FEATURES AND COMPARE SPATIAL DISTANCES
# ------------------------------------------------------------

all_ligand_features = []
all_compatible_pairs = []
ligand_summary = []

for sdf_path in sdf_files:

    supplier = Chem.SDMolSupplier(
        str(sdf_path),
        removeHs=False,
        sanitize=True
    )

    mol = next((m for m in supplier if m is not None), None)

    if mol is None:
        print("WARNING: Could not read:", sdf_path.name)
        continue

    if mol.GetNumConformers() == 0:
        print("WARNING: No 3D conformer:", sdf_path.name)
        continue

    if mol.GetProp("_Name") if mol.HasProp("_Name") else False:
        ligand_name = mol.GetProp("_Name")
    else:
        ligand_name = sdf_path.stem

    pdb_id = mol.GetProp("PDB_ID") if mol.HasProp("PDB_ID") else sdf_path.stem
    ccd_id = (
        mol.GetProp("CCD_COMPONENT")
        if mol.HasProp("CCD_COMPONENT")
        else ""
    )

    conformer = mol.GetConformer()
    ligand_features = factory.GetFeaturesForMol(mol)

    ligand_records = []

    for feature_number, feature in enumerate(ligand_features):

        family = feature.GetFamily()

        if family not in ligand_family_map:
            continue

        ligand_type = ligand_family_map[family]

        position = feature.GetPos()

        ligand_xyz = np.array(
            [position.x, position.y, position.z],
            dtype=float
        )

        if not np.isfinite(ligand_xyz).all():
            continue

        record = {
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": ccd_id,
            "ligand_name": ligand_name,
            "ligand_feature_index": feature_number,
            "ligand_feature_family": family,
            "ligand_type": ligand_type,
            "ligand_atom_ids": ",".join(
                map(str, feature.GetAtomIds())
            ),
            "x": ligand_xyz[0],
            "y": ligand_xyz[1],
            "z": ligand_xyz[2]
        }

        ligand_records.append(record)
        all_ligand_features.append(record)

    if not ligand_records:
        print("WARNING: No supported pharmacophore features:", pdb_id)
        continue

    ligand_xyz_array = np.array([
        [r["x"], r["y"], r["z"]]
        for r in ligand_records
    ])

    distance_matrix = cdist(receptor_xyz, ligand_xyz_array)

    compatible_count = 0
    within_cutoff_count = 0

    for receptor_idx, receptor_row in receptor_df.iterrows():

        receptor_type = receptor_row["normalized_type"]

        for ligand_idx, ligand_record in enumerate(ligand_records):

            ligand_type = ligand_record["ligand_type"]

            if (receptor_type, ligand_type) not in compatible_pairs:
                continue

            distance = float(
                distance_matrix[
                    receptor_df.index.get_loc(receptor_idx),
                    ligand_idx
                ]
            )

            cutoff = get_pair_cutoff(
                receptor_type,
                ligand_type
            )

            if cutoff is None:
                continue

            within_cutoff = distance <= cutoff

            compatible_count += 1

            if within_cutoff:
                within_cutoff_count += 1

            all_compatible_pairs.append({
                "PDB_ID": pdb_id,
                "CCD_COMPONENT": ccd_id,
                "receptor_feature_index": receptor_row["feature_index"],
                "receptor_type": receptor_type,
                "receptor_identity": receptor_row["identity"],
                "ligand_feature_index": ligand_record["ligand_feature_index"],
                "ligand_type": ligand_type,
                "ligand_atom_ids": ligand_record["ligand_atom_ids"],
                "distance_A": distance,
                "screening_cutoff_A": cutoff,
                "within_screening_cutoff": within_cutoff
            })

    ligand_summary.append({
        "PDB_ID": pdb_id,
        "CCD_COMPONENT": ccd_id,
        "ligand_feature_count": len(ligand_records),
        "compatible_feature_pairs": compatible_count,
        "pairs_within_screening_cutoff": within_cutoff_count
    })

# ------------------------------------------------------------
# 6. CREATE RESULTS TABLES
# ------------------------------------------------------------

ligand_features_df = pd.DataFrame(all_ligand_features)
compatible_pairs_df = pd.DataFrame(all_compatible_pairs)
ligand_summary_df = pd.DataFrame(ligand_summary)

if ligand_summary_df.empty:
    raise RuntimeError("No experimental ligand features were successfully analyzed.")

# Only pairs satisfying the declared distance cutoffs.
spatial_matches_df = compatible_pairs_df[
    compatible_pairs_df["within_screening_cutoff"]
].copy()

# ------------------------------------------------------------
# 7. EXPORT RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ligand_features_df.to_csv(
    OUTPUT_DIR / "experimental_ligand_pharmacophore_features.csv",
    index=False
)

compatible_pairs_df.to_csv(
    OUTPUT_DIR / "all_compatible_feature_pairs.csv",
    index=False
)

spatial_matches_df.to_csv(
    OUTPUT_DIR / "spatially_compatible_feature_pairs.csv",
    index=False
)

ligand_summary_df.to_csv(
    OUTPUT_DIR / "ligand_spatial_pharmacophore_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========== CELL 83G-O RESULTS ==========")

print("\nSuccessfully analyzed ligands:")
print(len(ligand_summary_df), "/ 10")

print("\nExtracted ligand pharmacophore features:")
print(len(ligand_features_df))

print("\nChemically compatible feature pairs:")
print(len(compatible_pairs_df))

print("\nPairs within screening cutoffs:")
print(len(spatial_matches_df))

print("\nPer-ligand summary:")
display(ligand_summary_df)

print("\nSpatially compatible feature pairs:")
display(
    spatial_matches_df.sort_values(
        ["PDB_ID", "distance_A"]
    ).head(30)
)

print("\nFeature types extracted from experimental ligands:")
print(ligand_features_df["ligand_type"].value_counts())

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nCell 83G-O completed.")

M1 receptor features loaded: 95
Conformer used: 1
Feature types:
normalized_type
HYDROPHOBIC    45
DONOR          22
ACCEPTOR       18
POSITIVE        5
AROMATIC        4
NEGATIVE        1
Name: count, dtype: int64

========== CELL 83G-O RESULTS ==========

Successfully analyzed ligands:
10 / 10

Extracted ligand pharmacophore features:
145

Chemically compatible feature pairs:
3272

Pairs within screening cutoffs:
144

Per-ligand summary:


,PDB_ID,CCD_COMPONENT,ligand_feature_count,compatible_feature_pairs,pairs_within_screening_cutoff
0,5KPP,6WZ,17,398,13
1,5KPQ,6X2,18,443,13
2,5WRZ,7U3,14,331,19
3,5WS1,7U9,14,256,13
4,7AAC,78P,12,216,13
5,7ONS,VKT,14,319,22
6,7ONT,VKQ,15,358,20
7,8JNZ,ERV,8,182,10
8,9ETQ,A1H63,14,290,11
9,9ILN,A1D9M,19,479,10



Spatially compatible feature pairs:


,PDB_ID,CCD_COMPONENT,receptor_feature_index,receptor_type,receptor_identity,ligand_feature_index,ligand_type,ligand_atom_ids,distance_A,screening_cutoff_A,within_screening_cutoff
83,5KPP,6WZ,22,AROMATIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,10,AROMATIC,"10,11,13,14,16,21",3.000321,5.0,True
102,5KPP,6WZ,26,HYDROPHOBIC,A|907||TYR|CE2,14,HYDROPHOBIC,16,3.244552,4.5,True
104,5KPP,6WZ,26,HYDROPHOBIC,A|907||TYR|CE2,16,HYDROPHOBIC,"16,17,18,19,20,21",3.341870,4.5,True
92,5KPP,6WZ,24,HYDROPHOBIC,A|907||TYR|CD2,14,HYDROPHOBIC,16,3.366326,4.5,True
87,5KPP,6WZ,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,14,HYDROPHOBIC,16,3.652353,4.5,True
96,5KPP,6WZ,25,HYDROPHOBIC,A|907||TYR|CE1,13,HYDROPHOBIC,8,3.816478,4.5,True
94,5KPP,6WZ,24,HYDROPHOBIC,A|907||TYR|CD2,16,HYDROPHOBIC,"16,17,18,19,20,21",3.981664,4.5,True
84,5KPP,6WZ,22,AROMATIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,11,AROMATIC,"16,17,18,19,20,21",4.023336,5.0,True
89,5KPP,6WZ,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,16,HYDROPHOBIC,"16,17,18,19,20,21",4.023336,4.5,True
107,5KPP,6WZ,27,HYDROPHOBIC,A|907||TYR|CG,14,HYDROPHOBIC,16,4.024445,4.5,True



Feature types extracted from experimental ligands:
ligand_type
HYDROPHOBIC    40
DONOR          34
ACCEPTOR       34
AROMATIC       25
POSITIVE       12
Name: count, dtype: int64

Output directory:
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-O completed.


In [122]:
# ============================================================
# CELL 83G-P
# RECURRING EXPERIMENTAL LIGAND–POCKET INTERACTION ANALYSIS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. VALIDATE INPUT
# ------------------------------------------------------------

if "spatial_matches_df" not in globals():
    raise NameError("Run Cell 83G-O first.")

if spatial_matches_df.empty:
    raise ValueError("No spatially compatible pairs available.")

required_columns = {
    "PDB_ID",
    "CCD_COMPONENT",
    "receptor_feature_index",
    "receptor_type",
    "receptor_identity",
    "ligand_feature_index",
    "ligand_type",
    "distance_A"
}

missing = required_columns - set(spatial_matches_df.columns)

if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 2. EXPAND RECEPTOR IDENTITIES INTO RESIDUES
# ------------------------------------------------------------

expanded_records = []

for _, row in spatial_matches_df.iterrows():

    identities = str(row["receptor_identity"]).split(";")

    for identity in identities:

        parts = identity.split("|")

        if len(parts) < 5:
            continue

        chain = parts[0]
        residue_number = parts[1]
        insertion_code = parts[2]
        residue_name = parts[3]

        residue_key = (
            f"{chain}|{residue_number}|"
            f"{insertion_code}|{residue_name}"
        )

        expanded_records.append({
            "PDB_ID": row["PDB_ID"],
            "CCD_COMPONENT": row["CCD_COMPONENT"],
            "ligand_id": (
                f'{row["PDB_ID"]}_{row["CCD_COMPONENT"]}'
            ),
            "residue_key": residue_key,
            "chain": chain,
            "residue_number": residue_number,
            "insertion_code": insertion_code,
            "residue_name": residue_name,
            "receptor_feature_index": row["receptor_feature_index"],
            "receptor_type": row["receptor_type"],
            "ligand_feature_index": row["ligand_feature_index"],
            "ligand_type": row["ligand_type"],
            "distance_A": float(row["distance_A"])
        })

expanded_df = pd.DataFrame(expanded_records)

if expanded_df.empty:
    raise ValueError("Could not extract residue identities.")

# ------------------------------------------------------------
# 3. REMOVE DUPLICATE REPRESENTATIONS
# Keep the minimum distance for the same ligand/residue/
# receptor feature/ligand feature combination.
# ------------------------------------------------------------

pair_columns = [
    "ligand_id",
    "residue_key",
    "receptor_feature_index",
    "receptor_type",
    "ligand_feature_index",
    "ligand_type"
]

expanded_df = (
    expanded_df
    .groupby(pair_columns, as_index=False)
    .agg(
        distance_A=("distance_A", "min"),
        PDB_ID=("PDB_ID", "first"),
        CCD_COMPONENT=("CCD_COMPONENT", "first"),
        chain=("chain", "first"),
        residue_number=("residue_number", "first"),
        insertion_code=("insertion_code", "first"),
        residue_name=("residue_name", "first")
    )
)

# ------------------------------------------------------------
# 4. RECURRENT RESIDUE SUMMARY
# ------------------------------------------------------------

residue_summary = (
    expanded_df
    .groupby(
        [
            "residue_key",
            "chain",
            "residue_number",
            "residue_name"
        ],
        as_index=False
    )
    .agg(
        distinct_ligands=("ligand_id", "nunique"),
        total_feature_pairs=("ligand_id", "size"),
        receptor_feature_types=(
            "receptor_type",
            lambda x: ", ".join(sorted(set(x)))
        ),
        median_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min")
    )
)

residue_summary["ligand_frequency_percent"] = (
    residue_summary["distinct_ligands"] / 10 * 100
)

residue_summary = residue_summary.sort_values(
    ["distinct_ligands", "total_feature_pairs"],
    ascending=False
)

# ------------------------------------------------------------
# 5. RECURRING FEATURE-TYPE COMBINATIONS
# ------------------------------------------------------------

interaction_summary = (
    expanded_df
    .groupby(
        ["receptor_type", "ligand_type"],
        as_index=False
    )
    .agg(
        distinct_ligands=("ligand_id", "nunique"),
        total_pairs=("ligand_id", "size"),
        median_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min")
    )
)

interaction_summary["ligand_frequency_percent"] = (
    interaction_summary["distinct_ligands"] / 10 * 100
)

interaction_summary = interaction_summary.sort_values(
    ["distinct_ligands", "total_pairs"],
    ascending=False
)

# ------------------------------------------------------------
# 6. RESIDUE + FEATURE-TYPE COMBINATION SUMMARY
# ------------------------------------------------------------

residue_interaction_summary = (
    expanded_df
    .groupby(
        [
            "residue_key",
            "receptor_type",
            "ligand_type"
        ],
        as_index=False
    )
    .agg(
        distinct_ligands=("ligand_id", "nunique"),
        total_pairs=("ligand_id", "size"),
        median_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min")
    )
)

residue_interaction_summary["ligand_frequency_percent"] = (
    residue_interaction_summary["distinct_ligands"] / 10 * 100
)

residue_interaction_summary = (
    residue_interaction_summary.sort_values(
        ["distinct_ligands", "total_pairs"],
        ascending=False
    )
)

# ------------------------------------------------------------
# 7. PER-LIGAND RESIDUE COVERAGE
# ------------------------------------------------------------

per_ligand_summary = (
    expanded_df
    .groupby(["PDB_ID", "CCD_COMPONENT"], as_index=False)
    .agg(
        unique_residues=("residue_key", "nunique"),
        feature_pairs=("residue_key", "size"),
        median_distance_A=("distance_A", "median")
    )
)

# ------------------------------------------------------------
# 8. EXPORT RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

expanded_df.to_csv(
    OUTPUT_DIR / "expanded_residue_feature_pairs.csv",
    index=False
)

residue_summary.to_csv(
    OUTPUT_DIR / "recurring_residue_summary.csv",
    index=False
)

interaction_summary.to_csv(
    OUTPUT_DIR / "recurring_feature_type_summary.csv",
    index=False
)

residue_interaction_summary.to_csv(
    OUTPUT_DIR / "residue_feature_type_summary.csv",
    index=False
)

per_ligand_summary.to_csv(
    OUTPUT_DIR / "per_ligand_residue_coverage.csv",
    index=False
)

# ------------------------------------------------------------
# 9. DISPLAY
# ------------------------------------------------------------

print("========== CELL 83G-P RESULTS ==========")

print("\nDistinct receptor residues:")
print(residue_summary["residue_key"].nunique())

print("\nRecurring receptor residues:")
display(residue_summary)

print("\nRecurring feature-type combinations:")
display(interaction_summary)

print("\nResidue-specific feature combinations:")
display(residue_interaction_summary)

print("\nPer-ligand residue coverage:")
display(per_ligand_summary)

print("\nSaved results to:")
print(OUTPUT_DIR)

print("\nCell 83G-P completed.")

========== CELL 83G-P RESULTS ==========

Distinct receptor residues:
1

Recurring receptor residues:


,residue_key,chain,residue_number,residue_name,distinct_ligands,total_feature_pairs,receptor_feature_types,median_distance_A,minimum_distance_A,ligand_frequency_percent
0,A|907||TYR,A,907,TYR,10,144,"ACCEPTOR, AROMATIC, DONOR, HYDROPHOBIC",3.850223,2.565842,100.0



Recurring feature-type combinations:


,receptor_type,ligand_type,distinct_ligands,total_pairs,median_distance_A,minimum_distance_A,ligand_frequency_percent
3,HYDROPHOBIC,HYDROPHOBIC,10,118,3.916692,3.162338,100.0
1,AROMATIC,AROMATIC,10,19,3.684652,3.000321,100.0
0,ACCEPTOR,DONOR,3,5,3.117775,2.565842,30.0
2,DONOR,ACCEPTOR,2,2,2.890282,2.662788,20.0



Residue-specific feature combinations:


,residue_key,receptor_type,ligand_type,distinct_ligands,total_pairs,median_distance_A,minimum_distance_A,ligand_frequency_percent
3,A|907||TYR,HYDROPHOBIC,HYDROPHOBIC,10,118,3.916692,3.162338,100.0
1,A|907||TYR,AROMATIC,AROMATIC,10,19,3.684652,3.000321,100.0
0,A|907||TYR,ACCEPTOR,DONOR,3,5,3.117775,2.565842,30.0
2,A|907||TYR,DONOR,ACCEPTOR,2,2,2.890282,2.662788,20.0



Per-ligand residue coverage:


,PDB_ID,CCD_COMPONENT,unique_residues,feature_pairs,median_distance_A
0,5KPP,6WZ,1,13,3.981664
1,5KPQ,6X2,1,13,3.756890
2,5WRZ,7U3,1,19,3.983986
3,5WS1,7U9,1,13,3.632673
4,7AAC,78P,1,13,3.607457
5,7ONS,VKT,1,22,3.816181
6,7ONT,VKQ,1,20,3.919196
7,8JNZ,ERV,1,10,3.888079
8,9ETQ,A1H63,1,11,4.028053
9,9ILN,A1D9M,1,10,3.872239



Saved results to:
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P completed.


In [123]:
# ============================================================
# CELL 83G-P1
# AUDIT RECEPTOR FEATURE IDENTITIES AND SPATIAL MATCHES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. VALIDATE INPUTS
# ------------------------------------------------------------

assert "receptor_features_by_conformer" in globals()
assert "spatial_matches_df" in globals()

receptor_df = receptor_features_by_conformer[1].copy()
matches_df = spatial_matches_df.copy()

print("Total receptor features:", len(receptor_df))
print("Total spatial matches:", len(matches_df))

# ------------------------------------------------------------
# 2. EXTRACT RESIDUE IDENTITIES
# ------------------------------------------------------------

def extract_residues(identity):

    residues = set()

    for atom_identity in str(identity).split(";"):

        parts = atom_identity.split("|")

        if len(parts) >= 5:

            residue = (
                f"{parts[0]}|{parts[1]}|"
                f"{parts[2]}|{parts[3]}"
            )

            residues.add(residue)

    return sorted(residues)


receptor_df["residues"] = (
    receptor_df["identity"].apply(extract_residues)
)

receptor_df["residue_count"] = (
    receptor_df["residues"].apply(len)
)

# ------------------------------------------------------------
# 3. INVENTORY ALL 95 FEATURES
# ------------------------------------------------------------

inventory = []

for _, row in receptor_df.iterrows():

    for residue in row["residues"]:

        inventory.append({
            "feature_index": row["feature_index"],
            "feature_type": row["normalized_type"],
            "residue": residue,
            "identity": row["identity"]
        })

inventory_df = pd.DataFrame(inventory)

print("\nDistinct residues represented in all M1 features:")
print(inventory_df["residue"].nunique())

print("\nFeature counts by residue:")
display(
    inventory_df.groupby("residue")
    .agg(
        feature_count=("feature_index", "nunique"),
        feature_types=("feature_type",
                       lambda x: ", ".join(sorted(set(x))))
    )
    .sort_values("feature_count", ascending=False)
)

# ------------------------------------------------------------
# 4. IDENTIFY MATCHED RECEPTOR FEATURE INDICES
# ------------------------------------------------------------

matched_indices = set(
    matches_df["receptor_feature_index"].astype(int)
)

receptor_df["matched"] = (
    receptor_df["feature_index"].astype(int)
    .isin(matched_indices)
)

print("\nMatched receptor feature indices:", len(matched_indices))
print("Unmatched receptor features:", (~receptor_df["matched"]).sum())

print("\nMatched receptor features:")
display(
    receptor_df.loc[
        receptor_df["matched"],
        ["feature_index", "normalized_type", "identity"]
    ]
)

# ------------------------------------------------------------
# 5. COUNT MATCHES PER RECEPTOR FEATURE
# ------------------------------------------------------------

feature_match_counts = (
    matches_df.groupby(
        [
            "receptor_feature_index",
            "receptor_type",
            "receptor_identity"
        ],
        as_index=False
    )
    .agg(
        total_pairs=("distance_A", "size"),
        distinct_ligands=("PDB_ID", "nunique"),
        minimum_distance_A=("distance_A", "min"),
        median_distance_A=("distance_A", "median")
    )
    .sort_values(
        ["distinct_ligands", "total_pairs"],
        ascending=False
    )
)

print("\nMatches per receptor feature:")
display(feature_match_counts)

# ------------------------------------------------------------
# 6. CHECK RESIDUE DISTRIBUTION OF MATCHED FEATURES
# ------------------------------------------------------------

matched_residue_records = []

for _, row in feature_match_counts.iterrows():

    for residue in extract_residues(row["receptor_identity"]):

        matched_residue_records.append({
            "residue": residue,
            "feature_index": row["receptor_feature_index"],
            "distinct_ligands": row["distinct_ligands"],
            "total_pairs": row["total_pairs"]
        })

matched_residue_df = pd.DataFrame(
    matched_residue_records
)

print("\nMatched feature distribution by residue:")
display(
    matched_residue_df.groupby("residue")
    .agg(
        matched_features=("feature_index", "nunique"),
        total_pairs=("total_pairs", "sum"),
        maximum_ligand_coverage=("distinct_ligands", "max")
    )
    .sort_values("total_pairs", ascending=False)
)

# ------------------------------------------------------------
# 7. EXPORT AUDIT
# ------------------------------------------------------------

from pathlib import Path

AUDIT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

inventory_df.to_csv(
    AUDIT_DIR / "receptor_feature_identity_inventory.csv",
    index=False
)

feature_match_counts.to_csv(
    AUDIT_DIR / "receptor_feature_match_counts.csv",
    index=False
)

print("\nAudit files saved:", AUDIT_DIR)
print("\nCell 83G-P1 completed.")

Total receptor features: 95
Total spatial matches: 144


KeyError: 'normalized_type'

In [124]:
# ============================================================
# CELL 83G-P1 — CORRECTED RECEPTOR FEATURE MAPPING AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD INPUTS
# ------------------------------------------------------------

receptor_df = receptor_features_by_conformer[1].copy()
matches_df = spatial_matches_df.copy()

# Recreate normalized_type because the original DataFrame
# does not contain the column created locally in Cell 83G-O.

feature_code_map = {
    0: "DONOR",
    1: "ACCEPTOR",
    2: "HYDROPHOBIC",
    3: "AROMATIC",
    4: "POSITIVE",
    5: "NEGATIVE"
}

def normalize_feature_type(value):

    if isinstance(value, (int, np.integer)):
        return feature_code_map.get(int(value), str(value).upper())

    value = str(value).strip().upper()

    if value.isdigit():
        return feature_code_map.get(int(value), value)

    return value

type_column = next(
    (c for c in ["type", "feature_type", "feature"] if c in receptor_df.columns),
    None
)

if type_column is None:
    raise ValueError("Receptor feature type column not found.")

receptor_df["normalized_type"] = (
    receptor_df[type_column].apply(normalize_feature_type)
)

print("Total receptor features:", len(receptor_df))
print("Total spatial matches:", len(matches_df))

# ------------------------------------------------------------
# 2. EXTRACT RESIDUE IDENTITIES
# ------------------------------------------------------------

def extract_residues(identity):

    residues = set()

    for atom_identity in str(identity).split(";"):

        parts = atom_identity.split("|")

        if len(parts) >= 5:

            residues.add(
                f"{parts[0]}|{parts[1]}|{parts[2]}|{parts[3]}"
            )

    return sorted(residues)


receptor_df["residues"] = (
    receptor_df["identity"].apply(extract_residues)
)

# ------------------------------------------------------------
# 3. INVENTORY ALL RECEPTOR FEATURES
# ------------------------------------------------------------

inventory = []

for _, row in receptor_df.iterrows():

    for residue in row["residues"]:

        inventory.append({
            "feature_index": row["feature_index"],
            "feature_type": row["normalized_type"],
            "residue": residue,
            "identity": row["identity"]
        })

inventory_df = pd.DataFrame(inventory)

print("\nDistinct residues represented:")
print(inventory_df["residue"].nunique())

print("\nFeature counts by residue:")

display(
    inventory_df.groupby("residue")
    .agg(
        feature_count=("feature_index", "nunique"),
        feature_types=(
            "feature_type",
            lambda x: ", ".join(sorted(set(x)))
        )
    )
    .sort_values("feature_count", ascending=False)
)

# ------------------------------------------------------------
# 4. MATCHED RECEPTOR FEATURES
# ------------------------------------------------------------

matched_indices = set(
    matches_df["receptor_feature_index"].astype(int)
)

receptor_df["matched"] = (
    receptor_df["feature_index"].astype(int).isin(matched_indices)
)

print("\nMatched receptor feature indices:", len(matched_indices))
print("Unmatched receptor features:", (~receptor_df["matched"]).sum())

print("\nMatched receptor features:")

display(
    receptor_df.loc[
        receptor_df["matched"],
        ["feature_index", "normalized_type", "identity"]
    ]
)

# ------------------------------------------------------------
# 5. MATCH COUNTS PER RECEPTOR FEATURE
# ------------------------------------------------------------

feature_match_counts = (
    matches_df.groupby(
        [
            "receptor_feature_index",
            "receptor_type",
            "receptor_identity"
        ],
        as_index=False
    )
    .agg(
        total_pairs=("distance_A", "size"),
        distinct_ligands=("PDB_ID", "nunique"),
        minimum_distance_A=("distance_A", "min"),
        median_distance_A=("distance_A", "median")
    )
    .sort_values(
        ["distinct_ligands", "total_pairs"],
        ascending=False
    )
)

print("\nMatches per receptor feature:")
display(feature_match_counts)

# ------------------------------------------------------------
# 6. MATCHED FEATURES BY RESIDUE
# ------------------------------------------------------------

matched_records = []

for _, row in feature_match_counts.iterrows():

    for residue in extract_residues(row["receptor_identity"]):

        matched_records.append({
            "residue": residue,
            "feature_index": row["receptor_feature_index"],
            "distinct_ligands": row["distinct_ligands"],
            "total_pairs": row["total_pairs"]
        })

matched_residue_df = pd.DataFrame(matched_records)

print("\nMatched feature distribution by residue:")

display(
    matched_residue_df.groupby("residue")
    .agg(
        matched_features=("feature_index", "nunique"),
        total_pairs=("total_pairs", "sum"),
        maximum_ligand_coverage=("distinct_ligands", "max")
    )
    .sort_values("total_pairs", ascending=False)
)

# ------------------------------------------------------------
# 7. EXPORT
# ------------------------------------------------------------

AUDIT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

inventory_df.to_csv(
    AUDIT_DIR / "receptor_feature_identity_inventory.csv",
    index=False
)

feature_match_counts.to_csv(
    AUDIT_DIR / "receptor_feature_match_counts.csv",
    index=False
)

matched_residue_df.to_csv(
    AUDIT_DIR / "matched_feature_residue_records.csv",
    index=False
)

print("\nAudit files saved:", AUDIT_DIR)
print("\nCell 83G-P1 completed.")

Total receptor features: 95
Total spatial matches: 144

Distinct residues represented:
11

Feature counts by residue:


,feature_count,feature_types
residue,,
A|986||TYR,15,"ACCEPTOR, AROMATIC, DONOR, HYDROPHOBIC"
A|907||TYR,12,"ACCEPTOR, AROMATIC, DONOR, HYDROPHOBIC"
A|988||GLU,11,"ACCEPTOR, DONOR, HYDROPHOBIC, NEGATIVE"
A|985||LEU,9,"ACCEPTOR, DONOR, HYDROPHOBIC"
A|984||LEU,9,"ACCEPTOR, DONOR, HYDROPHOBIC"
A|826||HIS,8,"ACCEPTOR, AROMATIC, DONOR, HYDROPHOBIC, POSITIVE"
A|909||HIS,8,"ACCEPTOR, AROMATIC, DONOR, HYDROPHOBIC, POSITIVE"
A|983||SER,7,"ACCEPTOR, DONOR, HYDROPHOBIC"
A|903||LYS,7,"ACCEPTOR, DONOR, HYDROPHOBIC, POSITIVE"



Matched receptor feature indices: 9
Unmatched receptor features: 86

Matched receptor features:


,feature_index,normalized_type,identity
21,21,HYDROPHOBIC,A|907||TYR|CD1
22,22,AROMATIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...
23,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...
24,24,HYDROPHOBIC,A|907||TYR|CD2
25,25,HYDROPHOBIC,A|907||TYR|CE1
26,26,HYDROPHOBIC,A|907||TYR|CE2
27,27,HYDROPHOBIC,A|907||TYR|CG
30,30,ACCEPTOR,A|907||TYR|OH
31,31,DONOR,A|907||TYR|OH



Matches per receptor feature:


,receptor_feature_index,receptor_type,receptor_identity,total_pairs,distinct_ligands,minimum_distance_A,median_distance_A
3,24,HYDROPHOBIC,A|907||TYR|CD2,23,10,3.316134,3.948045
5,26,HYDROPHOBIC,A|907||TYR|CE2,23,10,3.162338,3.476687
4,25,HYDROPHOBIC,A|907||TYR|CE1,22,10,3.469077,4.290425
2,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,21,10,3.328064,3.789024
1,22,AROMATIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,19,10,3.000321,3.684652
6,27,HYDROPHOBIC,A|907||TYR|CG,16,10,3.843338,4.076582
0,21,HYDROPHOBIC,A|907||TYR|CD1,13,7,3.377388,3.995035
7,30,ACCEPTOR,A|907||TYR|OH,5,3,2.565842,3.117775
8,31,DONOR,A|907||TYR|OH,2,2,2.662788,2.890282



Matched feature distribution by residue:


,matched_features,total_pairs,maximum_ligand_coverage
residue,,,
A|907||TYR,9,144,10



Audit files saved: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P1 completed.


In [125]:
# ============================================================
# CELL 83G-P2
# ALL-RECEPTOR-FEATURE DISTANCE AUDIT
# ============================================================

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD AND VALIDATE DATA
# ------------------------------------------------------------

receptor_df = receptor_features_by_conformer[1].copy()
ligand_df = ligand_features_df.copy()

required_receptor = {
    "feature_index", "identity", "x", "y", "z"
}

required_ligand = {
    "PDB_ID", "CCD_COMPONENT",
    "ligand_feature_index", "ligand_type",
    "x", "y", "z"
}

assert required_receptor.issubset(receptor_df.columns)
assert required_ligand.issubset(ligand_df.columns)

# Reconstruct receptor feature types.

feature_code_map = {
    0: "DONOR",
    1: "ACCEPTOR",
    2: "HYDROPHOBIC",
    3: "AROMATIC",
    4: "POSITIVE",
    5: "NEGATIVE"
}

type_column = next(
    c for c in ["type", "feature_type", "feature"]
    if c in receptor_df.columns
)

def normalize_type(value):
    if isinstance(value, (int, np.integer)):
        return feature_code_map.get(int(value), str(value).upper())

    value = str(value).strip().upper()

    if value.isdigit():
        return feature_code_map.get(int(value), value)

    return value

receptor_df["receptor_type"] = (
    receptor_df[type_column].apply(normalize_type)
)

# ------------------------------------------------------------
# 2. CHEMICAL COMPATIBILITY RULES
# ------------------------------------------------------------

compatible_pairs = {
    ("HYDROPHOBIC", "HYDROPHOBIC"),
    ("AROMATIC", "AROMATIC"),
    ("DONOR", "ACCEPTOR"),
    ("ACCEPTOR", "DONOR"),
    ("POSITIVE", "NEGATIVE"),
    ("NEGATIVE", "POSITIVE")
}

def get_cutoff(rt, lt):

    if rt == "HYDROPHOBIC":
        return 4.5

    if rt == "AROMATIC":
        return 5.0

    if {rt, lt} == {"DONOR", "ACCEPTOR"}:
        return 3.5

    if {rt, lt} == {"POSITIVE", "NEGATIVE"}:
        return 4.0

    return np.nan

# ------------------------------------------------------------
# 3. COMPARE ALL 95 FEATURES AGAINST EACH LIGAND
# ------------------------------------------------------------

audit_records = []

for (pdb_id, ccd_id), ligand_group in ligand_df.groupby(
    ["PDB_ID", "CCD_COMPONENT"]
):

    receptor_xyz = receptor_df[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    ligand_xyz = ligand_group[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    distances = cdist(receptor_xyz, ligand_xyz)

    ligand_records = ligand_group.to_dict("records")

    for i, (_, receptor_row) in enumerate(receptor_df.iterrows()):

        closest_j = int(np.argmin(distances[i]))

        for j, ligand_record in enumerate(ligand_records):

            rt = receptor_row["receptor_type"]
            lt = ligand_record["ligand_type"]

            distance = float(distances[i, j])

            is_compatible = (rt, lt) in compatible_pairs

            cutoff = get_cutoff(rt, lt)

            audit_records.append({
                "PDB_ID": pdb_id,
                "CCD_COMPONENT": ccd_id,
                "receptor_feature_index": receptor_row["feature_index"],
                "receptor_identity": receptor_row["identity"],
                "receptor_type": rt,
                "ligand_feature_index": ligand_record["ligand_feature_index"],
                "ligand_type": lt,
                "distance_A": distance,
                "chemically_compatible": is_compatible,
                "screening_cutoff_A": cutoff,
                "within_cutoff": (
                    bool(distance <= cutoff)
                    if np.isfinite(cutoff)
                    else False
                ),
                "is_nearest_ligand_feature": (j == closest_j)
            })

audit_df = pd.DataFrame(audit_records)

# ------------------------------------------------------------
# 4. MINIMUM DISTANCE PER RECEPTOR FEATURE AND LIGAND
# ------------------------------------------------------------

nearest_df = (
    audit_df.sort_values("distance_A")
    .groupby(
        [
            "PDB_ID",
            "CCD_COMPONENT",
            "receptor_feature_index"
        ],
        as_index=False
    )
    .first()
)

# ------------------------------------------------------------
# 5. SUMMARIZE ALL RECEPTOR FEATURES
# ------------------------------------------------------------

feature_summary = (
    nearest_df.groupby(
        [
            "receptor_feature_index",
            "receptor_identity",
            "receptor_type"
        ],
        as_index=False
    )
    .agg(
        ligands_nearby=("PDB_ID", "nunique"),
        median_nearest_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min"),
        compatible_ligands=(
            "chemically_compatible", "sum"
        ),
        ligands_within_cutoff=("within_cutoff", "sum")
    )
    .sort_values(
        ["ligands_within_cutoff", "minimum_distance_A"],
        ascending=[False, True]
    )
)

# ------------------------------------------------------------
# 6. RESIDUE-LEVEL SUMMARY
# ------------------------------------------------------------

def residue_key(identity):

    first = str(identity).split(";")[0].split("|")

    if len(first) < 5:
        return "UNKNOWN"

    return "|".join(first[:4])

nearest_df["residue"] = (
    nearest_df["receptor_identity"].apply(residue_key)
)

residue_summary = (
    nearest_df.groupby("residue", as_index=False)
    .agg(
        receptor_features=("receptor_feature_index", "nunique"),
        ligand_observations=("PDB_ID", "nunique"),
        median_nearest_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min"),
        compatible_observations=(
            "chemically_compatible", "sum"
        ),
        observations_within_cutoff=("within_cutoff", "sum")
    )
    .sort_values("median_nearest_distance_A")
)

# ------------------------------------------------------------
# 7. EXPORT
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(
    OUTPUT_DIR / "all_receptor_ligand_feature_distances.csv",
    index=False
)

feature_summary.to_csv(
    OUTPUT_DIR / "all_receptor_feature_distance_summary.csv",
    index=False
)

residue_summary.to_csv(
    OUTPUT_DIR / "all_residue_distance_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. DISPLAY
# ------------------------------------------------------------

print("========== CELL 83G-P2 RESULTS ==========")

print("\nAll receptor-ligand feature comparisons:", len(audit_df))

print("\nReceptor feature summary:")
display(feature_summary)

print("\nResidue-level distance summary:")
display(residue_summary)

print("\nSaved to:", OUTPUT_DIR)

print("\nCell 83G-P2 completed.")

========== CELL 83G-P2 RESULTS ==========

All receptor-ligand feature comparisons: 13775

Receptor feature summary:


,receptor_feature_index,receptor_identity,receptor_type,ligands_nearby,median_nearest_distance_A,minimum_distance_A,compatible_ligands,ligands_within_cutoff
21,21,A|907||TYR|CD1,HYDROPHOBIC,10,3.439242,2.619213,1,10
22,22,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,AROMATIC,10,3.200094,2.718099,3,10
23,23,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,HYDROPHOBIC,10,3.200094,2.718099,1,10
25,25,A|907||TYR|CE1,HYDROPHOBIC,10,3.021155,2.725016,2,10
27,27,A|907||TYR|CG,HYDROPHOBIC,10,3.365240,2.973762,0,10
...,...,...,...,...,...,...,...,...
3,3,A|826||HIS|N,DONOR,10,14.062743,13.430427,2,0
93,93,B|721||SER|OG,ACCEPTOR,10,15.026753,13.614449,1,0
94,94,B|721||SER|OG,DONOR,10,15.026753,13.614449,2,0
91,91,B|721||SER|N,DONOR,10,15.333312,13.716915,1,0



Residue-level distance summary:


,residue,receptor_features,ligand_observations,median_nearest_distance_A,minimum_distance_A,compatible_observations,observations_within_cutoff
3,A|907||TYR,12,10,3.455420,2.565842,26,83
1,A|903||LYS,7,10,5.575117,3.611136,28,3
9,A|988||GLU,11,10,5.999644,3.336404,37,6
2,A|906||ASN,5,10,7.787908,6.636268,29,0
4,A|909||HIS,8,10,9.004021,6.658494,18,0
8,A|986||TYR,15,10,10.266823,6.248331,70,0
6,A|984||LEU,9,10,10.277276,6.897137,39,0
0,A|826||HIS,8,10,10.511479,9.268179,14,0
7,A|985||LEU,9,10,12.187436,9.332792,41,0
5,A|983||SER,7,10,13.114621,10.829933,24,0



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P2 completed.


In [126]:
# ============================================================
# CELL 83G-P3
# CORRECTED RESIDUE-LEVEL SPATIAL INTERACTION AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. VALIDATE INPUT
# ------------------------------------------------------------

if "audit_df" not in globals():
    raise NameError("Run Cell 83G-P2 first.")

df = audit_df.copy()

# ------------------------------------------------------------
# 2. IDENTIFY LIGAND AND RECEPTOR FEATURE PAIRS
# ------------------------------------------------------------

df["ligand_id"] = (
    df["PDB_ID"].astype(str)
    + "_"
    + df["CCD_COMPONENT"].astype(str)
)

df["compatible_within_cutoff"] = (
    df["chemically_compatible"]
    & df["within_cutoff"]
)

# ------------------------------------------------------------
# 3. AGGREGATE EACH RECEPTOR FEATURE PER LIGAND
# ------------------------------------------------------------

feature_ligand_summary = (
    df.groupby(
        [
            "ligand_id",
            "PDB_ID",
            "CCD_COMPONENT",
            "receptor_feature_index",
            "receptor_identity",
            "receptor_type"
        ],
        as_index=False
    )
    .agg(
        minimum_any_distance_A=("distance_A", "min"),
        minimum_compatible_distance_A=(
            "distance_A",
            lambda x: np.nan
        ),
        has_compatible_pair=("chemically_compatible", "any"),
        has_compatible_pair_within_cutoff=(
            "compatible_within_cutoff", "any"
        )
    )
)

# Calculate minimum distance among chemically compatible pairs.
compatible_min = (
    df[df["chemically_compatible"]]
    .groupby(
        [
            "ligand_id",
            "receptor_feature_index"
        ])["distance_A"]
    .min()
    .rename("minimum_compatible_distance_A")
    .reset_index()
)

feature_ligand_summary = (
    feature_ligand_summary.drop(
        columns="minimum_compatible_distance_A"
    )
    .merge(
        compatible_min,
        on=["ligand_id", "receptor_feature_index"],
        how="left"
    )
)

# ------------------------------------------------------------
# 4. EXPAND RECEPTOR FEATURES INTO RESIDUES
# ------------------------------------------------------------

records = []

for _, row in feature_ligand_summary.iterrows():

    identities = str(row["receptor_identity"]).split(";")

    residues = set()

    for identity in identities:

        parts = identity.split("|")

        if len(parts) >= 5:

            residues.add(
                "|".join(parts[:4])
            )

    for residue in residues:

        records.append({
            "ligand_id": row["ligand_id"],
            "PDB_ID": row["PDB_ID"],
            "CCD_COMPONENT": row["CCD_COMPONENT"],
            "residue": residue,
            "receptor_feature_index": row["receptor_feature_index"],
            "receptor_type": row["receptor_type"],
            "minimum_any_distance_A": row["minimum_any_distance_A"],
            "minimum_compatible_distance_A": row["minimum_compatible_distance_A"],
            "has_compatible_pair": row["has_compatible_pair"],
            "has_compatible_pair_within_cutoff": row["has_compatible_pair_within_cutoff"]
        })

residue_feature_df = pd.DataFrame(records)

# ------------------------------------------------------------
# 5. AGGREGATE PER LIGAND AND RESIDUE
# Avoid counting multiple features as multiple ligands.
# ------------------------------------------------------------

residue_ligand_summary = (
    residue_feature_df.groupby(
        ["ligand_id", "PDB_ID", "CCD_COMPONENT", "residue"],
        as_index=False
    )
    .agg(
        minimum_any_distance_A=("minimum_any_distance_A", "min"),
        minimum_compatible_distance_A=(
            "minimum_compatible_distance_A", "min"
        ),
        has_compatible_pair=("has_compatible_pair", "any"),
        has_compatible_pair_within_cutoff=(
            "has_compatible_pair_within_cutoff", "any"
        )
    )
)

# ------------------------------------------------------------
# 6. FINAL RESIDUE SUMMARY
# ------------------------------------------------------------

final_residue_summary = (
    residue_ligand_summary.groupby(
        "residue",
        as_index=False
    )
    .agg(
        distinct_ligands=("ligand_id", "nunique"),
        ligands_with_compatible_pairs=(
            "has_compatible_pair", "sum"
        ),
        ligands_within_screening_cutoff=(
            "has_compatible_pair_within_cutoff", "sum"
        ),
        median_nearest_distance_A=(
            "minimum_any_distance_A", "median"
        ),
        median_compatible_distance_A=(
            "minimum_compatible_distance_A", "median"
        ),
        minimum_compatible_distance_A=(
            "minimum_compatible_distance_A", "min"
        )
    )
)

final_residue_summary["compatible_ligand_percent"] = (
    final_residue_summary["ligands_with_compatible_pairs"] / 10 * 100
)

final_residue_summary["within_cutoff_ligand_percent"] = (
    final_residue_summary["ligands_within_screening_cutoff"] / 10 * 100
)

final_residue_summary = final_residue_summary.sort_values(
    ["ligands_within_screening_cutoff",
     "ligands_with_compatible_pairs",
     "median_nearest_distance_A"],
    ascending=[False, False, True]
)

# ------------------------------------------------------------
# 7. EXPORT
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

residue_ligand_summary.to_csv(
    OUTPUT_DIR / "corrected_residue_ligand_summary.csv",
    index=False
)

final_residue_summary.to_csv(
    OUTPUT_DIR / "corrected_residue_interaction_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. DISPLAY
# ------------------------------------------------------------

print("========== CELL 83G-P3 RESULTS ==========")

print("\nDistinct receptor residues:")
print(final_residue_summary["residue"].nunique())

print("\nCorrected residue-level summary:")
display(final_residue_summary)

print("\nPer-ligand residue-level observations:")
display(residue_ligand_summary)

print("\nSaved to:", OUTPUT_DIR)

print("\nCell 83G-P3 completed.")

========== CELL 83G-P3 RESULTS ==========

Distinct receptor residues:
11

Corrected residue-level summary:


,residue,distinct_ligands,ligands_with_compatible_pairs,ligands_within_screening_cutoff,median_nearest_distance_A,median_compatible_distance_A,minimum_compatible_distance_A,compatible_ligand_percent,within_cutoff_ligand_percent
3,A|907||TYR,10,10,10,2.778594,3.092104,2.565842,100.0,100.0
1,A|903||LYS,10,10,0,4.743612,4.743612,3.682668,100.0,0.0
9,A|988||GLU,10,10,0,5.136014,5.333654,3.719068,100.0,0.0
2,A|906||ASN,10,10,0,6.890229,6.890229,6.636268,100.0,0.0
4,A|909||HIS,10,10,0,7.412733,8.302932,6.658494,100.0,0.0
6,A|984||LEU,10,10,0,8.474027,8.474027,8.098072,100.0,0.0
8,A|986||TYR,10,10,0,8.572544,8.901541,7.995127,100.0,0.0
0,A|826||HIS,10,10,0,10.068645,10.471572,9.268179,100.0,0.0
7,A|985||LEU,10,10,0,11.346673,11.449557,9.372936,100.0,0.0
5,A|983||SER,10,10,0,12.278277,12.658453,11.240160,100.0,0.0



Per-ligand residue-level observations:


,ligand_id,PDB_ID,CCD_COMPONENT,residue,minimum_any_distance_A,minimum_compatible_distance_A,has_compatible_pair,has_compatible_pair_within_cutoff
0,5KPP_6WZ,5KPP,6WZ,A|826||HIS,10.239140,10.468387,True,False
1,5KPP_6WZ,5KPP,6WZ,A|903||LYS,4.853528,5.179117,True,False
2,5KPP_6WZ,5KPP,6WZ,A|906||ASN,6.830460,6.830460,True,False
3,5KPP_6WZ,5KPP,6WZ,A|907||TYR,2.670575,3.000321,True,True
4,5KPP_6WZ,5KPP,6WZ,A|909||HIS,6.658494,6.658494,True,False
...,...,...,...,...,...,...,...,...
105,9ILN_A1D9M,9ILN,A1D9M,A|984||LEU,9.035110,9.035110,True,False
106,9ILN_A1D9M,9ILN,A1D9M,A|985||LEU,11.348229,11.348229,True,False
107,9ILN_A1D9M,9ILN,A1D9M,A|986||TYR,8.507301,8.694604,True,False
108,9ILN_A1D9M,9ILN,A1D9M,A|988||GLU,5.412007,5.412007,True,False



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P3 completed.


In [128]:
# ============================================================
# CELL 83G-P4
# TYR907 ATOM-LEVEL CONTACT VALIDATION
# ============================================================

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from pathlib import Path

# 1. Load existing data

ligand_df = pose_coordinates_df.copy()
protein_df = protein_atoms_by_conformer[1].copy()

# 2. Select TYR907 in M1 reference structure

tyr = protein_df[
    (protein_df["residue_name"].astype(str).str.upper() == "TYR") &
    (protein_df["residue_number"].astype(str).str.strip() == "907") &
    (protein_df["chain"].astype(str).str.strip() == "A")
].copy()

if tyr.empty:
    raise ValueError("TYR907 not found in M1 reference structure.")

# Heavy atoms only
heavy_elements = ["C", "N", "O", "S", "P"]

tyr = tyr[
    tyr["element"].astype(str).str.upper().isin(heavy_elements)
].copy()

print("TYR907 heavy atoms:", len(tyr))

display(tyr)

# 3. Calculate ligand–TYR907 distances

contact_records = []
summary_records = []

for (pdb_id, comp_id), group in ligand_df.groupby(
    ["pdb_id", "comp_id"]
):

    ligand_atoms = group[
        group["element"].astype(str).str.upper().isin(heavy_elements)
    ].copy()

    ligand_xyz = ligand_atoms[
        ["x_m1", "y_m1", "z_m1"]
    ].to_numpy(dtype=float)

    tyr_xyz = tyr[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    if not np.isfinite(ligand_xyz).all():
        raise ValueError(f"Invalid ligand coordinates: {pdb_id}")

    distances = cdist(ligand_xyz, tyr_xyz)

    # Record atom pairs within 5 Angstrom
    for i, j in np.argwhere(distances <= 5.0):

        contact_records.append({
            "pdb_id": pdb_id,
            "comp_id": comp_id,
            "ligand_atom_index": ligand_atoms.iloc[i]["atom_index"],
            "ligand_element": ligand_atoms.iloc[i]["element"],
            "TYR907_atom": tyr.iloc[j]["atom_name"],
            "TYR907_element": tyr.iloc[j]["element"],
            "distance_A": distances[i, j]
        })

    # Per-ligand summary
    summary_records.append({
        "pdb_id": pdb_id,
        "comp_id": comp_id,
        "minimum_distance_A": distances.min(),
        "atom_pairs_le_3_5A": int((distances <= 3.5).sum()),
        "atom_pairs_le_4A": int((distances <= 4.0).sum()),
        "atom_pairs_le_5A": int((distances <= 5.0).sum()),
        "unique_ligand_atoms_within_4A": int(
            (distances <= 4.0).any(axis=1).sum()
        ),
        "unique_TYR907_atoms_within_4A": int(
            (distances <= 4.0).any(axis=0).sum()
        )
    })

contacts_df = pd.DataFrame(contact_records)
summary_df = pd.DataFrame(summary_records)

# 4. Recurring atom-pair analysis

atom_pair_summary = (
    contacts_df.groupby(
        ["ligand_atom_index", "TYR907_atom"],
        as_index=False
    )
    .agg(
        ligand_count=("pdb_id", "nunique"),
        median_distance_A=("distance_A", "median"),
        minimum_distance_A=("distance_A", "min")
    )
    .sort_values(
        ["ligand_count", "median_distance_A"],
        ascending=[False, True]
    )
)

# 5. Export

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contacts_df.to_csv(
    OUTPUT_DIR / "TYR907_atom_contacts.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "TYR907_per_ligand_atom_summary.csv",
    index=False
)

atom_pair_summary.to_csv(
    OUTPUT_DIR / "TYR907_recurring_atom_pairs.csv",
    index=False
)

# 6. Display

print("\n========== CELL 83G-P4 RESULTS ==========")

print("Ligands analyzed:", len(summary_df))
print("Total atom pairs within 5 A:", len(contacts_df))

print("\nPer-ligand summary:")
display(summary_df)

print("\nRecurring atom contacts:")
display(atom_pair_summary.head(25))

print("\nSaved to:", OUTPUT_DIR)

print("\nCell 83G-P4 completed.")

TYR907 heavy atoms: 12


,atom_name,element,residue_name,residue_number,chain,x,y,z
1936,N,N,TYR,907,A,16.118999,40.526001,0.595
1937,CA,C,TYR,907,A,14.796000,40.326000,1.170
1938,C,C,TYR,907,A,13.655000,40.414001,0.160
1939,O,O,TYR,907,A,12.527000,40.124001,0.510
1940,CB,C,TYR,907,A,14.575000,41.251999,2.353
1941,CG,C,TYR,907,A,15.405000,40.830002,3.556
1942,CD1,C,TYR,907,A,14.916000,39.917000,4.484
1943,CD2,C,TYR,907,A,16.664000,41.368999,3.789
1944,CE1,C,TYR,907,A,15.665000,39.544998,5.600
1945,CE2,C,TYR,907,A,17.421000,41.007999,4.902



========== CELL 83G-P4 RESULTS ==========
Ligands analyzed: 10
Total atom pairs within 5 A: 827

Per-ligand summary:


,pdb_id,comp_id,minimum_distance_A,atom_pairs_le_3_5A,atom_pairs_le_4A,atom_pairs_le_5A,unique_ligand_atoms_within_4A,unique_TYR907_atoms_within_4A
0,5KPP,6WZ,2.456679,33,52,94,13,8
1,5KPQ,6X2,2.400476,34,51,93,13,8
2,5WRZ,7U3,2.944162,17,46,88,14,8
3,5WS1,7U9,2.565812,20,46,86,16,8
4,7AAC,78P,2.662797,23,43,81,14,8
5,7ONS,VKT,3.371240,6,33,77,11,7
6,7ONT,VKQ,3.454614,2,29,76,11,7
7,8JNZ,ERV,2.811423,21,38,73,12,8
8,9ETQ,A1H63,3.444867,5,26,76,10,7
9,9ILN,A1D9M,2.451098,28,46,83,11,8



Recurring atom contacts:


,ligand_atom_index,TYR907_atom,ligand_count,median_distance_A,minimum_distance_A
38,4,CZ,8,3.475070,2.842271
28,3,CE2,8,3.515056,2.877988
30,3,CZ,8,3.554034,2.900495
18,2,CD2,8,3.603674,3.316123
36,4,CE2,8,3.726692,3.091800
20,2,CE2,8,3.745892,3.186535
27,3,CE1,8,3.893473,3.194819
12,1,CE2,8,3.908266,3.740123
46,5,CZ,8,3.924495,2.860015
34,4,CD2,8,3.986627,3.378852



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P4 completed.


In [129]:
# ============================================================
# CELL 83G-P5
# TYR907 HYDROGEN-BOND GEOMETRY VALIDATION
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
from scipy.spatial.distance import cdist
import os

# ------------------------------------------------------------
# 1. INPUTS
# ------------------------------------------------------------

SDF_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "experimental_ligand_poses_m1_frame"
    / "validated_sdf"
)

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

if not SDF_DIR.exists():
    raise FileNotFoundError(SDF_DIR)

# ------------------------------------------------------------
# 2. LOAD TYR907 ATOMS
# ------------------------------------------------------------

protein_df = protein_atoms_by_conformer[1].copy()

tyr = protein_df[
    (protein_df["chain"].astype(str).str.strip() == "A") &
    (protein_df["residue_name"].astype(str).str.upper() == "TYR") &
    (protein_df["residue_number"].astype(str).str.strip() == "907")
].copy()

oh = tyr[tyr["atom_name"].astype(str).str.upper() == "OH"]
cz = tyr[tyr["atom_name"].astype(str).str.upper() == "CZ"]

if len(oh) != 1 or len(cz) != 1:
    raise ValueError("Could not uniquely identify TYR907 OH and CZ atoms.")

oh_xyz = oh[["x", "y", "z"]].to_numpy(float)[0]
cz_xyz = cz[["x", "y", "z"]].to_numpy(float)[0]

# ------------------------------------------------------------
# 3. RDKit DONOR/ACCEPTOR FEATURE FACTORY
# ------------------------------------------------------------

feature_file = os.path.join(
    RDConfig.RDDataDir,
    "BaseFeatures.fdef"
)

factory = ChemicalFeatures.BuildFeatureFactory(feature_file)

# ------------------------------------------------------------
# 4. HELPER: ANGLE CALCULATION
# ------------------------------------------------------------

def calculate_angle(a, vertex, c):

    v1 = np.array(a, dtype=float) - np.array(vertex, dtype=float)
    v2 = np.array(c, dtype=float) - np.array(vertex, dtype=float)

    denominator = np.linalg.norm(v1) * np.linalg.norm(v2)

    if denominator == 0:
        return np.nan

    cosine = np.dot(v1, v2) / denominator
    cosine = np.clip(cosine, -1.0, 1.0)

    return float(np.degrees(np.arccos(cosine)))


# ------------------------------------------------------------
# 5. LOAD AND ANALYZE ALL EXPERIMENTAL LIGANDS
# ------------------------------------------------------------

sdf_files = sorted(SDF_DIR.glob("*.sdf"))

if len(sdf_files) != 10:
    raise ValueError(
        f"Expected 10 validated ligand SDFs; found {len(sdf_files)}."
    )

interaction_records = []
ligand_summary = []

for sdf_file in sdf_files:

    supplier = Chem.SDMolSupplier(
        str(sdf_file),
        removeHs=False,
        sanitize=True
    )

    mol = next((m for m in supplier if m is not None), None)

    if mol is None:
        raise ValueError(f"Could not read valid molecule: {sdf_file}")

    if mol.GetNumConformers() == 0:
        raise ValueError(f"No 3D coordinates: {sdf_file}")

    pdb_id = mol.GetProp("PDB_ID") if mol.HasProp("PDB_ID") else sdf_file.stem
    comp_id = (
        mol.GetProp("CCD_COMPONENT")
        if mol.HasProp("CCD_COMPONENT")
        else "UNKNOWN"
    )

    # Identify ligand acceptor atoms
    features = factory.GetFeaturesForMol(mol)

    acceptor_atoms = set()

    for feature in features:
        if feature.GetFamily() == "Acceptor":
            acceptor_atoms.update(feature.GetAtomIds())

    conformer = mol.GetConformer()

    acceptor_records = []

    for atom_idx in sorted(acceptor_atoms):

        atom = mol.GetAtomWithIdx(atom_idx)

        if atom.GetSymbol() == "H":
            continue

        position = np.array(
            conformer.GetAtomPosition(atom_idx),
            dtype=float
        )

        distance = np.linalg.norm(position - oh_xyz)

        # CZ-OH-acceptor angle
        angle = calculate_angle(
            cz_xyz,
            oh_xyz,
            position
        )

        # Screening criteria:
        # OH...A distance <= 3.5 A
        # CZ-OH...A angle >= 120 degrees
        distance_pass = distance <= 3.5
        angle_pass = angle >= 120

        interaction_records.append({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_atom_index": atom_idx,
            "ligand_atom_symbol": atom.GetSymbol(),
            "ligand_atom_smarts_role": "ACCEPTOR",
            "TYR907_donor_atom": "OH",
            "OH_acceptor_distance_A": distance,
            "CZ_OH_A_angle_deg": angle,
            "distance_pass": distance_pass,
            "angle_pass": angle_pass,
            "geometry_screen_pass": distance_pass and angle_pass
        })

        acceptor_records.append({
            "distance": distance,
            "angle": angle,
            "pass": distance_pass and angle_pass
        })

    if acceptor_records:

        nearest = min(
            acceptor_records,
            key=lambda x: x["distance"]
        )

        passing = sum(
            item["pass"] for item in acceptor_records
        )

        ligand_summary.append({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_acceptor_atoms": len(acceptor_atoms),
            "nearest_OH_acceptor_distance_A": nearest["distance"],
            "nearest_CZ_OH_A_angle_deg": nearest["angle"],
            "potential_geometry_matches": passing
        })

    else:

        ligand_summary.append({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_acceptor_atoms": 0,
            "nearest_OH_acceptor_distance_A": np.nan,
            "nearest_CZ_OH_A_angle_deg": np.nan,
            "potential_geometry_matches": 0
        })


# ------------------------------------------------------------
# 6. DATAFRAMES
# ------------------------------------------------------------

interaction_df = pd.DataFrame(interaction_records)
summary_df = pd.DataFrame(ligand_summary)

if len(summary_df) != 10:
    raise ValueError("Not all 10 ligands were analyzed.")


# ------------------------------------------------------------
# 7. EXPORT
# ------------------------------------------------------------

interaction_df.to_csv(
    OUTPUT_DIR / "TYR907_hbond_geometry_details.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "TYR907_hbond_geometry_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n========== CELL 83G-P5 RESULTS ==========")

print("\nLigands analyzed:", len(summary_df))

print("\nPer-ligand hydrogen-bond geometry screening:")
display(summary_df)

print("\nIndividual ligand acceptor observations:")
display(
    interaction_df.sort_values(
        ["PDB_ID", "OH_acceptor_distance_A"]
    )
)

if not interaction_df.empty:

    print(
        "\nPotential geometry matches:",
        int(interaction_df["geometry_screen_pass"].sum())
    )

    print(
        "Distinct ligands with potential matches:",
        interaction_df.loc[
            interaction_df["geometry_screen_pass"],
            "PDB_ID"
        ].nunique()
    )

print("\nSaved to:", OUTPUT_DIR)
print("\nCell 83G-P5 completed.")


========== CELL 83G-P5 RESULTS ==========

Ligands analyzed: 10

Per-ligand hydrogen-bond geometry screening:


,PDB_ID,CCD_COMPONENT,ligand_acceptor_atoms,nearest_OH_acceptor_distance_A,nearest_CZ_OH_A_angle_deg,potential_geometry_matches
0,5KPP,6WZ,4,5.196768,36.142170,0
1,5KPQ,6X2,4,5.116398,36.026280,0
2,5WRZ,7U3,4,3.690598,68.609404,0
3,5WS1,7U9,3,3.117775,64.684960,0
4,7AAC,78P,2,2.662788,106.090782,0
5,7ONS,VKT,2,6.536104,41.720500,0
6,7ONT,VKQ,3,6.528920,40.868020,0
7,8JNZ,ERV,3,3.528904,61.672194,0
8,9ETQ,A1H63,4,4.422976,64.692289,0
9,9ILN,A1D9M,5,5.259641,38.308974,0



Individual ligand acceptor observations:


,PDB_ID,CCD_COMPONENT,ligand_atom_index,ligand_atom_symbol,ligand_atom_smarts_role,TYR907_donor_atom,OH_acceptor_distance_A,CZ_OH_A_angle_deg,distance_pass,angle_pass,geometry_screen_pass
1,5KPP,6WZ,12,O,ACCEPTOR,OH,5.196768,36.142170,False,False,False
2,5KPP,6WZ,15,O,ACCEPTOR,OH,6.340974,43.430918,False,False,False
3,5KPP,6WZ,25,F,ACCEPTOR,OH,8.055872,95.239412,False,False,False
0,5KPP,6WZ,5,O,ACCEPTOR,OH,8.118830,74.865749,False,False,False
5,5KPQ,6X2,15,O,ACCEPTOR,OH,5.116398,36.026280,False,False,False
6,5KPQ,6X2,18,O,ACCEPTOR,OH,6.385814,43.780675,False,False,False
7,5KPQ,6X2,28,F,ACCEPTOR,OH,7.874541,95.954263,False,False,False
4,5KPQ,6X2,8,O,ACCEPTOR,OH,8.152435,77.849195,False,False,False
10,5WRZ,7U3,5,O,ACCEPTOR,OH,3.690598,68.609404,False,False,False
9,5WRZ,7U3,3,N,ACCEPTOR,OH,4.826096,43.871194,False,False,False



Potential geometry matches: 0
Distinct ligands with potential matches: 0

Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P5 completed.


In [130]:
# ============================================================
# CELL 83G-P6
# TYR907 AROMATIC RING INTERACTION ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem

# ------------------------------------------------------------
# 1. PATHS AND INPUT VALIDATION
# ------------------------------------------------------------

SDF_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "experimental_ligand_poses_m1_frame"
    / "validated_sdf"
)

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

if not SDF_DIR.exists():
    raise FileNotFoundError(SDF_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. EXTRACT TYR907 AROMATIC RING COORDINATES
# ------------------------------------------------------------

protein_df = protein_atoms_by_conformer[1].copy()

tyr_ring_names = ["CG", "CD1", "CD2", "CE1", "CE2", "CZ"]

tyr_ring = protein_df[
    (protein_df["chain"].astype(str).str.strip() == "A")
    & (protein_df["residue_name"].astype(str).str.upper() == "TYR")
    & (protein_df["residue_number"].astype(str).str.strip() == "907")
    & (protein_df["atom_name"].astype(str).str.upper().isin(tyr_ring_names))
].copy()

tyr_ring["atom_name"] = tyr_ring["atom_name"].astype(str).str.upper()

if len(tyr_ring) != 6:
    raise ValueError(
        f"Expected 6 TYR907 aromatic ring atoms; found {len(tyr_ring)}."
    )

tyr_ring = tyr_ring.set_index("atom_name").loc[tyr_ring_names]

tyr_xyz = tyr_ring[["x", "y", "z"]].to_numpy(float)

# ------------------------------------------------------------
# 3. GEOMETRY FUNCTIONS
# ------------------------------------------------------------

def calculate_ring_geometry(coords):

    coords = np.asarray(coords, dtype=float)

    if len(coords) < 3:
        raise ValueError("At least 3 ring atoms are required.")

    centroid = coords.mean(axis=0)

    centered = coords - centroid

    # Best-fit plane using singular value decomposition
    _, _, vh = np.linalg.svd(centered, full_matrices=False)

    normal = vh[-1]

    normal = normal / np.linalg.norm(normal)

    return centroid, normal


def calculate_ring_relationship(
    centroid1,
    normal1,
    centroid2,
    normal2
):

    displacement = centroid2 - centroid1

    centroid_distance = np.linalg.norm(displacement)

    cosine = np.clip(
        abs(np.dot(normal1, normal2)),
        0.0,
        1.0
    )

    plane_angle = np.degrees(np.arccos(cosine))

    perpendicular_separation = abs(
        np.dot(displacement, normal1)
    )

    lateral_offset = np.sqrt(
        max(
            centroid_distance**2
            - perpendicular_separation**2,
            0.0
        )
    )

    return {
        "centroid_distance_A": centroid_distance,
        "ring_plane_angle_deg": plane_angle,
        "perpendicular_separation_A": perpendicular_separation,
        "lateral_offset_A": lateral_offset
    }


# TYR907 geometry
tyr_centroid, tyr_normal = calculate_ring_geometry(tyr_xyz)

print("TYR907 aromatic ring centroid:", tyr_centroid)
print("TYR907 aromatic ring normal:", tyr_normal)

# ------------------------------------------------------------
# 4. IDENTIFY LIGAND AROMATIC RINGS
# ------------------------------------------------------------

sdf_files = sorted(SDF_DIR.glob("*.sdf"))

if len(sdf_files) != 10:
    raise ValueError(
        f"Expected 10 SDF files; found {len(sdf_files)}."
    )

ring_records = []
ligand_summary = []

for sdf_file in sdf_files:

    supplier = Chem.SDMolSupplier(
        str(sdf_file),
        removeHs=False,
        sanitize=True
    )

    mol = next((m for m in supplier if m is not None), None)

    if mol is None:
        raise ValueError(f"Invalid molecule: {sdf_file}")

    if mol.GetNumConformers() == 0:
        raise ValueError(f"No 3D coordinates: {sdf_file}")

    pdb_id = (
        mol.GetProp("PDB_ID")
        if mol.HasProp("PDB_ID")
        else sdf_file.stem
    )

    comp_id = (
        mol.GetProp("CCD_COMPONENT")
        if mol.HasProp("CCD_COMPONENT")
        else "UNKNOWN"
    )

    conformer = mol.GetConformer()

    ring_info = mol.GetRingInfo()

    aromatic_rings = []

    for ring in ring_info.AtomRings():

        # Keep only rings whose atoms are aromatic
        if not all(
            mol.GetAtomWithIdx(idx).GetIsAromatic()
            for idx in ring
        ):
            continue

        # Confirm all ring bonds are aromatic
        ring_bonds = []

        for i in range(len(ring)):

            a = ring[i]
            b = ring[(i + 1) % len(ring)]

            bond = mol.GetBondBetweenAtoms(a, b)

            if bond is None:
                break

            ring_bonds.append(bond.GetIsAromatic())

        if len(ring_bonds) != len(ring):
            continue

        if not all(ring_bonds):
            continue

        aromatic_rings.append(ring)

    ligand_ring_count = len(aromatic_rings)

    best_distance = np.inf
    best_ring_result = None

    if ligand_ring_count == 0:

        ligand_summary.append({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_aromatic_ring_count": 0,
            "nearest_centroid_distance_A": np.nan,
            "nearest_ring_plane_angle_deg": np.nan,
            "nearest_perpendicular_separation_A": np.nan,
            "nearest_lateral_offset_A": np.nan
        })

        continue

    # --------------------------------------------------------
    # 5. COMPARE EVERY LIGAND AROMATIC RING WITH TYR907
    # --------------------------------------------------------

    for ring_number, ring in enumerate(aromatic_rings, start=1):

        ligand_ring_xyz = np.array([
            conformer.GetAtomPosition(idx)
            for idx in ring
        ], dtype=float)

        ligand_centroid, ligand_normal = (
            calculate_ring_geometry(ligand_ring_xyz)
        )

        metrics = calculate_ring_relationship(
            tyr_centroid,
            tyr_normal,
            ligand_centroid,
            ligand_normal
        )

        metrics.update({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_ring_number": ring_number,
            "ligand_ring_atom_indices": ",".join(
                map(str, ring)
            ),
            "ligand_ring_atom_count": len(ring),
            "ligand_ring_centroid_x": ligand_centroid[0],
            "ligand_ring_centroid_y": ligand_centroid[1],
            "ligand_ring_centroid_z": ligand_centroid[2],
            "TYR907_centroid_x": tyr_centroid[0],
            "TYR907_centroid_y": tyr_centroid[1],
            "TYR907_centroid_z": tyr_centroid[2]
        })

        ring_records.append(metrics)

        if metrics["centroid_distance_A"] < best_distance:

            best_distance = metrics["centroid_distance_A"]
            best_ring_result = metrics

    ligand_summary.append({
        "PDB_ID": pdb_id,
        "CCD_COMPONENT": comp_id,
        "ligand_aromatic_ring_count": ligand_ring_count,
        "nearest_centroid_distance_A":
            best_ring_result["centroid_distance_A"],
        "nearest_ring_plane_angle_deg":
            best_ring_result["ring_plane_angle_deg"],
        "nearest_perpendicular_separation_A":
            best_ring_result["perpendicular_separation_A"],
        "nearest_lateral_offset_A":
            best_ring_result["lateral_offset_A"]
    })


# ------------------------------------------------------------
# 6. CREATE RESULT TABLES
# ------------------------------------------------------------

ring_df = pd.DataFrame(ring_records)

summary_df = pd.DataFrame(ligand_summary)

if len(summary_df) != 10:
    raise ValueError("Not all 10 ligands were analyzed.")


# ------------------------------------------------------------
# 7. EXPORT RESULTS
# ------------------------------------------------------------

ring_df.to_csv(
    OUTPUT_DIR / "TYR907_aromatic_ring_pair_details.csv",
    index=False
)

summary_df.to_csv(
    OUTPUT_DIR / "TYR907_aromatic_interaction_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 8. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========== CELL 83G-P6 RESULTS ==========")

print("\nTYR907 aromatic ring atoms:")
display(tyr_ring)

print("\nTotal ligand aromatic rings analyzed:", len(ring_df))

print("\nPer-ligand aromatic interaction summary:")
display(
    summary_df.sort_values(
        "nearest_centroid_distance_A"
    )
)

print("\nIndividual aromatic ring comparisons:")
display(
    ring_df.sort_values(
        "centroid_distance_A"
    )
)

print("\nSaved to:", OUTPUT_DIR)

print("\nCell 83G-P6 completed.")

TYR907 aromatic ring centroid: [16.16299979 40.46216647  4.69166664]
TYR907 aromatic ring normal: [ 0.41526366 -0.74716771 -0.51893787]

========== CELL 83G-P6 RESULTS ==========

TYR907 aromatic ring atoms:


,element,residue_name,residue_number,chain,x,y,z
atom_name,,,,,,,
CG,C,TYR,907,A,15.405,40.830002,3.556
CD1,C,TYR,907,A,14.916,39.917000,4.484
CD2,C,TYR,907,A,16.664,41.368999,3.789
CE1,C,TYR,907,A,15.665,39.544998,5.600
CE2,C,TYR,907,A,17.421,41.007999,4.902
CZ,C,TYR,907,A,16.907,40.104000,5.819



Total ligand aromatic rings analyzed: 25

Per-ligand aromatic interaction summary:


,PDB_ID,CCD_COMPONENT,ligand_aromatic_ring_count,nearest_centroid_distance_A,nearest_ring_plane_angle_deg,nearest_perpendicular_separation_A,nearest_lateral_offset_A
0,5KPP,6WZ,3,3.000320,12.349233,2.927591,0.656610
9,9ILN,A1D9M,2,3.001120,12.891189,2.942379,0.590874
1,5KPQ,6X2,3,3.021870,13.873392,2.941523,0.692200
2,5WRZ,7U3,2,3.244129,7.026101,3.208396,0.480174
5,7ONS,VKT,3,3.485952,3.737407,3.399575,0.771202
8,9ETQ,A1H63,3,3.494057,2.044238,3.407607,0.772433
6,7ONT,VKQ,3,3.527114,2.348196,3.433662,0.806533
3,5WS1,7U9,2,3.599107,14.808772,2.690145,2.390960
4,7AAC,78P,2,3.607457,10.760699,2.754185,2.329852
7,8JNZ,ERV,2,3.684652,9.670722,2.853938,2.330599



Individual aromatic ring comparisons:


,centroid_distance_A,ring_plane_angle_deg,perpendicular_separation_A,lateral_offset_A,PDB_ID,CCD_COMPONENT,ligand_ring_number,ligand_ring_atom_indices,ligand_ring_atom_count,ligand_ring_centroid_x,ligand_ring_centroid_y,ligand_ring_centroid_z,TYR907_centroid_x,TYR907_centroid_y,TYR907_centroid_z
0,3.000320,12.349233,2.927591,0.656610,5KPP,6WZ,1,"10,11,13,14,16,21",6,15.211933,43.084583,5.796350,16.163,40.462166,4.691667
24,3.001120,12.891189,2.942379,0.590874,9ILN,A1D9M,2,"2,3,4,26,28,29",6,15.160200,43.050550,5.832450,16.163,40.462166,4.691667
3,3.021870,13.873392,2.941523,0.692200,5KPQ,6X2,1,"13,14,16,17,19,24",6,15.269817,43.120017,5.818500,16.163,40.462166,4.691667
6,3.244129,7.026101,3.208396,0.480174,5WRZ,7U3,1,"1,2,3,4,12,18",6,15.086050,43.177350,6.103167,16.163,40.462166,4.691667
13,3.485952,3.737407,3.399575,0.771202,7ONS,VKT,2,"2,3,4,23,24,25",6,14.917000,43.491833,5.883500,16.163,40.462166,4.691667
21,3.494057,2.044238,3.407607,0.772433,9ETQ,A1H63,2,"2,3,4,26,27,28",6,14.733083,43.443183,5.821850,16.163,40.462166,4.691667
16,3.527114,2.348196,3.433662,0.806533,7ONT,VKQ,2,"2,3,4,26,27,28",6,14.797967,43.509000,5.829217,16.163,40.462166,4.691667
8,3.599107,14.808772,2.690145,2.390960,5WS1,7U9,1,"7,8,9,18,19",5,16.357820,41.876160,7.995640,16.163,40.462166,4.691667
10,3.607457,10.760699,2.754185,2.329852,7AAC,78P,1,"2,3,4,12,13",5,16.297860,41.939380,7.980040,16.163,40.462166,4.691667
19,3.684652,9.670722,2.853938,2.330599,8JNZ,ERV,2,"1,2,3,4,11,12",6,16.294033,42.045717,8.016100,16.163,40.462166,4.691667



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P6 completed.


In [131]:
# ============================================================
# CELL 83G-P7
# HYDROPHOBIC INTERACTION ANALYSIS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
import os

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

SDF_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "experimental_ligand_poses_m1_frame"
    / "validated_sdf"
)

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. IDENTIFY HYDROPHOBIC PROTEIN SIDE-CHAIN ATOMS
# ------------------------------------------------------------

protein_df = protein_atoms_by_conformer[1].copy()

# Hydrophobic amino acid residues
hydrophobic_residues = {
    "ALA", "VAL", "LEU", "ILE",
    "MET", "PHE", "TRP", "PRO",
    "TYR", "CYS"
}

# Exclude backbone atoms
backbone_atoms = {"N", "CA", "C", "O", "OXT"}

protein_hydrophobic = protein_df[
    protein_df["residue_name"].astype(str).str.upper().isin(
        hydrophobic_residues
    )
    & ~protein_df["atom_name"].astype(str).str.upper().isin(
        backbone_atoms
    )
    & protein_df["element"].astype(str).str.upper().isin(
        {"C", "S"}
    )
].copy()

protein_hydrophobic["atom_name"] = (
    protein_hydrophobic["atom_name"].astype(str).str.strip()
)

protein_hydrophobic["residue_name"] = (
    protein_hydrophobic["residue_name"].astype(str).str.strip()
)

protein_hydrophobic["chain"] = (
    protein_hydrophobic["chain"].astype(str).str.strip()
)

protein_hydrophobic["residue_number"] = (
    protein_hydrophobic["residue_number"].astype(str).str.strip()
)

protein_xyz = protein_hydrophobic[
    ["x", "y", "z"]
].to_numpy(dtype=float)

if len(protein_xyz) == 0:
    raise ValueError("No hydrophobic protein atoms found.")

print("Hydrophobic protein atoms:", len(protein_hydrophobic))
print(
    "Hydrophobic protein residues:",
    protein_hydrophobic["residue_name"].nunique()
)

# ------------------------------------------------------------
# 3. RDKit FEATURE FACTORY
# ------------------------------------------------------------

fdef_path = os.path.join(
    RDConfig.RDDataDir,
    "BaseFeatures.fdef"
)

factory = ChemicalFeatures.BuildFeatureFactory(fdef_path)


# ------------------------------------------------------------
# 4. DISTANCE FUNCTION
# ------------------------------------------------------------

def pairwise_distances(coords1, coords2):

    difference = (
        coords1[:, None, :]
        - coords2[None, :, :]
    )

    return np.sqrt(
        np.sum(difference ** 2, axis=2)
    )


# ------------------------------------------------------------
# 5. ANALYZE ALL EXPERIMENTAL LIGANDS
# ------------------------------------------------------------

sdf_files = sorted(SDF_DIR.glob("*.sdf"))

if len(sdf_files) != 10:
    raise ValueError(
        f"Expected 10 SDF files; found {len(sdf_files)}."
    )

detail_records = []
summary_records = []

for sdf_file in sdf_files:

    supplier = Chem.SDMolSupplier(
        str(sdf_file),
        removeHs=False,
        sanitize=True
    )

    mol = next((m for m in supplier if m is not None), None)

    if mol is None:
        raise ValueError(f"Invalid SDF: {sdf_file}")

    if mol.GetNumConformers() == 0:
        raise ValueError(f"No 3D coordinates: {sdf_file}")

    pdb_id = (
        mol.GetProp("PDB_ID")
        if mol.HasProp("PDB_ID")
        else sdf_file.stem
    )

    comp_id = (
        mol.GetProp("CCD_COMPONENT")
        if mol.HasProp("CCD_COMPONENT")
        else "UNKNOWN"
    )

    conformer = mol.GetConformer()

    # --------------------------------------------------------
    # Extract hydrophobic ligand atoms using RDKit
    # --------------------------------------------------------

    hydrophobic_indices = set()

    for feature in factory.GetFeaturesForMol(mol):

        if feature.GetFamily() in {
            "Hydrophobe",
            "LumpedHydrophobe"
        }:

            hydrophobic_indices.update(
                feature.GetAtomIds()
            )

    # Retain carbon/sulfur heavy atoms only
    hydrophobic_indices = sorted(
        idx for idx in hydrophobic_indices
        if mol.GetAtomWithIdx(idx).GetSymbol() in {"C", "S"}
    )

    if not hydrophobic_indices:

        summary_records.append({
            "PDB_ID": pdb_id,
            "CCD_COMPONENT": comp_id,
            "ligand_hydrophobic_atom_count": 0,
            "protein_hydrophobic_atom_count": len(protein_xyz),
            "pairs_within_3_5A": 0,
            "pairs_within_4A": 0,
            "pairs_within_4_5A": 0,
            "unique_ligand_atoms_within_4_5A": 0,
            "unique_protein_atoms_within_4_5A": 0,
            "contacting_residue_count_within_4_5A": 0,
            "nearest_hydrophobic_distance_A": np.nan
        })

        continue

    ligand_xyz = np.array([
        conformer.GetAtomPosition(idx)
        for idx in hydrophobic_indices
    ], dtype=float)

    # --------------------------------------------------------
    # Calculate all ligand-protein hydrophobic distances
    # --------------------------------------------------------

    distances = pairwise_distances(
        ligand_xyz,
        protein_xyz
    )

    nearest_distance = float(distances.min())

    # --------------------------------------------------------
    # Record atom-pair contacts
    # --------------------------------------------------------

    ligand_contact_indices = set()
    protein_contact_indices = set()

    contacting_residues = set()

    for i, ligand_idx in enumerate(hydrophobic_indices):

        for j in range(len(protein_xyz)):

            distance = float(distances[i, j])

            if distance <= 4.5:

                protein_atom = protein_hydrophobic.iloc[j]

                ligand_atom = mol.GetAtomWithIdx(
                    ligand_idx
                )

                residue_identity = (
                    protein_atom["chain"],
                    protein_atom["residue_number"],
                    protein_atom["residue_name"]
                )

                ligand_contact_indices.add(ligand_idx)
                protein_contact_indices.add(j)
                contacting_residues.add(residue_identity)

                detail_records.append({

                    "PDB_ID": pdb_id,
                    "CCD_COMPONENT": comp_id,

                    "ligand_atom_index": ligand_idx,
                    "ligand_atom_symbol": ligand_atom.GetSymbol(),

                    "protein_chain": protein_atom["chain"],
                    "protein_residue_number":
                        protein_atom["residue_number"],
                    "protein_residue_name":
                        protein_atom["residue_name"],
                    "protein_atom_name":
                        protein_atom["atom_name"],

                    "distance_A": distance,

                    "within_3_5A": distance <= 3.5,
                    "within_4A": distance <= 4.0,
                    "within_4_5A": distance <= 4.5
                })

    # --------------------------------------------------------
    # Summary for this ligand
    # --------------------------------------------------------

    summary_records.append({

        "PDB_ID": pdb_id,
        "CCD_COMPONENT": comp_id,

        "ligand_hydrophobic_atom_count":
            len(hydrophobic_indices),

        "protein_hydrophobic_atom_count":
            len(protein_xyz),

        "pairs_within_3_5A":
            int(np.sum(distances <= 3.5)),

        "pairs_within_4A":
            int(np.sum(distances <= 4.0)),

        "pairs_within_4_5A":
            int(np.sum(distances <= 4.5)),

        "unique_ligand_atoms_within_4_5A":
            len(ligand_contact_indices),

        "unique_protein_atoms_within_4_5A":
            len(protein_contact_indices),

        "contacting_residue_count_within_4_5A":
            len(contacting_residues),

        "nearest_hydrophobic_distance_A":
            nearest_distance
    })


# ------------------------------------------------------------
# 6. CREATE DATAFRAMES
# ------------------------------------------------------------

hydrophobic_details_df = pd.DataFrame(detail_records)

hydrophobic_summary_df = pd.DataFrame(summary_records)

if len(hydrophobic_summary_df) != 10:
    raise ValueError("Expected 10 ligand summaries.")


# ------------------------------------------------------------
# 7. EXPORT
# ------------------------------------------------------------

hydrophobic_details_df.to_csv(
    OUTPUT_DIR / "hydrophobic_contact_details.csv",
    index=False
)

hydrophobic_summary_df.to_csv(
    OUTPUT_DIR / "hydrophobic_contact_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 8. DISPLAY
# ------------------------------------------------------------

print("\n========== CELL 83G-P7 RESULTS ==========")

print("\nHydrophobic protein atoms:", len(protein_hydrophobic))

print(
    "\nTotal ligand-protein hydrophobic atom pairs within 4.5 Å:",
    len(hydrophobic_details_df)
)

print("\nPer-ligand hydrophobic contact summary:")

display(
    hydrophobic_summary_df.sort_values(
        "nearest_hydrophobic_distance_A"
    )
)

print("\nIndividual hydrophobic contacts:")

display(
    hydrophobic_details_df.sort_values(
        ["PDB_ID", "distance_A"]
    ).head(50)
)

print("\nSaved to:", OUTPUT_DIR)

print("\nCell 83G-P7 completed.")

Hydrophobic protein atoms: 1228
Hydrophobic protein residues: 10

========== CELL 83G-P7 RESULTS ==========

Hydrophobic protein atoms: 1228

Total ligand-protein hydrophobic atom pairs within 4.5 Å: 350

Per-ligand hydrophobic contact summary:


,PDB_ID,CCD_COMPONENT,ligand_hydrophobic_atom_count,protein_hydrophobic_atom_count,pairs_within_3_5A,pairs_within_4A,pairs_within_4_5A,unique_ligand_atoms_within_4_5A,unique_protein_atoms_within_4_5A,contacting_residue_count_within_4_5A,nearest_hydrophobic_distance_A
0,5KPP,6WZ,12,1228,10,28,62,12,15,4,2.942942
1,5KPQ,6X2,13,1228,10,28,61,12,15,4,2.945340
4,7AAC,78P,6,1228,4,13,28,6,10,3,3.078819
3,5WS1,7U9,6,1228,4,13,30,6,10,3,3.087072
2,5WRZ,7U3,7,1228,5,19,35,7,10,3,3.090047
9,9ILN,A1D9M,9,1228,5,18,40,9,16,5,3.186539
7,8JNZ,ERV,2,1228,3,6,13,2,9,2,3.261209
5,7ONS,VKT,8,1228,5,23,36,8,9,3,3.371225
6,7ONT,VKQ,8,1228,1,18,34,8,9,3,3.494657
8,9ETQ,A1H63,3,1228,1,3,11,3,8,3,3.498673



Individual hydrophobic contacts:


,PDB_ID,CCD_COMPONENT,ligand_atom_index,ligand_atom_symbol,protein_chain,protein_residue_number,protein_residue_name,protein_atom_name,distance_A,within_3_5A,within_4A,within_4_5A
42,5KPP,6WZ,21,C,A,907,TYR,CE2,2.942942,True,True,True
43,5KPP,6WZ,21,C,A,907,TYR,CZ,2.951429,True,True,True
36,5KPP,6WZ,20,C,A,907,TYR,CZ,3.115277,True,True,True
58,5KPP,6WZ,24,C,A,896,TYR,CD1,3.216126,True,True,True
17,5KPP,6WZ,16,C,A,907,TYR,CE2,3.244552,True,True,True
35,5KPP,6WZ,20,C,A,907,TYR,CE2,3.353834,True,True,True
15,5KPP,6WZ,16,C,A,907,TYR,CD2,3.366326,True,True,True
40,5KPP,6WZ,21,C,A,907,TYR,CD2,3.404205,True,True,True
41,5KPP,6WZ,21,C,A,907,TYR,CE1,3.440620,True,True,True
60,5KPP,6WZ,24,C,A,896,TYR,CE1,3.460191,True,True,True



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P7 completed.


In [132]:
# ============================================================
# CELL 83G-P8
# INTEGRATED EXPERIMENTAL LIGAND INTERACTION FINGERPRINT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

# ------------------------------------------------------------
# 1. LOAD PREVIOUS ANALYSIS RESULTS
# ------------------------------------------------------------

hbond = pd.read_csv(
    OUTPUT_DIR / "TYR907_hbond_geometry_summary.csv"
)

aromatic = pd.read_csv(
    OUTPUT_DIR / "TYR907_aromatic_interaction_summary.csv"
)

hydrophobic = pd.read_csv(
    OUTPUT_DIR / "hydrophobic_contact_summary.csv"
)

# ------------------------------------------------------------
# 2. VERIFY INPUTS
# ------------------------------------------------------------

for name, df in [
    ("Hydrogen bond", hbond),
    ("Aromatic", aromatic),
    ("Hydrophobic", hydrophobic)
]:

    if len(df) != 10:
        raise ValueError(
            f"{name} dataset contains {len(df)} rows, expected 10."
        )

    if df.duplicated(
        subset=["PDB_ID", "CCD_COMPONENT"]
    ).any():
        raise ValueError(
            f"Duplicate ligand records found in {name} dataset."
        )

# ------------------------------------------------------------
# 3. MERGE RESULTS
# ------------------------------------------------------------

fingerprint = hbond.merge(
    aromatic,
    on=["PDB_ID", "CCD_COMPONENT"],
    how="inner",
    validate="one_to_one"
)

fingerprint = fingerprint.merge(
    hydrophobic,
    on=["PDB_ID", "CCD_COMPONENT"],
    how="inner",
    validate="one_to_one"
)

if len(fingerprint) != 10:
    raise ValueError(
        "Ligand records did not merge correctly."
    )

# ------------------------------------------------------------
# 4. SELECT IMPORTANT FEATURES
# ------------------------------------------------------------

selected_columns = [
    "PDB_ID",
    "CCD_COMPONENT",

    # Hydrogen-bond geometry
    "ligand_acceptor_atoms",
    "nearest_OH_acceptor_distance_A",
    "nearest_CZ_OH_A_angle_deg",
    "potential_geometry_matches",

    # Aromatic geometry
    "ligand_aromatic_ring_count",
    "nearest_centroid_distance_A",
    "nearest_ring_plane_angle_deg",
    "nearest_perpendicular_separation_A",
    "nearest_lateral_offset_A",

    # Hydrophobic contacts
    "ligand_hydrophobic_atom_count",
    "pairs_within_3_5A",
    "pairs_within_4A",
    "pairs_within_4_5A",
    "unique_ligand_atoms_within_4_5A",
    "unique_protein_atoms_within_4_5A",
    "contacting_residue_count_within_4_5A",
    "nearest_hydrophobic_distance_A"
]

missing = [
    col for col in selected_columns
    if col not in fingerprint.columns
]

if missing:
    raise KeyError(
        f"Missing expected columns: {missing}"
    )

fingerprint = fingerprint[selected_columns].copy()

# ------------------------------------------------------------
# 5. ADD INTERPRETATION FLAGS
# ------------------------------------------------------------

fingerprint["TYR907_hbond_geometry_detected"] = (
    fingerprint["potential_geometry_matches"] > 0
)

fingerprint["close_aromatic_geometry"] = (
    (fingerprint["nearest_centroid_distance_A"] <= 4.0)
    &
    (fingerprint["nearest_ring_plane_angle_deg"] <= 30.0)
)

fingerprint["hydrophobic_contacts_detected"] = (
    fingerprint["pairs_within_4_5A"] > 0
)

# This is a descriptive classification, not a binding score.
fingerprint["structural_observation"] = np.where(
    fingerprint["close_aromatic_geometry"]
    & fingerprint["hydrophobic_contacts_detected"],
    "AROMATIC_AND_HYDROPHOBIC_CONTACTS",
    "OTHER_CONTACT_PATTERN"
)

# ------------------------------------------------------------
# 6. EXPORT
# ------------------------------------------------------------

output_file = (
    OUTPUT_DIR
    / "integrated_experimental_ligand_interaction_fingerprint.csv"
)

fingerprint.to_csv(output_file, index=False)

# ------------------------------------------------------------
# 7. DISPLAY
# ------------------------------------------------------------

print("\n========== CELL 83G-P8 RESULTS ==========")

print("Experimental ligands integrated:", len(fingerprint))

print(
    "\nHydrogen-bond geometry matches:",
    int(fingerprint["TYR907_hbond_geometry_detected"].sum())
)

print(
    "Ligands with close aromatic geometry:",
    int(fingerprint["close_aromatic_geometry"].sum())
)

print(
    "Ligands with hydrophobic contacts:",
    int(fingerprint["hydrophobic_contacts_detected"].sum())
)

print("\nIntegrated interaction fingerprint:")

display(fingerprint)

print("\nSaved to:", output_file)

print("\nCell 83G-P8 completed.")


========== CELL 83G-P8 RESULTS ==========
Experimental ligands integrated: 10

Hydrogen-bond geometry matches: 0
Ligands with close aromatic geometry: 10
Ligands with hydrophobic contacts: 10

Integrated interaction fingerprint:


,PDB_ID,CCD_COMPONENT,ligand_acceptor_atoms,nearest_OH_acceptor_distance_A,nearest_CZ_OH_A_angle_deg,potential_geometry_matches,ligand_aromatic_ring_count,nearest_centroid_distance_A,nearest_ring_plane_angle_deg,nearest_perpendicular_separation_A,...,pairs_within_4A,pairs_within_4_5A,unique_ligand_atoms_within_4_5A,unique_protein_atoms_within_4_5A,contacting_residue_count_within_4_5A,nearest_hydrophobic_distance_A,TYR907_hbond_geometry_detected,close_aromatic_geometry,hydrophobic_contacts_detected,structural_observation
0,5KPP,6WZ,4,5.196768,36.142170,0,3,3.000320,12.349233,2.927591,...,28,62,12,15,4,2.942942,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
1,5KPQ,6X2,4,5.116398,36.026280,0,3,3.021870,13.873392,2.941523,...,28,61,12,15,4,2.945340,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
2,5WRZ,7U3,4,3.690598,68.609404,0,2,3.244129,7.026101,3.208396,...,19,35,7,10,3,3.090047,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
3,5WS1,7U9,3,3.117775,64.684960,0,2,3.599107,14.808772,2.690145,...,13,30,6,10,3,3.087072,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
4,7AAC,78P,2,2.662788,106.090782,0,2,3.607457,10.760699,2.754185,...,13,28,6,10,3,3.078819,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
5,7ONS,VKT,2,6.536104,41.720500,0,3,3.485952,3.737407,3.399575,...,23,36,8,9,3,3.371225,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
6,7ONT,VKQ,3,6.528920,40.868020,0,3,3.527114,2.348196,3.433662,...,18,34,8,9,3,3.494657,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
7,8JNZ,ERV,3,3.528904,61.672194,0,2,3.684652,9.670722,2.853938,...,6,13,2,9,2,3.261209,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
8,9ETQ,A1H63,4,4.422976,64.692289,0,3,3.494057,2.044238,3.407607,...,3,11,3,8,3,3.498673,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS
9,9ILN,A1D9M,5,5.259641,38.308974,0,2,3.001120,12.891189,2.942379,...,18,40,9,16,5,3.186539,False,True,True,AROMATIC_AND_HYDROPHOBIC_CONTACTS



Saved to: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis/integrated_experimental_ligand_interaction_fingerprint.csv

Cell 83G-P8 completed.


In [134]:
# ============================================================
# CELL 83G-P9
# M1 CONFORMER COORDINATE COMPATIBILITY AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member2_outputs"
    / "spatial_pharmacophore_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. VERIFY INPUTS
# ------------------------------------------------------------

if not isinstance(protein_atoms_by_conformer, dict):
    raise TypeError("protein_atoms_by_conformer must be a dictionary.")

expected_conformers = set(range(1, 6))

if set(protein_atoms_by_conformer.keys()) != expected_conformers:
    raise ValueError(
        f"Expected conformers {expected_conformers}; "
        f"found {set(protein_atoms_by_conformer.keys())}"
    )

required_columns = {
    "atom_name",
    "element",
    "residue_name",
    "residue_number",
    "chain",
    "x",
    "y",
    "z"
}

# ------------------------------------------------------------
# 2. PREPARE PROTEIN ATOM TABLES
# ------------------------------------------------------------

protein_tables = {}

for conf_id, df in protein_atoms_by_conformer.items():

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"Conformer {conf_id} missing columns: {missing}"
        )

    temp = df.copy()

    for col in [
        "atom_name",
        "element",
        "residue_name",
        "residue_number",
        "chain"
    ]:
        temp[col] = temp[col].astype(str).str.strip()

    # Retain protein heavy atoms
    temp = temp[
        temp["element"].str.upper().isin(
            {"C", "N", "O", "S", "P"}
        )
    ].copy()

    # Stable atom identity
    temp["atom_key"] = (
        temp["chain"] + "|" +
        temp["residue_number"] + "|" +
        temp["residue_name"] + "|" +
        temp["atom_name"]
    )

    if temp["atom_key"].duplicated().any():
        raise ValueError(
            f"Duplicate atom identities in conformer {conf_id}."
        )

    protein_tables[conf_id] = temp.set_index("atom_key")

# ------------------------------------------------------------
# 3. KABSCH ALIGNMENT FUNCTION
# ------------------------------------------------------------

def calculate_kabsch(reference_xyz, moving_xyz):

    reference_xyz = np.asarray(reference_xyz, dtype=float)
    moving_xyz = np.asarray(moving_xyz, dtype=float)

    if reference_xyz.shape != moving_xyz.shape:
        raise ValueError("Coordinate arrays have different shapes.")

    ref_center = reference_xyz.mean(axis=0)
    mov_center = moving_xyz.mean(axis=0)

    ref_centered = reference_xyz - ref_center
    mov_centered = moving_xyz - mov_center

    covariance = mov_centered.T @ ref_centered

    U, S, Vt = np.linalg.svd(covariance)

    correction = np.eye(3)

    if np.linalg.det(U @ Vt) < 0:
        correction[-1, -1] = -1

    rotation = U @ correction @ Vt

    translation = ref_center - mov_center @ rotation

    aligned = moving_xyz @ rotation + translation

    rmsd = np.sqrt(
        np.mean(
            np.sum(
                (aligned - reference_xyz) ** 2,
                axis=1
            )
        )
    )

    return rotation, translation, aligned, rmsd

# ------------------------------------------------------------
# 4. DEFINE REFERENCE CONFORMER
# ------------------------------------------------------------

reference_id = 1

reference_df = protein_tables[reference_id]

reference_xyz_all = reference_df[
    ["x", "y", "z"]
].to_numpy(dtype=float)

# Pocket residues from the M1 feature identities
import json

M1_JSON = (
    Path("/content/drive/MyDrive/Member2_Project")
    / "member1_input"
    / "module1 parp1 files"
    / "pharmacophore_tensors.json"
)

with open(M1_JSON, "r") as f:
    m1_metadata_json = json.load(f)

metadata = m1_metadata_json["1"]["feature_metadata"]

print("M1 feature metadata loaded:", len(metadata))
pocket_residues = set()

for record in metadata:

    identity = str(record.get("identity", ""))

    for atom_identity in identity.split(";"):

        fields = atom_identity.split("|")

        if len(fields) >= 4:

            chain = fields[0].strip()
            residue_number = fields[1].strip()
            residue_name = fields[3].strip()

            pocket_residues.add(
                (chain, residue_number, residue_name)
            )

if not pocket_residues:
    raise ValueError("Could not identify pocket residues from M1 metadata.")

# ------------------------------------------------------------
# 5. ALIGN EACH CONFORMER TO CONFORMER 1
# ------------------------------------------------------------

alignment_records = []
transformations = {}

for conf_id in range(1, 6):

    moving_df = protein_tables[conf_id]

    common_keys = sorted(
        set(reference_df.index)
        & set(moving_df.index)
    )

    if len(common_keys) < 3:
        raise ValueError(
            f"Too few common atoms for conformer {conf_id}."
        )

    ref_common = reference_df.loc[common_keys]
    mov_common = moving_df.loc[common_keys]

    ref_xyz = ref_common[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    mov_xyz = mov_common[
        ["x", "y", "z"]
    ].to_numpy(dtype=float)

    rotation, translation, aligned_xyz, global_rmsd = (
        calculate_kabsch(ref_xyz, mov_xyz)
    )

    # --------------------------------------------------------
    # Pocket-local alignment quality
    # --------------------------------------------------------

    pocket_mask = np.array([
        (
            row["chain"],
            row["residue_number"],
            row["residue_name"]
        ) in pocket_residues
        for _, row in ref_common.iterrows()
    ])

    if pocket_mask.sum() < 3:
        raise ValueError(
            f"Too few common pocket atoms in conformer {conf_id}."
        )

    pocket_rmsd = np.sqrt(
        np.mean(
            np.sum(
                (
                    aligned_xyz[pocket_mask]
                    - ref_xyz[pocket_mask]
                ) ** 2,
                axis=1
            )
        )
    )

    # Direct RMSD before alignment
    direct_rmsd = np.sqrt(
        np.mean(
            np.sum(
                (mov_xyz - ref_xyz) ** 2,
                axis=1
            )
        )
    )

    transformations[conf_id] = {
        "rotation": rotation,
        "translation": translation,
        "common_atom_keys": common_keys,
        "global_rmsd": global_rmsd,
        "pocket_rmsd": pocket_rmsd
    }

    alignment_records.append({

        "conformer_id": conf_id,

        "common_protein_atoms": len(common_keys),

        "common_pocket_atoms": int(pocket_mask.sum()),

        "direct_global_rmsd_A": direct_rmsd,

        "aligned_global_rmsd_A": global_rmsd,

        "aligned_pocket_rmsd_A": pocket_rmsd,

        "determinant": np.linalg.det(rotation),

        "alignment_status": "PASS"
    })

alignment_df = pd.DataFrame(alignment_records)

# ------------------------------------------------------------
# 6. EXPORT RESULTS
# ------------------------------------------------------------

alignment_df.to_csv(
    OUTPUT_DIR / "M1_conformer_coordinate_alignment_audit.csv",
    index=False
)

# Save transformation matrices and translations
transform_rows = []

for conf_id, data in transformations.items():

    rotation = data["rotation"]
    translation = data["translation"]

    transform_rows.append({

        "conformer_id": conf_id,

        "rotation_matrix": rotation.tolist(),

        "translation_vector": translation.tolist(),

        "global_rmsd_A": data["global_rmsd"],

        "pocket_rmsd_A": data["pocket_rmsd"]
    })

import json

with open(
    OUTPUT_DIR / "M1_conformer_alignment_transformations.json",
    "w"
) as f:

    json.dump(transform_rows, f, indent=2)

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n========== CELL 83G-P9 RESULTS ==========")

print("Reference conformer:", reference_id)

print("Pocket residues identified:", len(pocket_residues))

print("\nConformer alignment audit:")

display(alignment_df)

print("\nSaved alignment transformations.")

print("\nOutput directory:", OUTPUT_DIR)

print("\nCell 83G-P9 completed.")

M1 feature metadata loaded: 95

========== CELL 83G-P9 RESULTS ==========
Reference conformer: 1
Pocket residues identified: 11

Conformer alignment audit:


,conformer_id,common_protein_atoms,common_pocket_atoms,direct_global_rmsd_A,aligned_global_rmsd_A,aligned_pocket_rmsd_A,determinant,alignment_status
0,1,5445,98,0.000000,1.150809e-14,9.703304e-15,1.0,PASS
1,2,5445,98,0.602913,6.028700e-01,3.742515e-01,1.0,PASS
2,3,5445,98,0.876797,8.766404e-01,3.336639e-01,1.0,PASS
3,4,5445,98,0.691208,6.910264e-01,3.552911e-01,1.0,PASS
4,5,5445,98,0.438398,4.383200e-01,1.668324e-01,1.0,PASS



Saved alignment transformations.

Output directory: /content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis

Cell 83G-P9 completed.


In [135]:
print("Transformation variable:", type(transformations))
print("Available conformers:", list(transformations.keys()))

for key, value in transformations.items():
    print(f"\nConformer {key}:")
    print("Stored fields:", value.keys())
    break

Transformation variable: <class 'dict'>
Available conformers: [1, 2, 3, 4, 5]

Conformer 1:
Stored fields: dict_keys(['rotation', 'translation', 'common_atom_keys', 'global_rmsd', 'pocket_rmsd'])


In [136]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Member2_Project")

folders = [
    PROJECT_ROOT / "member2_outputs",
    PROJECT_ROOT / "checkpoints",
]

print("========== CELL 84: TRAINING ARTIFACT AUDIT ==========")

for folder in folders:
    print(f"\nFolder: {folder}")

    if not folder.exists():
        print("Folder not found.")
        continue

    for file in sorted(folder.rglob("*")):
        if file.is_file():
            size_mb = file.stat().st_size / (1024 * 1024)
            if file.suffix.lower() in [
                ".pt", ".pth", ".json", ".jsonl",
                ".csv", ".npy", ".npz", ".txt"
            ]:
                print(f"{file.relative_to(PROJECT_ROOT)} | {size_mb:.2f} MB")

========== CELL 84: TRAINING ARTIFACT AUDIT ==========

Folder: /content/drive/MyDrive/Member2_Project/member2_outputs
member2_outputs/chembl37_clean_smiles.txt | 2.62 MB
member2_outputs/chembl37_general_graphs.jsonl | 135.83 MB
member2_outputs/chembl37_general_summary.json | 0.00 MB
member2_outputs/chembl37_general_validation.csv | 3.72 MB
member2_outputs/chembl37_parp_activity_records.csv | 0.70 MB
member2_outputs/chembl37_parp_activity_summary.csv | 0.00 MB
member2_outputs/chembl37_parp_activity_with_structures.csv | 2.59 MB
member2_outputs/chembl37_parp_assay_metadata.csv | 0.06 MB
member2_outputs/chembl37_parp_integrated_dataset.csv | 1.70 MB
member2_outputs/chembl37_parp_invalid_structures.csv | 0.00 MB
member2_outputs/chembl37_parp_matched_structures.csv | 1.23 MB
member2_outputs/chembl37_parp_standardized_activity.csv | 2.21 MB
member2_outputs/chembl37_parp_target_records.csv | 0.00 MB
member2_outputs/chembl37_parp_target_records.json | 0.00 MB
member2_outputs/chembl37_parp_val

In [137]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUT = PROJECT_ROOT / "member2_outputs"

print("========== CELL 85: TRANSFORMER DATA AUDIT ==========")

# 1. Existing molecular tokenizer
tokenizer_path = OUT / "smiles_tokenizer.json"

with open(tokenizer_path) as f:
    tokenizer_data = json.load(f)

print("\n1. TOKENIZER")
print("Keys:", tokenizer_data.keys())

for key, value in tokenizer_data.items():
    if isinstance(value, dict):
        print(key, "entries:", len(value))
    else:
        print(key, ":", str(value)[:200])

# 2. Existing token arrays
print("\n2. TOKEN ARRAYS")

for filename in ["chembl_train_tokens.npy", "chembl_val_tokens.npy"]:
    arr = np.load(OUT / filename, mmap_mode="r")
    print(filename, "| Shape:", arr.shape, "| dtype:", arr.dtype)

# 3. PARP integrated dataset
print("\n3. PARP INTEGRATED DATASET")

parp_path = OUT / "chembl37_parp_integrated_dataset.csv"

if parp_path.exists():
    parp_df = pd.read_csv(parp_path)
    print("Shape:", parp_df.shape)
    print("Columns:", parp_df.columns.tolist())
    print("\nSample:")
    print(parp_df.head(3).to_string(index=False))
else:
    print("File not found.")

# 4. PARP provisional clean dataset
print("\n4. PARP LEVEL-A PROVISIONAL DATASET")

level_a_path = OUT / "parp_level_A_provisional_clean_75.csv"

if level_a_path.exists():
    level_a_df = pd.read_csv(level_a_path)
    print("Shape:", level_a_df.shape)
    print("Columns:", level_a_df.columns.tolist())
    print(level_a_df.head(3).to_string(index=False))
else:
    print("File not found.")

# 5. Experimental target-specific ligand graphs
print("\n5. VALIDATED PARP1 REFERENCE GRAPHS")

target_graph_path = OUT / "validated_target_ligand_graphs.jsonl"

with open(target_graph_path) as f:
    target_records = [json.loads(line) for line in f if line.strip()]

print("Reference graph count:", len(target_records))
print("First record keys:", target_records[0].keys())

# 6. Existing GRU checkpoint
print("\n6. EXISTING GRU CHECKPOINT")

checkpoint_path = PROJECT_ROOT / "checkpoints" / "molecular_gru_best.pt"
print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint size (MB):", round(checkpoint_path.stat().st_size / 1e6, 2))

========== CELL 85: TRANSFORMER DATA AUDIT ==========

1. TOKENIZER
Keys: dict_keys(['vocabulary', 'token_to_id', 'max_length', 'special_tokens'])
vocabulary : ['<PAD>', '<BOS>', '<EOS>', '<UNK>', '#', '%10', '(', ')', '-', '/', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', 'B', 'Br', 'C', 'Cl', 'F', 'I', 'N', 'O', 'P', 'S', '[10B]', '[11CH3]', '[11C]', 
token_to_id entries: 81
max_length : 143
special_tokens : ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

2. TOKEN ARRAYS
chembl_train_tokens.npy | Shape: (45000, 143) | dtype: int32
chembl_val_tokens.npy | Shape: (5000, 143) | dtype: int32

3. PARP INTEGRATED DATASET
Shape: (8065, 22)
Columns: ['protein', 'target_chembl_id', 'activity_id', 'molecule_chembl_id', 'assay_chembl_id', 'document_chembl_id', 'standard_type', 'standard_relation', 'standard_value', 'standard_units', 'pchembl_value', 'data_validity_comment', 'potential_duplicate', 'chembl_id', 'canonical_smiles', 'inchikey', 'molecular_weight', 'heavy_atoms', 'heavy_bonds', 'num_ring

In [138]:
import pandas as pd
from pathlib import Path

OUT = PROJECT_ROOT / "member2_outputs"

df = pd.read_csv(OUT / "chembl37_parp_integrated_dataset.csv")

print("========== CELL 86: PARP ACTIVITY AUDIT ==========")

print("\n1. TARGET DISTRIBUTION")
print(df["protein"].value_counts(dropna=False))

print("\n2. ACTIVITY TYPE")
print(df["standard_type"].value_counts(dropna=False))

print("\n3. ACTIVITY RELATION")
print(df["standard_relation"].value_counts(dropna=False))

print("\n4. MEASUREMENT UNITS")
print(df["standard_units"].value_counts(dropna=False))

print("\n5. UNIQUE MOLECULES")
print(df["molecule_chembl_id"].nunique())

print("\n6. UNIQUE MOLECULES BY TARGET")
print(
    df.groupby("protein")["molecule_chembl_id"]
    .nunique()
)

print("\n7. TESTED AGAINST BOTH TARGETS")
print(df["tested_against_both"].value_counts(dropna=False))

print("\n8. VALID pChEMBL VALUES")
print(df.groupby("protein")["pchembl_value"].agg(
    ["count", "min", "median", "max"]
))

print("\n9. DUPLICATE AND MISSING STRUCTURES")
print("Missing SMILES:", df["canonical_smiles"].isna().sum())
print("Duplicate activity records:", df["activity_id"].duplicated().sum())

========== CELL 86: PARP ACTIVITY AUDIT ==========

1. TARGET DISTRIBUTION
protein
PARP1    6721
PARP2    1344
Name: count, dtype: int64

2. ACTIVITY TYPE
standard_type
IC50    6070
Ki      1243
EC50     420
Kd       332
Name: count, dtype: int64

3. ACTIVITY RELATION
standard_relation
=    7224
>     621
<     219
~       1
Name: count, dtype: int64

4. MEASUREMENT UNITS
standard_units
nM             8057
ug.mL-1           3
10^-10mol/L       2
10'8nM            1
10'13nM           1
10^-8mol/L        1
Name: count, dtype: int64

5. UNIQUE MOLECULES
5160

6. UNIQUE MOLECULES BY TARGET
protein
PARP1    5072
PARP2    1027
Name: molecule_chembl_id, dtype: int64

7. TESTED AGAINST BOTH TARGETS
tested_against_both
False    5152
True     2913
Name: count, dtype: int64

8. VALID pChEMBL VALUES
         count   min  median   max
protein                           
PARP1     5933  4.00    7.44  10.7
PARP2     1083  4.01    6.52  10.3

9. DUPLICATE AND MISSING STRUCTURES
Missing SMILES: 0
Duplic

In [139]:
import numpy as np
import json
from pathlib import Path

OUT = PROJECT_ROOT / "member2_outputs"

print("========== CELL 87: TRANSFORMER INPUT VALIDATION ==========")

# Load tokenizer
with open(OUT / "smiles_tokenizer.json") as f:
    tokenizer = json.load(f)

vocab = tokenizer["vocabulary"]
token_to_id = tokenizer["token_to_id"]

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

print("\n1. VOCABULARY")
print("Vocabulary size:", len(vocab))
print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)
print("Maximum sequence length:", tokenizer["max_length"])

# Load arrays
train = np.load(OUT / "chembl_train_tokens.npy")
val = np.load(OUT / "chembl_val_tokens.npy")

print("\n2. ARRAY SHAPES")
print("Training:", train.shape)
print("Validation:", val.shape)

# Validate token IDs
print("\n3. TOKEN ID VALIDATION")
print("Training IDs in range:",
      train.min() >= 0 and train.max() < len(vocab))
print("Validation IDs in range:",
      val.min() >= 0 and val.max() < len(vocab))

# Special token counts
print("\n4. SPECIAL TOKEN COUNTS")

for name, token_id in [
    ("PAD", PAD_ID),
    ("BOS", BOS_ID),
    ("EOS", EOS_ID),
    ("UNK", UNK_ID)
]:
    print(
        name,
        "| Train:", int((train == token_id).sum()),
        "| Validation:", int((val == token_id).sum())
    )

# Sequence lengths
print("\n5. SEQUENCE LENGTHS")

train_lengths = (train != PAD_ID).sum(axis=1)
val_lengths = (val != PAD_ID).sum(axis=1)

print("Train - min/median/max:",
      train_lengths.min(),
      np.median(train_lengths),
      train_lengths.max())

print("Validation - min/median/max:",
      val_lengths.min(),
      np.median(val_lengths),
      val_lengths.max())

# Unknown token proportion
print("\n6. UNKNOWN TOKEN PROPORTION")

print("Train:", round((train == UNK_ID).sum() / train.size, 6))
print("Validation:", round((val == UNK_ID).sum() / val.size, 6))

print("\nCell 87 audit completed.")

========== CELL 87: TRANSFORMER INPUT VALIDATION ==========

1. VOCABULARY
Vocabulary size: 81
PAD: 0
BOS: 1
EOS: 2
UNK: 3
Maximum sequence length: 143

2. ARRAY SHAPES
Training: (45000, 143)
Validation: (5000, 143)

3. TOKEN ID VALIDATION
Training IDs in range: True
Validation IDs in range: True

4. SPECIAL TOKEN COUNTS
PAD | Train: 4123646 | Validation: 457723
BOS | Train: 45000 | Validation: 5000
EOS | Train: 45000 | Validation: 5000
UNK | Train: 0 | Validation: 0

5. SEQUENCE LENGTHS
Train - min/median/max: 9 49.0 143
Validation - min/median/max: 11 50.0 130

6. UNKNOWN TOKEN PROPORTION
Train: 0.0
Validation: 0.0

Cell 87 audit completed.


In [140]:
# ============================================================
# CELL 88: DYNAMIC M1 -> M2 INTERFACE VALIDATION
# ============================================================

import json
import hashlib
import re
import numpy as np

from pathlib import Path
from datetime import datetime, timezone

print("=" * 65)
print("CELL 88: DYNAMIC M1 -> M2 INTERFACE VALIDATION")
print("=" * 65)

PROJECT_ROOT = Path("/content/drive/MyDrive/Member2_Project")
M1_ROOT = PROJECT_ROOT / "member1_input"
OUTPUT_DIR = PROJECT_ROOT / "member2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. DISCOVER M1 METADATA
# ------------------------------------------------------------

metadata_files = list(M1_ROOT.rglob("pharmacophore_tensors.json"))

if len(metadata_files) != 1:
    raise RuntimeError(
        f"Expected exactly one M1 metadata JSON, found "
        f"{len(metadata_files)}: {metadata_files}"
    )

metadata_path = metadata_files[0]

with open(metadata_path, "r") as f:
    m1_json = json.load(f)

print("\n1. METADATA DISCOVERY")
print("Metadata file:", metadata_path)
print("Available pocket IDs:", list(m1_json.keys()))

# ------------------------------------------------------------
# 2. FILE HASH FUNCTION
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. DYNAMIC POCKET DISCOVERY AND VALIDATION
# ------------------------------------------------------------

pocket_contracts = {}
validation_results = []

for pocket_id, pocket_data in m1_json.items():

    print(f"\nProcessing pocket: {pocket_id}")

    if not isinstance(pocket_data, dict):
        raise TypeError(
            f"Pocket {pocket_id}: metadata must be a dictionary."
        )

    # Discover corresponding files.
    tensor_candidates = [
        p for p in M1_ROOT.rglob("*.npy")
        if re.search(
            rf"pocket[_-]?{re.escape(str(pocket_id))}.*pharmacophore",
            p.name,
            flags=re.IGNORECASE
        )
    ]

    mask_candidates = [
        p for p in M1_ROOT.rglob("*.npy")
        if re.search(
            rf"pocket[_-]?{re.escape(str(pocket_id))}.*mask",
            p.name,
            flags=re.IGNORECASE
        )
    ]

    if len(tensor_candidates) != 1:
        raise RuntimeError(
            f"Pocket {pocket_id}: expected one tensor file, "
            f"found {len(tensor_candidates)}: {tensor_candidates}"
        )

    if len(mask_candidates) != 1:
        raise RuntimeError(
            f"Pocket {pocket_id}: expected one mask file, "
            f"found {len(mask_candidates)}: {mask_candidates}"
        )

    tensor_path = tensor_candidates[0]
    mask_path = mask_candidates[0]

    # Load without assuming fixed dimensions.
    tensor = np.load(tensor_path, allow_pickle=False)
    mask = np.load(mask_path, allow_pickle=False)

    # Basic shape validation.
    if tensor.ndim != 3:
        raise ValueError(
            f"Pocket {pocket_id}: expected 3D tensor "
            f"(conformers, features, channels), got {tensor.shape}"
        )

    if mask.ndim != 2:
        raise ValueError(
            f"Pocket {pocket_id}: expected 2D mask, got {mask.shape}"
        )

    conformer_count, feature_count, channel_count = tensor.shape

    if mask.shape != tensor.shape[:2]:
        raise ValueError(
            f"Pocket {pocket_id}: mask shape {mask.shape} "
            f"does not match tensor dimensions {tensor.shape[:2]}"
        )

    # Metadata consistency.
    declared_conformers = pocket_data.get("num_conformations")
    declared_features = pocket_data.get("num_features")
    declared_tensor_shape = pocket_data.get("tensor_shape")
    declared_mask_shape = pocket_data.get("mask_shape")

    if declared_conformers is not None:
        assert int(declared_conformers) == conformer_count

    if declared_features is not None:
        assert int(declared_features) == feature_count

    if declared_tensor_shape is not None:
        assert tuple(declared_tensor_shape) == tensor.shape

    if declared_mask_shape is not None:
        assert tuple(declared_mask_shape) == mask.shape

    # Numeric validation.
    if not np.issubdtype(tensor.dtype, np.number):
        raise TypeError("Pocket tensor must contain numeric values.")

    if not np.isfinite(tensor).all():
        raise ValueError("Pocket tensor contains NaN or infinity.")

    if not np.isin(mask, [0, 1]).all():
        raise ValueError("Pocket mask must contain only 0 and 1.")

    # Feature metadata validation.
    feature_metadata = pocket_data.get("feature_metadata")

    if not isinstance(feature_metadata, list):
        raise ValueError("Feature metadata must be a list.")

    if len(feature_metadata) != feature_count:
        raise ValueError(
            f"Feature metadata count {len(feature_metadata)} "
            f"does not match tensor feature count {feature_count}."
        )

    # Verify feature identities.
    missing_identities = sum(
        not isinstance(item, dict)
        or not item.get("identity")
        for item in feature_metadata
    )

    # Check whether valid mask positions contain meaningful features.
    valid_positions = int(mask.sum())

    # Create stable pocket fingerprint.
    fingerprint = hashlib.sha256(
        (
            sha256_file(tensor_path)
            + sha256_file(mask_path)
            + sha256_file(metadata_path)
        ).encode()
    ).hexdigest()

    # Standardized contract.
    pocket_contracts[str(pocket_id)] = {
        "pocket_id": str(pocket_id),

        "tensor_path": str(tensor_path),
        "mask_path": str(mask_path),

        "tensor_shape": list(tensor.shape),
        "mask_shape": list(mask.shape),

        "num_conformers": int(conformer_count),
        "num_features": int(feature_count),
        "num_channels": int(channel_count),

        "tensor_dtype": str(tensor.dtype),
        "mask_dtype": str(mask.dtype),

        "valid_feature_positions": valid_positions,
        "masked_feature_positions": int(mask.size - valid_positions),

        "feature_metadata_count": len(feature_metadata),
        "missing_feature_identities": int(missing_identities),

        "feature_types": sorted(set(
            str(item.get("feature", item.get("identity", "UNKNOWN")))
            for item in feature_metadata
            if isinstance(item, dict)
        )),

        "tensor_sha256": sha256_file(tensor_path),
        "mask_sha256": sha256_file(mask_path),
        "metadata_sha256": sha256_file(metadata_path),

        "input_fingerprint": fingerprint
    }

    validation_results.append({
        "pocket_id": str(pocket_id),
        "tensor_shape": tensor.shape,
        "mask_shape": mask.shape,
        "conformers": conformer_count,
        "features": feature_count,
        "channels": channel_count,
        "valid_positions": valid_positions,
        "missing_identities": missing_identities,
        "status": "PASS"
    })

    print("Tensor shape:", tensor.shape)
    print("Mask shape:", mask.shape)
    print("Valid feature positions:", valid_positions)
    print("Missing feature identities:", missing_identities)
    print("Validation: PASS")


# ------------------------------------------------------------
# 4. CREATE STANDARDIZED M1 -> M2 CONTRACT
# ------------------------------------------------------------

interface_contract = {
    "interface_version": "1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "source_module": "Member-1",
    "destination_module": "Member-2",

    "metadata_file": str(metadata_path),

    "num_pockets": len(pocket_contracts),

    "pockets": pocket_contracts,

    "validation_status": (
        "PASS" if all(
            item["status"] == "PASS"
            for item in validation_results
        ) else "FAIL"
    )
}

contract_path = OUTPUT_DIR / "m1_to_m2_interface_contract.json"

with open(contract_path, "w") as f:
    json.dump(interface_contract, f, indent=2)

# ------------------------------------------------------------
# 5. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 88 FINAL REPORT")
print("=" * 65)

print("Pockets discovered:", len(pocket_contracts))
print("Metadata file:", metadata_path.name)

for result in validation_results:
    print("\nPocket:", result["pocket_id"])
    print("Tensor:", result["tensor_shape"])
    print("Mask:", result["mask_shape"])
    print("Conformers:", result["conformers"])
    print("Features:", result["features"])
    print("Channels:", result["channels"])
    print("Status:", result["status"])

print("\nOverall status:", interface_contract["validation_status"])
print("Contract saved:", contract_path)

print("\nM1 input files were only read, not modified.")
print("Cell 88 completed.")

CELL 88: DYNAMIC M1 -> M2 INTERFACE VALIDATION

1. METADATA DISCOVERY
Metadata file: /content/drive/MyDrive/Member2_Project/member1_input/module1 parp1 files/pharmacophore_tensors.json
Available pocket IDs: ['1']

Processing pocket: 1
Tensor shape: (5, 95, 4)
Mask shape: (5, 95)
Valid feature positions: 475
Missing feature identities: 0
Validation: PASS

CELL 88 FINAL REPORT
Pockets discovered: 1
Metadata file: pharmacophore_tensors.json

Pocket: 1
Tensor: (5, 95, 4)
Mask: (5, 95)
Conformers: 5
Features: 95
Channels: 4
Status: PASS

Overall status: PASS
Contract saved: /content/drive/MyDrive/Member2_Project/member2_outputs/m1_to_m2_interface_contract.json

M1 input files were only read, not modified.
Cell 88 completed.


In [141]:
# ============================================================
# CELL 89: DYNAMIC M1 POCKET LOADER
# ============================================================

import json
import hashlib
import numpy as np

from pathlib import Path

print("=" * 65)
print("CELL 89: DYNAMIC M1 POCKET LOADER")
print("=" * 65)

CONTRACT_PATH = (
    PROJECT_ROOT / "member2_outputs"
    / "m1_to_m2_interface_contract.json"
)

# ------------------------------------------------------------
# 1. LOAD INTERFACE CONTRACT
# ------------------------------------------------------------

with open(CONTRACT_PATH) as f:
    contract = json.load(f)

if contract["validation_status"] != "PASS":
    raise RuntimeError("M1 interface contract has not passed validation.")

# ------------------------------------------------------------
# 2. REUSABLE DYNAMIC LOADER
# ------------------------------------------------------------

def load_m1_pockets(contract_path):

    with open(contract_path) as f:
        interface = json.load(f)

    if interface["validation_status"] != "PASS":
        raise ValueError("Invalid M1 interface contract.")

    loaded_pockets = {}

    for pocket_id, info in interface["pockets"].items():

        tensor_path = Path(info["tensor_path"])
        mask_path = Path(info["mask_path"])
        metadata_path = Path(interface["metadata_file"])

        # Verify source files still match the contract.
        def file_hash(path):
            h = hashlib.sha256()

            with open(path, "rb") as f:
                for chunk in iter(
                    lambda: f.read(1024 * 1024), b""
                ):
                    h.update(chunk)

            return h.hexdigest()

        if file_hash(tensor_path) != info["tensor_sha256"]:
            raise RuntimeError(
                f"Pocket {pocket_id}: tensor changed after validation."
            )

        if file_hash(mask_path) != info["mask_sha256"]:
            raise RuntimeError(
                f"Pocket {pocket_id}: mask changed after validation."
            )

        if file_hash(metadata_path) != info["metadata_sha256"]:
            raise RuntimeError(
                f"Pocket {pocket_id}: metadata changed after validation."
            )

        # Load arrays.
        tensor = np.load(tensor_path, allow_pickle=False)
        mask = np.load(mask_path, allow_pickle=False)

        with open(metadata_path) as f:
            metadata_json = json.load(f)

        pocket_metadata = metadata_json[pocket_id]
        feature_metadata = pocket_metadata["feature_metadata"]

        # Dynamic shape validation.
        if tensor.ndim != 3:
            raise ValueError("Pocket tensor must have 3 dimensions.")

        if mask.shape != tensor.shape[:2]:
            raise ValueError("Tensor and mask dimensions do not match.")

        if len(feature_metadata) != tensor.shape[1]:
            raise ValueError("Feature metadata and tensor do not match.")

        if not np.isfinite(tensor).all():
            raise ValueError("Non-finite tensor values detected.")

        # Dynamically construct feature-type vocabulary.
        feature_types = [
            str(item.get("feature", "UNKNOWN"))
            for item in feature_metadata
        ]

        unique_types = sorted(set(feature_types))

        feature_type_to_id = {
            name: index
            for index, name in enumerate(unique_types)
        }

        feature_type_ids = np.array(
            [feature_type_to_id[name] for name in feature_types],
            dtype=np.int64
        )

        # Keep feature identities and metadata intact.
        loaded_pockets[str(pocket_id)] = {

            "pocket_id": str(pocket_id),

            "tensor": tensor.astype(np.float32),

            "mask": mask.astype(bool),

            "feature_metadata": feature_metadata,

            "feature_types": feature_types,

            "feature_type_ids": feature_type_ids,

            "feature_type_to_id": feature_type_to_id,

            "num_conformers": tensor.shape[0],

            "num_features": tensor.shape[1],

            "num_channels": tensor.shape[2],

            "input_fingerprint": info["input_fingerprint"]
        }

    return loaded_pockets


# ------------------------------------------------------------
# 3. LOAD ALL AVAILABLE POCKETS
# ------------------------------------------------------------

m1_pockets = load_m1_pockets(CONTRACT_PATH)

# ------------------------------------------------------------
# 4. REPORT
# ------------------------------------------------------------

print("\nLOADER RESULTS")

for pocket_id, pocket in m1_pockets.items():

    print("\nPocket:", pocket_id)

    print("Tensor:", pocket["tensor"].shape)
    print("Mask:", pocket["mask"].shape)

    print("Conformers:", pocket["num_conformers"])
    print("Features:", pocket["num_features"])
    print("Channels:", pocket["num_channels"])

    print(
        "Feature vocabulary:",
        pocket["feature_type_to_id"]
    )

    print(
        "Valid positions:",
        int(pocket["mask"].sum())
    )

    print(
        "Input fingerprint:",
        pocket["input_fingerprint"][:16]
    )

print("\nTotal pockets loaded:", len(m1_pockets))
print("M1 files modified: NO")
print("\nCELL 89 COMPLETED")

CELL 89: DYNAMIC M1 POCKET LOADER

LOADER RESULTS

Pocket: 1
Tensor: (5, 95, 4)
Mask: (5, 95)
Conformers: 5
Features: 95
Channels: 4
Feature vocabulary: {'ACCEPTOR': 0, 'AROMATIC': 1, 'DONOR': 2, 'HYDROPHOBIC': 3, 'NEGATIVE': 4, 'POSITIVE': 5}
Valid positions: 475
Input fingerprint: 912feb501681aa4c

Total pockets loaded: 1
M1 files modified: NO

CELL 89 COMPLETED


In [142]:
# ============================================================
# CELL 90: M1 TENSOR CHANNEL SEMANTICS AUDIT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 90: M1 TENSOR CHANNEL SEMANTICS AUDIT")
print("=" * 65)

for pocket_id, pocket in m1_pockets.items():

    tensor = pocket["tensor"]
    mask = pocket["mask"]
    metadata = pocket["feature_metadata"]

    print(f"\nPOCKET: {pocket_id}")
    print("Tensor shape:", tensor.shape)

    # --------------------------------------------------------
    # 1. CHANNEL STATISTICS
    # --------------------------------------------------------

    print("\n1. CHANNEL STATISTICS")

    for channel in range(tensor.shape[-1]):

        values = tensor[:, :, channel][mask]

        print(
            f"Channel {channel}: "
            f"min={values.min():.5f}, "
            f"max={values.max():.5f}, "
            f"mean={values.mean():.5f}, "
            f"std={values.std():.5f}, "
            f"unique={len(np.unique(values))}"
        )

    # --------------------------------------------------------
    # 2. CHECK FOURTH CHANNEL AGAINST FEATURE IDS
    # --------------------------------------------------------

    print("\n2. FEATURE ID COMPARISON")

    metadata_ids = np.array([
        int(item["feature_id"])
        for item in metadata
    ])

    tensor_ids = tensor[0, :, 3]

    print("Metadata feature IDs:", np.unique(metadata_ids))
    print("Tensor channel-3 values:", np.unique(tensor_ids))

    print(
        "Channel 3 matches metadata feature IDs:",
        np.array_equal(
            tensor_ids.astype(int),
            metadata_ids
        )
    )

    # --------------------------------------------------------
    # 3. DISPLAY SAMPLE FEATURES
    # --------------------------------------------------------

    print("\n3. SAMPLE FEATURE COMPARISON")

    rows = []

    for index in range(min(10, tensor.shape[1])):

        rows.append({
            "feature_index": index,
            "metadata_type": metadata[index].get("feature"),
            "metadata_feature_id": metadata[index].get("feature_id"),
            "tensor_x": tensor[0, index, 0],
            "tensor_y": tensor[0, index, 1],
            "tensor_z": tensor[0, index, 2],
            "tensor_channel_4": tensor[0, index, 3]
        })

    display(pd.DataFrame(rows))

    # --------------------------------------------------------
    # 4. FEATURE TYPE COUNTS
    # --------------------------------------------------------

    print("\n4. FEATURE TYPE COUNTS")

    feature_counts = pd.Series(
        pocket["feature_types"]
    ).value_counts()

    print(feature_counts)

    # --------------------------------------------------------
    # 5. COORDINATE FINITENESS
    # --------------------------------------------------------

    xyz = tensor[:, :, :3]

    print("\n5. COORDINATE VALIDATION")
    print("All coordinates finite:", np.isfinite(xyz).all())
    print("Coordinate shape:", xyz.shape)

print("\nCELL 90 COMPLETED")

CELL 90: M1 TENSOR CHANNEL SEMANTICS AUDIT

POCKET: 1
Tensor shape: (5, 95, 4)

1. CHANNEL STATISTICS
Channel 0: min=11.56100, max=31.00200, mean=23.24983, std=5.28522, unique=413
Channel 1: min=32.66600, max=48.17200, mean=41.14394, std=3.01240, unique=405
Channel 2: min=-5.61700, max=15.02400, mean=5.72074, std=5.05067, unique=410
Channel 3: min=0.00000, max=5.00000, mean=1.52632, std=1.11301, unique=6

2. FEATURE ID COMPARISON
Metadata feature IDs: [0 1 2 3 4 5]
Tensor channel-3 values: [0. 1. 2. 3. 4. 5.]
Channel 3 matches metadata feature IDs: True

3. SAMPLE FEATURE COMPARISON


,feature_index,metadata_type,metadata_feature_id,tensor_x,tensor_y,tensor_z,tensor_channel_4
0,0,HYDROPHOBIC,2,25.072001,40.769001,-2.4720,2.0
1,1,AROMATIC,3,24.822001,40.862000,0.1456,3.0
2,2,POSITIVE,4,24.822001,40.862000,0.1456,4.0
3,3,DONOR,0,27.455000,40.000999,-2.4300,0.0
4,4,DONOR,0,24.573999,41.905998,-0.3050,0.0
5,5,POSITIVE,4,24.573999,41.905998,-0.3050,4.0
6,6,DONOR,0,24.884001,40.306000,1.1480,0.0
7,7,ACCEPTOR,1,25.372999,40.436001,-5.2010,1.0
8,8,HYDROPHOBIC,2,22.134001,44.251999,3.2230,2.0
9,9,HYDROPHOBIC,2,22.655001,43.909000,5.6760,2.0



4. FEATURE TYPE COUNTS
HYDROPHOBIC    45
DONOR          22
ACCEPTOR       18
POSITIVE        5
AROMATIC        4
NEGATIVE        1
Name: count, dtype: int64

5. COORDINATE VALIDATION
All coordinates finite: True
Coordinate shape: (5, 95, 3)

CELL 90 COMPLETED


In [143]:
# ============================================================
# CELL 91: DYNAMIC M1 POCKET ENCODER
# ============================================================

import torch
import torch.nn as nn
import numpy as np

print("=" * 65)
print("CELL 91: DYNAMIC M1 POCKET ENCODER")
print("=" * 65)

# ------------------------------------------------------------
# 1. DEVICE
# ------------------------------------------------------------

device = torch.device("cpu")

print("Device:", device)

# ------------------------------------------------------------
# 2. POCKET ENCODER
# ------------------------------------------------------------

class DynamicPocketEncoder(nn.Module):

    def __init__(
        self,
        num_feature_types,
        embedding_dim=128
    ):
        super().__init__()

        self.embedding_dim = embedding_dim

        # Learnable representation of pharmacophore types
        self.feature_embedding = nn.Embedding(
            num_embeddings=num_feature_types,
            embedding_dim=embedding_dim
        )

        # Convert XYZ coordinates into embedding space
        self.coordinate_projection = nn.Sequential(
            nn.Linear(3, embedding_dim),
            nn.GELU(),
            nn.Linear(embedding_dim, embedding_dim)
        )

        # Combine coordinate and feature representations
        self.feature_fusion = nn.Sequential(
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.GELU(),
            nn.LayerNorm(embedding_dim)
        )

    def forward(self, coordinates, feature_ids, mask):

        coordinate_embeddings = self.coordinate_projection(
            coordinates
        )

        feature_embeddings = self.feature_embedding(
            feature_ids
        )

        combined = torch.cat(
            [
                coordinate_embeddings,
                feature_embeddings
            ],
            dim=-1
        )

        pocket_embeddings = self.feature_fusion(combined)

        # Remove invalid positions
        pocket_embeddings = pocket_embeddings * mask.unsqueeze(-1)

        return pocket_embeddings


# ------------------------------------------------------------
# 3. DYNAMIC FEATURE VOCABULARY
# ------------------------------------------------------------

feature_names = sorted({
    feature
    for pocket in m1_pockets.values()
    for feature in pocket["feature_types"]
})

feature_to_id = {
    name: index
    for index, name in enumerate(feature_names)
}

print("\nFeature vocabulary:")
print(feature_to_id)

# ------------------------------------------------------------
# 4. INITIALIZE ENCODER
# ------------------------------------------------------------

pocket_encoder = DynamicPocketEncoder(
    num_feature_types=len(feature_to_id),
    embedding_dim=128
).to(device)

# ------------------------------------------------------------
# 5. PROCESS ALL POCKETS
# ------------------------------------------------------------

pocket_embeddings_by_id = {}

for pocket_id, pocket in m1_pockets.items():

    tensor = pocket["tensor"]

    # Extract coordinates
    coordinates = tensor[:, :, :3].copy()

    # Extract feature types from metadata
    feature_ids = np.array([
        feature_to_id[item["feature"]]
        for item in pocket["feature_metadata"]
    ])

    # Convert mask
    mask = pocket["mask"].astype(bool)

    # Coordinate normalization
    coordinate_mean = coordinates[mask].mean(axis=0)
    coordinate_std = coordinates[mask].std(axis=0)

    coordinate_std = np.where(
        coordinate_std < 1e-6,
        1.0,
        coordinate_std
    )

    normalized_coordinates = (
        coordinates - coordinate_mean
    ) / coordinate_std

    # Convert to PyTorch tensors
    coordinates_tensor = torch.tensor(
        normalized_coordinates,
        dtype=torch.float32,
        device=device
    )

    feature_ids_tensor = torch.tensor(
        feature_ids,
        dtype=torch.long,
        device=device
    )

    mask_tensor = torch.tensor(
        mask,
        dtype=torch.bool,
        device=device
    )

    # Encode
    pocket_encoder.eval()

    with torch.no_grad():

        embeddings = pocket_encoder(
            coordinates_tensor,
            feature_ids_tensor.unsqueeze(0).expand(
                coordinates_tensor.shape[0], -1
            ),
            mask_tensor
        )

    pocket_embeddings_by_id[pocket_id] = {
        "embeddings": embeddings,
        "mask": mask_tensor,
        "coordinate_mean": coordinate_mean,
        "coordinate_std": coordinate_std,
        "feature_vocabulary": feature_to_id
    }

    print(f"\nPocket: {pocket_id}")
    print("Input shape:", tensor.shape)
    print("Embedding shape:", tuple(embeddings.shape))
    print("Mask shape:", tuple(mask_tensor.shape))


# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ENCODER VALIDATION")
print("=" * 65)

all_passed = True

for pocket_id, result in pocket_embeddings_by_id.items():

    embeddings = result["embeddings"]
    mask = result["mask"]

    checks = {
        "Correct conformer count": embeddings.shape[0] == m1_pockets[pocket_id]["tensor"].shape[0],
        "Correct feature count": embeddings.shape[1] == m1_pockets[pocket_id]["tensor"].shape[1],
        "Embedding dimension is 128": embeddings.shape[2] == 128,
        "All embeddings finite": torch.isfinite(embeddings).all().item(),
        "Invalid positions zeroed": torch.all(
            embeddings[~mask] == 0
        ).item()
    }

    for name, passed in checks.items():

        print(f"{name}: {'PASS' if passed else 'FAIL'}")

        if not passed:
            all_passed = False

print("\nFINAL STATUS:",
      "PASS" if all_passed else "FAIL")

print("\nCELL 91 COMPLETED")

CELL 91: DYNAMIC M1 POCKET ENCODER
Device: cpu

Feature vocabulary:
{'ACCEPTOR': 0, 'AROMATIC': 1, 'DONOR': 2, 'HYDROPHOBIC': 3, 'NEGATIVE': 4, 'POSITIVE': 5}

Pocket: 1
Input shape: (5, 95, 4)
Embedding shape: (5, 95, 128)
Mask shape: (5, 95)

ENCODER VALIDATION
Correct conformer count: PASS
Correct feature count: PASS
Embedding dimension is 128: PASS
All embeddings finite: PASS
Invalid positions zeroed: PASS

FINAL STATUS: PASS

CELL 91 COMPLETED


In [144]:
# ============================================================
# CELL 92: POCKET EMBEDDING QUALITY AUDIT
# ============================================================

import torch
import numpy as np
import pandas as pd
from itertools import combinations
import torch.nn.functional as F

print("=" * 65)
print("CELL 92: POCKET EMBEDDING QUALITY AUDIT")
print("=" * 65)

audit_results = []

for pocket_id, result in pocket_embeddings_by_id.items():

    embeddings = result["embeddings"]
    mask = result["mask"]

    num_conformers, num_features, embedding_dim = embeddings.shape

    print(f"\nPOCKET: {pocket_id}")
    print("Embedding shape:", tuple(embeddings.shape))

    # --------------------------------------------------------
    # 1. NUMERICAL QUALITY
    # --------------------------------------------------------

    valid_embeddings = embeddings[mask]

    print("\n1. NUMERICAL QUALITY")

    numerical_checks = {
        "All values finite": torch.isfinite(embeddings).all().item(),
        "No NaN values": not torch.isnan(embeddings).any().item(),
        "No infinite values": not torch.isinf(embeddings).any().item(),
        "Non-zero embedding variation": embeddings.std().item() > 0
    }

    for name, passed in numerical_checks.items():
        print(f"{name}: {'PASS' if passed else 'FAIL'}")

    # --------------------------------------------------------
    # 2. EMBEDDING MAGNITUDE
    # --------------------------------------------------------

    norms = torch.linalg.vector_norm(
        valid_embeddings,
        dim=-1
    )

    print("\n2. EMBEDDING MAGNITUDE")
    print("Mean norm:", round(norms.mean().item(), 5))
    print("Minimum norm:", round(norms.min().item(), 5))
    print("Maximum norm:", round(norms.max().item(), 5))

    # --------------------------------------------------------
    # 3. CONFORMER DIFFERENCE AUDIT
    # --------------------------------------------------------

    print("\n3. CONFORMER DIFFERENCE AUDIT")

    conformer_records = []

    for c1, c2 in combinations(range(num_conformers), 2):

        common_mask = mask[c1] & mask[c2]

        e1 = embeddings[c1][common_mask]
        e2 = embeddings[c2][common_mask]

        mean_difference = torch.mean(
            torch.linalg.vector_norm(e1 - e2, dim=-1)
        ).item()

        cosine_similarity = F.cosine_similarity(
            e1,
            e2,
            dim=-1
        ).mean().item()

        conformer_records.append({
            "conformer_1": c1 + 1,
            "conformer_2": c2 + 1,
            "mean_embedding_difference": mean_difference,
            "mean_cosine_similarity": cosine_similarity
        })

    conformer_df = pd.DataFrame(conformer_records)

    display(conformer_df)

    # --------------------------------------------------------
    # 4. FEATURE EMBEDDING CONSISTENCY
    # --------------------------------------------------------

    print("\n4. FEATURE EMBEDDING CONSISTENCY")

    feature_names = result["feature_vocabulary"]

    feature_records = []

    for feature_name, feature_id in feature_names.items():

        embedding_vector = pocket_encoder.feature_embedding.weight[
            feature_id
        ].detach()

        feature_records.append({
            "feature": feature_name,
            "feature_id": feature_id,
            "embedding_norm": torch.norm(embedding_vector).item()
        })

    feature_df = pd.DataFrame(feature_records)

    display(feature_df)

    # --------------------------------------------------------
    # 5. MASK VALIDATION
    # --------------------------------------------------------

    print("\n5. MASK VALIDATION")

    invalid_count = (~mask).sum().item()

    invalid_zero = torch.all(
        embeddings[~mask] == 0
    ).item()

    print("Valid positions:", mask.sum().item())
    print("Invalid positions:", invalid_count)
    print("Invalid embeddings zeroed:", invalid_zero)

    # --------------------------------------------------------
    # 6. FINAL AUDIT STATUS
    # --------------------------------------------------------

    checks = [
        all(numerical_checks.values()),
        invalid_zero,
        len(conformer_records) == (
            num_conformers * (num_conformers - 1) // 2
        )
    ]

    passed = all(checks)

    audit_results.append({
        "pocket_id": pocket_id,
        "conformers": num_conformers,
        "features": num_features,
        "embedding_dimension": embedding_dim,
        "mean_embedding_norm": norms.mean().item(),
        "mean_conformer_difference": conformer_df[
            "mean_embedding_difference"
        ].mean(),
        "status": "PASS" if passed else "FAIL"
    })


print("\n" + "=" * 65)
print("FINAL EMBEDDING AUDIT")
print("=" * 65)

audit_df = pd.DataFrame(audit_results)
display(audit_df)

print(
    "\nOVERALL STATUS:",
    "PASS" if all(x["status"] == "PASS" for x in audit_results)
    else "FAIL"
)

print("\nCELL 92 COMPLETED")

CELL 92: POCKET EMBEDDING QUALITY AUDIT

POCKET: 1
Embedding shape: (5, 95, 128)

1. NUMERICAL QUALITY
All values finite: PASS
No NaN values: PASS
No infinite values: PASS
Non-zero embedding variation: PASS

2. EMBEDDING MAGNITUDE
Mean norm: 11.31262
Minimum norm: 11.31195
Maximum norm: 11.31299

3. CONFORMER DIFFERENCE AUDIT


,conformer_1,conformer_2,mean_embedding_difference,mean_cosine_similarity
0,1,2,0.081701,0.999970
1,1,3,0.090623,0.999963
2,1,4,0.083372,0.999963
3,1,5,0.045221,0.999991
4,2,3,0.083362,0.999963
5,2,4,0.139513,0.999903
6,2,5,0.069690,0.999976
7,3,4,0.081726,0.999970
8,3,5,0.045406,0.999991
9,4,5,0.069844,0.999976



4. FEATURE EMBEDDING CONSISTENCY


,feature,feature_id,embedding_norm
0,ACCEPTOR,0,9.998911
1,AROMATIC,1,12.672548
2,DONOR,2,10.645140
3,HYDROPHOBIC,3,12.368545
4,NEGATIVE,4,11.036303
5,POSITIVE,5,12.297628



5. MASK VALIDATION
Valid positions: 475
Invalid positions: 0
Invalid embeddings zeroed: True

FINAL EMBEDDING AUDIT


,pocket_id,conformers,features,embedding_dimension,mean_embedding_norm,mean_conformer_difference,status
0,1,5,95,128,11.312623,0.079046,PASS



OVERALL STATUS: PASS

CELL 92 COMPLETED


In [145]:
# ============================================================
# CELL 93: MOLECULAR TRANSFORMER INPUT AUDIT
# ============================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 93: MOLECULAR TRANSFORMER INPUT AUDIT")
print("=" * 65)

ROOT = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")

CHECKPOINT_DIR = os.path.join(ROOT, "checkpoints")

# ------------------------------------------------------------
# 1. LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    OUTPUT_DIR,
    "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

print("\n1. TOKENIZER")

print("Tokenizer file:", tokenizer_path)
print("Top-level structure:", type(tokenizer_data).__name__)

if isinstance(tokenizer_data, dict):
    print("Tokenizer keys:", list(tokenizer_data.keys()))

# ------------------------------------------------------------
# 2. LOAD TRAINING AND VALIDATION ARRAYS
# ------------------------------------------------------------

train_path = os.path.join(
    OUTPUT_DIR,
    "chembl_train_tokens.npy"
)

val_path = os.path.join(
    OUTPUT_DIR,
    "chembl_val_tokens.npy"
)

train_tokens = np.load(train_path)
val_tokens = np.load(val_path)

print("\n2. TOKEN ARRAYS")

print("Training shape:", train_tokens.shape)
print("Validation shape:", val_tokens.shape)

print("Training dtype:", train_tokens.dtype)
print("Validation dtype:", val_tokens.dtype)

# ------------------------------------------------------------
# 3. TOKEN VALUE VALIDATION
# ------------------------------------------------------------

print("\n3. TOKEN VALUE VALIDATION")

print("Training minimum ID:", train_tokens.min())
print("Training maximum ID:", train_tokens.max())

print("Validation minimum ID:", val_tokens.min())
print("Validation maximum ID:", val_tokens.max())

print("Training unique tokens:", len(np.unique(train_tokens)))
print("Validation unique tokens:", len(np.unique(val_tokens)))

# ------------------------------------------------------------
# 4. SPECIAL TOKEN COUNTS
# ------------------------------------------------------------

print("\n4. SPECIAL TOKEN COUNTS")

special_tokens = {
    "PAD": 0,
    "BOS": 1,
    "EOS": 2,
    "UNK": 3
}

for name, token_id in special_tokens.items():

    print(
        f"{name}: "
        f"train={np.sum(train_tokens == token_id)}, "
        f"validation={np.sum(val_tokens == token_id)}"
    )

# ------------------------------------------------------------
# 5. SEQUENCE LENGTH AUDIT
# ------------------------------------------------------------

print("\n5. SEQUENCE LENGTH AUDIT")

train_lengths = np.sum(train_tokens != 0, axis=1)
val_lengths = np.sum(val_tokens != 0, axis=1)

print("Training lengths:")
print(pd.Series(train_lengths).describe())

print("\nValidation lengths:")
print(pd.Series(val_lengths).describe())

# ------------------------------------------------------------
# 6. CHECKPOINT AVAILABILITY
# ------------------------------------------------------------

print("\n6. EXISTING MODEL CHECKPOINT")

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "molecular_gru_best.pt"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "Checkpoint size (MB):",
        round(os.path.getsize(checkpoint_path) / (1024 ** 2), 2)
    )

# ------------------------------------------------------------
# 7. FINAL VALIDATION
# ------------------------------------------------------------

checks = {
    "Training data has 2 dimensions": train_tokens.ndim == 2,
    "Validation data has 2 dimensions": val_tokens.ndim == 2,
    "Same sequence length": train_tokens.shape[1] == val_tokens.shape[1],
    "No negative token IDs": train_tokens.min() >= 0 and val_tokens.min() >= 0,
    "BOS present in training": np.all(train_tokens[:, 0] == 1),
    "BOS present in validation": np.all(val_tokens[:, 0] == 1),
    "EOS present in training": np.all(np.any(train_tokens == 2, axis=1)),
    "EOS present in validation": np.all(np.any(val_tokens == 2, axis=1))
}

print("\n7. FINAL VALIDATION")

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print(
    "\nOVERALL:",
    "PASS" if all(checks.values()) else "CHECK REQUIRED"
)

print("\nCELL 93 COMPLETED")

CELL 93: MOLECULAR TRANSFORMER INPUT AUDIT

1. TOKENIZER
Tokenizer file: /content/drive/MyDrive/Member2_Project/member2_outputs/smiles_tokenizer.json
Top-level structure: dict
Tokenizer keys: ['vocabulary', 'token_to_id', 'max_length', 'special_tokens']

2. TOKEN ARRAYS
Training shape: (45000, 143)
Validation shape: (5000, 143)
Training dtype: int32
Validation dtype: int32

3. TOKEN VALUE VALIDATION
Training minimum ID: 0
Training maximum ID: 80
Validation minimum ID: 0
Validation maximum ID: 80
Training unique tokens: 79
Validation unique tokens: 53

4. SPECIAL TOKEN COUNTS
PAD: train=4123646, validation=457723
BOS: train=45000, validation=5000
EOS: train=45000, validation=5000
UNK: train=0, validation=0

5. SEQUENCE LENGTH AUDIT
Training lengths:
count    45000.000000
mean        51.363422
std         16.303996
min          9.000000
25%         40.000000
50%         49.000000
75%         60.000000
max        143.000000
dtype: float64

Validation lengths:
count    5000.000000
mean    

In [147]:
# Inspect actual tokenizer structure

print("SPECIAL TOKENS:")
print(tokenizer_data["special_tokens"])

print("\nTOKEN TO ID SAMPLE:")
print(list(tokenizer_data["token_to_id"].items())[:15])

print("\nVOCABULARY SIZE:")
print(len(tokenizer_data["token_to_id"]))

SPECIAL TOKENS:
['<PAD>', '<BOS>', '<EOS>', '<UNK>']

TOKEN TO ID SAMPLE:
[('<PAD>', 0), ('<BOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('#', 4), ('%10', 5), ('(', 6), (')', 7), ('-', 8), ('/', 9), ('1', 10), ('2', 11), ('3', 12), ('4', 13), ('5', 14)]

VOCABULARY SIZE:
81


In [148]:
# ============================================================
# CELL 94: POCKET-CONDITIONED MOLECULAR TRANSFORMER
# ============================================================

import torch
import torch.nn as nn
import json
import os

print("=" * 65)
print("CELL 94: POCKET-CONDITIONED MOLECULAR TRANSFORMER")
print("=" * 65)

device = torch.device("cpu")

# ------------------------------------------------------------
# 1. LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    OUTPUT_DIR,
    "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]

VOCAB_SIZE = len(token_to_id)
MAX_LENGTH = int(tokenizer_data["max_length"])

# Correctly retrieve IDs from token_to_id
PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

print("\nTOKENIZER CONFIGURATION")
print("Vocabulary size:", VOCAB_SIZE)
print("Maximum length:", MAX_LENGTH)
print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)

# Validate vocabulary IDs
assert sorted(token_to_id.values()) == list(range(VOCAB_SIZE))

# ------------------------------------------------------------
# 2. MODEL CONFIGURATION
# ------------------------------------------------------------

EMBEDDING_DIM = 128
NUM_HEADS = 4
NUM_DECODER_LAYERS = 3
FEED_FORWARD_DIM = 512
DROPOUT = 0.1

assert EMBEDDING_DIM % NUM_HEADS == 0

# ------------------------------------------------------------
# 3. POSITIONAL ENCODING
# ------------------------------------------------------------

class PositionalEncoding(nn.Module):

    def __init__(self, embedding_dim, max_length):

        super().__init__()

        self.position_embedding = nn.Embedding(
            max_length,
            embedding_dim
        )

    def forward(self, x):

        seq_length = x.size(1)

        positions = torch.arange(
            seq_length,
            device=x.device
        ).unsqueeze(0)

        return x + self.position_embedding(positions)


# ------------------------------------------------------------
# 4. POCKET-CONDITIONED TRANSFORMER
# ------------------------------------------------------------

class PocketConditionedTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBEDDING_DIM,
            padding_idx=PAD_ID
        )

        self.position_encoding = PositionalEncoding(
            EMBEDDING_DIM,
            MAX_LENGTH
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=EMBEDDING_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=FEED_FORWARD_DIM,
            dropout=DROPOUT,
            batch_first=True,
            norm_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=NUM_DECODER_LAYERS
        )

        self.output_layer = nn.Linear(
            EMBEDDING_DIM,
            VOCAB_SIZE
        )

    def forward(
        self,
        token_ids,
        pocket_memory,
        token_padding_mask,
        pocket_padding_mask
    ):

        # Molecular token representation
        token_embeddings = self.token_embedding(token_ids)

        token_embeddings = self.position_encoding(
            token_embeddings
        )

        seq_length = token_ids.size(1)

        # Causal mask: prevents future-token access
        causal_mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                dtype=torch.bool,
                device=token_ids.device
            ),
            diagonal=1
        )

        # Decoder:
        # 1. Molecular self-attention
        # 2. Pocket cross-attention
        # 3. Feed-forward network

        decoder_output = self.decoder(
            tgt=token_embeddings,
            memory=pocket_memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=token_padding_mask,
            memory_key_padding_mask=pocket_padding_mask
        )

        return self.output_layer(decoder_output)


# ------------------------------------------------------------
# 5. INITIALIZE MODEL
# ------------------------------------------------------------

molecular_transformer = PocketConditionedTransformer().to(device)

parameter_count = sum(
    p.numel()
    for p in molecular_transformer.parameters()
    if p.requires_grad
)

print("\nMODEL CONFIGURATION")
print("Embedding dimension:", EMBEDDING_DIM)
print("Attention heads:", NUM_HEADS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Feed-forward dimension:", FEED_FORWARD_DIM)
print("Trainable parameters:", f"{parameter_count:,}")


# ------------------------------------------------------------
# 6. PREPARE POCKET MEMORY
# ------------------------------------------------------------

pocket_id = next(iter(pocket_embeddings_by_id))

pocket_data = pocket_embeddings_by_id[pocket_id]

pocket_embeddings = pocket_data["embeddings"]
pocket_mask = pocket_data["mask"]

# (5, 95, 128) -> (1, 475, 128)

pocket_memory = pocket_embeddings.reshape(
    1,
    -1,
    EMBEDDING_DIM
)

pocket_padding_mask = ~pocket_mask.reshape(1, -1)

print("\nPOCKET MEMORY")
print("Pocket ID:", pocket_id)
print("Pocket memory:", tuple(pocket_memory.shape))
print("Pocket mask:", tuple(pocket_padding_mask.shape))


# ------------------------------------------------------------
# 7. PREPARE MOLECULAR INPUT
# ------------------------------------------------------------

sample_tokens = torch.tensor(
    train_tokens[:1, :-1].astype("int64"),
    dtype=torch.long,
    device=device
)

token_padding_mask = sample_tokens == PAD_ID

print("\nMOLECULAR INPUT")
print("Input shape:", tuple(sample_tokens.shape))


# ------------------------------------------------------------
# 8. FORWARD PASS
# ------------------------------------------------------------

molecular_transformer.eval()

with torch.no_grad():

    output_logits = molecular_transformer(
        token_ids=sample_tokens,
        pocket_memory=pocket_memory,
        token_padding_mask=token_padding_mask,
        pocket_padding_mask=pocket_padding_mask
    )


# ------------------------------------------------------------
# 9. VALIDATION
# ------------------------------------------------------------

print("\nFORWARD PASS VALIDATION")

checks = {

    "Pocket memory shape correct":
        pocket_memory.shape == (1, 475, 128),

    "All five conformers preserved":
        pocket_memory.shape[1] == 5 * 95,

    "Molecular input batch correct":
        sample_tokens.shape[0] == 1,

    "Output sequence length correct":
        output_logits.shape[1] == sample_tokens.shape[1],

    "Output vocabulary dimension correct":
        output_logits.shape[2] == VOCAB_SIZE,

    "All output values finite":
        torch.isfinite(output_logits).all().item()
}

for name, passed in checks.items():

    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nOUTPUT SHAPE:", tuple(output_logits.shape))

print(
    "\nFINAL STATUS:",
    "PASS" if all(checks.values()) else "FAIL"
)

print("\nCELL 94 COMPLETED")

CELL 94: POCKET-CONDITIONED MOLECULAR TRANSFORMER

TOKENIZER CONFIGURATION
Vocabulary size: 81
Maximum length: 143
PAD: 0
BOS: 1
EOS: 2
UNK: 3

MODEL CONFIGURATION
Embedding dimension: 128
Attention heads: 4
Decoder layers: 3
Feed-forward dimension: 512
Trainable parameters: 832,849

POCKET MEMORY
Pocket ID: 1
Pocket memory: (1, 475, 128)
Pocket mask: (1, 475)

MOLECULAR INPUT
Input shape: (1, 142)

FORWARD PASS VALIDATION
Pocket memory shape correct: PASS
All five conformers preserved: PASS
Molecular input batch correct: PASS
Output sequence length correct: PASS
Output vocabulary dimension correct: PASS
All output values finite: PASS

OUTPUT SHAPE: (1, 142, 81)

FINAL STATUS: PASS

CELL 94 COMPLETED


In [149]:
# ============================================================
# CELL 95: POCKET-CONDITIONED TRAINING DATA READINESS AUDIT
# ============================================================

import os
import json
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project"

OUTPUT = os.path.join(ROOT, "member2_outputs")

print("=" * 65)
print("CELL 95: TRAINING DATA READINESS AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. LOCATE IMPORTANT DATASETS
# ------------------------------------------------------------

important_files = [
    "crossdocked_ligand_graphs_v2.jsonl",
    "chembl37_general_graphs.jsonl",
    "validated_target_ligand_graphs.jsonl",
    "member2_to_member3_interface.json",
    "final_validated_495_mapping.csv",
    "final_selected_500_mapping.csv"
]

found_files = {}

for filename in important_files:

    matches = []

    for folder in [OUTPUT, ROOT]:

        if not os.path.exists(folder):
            continue

        for current_root, dirs, files in os.walk(folder):

            if filename in files:
                matches.append(
                    os.path.join(current_root, filename)
                )

    if matches:
        found_files[filename] = matches[0]

# ------------------------------------------------------------
# 2. DISPLAY FILE DISCOVERY
# ------------------------------------------------------------

print("\n1. DATASET FILE DISCOVERY")

for filename in important_files:

    if filename in found_files:
        print("FOUND:", filename)
        print("PATH:", found_files[filename])
    else:
        print("NOT FOUND:", filename)

# ------------------------------------------------------------
# 3. INSPECT JSONL SCHEMAS
# ------------------------------------------------------------

print("\n2. JSONL DATASET SCHEMA AUDIT")

jsonl_files = [
    "crossdocked_ligand_graphs_v2.jsonl",
    "chembl37_general_graphs.jsonl",
    "validated_target_ligand_graphs.jsonl"
]

for filename in jsonl_files:

    if filename not in found_files:
        continue

    path = found_files[filename]

    print("\nDATASET:", filename)

    with open(path, "r") as f:

        first_record = json.loads(f.readline())

    print("Top-level keys:", list(first_record.keys()))

    for key, value in first_record.items():

        if isinstance(value, dict):
            print(key, "-> dictionary keys:", list(value.keys())[:15])

        elif isinstance(value, list):
            print(key, "-> list length:", len(value))

        else:
            print(key, "->", str(value)[:150])

# ------------------------------------------------------------
# 4. INSPECT CROSSDOCKED MAPPING
# ------------------------------------------------------------

print("\n3. CROSSDOCKED PAIR MAPPING AUDIT")

mapping_filename = next(
    (
        name for name in [
            "final_validated_495_mapping.csv",
            "final_selected_500_mapping.csv"
        ]
        if name in found_files
    ),
    None
)

if mapping_filename:

    mapping_df = pd.read_csv(
        found_files[mapping_filename],
        nrows=5
    )

    print("Mapping file:", mapping_filename)
    print("Columns:", mapping_df.columns.tolist())

    display(mapping_df.head())

else:
    print("No mapping CSV found.")

# ------------------------------------------------------------
# 5. INSPECT M3 INTERFACE
# ------------------------------------------------------------

print("\n4. M3 INTERFACE AUDIT")

interface_name = "member2_to_member3_interface.json"

if interface_name in found_files:

    with open(found_files[interface_name], "r") as f:
        interface_data = json.load(f)

    print("Interface keys:", list(interface_data.keys()))

    for key, value in interface_data.items():

        if isinstance(value, list):
            print(key, "count:", len(value))

        elif isinstance(value, dict):
            print(key, "keys:", list(value.keys())[:15])

        else:
            print(key, ":", str(value)[:150])

else:
    print("M3 interface file not found.")

# ------------------------------------------------------------
# 6. FINAL READINESS SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DATA READINESS SUMMARY")
print("=" * 65)

print("CrossDocked graph file:",
      "AVAILABLE" if "crossdocked_ligand_graphs_v2.jsonl" in found_files
      else "MISSING")

print("ChEMBL graph file:",
      "AVAILABLE" if "chembl37_general_graphs.jsonl" in found_files
      else "MISSING")

print("PARP1 reference graphs:",
      "AVAILABLE" if "validated_target_ligand_graphs.jsonl" in found_files
      else "MISSING")

print("\nCELL 95 COMPLETED")

CELL 95: TRAINING DATA READINESS AUDIT

1. DATASET FILE DISCOVERY
FOUND: crossdocked_ligand_graphs_v2.jsonl
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_ligand_graphs_v2.jsonl
FOUND: chembl37_general_graphs.jsonl
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_graphs.jsonl
FOUND: validated_target_ligand_graphs.jsonl
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/validated_target_ligand_graphs.jsonl
FOUND: member2_to_member3_interface.json
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_to_member3_interface.json
FOUND: final_validated_495_mapping.csv
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/final_validated_495_mapping.csv
NOT FOUND: final_selected_500_mapping.csv

2. JSONL DATASET SCHEMA AUDIT

DATASET: crossdocked_ligand_graphs_v2.jsonl
Top-level keys: ['schema_version', 'pair_idx', 'ligand_file', 'receptor_pdb', 'smiles', 'num_atoms', 'num_bonds', 'node_features', 'node_fea

,pair_idx,ligand_file,receptor_pdb,complex_dir,ligand_exists,receptor_exists,pdb_has_atoms
0,158402,0158402.sdf.gz,1mdo_A_rec.pdb,ARNB_SALTY_1_385_0,True,True,True
1,159900,0159900.sdf.gz,1gg1_B_rec.pdb,AROG_ECOLI_1_350_0,True,True,True
2,84592,0084592.sdf.gz,5unf_B_rec.pdb,AGTR2_HUMAN_35_335_0,True,True,True
3,154374,0154374.sdf.gz,2pb2_B_rec.pdb,ARGD_SALTY_1_405_0,True,True,True
4,159426,0159426.sdf.gz,1rf4_C_rec.pdb,AROA_STRPN_1_427_0,True,True,True



4. M3 INTERFACE AUDIT
Interface keys: ['interface_version', 'producer', 'consumer', 'architecture_role', 'pocket_conditioning', 'reference_ligand_graphs', 'reference_graph_count', 'graph_schema', 'generated_candidate_graphs', 'generation_status', 'generator_trained', 'training_manifest', 'scientific_note']
interface_version : 1.0
producer : MEMBER_2
consumer : MEMBER_3
architecture_role : Pocket-conditioned 2D molecular topology preparation
pocket_conditioning keys: ['pocket_id', 'num_conformers', 'num_feature_slots', 'num_channels', 'mask_shape', 'tensor_shape', 'feature_vocabulary', 'tokens_per_conformer', 'conditioning_source', 'm1_fingerprint']
reference_ligand_graphs count: 10
reference_graph_count : 10
graph_schema keys: ['node_features', 'bond_features', 'edge_representation', 'coordinates']
generated_candidate_graphs count: 0
generation_status : NOT_GENERATED
generator_trained : False
training_manifest : member2_training_manifest.json
scientific_note : Reference ligand graphs 

In [150]:
# ============================================================
# CELL 96: CROSSDOCKED PAIR CONSISTENCY AUDIT
# ============================================================

import os
import json
import pandas as pd
from collections import Counter

print("=" * 65)
print("CELL 96: CROSSDOCKED PAIR CONSISTENCY AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. LOAD DATASETS
# ------------------------------------------------------------

mapping_path = os.path.join(
    OUTPUT_DIR,
    "final_validated_495_mapping.csv"
)

graph_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_ligand_graphs_v2.jsonl"
)

mapping_df = pd.read_csv(mapping_path)

print("\n1. DATASET INFORMATION")
print("Mapping records:", len(mapping_df))

# ------------------------------------------------------------
# 2. LOAD GRAPH RECORDS
# ------------------------------------------------------------

graph_records = []

with open(graph_path, "r") as f:

    for line in f:

        if line.strip():
            graph_records.append(json.loads(line))

print("Graph records:", len(graph_records))

# ------------------------------------------------------------
# 3. CHECK UNIQUE PAIR IDS
# ------------------------------------------------------------

print("\n2. PAIR ID VALIDATION")

mapping_unique = mapping_df["pair_idx"].is_unique

graph_pair_ids = [
    record["pair_idx"]
    for record in graph_records
]

graph_unique = len(graph_pair_ids) == len(set(graph_pair_ids))

print("Mapping pair IDs unique:", mapping_unique)
print("Graph pair IDs unique:", graph_unique)

mapping_ids = set(mapping_df["pair_idx"])
graph_ids = set(graph_pair_ids)

print("Mapping-only IDs:", len(mapping_ids - graph_ids))
print("Graph-only IDs:", len(graph_ids - mapping_ids))

# ------------------------------------------------------------
# 4. MATCH GRAPH RECORDS WITH MAPPING
# ------------------------------------------------------------

mapping_lookup = mapping_df.set_index("pair_idx").to_dict("index")

audit_records = []

for record in graph_records:

    pair_id = record["pair_idx"]

    mapping = mapping_lookup.get(pair_id)

    if mapping is None:
        continue

    ligand_match = (
        str(record.get("ligand_file")) ==
        str(mapping["ligand_file"])
    )

    receptor_match = (
        str(record.get("receptor_pdb")) ==
        str(mapping["receptor_pdb"])
    )

    smiles_valid = bool(record.get("smiles"))

    atoms_valid = (
        record.get("num_atoms", 0) > 0
        and len(record.get("node_features", []))
        == record.get("num_atoms")
    )

    bonds_valid = (
        record.get("num_bonds", 0) >= 0
        and len(record.get("edge_index", []))
        == 2 * record.get("num_bonds", 0)
    )

    coordinates_valid = (
        len(record.get("heavy_atom_coordinates", []))
        == record.get("num_atoms", 0)
    )

    audit_records.append({

        "pair_idx": pair_id,
        "ligand_match": ligand_match,
        "receptor_match": receptor_match,
        "smiles_valid": smiles_valid,
        "atoms_valid": atoms_valid,
        "bonds_valid": bonds_valid,
        "coordinates_valid": coordinates_valid,
        "receptor_pdb": record.get("receptor_pdb"),
        "complex_dir": mapping.get("complex_dir")

    })

audit_df = pd.DataFrame(audit_records)

# ------------------------------------------------------------
# 5. VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n3. PAIR CONSISTENCY RESULTS")

checks = [
    "ligand_match",
    "receptor_match",
    "smiles_valid",
    "atoms_valid",
    "bonds_valid",
    "coordinates_valid"
]

for check in checks:

    print(
        check,
        ":",
        int(audit_df[check].sum()),
        "/",
        len(audit_df),
        "PASS"
    )

# ------------------------------------------------------------
# 6. RECEPTOR DISTRIBUTION
# ------------------------------------------------------------

print("\n4. RECEPTOR DISTRIBUTION")

receptor_counts = (
    audit_df["receptor_pdb"]
    .value_counts()
)

print("Unique receptor PDB files:", len(receptor_counts))

print("\nTop receptor files:")
display(receptor_counts.head(20).to_frame("record_count"))

# ------------------------------------------------------------
# 7. PROTEIN NAME SCREENING
# ------------------------------------------------------------

print("\n5. PARP1 NAME SCREENING")

parp_matches = audit_df[
    audit_df["receptor_pdb"].str.contains(
        "PARP1|PARP-1|ARTD1",
        case=False,
        na=False
    )
]

print("Potential PARP1 filename matches:", len(parp_matches))

if len(parp_matches):
    display(parp_matches)
else:
    print("No obvious PARP1 filename matches.")

# ------------------------------------------------------------
# 8. FINAL REPORT
# ------------------------------------------------------------

all_checks_pass = (
    mapping_unique
    and graph_unique
    and len(mapping_ids - graph_ids) == 0
    and len(graph_ids - mapping_ids) == 0
    and len(audit_df) == len(graph_records)
    and all(audit_df[check].all() for check in checks)
)

print("\n" + "=" * 65)
print("FINAL AUDIT")
print("=" * 65)

print("Matched records:", len(audit_df))
print("Overall consistency:", "PASS" if all_checks_pass else "CHECK REQUIRED")

report_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_pair_consistency_audit.csv"
)

audit_df.to_csv(report_path, index=False)

print("Report saved:", report_path)
print("\nCELL 96 COMPLETED")

CELL 96: CROSSDOCKED PAIR CONSISTENCY AUDIT

1. DATASET INFORMATION
Mapping records: 495
Graph records: 495

2. PAIR ID VALIDATION
Mapping pair IDs unique: True
Graph pair IDs unique: True
Mapping-only IDs: 0
Graph-only IDs: 0

3. PAIR CONSISTENCY RESULTS
ligand_match : 495 / 495 PASS
receptor_match : 495 / 495 PASS
smiles_valid : 495 / 495 PASS
atoms_valid : 495 / 495 PASS
bonds_valid : 495 / 495 PASS
coordinates_valid : 495 / 495 PASS

4. RECEPTOR DISTRIBUTION
Unique receptor PDB files: 490

Top receptor files:


,record_count
receptor_pdb,
3eym_D_rec.pdb,2
3ut1_A_rec.pdb,2
1n46_A_rec.pdb,2
2nxw_A_rec.pdb,2
1bke_A_rec.pdb,2
1lpo_A_rec.pdb,1
1nxk_C_rec.pdb,1
1pxd_A_rec.pdb,1
2pno_A_rec.pdb,1



5. PARP1 NAME SCREENING
Potential PARP1 filename matches: 0
No obvious PARP1 filename matches.

FINAL AUDIT
Matched records: 495
Overall consistency: PASS
Report saved: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_pair_consistency_audit.csv

CELL 96 COMPLETED


In [151]:
# ============================================================
# CELL 97: LOCATE CROSSDOCKED RECEPTOR FILES
# ============================================================

import os

print("=" * 65)
print("CELL 97: CROSSDOCKED FILE LOCATION AUDIT")
print("=" * 65)

search_roots = [
    "/content/drive/MyDrive/Member2_Project",
    "/content/drive/MyDrive"
]

print("\n1. PROJECT ROOT DIRECTORIES")

project_root = "/content/drive/MyDrive/Member2_Project"

for item in os.listdir(project_root):
    path = os.path.join(project_root, item)

    if os.path.isdir(path):
        print("[DIR]", item)

# ------------------------------------------------------------
# 2. CHECK COMMON DATASET DIRECTORIES
# ------------------------------------------------------------

print("\n2. POSSIBLE CROSSDOCKED DIRECTORIES")

for root in search_roots:

    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):

        dirs[:] = [
            d for d in dirs
            if d not in {
                ".ipynb_checkpoints",
                "__pycache__"
            }
        ]

        folder_name = os.path.basename(current).lower()

        if "crossdock" in folder_name:
            print(current)

        if current.count(os.sep) - root.count(os.sep) >= 5:
            dirs[:] = []

print("\nCELL 97 COMPLETED")

CELL 97: CROSSDOCKED FILE LOCATION AUDIT

1. PROJECT ROOT DIRECTORIES
[DIR] member1_input
[DIR] crossdocked_subset
[DIR] checkpoints
[DIR] final_selected_500
[DIR] member2_outputs
[DIR] chembl37_data

2. POSSIBLE CROSSDOCKED DIRECTORIES
/content/drive/MyDrive/Member2_Project/crossdocked_subset
/content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_mapping_audit
/content/drive/MyDrive/Member2_Project/crossdocked_subset
/content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_mapping_audit

CELL 97 COMPLETED


In [152]:
# ============================================================
# CELL 98: CROSSDOCKED RECEPTOR FILE INVENTORY
# ============================================================

import os
import pandas as pd

print("=" * 65)
print("CELL 98: CROSSDOCKED RECEPTOR FILE INVENTORY")
print("=" * 65)

DATA_DIR = "/content/drive/MyDrive/Member2_Project/crossdocked_subset"

mapping_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/final_validated_495_mapping.csv"
)

mapping_df = pd.read_csv(mapping_path)

# ------------------------------------------------------------
# 1. DIRECTORY STRUCTURE
# ------------------------------------------------------------

print("\n1. TOP-LEVEL DIRECTORY CONTENTS")

for item in os.listdir(DATA_DIR):

    path = os.path.join(DATA_DIR, item)

    if os.path.isdir(path):
        print("[DIR]", item)
    else:
        print("[FILE]", item)

# ------------------------------------------------------------
# 2. INVENTORY PDB AND SDF FILES
# ------------------------------------------------------------

print("\n2. STRUCTURE FILE INVENTORY")

pdb_files = []
sdf_files = []
other_structure_files = []

for current, dirs, files in os.walk(DATA_DIR):

    for filename in files:

        path = os.path.join(current, filename)
        lower = filename.lower()

        if lower.endswith((".pdb", ".pdb.gz")):
            pdb_files.append(path)

        elif lower.endswith((".sdf", ".sdf.gz")):
            sdf_files.append(path)

        elif lower.endswith((".cif", ".cif.gz")):
            other_structure_files.append(path)

print("PDB files:", len(pdb_files))
print("SDF files:", len(sdf_files))
print("CIF files:", len(other_structure_files))

# ------------------------------------------------------------
# 3. MATCH MAPPED RECEPTORS
# ------------------------------------------------------------

print("\n3. MAPPED RECEPTOR MATCHING")

pdb_lookup = {}

for path in pdb_files:

    filename = os.path.basename(path)

    pdb_lookup.setdefault(filename, []).append(path)

required_receptors = set(
    mapping_df["receptor_pdb"].dropna().astype(str)
)

found_receptors = required_receptors.intersection(pdb_lookup)

missing_receptors = required_receptors - found_receptors

print("Unique mapped receptors:", len(required_receptors))
print("Receptors located:", len(found_receptors))
print("Receptors missing:", len(missing_receptors))

if missing_receptors:
    print("\nMissing receptor examples:")
    print(list(missing_receptors)[:15])

# ------------------------------------------------------------
# 4. DISPLAY EXAMPLES
# ------------------------------------------------------------

print("\n4. RECEPTOR FILE EXAMPLES")

for filename in sorted(found_receptors)[:15]:

    print(filename, "->", pdb_lookup[filename][0])

print("\n5. LIGAND FILE EXAMPLES")

for path in sdf_files[:10]:
    print(path)

# ------------------------------------------------------------
# 6. SAVE INVENTORY
# ------------------------------------------------------------

inventory = pd.DataFrame({
    "receptor_pdb": sorted(required_receptors)
})

inventory["located"] = (
    inventory["receptor_pdb"].isin(found_receptors)
)

inventory["file_path"] = inventory["receptor_pdb"].map(
    lambda name: (
        pdb_lookup[name][0]
        if name in pdb_lookup else None
    )
)

save_path = os.path.join(
    "/content/drive/MyDrive/Member2_Project/member2_outputs",
    "crossdocked_receptor_inventory.csv"
)

inventory.to_csv(save_path, index=False)

print("\nInventory saved:", save_path)
print("\nCELL 98 COMPLETED")

CELL 98: CROSSDOCKED RECEPTOR FILE INVENTORY

1. TOP-LEVEL DIRECTORY CONTENTS
[DIR] ligands
[FILE] shard_13_extraction.json
[FILE] extraction_failures.json
[FILE] extraction_manifest.json

2. STRUCTURE FILE INVENTORY
PDB files: 0
SDF files: 500
CIF files: 0

3. MAPPED RECEPTOR MATCHING
Unique mapped receptors: 490
Receptors located: 0
Receptors missing: 490

Missing receptor examples:
['1jr7_A_rec.pdb', '1kmm_A_rec.pdb', '1ljr_A_rec.pdb', '1bw9_A_rec.pdb', '1cqx_A_rec.pdb', '1dhn_A_rec.pdb', '2dh4_A_rec.pdb', '3pds_A_rec.pdb', '2fr7_A_rec.pdb', '3odj_A_rec.pdb', '5u9a_A_rec.pdb', '5ern_A_rec.pdb', '1auw_A_rec.pdb', '2hor_A_rec.pdb', '1rf4_C_rec.pdb']

4. RECEPTOR FILE EXAMPLES

5. LIGAND FILE EXAMPLES
/content/drive/MyDrive/Member2_Project/crossdocked_subset/ligands/0474959.sdf.gz
/content/drive/MyDrive/Member2_Project/crossdocked_subset/ligands/0528368.sdf.gz
/content/drive/MyDrive/Member2_Project/crossdocked_subset/ligands/0528410.sdf.gz
/content/drive/MyDrive/Member2_Project/crossdo

In [153]:
# ============================================================
# CELL 99: CROSSDOCKED EXTRACTION METADATA AUDIT
# ============================================================

import os
import json

BASE_DIR = "/content/drive/MyDrive/Member2_Project/crossdocked_subset"

metadata_files = [
    "extraction_manifest.json",
    "extraction_failures.json",
    "shard_13_extraction.json"
]

for filename in metadata_files:

    path = os.path.join(BASE_DIR, filename)

    print("\n" + "=" * 65)
    print("FILE:", filename)
    print("=" * 65)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    with open(path, "r") as f:
        data = json.load(f)

    print("Python type:", type(data).__name__)

    if isinstance(data, dict):

        print("Top-level keys:", list(data.keys()))

        for key, value in data.items():

            print(f"\n{key}:")

            if isinstance(value, (dict, list)):
                print("Type:", type(value).__name__)
                print("Length:", len(value))
                print("Preview:", str(value)[:1500])
            else:
                print(value)

    elif isinstance(data, list):

        print("Number of records:", len(data))
        print("First records:", data[:5])

    else:
        print(str(data)[:3000])

print("\nCELL 99 COMPLETED")


FILE: extraction_manifest.json
Python type: list
Number of records: 107
First records: [{'pair_idx': 996704, 'ligand_file': '0996704.sdf.gz', 'receptor_pdb': '4eqh_B_rec.pdb', 'complex_dir': 'HINT1_HUMAN_2_126_0', 'metadata': {'pair_idx': 996704, 'receptor_path': 'HINT1_HUMAN_2_126_0/4eqh_B_rec.pdb', 'complex_dir': 'HINT1_HUMAN_2_126_0', 'ligand_original_name': '4eqh_B_rec_5ipe_6cg_lig_tt_min.sdf.gz', 'source_type': 'cdonly'}}, {'pair_idx': 979695, 'ligand_file': '0979695.sdf.gz', 'receptor_pdb': '4uwt_A_rec.pdb', 'complex_dir': 'GUX1_HYPJE_18_451_0', 'metadata': {'pair_idx': 979695, 'receptor_path': 'GUX1_HYPJE_18_451_0/4uwt_A_rec.pdb', 'complex_dir': 'GUX1_HYPJE_18_451_0', 'ligand_original_name': '4uwt_A_rec_1cel_ibz_lig_tt_min.sdf.gz', 'source_type': 'cdonly'}}, {'pair_idx': 993139, 'ligand_file': '0993139.sdf.gz', 'receptor_pdb': '1qk3_C_rec.pdb', 'complex_dir': 'HGXR_TOXGO_2_230_0', 'metadata': {'pair_idx': 993139, 'receptor_path': 'HGXR_TOXGO_2_230_0/1qk3_C_rec.pdb', 'complex_di

In [155]:
# ============================================================
# CELL 100: EXTRACTION MANIFEST RECONCILIATION (FIXED)
# ============================================================

import os
import json
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
DATA_DIR = os.path.join(BASE_DIR, "crossdocked_subset")
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

# ------------------------------------------------------------
# 1. LOAD FILES
# ------------------------------------------------------------

mapping = pd.read_csv(
    os.path.join(OUTPUT_DIR, "final_validated_495_mapping.csv")
)

with open(
    os.path.join(DATA_DIR, "extraction_manifest.json")
) as f:
    manifest = json.load(f)

manifest_df = pd.DataFrame(manifest)

print("=" * 65)
print("CELL 100: EXTRACTION MANIFEST RECONCILIATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. RECORD COUNTS
# ------------------------------------------------------------

print("\n1. RECORD COUNTS")

print("Selected mapping records:", len(mapping))
print("Manifest records:", len(manifest_df))
print("Unique manifest pair IDs:", manifest_df["pair_idx"].nunique())

# ------------------------------------------------------------
# 3. NORMALIZE PAIR IDS
# ------------------------------------------------------------

mapping["pair_idx"] = pd.to_numeric(
    mapping["pair_idx"], errors="coerce"
)

manifest_df["pair_idx"] = pd.to_numeric(
    manifest_df["pair_idx"], errors="coerce"
)

mapping_ids = set(mapping["pair_idx"].dropna())
manifest_ids = set(manifest_df["pair_idx"].dropna())

print("\n2. PAIR ID RECONCILIATION")

print("Mapping IDs:", len(mapping_ids))
print("Manifest IDs:", len(manifest_ids))
print("Common IDs:", len(mapping_ids & manifest_ids))
print("Mapping-only IDs:", len(mapping_ids - manifest_ids))
print("Manifest-only IDs:", len(manifest_ids - mapping_ids))

# ------------------------------------------------------------
# 4. SAFE METADATA HANDLING
# ------------------------------------------------------------

def get_metadata(value):

    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, dict) else {}
        except:
            return {}

    return {}


manifest_df["metadata_normalized"] = manifest_df[
    "metadata"
].apply(get_metadata)


def get_receptor_path(row):

    metadata = row["metadata_normalized"]

    relative_path = metadata.get("receptor_path")

    if not relative_path:
        return None

    return os.path.join(DATA_DIR, relative_path)


manifest_df["receptor_path_checked"] = manifest_df.apply(
    get_receptor_path,
    axis=1
)

manifest_df["receptor_exists"] = manifest_df[
    "receptor_path_checked"
].apply(
    lambda path: (
        os.path.isfile(path)
        if isinstance(path, str)
        else False
    )
)

print("\n3. RECEPTOR PATH CHECK")

print(
    "Receptor paths available:",
    manifest_df["receptor_exists"].sum()
)

print(
    "Receptor paths missing:",
    (~manifest_df["receptor_exists"]).sum()
)

# ------------------------------------------------------------
# 5. COMPLEX DIRECTORY CHECK
# ------------------------------------------------------------

manifest_df["complex_dir_exists"] = manifest_df[
    "complex_dir"
].apply(
    lambda name: (
        os.path.isdir(os.path.join(DATA_DIR, str(name)))
        if pd.notna(name) else False
    )
)

print("\n4. COMPLEX DIRECTORY CHECK")

print(
    "Existing complex directories:",
    manifest_df["complex_dir_exists"].sum()
)

# ------------------------------------------------------------
# 6. SOURCE TYPE
# ------------------------------------------------------------

manifest_df["source_type"] = manifest_df[
    "metadata_normalized"
].apply(
    lambda metadata: metadata.get("source_type", "UNKNOWN")
)

print("\n5. SOURCE TYPE DISTRIBUTION")

print(manifest_df["source_type"].value_counts())

# ------------------------------------------------------------
# 7. CHECK MISSING METADATA
# ------------------------------------------------------------

print("\n6. METADATA QUALITY")

print(
    "Missing/invalid metadata records:",
    (manifest_df["metadata_normalized"].apply(
        lambda x: len(x) == 0
    )).sum()
)

# ------------------------------------------------------------
# 8. SAVE AUDIT
# ------------------------------------------------------------

audit_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_extraction_reconciliation.csv"
)

manifest_df.to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path)

print("\nCELL 100 COMPLETED")

CELL 100: EXTRACTION MANIFEST RECONCILIATION

1. RECORD COUNTS
Selected mapping records: 495
Manifest records: 107
Unique manifest pair IDs: 107

2. PAIR ID RECONCILIATION
Mapping IDs: 495
Manifest IDs: 107
Common IDs: 13
Mapping-only IDs: 482
Manifest-only IDs: 94

3. RECEPTOR PATH CHECK
Receptor paths available: 0
Receptor paths missing: 107

4. COMPLEX DIRECTORY CHECK
Existing complex directories: 0

5. SOURCE TYPE DISTRIBUTION
source_type
UNKNOWN    85
cdonly     22
Name: count, dtype: int64

6. METADATA QUALITY
Missing/invalid metadata records: 85

Audit saved: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_extraction_reconciliation.csv

CELL 100 COMPLETED


In [156]:
# ============================================================
# CELL 101: LOCAL LIGAND FILE RECONCILIATION
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
DATA_DIR = os.path.join(BASE_DIR, "crossdocked_subset")
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

mapping = pd.read_csv(
    os.path.join(OUTPUT_DIR, "final_validated_495_mapping.csv")
)

LIGAND_DIR = os.path.join(DATA_DIR, "ligands")

# ------------------------------------------------------------
# 1. LOCAL LIGAND INVENTORY
# ------------------------------------------------------------

local_files = set(
    f for f in os.listdir(LIGAND_DIR)
    if f.endswith(".sdf.gz")
)

print("=" * 65)
print("CELL 101: LOCAL LIGAND RECONCILIATION")
print("=" * 65)

print("\n1. LOCAL FILE COUNTS")
print("Local ligand files:", len(local_files))
print("Mapping records:", len(mapping))

# ------------------------------------------------------------
# 2. MATCH MAPPING FILENAMES
# ------------------------------------------------------------

mapping["local_ligand_exists"] = mapping[
    "ligand_file"
].isin(local_files)

print("\n2. MAPPING-TO-FILE MATCH")

print(
    "Mapped ligand files found:",
    mapping["local_ligand_exists"].sum()
)

print(
    "Mapped ligand files missing:",
    (~mapping["local_ligand_exists"]).sum()
)

# ------------------------------------------------------------
# 3. EXTRA FILES
# ------------------------------------------------------------

mapped_files = set(mapping["ligand_file"].dropna())

extra_files = local_files - mapped_files

print("\n3. EXTRA LOCAL FILES")
print("Files not in mapping:", len(extra_files))
print("Examples:", list(extra_files)[:10])

# ------------------------------------------------------------
# 4. PAIR ID FROM LIGAND FILENAME
# ------------------------------------------------------------

mapping["pair_id_from_filename"] = pd.to_numeric(
    mapping["ligand_file"].str.replace(
        ".sdf.gz", "", regex=False
    ),
    errors="coerce"
)

mapping["filename_pair_id_matches"] = (
    mapping["pair_idx"] == mapping["pair_id_from_filename"]
)

print("\n4. PAIR ID CHECK")

print(
    "Filename pair IDs matching mapping:",
    mapping["filename_pair_id_matches"].sum()
)

print(
    "Filename pair IDs not matching:",
    (~mapping["filename_pair_id_matches"]).sum()
)

# ------------------------------------------------------------
# 5. SAVE REPORT
# ------------------------------------------------------------

report_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_local_ligand_reconciliation.csv"
)

mapping.to_csv(report_path, index=False)

print("\nReport saved:", report_path)
print("\nCELL 101 COMPLETED")

CELL 101: LOCAL LIGAND RECONCILIATION

1. LOCAL FILE COUNTS
Local ligand files: 500
Mapping records: 495

2. MAPPING-TO-FILE MATCH
Mapped ligand files found: 39
Mapped ligand files missing: 456

3. EXTRA LOCAL FILES
Files not in mapping: 461
Examples: ['0890236.sdf.gz', '1148943.sdf.gz', '1302526.sdf.gz', '0775332.sdf.gz', '1477654.sdf.gz', '2274203.sdf.gz', '1838134.sdf.gz', '1464745.sdf.gz', '0093950.sdf.gz', '2148111.sdf.gz']

4. PAIR ID CHECK
Filename pair IDs matching mapping: 495
Filename pair IDs not matching: 0

Report saved: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_local_ligand_reconciliation.csv

CELL 101 COMPLETED


In [157]:
# ============================================================
# CELL 102: CROSSDOCKED DATASET PROVENANCE AUDIT
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
DATA_DIR = os.path.join(BASE_DIR, "crossdocked_subset")
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

mapping = pd.read_csv(
    os.path.join(OUTPUT_DIR, "final_validated_495_mapping.csv")
)

graph_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_ligand_graphs_v2.jsonl"
)

LIGAND_DIR = os.path.join(DATA_DIR, "ligands")

local_files = set(os.listdir(LIGAND_DIR))

# ------------------------------------------------------------
# 1. DISPLAY MAPPING RECORDS
# ------------------------------------------------------------

print("=" * 65)
print("CELL 102: DATASET PROVENANCE AUDIT")
print("=" * 65)

print("\n1. FIRST 15 MAPPING RECORDS")

print(
    mapping[
        ["pair_idx", "ligand_file", "receptor_pdb", "complex_dir"]
    ].head(15).to_string(index=False)
)

# ------------------------------------------------------------
# 2. DISPLAY LOCAL FILES
# ------------------------------------------------------------

print("\n2. FIRST 15 LOCAL LIGAND FILES")

for filename in sorted(local_files)[:15]:
    print(filename)

# ------------------------------------------------------------
# 3. OVERLAPPING RECORDS
# ------------------------------------------------------------

mapping["exists_locally"] = mapping[
    "ligand_file"
].isin(local_files)

overlap = mapping[mapping["exists_locally"]]

print("\n3. OVERLAPPING RECORDS")

print("Overlap count:", len(overlap))

print(
    overlap[
        ["pair_idx", "ligand_file", "receptor_pdb"]
    ].head(20).to_string(index=False)
)

# ------------------------------------------------------------
# 4. CHECK LOCAL FILE PAIR IDS
# ------------------------------------------------------------

local_pair_ids = {
    int(filename.replace(".sdf.gz", ""))
    for filename in local_files
    if filename.replace(".sdf.gz", "").isdigit()
}

mapping_pair_ids = set(mapping["pair_idx"].astype(int))

print("\n4. PAIR ID OVERLAP")

print("Local pair IDs:", len(local_pair_ids))
print("Mapping pair IDs:", len(mapping_pair_ids))
print("Common pair IDs:", len(local_pair_ids & mapping_pair_ids))

# ------------------------------------------------------------
# 5. GRAPH FILE SAMPLE
# ------------------------------------------------------------

print("\n5. GRAPH FILE SAMPLE")

with open(graph_path, "r") as f:
    for i in range(3):
        line = f.readline()

        if not line:
            break

        import json
        record = json.loads(line)

        print({
            "pair_idx": record.get("pair_idx"),
            "ligand_file": record.get("ligand_file"),
            "receptor_pdb": record.get("receptor_pdb"),
            "smiles": record.get("smiles")
        })

print("\nCELL 102 COMPLETED")

CELL 102: DATASET PROVENANCE AUDIT

1. FIRST 15 MAPPING RECORDS
 pair_idx    ligand_file   receptor_pdb                complex_dir
   158402 0158402.sdf.gz 1mdo_A_rec.pdb         ARNB_SALTY_1_385_0
   159900 0159900.sdf.gz 1gg1_B_rec.pdb         AROG_ECOLI_1_350_0
    84592 0084592.sdf.gz 5unf_B_rec.pdb       AGTR2_HUMAN_35_335_0
   154374 0154374.sdf.gz 2pb2_B_rec.pdb         ARGD_SALTY_1_405_0
   159426 0159426.sdf.gz 1rf4_C_rec.pdb         AROA_STRPN_1_427_0
    84000 0084000.sdf.gz 4fnp_D_rec.pdb         AGAA_GEOSE_1_729_0
   158426 0158426.sdf.gz 1nrx_B_rec.pdb         ARO1_EMENI_1_393_0
   154052 0154052.sdf.gz 3zzg_A_rec.pdb       ARG56_YEAST_58_356_0
   157557 0157557.sdf.gz 1pq3_C_rec.pdb       ARGI2_HUMAN_24_329_0
    83512 0083512.sdf.gz 3pds_A_rec.pdb ADRB2_HUMAN_26_357_EH234_0
   157711 0157711.sdf.gz 3bue_D_rec.pdb        ARGR_MYCTU_92_170_0
   152504 0152504.sdf.gz 1zn7_A_rec.pdb          APT_HUMAN_1_180_0
   145324 0145324.sdf.gz 1b2y_A_rec.pdb        AMYP_HUMAN_16_511_

In [158]:
# ============================================================
# CELL 103: HUGGING FACE CACHE INSPECTION
# ============================================================

import os
from pathlib import Path

print("=" * 70)
print("CELL 103: HUGGING FACE CACHE INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CACHE LOCATIONS
# ------------------------------------------------------------

cache_locations = [
    Path.home() / ".cache" / "huggingface",
    Path("/root/.cache/huggingface"),
    Path("/content/.cache/huggingface"),
    Path("/content/drive/MyDrive/.cache/huggingface")
]

for variable in ["HF_HOME", "HF_HUB_CACHE", "HF_DATASETS_CACHE"]:
    value = os.environ.get(variable)
    if value:
        cache_locations.append(Path(value))

cache_locations = list(dict.fromkeys(cache_locations))

print("\n1. CACHE DIRECTORIES")

existing_locations = []

for path in cache_locations:
    if path.exists():
        existing_locations.append(path)
        print("[FOUND]", path)
    else:
        print("[NOT FOUND]", path)

# ------------------------------------------------------------
# 2. FIND CROSSDOCKED DATASET DIRECTORIES
# ------------------------------------------------------------

print("\n2. CROSSDOCKED DATASET DIRECTORIES")

dataset_dirs = []

for root in existing_locations:

    for current, dirs, files in os.walk(root):

        current_path = Path(current)

        if "crossdocked" in str(current_path).lower():
            dataset_dirs.append(current_path)
            print(current_path)

        # Avoid unnecessarily deep traversal
        if len(current_path.relative_to(root).parts) >= 6:
            dirs[:] = []

dataset_dirs = list(dict.fromkeys(dataset_dirs))

# ------------------------------------------------------------
# 3. INSPECT HUB SNAPSHOTS
# ------------------------------------------------------------

print("\n3. HUGGING FACE HUB SNAPSHOTS")

for root in existing_locations:

    hub_dir = root / "hub"

    if not hub_dir.exists():
        continue

    for repo in hub_dir.iterdir():

        if "crossdocked" in repo.name.lower():

            print("\nRepository:", repo)

            snapshots = repo / "snapshots"

            if snapshots.exists():

                for snapshot in snapshots.iterdir():
                    print("Snapshot:", snapshot)

                    for item in list(snapshot.iterdir())[:30]:
                        print("  ", item.name)

# ------------------------------------------------------------
# 4. SEARCH FOR STRUCTURE AND DATA FILES
# ------------------------------------------------------------

print("\n4. CACHED STRUCTURE/DATA FILES")

extensions = (
    ".pdb", ".pdb.gz",
    ".cif", ".cif.gz",
    ".parquet",
    ".tar", ".tar.gz",
    ".zip"
)

found_files = []
total_size = 0

for root in existing_locations:

    for current, dirs, files in os.walk(root):

        for filename in files:

            lower = filename.lower()

            if lower.endswith(extensions):

                path = Path(current) / filename

                try:
                    size = path.stat().st_size
                except OSError:
                    size = 0

                found_files.append((path, size))
                total_size += size

        if len(Path(current).relative_to(root).parts) >= 7:
            dirs[:] = []

print("Matching files:", len(found_files))
print("Total size (GB):", round(total_size / (1024**3), 3))

for path, size in found_files[:40]:
    print(path, "|", round(size / (1024**2), 2), "MB")

# ------------------------------------------------------------
# 5. FINAL SUMMARY
# ------------------------------------------------------------

print("\n5. FINAL SUMMARY")

print("Existing cache roots:", len(existing_locations))
print("CrossDocked directories:", len(dataset_dirs))
print("Structure/data files found:", len(found_files))

if len(found_files) == 0:
    print("No recognizable structure/data files found in inspected cache.")

print("\nCELL 103 COMPLETED")

CELL 103: HUGGING FACE CACHE INSPECTION

1. CACHE DIRECTORIES
[NOT FOUND] /root/.cache/huggingface
[NOT FOUND] /content/.cache/huggingface
[NOT FOUND] /content/drive/MyDrive/.cache/huggingface

2. CROSSDOCKED DATASET DIRECTORIES

3. HUGGING FACE HUB SNAPSHOTS

4. CACHED STRUCTURE/DATA FILES
Matching files: 0
Total size (GB): 0.0

5. FINAL SUMMARY
Existing cache roots: 0
CrossDocked directories: 0
Structure/data files found: 0
No recognizable structure/data files found in inspected cache.

CELL 103 COMPLETED


In [159]:
# ============================================================
# CELL 104: ORIGINAL CROSSDOCKED REPOSITORY INSPECTION
# ============================================================

import importlib.util
import sys

print("=" * 70)
print("CELL 104: ORIGINAL CROSSDOCKED REPOSITORY INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK REQUIRED PACKAGE
# ------------------------------------------------------------

if importlib.util.find_spec("huggingface_hub") is None:
    %pip install -q huggingface_hub

from huggingface_hub import HfApi

# ------------------------------------------------------------
# 2. CONNECT TO REPOSITORY
# ------------------------------------------------------------

REPO_ID = "kohbanye/crossdocked2020"

api = HfApi()

print("\n1. REPOSITORY CONNECTION")

try:
    repo_info = api.dataset_info(REPO_ID)

    print("Repository:", repo_info.id)
    print("Private:", repo_info.private)
    print("Gated:", repo_info.gated)
    print("Connection: SUCCESS")

except Exception as e:
    print("Repository connection failed:")
    print(type(e).__name__, str(e))
    raise

# ------------------------------------------------------------
# 3. LIST REPOSITORY FILES
# ------------------------------------------------------------

print("\n2. RETRIEVING FILE INVENTORY")

try:
    repo_files = list(
        api.list_repo_tree(
            repo_id=REPO_ID,
            repo_type="dataset",
            recursive=True
        )
    )

    print("Total repository entries:", len(repo_files))

except Exception as e:
    print("File listing failed:")
    print(type(e).__name__, str(e))
    raise

# ------------------------------------------------------------
# 4. CLASSIFY FILES
# ------------------------------------------------------------

file_records = []

for item in repo_files:

    path = getattr(item, "path", "")
    size = getattr(item, "size", None)

    if not path:
        continue

    file_records.append({
        "path": path,
        "size_bytes": size,
        "extension": path.lower().split(".")[-1]
    })

import pandas as pd

files_df = pd.DataFrame(file_records)

print("\n3. FILE TYPE DISTRIBUTION")

print(
    files_df["extension"].value_counts().head(25)
)

# ------------------------------------------------------------
# 5. FIND RECEPTOR STRUCTURES AND ARCHIVES
# ------------------------------------------------------------

print("\n4. STRUCTURE AND ARCHIVE FILES")

keywords = (
    ".pdb", ".cif", ".ent",
    ".tar", ".gz", ".zip",
    ".parquet"
)

relevant = files_df[
    files_df["path"].str.lower().str.endswith(keywords)
]

print("Relevant files:", len(relevant))

for _, row in relevant.head(100).iterrows():

    size_mb = (
        round(row["size_bytes"] / (1024**2), 2)
        if pd.notna(row["size_bytes"])
        else "Unknown"
    )

    print(row["path"], "|", size_mb, "MB")

# ------------------------------------------------------------
# 6. RECEPTOR-SPECIFIC SEARCH
# ------------------------------------------------------------

print("\n5. RECEPTOR FILE SEARCH")

receptor_matches = files_df[
    files_df["path"].str.lower().str.contains(
        r"receptor|_rec\.pdb|\.pdb\.gz|receptor.*tar",
        regex=True,
        na=False
    )
]

print("Potential receptor files:", len(receptor_matches))

for path in receptor_matches["path"].head(50):
    print(path)

# ------------------------------------------------------------
# 7. SAVE INVENTORY
# ------------------------------------------------------------

save_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/huggingface_crossdocked_repository_inventory.csv"
)

files_df.to_csv(save_path, index=False)

print("\nInventory saved:", save_path)
print("\nCELL 104 COMPLETED")

CELL 104: ORIGINAL CROSSDOCKED REPOSITORY INSPECTION

1. REPOSITORY CONNECTION
Repository: kohbanye/crossdocked2020
Private: False
Gated: False
Connection: SUCCESS

2. RETRIEVING FILE INVENTORY
Total repository entries: 41

3. FILE TYPE DISTRIBUTION
extension
tar              35
receptors         1
ligands           1
gitattributes     1
md                1
parquet           1
gz                1
Name: count, dtype: int64

4. STRUCTURE AND ARCHIVE FILES
Relevant files: 37
ligands/000000.tar | 615.7 MB
ligands/000001.tar | 626.15 MB
ligands/000002.tar | 603.29 MB
ligands/000003.tar | 620.02 MB
ligands/000004.tar | 617.24 MB
ligands/000005.tar | 609.65 MB
ligands/000006.tar | 616.13 MB
ligands/000007.tar | 623.62 MB
ligands/000008.tar | 592.9 MB
ligands/000009.tar | 600.83 MB
ligands/000010.tar | 618.13 MB
ligands/000011.tar | 602.41 MB
ligands/000012.tar | 611.76 MB
ligands/000013.tar | 605.67 MB
ligands/000014.tar | 613.21 MB
ligands/000015.tar | 595.74 MB
ligands/000016.tar | 618.99 M

In [160]:
# ============================================================
# CELL 105: ORIGINAL MANIFEST PAIR MATCHING
# ============================================================

import os
import pandas as pd
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

print("=" * 70)
print("CELL 105: ORIGINAL CROSSDOCKED MANIFEST AUDIT")
print("=" * 70)

REPO_ID = "kohbanye/crossdocked2020"

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

# ------------------------------------------------------------
# 1. DOWNLOAD ONLY THE SMALL MANIFEST
# ------------------------------------------------------------

print("\n1. ACCESSING ORIGINAL MANIFEST")

manifest_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="manifest.parquet",
    repo_type="dataset"
)

print("Manifest path:", manifest_path)
print(
    "Manifest size (MB):",
    round(os.path.getsize(manifest_path) / (1024**2), 2)
)

# ------------------------------------------------------------
# 2. INSPECT SCHEMA
# ------------------------------------------------------------

print("\n2. MANIFEST SCHEMA")

parquet_file = pq.ParquetFile(manifest_path)

print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.schema.names)

# ------------------------------------------------------------
# 3. LOAD OUR 495 MAPPING RECORDS
# ------------------------------------------------------------

mapping_path = os.path.join(
    OUTPUT_DIR,
    "final_validated_495_mapping.csv"
)

mapping = pd.read_csv(mapping_path)

# ------------------------------------------------------------
# 4. LOAD ORIGINAL MANIFEST
# ------------------------------------------------------------

print("\n3. LOADING ORIGINAL MANIFEST")

original = pd.read_parquet(manifest_path)

print("Original records:", len(original))
print("Original columns:", list(original.columns))

# ------------------------------------------------------------
# 5. IDENTIFY PAIR ID COLUMN
# ------------------------------------------------------------

possible_id_columns = [
    col for col in original.columns
    if col.lower() in ["pair_idx", "pair_id", "index"]
]

print("\n4. POSSIBLE PAIR ID COLUMNS")
print(possible_id_columns)

if "pair_idx" not in original.columns:
    raise ValueError(
        "pair_idx not found. Inspect schema before proceeding."
    )

# ------------------------------------------------------------
# 6. MATCH OUR 495 PAIRS
# ------------------------------------------------------------

mapping["pair_idx"] = pd.to_numeric(
    mapping["pair_idx"], errors="coerce"
)

original["pair_idx"] = pd.to_numeric(
    original["pair_idx"], errors="coerce"
)

matched = original[
    original["pair_idx"].isin(mapping["pair_idx"])
].copy()

print("\n5. PAIR MATCHING")

print("Our selected pairs:", len(mapping))
print("Original manifest matches:", len(matched))
print(
    "Missing from original manifest:",
    len(mapping) - matched["pair_idx"].nunique()
)

# ------------------------------------------------------------
# 7. DISPLAY MATCHED RECORDS
# ------------------------------------------------------------

print("\n6. MATCHED RECORD PREVIEW")

print(matched.head(10).to_string(index=False))

# ------------------------------------------------------------
# 8. DISPLAY RECEPTOR-RELATED COLUMNS
# ------------------------------------------------------------

print("\n7. RECEPTOR-RELATED COLUMNS")

receptor_columns = [
    col for col in matched.columns
    if any(
        term in col.lower()
        for term in [
            "receptor", "protein", "pdb",
            "path", "shard", "complex"
        ]
    )
]

print(receptor_columns)

if receptor_columns:
    print(
        matched[["pair_idx"] + receptor_columns]
        .head(15)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 9. SAVE MATCHED ORIGINAL RECORDS
# ------------------------------------------------------------

save_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_original_manifest_495_matches.csv"
)

matched.to_csv(save_path, index=False)

print("\nMatched records saved:", save_path)
print("\nCELL 105 COMPLETED")

CELL 105: ORIGINAL CROSSDOCKED MANIFEST AUDIT

1. ACCESSING ORIGINAL MANIFEST


manifest.parquet: reconstructing file:   0%|          |  0.00B / 25.6MB            

manifest.parquet: downloading bytes:           |  0.00B            

Manifest path: /root/.cache/huggingface/hub/datasets--kohbanye--crossdocked2020/snapshots/017d8a19301c901471bba9f139331f86cd9ad278/manifest.parquet
Manifest size (MB): 24.4

2. MANIFEST SCHEMA
Rows: 2533288
Columns: ['pair_idx', 'complex_dir', 'receptor_pdb', 'ligand_sdf_gz', 'source_type', 'shard_idx', 'label', 'score1', 'score2', 'cdonly_fold0', 'cdonly_fold1', 'cdonly_fold2', 'it0_fold0', 'it0_fold1', 'it0_fold2', 'it2_fold0', 'it2_fold1', 'it2_fold2', 'it2_redocked_fold0', 'it2_redocked_fold1', 'it2_redocked_fold2', 'it2_10p20n_fold0', 'it2_10p20n_fold1', 'it2_10p20n_fold2', 'mod_it2_fold0', 'mod_it2_fold1', 'mod_it2_fold2', 'it2_completeset_fold0', 'it2_completeset_fold1', 'it2_completeset_fold2', 'it2_redocked_completeset_fold0', 'it2_redocked_completeset_fold1', 'it2_redocked_completeset_fold2', 'mod_it2_completeset_fold0', 'mod_it2_completeset_fold1', 'mod_it2_completeset_fold2']

3. LOADING ORIGINAL MANIFEST
Original records: 2533288
Original columns: ['pair_idx', 'complex_dir

In [161]:
# ============================================================
# CELL 106: CROSSDOCKED TARGET RELEVANCE AUDIT
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

print("=" * 70)
print("CELL 106: CROSSDOCKED TARGET RELEVANCE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD ORIGINAL MANIFEST AND SELECTED RECORDS
# ------------------------------------------------------------

manifest_path = (
    "/root/.cache/huggingface/hub/"
    "datasets--kohbanye--crossdocked2020/"
    "snapshots/017d8a19301c901471bba9f139331f86cd9ad278/"
    "manifest.parquet"
)

if not os.path.exists(manifest_path):
    raise FileNotFoundError(
        "Original manifest not found in Hugging Face cache."
    )

original = pd.read_parquet(manifest_path)

selected = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "crossdocked_original_manifest_495_matches.csv"
    )
)

print("\n1. DATASET COUNTS")

print("Complete manifest:", len(original))
print("Selected records:", len(selected))

# ------------------------------------------------------------
# 2. SELECTED PROTEIN DISTRIBUTION
# ------------------------------------------------------------

print("\n2. SELECTED SOURCE TYPE DISTRIBUTION")

print(selected["source_type"].value_counts(dropna=False))

print("\n3. SELECTED SHARD DISTRIBUTION")

print(selected["shard_idx"].value_counts(dropna=False))

print("\n4. UNIQUE PROTEINS")

print(
    "Unique complex directories:",
    selected["complex_dir"].nunique()
)

print(
    "Unique receptor files:",
    selected["receptor_pdb"].nunique()
)

# ------------------------------------------------------------
# 3. SEARCH SELECTED RECORDS FOR TARGET-RELATED NAMES
# ------------------------------------------------------------

target_pattern = (
    r"PARP1|PARP-1|ARTD1|POLY.?ADP.?RIBOSE"
)

selected["target_name_match"] = (
    selected["complex_dir"]
    .fillna("")
    .str.contains(target_pattern, case=False, regex=True)
)

selected_target = selected[
    selected["target_name_match"]
].copy()

print("\n5. SELECTED PARP1 NAME MATCHES")

print("Potential matches:", len(selected_target))

if len(selected_target):
    print(
        selected_target[
            [
                "pair_idx",
                "complex_dir",
                "receptor_pdb",
                "ligand_sdf_gz",
                "source_type",
                "shard_idx"
            ]
        ].to_string(index=False)
    )
else:
    print("No obvious PARP1 names in selected records.")

# ------------------------------------------------------------
# 4. SEARCH COMPLETE MANIFEST FOR PARP1
# ------------------------------------------------------------

print("\n6. COMPLETE DATASET PARP1 SEARCH")

original["target_name_match"] = (
    original["complex_dir"]
    .fillna("")
    .str.contains(target_pattern, case=False, regex=True)
)

all_target = original[
    original["target_name_match"]
].copy()

print("Potential PARP1-related records:", len(all_target))

if len(all_target):
    print(
        all_target[
            [
                "pair_idx",
                "complex_dir",
                "receptor_pdb",
                "ligand_sdf_gz",
                "source_type",
                "shard_idx"
            ]
        ].head(30).to_string(index=False)
    )

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

selected_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_selected_target_audit.csv"
)

complete_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_complete_parp1_name_matches.csv"
)

selected.to_csv(selected_path, index=False)
all_target.to_csv(complete_path, index=False)

print("\n7. REPORTS SAVED")

print(selected_path)
print(complete_path)

print("\nCELL 106 COMPLETED")

CELL 106: CROSSDOCKED TARGET RELEVANCE AUDIT

1. DATASET COUNTS
Complete manifest: 2533288
Selected records: 495

2. SELECTED SOURCE TYPE DISTRIBUTION
source_type
cdonly    495
Name: count, dtype: int64

3. SELECTED SHARD DISTRIBUTION
shard_idx
7     90
17    87
12    83
1     81
13    77
31    77
Name: count, dtype: int64

4. UNIQUE PROTEINS
Unique complex directories: 495
Unique receptor files: 490

5. SELECTED PARP1 NAME MATCHES
Potential matches: 0
No obvious PARP1 names in selected records.

6. COMPLETE DATASET PARP1 SEARCH
Potential PARP1-related records: 3750
 pair_idx            complex_dir   receptor_pdb                                    ligand_sdf_gz  source_type  shard_idx
  1735535 PARP1_CHICK_652_1011_0 1a26_A_rec.pdb     1a26_A_rec_1efy_bzc_lig_it1_tt_docked.sdf.gz       cdonly         23
  1735536 PARP1_CHICK_652_1011_0 1a26_A_rec.pdb     1a26_A_rec_1efy_bzc_lig_it2_tt_docked.sdf.gz       cdonly         23
  1735537 PARP1_CHICK_652_1011_0 1a26_A_rec.pdb         1a26_A_r

In [162]:
# ============================================================
# CELL 107: PARP1 DATASET SPECIES AND UNIQUENESS AUDIT
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

manifest_path = (
    "/root/.cache/huggingface/hub/"
    "datasets--kohbanye--crossdocked2020/"
    "snapshots/017d8a19301c901471bba9f139331f86cd9ad278/"
    "manifest.parquet"
)

df = pd.read_parquet(manifest_path)

pattern = r"PARP1|PARP-1|ARTD1|POLY.?ADP.?RIBOSE"

parp = df[
    df["complex_dir"]
    .fillna("")
    .str.contains(pattern, case=False, regex=True)
].copy()

print("=" * 70)
print("CELL 107: PARP1 SPECIES AND UNIQUENESS AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC COUNTS
# ------------------------------------------------------------

print("\n1. TOTAL COUNTS")

print("PARP1-labelled records:", len(parp))
print("Unique complex directories:", parp["complex_dir"].nunique())
print("Unique receptor PDB files:", parp["receptor_pdb"].nunique())
print("Unique ligand files:", parp["ligand_sdf_gz"].nunique())

# ------------------------------------------------------------
# 2. SPECIES DISTRIBUTION
# ------------------------------------------------------------

parp["species_label"] = (
    parp["complex_dir"]
    .str.extract(r"PARP1_([^_]+)_", expand=False)
    .fillna("UNKNOWN")
)

print("\n2. SPECIES LABEL DISTRIBUTION")

print(
    parp.groupby("species_label")
    .agg(
        records=("pair_idx", "count"),
        unique_complexes=("complex_dir", "nunique"),
        unique_receptors=("receptor_pdb", "nunique"),
        unique_ligands=("ligand_sdf_gz", "nunique")
    )
    .sort_values("records", ascending=False)
    .to_string()
)

# ------------------------------------------------------------
# 3. HUMAN PARP1 RECORDS
# ------------------------------------------------------------

human = parp[
    parp["species_label"].str.upper().isin(
        ["HUMAN", "HOMO", "H_SAPIENS"]
    )
].copy()

print("\n3. HUMAN PARP1")

print("Records:", len(human))
print("Unique complexes:", human["complex_dir"].nunique())
print("Unique receptors:", human["receptor_pdb"].nunique())
print("Unique ligand files:", human["ligand_sdf_gz"].nunique())

if len(human):
    print("\nHuman PARP1 examples:")
    print(
        human[
            [
                "pair_idx",
                "complex_dir",
                "receptor_pdb",
                "ligand_sdf_gz",
                "source_type",
                "shard_idx"
            ]
        ].head(20).to_string(index=False)
    )

# ------------------------------------------------------------
# 4. UNIQUE RECEPTOR INVENTORY
# ------------------------------------------------------------

print("\n4. UNIQUE RECEPTOR INVENTORY")

receptors = (
    parp.groupby(
        ["species_label", "complex_dir", "receptor_pdb"],
        dropna=False
    )
    .agg(
        records=("pair_idx", "count"),
        unique_ligands=("ligand_sdf_gz", "nunique"),
        source_types=("source_type", lambda x: ",".join(sorted(set(x.dropna())))),
        shard_indices=("shard_idx", lambda x: ",".join(map(str, sorted(set(x.dropna())))))
    )
    .reset_index()
    .sort_values(
        ["species_label", "records"],
        ascending=[True, False]
    )
)

print("Unique receptor entries:", len(receptors))
print(receptors.head(30).to_string(index=False))

# ------------------------------------------------------------
# 5. SOURCE TYPE DISTRIBUTION
# ------------------------------------------------------------

print("\n5. SOURCE TYPE DISTRIBUTION")

print(parp["source_type"].value_counts(dropna=False))

# ------------------------------------------------------------
# 6. SAVE AUDIT
# ------------------------------------------------------------

all_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_parp1_unique_receptors.csv"
)

human_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_human_parp1_records.csv"
)

receptors.to_csv(all_path, index=False)
human.to_csv(human_path, index=False)

print("\n6. FILES SAVED")
print(all_path)
print(human_path)

print("\nCELL 107 COMPLETED")

CELL 107: PARP1 SPECIES AND UNIQUENESS AUDIT

1. TOTAL COUNTS
PARP1-labelled records: 3750
Unique complex directories: 2
Unique receptor PDB files: 41
Unique ligand files: 3750

2. SPECIES LABEL DISTRIBUTION
               records  unique_complexes  unique_receptors  unique_ligands
species_label                                                             
HUMAN             3609                 1                35            3609
CHICK              141                 1                 6             141

3. HUMAN PARP1
Records: 3609
Unique complexes: 1
Unique receptors: 35
Unique ligand files: 3609

Human PARP1 examples:
 pair_idx            complex_dir   receptor_pdb                                    ligand_sdf_gz source_type  shard_idx
  1735676 PARP1_HUMAN_775_1012_0 1uk0_A_rec.pdb            1uk0_A_rec_1uk0_frm_lig_tt_min.sdf.gz         it0         23
  1735677 PARP1_HUMAN_775_1012_0 1uk0_A_rec.pdb            1uk0_A_rec_1uk1_frq_lig_tt_min.sdf.gz      cdonly         23
  1735678 PA

In [163]:
# ============================================================
# CELL 108: PARP1 LIGAND IDENTITY AND M1 STRUCTURE OVERLAP
# ============================================================

import os
import re
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

input_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_human_parp1_records.csv"
)

df = pd.read_csv(input_path)

print("=" * 70)
print("CELL 108: LIGAND IDENTITY AND M1 STRUCTURE OVERLAP")
print("=" * 70)

# ------------------------------------------------------------
# 1. EXTRACT SOURCE LIGAND IDENTIFIERS
# ------------------------------------------------------------

def extract_ligand_identity(filename):

    if pd.isna(filename):
        return None

    name = os.path.basename(str(filename))

    # Expected examples:
    # 1uk0_A_rec_1uk0_frm_lig_tt_min.sdf.gz
    # 1uk0_A_rec_4hhy_15r_lig_it1_tt_docked.sdf.gz

    match = re.search(
        r"_rec_([0-9][a-zA-Z0-9]{3})_([a-zA-Z0-9]{3})_lig",
        name
    )

    if match:
        return (
            match.group(1).lower()
            + "_"
            + match.group(2).lower()
        )

    return "UNPARSED"

df["ligand_identity"] = (
    df["ligand_sdf_gz"].apply(extract_ligand_identity)
)

print("\n1. LIGAND IDENTITY COUNTS")

print("Total ligand files:", len(df))
print(
    "Unique extracted ligand identities:",
    df["ligand_identity"].nunique()
)
print(
    "Unparsed filenames:",
    (df["ligand_identity"] == "UNPARSED").sum()
)

print("\nMost frequent ligand identities:")

print(
    df["ligand_identity"]
    .value_counts()
    .head(20)
    .to_string()
)

# ------------------------------------------------------------
# 2. RECEPTOR STRUCTURE OVERLAP WITH M1
# ------------------------------------------------------------

m1_structures = {
    "5KPP",
    "5KPQ",
    "5WRZ",
    "5WS1",
    "7AAC",
    "7ONS",
    "7ONT",
    "8JNZ",
    "9ETQ",
    "9ILN"
}

df["receptor_structure_id"] = (
    df["receptor_pdb"]
    .str.extract(r"^([0-9A-Za-z]{4})_", expand=False)
    .str.upper()
)

df["overlaps_m1_structure"] = (
    df["receptor_structure_id"].isin(m1_structures)
)

overlap = df[
    df["overlaps_m1_structure"]
].copy()

print("\n2. M1 STRUCTURE OVERLAP")

print("Matching records:", len(overlap))

print(
    "Matching receptor structures:",
    overlap["receptor_structure_id"].nunique()
)

print(
    overlap.groupby("receptor_structure_id")
    .agg(
        records=("pair_idx", "count"),
        unique_ligands=("ligand_identity", "nunique"),
        receptor_files=("receptor_pdb", "nunique")
    )
    .to_string()
)

# ------------------------------------------------------------
# 3. UNIQUE LIGAND IDENTITIES PER RECEPTOR
# ------------------------------------------------------------

print("\n3. UNIQUE LIGAND IDENTITIES PER RECEPTOR")

receptor_summary = (
    df.groupby(
        ["receptor_structure_id", "receptor_pdb"]
    )
    .agg(
        records=("pair_idx", "count"),
        unique_ligand_identities=("ligand_identity", "nunique")
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print(receptor_summary.to_string(index=False))

# ------------------------------------------------------------
# 4. SAVE RESULTS
# ------------------------------------------------------------

identity_path = os.path.join(
    OUTPUT_DIR,
    "human_parp1_ligand_identity_audit.csv"
)

overlap_path = os.path.join(
    OUTPUT_DIR,
    "human_parp1_m1_structure_overlap.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "human_parp1_receptor_summary.csv"
)

df.to_csv(identity_path, index=False)
overlap.to_csv(overlap_path, index=False)
receptor_summary.to_csv(summary_path, index=False)

print("\n4. FILES SAVED")
print(identity_path)
print(overlap_path)
print(summary_path)

print("\nCELL 108 COMPLETED")

CELL 108: LIGAND IDENTITY AND M1 STRUCTURE OVERLAP

1. LIGAND IDENTITY COUNTS
Total ligand files: 3609
Unique extracted ligand identities: 32
Unparsed filenames: 0

Most frequent ligand identities:
ligand_identity
4hhy_15r    174
1uk0_frm    117
2rd6_78p    117
1uk1_frq    117
4pjt_2yq    117
5wry_4yr    117
5wtc_7ul    117
5wrz_7u3    117
5kpp_6wz    117
5kpq_6x2    117
5wrq_7tx    117
5kpo_6wy    117
5ds3_09l    117
5kpn_6wx    117
4zzz_fsu    117
4und_2yq    117
4gv7_mew    116
4uxb_p34    116
5ws0_7u6    113
3gn7_3gn    112

2. M1 STRUCTURE OVERLAP
Matching records: 468
Matching receptor structures: 4
                       records  unique_ligands  receptor_files
receptor_structure_id                                         
5KPP                       155              32               1
5KPQ                       125              32               1
5WRZ                        36              32               1
5WS1                       152              32               1

3. UNIQU

In [164]:
# ============================================================
# CELL 109: CROSSDOCKED LIGAND OVERLAP WITH EXPERIMENTAL REFERENCES
# ============================================================

import os
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

INPUT = os.path.join(
    OUTPUT_DIR,
    "human_parp1_ligand_identity_audit.csv"
)

df = pd.read_csv(INPUT)

print("=" * 70)
print("CELL 109: EXPERIMENTAL LIGAND OVERLAP AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. EXPERIMENTAL REFERENCE COMPONENTS
# ------------------------------------------------------------

reference_components = {
    "6WZ", "6X2", "7U3", "7U9", "78P",
    "VKT", "VKQ", "ERV", "A1H63", "A1D9M"
}

# Extract component code from identity: 5kpp_6wz -> 6WZ
df["component_id"] = (
    df["ligand_identity"]
    .astype(str)
    .str.split("_")
    .str[-1]
    .str.upper()
)

df["matches_experimental_reference"] = (
    df["component_id"].isin(reference_components)
)

# ------------------------------------------------------------
# 2. COMPONENT IDENTITY SUMMARY
# ------------------------------------------------------------

summary = (
    df.groupby(
        ["ligand_identity", "component_id"],
        dropna=False
    )
    .agg(
        total_records=("pair_idx", "count"),
        receptor_structures=("receptor_structure_id", "nunique"),
        source_types=(
            "source_type",
            lambda x: ",".join(sorted(set(x.dropna())))
        ),
        m1_overlap_records=("overlaps_m1_structure", "sum"),
        experimental_reference_match=(
            "matches_experimental_reference", "max"
        )
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

print("\n1. ALL DISTINCT LIGAND IDENTITIES")
print(summary.to_string(index=False))

# ------------------------------------------------------------
# 3. EXPERIMENTAL REFERENCE OVERLAP
# ------------------------------------------------------------

matched = summary[
    summary["experimental_reference_match"]
].copy()

print("\n2. MATCHES TO OUR 10 EXPERIMENTAL REFERENCES")

print("Matching ligand identities:", len(matched))

if len(matched):
    print(matched.to_string(index=False))
else:
    print("No exact component-code matches.")

# ------------------------------------------------------------
# 4. LIGANDS IN THE FOUR M1-OVERLAPPING RECEPTORS
# ------------------------------------------------------------

m1_receptors = {"5KPP", "5KPQ", "5WRZ", "5WS1"}

m1_records = df[
    df["receptor_structure_id"].isin(m1_receptors)
].copy()

m1_summary = (
    m1_records.groupby(
        ["receptor_structure_id", "ligand_identity"]
    )
    .agg(
        records=("pair_idx", "count"),
        source_types=(
            "source_type",
            lambda x: ",".join(sorted(set(x.dropna())))
        )
    )
    .reset_index()
)

print("\n3. LIGAND COVERAGE IN M1-OVERLAPPING STRUCTURES")

print(
    m1_summary.groupby("receptor_structure_id")
    .agg(
        distinct_ligands=("ligand_identity", "nunique"),
        total_records=("records", "sum")
    )
    .to_string()
)

# ------------------------------------------------------------
# 5. SAVE REPORTS
# ------------------------------------------------------------

summary_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_32_ligand_identity_summary.csv"
)

matched_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_experimental_reference_overlap.csv"
)

m1_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_m1_receptor_ligand_coverage.csv"
)

summary.to_csv(summary_path, index=False)
matched.to_csv(matched_path, index=False)
m1_summary.to_csv(m1_path, index=False)

print("\n4. REPORTS SAVED")
print(summary_path)
print(matched_path)
print(m1_path)

print("\nCELL 109 COMPLETED")

CELL 109: EXPERIMENTAL LIGAND OVERLAP AUDIT

1. ALL DISTINCT LIGAND IDENTITIES
ligand_identity component_id  total_records  receptor_structures            source_types  m1_overlap_records  experimental_reference_match
       4hhy_15r          15R            174                   35 cdonly,it0,it2_redocked                  20                         False
       1uk0_frm          FRM            117                   33              cdonly,it0                  16                         False
       2rd6_78p          78P            117                   33 cdonly,it0,it2_redocked                  16                          True
       1uk1_frq          FRQ            117                   33              cdonly,it0                  16                         False
       4pjt_2yq          2YQ            117                   33 cdonly,it0,it2_redocked                  16                         False
       5wry_4yr          4YR            117                   33              cdonly,it

In [165]:
# ============================================================
# CELL 110: LOCATE PARP1 LIGAND ARCHIVE
# ============================================================

import os
import pandas as pd
from huggingface_hub import HfApi

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")

REPO_ID = "kohbanye/crossdocked2020"

print("=" * 70)
print("CELL 110: LOCATE PARP1 LIGAND ARCHIVE")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD HUMAN PARP1 RECORDS
# ------------------------------------------------------------

records_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_human_parp1_records.csv"
)

df = pd.read_csv(records_path)

print("\n1. HUMAN PARP1 RECORDS")
print("Total records:", len(df))

# ------------------------------------------------------------
# 2. SELECT THE 32 DISTINCT LIGAND IDENTITIES
# ------------------------------------------------------------

identities = sorted(df["ligand_identity"].dropna().unique())

print("\n2. LIGAND IDENTITIES")
print("Unique identities:", len(identities))

# Select one representative SDF file per identity.
retrieval = (
    df.sort_values("pair_idx")
    .drop_duplicates("ligand_identity")
    [[
        "ligand_identity",
        "pair_idx",
        "ligand_sdf_gz",
        "receptor_pdb",
        "source_type",
        "shard_idx"
    ]]
    .copy()
)

print("Representative ligand files:", len(retrieval))

# ------------------------------------------------------------
# 3. VERIFY SHARD
# ------------------------------------------------------------

shards = sorted(retrieval["shard_idx"].dropna().unique())

print("\n3. SHARD INDICES")
print(shards)

if len(shards) != 1 or int(shards[0]) != 23:
    raise ValueError(
        "Expected all selected ligand records to belong to shard 23."
    )

# ------------------------------------------------------------
# 4. INSPECT HUGGING FACE REPOSITORY FILES
# ------------------------------------------------------------

print("\n4. INSPECTING HUGGING FACE REPOSITORY")

api = HfApi()

repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

archives = [
    f for f in repo_files
    if f.endswith((".tar.gz", ".tgz"))
    and "ligand" in f.lower()
]

print("Ligand archives found:", len(archives))

print("\nAvailable ligand archives:")
for archive in archives:
    print(archive)

# ------------------------------------------------------------
# 5. IDENTIFY SHARD-23 ARCHIVE
# ------------------------------------------------------------

candidates = [
    f for f in archives
    if "023" in os.path.basename(f)
    or "shard-23" in os.path.basename(f).lower()
    or "shard_23" in os.path.basename(f).lower()
]

if len(candidates) != 1:
    print("\nCould not uniquely identify shard-23 archive.")
    print("Candidate matches:", candidates)
    raise RuntimeError(
        "Stop here. Do not download an archive until its identity is verified."
    )

archive_path = candidates[0]

print("\n5. IDENTIFIED ARCHIVE")
print(archive_path)

# ------------------------------------------------------------
# 6. CHECK ARCHIVE SIZE
# ------------------------------------------------------------

info = api.get_paths_info(
    repo_id=REPO_ID,
    paths=[archive_path],
    repo_type="dataset"
)

archive_size = info[0].size

print("\n6. ARCHIVE SIZE")
print(f"{archive_size / (1024**2):.2f} MB")
print(f"{archive_size / (1024**3):.3f} GB")

# ------------------------------------------------------------
# 7. SAVE RETRIEVAL MANIFEST
# ------------------------------------------------------------

retrieval["archive_path"] = archive_path

manifest_output = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

retrieval.to_csv(manifest_output, index=False)

print("\n7. RETRIEVAL MANIFEST SAVED")
print(manifest_output)

print("\n8. REPRESENTATIVE FILES")

print(
    retrieval[
        ["ligand_identity", "ligand_sdf_gz", "pair_idx"]
    ].to_string(index=False)
)

print("\nCELL 110 COMPLETED")

CELL 110: LOCATE PARP1 LIGAND ARCHIVE

1. HUMAN PARP1 RECORDS
Total records: 3609


KeyError: 'ligand_identity'

In [166]:
# ============================================================
# CELL 110A: INSPECT EXISTING PARP1 IDENTITY FILES
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/content/drive/MyDrive/Member2_Project/member2_outputs"

files_to_check = [
    "crossdocked_human_parp1_records.csv",
    "human_parp1_ligand_identity_audit.csv",
    "crossdocked_32_ligand_identity_summary.csv",
    "human_parp1_m1_structure_overlap.csv"
]

for filename in files_to_check:

    path = os.path.join(OUTPUT_DIR, filename)

    print("\n" + "=" * 70)
    print("FILE:", filename)
    print("=" * 70)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    data = pd.read_csv(path)

    print("Shape:", data.shape)
    print("\nColumns:")
    print(data.columns.tolist())

    print("\nFirst 3 records:")
    display(data.head(3))

print("\nINSPECTION COMPLETED")


FILE: crossdocked_human_parp1_records.csv
Shape: (3609, 37)

Columns:
['pair_idx', 'complex_dir', 'receptor_pdb', 'ligand_sdf_gz', 'source_type', 'shard_idx', 'label', 'score1', 'score2', 'cdonly_fold0', 'cdonly_fold1', 'cdonly_fold2', 'it0_fold0', 'it0_fold1', 'it0_fold2', 'it2_fold0', 'it2_fold1', 'it2_fold2', 'it2_redocked_fold0', 'it2_redocked_fold1', 'it2_redocked_fold2', 'it2_10p20n_fold0', 'it2_10p20n_fold1', 'it2_10p20n_fold2', 'mod_it2_fold0', 'mod_it2_fold1', 'mod_it2_fold2', 'it2_completeset_fold0', 'it2_completeset_fold1', 'it2_completeset_fold2', 'it2_redocked_completeset_fold0', 'it2_redocked_completeset_fold1', 'it2_redocked_completeset_fold2', 'mod_it2_completeset_fold0', 'mod_it2_completeset_fold1', 'mod_it2_completeset_fold2', 'species_label']

First 3 records:


,pair_idx,complex_dir,receptor_pdb,ligand_sdf_gz,source_type,shard_idx,label,score1,score2,cdonly_fold0,...,it2_completeset_fold0,it2_completeset_fold1,it2_completeset_fold2,it2_redocked_completeset_fold0,it2_redocked_completeset_fold1,it2_redocked_completeset_fold2,mod_it2_completeset_fold0,mod_it2_completeset_fold1,mod_it2_completeset_fold2,species_label
0,1735676,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1uk0_frm_lig_tt_min.sdf.gz,it0,23,1,8.0969,0.7188,NaN,...,train,NaN,NaN,train,NaN,NaN,train,NaN,NaN,HUMAN
1,1735677,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1uk1_frq_lig_tt_min.sdf.gz,cdonly,23,1,7.2218,0.5831,test,...,train,NaN,NaN,NaN,NaN,NaN,train,NaN,NaN,HUMAN
2,1735678,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1wok_cnq_lig_tt_min.sdf.gz,cdonly,23,1,7.4815,1.6894,test,...,train,NaN,NaN,NaN,NaN,NaN,train,NaN,NaN,HUMAN



FILE: human_parp1_ligand_identity_audit.csv
Shape: (3609, 40)

Columns:
['pair_idx', 'complex_dir', 'receptor_pdb', 'ligand_sdf_gz', 'source_type', 'shard_idx', 'label', 'score1', 'score2', 'cdonly_fold0', 'cdonly_fold1', 'cdonly_fold2', 'it0_fold0', 'it0_fold1', 'it0_fold2', 'it2_fold0', 'it2_fold1', 'it2_fold2', 'it2_redocked_fold0', 'it2_redocked_fold1', 'it2_redocked_fold2', 'it2_10p20n_fold0', 'it2_10p20n_fold1', 'it2_10p20n_fold2', 'mod_it2_fold0', 'mod_it2_fold1', 'mod_it2_fold2', 'it2_completeset_fold0', 'it2_completeset_fold1', 'it2_completeset_fold2', 'it2_redocked_completeset_fold0', 'it2_redocked_completeset_fold1', 'it2_redocked_completeset_fold2', 'mod_it2_completeset_fold0', 'mod_it2_completeset_fold1', 'mod_it2_completeset_fold2', 'species_label', 'ligand_identity', 'receptor_structure_id', 'overlaps_m1_structure']

First 3 records:


,pair_idx,complex_dir,receptor_pdb,ligand_sdf_gz,source_type,shard_idx,label,score1,score2,cdonly_fold0,...,it2_redocked_completeset_fold0,it2_redocked_completeset_fold1,it2_redocked_completeset_fold2,mod_it2_completeset_fold0,mod_it2_completeset_fold1,mod_it2_completeset_fold2,species_label,ligand_identity,receptor_structure_id,overlaps_m1_structure
0,1735676,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1uk0_frm_lig_tt_min.sdf.gz,it0,23,1,8.0969,0.7188,NaN,...,train,NaN,NaN,train,NaN,NaN,HUMAN,1uk0_frm,1UK0,False
1,1735677,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1uk1_frq_lig_tt_min.sdf.gz,cdonly,23,1,7.2218,0.5831,test,...,NaN,NaN,NaN,train,NaN,NaN,HUMAN,1uk1_frq,1UK0,False
2,1735678,PARP1_HUMAN_775_1012_0,1uk0_A_rec.pdb,1uk0_A_rec_1wok_cnq_lig_tt_min.sdf.gz,cdonly,23,1,7.4815,1.6894,test,...,NaN,NaN,NaN,train,NaN,NaN,HUMAN,1wok_cnq,1UK0,False



FILE: crossdocked_32_ligand_identity_summary.csv
Shape: (32, 7)

Columns:
['ligand_identity', 'component_id', 'total_records', 'receptor_structures', 'source_types', 'm1_overlap_records', 'experimental_reference_match']

First 3 records:


,ligand_identity,component_id,total_records,receptor_structures,source_types,m1_overlap_records,experimental_reference_match
0,4hhy_15r,15R,174,35,"cdonly,it0,it2_redocked",20,False
1,1uk0_frm,FRM,117,33,"cdonly,it0",16,False
2,2rd6_78p,78P,117,33,"cdonly,it0,it2_redocked",16,True



FILE: human_parp1_m1_structure_overlap.csv
Shape: (468, 40)

Columns:
['pair_idx', 'complex_dir', 'receptor_pdb', 'ligand_sdf_gz', 'source_type', 'shard_idx', 'label', 'score1', 'score2', 'cdonly_fold0', 'cdonly_fold1', 'cdonly_fold2', 'it0_fold0', 'it0_fold1', 'it0_fold2', 'it2_fold0', 'it2_fold1', 'it2_fold2', 'it2_redocked_fold0', 'it2_redocked_fold1', 'it2_redocked_fold2', 'it2_10p20n_fold0', 'it2_10p20n_fold1', 'it2_10p20n_fold2', 'mod_it2_fold0', 'mod_it2_fold1', 'mod_it2_fold2', 'it2_completeset_fold0', 'it2_completeset_fold1', 'it2_completeset_fold2', 'it2_redocked_completeset_fold0', 'it2_redocked_completeset_fold1', 'it2_redocked_completeset_fold2', 'mod_it2_completeset_fold0', 'mod_it2_completeset_fold1', 'mod_it2_completeset_fold2', 'species_label', 'ligand_identity', 'receptor_structure_id', 'overlaps_m1_structure']

First 3 records:


,pair_idx,complex_dir,receptor_pdb,ligand_sdf_gz,source_type,shard_idx,label,score1,score2,cdonly_fold0,...,it2_redocked_completeset_fold0,it2_redocked_completeset_fold1,it2_redocked_completeset_fold2,mod_it2_completeset_fold0,mod_it2_completeset_fold1,mod_it2_completeset_fold2,species_label,ligand_identity,receptor_structure_id,overlaps_m1_structure
0,1738443,PARP1_HUMAN_775_1012_0,5kpp_A_rec.pdb,5kpp_A_rec_1uk0_frm_lig_it1_it2_tt_docked.sdf.gz,cdonly,23,0,-8.0969,8.5085,test,...,NaN,NaN,NaN,train,NaN,NaN,HUMAN,1uk0_frm,5KPP,True
1,1738444,PARP1_HUMAN_775_1012_0,5kpp_A_rec.pdb,5kpp_A_rec_1uk0_frm_lig_it1_tt_docked.sdf.gz,cdonly,23,0,-8.0969,5.9458,test,...,NaN,NaN,NaN,train,NaN,NaN,HUMAN,1uk0_frm,5KPP,True
2,1738445,PARP1_HUMAN_775_1012_0,5kpp_A_rec.pdb,5kpp_A_rec_1uk0_frm_lig_it2_tt_docked.sdf.gz,cdonly,23,0,-8.0969,3.4387,test,...,NaN,NaN,NaN,train,NaN,NaN,HUMAN,1uk0_frm,5KPP,True



INSPECTION COMPLETED


In [167]:
# ============================================================
# CELL 110: IDENTIFY THE PARP1 LIGAND ARCHIVE
# ============================================================

import os
import pandas as pd
from huggingface_hub import HfApi

OUTPUT_DIR = (
    "/content/drive/MyDrive/Member2_Project/member2_outputs"
)

REPO_ID = "kohbanye/crossdocked2020"

print("=" * 70)
print("CELL 110: PARP1 LIGAND ARCHIVE IDENTIFICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD THE CORRECT IDENTITY AUDIT
# ------------------------------------------------------------

identity_file = os.path.join(
    OUTPUT_DIR,
    "human_parp1_ligand_identity_audit.csv"
)

df = pd.read_csv(identity_file)

required_columns = [
    "pair_idx",
    "ligand_identity",
    "ligand_sdf_gz",
    "receptor_pdb",
    "shard_idx"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("\n1. HUMAN PARP1 IDENTITY DATA")
print("Records:", len(df))
print("Unique ligand identities:", df["ligand_identity"].nunique())
print("Shard indices:", sorted(df["shard_idx"].dropna().unique()))

# ------------------------------------------------------------
# 2. SELECT ONE REPRESENTATIVE FILE PER IDENTITY
# ------------------------------------------------------------

retrieval = (
    df.sort_values(["pair_idx"])
      .drop_duplicates("ligand_identity")
      [required_columns]
      .copy()
      .reset_index(drop=True)
)

print("\n2. REPRESENTATIVE LIGAND FILES")
print("Selected:", len(retrieval))

display(
    retrieval[
        ["ligand_identity", "pair_idx", "ligand_sdf_gz"]
    ]
)

# ------------------------------------------------------------
# 3. CHECK SHARD NUMBER
# ------------------------------------------------------------

shards = retrieval["shard_idx"].dropna().unique()

if len(shards) != 1 or int(shards[0]) != 23:
    raise ValueError(
        f"Unexpected shard assignment: {shards}"
    )

# ------------------------------------------------------------
# 4. INSPECT HUGGING FACE REPOSITORY
# ------------------------------------------------------------

print("\n3. INSPECTING HUGGING FACE REPOSITORY")

api = HfApi()

repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

ligand_archives = [
    path for path in repo_files
    if "ligand" in path.lower()
    and path.endswith((".tar.gz", ".tgz"))
]

print("Ligand archives found:", len(ligand_archives))

for path in ligand_archives:
    print(path)

# ------------------------------------------------------------
# 5. GET ARCHIVE SIZES
# ------------------------------------------------------------

print("\n4. CHECKING ARCHIVE SIZES")

if ligand_archives:

    archive_info = api.get_paths_info(
        repo_id=REPO_ID,
        paths=ligand_archives,
        repo_type="dataset"
    )

    archive_table = pd.DataFrame([
        {
            "archive": item.path,
            "size_MB": round(item.size / (1024 ** 2), 2),
            "size_GB": round(item.size / (1024 ** 3), 3)
        }
        for item in archive_info
    ])

    display(archive_table)

else:
    archive_table = pd.DataFrame()
    print("No ligand tar archives found.")

# ------------------------------------------------------------
# 6. SAVE RETRIEVAL MANIFEST
# ------------------------------------------------------------

manifest_path = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

retrieval.to_csv(manifest_path, index=False)

print("\n5. RETRIEVAL MANIFEST SAVED")
print(manifest_path)

print("\nCELL 110 COMPLETED")
print("No large files downloaded.")

CELL 110: PARP1 LIGAND ARCHIVE IDENTIFICATION

1. HUMAN PARP1 IDENTITY DATA
Records: 3609
Unique ligand identities: 32
Shard indices: [np.int64(23)]

2. REPRESENTATIVE LIGAND FILES
Selected: 32


,ligand_identity,pair_idx,ligand_sdf_gz
0,1uk0_frm,1735676,1uk0_A_rec_1uk0_frm_lig_tt_min.sdf.gz
1,1uk1_frq,1735677,1uk0_A_rec_1uk1_frq_lig_tt_min.sdf.gz
2,1wok_cnq,1735678,1uk0_A_rec_1wok_cnq_lig_tt_min.sdf.gz
3,2rcw_aai,1735679,1uk0_A_rec_2rcw_aai_lig_tt_min.sdf.gz
4,2rd6_78p,1735680,1uk0_A_rec_2rd6_78p_lig_tt_min.sdf.gz
5,3gjw_gjw,1735681,1uk0_A_rec_3gjw_gjw_lig_tt_min.sdf.gz
6,3gn7_3gn,1735682,1uk0_A_rec_3gn7_3gn_lig_tt_min.sdf.gz
7,3l3l_l3l,1735683,1uk0_A_rec_3l3l_l3l_lig_tt_min.sdf.gz
8,3l3m_a92,1735684,1uk0_A_rec_3l3m_a92_lig_tt_min.sdf.gz
9,4gv7_mew,1735685,1uk0_A_rec_4gv7_mew_lig_tt_min.sdf.gz



3. INSPECTING HUGGING FACE REPOSITORY
Ligand archives found: 0

4. CHECKING ARCHIVE SIZES
No ligand tar archives found.

5. RETRIEVAL MANIFEST SAVED
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_ligand_retrieval_manifest.csv

CELL 110 COMPLETED
No large files downloaded.


In [168]:
# ============================================================
# CELL 111: COMPLETE HUGGING FACE ARCHIVE DISCOVERY
# ============================================================

import os
import pandas as pd
from huggingface_hub import HfApi

REPO_ID = "kohbanye/crossdocked2020"
OUTPUT_DIR = "/content/drive/MyDrive/Member2_Project/member2_outputs"

api = HfApi()

print("=" * 70)
print("CELL 111: COMPLETE REPOSITORY ARCHIVE DISCOVERY")
print("=" * 70)

# 1. Retrieve complete repository file list
repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

print("\n1. TOTAL REPOSITORY FILES:", len(repo_files))

# 2. Display every archive
archives = [
    f for f in repo_files
    if f.lower().endswith((".tar", ".tar.gz", ".tgz", ".zip"))
]

print("\n2. ALL ARCHIVE FILES:", len(archives))

for f in archives:
    print(f)

# 3. Display all repository paths containing shard references
print("\n3. PATHS CONTAINING 'SHARD'")

for f in repo_files:
    if "shard" in f.lower():
        print(f)

# 4. Inspect README/documentation files
print("\n4. REPOSITORY DOCUMENTATION FILES")

docs = [
    f for f in repo_files
    if any(x in os.path.basename(f).lower()
           for x in ["readme", "metadata", "index", "manifest"])
]

for f in docs:
    print(f)

# 5. Inspect previously saved repository inventory
inventory_path = os.path.join(
    OUTPUT_DIR,
    "huggingface_crossdocked_repository_inventory.csv"
)

if os.path.exists(inventory_path):
    print("\n5. PREVIOUS REPOSITORY INVENTORY")

    inventory = pd.read_csv(inventory_path)

    print("Shape:", inventory.shape)
    print("Columns:", inventory.columns.tolist())

    display(inventory.head(50))
else:
    print("\nPrevious inventory CSV not found.")

# 6. Save complete repository listing
listing_path = os.path.join(
    OUTPUT_DIR,
    "crossdocked_complete_repository_file_list.csv"
)

pd.DataFrame({"repository_path": repo_files}).to_csv(
    listing_path,
    index=False
)

print("\n6. COMPLETE FILE LIST SAVED:")
print(listing_path)

print("\nCELL 111 COMPLETED")
print("No archive downloaded.")

CELL 111: COMPLETE REPOSITORY ARCHIVE DISCOVERY

1. TOTAL REPOSITORY FILES: 39

2. ALL ARCHIVE FILES: 36
ligands/000000.tar
ligands/000001.tar
ligands/000002.tar
ligands/000003.tar
ligands/000004.tar
ligands/000005.tar
ligands/000006.tar
ligands/000007.tar
ligands/000008.tar
ligands/000009.tar
ligands/000010.tar
ligands/000011.tar
ligands/000012.tar
ligands/000013.tar
ligands/000014.tar
ligands/000015.tar
ligands/000016.tar
ligands/000017.tar
ligands/000018.tar
ligands/000019.tar
ligands/000020.tar
ligands/000021.tar
ligands/000022.tar
ligands/000023.tar
ligands/000024.tar
ligands/000025.tar
ligands/000026.tar
ligands/000027.tar
ligands/000028.tar
ligands/000029.tar
ligands/000030.tar
ligands/000031.tar
ligands/000032.tar
ligands/000033.tar
ligands/000034.tar
receptors/shard-000.tar.gz

3. PATHS CONTAINING 'SHARD'
receptors/shard-000.tar.gz

4. REPOSITORY DOCUMENTATION FILES
README.md
manifest.parquet

5. PREVIOUS REPOSITORY INVENTORY
Shape: (41, 3)
Columns: ['path', 'size_bytes', 'ext

,path,size_bytes,extension
0,ligands,NaN,ligands
1,receptors,NaN,receptors
2,.gitattributes,2.504000e+03,gitattributes
3,README.md,2.734000e+03,md
4,ligands/000000.tar,6.456115e+08,tar
5,ligands/000001.tar,6.565683e+08,tar
6,ligands/000002.tar,6.325965e+08,tar
7,ligands/000003.tar,6.501376e+08,tar
8,ligands/000004.tar,6.472192e+08,tar
9,ligands/000005.tar,6.392627e+08,tar



6. COMPLETE FILE LIST SAVED:
/content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_complete_repository_file_list.csv

CELL 111 COMPLETED
No archive downloaded.


In [169]:
# ============================================================
# CELL 112: VERIFY SHARD 23 ARCHIVE
# ============================================================

import os
import shutil
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "kohbanye/crossdocked2020"
ARCHIVE = "ligands/000023.tar"

print("=" * 70)
print("CELL 112: SHARD-23 ARCHIVE VERIFICATION")
print("=" * 70)

# 1. Download only the small README
readme_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="README.md",
    repo_type="dataset"
)

print("\n1. REPOSITORY README")
with open(readme_path, "r", encoding="utf-8") as f:
    readme = f.read()

print(readme)

# 2. Verify archive metadata
api = HfApi()

info = api.get_paths_info(
    repo_id=REPO_ID,
    paths=[ARCHIVE],
    repo_type="dataset"
)[0]

print("\n2. ARCHIVE METADATA")
print("Archive:", info.path)
print(f"Size: {info.size / (1024**2):.2f} MB")

# 3. Check available Colab disk space
usage = shutil.disk_usage("/content")

print("\n3. COLAB DISK SPACE")
print(f"Total: {usage.total / (1024**3):.2f} GB")
print(f"Used: {usage.used / (1024**3):.2f} GB")
print(f"Free: {usage.free / (1024**3):.2f} GB")

# 4. Confirm shard-23 records
manifest_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/parp1_ligand_retrieval_manifest.csv"
)

import pandas as pd

retrieval = pd.read_csv(manifest_path)

print("\n4. RETRIEVAL MANIFEST")
print("Selected ligand identities:", len(retrieval))
print("Shard values:", retrieval["shard_idx"].unique())

print("\nCELL 112 COMPLETED")
print("No large archive downloaded.")

CELL 112: SHARD-23 ARCHIVE VERIFICATION


README.md:   0%|          | 0.00/2.73k [00:00<?, ?B/s]


1. REPOSITORY README
---
license: cc0-1.0
task_categories:
  - other
tags:
  - drug-discovery
  - molecular-generation
  - protein-ligand
  - structural-biology
size_categories:
  - 1M<n<10M
---

# CrossDocked2020

Pre-processed CrossDocked2020 dataset containing raw receptor PDB and ligand
SDF.gz files, organized for efficient loading.

## Dataset Summary

- **Unique pairs**: 2,533,288
- **Unique receptor PDB files**: 24,533
- **Source types**: cdonly, it0, it2, it2_redocked, it2_10p20n, mod_it2,
  it2_completeset, it2_redocked_completeset, mod_it2_completeset
- **Fold splits**: 3 folds (0, 1, 2) per source type category
  (completeset categories only populate fold0)


## Pose Counts

Each ligand `.sdf.gz` contains **multiple docked poses** concatenated as SDF
records. The 5 filename patterns correspond to different gnina iterations
(`tt_min` is the energy-minimized crystal pose; the others are docked poses
from successive scoring iterations). Pose totals across the 2,533,288
uploade

In [170]:
# ============================================================
# CELL 113: DOWNLOAD SHARD 23 AND EXTRACT SELECTED LIGANDS
# ============================================================

import os
import json
import tarfile
import shutil
import gzip
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "kohbanye/crossdocked2020"
ARCHIVE_NAME = "ligands/000023.tar"

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "member2_outputs"
)

MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

EXTRACT_DIR = os.path.join(
    OUTPUT_DIR,
    "crossdocked_parp1_ligands_shard23"
)

os.makedirs(EXTRACT_DIR, exist_ok=True)

print("=" * 70)
print("CELL 113: SELECTIVE PARP1 LIGAND EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD SELECTED PAIR IDS
# ------------------------------------------------------------

import pandas as pd

retrieval = pd.read_csv(MANIFEST_PATH)

expected_ids = {
    f"{int(pair_id):07d}"
    for pair_id in retrieval["pair_idx"]
}

print("\n1. SELECTED RECORDS")
print("Expected ligand records:", len(expected_ids))

# ------------------------------------------------------------
# 2. DOWNLOAD SHARD 23
# ------------------------------------------------------------

print("\n2. DOWNLOADING SHARD 23")
print("Archive:", ARCHIVE_NAME)

archive_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=ARCHIVE_NAME,
    repo_type="dataset"
)

print("Download completed.")
print("Cached archive:", archive_path)
print(
    "Archive size:",
    round(os.path.getsize(archive_path) / (1024**2), 2),
    "MB"
)

# ------------------------------------------------------------
# 3. INSPECT TAR MEMBERS
# ------------------------------------------------------------

print("\n3. INSPECTING TAR MEMBERS")

with tarfile.open(archive_path, "r:") as tar:

    members = tar.getmembers()

    member_map = {}

    for member in members:
        if not member.isfile():
            continue

        basename = os.path.basename(member.name)

        if basename.endswith(".sdf.gz"):
            member_map[basename] = member

        elif basename.endswith(".json"):
            member_map[basename] = member

    print("Total regular files:", len(member_map))

    # --------------------------------------------------------
    # 4. SELECT ONLY THE 32 REQUIRED PAIRS
    # --------------------------------------------------------

    extracted_records = []
    missing_files = []

    print("\n4. EXTRACTING SELECTED RECORDS")

    for pair_id in sorted(expected_ids):

        sdf_name = f"{pair_id}.sdf.gz"
        json_name = f"{pair_id}.json"

        sdf_member = member_map.get(sdf_name)
        json_member = member_map.get(json_name)

        if sdf_member is None or json_member is None:
            missing_files.append(pair_id)
            continue

        # Extract SDF.gz safely
        sdf_output = os.path.join(
            EXTRACT_DIR,
            sdf_name
        )

        with tar.extractfile(sdf_member) as source:
            with open(sdf_output, "wb") as destination:
                shutil.copyfileobj(source, destination)

        # Extract JSON metadata safely
        json_output = os.path.join(
            EXTRACT_DIR,
            json_name
        )

        with tar.extractfile(json_member) as source:
            with open(json_output, "wb") as destination:
                shutil.copyfileobj(source, destination)

        # Verify gzip integrity
        try:
            with gzip.open(sdf_output, "rb") as gz:
                first_bytes = gz.read(100)

            gzip_valid = len(first_bytes) > 0

        except Exception:
            gzip_valid = False

        # Read metadata
        with open(json_output, "r") as f:
            metadata = json.load(f)

        extracted_records.append({
            "pair_idx": int(pair_id),
            "sdf_file": sdf_name,
            "json_file": json_name,
            "sdf_size_bytes": os.path.getsize(sdf_output),
            "gzip_valid": gzip_valid,
            "metadata": metadata
        })

    # --------------------------------------------------------
    # 5. REPORT
    # --------------------------------------------------------

    print("\n5. EXTRACTION RESULTS")

    print("Expected:", len(expected_ids))
    print("Extracted:", len(extracted_records))
    print("Missing:", len(missing_files))

    if missing_files:
        print("Missing pair IDs:", missing_files)

    # --------------------------------------------------------
    # 6. SAVE EXTRACTION AUDIT
    # --------------------------------------------------------

    audit_rows = []

    for record in extracted_records:

        audit_rows.append({
            "pair_idx": record["pair_idx"],
            "sdf_file": record["sdf_file"],
            "json_file": record["json_file"],
            "sdf_size_bytes": record["sdf_size_bytes"],
            "gzip_valid": record["gzip_valid"],
            "metadata": json.dumps(record["metadata"])
        })

    audit_df = pd.DataFrame(audit_rows)

    audit_path = os.path.join(
        OUTPUT_DIR,
        "parp1_shard23_extraction_audit.csv"
    )

    audit_df.to_csv(audit_path, index=False)

print("\n6. OUTPUT DIRECTORY")
print(EXTRACT_DIR)

print("\n7. AUDIT FILE")
print(audit_path)

print("\n8. GZIP VALIDATION")
if len(audit_df):
    print(audit_df["gzip_valid"].value_counts(dropna=False))

print("\nCELL 113 COMPLETED")

CELL 113: SELECTIVE PARP1 LIGAND EXTRACTION

1. SELECTED RECORDS
Expected ligand records: 32

2. DOWNLOADING SHARD 23
Archive: ligands/000023.tar


ligands/000023.tar: reconstructing file:   0%|          |  0.00B /  666MB            

ligands/000023.tar: downloading bytes:           |  0.00B            

Download completed.
Cached archive: /root/.cache/huggingface/hub/datasets--kohbanye--crossdocked2020/snapshots/017d8a19301c901471bba9f139331f86cd9ad278/ligands/000023.tar
Archive size: 634.69 MB

3. INSPECTING TAR MEMBERS
Total regular files: 179260

4. EXTRACTING SELECTED RECORDS

5. EXTRACTION RESULTS
Expected: 32
Extracted: 32
Missing: 0

6. OUTPUT DIRECTORY
/content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_parp1_ligands_shard23

7. AUDIT FILE
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_shard23_extraction_audit.csv

8. GZIP VALIDATION
gzip_valid
True    32
Name: count, dtype: int64

CELL 113 COMPLETED


In [171]:
# ============================================================
# CELL 114: VALIDATE EXTRACTED PARP1 LIGAND FILES
# ============================================================

import os
import json
import gzip
import pandas as pd

from rdkit import Chem

OUTPUT_DIR = (
    "/content/drive/MyDrive/Member2_Project/member2_outputs"
)

EXTRACT_DIR = os.path.join(
    OUTPUT_DIR,
    "crossdocked_parp1_ligands_shard23"
)

MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

print("=" * 70)
print("CELL 114: EXTRACTED LIGAND VALIDATION")
print("=" * 70)

retrieval = pd.read_csv(MANIFEST_PATH)

results = []

# ------------------------------------------------------------
# 1. CHECK EACH EXTRACTED FILE
# ------------------------------------------------------------

for _, row in retrieval.iterrows():

    pair_id = int(row["pair_idx"])
    pair_name = f"{pair_id:07d}"

    sdf_path = os.path.join(
        EXTRACT_DIR,
        pair_name + ".sdf.gz"
    )

    json_path = os.path.join(
        EXTRACT_DIR,
        pair_name + ".json"
    )

    sdf_exists = os.path.isfile(sdf_path)
    json_exists = os.path.isfile(json_path)

    gzip_valid = False
    pose_count = 0
    valid_poses = 0
    invalid_poses = 0
    metadata_valid = False
    error_message = ""

    # --------------------------------------------------------
    # 2. VALIDATE JSON METADATA
    # --------------------------------------------------------

    if json_exists:
        try:
            with open(json_path, "r") as f:
                metadata = json.load(f)

            metadata_valid = isinstance(metadata, dict)

        except Exception as e:
            error_message = "JSON: " + str(e)

    # --------------------------------------------------------
    # 3. VALIDATE GZIP AND SDF WITH RDKIT
    # --------------------------------------------------------

    if sdf_exists:

        try:
            with gzip.open(sdf_path, "rb") as gz:

                # Read all compressed bytes to verify integrity
                sdf_bytes = gz.read()

            gzip_valid = len(sdf_bytes) > 0

            from io import BytesIO

            supplier = Chem.ForwardSDMolSupplier(
                BytesIO(sdf_bytes),
                sanitize=True,
                removeHs=False
            )

            for mol in supplier:

                pose_count += 1

                if mol is None:
                    invalid_poses += 1
                else:
                    valid_poses += 1

        except Exception as e:
            error_message = str(e)

    results.append({
        "pair_idx": pair_id,
        "ligand_identity": row["ligand_identity"],
        "sdf_exists": sdf_exists,
        "json_exists": json_exists,
        "gzip_valid": gzip_valid,
        "metadata_valid": metadata_valid,
        "total_poses": pose_count,
        "valid_poses": valid_poses,
        "invalid_poses": invalid_poses,
        "error": error_message
    })

# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------

audit = pd.DataFrame(results)

print("\n1. FILE VALIDATION")
print("Expected records:", len(retrieval))
print("SDF files found:", audit["sdf_exists"].sum())
print("JSON files found:", audit["json_exists"].sum())
print("Valid gzip files:", audit["gzip_valid"].sum())
print("Valid metadata files:", audit["metadata_valid"].sum())

print("\n2. MOLECULAR POSE VALIDATION")
print("Total poses:", audit["total_poses"].sum())
print("Valid RDKit poses:", audit["valid_poses"].sum())
print("Invalid poses:", audit["invalid_poses"].sum())

print("\n3. PER-LIGAND RESULTS")

display(audit)

# ------------------------------------------------------------
# 5. SAVE AUDIT
# ------------------------------------------------------------

audit_path = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_rdkit_validation.csv"
)

audit.to_csv(audit_path, index=False)

print("\n4. AUDIT SAVED")
print(audit_path)

print("\nCELL 114 COMPLETED")

CELL 114: EXTRACTED LIGAND VALIDATION

1. FILE VALIDATION
Expected records: 32
SDF files found: 32
JSON files found: 32
Valid gzip files: 32
Valid metadata files: 32

2. MOLECULAR POSE VALIDATION
Total poses: 51
Valid RDKit poses: 51
Invalid poses: 0

3. PER-LIGAND RESULTS


,pair_idx,ligand_identity,sdf_exists,json_exists,gzip_valid,metadata_valid,total_poses,valid_poses,invalid_poses,error
0,1735676,1uk0_frm,True,True,True,True,1,1,0,
1,1735677,1uk1_frq,True,True,True,True,1,1,0,
2,1735678,1wok_cnq,True,True,True,True,1,1,0,
3,1735679,2rcw_aai,True,True,True,True,1,1,0,
4,1735680,2rd6_78p,True,True,True,True,1,1,0,
5,1735681,3gjw_gjw,True,True,True,True,1,1,0,
6,1735682,3gn7_3gn,True,True,True,True,1,1,0,
7,1735683,3l3l_l3l,True,True,True,True,1,1,0,
8,1735684,3l3m_a92,True,True,True,True,1,1,0,
9,1735685,4gv7_mew,True,True,True,True,1,1,0,



4. AUDIT SAVED
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_ligand_rdkit_validation.csv

CELL 114 COMPLETED


In [172]:
# ============================================================
# CELL 115: PARP1 LIGAND CHEMICAL IDENTITY ANALYSIS
# ============================================================

import os
import json
import gzip
import pandas as pd

from io import BytesIO
from rdkit import Chem
from rdkit.Chem import inchi

OUTPUT_DIR = (
    "/content/drive/MyDrive/Member2_Project/member2_outputs"
)

EXTRACT_DIR = os.path.join(
    OUTPUT_DIR,
    "crossdocked_parp1_ligands_shard23"
)

RETRIEVAL_PATH = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

REFERENCE_PATH = os.path.join(
    OUTPUT_DIR,
    "validated_target_ligand_graphs.jsonl"
)

print("=" * 70)
print("CELL 115: CHEMICAL IDENTITY AND DUPLICATE ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD RETRIEVAL MANIFEST
# ------------------------------------------------------------

retrieval = pd.read_csv(RETRIEVAL_PATH)

# ------------------------------------------------------------
# 2. EXTRACT CHEMICAL IDENTITIES FROM ALL POSES
# ------------------------------------------------------------

pose_records = []

for _, row in retrieval.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    sdf_path = os.path.join(
        EXTRACT_DIR,
        f"{pair_id:07d}.sdf.gz"
    )

    with gzip.open(sdf_path, "rb") as gz:
        sdf_bytes = gz.read()

    supplier = Chem.ForwardSDMolSupplier(
        BytesIO(sdf_bytes),
        sanitize=True,
        removeHs=False
    )

    for pose_number, mol in enumerate(supplier, start=1):

        if mol is None:
            continue

        # Remove explicit hydrogens for identity comparison
        mol_no_h = Chem.RemoveHs(mol)

        canonical_smiles = Chem.MolToSmiles(
            mol_no_h,
            canonical=True,
            isomericSmiles=True
        )

        inchikey = inchi.MolToInchiKey(mol_no_h)

        pose_records.append({
            "pair_idx": pair_id,
            "ligand_identity": identity,
            "pose_number": pose_number,
            "canonical_smiles": canonical_smiles,
            "inchikey": inchikey,
            "heavy_atoms": mol_no_h.GetNumHeavyAtoms(),
            "bonds": mol_no_h.GetNumBonds()
        })

poses_df = pd.DataFrame(pose_records)

print("\n1. CROSSDOCKED CHEMICAL IDENTITIES")
print("Valid poses:", len(poses_df))
print("Unique ligand records:", poses_df["pair_idx"].nunique())
print("Unique canonical SMILES:", poses_df["canonical_smiles"].nunique())
print("Unique InChIKeys:", poses_df["inchikey"].nunique())

# ------------------------------------------------------------
# 3. SUMMARIZE EACH LIGAND RECORD
# ------------------------------------------------------------

ligand_summary = (
    poses_df.groupby(
        ["pair_idx", "ligand_identity"],
        as_index=False
    )
    .agg(
        pose_count=("pose_number", "count"),
        unique_smiles=("canonical_smiles", "nunique"),
        unique_inchikeys=("inchikey", "nunique"),
        canonical_smiles=("canonical_smiles", "first"),
        inchikey=("inchikey", "first")
    )
)

print("\n2. PER-LIGAND SUMMARY")
display(ligand_summary)

# ------------------------------------------------------------
# 4. LOAD EXPERIMENTAL REFERENCE LIGANDS
# ------------------------------------------------------------

if not os.path.exists(REFERENCE_PATH):
    raise FileNotFoundError(REFERENCE_PATH)

reference_records = []

with open(REFERENCE_PATH, "r") as f:

    for line_number, line in enumerate(f, start=1):

        if not line.strip():
            continue

        record = json.loads(line)

        smiles = None

        # Detect likely SMILES field
        for key in [
            "canonical_smiles",
            "smiles",
            "canonical_isomeric_smiles"
        ]:
            if isinstance(record.get(key), str):
                smiles = record[key]
                break

        if smiles is None:
            continue

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        canonical = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        reference_records.append({
            "reference_record": line_number,
            "reference_smiles": canonical,
            "reference_inchikey": inchi.MolToInchiKey(mol),
            "reference_metadata": json.dumps(record)
        })

reference_df = pd.DataFrame(reference_records)

print("\n3. EXPERIMENTAL REFERENCE LIGANDS")
print("Valid reference records:", len(reference_df))
print(
    "Unique reference structures:",
    reference_df["reference_inchikey"].nunique()
)

if reference_df.empty:
    raise ValueError(
        "No usable SMILES found in reference graph file. "
        "Inspect its schema before continuing."
    )

# ------------------------------------------------------------
# 5. COMPARE INCHIKEYS
# ------------------------------------------------------------

reference_keys = set(reference_df["reference_inchikey"])

ligand_summary["matches_experimental_reference"] = (
    ligand_summary["inchikey"].isin(reference_keys)
)

matched = ligand_summary[
    ligand_summary["matches_experimental_reference"]
]

unmatched = ligand_summary[
    ~ligand_summary["matches_experimental_reference"]
]

print("\n4. REFERENCE OVERLAP")
print("CrossDocked identities matching references:", len(matched))
print("Not matching references:", len(unmatched))

print("\nMatching identities:")
display(matched)

print("\nNon-matching identities:")
display(unmatched)

# ------------------------------------------------------------
# 6. DETECT DUPLICATES ACROSS LIGAND IDENTITIES
# ------------------------------------------------------------

duplicate_groups = (
    ligand_summary.groupby("inchikey")
    .agg(
        identities=("ligand_identity", lambda x: list(x)),
        identity_count=("ligand_identity", "nunique")
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["identity_count"] > 1
]

print("\n5. DUPLICATES BETWEEN DIFFERENT LIGAND IDENTITIES")
print("Duplicate chemical groups:", len(duplicate_groups))

display(duplicate_groups)

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

pose_output = os.path.join(
    OUTPUT_DIR,
    "parp1_crossdocked_pose_chemical_identities.csv"
)

summary_output = os.path.join(
    OUTPUT_DIR,
    "parp1_crossdocked_ligand_identity_summary.csv"
)

duplicate_output = os.path.join(
    OUTPUT_DIR,
    "parp1_crossdocked_duplicate_chemical_groups.csv"
)

poses_df.to_csv(pose_output, index=False)
ligand_summary.to_csv(summary_output, index=False)
duplicate_groups.to_csv(duplicate_output, index=False)

print("\n6. FILES SAVED")
print(pose_output)
print(summary_output)
print(duplicate_output)

print("\nCELL 115 COMPLETED")

CELL 115: CHEMICAL IDENTITY AND DUPLICATE ANALYSIS

1. CROSSDOCKED CHEMICAL IDENTITIES
Valid poses: 51
Unique ligand records: 32
Unique canonical SMILES: 32
Unique InChIKeys: 32

2. PER-LIGAND SUMMARY


,pair_idx,ligand_identity,pose_count,unique_smiles,unique_inchikeys,canonical_smiles,inchikey
0,1735676,1uk0_frm,1,1,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,LOFDUAJQRUYHBR-UHFFFAOYSA-N
1,1735677,1uk1_frq,1,1,1,O=c1[nH]c(=O)n(CCCCN2CC=C(c3ccccc3)CC2)c2cccc(...,PNPFDRCIGCUCMN-UHFFFAOYSA-N
2,1735678,1wok_cnq,1,1,1,NC(=O)c1cccc2ncc(-c3ccc(Cl)cc3)nc12,FLYGLPYJEQPCFY-UHFFFAOYSA-N
3,1735679,2rcw_aai,1,1,1,CCCN1CCC(c2nc3c(C(N)=O)cccc3[nH]2)CC1,KXSIHXHEHABEJX-UHFFFAOYSA-N
4,1735680,2rd6_78p,1,1,1,C[C@]1(c2nc3cccc(C(N)=O)c3[nH]2)CCCN1,JNAHVYVRKWKWKQ-CYBMUJFWSA-N
5,1735681,3gjw_gjw,1,1,1,O=c1[nH]c2cc(CN3CCCC3)ccc2n2cccc12,ZLYVTQGSZRQZLA-UHFFFAOYSA-N
6,1735682,3gn7_3gn,1,1,1,C[C@@]1(c2nc3cccc(C(N)=O)c3[nH]2)CCCN1,JNAHVYVRKWKWKQ-ZDUSSCGKSA-N
7,1735683,3l3l_l3l,1,1,1,NC(=O)c1cccc2c1C(=O)N(C1CCNCC1)C2,KISOFIKBJFDVKM-UHFFFAOYSA-N
8,1735684,3l3m_a92,1,1,1,NC(=O)c1cccc2[nH]c(-c3ccc([C@@H]4CCCCN4)cc3F)nc12,LGVIXHMVSRYWJL-HNNXBMFYSA-N
9,1735685,4gv7_mew,1,1,1,Cc1nc2ccccc2c(=O)[nH]1,FIEYHAAMDAPVCH-UHFFFAOYSA-N



3. EXPERIMENTAL REFERENCE LIGANDS
Valid reference records: 10
Unique reference structures: 10

4. REFERENCE OVERLAP
CrossDocked identities matching references: 5
Not matching references: 27

Matching identities:


,pair_idx,ligand_identity,pose_count,unique_smiles,unique_inchikeys,canonical_smiles,inchikey,matches_experimental_reference
4,1735680,2rd6_78p,1,1,1,C[C@]1(c2nc3cccc(C(N)=O)c3[nH]2)CCCN1,JNAHVYVRKWKWKQ-CYBMUJFWSA-N,True
24,1735704,5kpp_6wz,1,1,1,C[C@@H]1CN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc4...,WJSBPAZKQYZPAE-CQSZACIVSA-N,True
25,1735705,5kpq_6x2,1,1,1,CCCN1CCN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc43)...,LVZZNRAHJDTKAL-MRXNPFEDSA-N,True
28,1735708,5wrz_7u3,1,1,1,O=c1[nH]nc(OC[C@@H]2CCNC2)c2ccc(F)cc12,PARCVGQDXZYJOY-MRVPVSSYSA-N,True
30,1735710,5ws1_7u9,1,1,1,NC(=O)c1cccc2nc(C(=O)N3CC[C@@H](N)C3)[nH]c12,WHEDMAICUFWICE-SSDOTTSWSA-N,True



Non-matching identities:


,pair_idx,ligand_identity,pose_count,unique_smiles,unique_inchikeys,canonical_smiles,inchikey,matches_experimental_reference
0,1735676,1uk0_frm,1,1,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,LOFDUAJQRUYHBR-UHFFFAOYSA-N,False
1,1735677,1uk1_frq,1,1,1,O=c1[nH]c(=O)n(CCCCN2CC=C(c3ccccc3)CC2)c2cccc(...,PNPFDRCIGCUCMN-UHFFFAOYSA-N,False
2,1735678,1wok_cnq,1,1,1,NC(=O)c1cccc2ncc(-c3ccc(Cl)cc3)nc12,FLYGLPYJEQPCFY-UHFFFAOYSA-N,False
3,1735679,2rcw_aai,1,1,1,CCCN1CCC(c2nc3c(C(N)=O)cccc3[nH]2)CC1,KXSIHXHEHABEJX-UHFFFAOYSA-N,False
5,1735681,3gjw_gjw,1,1,1,O=c1[nH]c2cc(CN3CCCC3)ccc2n2cccc12,ZLYVTQGSZRQZLA-UHFFFAOYSA-N,False
6,1735682,3gn7_3gn,1,1,1,C[C@@]1(c2nc3cccc(C(N)=O)c3[nH]2)CCCN1,JNAHVYVRKWKWKQ-ZDUSSCGKSA-N,False
7,1735683,3l3l_l3l,1,1,1,NC(=O)c1cccc2c1C(=O)N(C1CCNCC1)C2,KISOFIKBJFDVKM-UHFFFAOYSA-N,False
8,1735684,3l3m_a92,1,1,1,NC(=O)c1cccc2[nH]c(-c3ccc([C@@H]4CCCCN4)cc3F)nc12,LGVIXHMVSRYWJL-HNNXBMFYSA-N,False
9,1735685,4gv7_mew,1,1,1,Cc1nc2ccccc2c(=O)[nH]1,FIEYHAAMDAPVCH-UHFFFAOYSA-N,False
10,1735686,4hhy_15r,20,1,1,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,CSCBNPVHZXDTDS-NDEPHWFRSA-N,False



5. DUPLICATES BETWEEN DIFFERENT LIGAND IDENTITIES
Duplicate chemical groups: 0


,inchikey,identities,identity_count



6. FILES SAVED
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_crossdocked_pose_chemical_identities.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_crossdocked_ligand_identity_summary.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_crossdocked_duplicate_chemical_groups.csv

CELL 115 COMPLETED


In [173]:
# ============================================================
# CELL 116: VALIDATE LIGAND-RECEPTOR PAIR PROVENANCE
# ============================================================

import os
import json
import pandas as pd

OUTPUT_DIR = (
    "/content/drive/MyDrive/Member2_Project/member2_outputs"
)

EXTRACT_DIR = os.path.join(
    OUTPUT_DIR,
    "crossdocked_parp1_ligands_shard23"
)

MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_retrieval_manifest.csv"
)

print("=" * 70)
print("CELL 116: LIGAND-RECEPTOR PROVENANCE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD SELECTED RECORDS
# ------------------------------------------------------------

retrieval = pd.read_csv(MANIFEST_PATH)

# ------------------------------------------------------------
# 2. READ METADATA FOR EVERY SELECTED PAIR
# ------------------------------------------------------------

records = []

for _, row in retrieval.iterrows():

    pair_id = int(row["pair_idx"])
    pair_name = f"{pair_id:07d}"

    json_path = os.path.join(
        EXTRACT_DIR,
        pair_name + ".json"
    )

    if not os.path.exists(json_path):
        records.append({
            "pair_idx": pair_id,
            "ligand_identity": row["ligand_identity"],
            "metadata_found": False
        })
        continue

    with open(json_path, "r") as f:
        metadata = json.load(f)

    records.append({
        "pair_idx": pair_id,
        "ligand_identity": row["ligand_identity"],
        "metadata_found": True,
        "manifest_receptor": row["receptor_pdb"],
        "manifest_ligand": row["ligand_sdf_gz"],
        "manifest_complex_dir": row.get("complex_dir", None),
        "metadata": json.dumps(metadata),
        "metadata_receptor_path": metadata.get("receptor_path"),
        "metadata_complex_dir": metadata.get("complex_dir"),
        "metadata_source_type": metadata.get("source_type")
    })

audit = pd.DataFrame(records)

# ------------------------------------------------------------
# 3. COMPARE RECEPTOR AND COMPLEX IDENTITIES
# ------------------------------------------------------------

def normalize_filename(value):
    if not isinstance(value, str):
        return None
    return os.path.basename(value).lower()

audit["receptor_filename"] = audit[
    "manifest_receptor"
].apply(normalize_filename)

audit["metadata_receptor_filename"] = audit[
    "metadata_receptor_path"
].apply(normalize_filename)

audit["receptor_filename_match"] = (
    audit["receptor_filename"]
    == audit["metadata_receptor_filename"]
)

audit["complex_dir_match"] = (
    audit["manifest_complex_dir"]
    == audit["metadata_complex_dir"]
)

print("\n1. METADATA AVAILABILITY")
print("Records:", len(audit))
print("Metadata found:", audit["metadata_found"].sum())

print("\n2. RECEPTOR PATH MATCHING")
print(
    "Matching receptor filenames:",
    audit["receptor_filename_match"].sum()
)

print(
    "Matching complex directories:",
    audit["complex_dir_match"].sum()
)

# ------------------------------------------------------------
# 4. DISPLAY COMPLETE PAIR AUDIT
# ------------------------------------------------------------

print("\n3. PAIR-LEVEL AUDIT")

display(
    audit[
        [
            "pair_idx",
            "ligand_identity",
            "manifest_receptor",
            "metadata_receptor_path",
            "receptor_filename_match",
            "manifest_complex_dir",
            "metadata_complex_dir",
            "complex_dir_match"
        ]
    ]
)

# ------------------------------------------------------------
# 5. SAVE AUDIT
# ------------------------------------------------------------

audit_path = os.path.join(
    OUTPUT_DIR,
    "parp1_ligand_receptor_provenance_audit.csv"
)

audit.to_csv(audit_path, index=False)

print("\n4. AUDIT SAVED")
print(audit_path)

print("\nCELL 116 COMPLETED")

CELL 116: LIGAND-RECEPTOR PROVENANCE VALIDATION

1. METADATA AVAILABILITY
Records: 32
Metadata found: 32

2. RECEPTOR PATH MATCHING
Matching receptor filenames: 32
Matching complex directories: 0

3. PAIR-LEVEL AUDIT


,pair_idx,ligand_identity,manifest_receptor,metadata_receptor_path,receptor_filename_match,manifest_complex_dir,metadata_complex_dir,complex_dir_match
0,1735676,1uk0_frm,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
1,1735677,1uk1_frq,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
2,1735678,1wok_cnq,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
3,1735679,2rcw_aai,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
4,1735680,2rd6_78p,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
5,1735681,3gjw_gjw,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
6,1735682,3gn7_3gn,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
7,1735683,3l3l_l3l,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
8,1735684,3l3m_a92,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False
9,1735685,4gv7_mew,1uk0_A_rec.pdb,PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb,True,None,PARP1_HUMAN_775_1012_0,False



4. AUDIT SAVED
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_ligand_receptor_provenance_audit.csv

CELL 116 COMPLETED


In [174]:
# ============================================================
# CELL 117: EXTRACT EXACT PARP1 RECEPTOR
# ============================================================

import os
import tarfile
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "kohbanye/crossdocked2020"

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT_DIR = ROOT / "member2_outputs" / "parp1_receptor_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RECEPTOR = "1uk0_A_rec.pdb"
RECEPTOR_PATH = OUT_DIR / TARGET_RECEPTOR

print("=" * 65)
print("CELL 117: EXTRACT EXACT PARP1 RECEPTOR")
print("=" * 65)

# Reuse an already extracted receptor
if RECEPTOR_PATH.exists() and RECEPTOR_PATH.stat().st_size > 0:
    print("Receptor already extracted:", RECEPTOR_PATH)
else:
    print("Downloading receptor archive if not cached...")

    archive_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="receptors/shard-000.tar.gz",
        repo_type="dataset"
    )

    print("Archive:", archive_path)
    print("Archive size (GB):",
          round(os.path.getsize(archive_path) / (1024**3), 3))

    print("\nSearching archive for exact receptor...")

    with tarfile.open(archive_path, mode="r:gz") as tar:
        matches = [
            member for member in tar.getmembers()
            if os.path.basename(member.name) == TARGET_RECEPTOR
            and member.isfile()
        ]

        print("Matching archive members:", len(matches))

        if len(matches) == 0:
            raise FileNotFoundError(
                f"{TARGET_RECEPTOR} was not found in receptor archive."
            )

        if len(matches) > 1:
            print("Multiple matches:")
            for member in matches:
                print(member.name)
            raise RuntimeError(
                "Multiple receptor files found. Need to resolve exact path."
            )

        member = matches[0]
        print("Selected member:", member.name)

        source = tar.extractfile(member)
        if source is None:
            raise RuntimeError("Could not read receptor archive member.")

        with open(RECEPTOR_PATH, "wb") as output:
            output.write(source.read())

print("\nRECEPTOR FILE CHECK")
print("Path:", RECEPTOR_PATH)
print("Exists:", RECEPTOR_PATH.exists())
print("Size (KB):", round(RECEPTOR_PATH.stat().st_size / 1024, 2))

if RECEPTOR_PATH.stat().st_size == 0:
    raise RuntimeError("Extracted receptor file is empty.")

print("\nCELL 117 COMPLETED")

CELL 117: EXTRACT EXACT PARP1 RECEPTOR


receptors/shard-000.tar.gz: reconstructing file:   0%|          |  0.00B / 1.84GB            

receptors/shard-000.tar.gz: downloading bytes:           |  0.00B            

Archive: /root/.cache/huggingface/hub/datasets--kohbanye--crossdocked2020/snapshots/017d8a19301c901471bba9f139331f86cd9ad278/receptors/shard-000.tar.gz
Archive size (GB): 1.712

Searching archive for exact receptor...
Matching archive members: 1
Selected member: PARP1_HUMAN_775_1012_0/1uk0_A_rec.pdb

RECEPTOR FILE CHECK
Path: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/1uk0_A_rec.pdb
Exists: True
Size (KB): 217.9

CELL 117 COMPLETED


In [175]:
# ============================================================
# CELL 118: RECEPTOR STRUCTURE AND SEQUENCE VALIDATION
# ============================================================

import os
import pandas as pd

from Bio.PDB import PDBParser, is_aa
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner

ROOT = "/content/drive/MyDrive/Member2_Project"

RECEPTOR_PATH = os.path.join(
    ROOT,
    "member2_outputs",
    "parp1_receptor_validation",
    "1uk0_A_rec.pdb"
)

M1_DIR = os.path.join(
    ROOT,
    "member1_input",
    "module1 parp1 files"
)

M1_PDB = os.path.join(M1_DIR, "conformer_001.pdb")

assert os.path.exists(RECEPTOR_PATH), "Receptor PDB missing"
assert os.path.exists(M1_PDB), "M1 reference PDB missing"

parser = PDBParser(QUIET=True)

receptor_structure = parser.get_structure(
    "crossdocked_receptor",
    RECEPTOR_PATH
)

m1_structure = parser.get_structure(
    "m1_reference",
    M1_PDB
)

# ------------------------------------------------------------
# 1. EXTRACT PROTEIN SEQUENCES
# ------------------------------------------------------------

def extract_chain_information(structure):

    information = {}

    for chain in structure[0]:

        residues = [
            residue
            for residue in chain
            if is_aa(residue, standard=False)
        ]

        sequence = "".join(
            seq1(
                residue.resname,
                custom_map={"MSE": "M"},
                undef_code="X"
            )
            for residue in residues
        )

        protein_atoms = [
            atom
            for residue in residues
            for atom in residue
            if atom.element.upper() in ["C", "N", "O", "S", "P"]
        ]

        information[chain.id] = {
            "sequence": sequence,
            "residue_count": len(residues),
            "atom_count": len(protein_atoms),
            "residues": residues
        }

    return information


receptor_chains = extract_chain_information(
    receptor_structure
)

m1_chains = extract_chain_information(
    m1_structure
)

# ------------------------------------------------------------
# 2. DISPLAY RECEPTOR STRUCTURE
# ------------------------------------------------------------

print("=" * 65)
print("CELL 118: RECEPTOR STRUCTURE AUDIT")
print("=" * 65)

print("\nRECEPTOR CHAINS")

for chain_id, data in receptor_chains.items():
    print(
        f"Chain {chain_id}: "
        f"{data['residue_count']} residues, "
        f"{data['atom_count']} protein heavy atoms"
    )
    print("Sequence:", data["sequence"][:120], "...")

print("\nM1 CHAINS")

for chain_id, data in m1_chains.items():
    print(
        f"Chain {chain_id}: "
        f"{data['residue_count']} residues, "
        f"{data['atom_count']} protein heavy atoms"
    )

# ------------------------------------------------------------
# 3. LOCAL SEQUENCE ALIGNMENT
# ------------------------------------------------------------

aligner = PairwiseAligner()
aligner.mode = "local"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -5
aligner.extend_gap_score = -0.5

comparison_rows = []

for receptor_chain, receptor_data in receptor_chains.items():

    receptor_sequence = receptor_data["sequence"]

    for m1_chain, m1_data in m1_chains.items():

        m1_sequence = m1_data["sequence"]

        alignments = aligner.align(
            m1_sequence,
            receptor_sequence
        )

        alignment = alignments[0]
        indices = alignment.indices

        paired = [
            (i, j)
            for i, j in zip(indices[0], indices[1])
            if i >= 0 and j >= 0
        ]

        aligned_length = len(paired)

        identical = sum(
            m1_sequence[i] == receptor_sequence[j]
            for i, j in paired
        )

        identity = (
            100 * identical / aligned_length
            if aligned_length else 0
        )

        receptor_coverage = (
            100 * aligned_length / len(receptor_sequence)
            if receptor_sequence else 0
        )

        m1_coverage = (
            100 * aligned_length / len(m1_sequence)
            if m1_sequence else 0
        )

        comparison_rows.append({
            "receptor_chain": receptor_chain,
            "M1_chain": m1_chain,
            "receptor_length": len(receptor_sequence),
            "M1_length": len(m1_sequence),
            "aligned_residues": aligned_length,
            "identical_residues": identical,
            "sequence_identity_percent": round(identity, 3),
            "receptor_coverage_percent": round(
                receptor_coverage, 3
            ),
            "M1_coverage_percent": round(m1_coverage, 3)
        })

comparison_df = pd.DataFrame(comparison_rows)

print("\nSEQUENCE ALIGNMENT RESULTS")

display(
    comparison_df.sort_values(
        "sequence_identity_percent",
        ascending=False
    ).reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = os.path.dirname(RECEPTOR_PATH)

comparison_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "receptor_m1_sequence_comparison.csv"
    ),
    index=False
)

print("\nSaved sequence comparison.")
print("CELL 118 COMPLETED")

CELL 118: RECEPTOR STRUCTURE AUDIT

RECEPTOR CHAINS
Chain A: 350 residues, 2754 protein heavy atoms
Sequence: KSKLPKPVQDLIKMIFDVESMKKAMVEYEIDLQKMPLGKLSKRQIQAAYSILSEVQQAVSQGSSDSQILDLSNRFYTLIPHDFGMKKPPLLNNADSVQAKVEMLDNLLDIEVAYSLLRGG ...

M1 CHAINS
Chain A: 350 residues, 2735 protein heavy atoms
Chain B: 348 residues, 2710 protein heavy atoms

SEQUENCE ALIGNMENT RESULTS


,receptor_chain,M1_chain,receptor_length,M1_length,aligned_residues,identical_residues,sequence_identity_percent,receptor_coverage_percent,M1_coverage_percent
0,A,A,350,350,350,349,99.714,100.000,100.0
1,A,B,350,348,348,347,99.713,99.429,100.0



Saved sequence comparison.
CELL 118 COMPLETED


In [176]:
# ============================================================
# CELL 119: ALIGN CROSSDOCKED RECEPTOR TO M1 REFERENCE
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from Bio.PDB import PDBParser, Superimposer
from Bio.PDB import is_aa
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner

ROOT = "/content/drive/MyDrive/Member2_Project"

RECEPTOR_PATH = os.path.join(
    ROOT, "member2_outputs",
    "parp1_receptor_validation",
    "1uk0_A_rec.pdb"
)

M1_PDB = os.path.join(
    ROOT, "member1_input",
    "module1 parp1 files",
    "conformer_001.pdb"
)

OUT_DIR = os.path.dirname(RECEPTOR_PATH)

parser = PDBParser(QUIET=True)

receptor = parser.get_structure("REC", RECEPTOR_PATH)[0]["A"]
m1 = parser.get_structure("M1", M1_PDB)[0]["A"]

# ------------------------------------------------------------
# 1. GET STANDARD AMINO-ACID RESIDUES
# ------------------------------------------------------------

def get_residues(chain):
    return [
        r for r in chain
        if is_aa(r, standard=False)
    ]

rec_res = get_residues(receptor)
m1_res = get_residues(m1)

def get_sequence(residues):
    return "".join(
        seq1(r.resname, custom_map={"MSE": "M"}, undef_code="X")
        for r in residues
    )

rec_seq = get_sequence(rec_res)
m1_seq = get_sequence(m1_res)

# ------------------------------------------------------------
# 2. MAP RESIDUES USING SEQUENCE ALIGNMENT
# ------------------------------------------------------------

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -5
aligner.extend_gap_score = -0.5

alignment = aligner.align(m1_seq, rec_seq)[0]

pairs = [
    (i, j)
    for i, j in zip(
        alignment.indices[0],
        alignment.indices[1]
    )
    if i >= 0 and j >= 0
]

# ------------------------------------------------------------
# 3. COLLECT MATCHING C-ALPHA ATOMS
# ------------------------------------------------------------

fixed_ca = []
moving_ca = []

for m1_index, rec_index in pairs:

    m1_residue = m1_res[m1_index]
    rec_residue = rec_res[rec_index]

    if "CA" in m1_residue and "CA" in rec_residue:
        fixed_ca.append(m1_residue["CA"])
        moving_ca.append(rec_residue["CA"])

print("=" * 65)
print("CELL 119: RECEPTOR TO M1 STRUCTURAL ALIGNMENT")
print("=" * 65)

print("Sequence-mapped residues:", len(pairs))
print("Common C-alpha atoms:", len(fixed_ca))

# ------------------------------------------------------------
# 4. CALCULATE C-ALPHA SUPERPOSITION
# ------------------------------------------------------------

sup = Superimposer()
sup.set_atoms(fixed_ca, moving_ca)

ca_rmsd = sup.rms

rotation, translation = sup.rotran

# ------------------------------------------------------------
# 5. CALCULATE BACKBONE RMSD USING SAME TRANSFORM
# ------------------------------------------------------------

fixed_backbone = []
moving_backbone = []

for m1_index, rec_index in pairs:

    mr = m1_res[m1_index]
    rr = rec_res[rec_index]

    for atom_name in ["N", "CA", "C", "O"]:

        if atom_name in mr and atom_name in rr:
            fixed_backbone.append(mr[atom_name].coord)
            moving_backbone.append(rr[atom_name].coord)

fixed_backbone = np.array(fixed_backbone)
moving_backbone = np.array(moving_backbone)

aligned_backbone = (
    moving_backbone @ rotation + translation
)

backbone_rmsd = np.sqrt(
    np.mean(
        np.sum(
            (fixed_backbone - aligned_backbone) ** 2,
            axis=1
        )
    )
)

# ------------------------------------------------------------
# 6. SAVE TRANSFORMATION
# ------------------------------------------------------------

transform = {
    "source_structure": "1uk0_A_rec.pdb",
    "target_structure": "M1_conformer_001.pdb",
    "source_chain": "A",
    "target_chain": "A",
    "mapped_residues": len(pairs),
    "common_ca_atoms": len(fixed_ca),
    "ca_rmsd_angstrom": float(ca_rmsd),
    "backbone_rmsd_angstrom": float(backbone_rmsd),
    "rotation": rotation.tolist(),
    "translation": translation.tolist()
}

transform_path = os.path.join(
    OUT_DIR,
    "crossdocked_receptor_to_m1_transform.json"
)

with open(transform_path, "w") as f:
    json.dump(transform, f, indent=2)

# ------------------------------------------------------------
# 7. REPORT
# ------------------------------------------------------------

print("\nSTRUCTURAL ALIGNMENT RESULTS")
print("C-alpha RMSD:", round(ca_rmsd, 4), "Å")
print("Backbone RMSD:", round(backbone_rmsd, 4), "Å")

print("\nTRANSFORMATION")
print("Rotation shape:", rotation.shape)
print("Translation:", translation)

print("\nSaved:", transform_path)

print("\nCELL 119 COMPLETED")

CELL 119: RECEPTOR TO M1 STRUCTURAL ALIGNMENT
Sequence-mapped residues: 350
Common C-alpha atoms: 350

STRUCTURAL ALIGNMENT RESULTS
C-alpha RMSD: 1.3496 Å
Backbone RMSD: 1.4027 Å

TRANSFORMATION
Rotation shape: (3, 3)
Translation: [-25.46410339  45.10188131 -37.07815514]

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_receptor_to_m1_transform.json

CELL 119 COMPLETED


In [177]:
# ============================================================
# CELL 120: LIGAND-RECEPTOR POSE GEOMETRY VALIDATION
# ============================================================

import os
import gzip
import numpy as np
import pandas as pd

from pathlib import Path
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist
from rdkit import Chem

ROOT = Path("/content/drive/MyDrive/Member2_Project")

RECEPTOR_PATH = (
    ROOT / "member2_outputs/parp1_receptor_validation/1uk0_A_rec.pdb"
)

LIGAND_DIR = (
    ROOT / "member2_outputs/crossdocked_parp1_ligands_shard23"
)

OUTPUT_DIR = ROOT / "member2_outputs/parp1_receptor_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = (
    ROOT / "member2_outputs/parp1_ligand_retrieval_manifest.csv"
)

# ------------------------------------------------------------
# 1. LOAD RECEPTOR HEAVY ATOMS
# ------------------------------------------------------------

parser = PDBParser(QUIET=True)
structure = parser.get_structure("REC", str(RECEPTOR_PATH))

receptor_atoms = [
    atom for atom in structure.get_atoms()
    if atom.element.upper() in ["C", "N", "O", "S", "P"]
]

receptor_xyz = np.array([
    atom.coord for atom in receptor_atoms
])

print("=" * 65)
print("CELL 120: LIGAND-RECEPTOR GEOMETRY VALIDATION")
print("=" * 65)

print("Receptor heavy atoms:", len(receptor_xyz))

# ------------------------------------------------------------
# 2. LOAD ALL 32 LIGAND FILES AND THEIR POSES
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_PATH)

results = []

for _, row in manifest.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    sdf_path = LIGAND_DIR / f"{pair_id:07d}.sdf.gz"

    if not sdf_path.exists():
        print("Missing:", sdf_path.name)
        continue

    with gzip.open(sdf_path, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            f,
            sanitize=True,
            removeHs=True
        )

        for pose_index, mol in enumerate(supplier):

            if mol is None or mol.GetNumConformers() == 0:
                continue

            ligand_xyz = np.asarray(
                mol.GetConformer().GetPositions()
            )

            distances = cdist(ligand_xyz, receptor_xyz)

            min_distance = float(distances.min())

            pair_counts = {
                cutoff: int(np.sum(distances <= cutoff))
                for cutoff in [2, 3, 4, 5]
            }

            ligand_atom_counts = {
                cutoff: int(
                    np.sum(np.min(distances, axis=1) <= cutoff)
                )
                for cutoff in [3, 4, 5]
            }

            results.append({
                "pair_idx": pair_id,
                "ligand_identity": identity,
                "pose_index": pose_index,
                "ligand_atoms": len(ligand_xyz),
                "min_distance_A": min_distance,
                "atom_pairs_le_2A": pair_counts[2],
                "atom_pairs_le_3A": pair_counts[3],
                "atom_pairs_le_4A": pair_counts[4],
                "atom_pairs_le_5A": pair_counts[5],
                "ligand_atoms_le_3A": ligand_atom_counts[3],
                "ligand_atoms_le_4A": ligand_atom_counts[4],
                "ligand_atoms_le_5A": ligand_atom_counts[5],
                "severe_overlap": min_distance < 2.0,
                "has_receptor_contacts": pair_counts[5] > 0
            })

# ------------------------------------------------------------
# 3. BUILD RESULTS
# ------------------------------------------------------------

pose_df = pd.DataFrame(results)

if pose_df.empty:
    raise RuntimeError("No valid ligand poses were analyzed.")

print("\n1. POSE VALIDATION SUMMARY")
print("Valid poses analyzed:", len(pose_df))
print("Unique ligand records:", pose_df["pair_idx"].nunique())
print(
    "Poses with receptor contacts:",
    pose_df["has_receptor_contacts"].sum()
)
print(
    "Poses with severe overlaps (<2 Å):",
    pose_df["severe_overlap"].sum()
)

print("\n2. DISTANCE SUMMARY")
display(
    pose_df[
        [
            "min_distance_A",
            "atom_pairs_le_2A",
            "atom_pairs_le_3A",
            "atom_pairs_le_4A",
            "atom_pairs_le_5A"
        ]
    ].describe().round(3)
)

print("\n3. PER-LIGAND SUMMARY")

ligand_summary = (
    pose_df.groupby("ligand_identity")
    .agg(
        pose_count=("pose_index", "count"),
        minimum_distance_A=("min_distance_A", "min"),
        maximum_contacts_5A=("atom_pairs_le_5A", "max"),
        poses_with_contacts=("has_receptor_contacts", "sum"),
        severe_overlap_poses=("severe_overlap", "sum")
    )
    .reset_index()
)

display(ligand_summary)

# ------------------------------------------------------------
# 4. SAVE RESULTS
# ------------------------------------------------------------

pose_path = OUTPUT_DIR / "crossdocked_pose_receptor_geometry.csv"
summary_path = OUTPUT_DIR / "crossdocked_ligand_receptor_geometry_summary.csv"

pose_df.to_csv(pose_path, index=False)
ligand_summary.to_csv(summary_path, index=False)

print("\n4. FILES SAVED")
print(pose_path)
print(summary_path)

print("\nCELL 120 COMPLETED")

CELL 120: LIGAND-RECEPTOR GEOMETRY VALIDATION
Receptor heavy atoms: 2754

1. POSE VALIDATION SUMMARY
Valid poses analyzed: 51
Unique ligand records: 32
Poses with receptor contacts: 51
Poses with severe overlaps (<2 Å): 12

2. DISTANCE SUMMARY


,min_distance_A,atom_pairs_le_2A,atom_pairs_le_3A,atom_pairs_le_4A,atom_pairs_le_5A
count,51.000,51.000,51.000,51.000,51.000
mean,2.432,1.471,12.255,110.431,381.157
std,0.656,5.944,23.906,52.601,111.477
min,0.399,0.000,0.000,60.000,216.000
25%,2.115,0.000,1.500,75.000,283.000
50%,2.811,0.000,4.000,103.000,386.000
75%,2.890,0.000,15.500,125.000,436.000
max,3.139,41.000,148.000,372.000,790.000



3. PER-LIGAND SUMMARY


,ligand_identity,pose_count,minimum_distance_A,maximum_contacts_5A,poses_with_contacts,severe_overlap_poses
0,1uk0_frm,1,2.811002,371,1,0
1,1uk1_frq,1,2.802332,372,1,0
2,1wok_cnq,1,2.958304,262,1,0
3,2rcw_aai,1,2.686815,323,1,0
4,2rd6_78p,1,2.899166,250,1,0
5,3gjw_gjw,1,2.830066,264,1,0
6,3gn7_3gn,1,2.899287,246,1,0
7,3l3l_l3l,1,2.951474,268,1,0
8,3l3m_a92,1,3.022459,306,1,0
9,4gv7_mew,1,2.897480,219,1,0



4. FILES SAVED
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_pose_receptor_geometry.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_ligand_receptor_geometry_summary.csv

CELL 120 COMPLETED


In [178]:
# ============================================================
# CELL 121: TRANSFORM LIGAND POSES INTO M1 COORDINATE FRAME
# ============================================================

import os
import json
import gzip
import numpy as np
import pandas as pd

from pathlib import Path
from io import BytesIO
from scipy.spatial.distance import cdist
from Bio.PDB import PDBParser
from rdkit import Chem

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUTPUT_DIR = (
    ROOT / "member2_outputs/parp1_receptor_validation"
)

LIGAND_DIR = (
    ROOT / "member2_outputs/crossdocked_parp1_ligands_shard23"
)

MANIFEST_PATH = (
    ROOT / "member2_outputs/parp1_ligand_retrieval_manifest.csv"
)

RECEPTOR_PATH = OUTPUT_DIR / "1uk0_A_rec.pdb"

M1_PDB = (
    ROOT / "member1_input/module1 parp1 files/conformer_001.pdb"
)

M1_TENSOR = (
    ROOT / "member1_input/module1 parp1 files/"
    "pocket_1_pharmacophore(1).npy"
)

TRANSFORM_PATH = (
    OUTPUT_DIR / "crossdocked_receptor_to_m1_transform.json"
)

# ------------------------------------------------------------
# 1. LOAD TRANSFORMATION
# ------------------------------------------------------------

with open(TRANSFORM_PATH) as f:
    transform = json.load(f)

rotation = np.asarray(transform["rotation"])
translation = np.asarray(transform["translation"])

# ------------------------------------------------------------
# 2. LOAD M1 PROTEIN HEAVY ATOMS
# ------------------------------------------------------------

parser = PDBParser(QUIET=True)
m1_structure = parser.get_structure("M1", str(M1_PDB))

m1_chain = m1_structure[0]["A"]

m1_atoms = [
    atom for atom in m1_chain.get_atoms()
    if atom.element.upper() in ["C", "N", "O", "S", "P"]
]

m1_xyz = np.array([atom.coord for atom in m1_atoms])

# ------------------------------------------------------------
# 3. IDENTIFY M1 POCKET ATOMS
# ------------------------------------------------------------

pocket_tensor = np.load(M1_TENSOR)

if pocket_tensor.ndim != 3 or pocket_tensor.shape[2] < 4:
    raise ValueError("Unexpected M1 pocket tensor format.")

# First conformer: XYZ coordinates
pocket_feature_xyz = pocket_tensor[0, :, :3]

# Identify protein atoms within 8 Å of at least one
# M1 pharmacophore feature coordinate.
atom_feature_distances = cdist(
    m1_xyz,
    pocket_feature_xyz
)

pocket_atom_mask = (
    atom_feature_distances.min(axis=1) <= 8.0
)

pocket_xyz = m1_xyz[pocket_atom_mask]

print("=" * 65)
print("CELL 121: LIGAND POSES IN M1 COORDINATE FRAME")
print("=" * 65)

print("M1 protein heavy atoms:", len(m1_xyz))
print("M1 pocket-region atoms:", len(pocket_xyz))
print("Pocket features:", len(pocket_feature_xyz))

# ------------------------------------------------------------
# 4. LOAD MANIFEST AND TRANSFORM ALL LIGAND POSES
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_PATH)

transformed_dir = OUTPUT_DIR / "transformed_ligand_poses"
transformed_dir.mkdir(parents=True, exist_ok=True)

pose_results = []

for _, row in manifest.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    sdf_path = LIGAND_DIR / f"{pair_id:07d}.sdf.gz"

    with gzip.open(sdf_path, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            BytesIO(f.read()),
            sanitize=True,
            removeHs=True
        )

        pose_molecules = list(supplier)

    output_sdf = transformed_dir / f"{pair_id:07d}_M1.sdf"

    writer = Chem.SDWriter(str(output_sdf))

    for pose_index, mol in enumerate(pose_molecules):

        if mol is None or mol.GetNumConformers() == 0:
            continue

        original_xyz = np.asarray(
            mol.GetConformer().GetPositions()
        )

        # Receptor frame -> M1 frame
        transformed_xyz = (
            original_xyz @ rotation + translation
        )

        # Verify rigid transformation preserves internal distances
        internal_before = cdist(original_xyz, original_xyz)
        internal_after = cdist(transformed_xyz, transformed_xyz)

        internal_error = float(
            np.max(np.abs(internal_before - internal_after))
        )

        # Verify receptor-frame and M1-frame distances
        # using the receptor transformation.
        receptor_parser = PDBParser(QUIET=True)
        receptor_structure = receptor_parser.get_structure(
            "REC", str(RECEPTOR_PATH)
        )

        receptor_atoms = [
            atom for atom in receptor_structure.get_atoms()
            if atom.element.upper() in ["C", "N", "O", "S", "P"]
        ]

        receptor_xyz = np.array([
            atom.coord for atom in receptor_atoms
        ])

        original_distances = cdist(
            original_xyz, receptor_xyz
        )

        transformed_receptor_xyz = (
            receptor_xyz @ rotation + translation
        )

        transformed_receptor_distances = cdist(
            transformed_xyz, transformed_receptor_xyz
        )

        distance_invariance_error = float(
            np.max(
                np.abs(
                    original_distances -
                    transformed_receptor_distances
                )
            )
        )

        # Contacts against M1 protein
        m1_distances = cdist(transformed_xyz, m1_xyz)

        # Contacts against M1 pocket-region atoms
        pocket_distances = cdist(transformed_xyz, pocket_xyz)

        min_m1_distance = float(m1_distances.min())
        min_pocket_distance = float(pocket_distances.min())

        # Update molecule coordinates
        conformer = mol.GetConformer()

        for atom_index, xyz in enumerate(transformed_xyz):
            conformer.SetAtomPosition(atom_index, xyz.tolist())

        mol.SetProp("PAIR_ID", str(pair_id))
        mol.SetProp("LIGAND_IDENTITY", identity)
        mol.SetProp("POSE_INDEX", str(pose_index))
        mol.SetProp("COORDINATE_FRAME", "M1_CONFORMER_001")

        writer.write(mol)

        pose_results.append({
            "pair_idx": pair_id,
            "ligand_identity": identity,
            "pose_index": pose_index,
            "ligand_atoms": len(original_xyz),
            "min_M1_protein_distance_A": min_m1_distance,
            "min_M1_pocket_distance_A": min_pocket_distance,
            "M1_pocket_pairs_le_4A": int(
                np.sum(pocket_distances <= 4.0)
            ),
            "M1_pocket_pairs_le_5A": int(
                np.sum(pocket_distances <= 5.0)
            ),
            "M1_pocket_ligand_atoms_le_5A": int(
                np.sum(np.min(pocket_distances, axis=1) <= 5.0)
            ),
            "internal_distance_error_A": internal_error,
            "receptor_distance_invariance_error_A":
                distance_invariance_error
        })

    writer.close()

# ------------------------------------------------------------
# 5. VALIDATE AND SAVE
# ------------------------------------------------------------

results_df = pd.DataFrame(pose_results)

if len(results_df) != 51:
    raise RuntimeError(
        f"Expected 51 transformed poses; got {len(results_df)}"
    )

results_path = (
    OUTPUT_DIR / "crossdocked_ligands_M1_frame_geometry.csv"
)

results_df.to_csv(results_path, index=False)

print("\nTRANSFORMATION SUMMARY")
print("Transformed poses:", len(results_df))
print(
    "Maximum internal-distance error:",
    results_df["internal_distance_error_A"].max()
)
print(
    "Maximum receptor-distance invariance error:",
    results_df["receptor_distance_invariance_error_A"].max()
)

print("\nM1 POCKET CONTACT SUMMARY")
display(
    results_df[
        [
            "ligand_identity",
            "pose_index",
            "min_M1_pocket_distance_A",
            "M1_pocket_pairs_le_4A",
            "M1_pocket_pairs_le_5A"
        ]
    ].round(3)
)

print("\nSAVED TRANSFORMED POSES:")
print(transformed_dir)

print("\nSAVED GEOMETRY REPORT:")
print(results_path)

print("\nCELL 121 COMPLETED")

CELL 121: LIGAND POSES IN M1 COORDINATE FRAME
M1 protein heavy atoms: 2735
M1 pocket-region atoms: 523
Pocket features: 95

TRANSFORMATION SUMMARY
Transformed poses: 51
Maximum internal-distance error: 1.7763568394002505e-14
Maximum receptor-distance invariance error: 4.263256414560601e-14

M1 POCKET CONTACT SUMMARY


,ligand_identity,pose_index,min_M1_pocket_distance_A,M1_pocket_pairs_le_4A,M1_pocket_pairs_le_5A
0,1uk0_frm,0,2.653,76,246
1,1uk1_frq,0,2.750,81,259
2,1wok_cnq,0,2.600,77,256
3,2rcw_aai,0,2.378,89,276
4,2rd6_78p,0,2.613,76,248
5,3gjw_gjw,0,2.691,73,253
6,3gn7_3gn,0,2.744,73,255
7,3l3l_l3l,0,2.715,78,251
8,3l3m_a92,0,2.437,71,282
9,4gv7_mew,0,2.847,70,220



SAVED TRANSFORMED POSES:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/transformed_ligand_poses

SAVED GEOMETRY REPORT:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_ligands_M1_frame_geometry.csv

CELL 121 COMPLETED


In [179]:
# ============================================================
# CELL 122: LOCAL PARP1 POCKET ALIGNMENT VALIDATION
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from Bio.PDB import PDBParser, is_aa
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner
from scipy.spatial.distance import cdist

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"

receptor_path = OUT / "1uk0_A_rec.pdb"
m1_path = ROOT / "member1_input/module1 parp1 files/conformer_001.pdb"

# Load transformation
import json

with open(OUT / "crossdocked_receptor_to_m1_transform.json") as f:
    transform = json.load(f)

R = np.asarray(transform["rotation"])
t = np.asarray(transform["translation"])

parser = PDBParser(QUIET=True)

rec = parser.get_structure("REC", str(receptor_path))[0]["A"]
m1 = parser.get_structure("M1", str(m1_path))[0]["A"]

# Extract standard amino-acid residues
def get_residues(chain):
    return [
        r for r in chain
        if is_aa(r, standard=True)
    ]

rec_res = get_residues(rec)
m1_res = get_residues(m1)

rec_seq = "".join(seq1(r.resname) for r in rec_res)
m1_seq = "".join(seq1(r.resname) for r in m1_res)

# Sequence alignment
aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -5
aligner.extend_gap_score = -0.5

alignment = aligner.align(rec_seq, m1_seq)[0]

rec_indices, m1_indices = alignment.indices

matched = []

for ri, mi in zip(rec_indices, m1_indices):

    if ri < 0 or mi < 0:
        continue

    rr = rec_res[ri]
    mr = m1_res[mi]

    # Only compare matching residue identities
    if rr.resname != mr.resname:
        continue

    for atom_name in ["N", "CA", "C", "O", "CB"]:

        if atom_name not in rr or atom_name not in mr:
            continue

        rec_xyz = np.asarray(rr[atom_name].coord)
        m1_xyz = np.asarray(mr[atom_name].coord)

        transformed_rec_xyz = rec_xyz @ R + t

        matched.append({
            "residue_number": mr.id[1],
            "residue_name": mr.resname,
            "atom_name": atom_name,
            "receptor_x": transformed_rec_xyz[0],
            "receptor_y": transformed_rec_xyz[1],
            "receptor_z": transformed_rec_xyz[2],
            "m1_x": m1_xyz[0],
            "m1_y": m1_xyz[1],
            "m1_z": m1_xyz[2],
            "distance_A": np.linalg.norm(
                transformed_rec_xyz - m1_xyz
            )
        })

df = pd.DataFrame(matched)

# Define M1 pocket residues from the 8 A feature region
pocket_tensor = np.load(
    ROOT / "member1_input/module1 parp1 files/"
    "pocket_1_pharmacophore(1).npy"
)

feature_xyz = pocket_tensor[0, :, :3]

pocket_residue_numbers = set()

for residue in m1_res:

    heavy_atoms = [
        a for a in residue
        if a.element.upper() in ["C", "N", "O", "S", "P"]
    ]

    if not heavy_atoms:
        continue

    coords = np.array([a.coord for a in heavy_atoms])

    if cdist(coords, feature_xyz).min() <= 8.0:
        pocket_residue_numbers.add(residue.id[1])

df["is_pocket_residue"] = (
    df["residue_number"].isin(pocket_residue_numbers)
)

# Summary
pocket_df = df[df["is_pocket_residue"]]
global_df = df

print("=" * 65)
print("CELL 122: LOCAL POCKET ALIGNMENT")
print("=" * 65)

print("Matched backbone/side-chain atoms:", len(df))
print("Matched pocket atoms:", len(pocket_df))

print("\nGLOBAL MATCHED ATOM RMSD:")
print(np.sqrt(np.mean(global_df["distance_A"] ** 2)))

print("\nLOCAL POCKET MATCHED ATOM RMSD:")
print(np.sqrt(np.mean(pocket_df["distance_A"] ** 2)))

print("\nLOCAL POCKET DISTANCE DISTRIBUTION:")
print(pocket_df["distance_A"].describe().round(3))

print("\nPOCKET RESIDUE ALIGNMENT:")
display(
    pocket_df.groupby(
        ["residue_number", "residue_name"]
    )["distance_A"]
    .agg(["count", "mean", "max"])
    .round(3)
    .sort_values("mean")
)

# Save
output_csv = OUT / "local_pocket_alignment_audit.csv"
df.to_csv(output_csv, index=False)

print("\nSaved:", output_csv)
print("\nCELL 122 COMPLETED")

CELL 122: LOCAL POCKET ALIGNMENT
Matched backbone/side-chain atoms: 1722
Matched pocket atoms: 464

GLOBAL MATCHED ATOM RMSD:
1.4278044273047847

LOCAL POCKET MATCHED ATOM RMSD:
0.921627195528951

LOCAL POCKET DISTANCE DISTRIBUTION:
count    464.000
mean       0.710
std        0.588
min        0.092
25%        0.349
50%        0.534
75%        0.875
max        4.862
Name: distance_A, dtype: float64

POCKET RESIDUE ALIGNMENT:


,,count,mean,max
residue_number,residue_name,,,
990,ILE,5,0.205,0.351
896,TYR,5,0.209,0.284
905,ALA,5,0.218,0.387
911,SER,5,0.261,0.349
900,MET,5,0.264,0.315
...,...,...,...,...
977,SER,5,1.821,4.180
828,ALA,5,1.989,4.076
827,ASN,5,2.005,2.776



Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/local_pocket_alignment_audit.csv

CELL 122 COMPLETED


In [180]:
# ============================================================
# CELL 123: ROBUST LOCAL POCKET TRANSFORMATION
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs/parp1_receptor_validation"

df = pd.read_csv(OUT / "local_pocket_alignment_audit.csv")

# Use only matched pocket atoms
pocket = df[df["is_pocket_residue"] == True].copy()

X = pocket[
    ["receptor_x", "receptor_y", "receptor_z"]
].to_numpy()

Y = pocket[
    ["m1_x", "m1_y", "m1_z"]
].to_numpy()

def kabsch(source, target):
    source_center = source.mean(axis=0)
    target_center = target.mean(axis=0)

    Xc = source - source_center
    Yc = target - target_center

    U, S, Vt = np.linalg.svd(Xc.T @ Yc)

    correction = np.eye(3)
    correction[-1, -1] = np.linalg.det(U @ Vt)

    R = U @ correction @ Vt
    t = target_center - source_center @ R

    return R, t

# Initial local fit
R, t = kabsch(X, Y)

distances = np.linalg.norm(X @ R + t - Y, axis=1)

# Iteratively trim large residuals
keep = np.ones(len(X), dtype=bool)

for iteration in range(10):
    R, t = kabsch(X[keep], Y[keep])

    distances = np.linalg.norm(X @ R + t - Y, axis=1)

    new_keep = distances <= 2.0

    if new_keep.sum() < 20:
        raise RuntimeError("Too few atoms remain after trimming.")

    if np.array_equal(new_keep, keep):
        break

    keep = new_keep

# Final fit
R, t = kabsch(X[keep], Y[keep])

final_distances = np.linalg.norm(X @ R + t - Y, axis=1)

# Metrics
local_rmsd = np.sqrt(np.mean(final_distances[keep] ** 2))
all_atom_rmsd = np.sqrt(np.mean(final_distances ** 2))

print("=" * 65)
print("CELL 123: REFINED LOCAL POCKET ALIGNMENT")
print("=" * 65)

print("Pocket atom correspondences:", len(X))
print("Atoms retained:", int(keep.sum()))
print("Atoms excluded:", int((~keep).sum()))

print("\nREFINED ALIGNMENT")
print("Inlier RMSD:", round(local_rmsd, 4), "Å")
print("All-pocket RMSD:", round(all_atom_rmsd, 4), "Å")
print("Median deviation:", round(float(np.median(final_distances)), 4), "Å")
print("Maximum deviation:", round(float(final_distances.max()), 4), "Å")

print("\nRESIDUAL DISTRIBUTION")
print(pd.Series(final_distances).describe().round(3))

# Save separately; preserve original global transform
result = {
    "method": "Kabsch local pocket fit with iterative 2A trimming",
    "rotation": R.tolist(),
    "translation": t.tolist(),
    "pocket_atom_correspondences": int(len(X)),
    "retained_atoms": int(keep.sum()),
    "excluded_atoms": int((~keep).sum()),
    "inlier_rmsd_A": float(local_rmsd),
    "all_pocket_rmsd_A": float(all_atom_rmsd),
    "median_deviation_A": float(np.median(final_distances)),
    "maximum_deviation_A": float(final_distances.max())
}

path = OUT / "crossdocked_receptor_to_m1_local_transform.json"

with open(path, "w") as f:
    json.dump(result, f, indent=2)

print("\nSaved:", path)
print("\nCELL 123 COMPLETED")

CELL 123: REFINED LOCAL POCKET ALIGNMENT
Pocket atom correspondences: 464
Atoms retained: 453
Atoms excluded: 11

REFINED ALIGNMENT
Inlier RMSD: 0.6847 Å
All-pocket RMSD: 0.8328 Å
Median deviation: 0.4639 Å
Maximum deviation: 4.3589 Å

RESIDUAL DISTRIBUTION
count    464.000
mean       0.624
std        0.552
min        0.049
25%        0.292
50%        0.464
75%        0.766
max        4.359
dtype: float64

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_receptor_to_m1_local_transform.json

CELL 123 COMPLETED


In [181]:
# ============================================================
# CELL 124: GLOBAL VS LOCAL LIGAND PLACEMENT COMPARISON
# ============================================================

import gzip
import json
import numpy as np
import pandas as pd

from pathlib import Path
from io import BytesIO
from scipy.spatial.distance import cdist
from rdkit import Chem

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"
LIGAND_DIR = ROOT / "member2_outputs/crossdocked_parp1_ligands_shard23"

MANIFEST_PATH = ROOT / "member2_outputs/parp1_ligand_retrieval_manifest.csv"
TENSOR_PATH = ROOT / "member1_input/module1 parp1 files/pocket_1_pharmacophore(1).npy"

# ------------------------------------------------------------
# 1. LOAD BOTH TRANSFORMATIONS
# ------------------------------------------------------------

with open(OUT / "crossdocked_receptor_to_m1_transform.json") as f:
    global_transform = json.load(f)

with open(OUT / "crossdocked_receptor_to_m1_local_transform.json") as f:
    local_transform = json.load(f)

Rg = np.asarray(global_transform["rotation"])
tg = np.asarray(global_transform["translation"])

Rl = np.asarray(local_transform["rotation"])
tl = np.asarray(local_transform["translation"])

# ------------------------------------------------------------
# 2. LOAD M1 POCKET COORDINATES
# ------------------------------------------------------------

pocket_tensor = np.load(TENSOR_PATH)
pocket_xyz = pocket_tensor[0, :, :3]

# Load M1 conformer 001 protein atoms
from Bio.PDB import PDBParser

m1_path = ROOT / "member1_input/module1 parp1 files/conformer_001.pdb"

structure = PDBParser(QUIET=True).get_structure("M1", str(m1_path))
chain = structure[0]["A"]

m1_atoms = [
    atom for atom in chain.get_atoms()
    if atom.element.upper() in ["C", "N", "O", "S", "P"]
]

m1_xyz = np.asarray([atom.coord for atom in m1_atoms])

# Same operational 8 A pocket-region definition as Cell 121
pocket_mask = cdist(m1_xyz, pocket_xyz).min(axis=1) <= 8.0
m1_pocket_atoms = m1_xyz[pocket_mask]

print("M1 pocket-region atoms:", len(m1_pocket_atoms))

# ------------------------------------------------------------
# 3. LOAD LIGAND POSES AND COMPARE TRANSFORMATIONS
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_PATH)

records = []

for _, row in manifest.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    sdf_path = LIGAND_DIR / f"{pair_id:07d}.sdf.gz"

    with gzip.open(sdf_path, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            BytesIO(f.read()),
            sanitize=True,
            removeHs=True
        )

        molecules = list(supplier)

    for pose_index, mol in enumerate(molecules):

        if mol is None or mol.GetNumConformers() == 0:
            continue

        original_xyz = np.asarray(
            mol.GetConformer().GetPositions()
        )

        global_xyz = original_xyz @ Rg + tg
        local_xyz = original_xyz @ Rl + tl

        global_dist = cdist(global_xyz, m1_pocket_atoms)
        local_dist = cdist(local_xyz, m1_pocket_atoms)

        records.append({
            "pair_idx": pair_id,
            "ligand_identity": identity,
            "pose_index": pose_index,
            "ligand_atoms": len(original_xyz),

            "global_min_distance_A": float(global_dist.min()),
            "local_min_distance_A": float(local_dist.min()),

            "global_pairs_le_4A": int((global_dist <= 4).sum()),
            "local_pairs_le_4A": int((local_dist <= 4).sum()),

            "global_pairs_le_5A": int((global_dist <= 5).sum()),
            "local_pairs_le_5A": int((local_dist <= 5).sum()),

            "global_ligand_atoms_le_5A": int(
                (global_dist.min(axis=1) <= 5).sum()
            ),
            "local_ligand_atoms_le_5A": int(
                (local_dist.min(axis=1) <= 5).sum()
            ),

            "mean_coordinate_shift_A": float(
                np.linalg.norm(global_xyz - local_xyz, axis=1).mean()
            ),

            "max_coordinate_shift_A": float(
                np.linalg.norm(global_xyz - local_xyz, axis=1).max()
            )
        })

# ------------------------------------------------------------
# 4. RESULTS
# ------------------------------------------------------------

comparison = pd.DataFrame(records)

if len(comparison) != 51:
    raise RuntimeError(
        f"Expected 51 poses, found {len(comparison)}"
    )

print("\n" + "=" * 65)
print("GLOBAL VS LOCAL TRANSFORMATION")
print("=" * 65)

print("Poses compared:", len(comparison))

print("\nMEAN COMPARISON")

metrics = [
    "global_min_distance_A",
    "local_min_distance_A",
    "global_pairs_le_4A",
    "local_pairs_le_4A",
    "global_pairs_le_5A",
    "local_pairs_le_5A",
    "mean_coordinate_shift_A"
]

display(comparison[metrics].mean().to_frame("mean").round(3))

print("\nPER-POSE COMPARISON")

display(
    comparison[
        [
            "ligand_identity",
            "pose_index",
            "global_min_distance_A",
            "local_min_distance_A",
            "global_pairs_le_4A",
            "local_pairs_le_4A",
            "mean_coordinate_shift_A"
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 5. SAVE
# ------------------------------------------------------------

output_path = OUT / "global_vs_local_ligand_pose_comparison.csv"

comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("\nCELL 124 COMPLETED")

M1 pocket-region atoms: 523

GLOBAL VS LOCAL TRANSFORMATION
Poses compared: 51

MEAN COMPARISON


,mean
global_min_distance_A,2.385
local_min_distance_A,72.996
global_pairs_le_4A,81.157
local_pairs_le_4A,0.000
global_pairs_le_5A,258.039
local_pairs_le_5A,0.000
mean_coordinate_shift_A,94.465



PER-POSE COMPARISON


,ligand_identity,pose_index,global_min_distance_A,local_min_distance_A,global_pairs_le_4A,local_pairs_le_4A,mean_coordinate_shift_A
0,1uk0_frm,0,2.653,73.110,76,0,93.127
1,1uk1_frq,0,2.750,73.059,81,0,94.580
2,1wok_cnq,0,2.600,73.265,77,0,94.236
3,2rcw_aai,0,2.378,73.415,89,0,96.410
4,2rd6_78p,0,2.613,73.371,76,0,96.216
5,3gjw_gjw,0,2.691,73.159,73,0,93.251
6,3gn7_3gn,0,2.744,73.375,73,0,95.737
7,3l3l_l3l,0,2.715,73.698,78,0,97.210
8,3l3m_a92,0,2.437,73.646,71,0,99.228
9,4gv7_mew,0,2.847,73.344,70,0,95.362



Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/global_vs_local_ligand_pose_comparison.csv

CELL 124 COMPLETED


In [182]:
# ============================================================
# CELL 125: COMPOSE GLOBAL + LOCAL TRANSFORMATIONS
# ============================================================

import json
import gzip
import numpy as np
import pandas as pd

from pathlib import Path
from io import BytesIO
from scipy.spatial.distance import cdist
from rdkit import Chem
from Bio.PDB import PDBParser

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs/parp1_receptor_validation"
LIGAND_DIR = ROOT / "member2_outputs/crossdocked_parp1_ligands_shard23"

# ------------------------------------------------------------
# 1. LOAD TRANSFORMATIONS
# ------------------------------------------------------------

with open(OUT / "crossdocked_receptor_to_m1_transform.json") as f:
    global_T = json.load(f)

with open(OUT / "crossdocked_receptor_to_m1_local_transform.json") as f:
    local_T = json.load(f)

Rg = np.asarray(global_T["rotation"])
tg = np.asarray(global_T["translation"])

Rl = np.asarray(local_T["rotation"])
tl = np.asarray(local_T["translation"])

# Compose row-vector transformations
R_final = Rg @ Rl
t_final = tg @ Rl + tl

print("Transformation determinant:", np.linalg.det(R_final))
print("Orthogonality error:", np.max(
    np.abs(R_final.T @ R_final - np.eye(3))
))

# ------------------------------------------------------------
# 2. LOAD M1 POCKET
# ------------------------------------------------------------

tensor_path = (
    ROOT / "member1_input/module1 parp1 files/"
    "pocket_1_pharmacophore(1).npy"
)

pdb_path = (
    ROOT / "member1_input/module1 parp1 files/"
    "conformer_001.pdb"
)

tensor = np.load(tensor_path)
feature_xyz = tensor[0, :, :3]

structure = PDBParser(QUIET=True).get_structure(
    "M1", str(pdb_path)
)

chain = structure[0]["A"]

atoms = [
    a for a in chain.get_atoms()
    if a.element.upper() in ["C", "N", "O", "S", "P"]
]

m1_xyz = np.array([a.coord for a in atoms])

pocket_mask = cdist(m1_xyz, feature_xyz).min(axis=1) <= 8.0
pocket_xyz = m1_xyz[pocket_mask]

# ------------------------------------------------------------
# 3. LOAD LIGANDS AND APPLY COMPOSED TRANSFORMATION
# ------------------------------------------------------------

manifest = pd.read_csv(
    ROOT / "member2_outputs/parp1_ligand_retrieval_manifest.csv"
)

results = []

for _, row in manifest.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    sdf_path = LIGAND_DIR / f"{pair_id:07d}.sdf.gz"

    with gzip.open(sdf_path, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            BytesIO(f.read()),
            sanitize=True,
            removeHs=True
        )
        molecules = list(supplier)

    for pose_index, mol in enumerate(molecules):

        if mol is None or mol.GetNumConformers() == 0:
            continue

        xyz = np.asarray(mol.GetConformer().GetPositions())

        global_xyz = xyz @ Rg + tg
        final_xyz = xyz @ R_final + t_final

        distances = cdist(final_xyz, pocket_xyz)

        results.append({
            "pair_idx": pair_id,
            "ligand_identity": identity,
            "pose_index": pose_index,
            "min_pocket_distance_A": float(distances.min()),
            "pocket_pairs_le_4A": int((distances <= 4).sum()),
            "pocket_pairs_le_5A": int((distances <= 5).sum()),
            "ligand_atoms_le_5A": int(
                (distances.min(axis=1) <= 5).sum()
            ),
            "mean_shift_from_global_A": float(
                np.linalg.norm(final_xyz - global_xyz, axis=1).mean()
            ),
            "max_shift_from_global_A": float(
                np.linalg.norm(final_xyz - global_xyz, axis=1).max()
            )
        })

results_df = pd.DataFrame(results)

if len(results_df) != 51:
    raise RuntimeError(
        f"Expected 51 poses; found {len(results_df)}"
    )

# ------------------------------------------------------------
# 4. SAVE COMPOSED TRANSFORMATION
# ------------------------------------------------------------

final_transform = {
    "method": "Global receptor-to-M1 transform composed with local pocket correction",
    "rotation": R_final.tolist(),
    "translation": t_final.tolist(),
    "global_rmsd_A": global_T.get("global_rmsd"),
    "local_inlier_rmsd_A": local_T["inlier_rmsd_A"],
    "local_all_pocket_rmsd_A": local_T["all_pocket_rmsd_A"]
}

with open(OUT / "crossdocked_receptor_to_m1_composed_transform.json", "w") as f:
    json.dump(final_transform, f, indent=2)

results_df.to_csv(
    OUT / "composed_transform_ligand_geometry.csv",
    index=False
)

# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 125: COMPOSED TRANSFORMATION RESULTS")
print("=" * 65)

print("Poses transformed:", len(results_df))

display(
    results_df[
        [
            "ligand_identity",
            "pose_index",
            "min_pocket_distance_A",
            "pocket_pairs_le_4A",
            "pocket_pairs_le_5A",
            "mean_shift_from_global_A"
        ]
    ].round(3)
)

print("\nMEAN METRICS")
display(results_df.select_dtypes(include="number").mean().to_frame("mean").round(3))

print("\nSaved composed transformation:")
print(OUT / "crossdocked_receptor_to_m1_composed_transform.json")

print("\nCELL 125 COMPLETED")

Transformation determinant: 1.0000000000000016
Orthogonality error: 1.9984014443252818e-15

CELL 125: COMPOSED TRANSFORMATION RESULTS
Poses transformed: 51


,ligand_identity,pose_index,min_pocket_distance_A,pocket_pairs_le_4A,pocket_pairs_le_5A,mean_shift_from_global_A
0,1uk0_frm,0,2.901,75,250,0.229
1,1uk1_frq,0,2.775,85,265,0.240
2,1wok_cnq,0,2.633,79,251,0.229
3,2rcw_aai,0,2.630,86,274,0.263
4,2rd6_78p,0,2.843,78,252,0.244
5,3gjw_gjw,0,2.629,73,257,0.219
6,3gn7_3gn,0,2.712,73,255,0.239
7,3l3l_l3l,0,2.862,76,266,0.262
8,3l3m_a92,0,2.720,85,305,0.307
9,4gv7_mew,0,2.789,73,230,0.218



MEAN METRICS


,mean
pair_idx,1735691.098
pose_index,3.725
min_pocket_distance_A,2.472
pocket_pairs_le_4A,83.118
pocket_pairs_le_5A,263.961
ligand_atoms_le_5A,25.588
mean_shift_from_global_A,0.256
max_shift_from_global_A,0.334



Saved composed transformation:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_receptor_to_m1_composed_transform.json

CELL 125 COMPLETED


In [183]:
# ============================================================
# CELL 126: FINAL COMPOSED TRANSFORMATION VALIDATION
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.spatial.distance import cdist

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs/parp1_receptor_validation"

# ------------------------------------------------------------
# 1. LOAD AUDIT AND COMPOSED TRANSFORMATION
# ------------------------------------------------------------

audit = pd.read_csv(
    OUT / "local_pocket_alignment_audit.csv"
)

with open(
    OUT / "crossdocked_receptor_to_m1_composed_transform.json"
) as f:
    transform = json.load(f)

R = np.asarray(transform["rotation"])
t = np.asarray(transform["translation"])

# Original audit receptor coordinates were already
# transformed using the global transformation.
# Apply only the local correction to those coordinates.

pocket = audit[
    audit["is_pocket_residue"] == True
].copy()

X = pocket[
    ["receptor_x", "receptor_y", "receptor_z"]
].to_numpy()

Y = pocket[
    ["m1_x", "m1_y", "m1_z"]
].to_numpy()

# Recover the local correction from the composed transform
with open(
    OUT / "crossdocked_receptor_to_m1_local_transform.json"
) as f:
    local_transform = json.load(f)

Rl = np.asarray(local_transform["rotation"])
tl = np.asarray(local_transform["translation"])

corrected_xyz = X @ Rl + tl

residuals = np.linalg.norm(corrected_xyz - Y, axis=1)

# ------------------------------------------------------------
# 2. VALIDATION METRICS
# ------------------------------------------------------------

inlier_mask = residuals <= 2.0

inlier_rmsd = np.sqrt(
    np.mean(residuals[inlier_mask] ** 2)
)

all_rmsd = np.sqrt(
    np.mean(residuals ** 2)
)

median_error = np.median(residuals)
p95_error = np.percentile(residuals, 95)

# ------------------------------------------------------------
# 3. ROTATION VALIDATION
# ------------------------------------------------------------

determinant = np.linalg.det(R)

orthogonality_error = np.max(
    np.abs(R.T @ R - np.eye(3))
)

# ------------------------------------------------------------
# 4. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 65)
print("CELL 126: FINAL TRANSFORMATION VALIDATION")
print("=" * 65)

print("Pocket atom correspondences:", len(residuals))
print("Atoms within 2 A:", int(inlier_mask.sum()))
print("Atoms outside 2 A:", int((~inlier_mask).sum()))

print("\nALIGNMENT QUALITY")
print("Inlier RMSD:", round(float(inlier_rmsd), 4), "A")
print("All-pocket RMSD:", round(float(all_rmsd), 4), "A")
print("Median deviation:", round(float(median_error), 4), "A")
print("95th percentile:", round(float(p95_error), 4), "A")
print("Maximum deviation:", round(float(residuals.max()), 4), "A")

print("\nROTATION VALIDATION")
print("Determinant:", determinant)
print("Orthogonality error:", orthogonality_error)

# ------------------------------------------------------------
# 5. SAVE RESIDUAL AUDIT
# ------------------------------------------------------------

pocket["composed_transform_residual_A"] = residuals
pocket["within_2A"] = inlier_mask

output_path = OUT / "final_composed_transform_validation.csv"

pocket.to_csv(output_path, index=False)

print("\nSaved:", output_path)

# ------------------------------------------------------------
# 6. VALIDATION STATUS
# ------------------------------------------------------------

passed = (
    abs(determinant - 1.0) < 1e-5
    and orthogonality_error < 1e-5
    and inlier_rmsd < 1.0
    and inlier_mask.sum() >= 400
)

print("\nFINAL STATUS:", "PASS" if passed else "REVIEW REQUIRED")
print("\nCELL 126 COMPLETED")

CELL 126: FINAL TRANSFORMATION VALIDATION
Pocket atom correspondences: 464
Atoms within 2 A: 453
Atoms outside 2 A: 11

ALIGNMENT QUALITY
Inlier RMSD: 0.6847 A
All-pocket RMSD: 0.8328 A
Median deviation: 0.4639 A
95th percentile: 1.6528 A
Maximum deviation: 4.3589 A

ROTATION VALIDATION
Determinant: 1.0000000000000016
Orthogonality error: 1.9984014443252818e-15

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/final_composed_transform_validation.csv

FINAL STATUS: PASS

CELL 126 COMPLETED


In [184]:
# ============================================================
# CELL 127: EXPORT VALIDATED LIGAND POSES IN M1 FRAME
# ============================================================

import gzip
import json
import numpy as np
import pandas as pd

from pathlib import Path
from io import BytesIO
from rdkit import Chem

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"

LIGAND_DIR = ROOT / "member2_outputs/crossdocked_parp1_ligands_shard23"

MANIFEST_PATH = ROOT / "member2_outputs/parp1_ligand_retrieval_manifest.csv"

# ------------------------------------------------------------
# 1. CHECK FINAL TRANSFORMATION VALIDATION
# ------------------------------------------------------------

validation = pd.read_csv(
    OUT / "final_composed_transform_validation.csv"
)

if len(validation) != 464:
    raise RuntimeError("Unexpected pocket validation record count.")

# ------------------------------------------------------------
# 2. LOAD COMPOSED TRANSFORMATION
# ------------------------------------------------------------

with open(
    OUT / "crossdocked_receptor_to_m1_composed_transform.json"
) as f:
    transform = json.load(f)

R = np.asarray(transform["rotation"])
t = np.asarray(transform["translation"])

# ------------------------------------------------------------
# 3. PREPARE OUTPUT
# ------------------------------------------------------------

manifest = pd.read_csv(MANIFEST_PATH)

export_dir = OUT / "validated_composed_M1_frame_SDF"
export_dir.mkdir(parents=True, exist_ok=True)

export_records = []

# ------------------------------------------------------------
# 4. TRANSFORM AND EXPORT EACH POSE
# ------------------------------------------------------------

for _, row in manifest.iterrows():

    pair_id = int(row["pair_idx"])
    identity = row["ligand_identity"]

    source = LIGAND_DIR / f"{pair_id:07d}.sdf.gz"

    with gzip.open(source, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            BytesIO(f.read()),
            sanitize=True,
            removeHs=True
        )

        molecules = list(supplier)

    for pose_index, mol in enumerate(molecules):

        if mol is None or mol.GetNumConformers() == 0:
            continue

        original_xyz = np.asarray(
            mol.GetConformer().GetPositions()
        )

        transformed_xyz = original_xyz @ R + t

        # Internal geometry preservation check
        before = np.linalg.norm(
            original_xyz[:, None, :] - original_xyz[None, :, :],
            axis=2
        )

        after = np.linalg.norm(
            transformed_xyz[:, None, :] - transformed_xyz[None, :, :],
            axis=2
        )

        geometry_error = float(
            np.max(np.abs(before - after))
        )

        # Update coordinates
        conformer = mol.GetConformer()

        for atom_idx, xyz in enumerate(transformed_xyz):
            conformer.SetAtomPosition(atom_idx, xyz.tolist())

        mol.SetProp("PAIR_ID", str(pair_id))
        mol.SetProp("LIGAND_IDENTITY", identity)
        mol.SetProp("POSE_INDEX", str(pose_index))
        mol.SetProp("COORDINATE_FRAME", "M1_CONFORMER_001")
        mol.SetProp("TRANSFORMATION", "GLOBAL_PLUS_LOCAL_COMPOSED")

        output_file = (
            export_dir /
            f"{identity}_pair{pair_id}_pose{pose_index}.sdf"
        )

        writer = Chem.SDWriter(str(output_file))
        writer.write(mol)
        writer.close()

        # Reopen and validate
        check = Chem.SDMolSupplier(
            str(output_file),
            sanitize=True,
            removeHs=True
        )

        reopened = check[0]

        valid = (
            reopened is not None
            and reopened.GetNumAtoms() == mol.GetNumAtoms()
            and reopened.GetNumBonds() == mol.GetNumBonds()
            and reopened.GetNumConformers() == 1
        )

        export_records.append({
            "pair_idx": pair_id,
            "ligand_identity": identity,
            "pose_index": pose_index,
            "atom_count": mol.GetNumAtoms(),
            "bond_count": mol.GetNumBonds(),
            "geometry_error_A": geometry_error,
            "reopened_validation": valid,
            "output_file": str(output_file)
        })

# ------------------------------------------------------------
# 5. FINAL EXPORT AUDIT
# ------------------------------------------------------------

export_df = pd.DataFrame(export_records)

if len(export_df) != 51:
    raise RuntimeError(
        f"Expected 51 exported poses; got {len(export_df)}"
    )

if not export_df["reopened_validation"].all():
    raise RuntimeError("Some exported SDF files failed validation.")

if export_df["geometry_error_A"].max() > 1e-6:
    raise RuntimeError("Molecular geometry preservation failed.")

audit_path = OUT / "composed_M1_frame_SDF_export_audit.csv"

export_df.to_csv(audit_path, index=False)

print("=" * 65)
print("CELL 127: LIGAND POSE EXPORT")
print("=" * 65)

print("Exported poses:", len(export_df))
print("Unique ligand records:", export_df["ligand_identity"].nunique())
print("Successfully reopened:", export_df["reopened_validation"].sum())
print("Maximum geometry error:", export_df["geometry_error_A"].max())

print("\nOutput directory:")
print(export_dir)

print("\nAudit file:")
print(audit_path)

print("\nCELL 127 COMPLETED")

CELL 127: LIGAND POSE EXPORT
Exported poses: 51
Unique ligand records: 32
Successfully reopened: 51
Maximum geometry error: 2.4868995751603507e-14

Output directory:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/validated_composed_M1_frame_SDF

Audit file:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/composed_M1_frame_SDF_export_audit.csv

CELL 127 COMPLETED


In [185]:
# ============================================================
# CELL 128: POSE QUALITY CLASSIFICATION
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs/parp1_receptor_validation"

geometry_path = OUT / "crossdocked_pose_receptor_geometry.csv"
export_path = OUT / "composed_M1_frame_SDF_export_audit.csv"

geometry = pd.read_csv(geometry_path)
exports = pd.read_csv(export_path)

# Identify minimum-distance column
distance_col = next(
    (c for c in geometry.columns
     if c.lower() in ["min_distance_a", "minimum_distance_a"]),
    None
)

if distance_col is None:
    raise ValueError(
        f"Minimum-distance column not found. Columns: {list(geometry.columns)}"
    )

# Identify pose-index column
pose_col = next(
    (c for c in geometry.columns
     if c.lower() in ["pose_index", "pose_id"]),
    None
)

if pose_col is None:
    raise ValueError(
        f"Pose-index column not found. Columns: {list(geometry.columns)}"
    )

# Merge using pair ID and pose index
qc = exports.merge(
    geometry,
    on=["pair_idx", "ligand_identity", pose_col],
    how="left",
    suffixes=("_export", "_geometry")
)

if qc[distance_col].isna().any():
    raise RuntimeError("Some exported poses have no geometry record.")

# Apply the same <2 A severe-overlap criterion used in Cell 120
qc["pose_quality"] = qc[distance_col].apply(
    lambda d: "SEVERE_OVERLAP" if d < 2.0 else "PASS"
)

qc["usable_for_spatial_screening"] = (
    qc["pose_quality"] == "PASS"
) & qc["reopened_validation"].astype(bool)

# Per-ligand summary
summary = (
    qc.groupby("ligand_identity")
    .agg(
        total_poses=("pose_index", "count"),
        usable_poses=("usable_for_spatial_screening", "sum"),
        severe_overlap_poses=(
            "pose_quality",
            lambda x: (x == "SEVERE_OVERLAP").sum()
        )
    )
    .reset_index()
)

# Save outputs
qc_path = OUT / "crossdocked_pose_quality_manifest.csv"
summary_path = OUT / "crossdocked_pose_quality_summary.csv"

qc.to_csv(qc_path, index=False)
summary.to_csv(summary_path, index=False)

print("=" * 65)
print("CELL 128: POSE QUALITY CLASSIFICATION")
print("=" * 65)

print("Total poses:", len(qc))
print("Unique ligands:", qc["ligand_identity"].nunique())

print("\nPOSE CLASSIFICATION")
print(qc["pose_quality"].value_counts())

print("\nUSABLE POSES:", int(qc["usable_for_spatial_screening"].sum()))

print("\nPER-LIGAND SUMMARY")
display(summary)

print("\nSAVED:")
print(qc_path)
print(summary_path)

print("\nCELL 128 COMPLETED")

CELL 128: POSE QUALITY CLASSIFICATION
Total poses: 51
Unique ligands: 32

POSE CLASSIFICATION
pose_quality
PASS              39
SEVERE_OVERLAP    12
Name: count, dtype: int64

USABLE POSES: 39

PER-LIGAND SUMMARY


,ligand_identity,total_poses,usable_poses,severe_overlap_poses
0,1uk0_frm,1,1,0
1,1uk1_frq,1,1,0
2,1wok_cnq,1,1,0
3,2rcw_aai,1,1,0
4,2rd6_78p,1,1,0
5,3gjw_gjw,1,1,0
6,3gn7_3gn,1,1,0
7,3l3l_l3l,1,1,0
8,3l3m_a92,1,1,0
9,4gv7_mew,1,1,0



SAVED:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_pose_quality_manifest.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_pose_quality_summary.csv

CELL 128 COMPLETED


In [186]:
# ============================================================
# CELL 129: M1 CONFORMER COORDINATE-FRAME AUDIT
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 129: CONFORMER COORDINATE-FRAME AUDIT")
print("=" * 65)

# 1. Check required variables
required = [
    "transformations",
    "protein_atoms_by_conformer",
    "receptor_features_by_conformer"
]

for name in required:
    if name not in globals():
        raise RuntimeError(f"Required variable missing: {name}")

print("\nAVAILABLE CONFORMERS")
print("Transformations:", sorted(transformations.keys()))
print("Protein atom tables:", sorted(protein_atoms_by_conformer.keys()))
print("Receptor feature tables:", sorted(receptor_features_by_conformer.keys()))

# 2. Inspect transformation structure
print("\nTRANSFORMATION DETAILS")

for conf_id in sorted(transformations.keys()):
    item = transformations[conf_id]

    print(f"\nConformer {conf_id}")
    print("Fields:", list(item.keys()))

    if "global_rmsd" in item:
        print("Global RMSD:", item["global_rmsd"])

    if "pocket_rmsd" in item:
        print("Pocket RMSD:", item["pocket_rmsd"])

    if "rotation" in item:
        R = np.asarray(item["rotation"])
        t = np.asarray(item["translation"])

        print("Rotation shape:", R.shape)
        print("Translation shape:", t.shape)
        print("Determinant:", np.linalg.det(R))
        print("Orthogonality error:",
              np.max(np.abs(R.T @ R - np.eye(3))))

# 3. Inspect coordinate-table schemas
print("\nPROTEIN ATOM TABLE SCHEMA")

for conf_id, df in protein_atoms_by_conformer.items():
    print(
        f"Conformer {conf_id}:",
        df.shape,
        list(df.columns)
    )
    break

print("\nRECEPTOR FEATURE TABLE SCHEMA")

for conf_id, df in receptor_features_by_conformer.items():
    print(
        f"Conformer {conf_id}:",
        df.shape,
        list(df.columns)
    )
    break

# 4. Confirm pose-quality input
qc_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/parp1_receptor_validation/"
    "crossdocked_pose_quality_manifest.csv"
)

pose_qc = pd.read_csv(qc_path)

usable = pose_qc[
    pose_qc["usable_for_spatial_screening"] == True
].copy()

print("\nPOSE INPUT")
print("Total poses:", len(pose_qc))
print("Usable poses:", len(usable))
print("Unique usable ligands:", usable["ligand_identity"].nunique())

print("\nCELL 129 AUDIT COMPLETED")

CELL 129: CONFORMER COORDINATE-FRAME AUDIT

AVAILABLE CONFORMERS
Transformations: [1, 2, 3, 4, 5]
Protein atom tables: [1, 2, 3, 4, 5]
Receptor feature tables: [1, 2, 3, 4, 5]

TRANSFORMATION DETAILS

Conformer 1
Fields: ['rotation', 'translation', 'common_atom_keys', 'global_rmsd', 'pocket_rmsd']
Global RMSD: 1.1508085959435004e-14
Pocket RMSD: 9.70330409893131e-15
Rotation shape: (3, 3)
Translation shape: (3,)
Determinant: 0.9999999999999998
Orthogonality error: 5.826923063072086e-16

Conformer 2
Fields: ['rotation', 'translation', 'common_atom_keys', 'global_rmsd', 'pocket_rmsd']
Global RMSD: 0.6028700462859085
Pocket RMSD: 0.3742514752296577
Rotation shape: (3, 3)
Translation shape: (3,)
Determinant: 1.0000000000000002
Orthogonality error: 5.195573506171172e-16

Conformer 3
Fields: ['rotation', 'translation', 'common_atom_keys', 'global_rmsd', 'pocket_rmsd']
Global RMSD: 0.8766404293774209
Pocket RMSD: 0.33366393377140696
Rotation shape: (3, 3)
Translation shape: (3,)
Determinant: 

In [187]:
# ============================================================
# CELL 130: VERIFY CONFORMER ALIGNMENT DIRECTION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 130: TRANSFORMATION DIRECTION VALIDATION")
print("=" * 65)

# Inspect common atom key format
sample_keys = transformations[2]["common_atom_keys"]

print("Common atom key type:", type(sample_keys))
print("Number of common atoms:", len(sample_keys))
print("First five keys:")

for key in list(sample_keys)[:5]:
    print(key)

print("\nConformer 1 sample:")
display(protein_atoms_by_conformer[1].head())

print("\nConformer 2 sample:")
display(protein_atoms_by_conformer[2].head())

print("\nRecorded alignment RMSD values:")

for conf_id in sorted(transformations):
    print(
        f"Conformer {conf_id}: "
        f"global={transformations[conf_id]['global_rmsd']:.6f} Å, "
        f"pocket={transformations[conf_id]['pocket_rmsd']:.6f} Å"
    )

print("\nCELL 130 DIAGNOSTIC COMPLETED")

CELL 130: TRANSFORMATION DIRECTION VALIDATION
Common atom key type: <class 'list'>
Number of common atoms: 5445
First five keys:
A|1000|LYS|C
A|1000|LYS|CA
A|1000|LYS|CB
A|1000|LYS|CD
A|1000|LYS|CE

Conformer 1 sample:


,atom_name,element,residue_name,residue_number,chain,x,y,z
0,N,N,LYS,662,A,-10.953,54.203999,-18.306000
1,CA,C,LYS,662,A,-9.504,53.925999,-18.280001
2,C,C,LYS,662,A,-8.795,54.866001,-17.275000
3,O,O,LYS,662,A,-9.120,56.070999,-17.143000
4,CB,C,LYS,662,A,-8.932,54.007000,-19.677000



Conformer 2 sample:


,atom_name,element,residue_name,residue_number,chain,x,y,z
0,N,N,LYS,662,A,-11.448,54.404999,-18.180000
1,CA,C,LYS,662,A,-10.004,54.110001,-18.162001
2,C,C,LYS,662,A,-9.284,55.074001,-17.158001
3,O,O,LYS,662,A,-9.603,56.290001,-17.028000
4,CB,C,LYS,662,A,-9.427,54.203999,-19.558001



Recorded alignment RMSD values:
Conformer 1: global=0.000000 Å, pocket=0.000000 Å
Conformer 2: global=0.602870 Å, pocket=0.374251 Å
Conformer 3: global=0.876640 Å, pocket=0.333664 Å
Conformer 4: global=0.691026 Å, pocket=0.355291 Å
Conformer 5: global=0.438320 Å, pocket=0.166832 Å

CELL 130 DIAGNOSTIC COMPLETED


In [188]:
# ============================================================
# CELL 131: NUMERICAL TRANSFORMATION DIRECTION VALIDATION
# ============================================================

import numpy as np
import pandas as pd

def make_atom_keys(df):
    return (
        df["chain"].astype(str) + "|" +
        df["residue_number"].astype(str) + "|" +
        df["residue_name"].astype(str) + "|" +
        df["atom_name"].astype(str)
    )

reference = protein_atoms_by_conformer[1].copy()
reference["atom_key"] = make_atom_keys(reference)

if reference["atom_key"].duplicated().any():
    raise RuntimeError("Duplicate atom keys in reference conformer.")

reference = reference.set_index("atom_key")

results = []

for conf_id in sorted(transformations):

    current = protein_atoms_by_conformer[conf_id].copy()
    current["atom_key"] = make_atom_keys(current)

    if current["atom_key"].duplicated().any():
        raise RuntimeError(f"Duplicate atom keys in conformer {conf_id}.")

    current = current.set_index("atom_key")

    common_keys = transformations[conf_id]["common_atom_keys"]

    X = current.loc[common_keys, ["x", "y", "z"]].to_numpy(float)
    Y = reference.loc[common_keys, ["x", "y", "z"]].to_numpy(float)

    R = np.asarray(transformations[conf_id]["rotation"])
    t = np.asarray(transformations[conf_id]["translation"])

    # Test 1: current conformer -> conformer 1
    forward = X @ R + t
    forward_rmsd = np.sqrt(np.mean(np.sum((forward - Y)**2, axis=1)))

    # Test 2: opposite direction using inverse rigid transform
    reverse = (Y - t) @ R.T
    reverse_rmsd = np.sqrt(np.mean(np.sum((reverse - X)**2, axis=1)))

    recorded = transformations[conf_id]["global_rmsd"]

    results.append({
        "conformer": conf_id,
        "forward_rmsd_A": forward_rmsd,
        "reverse_rmsd_A": reverse_rmsd,
        "recorded_rmsd_A": recorded,
        "forward_matches_recorded": abs(forward_rmsd - recorded) < 1e-4
    })

results_df = pd.DataFrame(results)

display(results_df)

if not results_df["forward_matches_recorded"].all():
    raise RuntimeError("Transformation direction did not match recorded RMSD.")

print("\nALL TRANSFORMATIONS VERIFIED")
print("Direction: conformer coordinates -> conformer 1 coordinates")
print("Formula: X_aligned = X @ R + t")
print("\nCELL 131 COMPLETED")

,conformer,forward_rmsd_A,reverse_rmsd_A,recorded_rmsd_A,forward_matches_recorded
0,1,1.150809e-14,1.283538e-14,1.150809e-14,True
1,2,6.028700e-01,6.028700e-01,6.028700e-01,True
2,3,8.766404e-01,8.766404e-01,8.766404e-01,True
3,4,6.910264e-01,6.910264e-01,6.910264e-01,True
4,5,4.383200e-01,4.383200e-01,4.383200e-01,True



ALL TRANSFORMATIONS VERIFIED
Direction: conformer coordinates -> conformer 1 coordinates
Formula: X_aligned = X @ R + t

CELL 131 COMPLETED


In [189]:
# ============================================================
# CELL 132: ALIGN ALL M1 CONFORMERS TO CONFORMER 1
# ============================================================

import numpy as np
import pandas as pd

aligned_protein_atoms = {}
aligned_receptor_features = {}

alignment_checks = []

for conf_id in sorted(transformations):

    R = np.asarray(transformations[conf_id]["rotation"])
    t = np.asarray(transformations[conf_id]["translation"])

    # --------------------------------------------
    # 1. ALIGN PROTEIN ATOMS
    # --------------------------------------------

    protein = protein_atoms_by_conformer[conf_id].copy()

    original_xyz = protein[["x", "y", "z"]].to_numpy(float)
    aligned_xyz = original_xyz @ R + t

    protein[["x", "y", "z"]] = aligned_xyz

    aligned_protein_atoms[conf_id] = protein

    # --------------------------------------------
    # 2. ALIGN RECEPTOR PHARMACOPHORE FEATURES
    # --------------------------------------------

    features = receptor_features_by_conformer[conf_id].copy()

    original_feature_xyz = features[
        ["x", "y", "z"]
    ].to_numpy(float)

    aligned_feature_xyz = original_feature_xyz @ R + t

    features[["x", "y", "z"]] = aligned_feature_xyz

    aligned_receptor_features[conf_id] = features

    # --------------------------------------------
    # 3. VALIDATE
    # --------------------------------------------

    if not np.isfinite(aligned_xyz).all():
        raise RuntimeError(f"Invalid protein coordinates: {conf_id}")

    if not np.isfinite(aligned_feature_xyz).all():
        raise RuntimeError(f"Invalid feature coordinates: {conf_id}")

    if len(features) != len(receptor_features_by_conformer[1]):
        raise RuntimeError(f"Feature count mismatch: {conf_id}")

    if conf_id == 1:
        protein_error = np.max(
            np.abs(aligned_xyz - original_xyz)
        )
        feature_error = np.max(
            np.abs(aligned_feature_xyz - original_feature_xyz)
        )
    else:
        protein_error = np.nan
        feature_error = np.nan

    alignment_checks.append({
        "conformer": conf_id,
        "protein_atoms": len(protein),
        "receptor_features": len(features),
        "protein_coordinate_error_conf1_A": protein_error,
        "feature_coordinate_error_conf1_A": feature_error,
        "finite_coordinates": True
    })

# --------------------------------------------
# 4. FINAL AUDIT
# --------------------------------------------

alignment_df = pd.DataFrame(alignment_checks)

display(alignment_df)

assert len(aligned_protein_atoms) == 5
assert len(aligned_receptor_features) == 5
assert alignment_df["finite_coordinates"].all()

print("\nAligned conformers:", len(aligned_protein_atoms))
print("Total aligned receptor features:",
      sum(len(x) for x in aligned_receptor_features.values()))

print("\nCELL 132 COMPLETED")

,conformer,protein_atoms,receptor_features,protein_coordinate_error_conf1_A,feature_coordinate_error_conf1_A,finite_coordinates
0,1,5445,95,2.131628e-14,1.421085e-14,True
1,2,5445,95,NaN,NaN,True
2,3,5445,95,NaN,NaN,True
3,4,5445,95,NaN,NaN,True
4,5,5445,95,NaN,NaN,True



Aligned conformers: 5
Total aligned receptor features: 475

CELL 132 COMPLETED


In [190]:
# ============================================================
# CELL 133: SPATIAL PHARMACOPHORE MATCHING
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import ChemicalFeatures
from rdkit import RDConfig
from scipy.spatial.distance import cdist

# ------------------------------------------------------------
# 1. PATHS AND INPUTS
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"

qc_path = OUT / "crossdocked_pose_quality_manifest.csv"

pose_qc = pd.read_csv(qc_path)

usable = pose_qc[
    pose_qc["usable_for_spatial_screening"].astype(bool)
].copy()

if len(usable) != 39:
    raise RuntimeError(f"Expected 39 usable poses, found {len(usable)}")

# Find exported SDF path column
path_col = next(
    (c for c in usable.columns
     if c.lower() in ["output_file", "output_file_export"]),
    None
)

if path_col is None:
    raise RuntimeError(
        f"SDF path column not found. Available columns: {list(usable.columns)}"
    )

# ------------------------------------------------------------
# 2. RDKit PHARMACOPHORE FEATURE FACTORY
# ------------------------------------------------------------

fdef_path = str(
    Path(RDConfig.RDDataDir) / "BaseFeatures.fdef"
)

factory = ChemicalFeatures.BuildFeatureFactory(fdef_path)

family_mapping = {
    "Hydrophobe": "HYDROPHOBIC",
    "LumpedHydrophobe": "HYDROPHOBIC",
    "Aromatic": "AROMATIC",
    "Donor": "DONOR",
    "Acceptor": "ACCEPTOR",
    "PosIonizable": "POSITIVE",
    "NegIonizable": "NEGATIVE"
}

compatibility = {
    "HYDROPHOBIC": {"HYDROPHOBIC": 4.5},
    "AROMATIC": {"AROMATIC": 5.0},
    "DONOR": {"ACCEPTOR": 3.5},
    "ACCEPTOR": {"DONOR": 3.5},
    "POSITIVE": {"NEGATIVE": 4.0},
    "NEGATIVE": {"POSITIVE": 4.0}
}

# ------------------------------------------------------------
# 3. EXTRACT LIGAND PHARMACOPHORE FEATURES
# ------------------------------------------------------------

def extract_ligand_features(mol):

    if mol is None or mol.GetNumConformers() == 0:
        raise ValueError("Invalid molecule or missing 3D coordinates.")

    conf = mol.GetConformer()
    features = factory.GetFeaturesForMol(mol)

    records = []

    for feature_index, feature in enumerate(features):

        family = feature.GetFamily()

        if family not in family_mapping:
            continue

        feature_type = family_mapping[family]

        atom_ids = list(feature.GetAtomIds())

        coords = np.array([
            conf.GetAtomPosition(i)
            for i in atom_ids
        ], dtype=float)

        centroid = coords.mean(axis=0)

        records.append({
            "ligand_feature_index": feature_index,
            "ligand_feature_type": feature_type,
            "ligand_atom_ids": ";".join(map(str, atom_ids)),
            "x": centroid[0],
            "y": centroid[1],
            "z": centroid[2]
        })

    return pd.DataFrame(records)

# ------------------------------------------------------------
# 4. COMPARE ONE LIGAND WITH ONE CONFORMER
# ------------------------------------------------------------

def match_features(ligand_features, receptor_features, conf_id,
                   pair_id, ligand_identity, pose_index):

    matches = []

    for ligand_type, receptor_types in compatibility.items():

        ligand_subset = ligand_features[
            ligand_features["ligand_feature_type"] == ligand_type
        ]

        if ligand_subset.empty:
            continue

        for receptor_type, cutoff in receptor_types.items():

            receptor_subset = receptor_features[
                receptor_features["feature_type"] == receptor_type
            ]

            if receptor_subset.empty:
                continue

            ligand_xyz = ligand_subset[
                ["x", "y", "z"]
            ].to_numpy(float)

            receptor_xyz = receptor_subset[
                ["x", "y", "z"]
            ].to_numpy(float)

            distances = cdist(ligand_xyz, receptor_xyz)

            ligand_indices, receptor_indices = np.where(
                distances <= cutoff
            )

            for li, ri in zip(ligand_indices, receptor_indices):

                lf = ligand_subset.iloc[li]
                rf = receptor_subset.iloc[ri]

                matches.append({
                    "pair_idx": pair_id,
                    "ligand_identity": ligand_identity,
                    "pose_index": pose_index,
                    "conformer": conf_id,
                    "ligand_feature_index": int(
                        lf["ligand_feature_index"]
                    ),
                    "ligand_feature_type": ligand_type,
                    "ligand_atom_ids": lf["ligand_atom_ids"],
                    "receptor_feature_index": int(
                        rf["feature_index"]
                    ),
                    "receptor_feature_type": receptor_type,
                    "receptor_identity": rf["identity"],
                    "distance_A": float(distances[li, ri]),
                    "cutoff_A": cutoff
                })

    return matches

# ------------------------------------------------------------
# 5. PROCESS ALL 39 POSES AGAINST ALL 5 CONFORMERS
# ------------------------------------------------------------

all_matches = []
pose_processing = []

for _, row in usable.iterrows():

    sdf_path = Path(row[path_col])

    if not sdf_path.exists():
        raise FileNotFoundError(sdf_path)

    supplier = Chem.SDMolSupplier(
        str(sdf_path),
        sanitize=True,
        removeHs=True
    )

    mol = supplier[0]

    if mol is None:
        raise RuntimeError(f"Could not read SDF: {sdf_path}")

    ligand_features = extract_ligand_features(mol)

    pose_match_count = 0

    for conf_id in sorted(aligned_receptor_features):

        receptor_features = aligned_receptor_features[conf_id]

        matches = match_features(
            ligand_features,
            receptor_features,
            conf_id,
            int(row["pair_idx"]),
            row["ligand_identity"],
            int(row["pose_index"])
        )

        all_matches.extend(matches)
        pose_match_count += len(matches)

    pose_processing.append({
        "pair_idx": int(row["pair_idx"]),
        "ligand_identity": row["ligand_identity"],
        "pose_index": int(row["pose_index"]),
        "ligand_feature_count": len(ligand_features),
        "matches_across_five_conformers": pose_match_count
    })

# ------------------------------------------------------------
# 6. BUILD RESULTS
# ------------------------------------------------------------

matches_df = pd.DataFrame(all_matches)
processing_df = pd.DataFrame(pose_processing)

if len(processing_df) != 39:
    raise RuntimeError("Not all 39 usable poses were processed.")

# Summary per conformer
if len(matches_df):

    conformer_summary = (
        matches_df.groupby("conformer")
        .agg(
            compatible_feature_matches=("distance_A", "size"),
            unique_ligands=("ligand_identity", "nunique"),
            unique_poses=("pair_idx", "count"),
            mean_distance_A=("distance_A", "mean"),
            median_distance_A=("distance_A", "median")
        )
        .reset_index()
    )

else:

    conformer_summary = pd.DataFrame(
        columns=[
            "conformer",
            "compatible_feature_matches",
            "unique_ligands",
            "unique_poses",
            "mean_distance_A",
            "median_distance_A"
        ]
    )

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

matches_path = OUT / "spatial_pharmacophore_matches_5conformers.csv"

processing_path = OUT / "spatial_pharmacophore_pose_processing.csv"

summary_path = OUT / "spatial_pharmacophore_conformer_summary.csv"

matches_df.to_csv(matches_path, index=False)
processing_df.to_csv(processing_path, index=False)
conformer_summary.to_csv(summary_path, index=False)

# ------------------------------------------------------------
# 8. REPORT
# ------------------------------------------------------------

print("=" * 65)
print("CELL 133: SPATIAL PHARMACOPHORE MATCHING")
print("=" * 65)

print("Usable poses processed:", len(processing_df))
print("Unique ligand records:", processing_df["ligand_identity"].nunique())
print("Conformers evaluated:", len(aligned_receptor_features))
print("Total compatible spatial matches:", len(matches_df))

print("\nMATCHES BY CONFORMER")
display(conformer_summary)

print("\nMATCHES BY INTERACTION TYPE")

if len(matches_df):
    display(
        matches_df.groupby(
            ["ligand_feature_type", "receptor_feature_type"]
        ).size().reset_index(name="match_count")
    )

print("\nSAVED:")
print(matches_path)
print(processing_path)
print(summary_path)

print("\nCELL 133 COMPLETED")

CELL 133: SPATIAL PHARMACOPHORE MATCHING
Usable poses processed: 39
Unique ligand records: 32
Conformers evaluated: 5
Total compatible spatial matches: 2314

MATCHES BY CONFORMER


,conformer,compatible_feature_matches,unique_ligands,unique_poses,mean_distance_A,median_distance_A
0,1,514,32,514,3.956031,3.977736
1,2,439,32,439,3.951412,4.000870
2,3,413,32,413,4.003876,4.030564
3,4,480,32,480,4.003383,4.017053
4,5,468,32,468,3.985159,4.006742



MATCHES BY INTERACTION TYPE


,ligand_feature_type,receptor_feature_type,match_count
0,ACCEPTOR,DONOR,66
1,AROMATIC,AROMATIC,326
2,DONOR,ACCEPTOR,67
3,HYDROPHOBIC,HYDROPHOBIC,1855



SAVED:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/spatial_pharmacophore_matches_5conformers.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/spatial_pharmacophore_pose_processing.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/spatial_pharmacophore_conformer_summary.csv

CELL 133 COMPLETED


In [191]:
# ============================================================
# CELL 134: NORMALIZED SPATIAL MATCHING AUDIT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"

matches = pd.read_csv(
    OUT / "spatial_pharmacophore_matches_5conformers.csv"
)

processing = pd.read_csv(
    OUT / "spatial_pharmacophore_pose_processing.csv"
)

# ------------------------------------------------------------
# 1. VALIDATE INPUT
# ------------------------------------------------------------

required_match_columns = [
    "pair_idx",
    "ligand_identity",
    "pose_index",
    "conformer",
    "ligand_feature_index",
    "ligand_feature_type",
    "receptor_feature_index",
    "receptor_feature_type",
    "distance_A"
]

missing = set(required_match_columns) - set(matches.columns)

if missing:
    raise RuntimeError(f"Missing columns: {missing}")

if matches["conformer"].nunique() != 5:
    raise RuntimeError("Expected five conformers.")

# ------------------------------------------------------------
# 2. COUNT UNIQUE FEATURE PAIRS
# ------------------------------------------------------------

pair_summary = (
    matches.groupby(
        ["ligand_identity", "pair_idx", "pose_index", "conformer"]
    )
    .agg(
        total_match_records=("distance_A", "size"),

        unique_feature_pairs=(
            ["ligand_feature_index", "receptor_feature_index"],
            "size"
        ) if False else ("distance_A", "size"),

        unique_ligand_features=(
            "ligand_feature_index", "nunique"
        ),

        unique_receptor_features=(
            "receptor_feature_index", "nunique"
        ),

        mean_distance_A=("distance_A", "mean"),

        median_distance_A=("distance_A", "median")
    )
    .reset_index()
)

# Correct unique feature-pair count separately
unique_pairs = (
    matches.drop_duplicates(
        subset=[
            "ligand_identity",
            "pair_idx",
            "pose_index",
            "conformer",
            "ligand_feature_index",
            "receptor_feature_index"
        ]
    )
    .groupby(
        ["ligand_identity", "pair_idx", "pose_index", "conformer"]
    )
    .size()
    .reset_index(name="unique_feature_pairs")
)

pair_summary = pair_summary.drop(
    columns=["unique_feature_pairs"]
).merge(
    unique_pairs,
    on=["ligand_identity", "pair_idx", "pose_index", "conformer"],
    how="left"
)

# ------------------------------------------------------------
# 3. NORMALIZE BY LIGAND FEATURE COUNT
# ------------------------------------------------------------

pair_summary = pair_summary.merge(
    processing[
        [
            "pair_idx",
            "ligand_identity",
            "pose_index",
            "ligand_feature_count"
        ]
    ],
    on=["pair_idx", "ligand_identity", "pose_index"],
    how="left"
)

if pair_summary["ligand_feature_count"].isna().any():
    raise RuntimeError("Missing ligand feature counts.")

pair_summary["matches_per_ligand_feature"] = (
    pair_summary["unique_feature_pairs"] /
    pair_summary["ligand_feature_count"]
)

# ------------------------------------------------------------
# 4. LIGAND-LEVEL FIVE-CONFORMER SUMMARY
# ------------------------------------------------------------

ligand_summary = (
    pair_summary.groupby("ligand_identity")
    .agg(
        pair_id=("pair_idx", "first"),
        ligand_feature_count=("ligand_feature_count", "first"),

        conformers_with_matches=("conformer", "nunique"),

        total_unique_feature_pairs=("unique_feature_pairs", "sum"),

        mean_matches_per_feature=(
            "matches_per_ligand_feature", "mean"
        ),

        median_matches_per_feature=(
            "matches_per_ligand_feature", "median"
        ),

        mean_matched_ligand_features=(
            "unique_ligand_features", "mean"
        ),

        mean_distance_A=("mean_distance_A", "mean")
    )
    .reset_index()
)

ligand_summary["conformer_coverage_percent"] = (
    ligand_summary["conformers_with_matches"] / 5 * 100
)

# ------------------------------------------------------------
# 5. CONFORMER-LEVEL CORRECTED SUMMARY
# ------------------------------------------------------------

conformer_summary = (
    pair_summary.groupby("conformer")
    .agg(
        ligand_records=("ligand_identity", "nunique"),
        unique_poses=("pose_index", "count"),
        unique_feature_pairs=("unique_feature_pairs", "sum"),
        mean_matches_per_feature=(
            "matches_per_ligand_feature", "mean"
        ),
        mean_distance_A=("mean_distance_A", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

pair_path = OUT / "normalized_matching_per_pose_conformer.csv"
ligand_path = OUT / "normalized_matching_per_ligand.csv"
conf_path = OUT / "normalized_matching_per_conformer.csv"

pair_summary.to_csv(pair_path, index=False)
ligand_summary.to_csv(ligand_path, index=False)
conformer_summary.to_csv(conf_path, index=False)

# ------------------------------------------------------------
# 7. REPORT
# ------------------------------------------------------------

print("=" * 65)
print("CELL 134: NORMALIZED SPATIAL MATCHING AUDIT")
print("=" * 65)

print("Original match records:", len(matches))
print("Unique ligand records:", ligand_summary.shape[0])
print("Pose-conformer records:", len(pair_summary))

print("\nCONFORMER SUMMARY")
display(conformer_summary)

print("\nLIGAND SUMMARY")
display(
    ligand_summary.sort_values(
        "mean_matches_per_feature",
        ascending=False
    )
)

print("\nSAVED:")
print(pair_path)
print(ligand_path)
print(conf_path)

print("\nCELL 134 COMPLETED")

CELL 134: NORMALIZED SPATIAL MATCHING AUDIT
Original match records: 2314
Unique ligand records: 32
Pose-conformer records: 184

CONFORMER SUMMARY


,conformer,ligand_records,unique_poses,unique_feature_pairs,mean_matches_per_feature,mean_distance_A
0,1,32,37,514,0.899182,3.967339
1,2,32,37,439,0.772676,3.957822
2,3,32,37,413,0.727190,4.014352
3,4,32,36,480,0.866377,4.008852
4,5,32,37,468,0.823425,3.998765



LIGAND SUMMARY


,ligand_identity,pair_id,ligand_feature_count,conformers_with_matches,total_unique_feature_pairs,mean_matches_per_feature,median_matches_per_feature,mean_matched_ligand_features,mean_distance_A,conformer_coverage_percent
18,4uxb_p34,1735698,14,5,121,1.728571,1.714286,8.000000,3.877675,100.0
5,3gjw_gjw,1735681,9,5,63,1.400000,1.444444,5.000000,3.886316,100.0
14,4r5w_xav,1735694,6,5,42,1.400000,1.333333,3.200000,4.135113,100.0
19,4zzz_fsu,1735699,11,5,74,1.345455,1.363636,5.600000,4.058375,100.0
7,3l3l_l3l,1735683,11,5,73,1.327273,1.272727,6.000000,4.000864,100.0
20,5a00_d7n,1735700,12,5,77,1.283333,1.250000,6.000000,4.046714,100.0
16,4rv6_rpb,1735696,14,5,88,1.257143,1.285714,7.000000,4.049225,100.0
17,4und_2yq,1735697,16,5,84,1.050000,1.000000,7.600000,4.025612,100.0
2,1wok_cnq,1735678,12,5,58,0.966667,1.000000,6.600000,3.976460,100.0
28,5wrz_7u3,1735708,14,5,64,0.914286,1.000000,5.000000,4.037118,100.0



SAVED:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/normalized_matching_per_pose_conformer.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/normalized_matching_per_ligand.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/normalized_matching_per_conformer.csv

CELL 134 COMPLETED


In [192]:
# ============================================================
# CELL 135: COMPLETE POSE-CONFORMER MATCHING AUDIT
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs/parp1_receptor_validation"

matches = pd.read_csv(
    OUT / "spatial_pharmacophore_matches_5conformers.csv"
)

processing = pd.read_csv(
    OUT / "spatial_pharmacophore_pose_processing.csv"
)

# ------------------------------------------------------------
# 1. CREATE COMPLETE 39 x 5 COMBINATION GRID
# ------------------------------------------------------------

conformers = pd.DataFrame({
    "conformer": [1, 2, 3, 4, 5]
})

processing["pose_key"] = (
    processing["pair_idx"].astype(str) + "_" +
    processing["pose_index"].astype(str)
)

grid = processing.merge(conformers, how="cross")

if len(grid) != 195:
    raise RuntimeError(f"Expected 195 combinations, found {len(grid)}")

# ------------------------------------------------------------
# 2. COUNT UNIQUE FEATURE PAIRS
# ------------------------------------------------------------

unique_matches = matches.drop_duplicates(
    subset=[
        "pair_idx",
        "pose_index",
        "conformer",
        "ligand_feature_index",
        "receptor_feature_index"
    ]
)

pair_counts = (
    unique_matches.groupby(
        ["pair_idx", "pose_index", "conformer"]
    )
    .agg(
        unique_feature_pairs=("receptor_feature_index", "size"),
        unique_ligand_features=("ligand_feature_index", "nunique"),
        unique_receptor_features=("receptor_feature_index", "nunique"),
        mean_distance_A=("distance_A", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. MERGE WITH COMPLETE GRID
# ------------------------------------------------------------

complete = grid.merge(
    pair_counts,
    on=["pair_idx", "pose_index", "conformer"],
    how="left"
)

count_columns = [
    "unique_feature_pairs",
    "unique_ligand_features",
    "unique_receptor_features"
]

complete[count_columns] = (
    complete[count_columns].fillna(0).astype(int)
)

complete["mean_distance_A"] = (
    complete["mean_distance_A"].fillna(np.nan)
)

complete["matches_per_ligand_feature"] = (
    complete["unique_feature_pairs"] /
    complete["ligand_feature_count"]
)

complete["has_spatial_match"] = (
    complete["unique_feature_pairs"] > 0
)

# ------------------------------------------------------------
# 4. POSE-LEVEL SUMMARY ACROSS FIVE CONFORMERS
# ------------------------------------------------------------

pose_summary = (
    complete.groupby(
        ["ligand_identity", "pair_idx", "pose_index"]
    )
    .agg(
        ligand_feature_count=("ligand_feature_count", "first"),
        conformers_with_matches=("has_spatial_match", "sum"),
        total_unique_feature_pairs=("unique_feature_pairs", "sum"),
        mean_matches_per_feature=("matches_per_ligand_feature", "mean"),
        mean_distance_A=("mean_distance_A", "mean")
    )
    .reset_index()
)

pose_summary["conformer_coverage_percent"] = (
    pose_summary["conformers_with_matches"] / 5 * 100
)

# ------------------------------------------------------------
# 5. FAIR LIGAND-LEVEL SUMMARY
# First aggregate within each pose, then across poses.
# ------------------------------------------------------------

ligand_summary = (
    pose_summary.groupby("ligand_identity")
    .agg(
        pair_id=("pair_idx", "first"),
        usable_pose_count=("pose_index", "nunique"),
        mean_pose_score=("mean_matches_per_feature", "mean"),
        median_pose_score=("mean_matches_per_feature", "median"),
        mean_pose_coverage=("conformer_coverage_percent", "mean"),
        mean_pose_feature_pairs=("total_unique_feature_pairs", "mean"),
        mean_pose_distance_A=("mean_distance_A", "mean")
    )
    .reset_index()
    .sort_values("mean_pose_score", ascending=False)
)

# ------------------------------------------------------------
# 6. CONFORMER SUMMARY INCLUDING ZERO-MATCH POSES
# ------------------------------------------------------------

conformer_summary = (
    complete.groupby("conformer")
    .agg(
        total_poses=("pose_index", "count"),
        poses_with_matches=("has_spatial_match", "sum"),
        total_unique_feature_pairs=("unique_feature_pairs", "sum"),
        mean_normalized_score=("matches_per_ligand_feature", "mean")
    )
    .reset_index()
)

conformer_summary["pose_match_rate_percent"] = (
    conformer_summary["poses_with_matches"] /
    conformer_summary["total_poses"] * 100
)

# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

complete.to_csv(
    OUT / "complete_pose_conformer_audit.csv",
    index=False
)

pose_summary.to_csv(
    OUT / "corrected_pose_level_summary.csv",
    index=False
)

ligand_summary.to_csv(
    OUT / "corrected_ligand_level_summary.csv",
    index=False
)

conformer_summary.to_csv(
    OUT / "corrected_conformer_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 8. REPORT
# ------------------------------------------------------------

print("=" * 65)
print("CELL 135: COMPLETE MATCHING AUDIT")
print("=" * 65)

print("Pose-conformer combinations:", len(complete))
print("Expected combinations: 195")
print("Zero-match combinations:",
      int((~complete["has_spatial_match"]).sum()))

print("\nCONFORMER SUMMARY")
display(conformer_summary)

print("\nPOSE SUMMARY")
display(pose_summary.head(12))

print("\nFAIR LIGAND-LEVEL SUMMARY")
display(ligand_summary)

print("\nSAVED FOUR AUDIT FILES")

print("\nCELL 135 COMPLETED")

CELL 135: COMPLETE MATCHING AUDIT
Pose-conformer combinations: 195
Expected combinations: 195
Zero-match combinations: 11

CONFORMER SUMMARY


,conformer,total_poses,poses_with_matches,total_unique_feature_pairs,mean_normalized_score,pose_match_rate_percent
0,1,39,37,514,0.853070,94.871795
1,2,39,37,439,0.733051,94.871795
2,3,39,37,413,0.689898,94.871795
3,4,39,36,480,0.799732,92.307692
4,5,39,37,468,0.781198,94.871795



POSE SUMMARY


,ligand_identity,pair_idx,pose_index,ligand_feature_count,conformers_with_matches,total_unique_feature_pairs,mean_matches_per_feature,mean_distance_A,conformer_coverage_percent
0,1uk0_frm,1735676,0,18,5,82,0.911111,4.017035,100.0
1,1uk1_frq,1735677,0,17,5,40,0.470588,4.041123,100.0
2,1wok_cnq,1735678,0,12,5,58,0.966667,3.976460,100.0
3,2rcw_aai,1735679,0,14,5,49,0.700000,3.993377,100.0
4,2rd6_78p,1735680,0,12,5,46,0.766667,3.828624,100.0
5,3gjw_gjw,1735681,0,9,5,63,1.400000,3.886316,100.0
6,3gn7_3gn,1735682,0,12,5,50,0.833333,3.874769,100.0
7,3l3l_l3l,1735683,0,11,5,73,1.327273,4.000864,100.0
8,3l3m_a92,1735684,0,17,5,74,0.870588,3.722830,100.0
9,4gv7_mew,1735685,0,8,5,36,0.900000,4.065516,100.0



FAIR LIGAND-LEVEL SUMMARY


,ligand_identity,pair_id,usable_pose_count,mean_pose_score,median_pose_score,mean_pose_coverage,mean_pose_feature_pairs,mean_pose_distance_A
18,4uxb_p34,1735698,1,1.728571,1.728571,100.0,121.00,3.877675
5,3gjw_gjw,1735681,1,1.400000,1.400000,100.0,63.00,3.886316
14,4r5w_xav,1735694,1,1.400000,1.400000,100.0,42.00,4.135113
19,4zzz_fsu,1735699,1,1.345455,1.345455,100.0,74.00,4.058375
7,3l3l_l3l,1735683,1,1.327273,1.327273,100.0,73.00,4.000864
20,5a00_d7n,1735700,1,1.283333,1.283333,100.0,77.00,4.046714
16,4rv6_rpb,1735696,1,1.257143,1.257143,100.0,88.00,4.049225
17,4und_2yq,1735697,1,1.050000,1.050000,100.0,84.00,4.025612
2,1wok_cnq,1735678,1,0.966667,0.966667,100.0,58.00,3.976460
28,5wrz_7u3,1735708,1,0.914286,0.914286,100.0,64.00,4.037118



SAVED FOUR AUDIT FILES

CELL 135 COMPLETED


In [193]:
# ============================================================
# CELL 136: RESIDUE-LEVEL PHARMACOPHORE CONSISTENCY
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")

OUT = ROOT / "member2_outputs/parp1_receptor_validation"

matches = pd.read_csv(
    OUT / "spatial_pharmacophore_matches_5conformers.csv"
)

# ------------------------------------------------------------
# 1. EXTRACT RECEPTOR RESIDUE IDENTITY
# ------------------------------------------------------------

def extract_residue(identity):

    parts = str(identity).split("|")

    if len(parts) < 4:
        return None

    chain = parts[0]
    residue_number = parts[1]
    insertion_code = parts[2]
    residue_name = parts[3]

    return (
        f"{chain}|{residue_number}|"
        f"{insertion_code}|{residue_name}"
    )

matches["receptor_residue"] = (
    matches["receptor_identity"].apply(extract_residue)
)

if matches["receptor_residue"].isna().any():
    raise RuntimeError("Some receptor identities could not be parsed.")

# ------------------------------------------------------------
# 2. COUNT UNIQUE LIGAND-RESIDUE MATCHES
# ------------------------------------------------------------

residue_ligand = matches.drop_duplicates(
    subset=[
        "ligand_identity",
        "conformer",
        "receptor_residue"
    ]
)

# ------------------------------------------------------------
# 3. RESIDUE-LEVEL SUMMARY
# ------------------------------------------------------------

residue_summary = (
    residue_ligand.groupby("receptor_residue")
    .agg(
        unique_ligands=("ligand_identity", "nunique"),
        conformers_with_matches=("conformer", "nunique")
    )
    .reset_index()
)

residue_summary["conformer_coverage_percent"] = (
    residue_summary["conformers_with_matches"] / 5 * 100
)

# Count ligand-residue matches by interaction type
interaction_summary = (
    matches.drop_duplicates(
        subset=[
            "ligand_identity",
            "conformer",
            "receptor_residue",
            "ligand_feature_type",
            "receptor_feature_type"
        ]
    )
    .groupby(
        ["receptor_residue", "ligand_feature_type",
         "receptor_feature_type"]
    )
    .size()
    .reset_index(name="distinct_ligand_conformer_records")
)

residue_summary = residue_summary.sort_values(
    ["unique_ligands", "conformers_with_matches"],
    ascending=False
)

# ------------------------------------------------------------
# 4. RESIDUE CONSISTENCY ACROSS CONFORMERS
# ------------------------------------------------------------

residue_conformer = (
    residue_ligand.groupby(
        ["receptor_residue", "conformer"]
    )
    .agg(
        unique_ligands=("ligand_identity", "nunique")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. SAVE
# ------------------------------------------------------------

residue_summary.to_csv(
    OUT / "residue_level_pharmacophore_consistency.csv",
    index=False
)

interaction_summary.to_csv(
    OUT / "residue_interaction_type_summary.csv",
    index=False
)

residue_conformer.to_csv(
    OUT / "residue_conformer_coverage.csv",
    index=False
)

# ------------------------------------------------------------
# 6. REPORT
# ------------------------------------------------------------

print("=" * 65)
print("CELL 136: RESIDUE-LEVEL PHARMACOPHORE CONSISTENCY")
print("=" * 65)

print("Unique receptor residues matched:",
      residue_summary["receptor_residue"].nunique())

print("\nTOP RECEPTOR RESIDUES")
display(residue_summary.head(20))

print("\nINTERACTION TYPES BY RESIDUE")
display(interaction_summary.sort_values(
    "distinct_ligand_conformer_records",
    ascending=False
).head(25))

print("\nRESIDUE MATCHES BY CONFORMER")
display(
    residue_conformer.pivot(
        index="receptor_residue",
        columns="conformer",
        values="unique_ligands"
    ).fillna(0).astype(int)
)

print("\nSAVED:")
print(OUT / "residue_level_pharmacophore_consistency.csv")
print(OUT / "residue_interaction_type_summary.csv")
print(OUT / "residue_conformer_coverage.csv")

print("\nCELL 136 COMPLETED")

CELL 136: RESIDUE-LEVEL PHARMACOPHORE CONSISTENCY
Unique receptor residues matched: 4

TOP RECEPTOR RESIDUES


,receptor_residue,unique_ligands,conformers_with_matches,conformer_coverage_percent
1,A|907||TYR,32,5,100.0
3,A|988||GLU,6,5,100.0
0,A|903||LYS,3,5,100.0
2,A|909||HIS,1,1,20.0



INTERACTION TYPES BY RESIDUE


,receptor_residue,ligand_feature_type,receptor_feature_type,distinct_ligand_conformer_records
5,A|907||TYR,HYDROPHOBIC,HYDROPHOBIC,160
3,A|907||TYR,AROMATIC,AROMATIC,157
4,A|907||TYR,DONOR,ACCEPTOR,53
2,A|907||TYR,ACCEPTOR,DONOR,39
9,A|988||GLU,HYDROPHOBIC,HYDROPHOBIC,30
1,A|903||LYS,HYDROPHOBIC,HYDROPHOBIC,8
7,A|988||GLU,ACCEPTOR,DONOR,7
0,A|903||LYS,ACCEPTOR,DONOR,5
8,A|988||GLU,DONOR,ACCEPTOR,5
6,A|909||HIS,ACCEPTOR,DONOR,1



RESIDUE MATCHES BY CONFORMER


conformer,1,2,3,4,5
receptor_residue,,,,,
A|903||LYS,3,3,2,2,3
A|907||TYR,32,32,32,32,32
A|909||HIS,0,1,0,0,0
A|988||GLU,6,6,6,6,6



SAVED:
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/residue_level_pharmacophore_consistency.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/residue_interaction_type_summary.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/residue_conformer_coverage.csv

CELL 136 COMPLETED


In [194]:
# ============================================================
# CELL 137: TRANSFORMER TRAINING DATA AUDIT
# ============================================================

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"
PARP_OUT = OUT / "parp1_receptor_validation"

files = {
    "pose_quality": PARP_OUT / "crossdocked_pose_quality_manifest.csv",
    "chemical_identities": OUT / "parp1_crossdocked_pose_chemical_identities.csv",
    "experimental_graphs": OUT / "validated_target_ligand_graphs.jsonl",
    "chembl_graphs": OUT / "chembl37_general_graphs.jsonl",
    "pocket_interface": OUT / "member2_to_member3_interface.json",
    "tokenizer": OUT / "smiles_tokenizer.json"
}

print("=" * 70)
print("CELL 137: TRANSFORMER TRAINING DATA AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK FILE AVAILABILITY
# ------------------------------------------------------------

for name, path in files.items():
    print(f"{name:25s}: {'FOUND' if path.exists() else 'MISSING'}")
    print(" ", path)

# ------------------------------------------------------------
# 2. INSPECT POSE QUALITY MANIFEST
# ------------------------------------------------------------

if not files["pose_quality"].exists():
    raise FileNotFoundError(files["pose_quality"])

pose_df = pd.read_csv(files["pose_quality"])

print("\nPOSE QUALITY MANIFEST")
print("Shape:", pose_df.shape)
print("Columns:", pose_df.columns.tolist())
display(pose_df.head(3))

# ------------------------------------------------------------
# 3. INSPECT CHEMICAL IDENTITIES
# ------------------------------------------------------------

if not files["chemical_identities"].exists():
    raise FileNotFoundError(files["chemical_identities"])

identity_df = pd.read_csv(files["chemical_identities"])

print("\nCHEMICAL IDENTITY DATA")
print("Shape:", identity_df.shape)
print("Columns:", identity_df.columns.tolist())
display(identity_df.head(3))

# ------------------------------------------------------------
# 4. INSPECT EXPERIMENTAL PARP1 GRAPHS
# ------------------------------------------------------------

if not files["experimental_graphs"].exists():
    raise FileNotFoundError(files["experimental_graphs"])

with open(files["experimental_graphs"]) as f:
    experimental_record = json.loads(f.readline())

print("\nEXPERIMENTAL PARP1 GRAPH")
print("Keys:", list(experimental_record.keys()))

for key, value in experimental_record.items():
    if isinstance(value, (list, dict)):
        print(key, ":", type(value).__name__, "length:", len(value))
    else:
        print(key, ":", str(value)[:150])

# ------------------------------------------------------------
# 5. INSPECT GENERAL CHEMBL GRAPHS
# ------------------------------------------------------------

if not files["chembl_graphs"].exists():
    raise FileNotFoundError(files["chembl_graphs"])

with open(files["chembl_graphs"]) as f:
    chembl_record = json.loads(f.readline())

print("\nCHEMBL GRAPH")
print("Keys:", list(chembl_record.keys()))

for key, value in chembl_record.items():
    if isinstance(value, (list, dict)):
        print(key, ":", type(value).__name__, "length:", len(value))
    else:
        print(key, ":", str(value)[:150])

# ------------------------------------------------------------
# 6. INSPECT POCKET INTERFACE
# ------------------------------------------------------------

if files["pocket_interface"].exists():

    with open(files["pocket_interface"]) as f:
        interface = json.load(f)

    print("\nM1 / M3 INTERFACE")
    print("Keys:", list(interface.keys()))

    for key, value in interface.items():
        if isinstance(value, (list, dict)):
            print(key, ":", type(value).__name__, "length:", len(value))
        else:
            print(key, ":", str(value)[:150])

# ------------------------------------------------------------
# 7. INSPECT TOKENIZER
# ------------------------------------------------------------

if files["tokenizer"].exists():

    with open(files["tokenizer"]) as f:
        tokenizer = json.load(f)

    print("\nTOKENIZER")
    print("Keys:", list(tokenizer.keys()))
    print("Vocabulary size:", len(tokenizer["token_to_id"]))
    print("Maximum length:", tokenizer["max_length"])

# ------------------------------------------------------------
# 8. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CELL 137 AUDIT COMPLETED")
print("=" * 70)

print("Usable pose records:", len(pose_df))
print("Unique ligand identities:",
      pose_df["ligand_identity"].nunique()
      if "ligand_identity" in pose_df.columns else "Column missing")

print("Next: construct and validate Transformer training records.")

CELL 137: TRANSFORMER TRAINING DATA AUDIT
pose_quality             : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_pose_quality_manifest.csv
chemical_identities      : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_crossdocked_pose_chemical_identities.csv
experimental_graphs      : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/validated_target_ligand_graphs.jsonl
chembl_graphs            : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_graphs.jsonl
pocket_interface         : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/member2_to_member3_interface.json
tokenizer                : FOUND
  /content/drive/MyDrive/Member2_Project/member2_outputs/smiles_tokenizer.json

POSE QUALITY MANIFEST
Shape: (51, 21)
Columns: ['pair_idx', 'ligand_identity', 'pose_index', 'atom_count', 'bond_count', 'geometry_error_A', 'reopened_validation', 'output_file', 'lig

,pair_idx,ligand_identity,pose_index,atom_count,bond_count,geometry_error_A,reopened_validation,output_file,ligand_atoms,min_distance_A,...,atom_pairs_le_3A,atom_pairs_le_4A,atom_pairs_le_5A,ligand_atoms_le_3A,ligand_atoms_le_4A,ligand_atoms_le_5A,severe_overlap,has_receptor_contacts,pose_quality,usable_for_spatial_screening
0,1735676,1uk0_frm,0,28,31,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,28,2.811002,...,1,101,371,1,26,28,False,True,PASS,True
1,1735677,1uk1_frq,0,29,32,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,29,2.802332,...,1,99,372,1,27,29,False,True,PASS,True
2,1735678,1wok_cnq,0,20,22,1.065814e-14,True,/content/drive/MyDrive/Member2_Project/member2...,20,2.958304,...,2,61,262,2,15,20,False,True,PASS,True



CHEMICAL IDENTITY DATA
Shape: (51, 7)
Columns: ['pair_idx', 'ligand_identity', 'pose_number', 'canonical_smiles', 'inchikey', 'heavy_atoms', 'bonds']


,pair_idx,ligand_identity,pose_number,canonical_smiles,inchikey,heavy_atoms,bonds
0,1735676,1uk0_frm,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,LOFDUAJQRUYHBR-UHFFFAOYSA-N,28,31
1,1735677,1uk1_frq,1,O=c1[nH]c(=O)n(CCCCN2CC=C(c3ccccc3)CC2)c2cccc(...,PNPFDRCIGCUCMN-UHFFFAOYSA-N,29,32
2,1735678,1wok_cnq,1,NC(=O)c1cccc2ncc(-c3ccc(Cl)cc3)nc12,FLYGLPYJEQPCFY-UHFFFAOYSA-N,20,22



EXPERIMENTAL PARP1 GRAPH
Keys: ['graph_id', 'comp_id', 'source', 'source_pdb_ids', 'canonical_smiles', 'smiles_descriptor_type', 'smiles_descriptor_program', 'molecular_formula', 'molecular_name', 'molecular_weight', 'validated_by_rdkit', 'num_atoms', 'num_bonds', 'node_features', 'bonds', 'edge_index', 'edge_features']
graph_id : CCD_6WZ
comp_id : 6WZ
source : RCSB_Chemical_Component_Dictionary
source_pdb_ids : list length: 1
canonical_smiles : C[C@@H]1CN(C(=O)c2cc(Cn3c(=O)[nH]c(=O)c4ccccc43)ccc2F)CCN1CC(F)(F)F
smiles_descriptor_type : SMILES_CANONICAL
smiles_descriptor_program : OPENEYE OETOOLKITS
molecular_formula : C23 H22 F4 N4 O3
molecular_name : 1-[[4-fluoranyl-3-[(3R)-3-methyl-4-[2,2,2-tris(fluoranyl)ethyl]piperazin-1-yl]carbonyl-phenyl]methyl]quinazoline-2,4-dione
molecular_weight : 478.4460000000002
validated_by_rdkit : True
num_atoms : 34
num_bonds : 37
node_features : list length: 34
bonds : list length: 37
edge_index : list length: 74
edge_features : list length: 74

CHEM

In [195]:
# ============================================================
# CELL 138: BUILD POCKET-CONDITIONED TRAINING RECORDS
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"
PARP_OUT = OUT / "parp1_receptor_validation"

# ------------------------------------------------------------
# 1. CHECK REQUIRED IN-MEMORY DATA
# ------------------------------------------------------------

required_variables = [
    "aligned_receptor_features"
]

for name in required_variables:
    if name not in globals():
        raise RuntimeError(
            f"{name} is missing. Run the completed conformer-alignment cell first."
        )

if len(aligned_receptor_features) != 5:
    raise RuntimeError("Expected five aligned M1 conformers.")

# ------------------------------------------------------------
# 2. LOAD VALIDATED POSES AND CHEMICAL IDENTITIES
# ------------------------------------------------------------

pose_df = pd.read_csv(
    PARP_OUT / "crossdocked_pose_quality_manifest.csv"
)

identity_df = pd.read_csv(
    OUT / "parp1_crossdocked_pose_chemical_identities.csv"
)

usable = pose_df[
    (pose_df["usable_for_spatial_screening"] == True) &
    (pose_df["reopened_validation"] == True) &
    (pose_df["severe_overlap"] == False)
].copy()

# Convert zero-based pose index to one-based pose number.
usable["pose_number"] = usable["pose_index"] + 1

merged = usable.merge(
    identity_df,
    on=["pair_idx", "ligand_identity", "pose_number"],
    how="left",
    validate="one_to_one"
)

if merged["canonical_smiles"].isna().any():
    raise RuntimeError("Some usable poses lack validated canonical SMILES.")

if merged["inchikey"].isna().any():
    raise RuntimeError("Some usable poses lack InChIKeys.")

print("Usable poses:", len(merged))
print("Unique ligand identities:", merged["ligand_identity"].nunique())
print("Unique chemical structures:", merged["inchikey"].nunique())

# ------------------------------------------------------------
# 3. BUILD ALIGNED POCKET FEATURE ARRAYS
# ------------------------------------------------------------

feature_id_map = {
    "DONOR": 0,
    "ACCEPTOR": 1,
    "HYDROPHOBIC": 2,
    "AROMATIC": 3,
    "POSITIVE": 4,
    "NEGATIVE": 5
}

pocket_arrays = {}

for conformer_id in sorted(aligned_receptor_features):

    df = aligned_receptor_features[conformer_id].copy()

    required_columns = [
        "feature_index", "feature_type",
        "x", "y", "z"
    ]

    missing = set(required_columns) - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Conformer {conformer_id}: missing columns {missing}"
        )

    df = df.sort_values("feature_index")

    if len(df) != 95:
        raise RuntimeError(
            f"Conformer {conformer_id}: expected 95 features, got {len(df)}"
        )

    if df["feature_index"].duplicated().any():
        raise RuntimeError("Duplicate feature indices detected.")

    if not df["feature_type"].isin(feature_id_map).all():
        raise RuntimeError("Unknown pharmacophore feature type detected.")

    array = np.column_stack([
        df["x"].to_numpy(dtype=np.float32),
        df["y"].to_numpy(dtype=np.float32),
        df["z"].to_numpy(dtype=np.float32),
        df["feature_type"].map(feature_id_map).to_numpy(dtype=np.float32)
    ])

    if not np.isfinite(array).all():
        raise RuntimeError("Non-finite pocket features detected.")

    pocket_arrays[conformer_id] = array

print("\nPocket arrays:")
for cid, array in pocket_arrays.items():
    print(f"Conformer {cid}: {array.shape}")

# ------------------------------------------------------------
# 4. CREATE LIGAND-POCKET TRAINING RECORDS
# ------------------------------------------------------------

records = []

for _, row in merged.iterrows():

    for conformer_id, pocket_array in pocket_arrays.items():

        records.append({
            "pair_idx": int(row["pair_idx"]),
            "ligand_identity": row["ligand_identity"],
            "inchikey": row["inchikey"],
            "pose_index": int(row["pose_index"]),
            "conformer_id": int(conformer_id),
            "canonical_smiles": row["canonical_smiles"],
            "ligand_atoms": int(row["atom_count"]),
            "ligand_bonds": int(row["bond_count"]),
            "pocket_features": pocket_array.tolist()
        })

training_df = pd.DataFrame(records)

expected_records = len(merged) * 5

if len(training_df) != expected_records:
    raise RuntimeError(
        f"Expected {expected_records} records, got {len(training_df)}"
    )

# ------------------------------------------------------------
# 5. SPLIT BY LIGAND IDENTITY
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        training_df,
        groups=training_df["inchikey"]
    )
)

training_df["split"] = "validation"
training_df.loc[train_idx, "split"] = "train"

train_keys = set(training_df.loc[
    training_df["split"] == "train", "inchikey"
])

val_keys = set(training_df.loc[
    training_df["split"] == "validation", "inchikey"
])

if train_keys.intersection(val_keys):
    raise RuntimeError("Data leakage: ligand appears in both splits.")

# ------------------------------------------------------------
# 6. SAVE DATASET
# ------------------------------------------------------------

output_file = OUT / "pocket_conditioned_training_records.jsonl"

with open(output_file, "w") as f:
    for record in training_df.to_dict(orient="records"):
        f.write(json.dumps(record) + "\n")

# Save split manifest without duplicating large pocket arrays.
split_manifest = training_df.drop(
    columns=["pocket_features"]
)

split_manifest.to_csv(
    OUT / "pocket_conditioned_training_manifest.csv",
    index=False
)

# ------------------------------------------------------------
# 7. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 138: TRAINING DATASET SUMMARY")
print("=" * 65)

print("Usable poses:", len(merged))
print("Unique ligands:", merged["inchikey"].nunique())
print("Pocket conformers:", len(pocket_arrays))
print("Total ligand-pocket records:", len(training_df))

print("\nTRAINING")
print("Records:", (training_df["split"] == "train").sum())
print("Unique ligands:", len(train_keys))

print("\nVALIDATION")
print("Records:", (training_df["split"] == "validation").sum())
print("Unique ligands:", len(val_keys))

print("\nLeakage check:", "PASS" if not train_keys.intersection(val_keys) else "FAIL")

print("\nSaved:", output_file)
print("CELL 138 COMPLETED")

Usable poses: 39
Unique ligand identities: 32
Unique chemical structures: 32

Pocket arrays:
Conformer 1: (95, 4)
Conformer 2: (95, 4)
Conformer 3: (95, 4)
Conformer 4: (95, 4)
Conformer 5: (95, 4)

CELL 138: TRAINING DATASET SUMMARY
Usable poses: 39
Unique ligands: 32
Pocket conformers: 5
Total ligand-pocket records: 195

TRAINING
Records: 160
Unique ligands: 25

VALIDATION
Records: 35
Unique ligands: 7

Leakage check: PASS

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/pocket_conditioned_training_records.jsonl
CELL 138 COMPLETED


In [196]:
# ============================================================
# CELL 139: TRANSFORMER INPUT TENSOR PREPARATION
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Member2_Project")
OUT = ROOT / "member2_outputs"
PARP_OUT = OUT / "parp1_receptor_validation"

# ------------------------------------------------------------
# 1. LOAD TOKENIZER
# ------------------------------------------------------------

with open(OUT / "smiles_tokenizer.json") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
MAX_LEN = int(tokenizer["max_length"])

PAD = token_to_id["<PAD>"]
BOS = token_to_id["<BOS>"]
EOS = token_to_id["<EOS>"]
UNK = token_to_id["<UNK>"]

# Greedy tokenization using the existing vocabulary.
# This avoids introducing a different tokenizer.

chemical_tokens = sorted(
    [
        token for token in token_to_id
        if token not in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
    ],
    key=len,
    reverse=True
)

def tokenize_smiles(smiles):

    tokens = []
    position = 0

    while position < len(smiles):

        matched = None

        for token in chemical_tokens:
            if smiles.startswith(token, position):
                matched = token
                break

        if matched is None:
            raise ValueError(
                f"Unknown SMILES token at position {position}: "
                f"{smiles[position:position+10]}"
            )

        tokens.append(matched)
        position += len(matched)

    return tokens


def encode_smiles(smiles):

    tokens = tokenize_smiles(smiles)

    ids = [BOS] + [
        token_to_id[token] for token in tokens
    ] + [EOS]

    if len(ids) > MAX_LEN:
        raise ValueError(
            f"SMILES exceeds maximum length: {len(ids)}"
        )

    ids += [PAD] * (MAX_LEN - len(ids))

    return np.asarray(ids, dtype=np.int64)


# ------------------------------------------------------------
# 2. LOAD UNIQUE PARP1 MOLECULES
# ------------------------------------------------------------

manifest = pd.read_csv(
    OUT / "pocket_conditioned_training_manifest.csv"
)

unique_ligands = (
    manifest[
        ["ligand_identity", "inchikey",
         "canonical_smiles", "split"]
    ]
    .drop_duplicates(subset="inchikey")
    .reset_index(drop=True)
)

# Every molecule must belong to exactly one split.
split_counts = (
    manifest.groupby("inchikey")["split"].nunique()
)

if (split_counts != 1).any():
    raise RuntimeError("A molecule appears in multiple splits.")

print("Unique molecules:", len(unique_ligands))

# ------------------------------------------------------------
# 3. ENCODE MOLECULAR SEQUENCES
# ------------------------------------------------------------

token_sequences = []

for smiles in unique_ligands["canonical_smiles"]:
    token_sequences.append(encode_smiles(smiles))

token_sequences = np.stack(token_sequences)

print("Molecular token tensor:", token_sequences.shape)

# ------------------------------------------------------------
# 4. CREATE COMPLETE FIVE-CONFORMER POCKET ENSEMBLE
# ------------------------------------------------------------

pocket_ensemble = np.stack([
    pocket_arrays[cid]
    for cid in sorted(pocket_arrays)
])

if pocket_ensemble.shape != (5, 95, 4):
    raise RuntimeError(
        f"Unexpected pocket ensemble shape: {pocket_ensemble.shape}"
    )

if not np.isfinite(pocket_ensemble).all():
    raise RuntimeError("Pocket contains invalid values.")

# Repeat the complete pocket ensemble for every molecule.
pocket_tensor = np.repeat(
    pocket_ensemble[np.newaxis, ...],
    len(unique_ligands),
    axis=0
)

# All 95 positions are valid in the current M1 input.
pocket_mask = np.ones(
    (len(unique_ligands), 5, 95),
    dtype=np.bool_
)

print("Pocket tensor:", pocket_tensor.shape)
print("Pocket mask:", pocket_mask.shape)

# ------------------------------------------------------------
# 5. CREATE TRAIN / VALIDATION INDICES
# ------------------------------------------------------------

train_indices = np.where(
    unique_ligands["split"].to_numpy() == "train"
)[0]

val_indices = np.where(
    unique_ligands["split"].to_numpy() == "validation"
)[0]

if set(train_indices).intersection(set(val_indices)):
    raise RuntimeError("Training/validation leakage detected.")

# ------------------------------------------------------------
# 6. VERIFY GENERAL CHEMBL PRETRAINING ARRAYS
# ------------------------------------------------------------

chembl_train_path = OUT / "chembl_train_tokens.npy"
chembl_val_path = OUT / "chembl_val_tokens.npy"

if not chembl_train_path.exists():
    raise FileNotFoundError(chembl_train_path)

if not chembl_val_path.exists():
    raise FileNotFoundError(chembl_val_path)

chembl_train = np.load(chembl_train_path, mmap_mode="r")
chembl_val = np.load(chembl_val_path, mmap_mode="r")

print("\nCHEMBL PRETRAINING DATA")
print("Training:", chembl_train.shape)
print("Validation:", chembl_val.shape)

if chembl_train.shape[1] != MAX_LEN:
    raise RuntimeError("ChEMBL training length mismatch.")

if chembl_val.shape[1] != MAX_LEN:
    raise RuntimeError("ChEMBL validation length mismatch.")

# ------------------------------------------------------------
# 7. SAVE TRANSFORMER INPUTS
# ------------------------------------------------------------

np.savez_compressed(
    OUT / "transformer_parp1_inputs.npz",
    token_sequences=token_sequences,
    pocket_tensor=pocket_tensor,
    pocket_mask=pocket_mask,
    train_indices=train_indices,
    val_indices=val_indices,
    ligand_identities=unique_ligands[
        "ligand_identity"
    ].to_numpy(dtype=str),
    inchikeys=unique_ligands[
        "inchikey"
    ].to_numpy(dtype=str),
    smiles=unique_ligands[
        "canonical_smiles"
    ].to_numpy(dtype=str)
)

# ------------------------------------------------------------
# 8. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 139: TRANSFORMER INPUT PREPARATION")
print("=" * 65)

print("Unique molecules:", len(unique_ligands))
print("Training molecules:", len(train_indices))
print("Validation molecules:", len(val_indices))
print("Token vocabulary:", len(token_to_id))
print("Maximum sequence length:", MAX_LEN)
print("Pocket ensemble:", pocket_tensor.shape)

print("\nChEMBL pretraining:")
print("Training sequences:", len(chembl_train))
print("Validation sequences:", len(chembl_val))

print("\nSaved:")
print(OUT / "transformer_parp1_inputs.npz")

print("\nCELL 139 COMPLETED")

Unique molecules: 32
Molecular token tensor: (32, 143)
Pocket tensor: (32, 5, 95, 4)
Pocket mask: (32, 5, 95)

CHEMBL PRETRAINING DATA
Training: (45000, 143)
Validation: (5000, 143)

CELL 139: TRANSFORMER INPUT PREPARATION
Unique molecules: 32
Training molecules: 25
Validation molecules: 7
Token vocabulary: 81
Maximum sequence length: 143
Pocket ensemble: (32, 5, 95, 4)

ChEMBL pretraining:
Training sequences: 45000
Validation sequences: 5000

Saved:
/content/drive/MyDrive/Member2_Project/member2_outputs/transformer_parp1_inputs.npz

CELL 139 COMPLETED


In [197]:
# ============================================================
# CELL 140: TRANSFORMER ARCHITECTURE AND CHECKPOINT AUDIT
# ============================================================

import torch
import torch.nn as nn
import inspect
import os

print("=" * 65)
print("CELL 140: TRANSFORMER ARCHITECTURE AUDIT")
print("=" * 65)

# 1. Find existing PyTorch models
print("\n1. EXISTING MODEL OBJECTS")

found_models = {}

for name, obj in globals().copy().items():
    if isinstance(obj, nn.Module):
        found_models[name] = obj
        print(f"\nModel variable: {name}")
        print(f"Class: {obj.__class__.__name__}")
        print(f"Parameters: {sum(p.numel() for p in obj.parameters()):,}")
        print(f"Forward signature: {inspect.signature(obj.forward)}")

# 2. Inspect model class definitions
print("\n2. TRANSFORMER CLASS DEFINITIONS")

for name, obj in globals().copy().items():
    if inspect.isclass(obj):
        try:
            if issubclass(obj, nn.Module):
                print(f"\nClass: {name}")
                print(f"Forward: {inspect.signature(obj.forward)}")
        except TypeError:
            pass

# 3. Inspect existing GRU checkpoint
print("\n3. EXISTING GRU CHECKPOINT")

checkpoint_path = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "molecular_gru_best.pt"
)

if os.path.exists(checkpoint_path):

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint found:", checkpoint_path)
    print("Checkpoint type:", type(checkpoint))

    if isinstance(checkpoint, dict):
        print("Checkpoint keys:", list(checkpoint.keys()))

        state = checkpoint.get(
            "model_state_dict",
            checkpoint.get("state_dict", checkpoint)
        )

        if isinstance(state, dict):
            print("\nFirst 15 parameter names:")
            for key in list(state.keys())[:15]:
                value = state[key]
                if torch.is_tensor(value):
                    print(f"{key}: {tuple(value.shape)}")
                else:
                    print(f"{key}: {type(value)}")

else:
    print("GRU checkpoint not found.")

# 4. Inspect pocket encoder
print("\n4. POCKET ENCODER")

for name, obj in found_models.items():
    if "pocket" in name.lower() or "encoder" in name.lower():
        print(f"\n{name}: {obj.__class__.__name__}")
        print(obj)

# 5. Summary
print("\n5. AUDIT SUMMARY")
print("Models discovered:", list(found_models.keys()))
print("GRU checkpoint exists:", os.path.exists(checkpoint_path))

print("\nCELL 140 AUDIT COMPLETED")

CELL 140: TRANSFORMER ARCHITECTURE AUDIT

1. EXISTING MODEL OBJECTS

Model variable: criterion
Class: CrossEntropyLoss
Parameters: 0
Forward signature: (input: torch.Tensor, target: torch.Tensor) -> torch.Tensor

Model variable: pocket_encoder
Class: DynamicPocketEncoder
Parameters: 50,944
Forward signature: (coordinates, feature_ids, mask)

Model variable: molecular_transformer
Class: PocketConditionedTransformer
Parameters: 832,849
Forward signature: (token_ids, pocket_memory, token_padding_mask, pocket_padding_mask)

2. TRANSFORMER CLASS DEFINITIONS

Class: MolecularGRU
Forward: (self, x)

Class: DynamicPocketEncoder
Forward: (self, coordinates, feature_ids, mask)

Class: PositionalEncoding
Forward: (self, x)

Class: PocketConditionedTransformer
Forward: (self, token_ids, pocket_memory, token_padding_mask, pocket_padding_mask)

3. EXISTING GRU CHECKPOINT
Checkpoint found: /content/drive/MyDrive/Member2_Project/checkpoints/molecular_gru_best.pt
Checkpoint type: <class 'dict'>
Checkpo

In [199]:
# ============================================================
# CELL 141B: RECOVER MODEL DEFINITIONS FROM COLAB HISTORY
# ============================================================

print("=" * 65)
print("RECOVERING MODEL DEFINITIONS")
print("=" * 65)

for keyword in [
    "class DynamicPocketEncoder",
    "class PocketConditionedTransformer"
]:

    print(f"\n\n{'=' * 65}")
    print("SEARCHING FOR:", keyword)
    print("=" * 65)

    found = False

    for cell_number, code in enumerate(In):

        if keyword in code:

            print(f"\nFound in executed cell: {cell_number}")
            print("-" * 65)
            print(code)
            found = True

    if not found:
        print("Definition not found in execution history.")

print("\n\nTRANSFORMER MODULE STRUCTURE")
print("=" * 65)

print(molecular_transformer)

print("\nNAMED MODULES")

for name, module in molecular_transformer.named_modules():

    if name:
        print(name, ":", module.__class__.__name__)

print("\nCELL 141B COMPLETED")

RECOVERING MODEL DEFINITIONS


SEARCHING FOR: class DynamicPocketEncoder

Found in executed cell: 143
-----------------------------------------------------------------
# ============================================================
# CELL 91: DYNAMIC M1 POCKET ENCODER
# ============================================================

import torch
import torch.nn as nn
import numpy as np

print("=" * 65)
print("CELL 91: DYNAMIC M1 POCKET ENCODER")
print("=" * 65)

# ------------------------------------------------------------
# 1. DEVICE
# ------------------------------------------------------------

device = torch.device("cpu")

print("Device:", device)

# ------------------------------------------------------------
# 2. POCKET ENCODER
# ------------------------------------------------------------

class DynamicPocketEncoder(nn.Module):

    def __init__(
        self,
        num_feature_types,
        embedding_dim=128
    ):
        super().__init__()

        self.embedding_dim = embed

In [200]:
# ============================================================
# CELL 142: POCKET FEATURE ENCODING AUDIT
# ============================================================

import numpy as np
import torch

print("=" * 65)
print("CELL 142: POCKET FEATURE ENCODING AUDIT")
print("=" * 65)

# 1. Inspect the pocket tensor
data = np.load(
    os.path.join(
        OUTPUT_DIR,
        "transformer_parp1_inputs.npz"
    ),
    allow_pickle=True
)

pocket_tensor = data["pocket_tensor"]

print("\nPocket tensor:", pocket_tensor.shape)

# 2. Unique feature IDs
feature_ids = pocket_tensor[..., 3].astype(int)

print("\nFeature IDs in pocket tensor:")
print(np.unique(feature_ids, return_counts=True))

# 3. Inspect M1 feature metadata
metadata_path = os.path.join(
    MEMBER1_INPUT_DIR,
    "pharmacophore_tensors.json"
)

print("\nFeature vocabulary from encoder:")

if "feature_to_id" in globals():
    print(feature_to_id)
else:
    print("feature_to_id not found.")

# 4. Inspect M1 metadata
import json

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("\nM1 metadata keys:")
print(metadata.keys())

print("\n5. Encoder embedding configuration")

print("Embedding vocabulary size:",
      pocket_encoder.feature_embedding.num_embeddings)

print("Embedding dimension:",
      pocket_encoder.feature_embedding.embedding_dim)

print("\n6. Feature ID range check")

valid_ids = (
    feature_ids.min() >= 0 and
    feature_ids.max() <
    pocket_encoder.feature_embedding.num_embeddings
)

print("All feature IDs fit encoder:",
      "PASS" if valid_ids else "FAIL")

print("\nCELL 142 COMPLETED")

CELL 142: POCKET FEATURE ENCODING AUDIT


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/transformer_parp1_inputs.npz'

In [201]:
# ============================================================
# CELL 142: POCKET FEATURE ENCODING AUDIT
# ============================================================

import os
import json
import numpy as np
import torch

print("=" * 65)
print("CELL 142: POCKET FEATURE ENCODING AUDIT")
print("=" * 65)

# 1. Correct input path
input_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "transformer_parp1_inputs.npz"
)

assert os.path.exists(input_path), (
    f"Input file not found: {input_path}"
)

data = np.load(input_path, allow_pickle=True)

pocket_tensor = data["pocket_tensor"]
pocket_mask = data["pocket_mask"]

print("\n1. POCKET INPUT")
print("Pocket tensor:", pocket_tensor.shape)
print("Pocket mask:", pocket_mask.shape)

# 2. Extract feature IDs only from valid positions
valid_positions = pocket_mask.astype(bool)

feature_ids = pocket_tensor[..., 3][valid_positions].astype(int)

unique_ids, counts = np.unique(
    feature_ids,
    return_counts=True
)

print("\n2. FEATURE ID DISTRIBUTION")

for fid, count in zip(unique_ids, counts):
    print(f"Feature ID {fid}: {count}")

# 3. Inspect M1 feature metadata
metadata_path = os.path.join(
    PROJECT_ROOT,
    "member1_input",
    "module1 parp1 files",
    "pharmacophore_tensors.json"
)

print("\n3. M1 METADATA")

if os.path.exists(metadata_path):

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    print("Metadata keys:", list(metadata.keys()))

else:
    print("Metadata JSON not found:", metadata_path)

# 4. Inspect encoder configuration
print("\n4. POCKET ENCODER")

embedding_size = (
    pocket_encoder.feature_embedding.num_embeddings
)

print("Embedding vocabulary size:", embedding_size)
print("Embedding dimension:",
      pocket_encoder.feature_embedding.embedding_dim)

# 5. Validate feature IDs
valid_ids = (
    len(feature_ids) > 0
    and feature_ids.min() >= 0
    and feature_ids.max() < embedding_size
)

print("\n5. FEATURE ID VALIDATION")
print("Minimum feature ID:", feature_ids.min())
print("Maximum feature ID:", feature_ids.max())
print("All IDs fit encoder:", "PASS" if valid_ids else "FAIL")

assert valid_ids, "Feature IDs are incompatible with the encoder."

print("\nCELL 142 COMPLETED")

CELL 142: POCKET FEATURE ENCODING AUDIT

1. POCKET INPUT
Pocket tensor: (32, 5, 95, 4)
Pocket mask: (32, 5, 95)

2. FEATURE ID DISTRIBUTION
Feature ID 0: 3520
Feature ID 1: 2880
Feature ID 2: 7200
Feature ID 3: 640
Feature ID 4: 800
Feature ID 5: 160

3. M1 METADATA
Metadata keys: ['1']

4. POCKET ENCODER
Embedding vocabulary size: 6
Embedding dimension: 128

5. FEATURE ID VALIDATION
Minimum feature ID: 0
Maximum feature ID: 5
All IDs fit encoder: PASS

CELL 142 COMPLETED


In [202]:
# ============================================================
# CELL 143: VERIFY PHARMACOPHORE FEATURE-ID MAPPING
# ============================================================

print("=" * 65)
print("CELL 143: FEATURE MAPPING CONSISTENCY")
print("=" * 65)

print("\n1. ENCODER FEATURE VOCABULARY")

print("feature_to_id:", feature_to_id)

id_to_feature = {
    value: key
    for key, value in feature_to_id.items()
}

print("id_to_feature:", id_to_feature)

print("\n2. M1 METADATA STRUCTURE")

metadata_path = os.path.join(
    PROJECT_ROOT,
    "member1_input",
    "module1 parp1 files",
    "pharmacophore_tensors.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

pocket_metadata = metadata["1"]

print("Pocket metadata type:", type(pocket_metadata))

if isinstance(pocket_metadata, dict):

    print("Pocket metadata keys:", list(pocket_metadata.keys()))

    for key, value in pocket_metadata.items():

        if isinstance(value, list):
            print(f"\n{key}: {len(value)} entries")
            print("First 3:", value[:3])

        elif isinstance(value, dict):
            print(f"\n{key}:")
            print(value)

        else:
            print(f"\n{key}: {value}")

print("\n3. CURRENT TENSOR FEATURE COUNTS")

unique_ids, counts = np.unique(
    feature_ids,
    return_counts=True
)

for fid, count in zip(unique_ids, counts):
    print(f"Tensor ID {fid}: {count}")

print("\nCELL 143 COMPLETED")

CELL 143: FEATURE MAPPING CONSISTENCY

1. ENCODER FEATURE VOCABULARY
feature_to_id: {'ACCEPTOR': 0, 'AROMATIC': 1, 'DONOR': 2, 'HYDROPHOBIC': 3, 'NEGATIVE': 4, 'POSITIVE': 5}
id_to_feature: {0: 'ACCEPTOR', 1: 'AROMATIC', 2: 'DONOR', 3: 'HYDROPHOBIC', 4: 'NEGATIVE', 5: 'POSITIVE'}

2. M1 METADATA STRUCTURE
Pocket metadata type: <class 'dict'>
Pocket metadata keys: ['pocket_id', 'num_conformations', 'num_features', 'tensor_shape', 'mask_shape', 'feature_tensor', 'mask', 'feature_metadata']

pocket_id: 1

num_conformations: 5

num_features: 95

tensor_shape: 3 entries
First 3: [5, 95, 4]

mask_shape: 2 entries
First 3: [5, 95]

feature_tensor: 5 entries
First 3: [[[25.07200050354004, 40.76900100708008, -2.4719998836517334, 2.0], [24.82200050354004, 40.86199951171875, 0.14559999108314514, 3.0], [24.82200050354004, 40.86199951171875, 0.14559999108314514, 4.0], [27.454999923706055, 40.000999450683594, -2.430000066757202, 0.0], [24.573999404907227, 41.90599822998047, -0.3050000071525574, 0.0]

In [203]:
# ============================================================
# CELL 144: M1 TO ENCODER FEATURE-ID MAPPING
# ============================================================

print("=" * 65)
print("CELL 144: FEATURE-ID MAPPING VALIDATION")
print("=" * 65)

# 1. Load M1 metadata
metadata_path = os.path.join(
    PROJECT_ROOT,
    "member1_input",
    "module1 parp1 files",
    "pharmacophore_tensors.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

feature_metadata = metadata["1"]["feature_metadata"]

# 2. Recover M1's original feature-ID mapping
m1_id_to_feature = {}

for item in feature_metadata:

    original_id = int(item["feature_id"])
    feature_name = item["feature"]

    if original_id in m1_id_to_feature:
        assert m1_id_to_feature[original_id] == feature_name, (
            f"Inconsistent M1 mapping for ID {original_id}"
        )

    m1_id_to_feature[original_id] = feature_name

# 3. Reverse encoder vocabulary
encoder_id_to_feature = {
    value: key
    for key, value in feature_to_id.items()
}

# 4. Create explicit conversion mapping
m1_to_encoder_id = {}

for m1_id, feature_name in m1_id_to_feature.items():

    assert feature_name in feature_to_id, (
        f"Unknown feature: {feature_name}"
    )

    m1_to_encoder_id[m1_id] = feature_to_id[feature_name]

print("\nM1 ID -> Feature -> Encoder ID")

for m1_id in sorted(m1_to_encoder_id):

    name = m1_id_to_feature[m1_id]
    encoder_id = m1_to_encoder_id[m1_id]

    print(f"{m1_id} -> {name} -> {encoder_id}")

# 5. Verify all six feature types
assert len(m1_to_encoder_id) == 6
assert set(m1_to_encoder_id.keys()) == set(range(6))
assert set(m1_to_encoder_id.values()) == set(range(6))

print("\nAll six feature types mapped:", "PASS")

# 6. Verify against actual pocket tensor
raw_ids = pocket_tensor[..., 3].astype(int)

mapped_ids = np.vectorize(
    m1_to_encoder_id.__getitem__
)(raw_ids)

print("\nOriginal M1 IDs:", np.unique(raw_ids))
print("Converted encoder IDs:", np.unique(mapped_ids))

print("\nMapping validation completed.")

CELL 144: FEATURE-ID MAPPING VALIDATION

M1 ID -> Feature -> Encoder ID
0 -> DONOR -> 2
1 -> ACCEPTOR -> 0
2 -> HYDROPHOBIC -> 3
3 -> AROMATIC -> 1
4 -> POSITIVE -> 5
5 -> NEGATIVE -> 4

All six feature types mapped: PASS

Original M1 IDs: [0 1 2 3 4 5]
Converted encoder IDs: [0 1 2 3 4 5]

Mapping validation completed.


In [204]:
# ============================================================
# CELL 145: CONVERT M1 FEATURE IDS FOR POCKET ENCODER
# ============================================================

import os
import numpy as np

print("=" * 65)
print("CELL 145: POCKET FEATURE-ID CONVERSION")
print("=" * 65)

input_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "transformer_parp1_inputs.npz"
)

data = np.load(input_path, allow_pickle=True)

# Preserve original data
pocket_tensor_original = data["pocket_tensor"].copy()

# Create converted copy
pocket_tensor_corrected = pocket_tensor_original.copy()

# Convert only feature-ID channel
for m1_id, encoder_id in m1_to_encoder_id.items():

    pocket_tensor_corrected[
        ..., 3
    ][pocket_tensor_original[..., 3] == m1_id] = encoder_id

# Validate conversion
converted_ids = pocket_tensor_corrected[..., 3].astype(int)

for m1_id, encoder_id in m1_to_encoder_id.items():

    original_count = np.sum(
        pocket_tensor_original[..., 3] == m1_id
    )

    converted_count = np.sum(
        converted_ids == encoder_id
    )

    assert original_count == converted_count

print("\nOriginal tensor:", pocket_tensor_original.shape)
print("Corrected tensor:", pocket_tensor_corrected.shape)

print("\nConverted feature distribution:")

for fid, count in zip(
    *np.unique(converted_ids, return_counts=True)
):
    print(f"Encoder ID {fid}: {count}")

# Save separate corrected artifact
corrected_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "transformer_parp1_inputs_corrected.npz"
)

save_data = {
    key: data[key]
    for key in data.files
}

save_data["pocket_tensor"] = pocket_tensor_corrected

np.savez_compressed(
    corrected_path,
    **save_data
)

print("\nSaved:", corrected_path)
print("\nOriginal file preserved:", os.path.exists(input_path))
print("CELL 145 COMPLETED")

CELL 145: POCKET FEATURE-ID CONVERSION

Original tensor: (32, 5, 95, 4)
Corrected tensor: (32, 5, 95, 4)

Converted feature distribution:
Encoder ID 0: 2880
Encoder ID 1: 640
Encoder ID 2: 3520
Encoder ID 3: 7200
Encoder ID 4: 160
Encoder ID 5: 800

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/transformer_parp1_inputs_corrected.npz

Original file preserved: True
CELL 145 COMPLETED


In [205]:
# ============================================================
# CELL 146: ENCODE CORRECTED M1 POCKET ENSEMBLE
# ============================================================

import os
import numpy as np
import torch

print("=" * 65)
print("CELL 146: POCKET ENCODING")
print("=" * 65)

# 1. Load corrected inputs
corrected_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "transformer_parp1_inputs_corrected.npz"
)

data = np.load(corrected_path, allow_pickle=True)

pocket_tensor = data["pocket_tensor"].astype(np.float32)
pocket_mask = data["pocket_mask"].astype(bool)

# 2. Separate coordinates and feature IDs
coordinates = pocket_tensor[..., :3]
feature_ids = pocket_tensor[..., 3].astype(np.int64)

# 3. Convert to PyTorch
coordinates_tensor = torch.tensor(
    coordinates,
    dtype=torch.float32
)

feature_ids_tensor = torch.tensor(
    feature_ids,
    dtype=torch.long
)

mask_tensor = torch.tensor(
    pocket_mask,
    dtype=torch.bool
)

# 4. Normalize coordinates consistently
# Use one shared normalization across the pocket ensemble.
valid_coordinates = coordinates_tensor[mask_tensor]

coordinate_mean = valid_coordinates.mean(dim=0)
coordinate_std = valid_coordinates.std(dim=0).clamp(min=1e-6)

coordinates_normalized = (
    coordinates_tensor - coordinate_mean
) / coordinate_std

# 5. Encode pocket features
pocket_encoder.eval()

with torch.no_grad():

    encoded_pocket = pocket_encoder(
        coordinates_normalized,
        feature_ids_tensor,
        mask_tensor
    )

# 6. Validate
print("\nINPUT SHAPES")
print("Coordinates:", tuple(coordinates_tensor.shape))
print("Feature IDs:", tuple(feature_ids_tensor.shape))
print("Mask:", tuple(mask_tensor.shape))

print("\nOUTPUT")
print("Encoded pocket:", tuple(encoded_pocket.shape))

checks = {
    "32 molecules represented":
        encoded_pocket.shape[0] == 32,

    "Five conformers preserved":
        encoded_pocket.shape[1] == 5,

    "95 features preserved":
        encoded_pocket.shape[2] == 95,

    "Embedding dimension is 128":
        encoded_pocket.shape[3] == 128,

    "All embeddings finite":
        torch.isfinite(encoded_pocket).all().item()
}

print("\nVALIDATION")

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values())

# 7. Save encoded representation
encoded_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "encoded_parp1_pocket_ensemble.pt"
)

torch.save({
    "pocket_embeddings": encoded_pocket,
    "pocket_mask": mask_tensor,
    "coordinate_mean": coordinate_mean,
    "coordinate_std": coordinate_std,
    "feature_mapping": m1_to_encoder_id
}, encoded_path)

print("\nSaved:", encoded_path)
print("\nCELL 146 COMPLETED")

CELL 146: POCKET ENCODING

INPUT SHAPES
Coordinates: (32, 5, 95, 3)
Feature IDs: (32, 5, 95)
Mask: (32, 5, 95)

OUTPUT
Encoded pocket: (32, 5, 95, 128)

VALIDATION
32 molecules represented: PASS
Five conformers preserved: PASS
95 features preserved: PASS
Embedding dimension is 128: PASS
All embeddings finite: PASS

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/encoded_parp1_pocket_ensemble.pt

CELL 146 COMPLETED


In [206]:
# ============================================================
# CELL 147: GRU AND TRANSFORMER TOKENIZER COMPATIBILITY
# ============================================================

import os
import json
import torch

print("=" * 65)
print("CELL 147: TOKENIZER COMPATIBILITY AUDIT")
print("=" * 65)

tokenizer_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "smiles_tokenizer.json"
)

gru_path = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "molecular_gru_best.pt"
)

# Load tokenizer
with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

print("\nTOKENIZER FILE")
print("Keys:", list(tokenizer_data.keys()))

# Load GRU checkpoint
checkpoint = torch.load(
    gru_path,
    map_location="cpu",
    weights_only=False
)

print("\nGRU CHECKPOINT")
print("Vocabulary size:", checkpoint.get("vocab_size"))
print("Embedding dimension:", checkpoint.get("embedding_dim"))
print("Tokenizer path:", checkpoint.get("tokenizer_path"))

# Inspect vocabulary structure
vocab = tokenizer_data.get("token_to_id")

if vocab is None:
    vocab = tokenizer_data.get("vocab")

if vocab is None:
    raise ValueError(
        "Could not identify token vocabulary in tokenizer JSON."
    )

print("\nTRANSFORMER TOKENIZER")
print("Vocabulary size:", len(vocab))

# Inspect embedding compatibility
embedding = checkpoint["model_state_dict"].get(
    "embedding.weight"
)

assert embedding is not None, "GRU embedding weights not found."

print("\nGRU TOKEN EMBEDDING")
print("Shape:", tuple(embedding.shape))

print("\nCOMPATIBILITY CHECKS")

print(
    "Vocabulary size matches:",
    len(vocab) == embedding.shape[0]
)

print(
    "Embedding dimension matches:",
    embedding.shape[1] == 128
)

assert len(vocab) == embedding.shape[0]
assert embedding.shape[1] == 128

print("\nToken examples:")
for token, token_id in list(vocab.items())[:15]:
    print(f"{token!r}: {token_id}")

print("\nCELL 147 COMPLETED")

CELL 147: TOKENIZER COMPATIBILITY AUDIT

TOKENIZER FILE
Keys: ['vocabulary', 'token_to_id', 'max_length', 'special_tokens']

GRU CHECKPOINT
Vocabulary size: 81
Embedding dimension: 128
Tokenizer path: /content/drive/MyDrive/Member2_Project/member2_outputs/smiles_tokenizer.json

TRANSFORMER TOKENIZER
Vocabulary size: 81

GRU TOKEN EMBEDDING
Shape: (81, 128)

COMPATIBILITY CHECKS
Vocabulary size matches: True
Embedding dimension matches: True

Token examples:
'<PAD>': 0
'<BOS>': 1
'<EOS>': 2
'<UNK>': 3
'#': 4
'%10': 5
'(': 6
')': 7
'-': 8
'/': 9
'1': 10
'2': 11
'3': 12
'4': 13
'5': 14

CELL 147 COMPLETED


In [207]:
# ============================================================
# CELL 148: CHECKPOINT TOKENIZER METADATA AUDIT
# ============================================================

print("=" * 65)
print("CELL 148: GRU CHECKPOINT METADATA AUDIT")
print("=" * 65)

print("\nCHECKPOINT KEYS:")
for key in checkpoint.keys():
    print("-", key)

print("\nTOKENIZER METADATA:")

for key in ["token_to_id", "vocabulary", "tokenizer", "vocab"]:
    if key in checkpoint:
        print(f"\nFound: {key}")
        print(str(checkpoint[key])[:1000])

print("\nTOKEN ID VALIDATION:")

token_ids = list(vocab.values())

print("Unique token IDs:", len(set(token_ids)))
print("Minimum ID:", min(token_ids))
print("Maximum ID:", max(token_ids))

assert len(set(token_ids)) == 81
assert sorted(token_ids) == list(range(81))

print("\nAll tokenizer IDs are unique and contiguous: PASS")
print("\nCELL 148 COMPLETED")

CELL 148: GRU CHECKPOINT METADATA AUDIT

CHECKPOINT KEYS:
- model_state_dict
- vocab_size
- embedding_dim
- hidden_dim
- num_layers
- tokenizer_path
- training_epoch
- validation_loss

TOKENIZER METADATA:

TOKEN ID VALIDATION:
Unique token IDs: 81
Minimum ID: 0
Maximum ID: 80

All tokenizer IDs are unique and contiguous: PASS

CELL 148 COMPLETED


In [208]:
# ============================================================
# CELL 149: TRANSFER GRU TOKEN EMBEDDINGS
# ============================================================

print("=" * 65)
print("CELL 149: GRU EMBEDDING TRANSFER")
print("=" * 65)

# Extract GRU embedding matrix
gru_embeddings = checkpoint["model_state_dict"]["embedding.weight"]

# Get Transformer token embedding layer
transformer_embeddings = molecular_transformer.token_embedding

# Validate dimensions
assert gru_embeddings.shape == transformer_embeddings.weight.shape

# Preserve original Transformer embeddings for comparison
original_embeddings = transformer_embeddings.weight.detach().clone()

# Transfer weights
with torch.no_grad():
    transformer_embeddings.weight.copy_(gru_embeddings)

# Validate transfer
transfer_difference = torch.max(
    torch.abs(
        transformer_embeddings.weight -
        gru_embeddings
    )
).item()

print("\nGRU embedding shape:", tuple(gru_embeddings.shape))
print("Transformer embedding shape:",
      tuple(transformer_embeddings.weight.shape))

print("Maximum transfer difference:", transfer_difference)

assert transfer_difference == 0.0

print("\nEmbedding transfer: PASS")
print("GRU recurrent layers transferred: NO")
print("Transformer architecture preserved: YES")

# Save transfer checkpoint
transfer_path = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "transformer_gru_embedding_initialization.pt"
)

torch.save({
    "token_embedding": transformer_embeddings.weight.detach().cpu(),
    "source_checkpoint": gru_path,
    "tokenizer_path": tokenizer_path,
    "vocabulary_size": len(vocab),
    "embedding_dimension": 128,
    "historical_tokenizer_verified": False
}, transfer_path)

print("\nSaved:", transfer_path)
print("\nCELL 149 COMPLETED")

CELL 149: GRU EMBEDDING TRANSFER

GRU embedding shape: (81, 128)
Transformer embedding shape: (81, 128)
Maximum transfer difference: 0.0

Embedding transfer: PASS
GRU recurrent layers transferred: NO
Transformer architecture preserved: YES

Saved: /content/drive/MyDrive/Member2_Project/checkpoints/transformer_gru_embedding_initialization.pt

CELL 149 COMPLETED


In [210]:
# ============================================================
# CELL 150: RELOAD AND AUDIT TRAINING DATA
# ============================================================

import os
import numpy as np
import torch

print("=" * 65)
print("CELL 150: TRAINING READINESS AUDIT")
print("=" * 65)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "member2_outputs")

# 1. Reload ChEMBL token arrays
chembl_train_path = os.path.join(
    OUTPUT_DIR, "chembl_train_tokens.npy"
)

chembl_val_path = os.path.join(
    OUTPUT_DIR, "chembl_val_tokens.npy"
)

assert os.path.exists(chembl_train_path)
assert os.path.exists(chembl_val_path)

chembl_train_tokens = np.load(chembl_train_path)
chembl_val_tokens = np.load(chembl_val_path)

print("\nCHEMBL DATA")
print("Training:", chembl_train_tokens.shape)
print("Validation:", chembl_val_tokens.shape)

# 2. Reload PARP1 training inputs
parp1_path = os.path.join(
    OUTPUT_DIR, "transformer_parp1_inputs.npz"
)

parp1_data = np.load(parp1_path, allow_pickle=True)

token_sequences = parp1_data["token_sequences"]
train_indices = np.asarray(parp1_data["train_indices"])
val_indices = np.asarray(parp1_data["val_indices"])

print("\nPARP1 DATA")
print("Unique ligand sequences:", token_sequences.shape)
print("Training ligands:", len(train_indices))
print("Validation ligands:", len(val_indices))

# 3. Reload encoded pocket representations
encoded_path = os.path.join(
    OUTPUT_DIR, "encoded_parp1_pocket_ensemble.pt"
)

encoded_data = torch.load(
    encoded_path,
    map_location="cpu",
    weights_only=False
)

encoded_pocket = encoded_data["pocket_embeddings"]

print("\nPOCKET REPRESENTATIONS")
print("Encoded pocket:", tuple(encoded_pocket.shape))

# 4. Model parameters
print("\nMODEL PARAMETERS")

pocket_params = sum(
    p.numel() for p in pocket_encoder.parameters()
)

transformer_params = sum(
    p.numel() for p in molecular_transformer.parameters()
)

print("Pocket encoder:", pocket_params)
print("Transformer:", transformer_params)
print("Total:", pocket_params + transformer_params)

# 5. Basic validation
assert chembl_train_tokens.shape[0] == 45000
assert chembl_val_tokens.shape[0] == 5000
assert token_sequences.shape[0] == 32
assert encoded_pocket.shape == (32, 5, 95, 128)

print("\nALL DATA RELOADED AND VALIDATED")
print("CELL 150 COMPLETED")

CELL 150: TRAINING READINESS AUDIT

CHEMBL DATA
Training: (45000, 143)
Validation: (5000, 143)

PARP1 DATA
Unique ligand sequences: (32, 143)
Training ligands: 25
Validation ligands: 7

POCKET REPRESENTATIONS
Encoded pocket: (32, 5, 95, 128)

MODEL PARAMETERS
Pocket encoder: 50944
Transformer: 832849
Total: 883793

ALL DATA RELOADED AND VALIDATED
CELL 150 COMPLETED


In [211]:
# ============================================================
# CELL 151: PARP1 SPLIT INTEGRITY AUDIT
# ============================================================

import json
import os
import numpy as np

print("=" * 65)
print("CELL 151: PARP1 SPLIT INTEGRITY")
print("=" * 65)

manifest_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "pocket_conditioned_training_manifest.csv"
)

import pandas as pd

manifest = pd.read_csv(manifest_path)

print("\nMANIFEST SHAPE:", manifest.shape)
print("COLUMNS:", manifest.columns.tolist())

# Inspect unique ligand identities and their assigned splits
identity_col = "inchikey"

assert identity_col in manifest.columns

split_col = "split"

assert split_col in manifest.columns

split_summary = (
    manifest.groupby(split_col)[identity_col]
    .nunique()
)

print("\nUNIQUE LIGAND IDENTITIES BY SPLIT:")
print(split_summary)

train_keys = set(
    manifest.loc[
        manifest[split_col] == "train",
        identity_col
    ]
)

val_keys = set(
    manifest.loc[
        manifest[split_col] == "validation",
        identity_col
    ]
)

overlap = train_keys.intersection(val_keys)

print("\nTRAIN IDENTITIES:", len(train_keys))
print("VALIDATION IDENTITIES:", len(val_keys))
print("OVERLAPPING IDENTITIES:", len(overlap))

assert len(overlap) == 0
assert len(train_keys) == 25
assert len(val_keys) == 7

print("\nSPLIT INTEGRITY: PASS")
print("CELL 151 COMPLETED")

CELL 151: PARP1 SPLIT INTEGRITY

MANIFEST SHAPE: (195, 9)
COLUMNS: ['pair_idx', 'ligand_identity', 'inchikey', 'pose_index', 'conformer_id', 'canonical_smiles', 'ligand_atoms', 'ligand_bonds', 'split']

UNIQUE LIGAND IDENTITIES BY SPLIT:
split
train         25
validation     7
Name: inchikey, dtype: int64

TRAIN IDENTITIES: 25
VALIDATION IDENTITIES: 7
OVERLAPPING IDENTITIES: 0

SPLIT INTEGRITY: PASS
CELL 151 COMPLETED


In [212]:
# ============================================================
# CELL 152: CHEMBL TOKEN SEQUENCE QUALITY AUDIT
# ============================================================

import numpy as np
import json
import os

print("=" * 65)
print("CELL 152: CHEMBL SEQUENCE QUALITY")
print("=" * 65)

tokenizer_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

vocab = tokenizer_data["token_to_id"]

PAD = vocab["<PAD>"]
BOS = vocab["<BOS>"]
EOS = vocab["<EOS>"]
UNK = vocab["<UNK>"]

def audit_sequences(sequences, name):

    print(f"\n{name}")

    sequences = np.asarray(sequences)

    print("Shape:", sequences.shape)
    print("Data type:", sequences.dtype)

    # Token range
    print("Minimum token ID:", sequences.min())
    print("Maximum token ID:", sequences.max())

    assert sequences.min() >= 0
    assert sequences.max() < len(vocab)

    # Special-token checks
    bos_count = np.sum(sequences[:, 0] == BOS)
    eos_count = np.sum(
        np.any(sequences == EOS, axis=1)
    )
    unk_count = np.sum(sequences == UNK)

    print("Sequences starting with BOS:", bos_count)
    print("Sequences containing EOS:", eos_count)
    print("Unknown-token occurrences:", unk_count)

    # Sequence lengths
    lengths = np.sum(sequences != PAD, axis=1)

    print("Minimum length:", lengths.min())
    print("Maximum length:", lengths.max())
    print("Mean length:", round(lengths.mean(), 2))

    assert bos_count == len(sequences)
    assert eos_count == len(sequences)
    assert unk_count == 0

    return lengths

train_lengths = audit_sequences(
    chembl_train_tokens,
    "CHEMBL TRAINING"
)

val_lengths = audit_sequences(
    chembl_val_tokens,
    "CHEMBL VALIDATION"
)

print("\nCELL 152 COMPLETED")

CELL 152: CHEMBL SEQUENCE QUALITY

CHEMBL TRAINING
Shape: (45000, 143)
Data type: int32
Minimum token ID: 0
Maximum token ID: 80
Sequences starting with BOS: 45000
Sequences containing EOS: 45000
Unknown-token occurrences: 0
Minimum length: 9
Maximum length: 143
Mean length: 51.36

CHEMBL VALIDATION
Shape: (5000, 143)
Data type: int32
Minimum token ID: 0
Maximum token ID: 80
Sequences starting with BOS: 5000
Sequences containing EOS: 5000
Unknown-token occurrences: 0
Minimum length: 11
Maximum length: 130
Mean length: 51.46

CELL 152 COMPLETED


In [214]:
# ============================================================
# CELL 153B: CORRECTED DECODER LAYER INSPECTION
# ============================================================

print("=" * 65)
print("CELL 153B: DECODER LAYER INSPECTION")
print("=" * 65)

decoder_layers = molecular_transformer.decoder.layers

print("Number of decoder layers:", len(decoder_layers))

for index, layer in enumerate(decoder_layers):

    print(f"\nDecoder layer {index + 1}")

    print("Self-attention:",
          type(layer.self_attn).__name__)

    print("Cross-attention:",
          type(layer.multihead_attn).__name__)

    print("Self-attention heads:",
          layer.self_attn.num_heads)

    print("Cross-attention heads:",
          layer.multihead_attn.num_heads)

    print("Feed-forward input:",
          layer.linear1.in_features)

    print("Feed-forward hidden:",
          layer.linear1.out_features)

print("\nCELL 153B COMPLETED")

CELL 153B: DECODER LAYER INSPECTION
Number of decoder layers: 3

Decoder layer 1
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Self-attention heads: 4
Cross-attention heads: 4
Feed-forward input: 128
Feed-forward hidden: 512

Decoder layer 2
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Self-attention heads: 4
Cross-attention heads: 4
Feed-forward input: 128
Feed-forward hidden: 512

Decoder layer 3
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Self-attention heads: 4
Cross-attention heads: 4
Feed-forward input: 128
Feed-forward hidden: 512

CELL 153B COMPLETED


In [215]:
# ============================================================
# CELL 154: PRETRAINING CONFIGURATION AUDIT
# ============================================================

import torch
import torch.nn as nn

print("=" * 65)
print("CELL 154: PRETRAINING CONFIGURATION AUDIT")
print("=" * 65)

layers = molecular_transformer.decoder.layers

for i, layer in enumerate(layers):

    print(f"\nDECODER LAYER {i + 1}")

    print("Norm first:", layer.norm_first)
    print("Self-attention batch first:",
          layer.self_attn.batch_first)
    print("Cross-attention batch first:",
          layer.multihead_attn.batch_first)

    print("Self-attention dropout:",
          layer.self_attn.dropout)

    print("Feed-forward dropout:",
          layer.dropout.p)

    print("Normalization 1:",
          type(layer.norm1).__name__)

    print("Normalization 2:",
          type(layer.norm2).__name__)

    print("Normalization 3:",
          type(layer.norm3).__name__)

# Verify causal sequence dimensions
print("\nMODEL CONFIGURATION")

print("Vocabulary size:",
      molecular_transformer.token_embedding.num_embeddings)

print("Embedding dimension:",
      molecular_transformer.token_embedding.embedding_dim)

print("Maximum sequence length:",
      molecular_transformer.position_encoding.position_embedding.num_embeddings)

print("Output vocabulary size:",
      molecular_transformer.output_layer.out_features)

print("\nCELL 154 COMPLETED")

CELL 154: PRETRAINING CONFIGURATION AUDIT

DECODER LAYER 1
Norm first: True
Self-attention batch first: True
Cross-attention batch first: True
Self-attention dropout: 0.1
Feed-forward dropout: 0.1
Normalization 1: LayerNorm
Normalization 2: LayerNorm
Normalization 3: LayerNorm

DECODER LAYER 2
Norm first: True
Self-attention batch first: True
Cross-attention batch first: True
Self-attention dropout: 0.1
Feed-forward dropout: 0.1
Normalization 1: LayerNorm
Normalization 2: LayerNorm
Normalization 3: LayerNorm

DECODER LAYER 3
Norm first: True
Self-attention batch first: True
Cross-attention batch first: True
Self-attention dropout: 0.1
Feed-forward dropout: 0.1
Normalization 1: LayerNorm
Normalization 2: LayerNorm
Normalization 3: LayerNorm

MODEL CONFIGURATION
Vocabulary size: 81
Embedding dimension: 128
Maximum sequence length: 143
Output vocabulary size: 81

CELL 154 COMPLETED


In [216]:
# ============================================================
# CELL 155: DECODER COMPONENT INSPECTION
# ============================================================

import torch.nn as nn

print("=" * 65)
print("CELL 155: DECODER COMPONENT INSPECTION")
print("=" * 65)

for i, layer in enumerate(molecular_transformer.decoder.layers):

    print(f"\nDECODER LAYER {i + 1}")

    print("Activation:", layer.activation)
    print("Dropout 1:", layer.dropout1)
    print("Dropout 2:", layer.dropout2)
    print("Dropout 3:", layer.dropout3)

    print("Self-attention output projection:",
          layer.self_attn.out_proj)

    print("Feed-forward layer 1:", layer.linear1)
    print("Feed-forward layer 2:", layer.linear2)

    print("Feed-forward activation:",
          layer.activation.__name__)

print("\nCELL 155 COMPLETED")

CELL 155: DECODER COMPONENT INSPECTION

DECODER LAYER 1
Activation: <function relu at 0x78003f944400>
Dropout 1: Dropout(p=0.1, inplace=False)
Dropout 2: Dropout(p=0.1, inplace=False)
Dropout 3: Dropout(p=0.1, inplace=False)
Self-attention output projection: NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
Feed-forward layer 1: Linear(in_features=128, out_features=512, bias=True)
Feed-forward layer 2: Linear(in_features=512, out_features=128, bias=True)
Feed-forward activation: relu

DECODER LAYER 2
Activation: <function relu at 0x78003f944400>
Dropout 1: Dropout(p=0.1, inplace=False)
Dropout 2: Dropout(p=0.1, inplace=False)
Dropout 3: Dropout(p=0.1, inplace=False)
Self-attention output projection: NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
Feed-forward layer 1: Linear(in_features=128, out_features=512, bias=True)
Feed-forward layer 2: Linear(in_features=512, out_features=128, bias=True)
Feed-forward activation: relu

DECO

In [217]:
# ============================================================
# CELL 156: MOLECULAR-ONLY TRANSFORMER FORWARD PASS
# ============================================================

import torch
import torch.nn as nn

print("=" * 65)
print("CELL 156: MOLECULAR-ONLY FORWARD PASS")
print("=" * 65)


def molecular_only_forward(model, token_ids):

    batch_size, seq_length = token_ids.shape

    # Token and positional embeddings
    x = model.token_embedding(token_ids)
    x = model.position_encoding(x)

    # Padding mask
    padding_mask = token_ids == 0

    # Causal mask: prevent looking at future tokens
    causal_mask = torch.triu(
        torch.ones(
            seq_length,
            seq_length,
            device=token_ids.device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    # Apply molecular self-attention and feed-forward layers
    for layer in model.decoder.layers:

        if layer.norm_first:

            # Self-attention
            normalized = layer.norm1(x)

            attention_output = layer.self_attn(
                normalized,
                normalized,
                normalized,
                attn_mask=causal_mask,
                key_padding_mask=padding_mask,
                need_weights=False
            )[0]

            x = x + layer.dropout1(attention_output)

            # Feed-forward
            normalized = layer.norm3(x)

            ff_output = layer.linear2(
                layer.dropout(
                    layer.activation(
                        layer.linear1(normalized)
                    )
                )
            )

            x = x + layer.dropout3(ff_output)

        else:
            raise NotImplementedError(
                "This implementation expects norm_first=True."
            )

    # Final vocabulary prediction
    logits = model.output_layer(x)

    return logits


# Test forward pass
test_batch = torch.tensor(
    chembl_train_tokens[:4],
    dtype=torch.long
)

molecular_transformer.eval()

with torch.no_grad():

    test_logits = molecular_only_forward(
        molecular_transformer,
        test_batch
    )

print("\nINPUT SHAPE:", tuple(test_batch.shape))
print("OUTPUT SHAPE:", tuple(test_logits.shape))

assert test_logits.shape == (
    4,
    143,
    81
)

assert torch.isfinite(test_logits).all()

print("\nPocket cross-attention used: NO")
print("Original cross-attention layers preserved: YES")
print("Output finite: PASS")

print("\nCELL 156 COMPLETED")

CELL 156: MOLECULAR-ONLY FORWARD PASS

INPUT SHAPE: (4, 143)
OUTPUT SHAPE: (4, 143, 81)

Pocket cross-attention used: NO
Original cross-attention layers preserved: YES
Output finite: PASS

CELL 156 COMPLETED


In [218]:
# ============================================================
# CELL 157: NEXT-TOKEN PREDICTION AND LOSS VALIDATION
# ============================================================

import torch
import torch.nn as nn

print("=" * 65)
print("CELL 157: NEXT-TOKEN LOSS VALIDATION")
print("=" * 65)

PAD = vocab["<PAD>"]

# Prepare a small batch
batch = torch.tensor(
    chembl_train_tokens[:8],
    dtype=torch.long
)

# Shift input and target
input_ids = batch[:, :-1]
target_ids = batch[:, 1:]

print("\nINPUT SHAPE:", tuple(input_ids.shape))
print("TARGET SHAPE:", tuple(target_ids.shape))

# Forward pass
molecular_transformer.eval()

with torch.no_grad():

    logits = molecular_only_forward(
        molecular_transformer,
        input_ids
    )

    # Exclude padding positions from loss
    loss_function = nn.CrossEntropyLoss(
        ignore_index=PAD
    )

    loss = loss_function(
        logits.reshape(-1, logits.shape[-1]),
        target_ids.reshape(-1)
    )

print("\nLOGITS SHAPE:", tuple(logits.shape))
print("Initial validation loss:", round(loss.item(), 4))

# Check prediction alignment
assert logits.shape[:2] == target_ids.shape
assert torch.isfinite(loss)

print("\nVALIDATION")
print("Input-target alignment: PASS")
print("Padding ignored: PASS")
print("Loss finite: PASS")

print("\nCELL 157 COMPLETED")

CELL 157: NEXT-TOKEN LOSS VALIDATION

INPUT SHAPE: (8, 142)
TARGET SHAPE: (8, 142)

LOGITS SHAPE: (8, 142, 81)
Initial validation loss: 5.084

VALIDATION
Input-target alignment: PASS
Padding ignored: PASS
Loss finite: PASS

CELL 157 COMPLETED


In [219]:
# ============================================================
# CELL 158: CH E M B L PRETRAINING CONFIGURATION
# ============================================================

import torch

print("=" * 65)
print("CELL 158: TRAINING CONFIGURATION")
print("=" * 65)

# Freeze pocket cross-attention parameters
for layer in molecular_transformer.decoder.layers:
    for parameter in layer.multihead_attn.parameters():
        parameter.requires_grad = False

# Keep molecular pathway trainable
for name, parameter in molecular_transformer.named_parameters():

    if "multihead_attn" not in name:
        parameter.requires_grad = True

# Training configuration
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
GRADIENT_CLIP = 1.0
EPOCHS = 5

# Optimizer includes only trainable parameters
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        molecular_transformer.parameters()
    ),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

trainable_params = sum(
    p.numel()
    for p in molecular_transformer.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in molecular_transformer.parameters()
    if not p.requires_grad
)

print("\nTRAINING SETTINGS")
print("Device: CPU")
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient clipping:", GRADIENT_CLIP)
print("Epochs:", EPOCHS)

print("\nPARAMETER STATUS")
print("Trainable:", trainable_params)
print("Frozen:", frozen_params)

assert frozen_params > 0
assert trainable_params > 0

print("\nCELL 158 COMPLETED")

CELL 158: TRAINING CONFIGURATION

TRAINING SETTINGS
Device: CPU
Batch size: 32
Learning rate: 0.0001
Weight decay: 1e-05
Gradient clipping: 1.0
Epochs: 5

PARAMETER STATUS
Trainable: 634705
Frozen: 198144

CELL 158 COMPLETED


In [220]:
# ============================================================
# CELL 158B: FREEZE UNUSED CROSS-ATTENTION COMPONENTS
# ============================================================

print("=" * 65)
print("CELL 158B: PARAMETER FREEZING CORRECTION")
print("=" * 65)

for layer in molecular_transformer.decoder.layers:

    # Cross-attention is not used during ChEMBL pretraining
    for parameter in layer.multihead_attn.parameters():
        parameter.requires_grad = False

    # norm2 belongs to the cross-attention residual block
    for parameter in layer.norm2.parameters():
        parameter.requires_grad = False

# Count parameters
trainable = sum(
    p.numel()
    for p in molecular_transformer.parameters()
    if p.requires_grad
)

frozen = sum(
    p.numel()
    for p in molecular_transformer.parameters()
    if not p.requires_grad
)

print("Trainable parameters:", trainable)
print("Frozen parameters:", frozen)
print("Total parameters:", trainable + frozen)

# Verify
for layer in molecular_transformer.decoder.layers:

    assert not any(
        p.requires_grad
        for p in layer.multihead_attn.parameters()
    )

    assert not any(
        p.requires_grad
        for p in layer.norm2.parameters()
    )

print("\nCross-attention frozen: PASS")
print("Unused normalization frozen: PASS")
print("\nCELL 158B COMPLETED")

CELL 158B: PARAMETER FREEZING CORRECTION
Trainable parameters: 633937
Frozen parameters: 198912
Total parameters: 832849

Cross-attention frozen: PASS
Unused normalization frozen: PASS

CELL 158B COMPLETED


In [222]:
# CELL 159A: TRAINING SPEED TEST

import time
import torch
import torch.nn as nn

DEVICE = torch.device("cpu")
PAD = vocab["<PAD>"]

optimizer = torch.optim.AdamW(
    [p for p in molecular_transformer.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-5
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

molecular_transformer.train()

test_batches = 20
start = time.time()

for i, (batch,) in enumerate(train_loader):

    if i >= test_batches:
        break

    batch = batch.to(DEVICE)

    inputs = batch[:, :-1]
    targets = batch[:, 1:]

    optimizer.zero_grad(set_to_none=True)

    logits = molecular_only_forward(
        molecular_transformer, inputs
    )

    loss = criterion(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        [p for p in molecular_transformer.parameters()
         if p.requires_grad],
        1.0
    )

    optimizer.step()

elapsed = time.time() - start

batches_per_epoch = len(train_loader)
estimated_epoch_time = elapsed / test_batches * batches_per_epoch

print("20 batches completed.")
print("Time:", round(elapsed, 2), "seconds")
print("Estimated time per epoch:", round(estimated_epoch_time / 60, 2), "minutes")
print("Estimated time for 5 epochs:",
      round(estimated_epoch_time * 5 / 60, 2), "minutes")
print("CELL 159A COMPLETED")

20 batches completed.
Time: 32.31 seconds
Estimated time per epoch: 37.89 minutes
Estimated time for 5 epochs: 189.43 minutes
CELL 159A COMPLETED


In [223]:
# ============================================================
# CELL 159B: REDUCED CHEMBL TRANSFORMER TRAINING
# ============================================================

import os
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("=" * 65)
print("CELL 159B: TRAINING ON 10,000 CHEMBL MOLECULES")
print("=" * 65)

# Configuration
DEVICE = torch.device("cpu")
BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 1e-4
PAD = vocab["<PAD>"]

# Select 10,000 training molecules
rng = np.random.default_rng(42)

selected_indices = rng.choice(
    len(chembl_train_tokens),
    size=10000,
    replace=False
)

train_subset = chembl_train_tokens[selected_indices]

print("Selected training molecules:", len(train_subset))
print("Validation molecules:", len(chembl_val_tokens))

# Prepare loaders
train_tensor = torch.tensor(train_subset, dtype=torch.long)
val_tensor = torch.tensor(chembl_val_tokens, dtype=torch.long)

train_loader_small = DataLoader(
    TensorDataset(train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(val_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Fresh optimizer
optimizer = torch.optim.AdamW(
    [p for p in molecular_transformer.parameters()
     if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

# Checkpoint
checkpoint_path = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "transformer_chembl37_10k_best.pt"
)

best_val_loss = float("inf")
history = []

# Training
for epoch in range(EPOCHS):

    start = time.time()

    molecular_transformer.train()
    train_loss_total = 0

    for batch_idx, (batch,) in enumerate(train_loader_small):

        batch = batch.to(DEVICE)

        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = molecular_only_forward(
            molecular_transformer,
            inputs
        )

        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            targets.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in molecular_transformer.parameters()
             if p.requires_grad],
            1.0
        )

        optimizer.step()

        train_loss_total += loss.item()

    train_loss = train_loss_total / len(train_loader_small)

    # Validation
    molecular_transformer.eval()
    val_loss_total = 0

    with torch.no_grad():

        for (batch,) in val_loader:

            batch = batch.to(DEVICE)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            logits = molecular_only_forward(
                molecular_transformer,
                inputs
            )

            loss = criterion(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1)
            )

            val_loss_total += loss.item()

    val_loss = val_loss_total / len(val_loader)

    elapsed = time.time() - start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f} | "
        f"Time: {elapsed/60:.2f} min"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "model_state_dict": molecular_transformer.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "training_molecules": 10000,
            "training_type": "chembl37_general_molecular_pretraining",
            "pocket_cross_attention_used": False,
            "vocabulary_size": len(vocab)
        }, checkpoint_path)

        print("Best checkpoint saved.")

print("\nTRAINING COMPLETED")
print("Best validation loss:", round(best_val_loss, 4))
print("Checkpoint:", checkpoint_path)

CELL 159B: TRAINING ON 10,000 CHEMBL MOLECULES
Selected training molecules: 10000
Validation molecules: 5000
Epoch 1/2 | Train Loss: 1.4113 | Validation Loss: 1.3472 | Time: 5.87 min
Best checkpoint saved.
Epoch 2/2 | Train Loss: 1.3710 | Validation Loss: 1.3104 | Time: 5.57 min
Best checkpoint saved.

TRAINING COMPLETED
Best validation loss: 1.3104
Checkpoint: /content/drive/MyDrive/Member2_Project/checkpoints/transformer_chembl37_10k_best.pt


In [224]:
# ============================================================
# CELL 160: MOLECULAR GENERATION AND VALIDATION
# ============================================================

import os
import json
import torch
import numpy as np
from rdkit import Chem

print("=" * 65)
print("CELL 160: MOLECULAR GENERATION")
print("=" * 65)

# 1. Load best checkpoint
checkpoint_path = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "transformer_chembl37_10k_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"]
)

molecular_transformer.eval()

print("Checkpoint loaded.")
print("Best epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["validation_loss"])

# 2. Token configuration
id_to_token = {
    index: token
    for token, index in vocab.items()
}

bos_id = vocab["<BOS>"]
eos_id = vocab["<EOS>"]
pad_id = vocab["<PAD>"]

MAX_LENGTH = 143
NUM_MOLECULES = 30
TEMPERATURE = 0.8
TOP_K = 20

# 3. Generate SMILES
generated_smiles = []

for molecule_index in range(NUM_MOLECULES):

    sequence = [bos_id]

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            input_ids = torch.tensor(
                [sequence],
                dtype=torch.long
            )

            logits = molecular_only_forward(
                molecular_transformer,
                input_ids
            )

            next_logits = logits[0, -1, :] / TEMPERATURE

            # Prevent special tokens from appearing inside molecules
            next_logits[pad_id] = float("-inf")
            next_logits[bos_id] = float("-inf")

            # Top-k sampling
            values, indices = torch.topk(
                next_logits,
                TOP_K
            )

            probabilities = torch.softmax(values, dim=-1)

            selected_position = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            next_token = indices[selected_position].item()

            if next_token == eos_id:
                break

            sequence.append(next_token)

    tokens = [
        id_to_token[token_id]
        for token_id in sequence[1:]
        if token_id != eos_id
    ]

    smiles = "".join(tokens)

    generated_smiles.append(smiles)

    if (molecule_index + 1) % 5 == 0:
        print(f"Generated {molecule_index + 1}/{NUM_MOLECULES}")

# 4. RDKit validation
valid_molecules = []
valid_smiles = []
canonical_smiles = []

for smiles in generated_smiles:

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is not None:

            canonical = Chem.MolToSmiles(
                mol,
                canonical=True
            )

            valid_molecules.append(mol)
            valid_smiles.append(smiles)
            canonical_smiles.append(canonical)

    except Exception:
        pass

unique_smiles = set(canonical_smiles)

validity = (
    len(valid_smiles) / NUM_MOLECULES
) * 100

uniqueness = (
    len(unique_smiles) / len(valid_smiles) * 100
    if valid_smiles else 0
)

# 5. Save results
output_dir = os.path.join(
    PROJECT_ROOT,
    "member2_outputs"
)

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "chembl_transformer_generated.jsonl"
)

with open(output_path, "w") as file:

    for smiles in valid_smiles:

        mol = Chem.MolFromSmiles(smiles)

        record = {
            "smiles": smiles,
            "canonical_smiles": Chem.MolToSmiles(mol),
            "source": "ChEMBL37_general_pretrained_transformer"
        }

        file.write(json.dumps(record) + "\n")

print("\n" + "=" * 65)
print("GENERATION RESULTS")
print("=" * 65)

print("Generated:", NUM_MOLECULES)
print("Valid:", len(valid_smiles))
print("Invalid:", NUM_MOLECULES - len(valid_smiles))
print("Validity:", round(validity, 2), "%")
print("Unique valid molecules:", len(unique_smiles))
print("Uniqueness:", round(uniqueness, 2), "%")

print("\nSample valid molecules:")

for smiles in valid_smiles[:5]:
    print(smiles)

print("\nSaved:", output_path)
print("\nCELL 160 COMPLETED")

CELL 160: MOLECULAR GENERATION
Checkpoint loaded.
Best epoch: 2
Validation loss: 1.3104469297797816
Generated 5/30
Generated 10/30
Generated 15/30
Generated 20/30
Generated 25/30
Generated 30/30

GENERATION RESULTS
Generated: 30
Valid: 0
Invalid: 30
Validity: 0.0 %
Unique valid molecules: 0
Uniqueness: 0 %

Sample valid molecules:

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/chembl_transformer_generated.jsonl

CELL 160 COMPLETED


In [225]:
# ============================================================
# CELL 161: GENERATED SMILES DIAGNOSTICS
# ============================================================

from rdkit import Chem
from rdkit import RDLogger

RDLogger.EnableLog("rdApp.error")

print("=" * 65)
print("CELL 161: SMILES DIAGNOSTICS")
print("=" * 65)

for i, smiles in enumerate(generated_smiles[:10], 1):

    print(f"\nMolecule {i}")
    print("Generated SMILES:", repr(smiles))

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print("Status: INVALID")
    else:
        print("Status: VALID")
        print("Canonical:", Chem.MolToSmiles(mol))

print("\nCELL 161 COMPLETED")

CELL 161: SMILES DIAGNOSTICS

Molecule 1
Generated SMILES: 'COc1cc(CN2CCC2)c(O)C(=O)NC(=O)c2ccccc1)CO'
Status: INVALID

Molecule 2
Generated SMILES: 'COc1ccc(CN(C(=O)NCCCCN2CCN(CC)CCCC(=O)N1CCCCC1C)C'
Status: INVALID

Molecule 3
Generated SMILES: 'CCCCCCCN(c1cccc2nc(-c3cccccc3)C(OC)c43)c1'
Status: INVALID

Molecule 4
Generated SMILES: 'C=C(NCC(=O)Nc1ccc(-c3ccccc(C(F)CC(C)C)c3)c2)c2)C2'
Status: INVALID

Molecule 5
Generated SMILES: 'Cc1ccc(CN(=O)c2cc(C(=O)c(C(=O)N2C)CCCC2)CC(C(Cc3)cc3)c(C)cc12'
Status: INVALID

Molecule 6
Generated SMILES: 'CC(C)c1cccc1OC(=O)Nc1ccc(Br)COc1cc(F)c2n(C)cc(O)c1'
Status: INVALID

Molecule 7
Generated SMILES: 'Cc1ccc(CN3CCC2C2)c(C(=O)C3)CN(C(CCc3C3)CC2)CCCCC3)C2ccc2)C1'
Status: INVALID

Molecule 8
Generated SMILES: 'CCC(C)Cn1c(OC(C)=O)NCCCCCC(=O)CC(C)CCCC)C1=O'
Status: INVALID

Molecule 9
Generated SMILES: 'CN1CC1CCC(=O)cc1cccc(CN=O)Nc1ccccc(N(C(=O)c2c(C)=O)c1'
Status: INVALID

Molecule 10
Generated SMILES: 'NC(CC(=O)Nc2ccc(Cl)ccc3)c(Cl)ccc(Cl)c2)n1'
Status: 

In [226]:
# ============================================================
# CELL 162: SMILES STRUCTURAL DIAGNOSTICS
# ============================================================

from collections import Counter
from rdkit import Chem
import re

print("=" * 65)
print("CELL 162: SMILES STRUCTURAL DIAGNOSTICS")
print("=" * 65)

results = Counter()

for i, smiles in enumerate(generated_smiles, 1):

    # Parentheses balance
    balance = 0
    min_balance = 0

    for ch in smiles:
        if ch == "(":
            balance += 1
        elif ch == ")":
            balance -= 1
            min_balance = min(min_balance, balance)

    parentheses_ok = balance == 0 and min_balance >= 0

    # Ring closure check
    ring_tokens = re.findall(r"%\d{2}|\d", smiles)
    ring_counts = Counter(ring_tokens)

    rings_ok = all(count % 2 == 0 for count in ring_counts.values())

    # RDKit validation
    mol = Chem.MolFromSmiles(smiles)
    rdkit_valid = mol is not None

    if not parentheses_ok:
        results["Unbalanced parentheses"] += 1

    if not rings_ok:
        results["Unclosed ring labels"] += 1

    if not rdkit_valid:
        results["RDKit invalid"] += 1

    print(
        f"Molecule {i:02d}: "
        f"Length={len(smiles):3d} | "
        f"Parentheses={'PASS' if parentheses_ok else 'FAIL'} | "
        f"Rings={'PASS' if rings_ok else 'FAIL'} | "
        f"RDKit={'PASS' if rdkit_valid else 'FAIL'}"
    )

print("\nSUMMARY")

for reason, count in results.items():
    print(f"{reason}: {count}/30")

print("\nCELL 162 COMPLETED")

CELL 162: SMILES STRUCTURAL DIAGNOSTICS
Molecule 01: Length= 41 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 02: Length= 49 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 03: Length= 41 | Parentheses=PASS | Rings=FAIL | RDKit=FAIL
Molecule 04: Length= 49 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 05: Length= 61 | Parentheses=FAIL | Rings=PASS | RDKit=FAIL
Molecule 06: Length= 50 | Parentheses=PASS | Rings=FAIL | RDKit=FAIL
Molecule 07: Length= 59 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 08: Length= 44 | Parentheses=FAIL | Rings=PASS | RDKit=FAIL
Molecule 09: Length= 53 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 10: Length= 41 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 11: Length= 35 | Parentheses=PASS | Rings=FAIL | RDKit=FAIL
Molecule 12: Length= 80 | Parentheses=FAIL | Rings=FAIL | RDKit=FAIL
Molecule 13: Length= 53 | Parentheses=FAIL | Rings=PASS | RDKit=FAIL
Molecule 14: Length= 29 | Parentheses=PASS | Rings=FAIL | RDKit

In [227]:
# ============================================================
# CELL 163: TOKENIZER ROUND-TRIP AUDIT
# ============================================================

import json
import os
from rdkit import Chem

print("=" * 65)
print("CELL 163: TOKENIZER VERIFICATION")
print("=" * 65)

tokenizer_path = os.path.join(
    PROJECT_ROOT,
    "member2_outputs",
    "smiles_tokenizer.json"
)

with open(tokenizer_path) as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]

id_to_token = {
    int(idx): token
    for token, idx in token_to_id.items()
}

print("Vocabulary size:", len(token_to_id))
print("Model vocabulary size:", len(vocab))
print("Tokenizer maximum length:", tokenizer_data["max_length"])

# Compare vocabulary mappings
mapping_matches = all(
    vocab[token] == idx
    for token, idx in token_to_id.items()
)

print("Vocabulary mapping matches:", mapping_matches)

# Decode original training sequences
sample_count = 20
valid_count = 0

for i in range(sample_count):

    ids = chembl_train_tokens[i]

    tokens = [
        id_to_token[int(idx)]
        for idx in ids
        if int(idx) not in (
            vocab["<PAD>"],
            vocab["<BOS>"],
            vocab["<EOS>"]
        )
    ]

    decoded = "".join(tokens)

    mol = Chem.MolFromSmiles(decoded)

    if mol is not None:
        valid_count += 1

    if i < 5:
        print("\nSample", i + 1)
        print("Decoded:", decoded)
        print("RDKit valid:", mol is not None)

print("\nRound-trip valid:", valid_count, "/", sample_count)

print("\nCELL 163 COMPLETED")

CELL 163: TOKENIZER VERIFICATION
Vocabulary size: 81
Model vocabulary size: 81
Tokenizer maximum length: 143
Vocabulary mapping matches: True

Sample 1
Decoded: Nc1ccc(C(=O)N/N=C/c2ccc3no[n+]([O-])c3c2)cc1
RDKit valid: True

Sample 2
Decoded: COCc1ccc(C(=O)N2CCC(c3ccc(C#N)cc3)CC2)cc1-c1[nH]c(C)nc1C
RDKit valid: True

Sample 3
Decoded: Cc1cc(C)c(C)c(OCCCN2CCC(O)CC2)c1
RDKit valid: True

Sample 4
Decoded: Cc1nc(N2CCC(C)(NCC3CCC(O)CC3)CC2)cc(=O)n1-c1cccc(Cl)c1Cl
RDKit valid: True

Sample 5
Decoded: COC(=O)CNc1nc(-c2cccc(O)c2)nc(N2CCOCC2)n1
RDKit valid: True

Round-trip valid: 20 / 20

CELL 163 COMPLETED


In [228]:
# ============================================================
# CELL 164: GENERATION BEHAVIOUR AUDIT
# ============================================================

from collections import Counter

print("=" * 65)
print("CELL 164: GENERATION BEHAVIOUR AUDIT")
print("=" * 65)

print("\n1. GENERATED SEQUENCE LENGTHS")

lengths = [len(s) for s in generated_smiles]

print("Minimum:", min(lengths))
print("Maximum:", max(lengths))
print("Average:", round(sum(lengths) / len(lengths), 2))

print("\n2. TOKEN FREQUENCY")

token_counts = Counter()

for smiles in generated_smiles:
    for token in smiles:
        token_counts[token] += 1

print("Most common characters:")
print(token_counts.most_common(20))

print("\n3. EOS GENERATION CHECK")

print("Generated sequences:", len(generated_smiles))
print("EOS ID:", eos_id)
print("Maximum generation length:", MAX_LENGTH)

print("\n4. TRAINING SEQUENCE COMPARISON")

for i in range(5):

    original = chembl_train_tokens[i]

    actual_length = int(
        np.sum(original != pad_id)
    )

    print(
        f"Training sequence {i+1}: "
        f"{actual_length} tokens"
    )

print("\nCELL 164 COMPLETED")

CELL 164: GENERATION BEHAVIOUR AUDIT

1. GENERATED SEQUENCE LENGTHS
Minimum: 22
Maximum: 89
Average: 49.5

2. TOKEN FREQUENCY
Most common characters:
[('c', 407), ('C', 302), (')', 160), ('(', 155), ('O', 90), ('1', 89), ('2', 58), ('N', 55), ('=', 55), ('3', 40), ('n', 16), ('-', 13), ('4', 7), ('l', 6), ('[', 5), (']', 5), ('F', 4), ('@', 4), ('H', 4), ('#', 2)]

3. EOS GENERATION CHECK
Generated sequences: 30
EOS ID: 2
Maximum generation length: 143

4. TRAINING SEQUENCE COMPARISON
Training sequence 1: 40 tokens
Training sequence 2: 55 tokens
Training sequence 3: 34 tokens
Training sequence 4: 56 tokens
Training sequence 5: 43 tokens

CELL 164 COMPLETED


In [229]:
# ============================================================
# CELL 165: EOS TERMINATION AUDIT
# ============================================================

import torch

print("=" * 65)
print("CELL 165: EOS TERMINATION AUDIT")
print("=" * 65)

termination_results = []

for i in range(30):

    sequence = [bos_id]
    terminated = False

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            input_ids = torch.tensor(
                [sequence],
                dtype=torch.long
            )

            logits = molecular_only_forward(
                molecular_transformer,
                input_ids
            )

            next_logits = logits[0, -1, :] / TEMPERATURE

            next_logits[pad_id] = float("-inf")
            next_logits[bos_id] = float("-inf")

            values, indices = torch.topk(
                next_logits, TOP_K
            )

            probabilities = torch.softmax(values, dim=-1)

            selected = torch.multinomial(
                probabilities, 1
            ).item()

            token_id = indices[selected].item()

            if token_id == eos_id:
                terminated = True
                break

            sequence.append(token_id)

    termination_results.append({
        "length": len(sequence) - 1,
        "eos_generated": terminated
    })

terminated_count = sum(
    x["eos_generated"] for x in termination_results
)

print("Total sequences:", len(termination_results))
print("EOS terminated:", terminated_count)
print("Maximum-length terminated:", 30 - terminated_count)

print("\nSequence details:")

for i, result in enumerate(termination_results, 1):
    print(
        f"{i:02d}: Length={result['length']} | "
        f"EOS={'YES' if result['eos_generated'] else 'NO'}"
    )

print("\nCELL 165 COMPLETED")

CELL 165: EOS TERMINATION AUDIT
Total sequences: 30
EOS terminated: 30
Maximum-length terminated: 0

Sequence details:
01: Length=39 | EOS=YES
02: Length=44 | EOS=YES
03: Length=57 | EOS=YES
04: Length=40 | EOS=YES
05: Length=34 | EOS=YES
06: Length=54 | EOS=YES
07: Length=59 | EOS=YES
08: Length=34 | EOS=YES
09: Length=71 | EOS=YES
10: Length=50 | EOS=YES
11: Length=47 | EOS=YES
12: Length=47 | EOS=YES
13: Length=67 | EOS=YES
14: Length=50 | EOS=YES
15: Length=53 | EOS=YES
16: Length=48 | EOS=YES
17: Length=74 | EOS=YES
18: Length=57 | EOS=YES
19: Length=50 | EOS=YES
20: Length=33 | EOS=YES
21: Length=50 | EOS=YES
22: Length=44 | EOS=YES
23: Length=48 | EOS=YES
24: Length=44 | EOS=YES
25: Length=47 | EOS=YES
26: Length=35 | EOS=YES
27: Length=40 | EOS=YES
28: Length=42 | EOS=YES
29: Length=39 | EOS=YES
30: Length=53 | EOS=YES

CELL 165 COMPLETED


In [230]:
# ============================================================
# CELL 166: GREEDY GENERATION TEST
# ============================================================

from rdkit import Chem
import torch

print("=" * 65)
print("CELL 166: GREEDY GENERATION")
print("=" * 65)

greedy_smiles = []

NUM_TEST = 30

for i in range(NUM_TEST):

    sequence = [bos_id]

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            input_ids = torch.tensor(
                [sequence], dtype=torch.long
            )

            logits = molecular_only_forward(
                molecular_transformer,
                input_ids
            )

            next_logits = logits[0, -1, :].clone()

            next_logits[pad_id] = float("-inf")
            next_logits[bos_id] = float("-inf")

            next_token = torch.argmax(next_logits).item()

            if next_token == eos_id:
                break

            sequence.append(next_token)

    tokens = [
        id_to_token[token_id]
        for token_id in sequence[1:]
    ]

    smiles = "".join(tokens)
    greedy_smiles.append(smiles)

# Validate
valid = []

for smiles in greedy_smiles:

    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        valid.append(Chem.MolToSmiles(mol))

print("\nGenerated:", len(greedy_smiles))
print("Valid:", len(valid))
print("Invalid:", len(greedy_smiles) - len(valid))
print("Validity:", round(len(valid) / NUM_TEST * 100, 2), "%")

print("\nFirst 5 generated molecules:")

for smiles in greedy_smiles[:5]:
    print(smiles)

print("\nCELL 166 COMPLETED")

CELL 166: GREEDY GENERATION

Generated: 30
Valid: 30
Invalid: 0
Validity: 100.0 %

First 5 generated molecules:
CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1
CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1
CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1
CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1
CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1

CELL 166 COMPLETED


In [231]:
# ============================================================
# CELL 167: GREEDY GENERATION UNIQUENESS AUDIT
# ============================================================

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem

print("=" * 65)
print("CELL 167: UNIQUENESS AND NOVELTY AUDIT")
print("=" * 65)

# Canonical generated molecules
generated_canonical = []

for smiles in greedy_smiles:
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        generated_canonical.append(
            Chem.MolToSmiles(mol, canonical=True)
        )

unique_generated = set(generated_canonical)

print("Total generated:", len(generated_canonical))
print("Unique generated:", len(unique_generated))

uniqueness = (
    len(unique_generated) / len(generated_canonical) * 100
)

print("Uniqueness:", round(uniqueness, 2), "%")

# Compare against ChEMBL training molecules
training_canonical = set()

for row in chembl_train_tokens:

    tokens = [
        id_to_token[int(idx)]
        for idx in row
        if int(idx) not in (pad_id, bos_id, eos_id)
    ]

    smiles = "".join(tokens)
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        training_canonical.add(
            Chem.MolToSmiles(mol, canonical=True)
        )

novel_generated = unique_generated - training_canonical

print("Unique training molecules:", len(training_canonical))
print("Novel generated molecules:", len(novel_generated))

print("\nCELL 167 COMPLETED")

CELL 167: UNIQUENESS AND NOVELTY AUDIT
Total generated: 30
Unique generated: 1
Uniqueness: 3.33 %
Unique training molecules: 45000
Novel generated molecules: 1

CELL 167 COMPLETED


In [232]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

print("=" * 65)
print("CELL 168: GENERATED MOLECULE DIAGNOSTICS")
print("=" * 65)

unique_smiles = list(set(greedy_smiles))

print("Total generated:", len(greedy_smiles))
print("Unique SMILES:", len(unique_smiles))

for i, smiles in enumerate(unique_smiles[:5], 1):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"\nMolecule {i}: Invalid")
        continue

    print(f"\nMolecule {i}")
    print("SMILES:", smiles)
    print("Atoms:", mol.GetNumAtoms())
    print("Molecular weight:", round(Descriptors.MolWt(mol), 2))
    print("LogP:", round(Descriptors.MolLogP(mol), 2))
    print("TPSA:", round(Descriptors.TPSA(mol), 2))
    print("H-bond donors:", Lipinski.NumHDonors(mol))
    print("H-bond acceptors:", Lipinski.NumHAcceptors(mol))
    print("Rotatable bonds:", Lipinski.NumRotatableBonds(mol))
    print("Rings:", rdMolDescriptors.CalcNumRings(mol))

print("\nCELL 168 COMPLETED")

CELL 168: GENERATED MOLECULE DIAGNOSTICS
Total generated: 30
Unique SMILES: 1

Molecule 1
SMILES: CC(C)(C)CC(=O)N1CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC1
Atoms: 52
Molecular weight: 730.35
LogP: 17.65
TPSA: 20.31
H-bond donors: 0
H-bond acceptors: 1
Rotatable bonds: 1
Rings: 1

CELL 168 COMPLETED


In [235]:
# ============================================================
# CELL 169: CHEMBL TRAINING DATA DISTRIBUTION AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

print("=" * 65)
print("CELL 169: CHEMBL MOLECULAR DISTRIBUTION AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Locate the cleaned ChEMBL dataset
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")

SMILES_FILE = (
    PROJECT_DIR
    / "member2_outputs"
    / "chembl37_clean_smiles.txt"
)

if not SMILES_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned ChEMBL file not found: {SMILES_FILE}"
    )

with open(SMILES_FILE, "r") as f:
    smiles_list = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("\nSTEP 1: DATASET")
print("Total SMILES records:", len(smiles_list))

# ------------------------------------------------------------
# 2. Validate molecules and calculate descriptors
# ------------------------------------------------------------

records = []
invalid_count = 0

for smi in smiles_list:

    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        invalid_count += 1
        continue

    canonical = Chem.MolToSmiles(
        mol,
        canonical=True
    )

    records.append({
        "SMILES": canonical,
        "Atoms": mol.GetNumAtoms(),
        "HeavyAtoms": Descriptors.HeavyAtomCount(mol),
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotatableBonds": Lipinski.NumRotatableBonds(mol),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "AromaticRings": Lipinski.NumAromaticRings(mol)
    })

df = pd.DataFrame(records)

print("\nSTEP 2: VALIDATION")
print("Valid molecules:", len(df))
print("Invalid molecules:", invalid_count)
print("Unique canonical molecules:", df["SMILES"].nunique())
print("Duplicate molecules:", len(df) - df["SMILES"].nunique())

# ------------------------------------------------------------
# 3. Numerical distribution
# ------------------------------------------------------------

properties = [
    "Atoms",
    "HeavyAtoms",
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "Rings",
    "AromaticRings"
]

print("\nSTEP 3: DESCRIPTOR DISTRIBUTION")

distribution = df[properties].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

print(distribution.round(3).to_string())

# ------------------------------------------------------------
# 4. Molecular property ranges
# ------------------------------------------------------------

print("\nSTEP 4: PROPERTY RANGE COUNTS")

checks = {
    "MW > 500": df["MW"] > 500,
    "MW > 700": df["MW"] > 700,
    "LogP > 5": df["LogP"] > 5,
    "LogP > 10": df["LogP"] > 10,
    "TPSA < 40": df["TPSA"] < 40,
    "TPSA > 140": df["TPSA"] > 140,
    "Heavy atoms > 50": df["HeavyAtoms"] > 50,
    "Rotatable bonds > 10": df["RotatableBonds"] > 10,
    "Rings > 4": df["Rings"] > 4
}

for label, condition in checks.items():

    count = int(condition.sum())
    percentage = count / len(df) * 100

    print(
        f"{label:25s}: "
        f"{count:7d} ({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# 5. Lipinski-style screening summary
# ------------------------------------------------------------

lipinski_violations = (
    (df["MW"] > 500).astype(int)
    + (df["LogP"] > 5).astype(int)
    + (df["HBD"] > 5).astype(int)
    + (df["HBA"] > 10).astype(int)
)

df["LipinskiViolations"] = lipinski_violations

print("\nSTEP 5: LIPINSKI VIOLATIONS")

print(
    df["LipinskiViolations"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 6. Inspect largest and most hydrophobic molecules
# ------------------------------------------------------------

print("\nSTEP 6: TOP 10 HIGHEST MOLECULAR WEIGHT")

print(
    df.nlargest(10, "MW")[
        ["SMILES", "MW", "LogP", "TPSA", "HeavyAtoms"]
    ].to_string(index=False)
)

print("\nTOP 10 HIGHEST LOGP")

print(
    df.nlargest(10, "LogP")[
        ["SMILES", "MW", "LogP", "TPSA", "HeavyAtoms"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 7. Save audit results
# ------------------------------------------------------------

OUTPUT_DIR = PROJECT_DIR / "member2_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

csv_path = OUTPUT_DIR / "chembl37_distribution_audit.csv"
summary_path = OUTPUT_DIR / "chembl37_distribution_summary.csv"

df.to_csv(csv_path, index=False)
distribution.to_csv(summary_path)

print("\nSTEP 7: SAVED")
print(csv_path)
print(summary_path)

print("\nCELL 169 COMPLETED")
print("=" * 65)

CELL 169: CHEMBL MOLECULAR DISTRIBUTION AUDIT

STEP 1: DATASET
Total SMILES records: 50000

STEP 2: VALIDATION
Valid molecules: 50000
Invalid molecules: 0
Unique canonical molecules: 50000
Duplicate molecules: 0

STEP 3: DESCRIPTOR DISTRIBUTION
                  count     mean      std      min      10%      25%      50%      75%      90%      95%      99%      max
Atoms           50000.0   29.252    8.881    5.000   19.000   23.000   28.000   34.000   40.000   45.000   59.000   75.000
HeavyAtoms      50000.0   29.252    8.881    5.000   19.000   23.000   28.000   34.000   40.000   45.000   59.000   75.000
MW              50000.0  414.782  124.846  101.113  276.286  330.334  400.197  478.407  566.646  637.746  838.419  999.543
LogP            50000.0    3.536    1.845  -10.288    1.386    2.429    3.502    4.628    5.752    6.553    8.377   16.466
TPSA            50000.0   84.603   41.469    0.000   41.570   58.200   78.670  102.740  131.262  155.100  233.641  474.900
HBD             5

In [234]:
# ============================================================
# CELL 169A: LOCATE CLEANED CHEMBL DATA
# ============================================================

import os
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}"
    )

print("Searching inside:", PROJECT_DIR)

matches = []

for root, dirs, files in os.walk(PROJECT_DIR):

    # Avoid searching unnecessary large data directories
    dirs[:] = [
        d for d in dirs
        if d not in {
            ".git",
            "__pycache__"
        }
    ]

    for filename in files:

        lower = filename.lower()

        if (
            "smiles" in lower
            or "chembl" in lower
            or lower.endswith(".smi")
        ):
            path = Path(root) / filename

            matches.append(path)

print("\nPOTENTIAL CHEMBL / SMILES FILES")
print("=" * 65)

if not matches:
    print("No matching files found.")

else:
    for i, path in enumerate(matches, 1):
        print(f"{i}. {path}")
        print(f"   Size: {path.stat().st_size / (1024**2):.2f} MB")

print("\nSEARCH COMPLETED")

Searching inside: /content/drive/MyDrive/Member2_Project

POTENTIAL CHEMBL / SMILES FILES
1. /content/drive/MyDrive/Member2_Project/checkpoints/transformer_chembl37_10k_best.pt
   Size: 8.07 MB
2. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl_parp_target_verification.json
   Size: 0.00 MB
3. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_graphs.jsonl
   Size: 135.83 MB
4. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_validation.csv
   Size: 3.72 MB
5. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_general_summary.json
   Size: 0.00 MB
6. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_parp_target_records.csv
   Size: 0.00 MB
7. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_parp_target_records.json
   Size: 0.00 MB
8. /content/drive/MyDrive/Member2_Project/member2_outputs/chembl37_parp_activity_records.csv
   Size: 0.70 MB
9. /content/drive/MyDrive/Member2_Project/member

In [236]:
# ============================================================
# CELL 170: AUDIT ACTUAL TRANSFORMER TRAINING INPUTS
# ============================================================

import json
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")
OUTPUT_DIR = PROJECT_DIR / "member2_outputs"

TRAIN_FILE = OUTPUT_DIR / "chembl_train_tokens.npy"
TOKENIZER_FILE = OUTPUT_DIR / "smiles_tokenizer.json"

print("=" * 65)
print("CELL 170: TRAINING INPUT AUDIT")
print("=" * 65)

# 1. Load training tokens

X_train = np.load(TRAIN_FILE)

print("\nTRAINING TOKEN ARRAY")
print("Shape:", X_train.shape)
print("Data type:", X_train.dtype)

# 2. Reproduce the exact 10,000-molecule selection

rng = np.random.default_rng(42)

selected_indices = rng.choice(
    len(X_train),
    size=min(10000, len(X_train)),
    replace=False
)

X_selected = X_train[selected_indices]

print("\nSELECTED TRAINING DATA")
print("Selected molecules:", len(X_selected))
print("Sequence length:", X_selected.shape[1])
print("Unique token sequences:", len(np.unique(X_selected, axis=0)))

# 3. Inspect tokenizer structure

with open(TOKENIZER_FILE, "r") as f:
    tokenizer_data = json.load(f)

print("\nTOKENIZER STRUCTURE")
print("Top-level keys:", list(tokenizer_data.keys()))

for key, value in tokenizer_data.items():

    if isinstance(value, dict):
        print(key, "-> dictionary with", len(value), "entries")

    elif isinstance(value, list):
        print(key, "-> list with", len(value), "entries")

    else:
        print(key, "->", type(value).__name__, str(value)[:100])

# 4. Token statistics

print("\nTOKEN STATISTICS")

for name, array in [
    ("Full training split", X_train),
    ("Selected 10k subset", X_selected)
]:

    nonzero = np.count_nonzero(array, axis=1)

    print(f"\n{name}")
    print("Mean nonzero tokens:", round(float(nonzero.mean()), 2))
    print("Median nonzero tokens:", float(np.median(nonzero)))
    print("Minimum:", int(nonzero.min()))
    print("Maximum:", int(nonzero.max()))

    print(
        "Sequences using >100 nonzero tokens:",
        int((nonzero > 100).sum())
    )

# 5. Token frequency

unique, counts = np.unique(X_selected, return_counts=True)

print("\nMOST FREQUENT TOKEN IDS")

order = np.argsort(counts)[::-1][:20]

for idx in order:
    print(
        f"Token ID {int(unique[idx])}: "
        f"{int(counts[idx])} occurrences"
    )

# 6. Save selection for reproducibility

np.save(
    OUTPUT_DIR / "transformer_selected_10k_indices.npy",
    selected_indices
)

print("\nSaved exact selected indices.")

print("\nCELL 170 COMPLETED")
print("=" * 65)

CELL 170: TRAINING INPUT AUDIT

TRAINING TOKEN ARRAY
Shape: (45000, 143)
Data type: int32

SELECTED TRAINING DATA
Selected molecules: 10000
Sequence length: 143
Unique token sequences: 10000

TOKENIZER STRUCTURE
Top-level keys: ['vocabulary', 'token_to_id', 'max_length', 'special_tokens']
vocabulary -> list with 81 entries
token_to_id -> dictionary with 81 entries
max_length -> int 143
special_tokens -> list with 4 entries

TOKEN STATISTICS

Full training split
Mean nonzero tokens: 51.36
Median nonzero tokens: 49.0
Minimum: 9
Maximum: 143
Sequences using >100 nonzero tokens: 646

Selected 10k subset
Mean nonzero tokens: 51.4
Median nonzero tokens: 50.0
Minimum: 11
Maximum: 136
Sequences using >100 nonzero tokens: 126

MOST FREQUENT TOKEN IDS
Token ID 0: 916018 occurrences
Token ID 77: 123802 occurrences
Token ID 22: 83692 occurrences
Token ID 6: 50819 occurrences
Token ID 7: 50819 occurrences
Token ID 10: 29558 occurrences
Token ID 27: 29398 occurrences
Token ID 11: 23968 occurrences
T

In [237]:
# ============================================================
# CELL 171: TOKEN MAPPING AND SEQUENCE DECODING AUDIT
# ============================================================

import json
import numpy as np
from pathlib import Path
from rdkit import Chem

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")
OUTPUT_DIR = PROJECT_DIR / "member2_outputs"

# Load data
X_train = np.load(OUTPUT_DIR / "chembl_train_tokens.npy")
indices = np.load(
    OUTPUT_DIR / "transformer_selected_10k_indices.npy"
)

X_selected = X_train[indices]

with open(OUTPUT_DIR / "smiles_tokenizer.json") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
id_to_token = {
    int(v): k for k, v in token_to_id.items()
}

special_tokens = tokenizer["special_tokens"]

print("=" * 65)
print("CELL 171: TOKEN MAPPING AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Special token identification
# ------------------------------------------------------------

print("\nSPECIAL TOKENS")

for token in special_tokens:
    print(repr(token), "-> ID:", token_to_id.get(token))

print("\nCOMPLETE TOKEN MAPPING")

for token_id in sorted(id_to_token):
    print(token_id, "->", repr(id_to_token[token_id]))

# ------------------------------------------------------------
# 2. Identify EOS and PAD
# ------------------------------------------------------------

pad_id = token_to_id.get("<PAD>")
bos_id = token_to_id.get("<BOS>")
eos_id = token_to_id.get("<EOS>")

print("\nSPECIAL TOKEN IDS")
print("PAD:", pad_id)
print("BOS:", bos_id)
print("EOS:", eos_id)

# ------------------------------------------------------------
# 3. Sequence structural audit
# ------------------------------------------------------------

eos_count = 0
missing_eos = 0
bos_count = 0
truncated = 0

for seq in X_selected:

    tokens = list(map(int, seq))

    if bos_id is not None and tokens[0] == bos_id:
        bos_count += 1

    if eos_id is not None and eos_id in tokens:
        eos_count += 1

    else:
        missing_eos += 1

    if pad_id is not None and pad_id not in tokens:
        truncated += 1

print("\nSEQUENCE STRUCTURE")
print("Sequences with BOS at position 0:", bos_count)
print("Sequences containing EOS:", eos_count)
print("Sequences missing EOS:", missing_eos)
print("Sequences without padding:", truncated)

# ------------------------------------------------------------
# 4. Decode sequences
# ------------------------------------------------------------

def decode_sequence(sequence):

    result = []

    for token_id in map(int, sequence):

        if token_id == pad_id:
            break

        if token_id == bos_id:
            continue

        if token_id == eos_id:
            break

        token = id_to_token.get(token_id)

        if token is None:
            return None

        result.append(token)

    return "".join(result)


valid = 0
invalid = 0
decoded_smiles = []

for seq in X_selected:

    smi = decode_sequence(seq)

    if smi is None:
        invalid += 1
        continue

    mol = Chem.MolFromSmiles(smi)

    if mol is not None:
        valid += 1
        decoded_smiles.append(smi)

    else:
        invalid += 1

print("\nDECODE VALIDATION")
print("Decoded valid molecules:", valid)
print("Invalid decoded molecules:", invalid)

print("\nFIRST 10 DECODED SEQUENCES")

for smi in decoded_smiles[:10]:
    print(smi)

# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

audit = {
    "selected_sequences": len(X_selected),
    "sequences_with_eos": eos_count,
    "sequences_missing_eos": missing_eos,
    "sequences_with_bos": bos_count,
    "valid_decoded_molecules": valid,
    "invalid_decoded_molecules": invalid,
    "special_tokens": special_tokens
}

with open(
    OUTPUT_DIR / "token_mapping_decoding_audit.json",
    "w"
) as f:
    json.dump(audit, f, indent=4)

print("\nCELL 171 COMPLETED")
print("=" * 65)

CELL 171: TOKEN MAPPING AUDIT

SPECIAL TOKENS
'<PAD>' -> ID: 0
'<BOS>' -> ID: 1
'<EOS>' -> ID: 2
'<UNK>' -> ID: 3

COMPLETE TOKEN MAPPING
0 -> '<PAD>'
1 -> '<BOS>'
2 -> '<EOS>'
3 -> '<UNK>'
4 -> '#'
5 -> '%10'
6 -> '('
7 -> ')'
8 -> '-'
9 -> '/'
10 -> '1'
11 -> '2'
12 -> '3'
13 -> '4'
14 -> '5'
15 -> '6'
16 -> '7'
17 -> '8'
18 -> '9'
19 -> '='
20 -> 'B'
21 -> 'Br'
22 -> 'C'
23 -> 'Cl'
24 -> 'F'
25 -> 'I'
26 -> 'N'
27 -> 'O'
28 -> 'P'
29 -> 'S'
30 -> '[10B]'
31 -> '[11CH3]'
32 -> '[11C]'
33 -> '[123I]'
34 -> '[125I]'
35 -> '[13C]'
36 -> '[14C]'
37 -> '[14cH]'
38 -> '[18F]'
39 -> '[B-]'
40 -> '[BH2-]'
41 -> '[C-]'
42 -> '[C@@H]'
43 -> '[C@@]'
44 -> '[C@H]'
45 -> '[C@]'
46 -> '[CH]'
47 -> '[N+]'
48 -> '[N-]'
49 -> '[N@+]'
50 -> '[N@@]'
51 -> '[N@]'
52 -> '[O+]'
53 -> '[O-]'
54 -> '[O]'
55 -> '[P+]'
56 -> '[P@+]'
57 -> '[P@@]'
58 -> '[P@]'
59 -> '[PH]'
60 -> '[S+]'
61 -> '[S-]'
62 -> '[S@+]'
63 -> '[S@@+]'
64 -> '[S@@]'
65 -> '[S@]'
66 -> '[SH+]'
67 -> '[Se]'
68 -> '[Si]'
69 -> '[c+]'
70 -

In [238]:
# ============================================================
# CELL 172: TRANSFORMER CHECKPOINT AND PREDICTION AUDIT
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Member2_Project")

CHECKPOINT = (
    PROJECT_DIR
    / "checkpoints"
    / "transformer_chembl37_10k_best.pt"
)

TRAIN_FILE = (
    PROJECT_DIR
    / "member2_outputs"
    / "chembl_train_tokens.npy"
)

DEVICE = torch.device("cpu")

print("=" * 65)
print("CELL 172: CHECKPOINT AND PREDICTION AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load checkpoint without modifying it
# ------------------------------------------------------------

if not CHECKPOINT.exists():
    raise FileNotFoundError(CHECKPOINT)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

print("\nCHECKPOINT TYPE:", type(checkpoint))

if isinstance(checkpoint, dict):

    print("CHECKPOINT KEYS:")

    for key, value in checkpoint.items():

        if isinstance(value, dict):
            print(key, "-> state dictionary:", len(value))

        elif torch.is_tensor(value):
            print(key, "-> tensor:", tuple(value.shape))

        else:
            print(key, "->", str(value)[:150])

else:
    raise TypeError("Unexpected checkpoint format")

# ------------------------------------------------------------
# 2. Restore the existing model architecture
# ------------------------------------------------------------

# Use the already-defined model class and existing model instance.
# This avoids guessing constructor arguments.

if "model" not in globals():
    raise RuntimeError(
        "Existing Transformer variable 'model' not found. "
        "Do not recreate the architecture yet."
    )

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

print("\nCheckpoint restored successfully.")

# ------------------------------------------------------------
# 3. Load a small training batch
# ------------------------------------------------------------

X_train = np.load(TRAIN_FILE)

batch = torch.tensor(
    X_train[:32],
    dtype=torch.long,
    device=DEVICE
)

# Teacher forcing:
# Input = BOS through penultimate token
# Target = next token

input_ids = batch[:, :-1]
target_ids = batch[:, 1:]

# ------------------------------------------------------------
# 4. Run molecular-only forward pass
# ------------------------------------------------------------

with torch.no_grad():

    logits = molecular_only_forward(
        model,
        input_ids
    )

    print("\nLOGIT SHAPE:", tuple(logits.shape))

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_ids.reshape(-1),
        ignore_index=0
    )

    predictions = logits.argmax(dim=-1)

    valid_positions = target_ids != 0

    correct = (
        (predictions == target_ids)
        & valid_positions
    ).sum()

    total = valid_positions.sum()

    accuracy = correct.float() / total.float()

    # --------------------------------------------------------
    # 5. Prediction entropy
    # --------------------------------------------------------

    probabilities = torch.softmax(logits, dim=-1)

    entropy = -(
        probabilities
        * torch.log(probabilities.clamp_min(1e-10))
    ).sum(dim=-1)

    entropy = entropy[valid_positions]

    # --------------------------------------------------------
    # 6. Token probability concentration
    # --------------------------------------------------------

    mean_probabilities = probabilities.mean(dim=(0, 1))

    top_probabilities, top_ids = torch.topk(
        mean_probabilities,
        k=10
    )

print("\nPREDICTION RESULTS")
print("Teacher-forced loss:", round(loss.item(), 4))
print("Teacher-forced token accuracy:", round(accuracy.item(), 4))
print("Mean prediction entropy:", round(entropy.mean().item(), 4))
print("Median prediction entropy:", round(entropy.median().item(), 4))

print("\nTOP 10 AVERAGE-PROBABILITY TOKENS")

for token_id, probability in zip(
    top_ids.tolist(),
    top_probabilities.tolist()
):
    print(
        f"Token ID {token_id}: "
        f"probability {probability:.5f}"
    )

print("\nCELL 172 COMPLETED")
print("=" * 65)

CELL 172: CHECKPOINT AND PREDICTION AUDIT

CHECKPOINT TYPE: <class 'dict'>
CHECKPOINT KEYS:
model_state_dict -> state dictionary: 58
optimizer_state_dict -> state dictionary: 2
epoch -> 2
train_loss -> 1.3710269425242854
validation_loss -> 1.3104469297797816
training_molecules -> 10000
training_type -> chembl37_general_molecular_pretraining
pocket_cross_attention_used -> False
vocabulary_size -> 81


AttributeError: 'Model' object has no attribute 'load_state_dict'

In [239]:
# ============================================================
# CELL 172A: IDENTIFY TRANSFORMER MODEL OBJECT
# ============================================================

import torch
import inspect

print("=" * 65)
print("CELL 172A: MODEL OBJECT IDENTIFICATION")
print("=" * 65)

print("\nOBJECTS NAMED model:")

if "model" in globals():
    print("Type:", type(model))
    print("Class:", model.__class__)
    print("Has load_state_dict:", hasattr(model, "load_state_dict"))

print("\nPYTORCH MODEL OBJECTS IN NOTEBOOK:")

for name, obj in list(globals().items()):

    try:
        if isinstance(obj, torch.nn.Module):

            print(
                f"\nVariable: {name}"
                f"\nClass: {obj.__class__.__name__}"
                f"\nParameters: {sum(p.numel() for p in obj.parameters()):,}"
            )

    except Exception:
        pass

print("\nMODEL CLASS DEFINITIONS:")

for name, obj in list(globals().items()):

    if inspect.isclass(obj):
        try:
            if issubclass(obj, torch.nn.Module):
                print(name)
        except TypeError:
            pass

print("\nCELL 172A COMPLETED")
print("=" * 65)

CELL 172A: MODEL OBJECT IDENTIFICATION

OBJECTS NAMED model:
Type: <class 'Bio.PDB.Model.Model'>
Class: <class 'Bio.PDB.Model.Model'>
Has load_state_dict: False

PYTORCH MODEL OBJECTS IN NOTEBOOK:

Variable: criterion
Class: CrossEntropyLoss
Parameters: 0

Variable: pocket_encoder
Class: DynamicPocketEncoder
Parameters: 50,944

Variable: molecular_transformer
Class: PocketConditionedTransformer
Parameters: 832,849

Variable: obj
Class: PocketConditionedTransformer
Parameters: 832,849

Variable: module
Class: Linear
Parameters: 10,449

Variable: transformer_embeddings
Class: Embedding
Parameters: 10,368

Variable: decoder_layers
Class: ModuleList
Parameters: 793,728

Variable: layer
Class: TransformerDecoderLayer
Parameters: 264,576

Variable: layers
Class: ModuleList
Parameters: 793,728

Variable: loss_function
Class: CrossEntropyLoss
Parameters: 0

MODEL CLASS DEFINITIONS:
MolecularGRU
DynamicPocketEncoder
PositionalEncoding
PocketConditionedTransformer

CELL 172A COMPLETED


In [240]:
# ============================================================
# CELL 172B: RESTORE TRANSFORMER AND AUDIT PREDICTIONS
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

CHECKPOINT_PATH = (
    PROJECT_DIR
    + "/checkpoints/transformer_chembl37_10k_best.pt"
)

TRAIN_PATH = (
    PROJECT_DIR
    + "/member2_outputs/chembl_train_tokens.npy"
)

print("=" * 65)
print("CELL 172B: TRANSFORMER PREDICTION AUDIT")
print("=" * 65)

# 1. Restore checkpoint

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"]
)

molecular_transformer.eval()

print("\nCHECKPOINT RESTORED")
print("Epoch:", checkpoint["epoch"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["validation_loss"])

# 2. Load a small batch

X_train = np.load(TRAIN_PATH)

batch = torch.tensor(
    X_train[:32],
    dtype=torch.long
)

input_ids = batch[:, :-1]
target_ids = batch[:, 1:]

# 3. Teacher-forced predictions

with torch.no_grad():

    logits = molecular_only_forward(
        molecular_transformer,
        input_ids
    )

    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_ids.reshape(-1),
        ignore_index=0
    )

    predictions = logits.argmax(dim=-1)

    valid_positions = target_ids != 0

    correct = (
        (predictions == target_ids)
        & valid_positions
    ).sum()

    total = valid_positions.sum()

    accuracy = correct.float() / total.float()

    # Prediction entropy

    probabilities = torch.softmax(logits, dim=-1)

    entropy = -(
        probabilities
        * torch.log(probabilities.clamp_min(1e-10))
    ).sum(dim=-1)

    entropy = entropy[valid_positions]

    # Average token probability

    mean_probabilities = probabilities.mean(dim=(0, 1))

    top_probabilities, top_ids = torch.topk(
        mean_probabilities,
        k=10
    )

# 4. Results

print("\nPREDICTION RESULTS")

print("Logit shape:", tuple(logits.shape))
print("Teacher-forced loss:", round(loss.item(), 4))
print("Token accuracy:", round(accuracy.item(), 4))
print("Mean entropy:", round(entropy.mean().item(), 4))
print("Median entropy:", round(entropy.median().item(), 4))

print("\nTOP 10 AVERAGE-PROBABILITY TOKENS")

for token_id, probability in zip(
    top_ids.tolist(),
    top_probabilities.tolist()
):
    print(
        f"Token ID {token_id}: "
        f"{probability:.5f}"
    )

print("\nCELL 172B COMPLETED")
print("=" * 65)

CELL 172B: TRANSFORMER PREDICTION AUDIT

CHECKPOINT RESTORED
Epoch: 2
Training loss: 1.3710269425242854
Validation loss: 1.3104469297797816

PREDICTION RESULTS
Logit shape: (32, 142, 81)
Teacher-forced loss: 1.2431
Token accuracy: 0.5859
Mean entropy: 1.3248
Median entropy: 1.4367

TOP 10 AVERAGE-PROBABILITY TOKENS
Token ID 77: 0.17016
Token ID 22: 0.16396
Token ID 2: 0.15733
Token ID 6: 0.14393
Token ID 10: 0.12840
Token ID 7: 0.04283
Token ID 11: 0.04272
Token ID 27: 0.02781
Token ID 26: 0.02166
Token ID 78: 0.02122

CELL 172B COMPLETED


In [241]:
# ============================================================
# CELL 173: TRANSFORMER CAUSAL MASKING INSPECTION
# ============================================================

import inspect
import torch

print("=" * 65)
print("CELL 173: TRANSFORMER IMPLEMENTATION AUDIT")
print("=" * 65)

# 1. Inspect molecular_only_forward()
print("\n1. MOLECULAR_ONLY_FORWARD FUNCTION")

try:
    print(inspect.getsource(molecular_only_forward))
except Exception as e:
    print("Could not retrieve function source:", e)

# 2. Inspect Transformer forward()
print("\n2. POCKET-CONDITIONED TRANSFORMER FORWARD")

try:
    print(inspect.getsource(
        molecular_transformer.__class__.forward
    ))
except Exception as e:
    print("Could not retrieve forward source:", e)

# 3. Inspect Transformer decoder configuration
print("\n3. DECODER CONFIGURATION")

print("Number of decoder layers:",
      len(molecular_transformer.decoder_layers))

for i, layer in enumerate(molecular_transformer.decoder_layers):
    print(f"\nLayer {i + 1}")
    print("Self-attention:", layer.self_attn)
    print("Cross-attention:", layer.multihead_attn)

# 4. Check model device and mode
print("\n4. MODEL STATUS")

print("Device:",
      next(molecular_transformer.parameters()).device)

print("Training mode:",
      molecular_transformer.training)

print("Vocabulary size:",
      molecular_transformer.output_projection.out_features
      if hasattr(molecular_transformer, "output_projection")
      else "Inspect output layer manually")

print("\nCELL 173 COMPLETED")
print("=" * 65)

CELL 173: TRANSFORMER IMPLEMENTATION AUDIT

1. MOLECULAR_ONLY_FORWARD FUNCTION
def molecular_only_forward(model, token_ids):

    batch_size, seq_length = token_ids.shape

    # Token and positional embeddings
    x = model.token_embedding(token_ids)
    x = model.position_encoding(x)

    # Padding mask
    padding_mask = token_ids == 0

    # Causal mask: prevent looking at future tokens
    causal_mask = torch.triu(
        torch.ones(
            seq_length,
            seq_length,
            device=token_ids.device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    # Apply molecular self-attention and feed-forward layers
    for layer in model.decoder.layers:

        if layer.norm_first:

            # Self-attention
            normalized = layer.norm1(x)

            attention_output = layer.self_attn(
                normalized,
                normalized,
                normalized,
                attn_mask=causal_mask,
                key_padding_mask=p

AttributeError: 'PocketConditionedTransformer' object has no attribute 'decoder_layers'

In [242]:
# ============================================================
# CELL 173B: CORRECTED TRANSFORMER DECODER INSPECTION
# ============================================================

print("=" * 65)
print("CELL 173B: DECODER CONFIGURATION")
print("=" * 65)

decoder_layers = molecular_transformer.decoder.layers

print("Number of decoder layers:", len(decoder_layers))

for i, layer in enumerate(decoder_layers):

    print(f"\nDECODER LAYER {i + 1}")

    print("Self-attention:", type(layer.self_attn).__name__)
    print("Cross-attention:", type(layer.multihead_attn).__name__)
    print("Feed-forward input:", layer.linear1.in_features)
    print("Feed-forward output:", layer.linear2.out_features)
    print("Norm first:", layer.norm_first)

    print("Self-attention heads:",
          layer.self_attn.num_heads)

    print("Dropout:", layer.dropout.p)

print("\nMODEL STATUS")

print("Device:",
      next(molecular_transformer.parameters()).device)

print("Training mode:",
      molecular_transformer.training)

print("Vocabulary size:",
      molecular_transformer.output_layer.out_features)

print("\nCELL 173B COMPLETED")
print("=" * 65)

CELL 173B: DECODER CONFIGURATION
Number of decoder layers: 3

DECODER LAYER 1
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Feed-forward input: 128
Feed-forward output: 128
Norm first: True
Self-attention heads: 4
Dropout: 0.1

DECODER LAYER 2
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Feed-forward input: 128
Feed-forward output: 128
Norm first: True
Self-attention heads: 4
Dropout: 0.1

DECODER LAYER 3
Self-attention: MultiheadAttention
Cross-attention: MultiheadAttention
Feed-forward input: 128
Feed-forward output: 128
Norm first: True
Self-attention heads: 4
Dropout: 0.1

MODEL STATUS
Device: cpu
Training mode: False
Vocabulary size: 81

CELL 173B COMPLETED


In [243]:
# ============================================================
# CELL 174: CAUSAL MASKING BEHAVIOUR TEST
# ============================================================

import torch

print("=" * 65)
print("CELL 174: CAUSAL MASKING VERIFICATION")
print("=" * 65)

molecular_transformer.eval()

# Select one real training sequence
test_input = torch.tensor(
    X_train[:1],
    dtype=torch.long,
    device=next(molecular_transformer.parameters()).device
)

# Create a modified copy
modified_input = test_input.clone()

# Select a prefix
prefix_length = 20

# Change tokens AFTER the prefix, keeping the prefix identical
# Use an existing molecular token ID
replacement_token = 22

modified_input[:, prefix_length + 1:] = replacement_token

with torch.no_grad():

    original_logits = molecular_only_forward(
        molecular_transformer,
        test_input
    )

    modified_logits = molecular_only_forward(
        molecular_transformer,
        modified_input
    )

# Compare predictions within the unchanged prefix
original_prefix = original_logits[:, :prefix_length, :]
modified_prefix = modified_logits[:, :prefix_length, :]

difference = torch.abs(
    original_prefix - modified_prefix
)

max_difference = difference.max().item()
mean_difference = difference.mean().item()

print("\nRESULTS")

print("Original input shape:", tuple(test_input.shape))
print("Modified input shape:", tuple(modified_input.shape))
print("Prefix length:", prefix_length)

print("Maximum prediction difference:", max_difference)
print("Mean prediction difference:", mean_difference)

if max_difference < 1e-5:
    print("\nPASS: Future tokens do not affect earlier predictions.")
else:
    print("\nWARNING: Earlier predictions changed.")
    print("Investigate causal masking implementation.")

print("\nCELL 174 COMPLETED")
print("=" * 65)

CELL 174: CAUSAL MASKING VERIFICATION

RESULTS
Original input shape: (1, 143)
Modified input shape: (1, 143)
Prefix length: 20
Maximum prediction difference: 0.0
Mean prediction difference: 0.0

PASS: Future tokens do not affect earlier predictions.

CELL 174 COMPLETED


In [244]:
# ============================================================
# CELL 175: TRAINING ALIGNMENT AND LOSS AUDIT
# ============================================================

import torch
import torch.nn as nn
import numpy as np

print("=" * 65)
print("CELL 175: TRAINING ALIGNMENT AND LOSS AUDIT")
print("=" * 65)

# 1. Inspect special tokens
print("\n1. SPECIAL TOKEN CONFIGURATION")

print("Tokenizer special tokens:")

for key, value in tokenizer.items():
    if "special" in key.lower():
        print(key, ":", value)

special_tokens = tokenizer.get("special_tokens", {})

print("\nSpecial token dictionary:", special_tokens)

# 2. Inspect training sequence
print("\n2. TRAINING SEQUENCE STRUCTURE")

sample = torch.tensor(X_train[:3], dtype=torch.long)

print("Original shape:", tuple(sample.shape))

input_ids = sample[:, :-1]
target_ids = sample[:, 1:]

print("Input shape:", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))

print("\nFirst sequence:")
print("Input tokens :", input_ids[0, :15].tolist())
print("Target tokens:", target_ids[0, :15].tolist())

print("\n3. SPECIAL TOKEN COUNTS")

for token_name, token_id in special_tokens.items():

    if isinstance(token_id, int):

        count = (sample == token_id).sum().item()

        print(f"{token_name}: ID={token_id}, count={count}")

# 4. Inspect loss function
print("\n4. LOSS FUNCTION INSPECTION")

print("Criterion type:", type(criterion).__name__)

if isinstance(criterion, nn.CrossEntropyLoss):

    print("Ignore index:", criterion.ignore_index)
    print("Reduction:", criterion.reduction)
    print("Label smoothing:", criterion.label_smoothing)

else:
    print("Criterion configuration requires manual inspection.")

# 5. Check padding positions in shifted targets
print("\n5. PADDING TARGET AUDIT")

pad_id = special_tokens.get("PAD", 0)

padding_targets = target_ids == pad_id

print("Total target positions:", target_ids.numel())
print("Padding target positions:", padding_targets.sum().item())
print("Non-padding target positions:", (~padding_targets).sum().item())

print("\n6. ALIGNMENT CHECK")

bos_id = special_tokens.get("BOS", 1)
eos_id = special_tokens.get("EOS", 2)

for i in range(3):

    sequence = sample[i].tolist()

    print(f"\nSequence {i + 1}")
    print("Starts with BOS:", sequence[0] == bos_id)
    print("Contains EOS:", eos_id in sequence)
    print("First input token:", input_ids[i, 0].item())
    print("First target token:", target_ids[i, 0].item())

print("\nCELL 175 COMPLETED")
print("=" * 65)

CELL 175: TRAINING ALIGNMENT AND LOSS AUDIT

1. SPECIAL TOKEN CONFIGURATION
Tokenizer special tokens:
special_tokens : ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

Special token dictionary: ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

2. TRAINING SEQUENCE STRUCTURE
Original shape: (3, 143)
Input shape: (3, 142)
Target shape: (3, 142)

First sequence:
Input tokens : [1, 26, 77, 10, 77, 77, 77, 6, 22, 6, 19, 27, 7, 26, 9]
Target tokens: [26, 77, 10, 77, 77, 77, 6, 22, 6, 19, 27, 7, 26, 9, 26]

3. SPECIAL TOKEN COUNTS


AttributeError: 'list' object has no attribute 'items'

In [245]:
# ============================================================
# CELL 175B: CORRECTED TRAINING ALIGNMENT AND LOSS AUDIT
# ============================================================

import torch
import torch.nn as nn

print("=" * 65)
print("CELL 175B: TRAINING ALIGNMENT AND LOSS AUDIT")
print("=" * 65)

# 1. Special token IDs
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

special_token_ids = {
    "PAD": PAD_ID,
    "BOS": BOS_ID,
    "EOS": EOS_ID,
    "UNK": UNK_ID
}

print("\n1. SPECIAL TOKEN COUNTS")

sample = torch.tensor(X_train[:3], dtype=torch.long)

for name, token_id in special_token_ids.items():
    count = (sample == token_id).sum().item()
    print(f"{name}: ID={token_id}, count={count}")

# 2. Input-target alignment
print("\n2. INPUT-TARGET ALIGNMENT")

input_ids = sample[:, :-1]
target_ids = sample[:, 1:]

print("Input shape:", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))

print("\nFirst 15 input tokens:")
print(input_ids[0, :15].tolist())

print("\nFirst 15 target tokens:")
print(target_ids[0, :15].tolist())

# 3. Check BOS and EOS positions
print("\n3. BOS AND EOS POSITION AUDIT")

for i in range(3):

    sequence = sample[i].tolist()

    bos_positions = [
        j for j, token in enumerate(sequence)
        if token == BOS_ID
    ]

    eos_positions = [
        j for j, token in enumerate(sequence)
        if token == EOS_ID
    ]

    print(f"\nSequence {i + 1}")
    print("BOS positions:", bos_positions)
    print("EOS positions:", eos_positions)

# 4. Padding targets
print("\n4. PADDING TARGET AUDIT")

padding_targets = target_ids == PAD_ID

print("Total target positions:", target_ids.numel())
print("Padding positions:", padding_targets.sum().item())
print("Non-padding positions:", (~padding_targets).sum().item())

# 5. Loss configuration
print("\n5. LOSS FUNCTION")

print("Criterion:", criterion)

if isinstance(criterion, nn.CrossEntropyLoss):

    print("Ignore index:", criterion.ignore_index)
    print("Reduction:", criterion.reduction)
    print("Label smoothing:", criterion.label_smoothing)

    if criterion.ignore_index == PAD_ID:
        print("PASS: PAD tokens are excluded by loss.")
    else:
        print("WARNING: PAD tokens are not excluded by ignore_index.")

# 6. Check actual training loop source if available
print("\n6. TRAINING FUNCTION INSPECTION")

import inspect

for function_name in [
    "train_one_epoch",
    "train_model",
    "molecular_only_forward"
]:

    function_obj = globals().get(function_name)

    if callable(function_obj):

        print(f"\n--- {function_name} ---")

        try:
            print(inspect.getsource(function_obj))
        except Exception as e:
            print("Source unavailable:", e)

print("\nCELL 175B COMPLETED")
print("=" * 65)

CELL 175B: TRAINING ALIGNMENT AND LOSS AUDIT

1. SPECIAL TOKEN COUNTS
PAD: ID=0, count=300
BOS: ID=1, count=3
EOS: ID=2, count=3
UNK: ID=3, count=0

2. INPUT-TARGET ALIGNMENT
Input shape: (3, 142)
Target shape: (3, 142)

First 15 input tokens:
[1, 26, 77, 10, 77, 77, 77, 6, 22, 6, 19, 27, 7, 26, 9]

First 15 target tokens:
[26, 77, 10, 77, 77, 77, 6, 22, 6, 19, 27, 7, 26, 9, 26]

3. BOS AND EOS POSITION AUDIT

Sequence 1
BOS positions: [0]
EOS positions: [39]

Sequence 2
BOS positions: [0]
EOS positions: [54]

Sequence 3
BOS positions: [0]
EOS positions: [33]

4. PADDING TARGET AUDIT
Total target positions: 426
Padding positions: 300
Non-padding positions: 126

5. LOSS FUNCTION
Criterion: CrossEntropyLoss()
Ignore index: 0
Reduction: mean
Label smoothing: 0.0
PASS: PAD tokens are excluded by loss.

6. TRAINING FUNCTION INSPECTION

--- molecular_only_forward ---
def molecular_only_forward(model, token_ids):

    batch_size, seq_length = token_ids.shape

    # Token and positional embedd

In [246]:
# ============================================================
# CELL 176: INDEPENDENT NEXT-TOKEN PREDICTION AUDIT
# ============================================================

import torch
import torch.nn.functional as F

print("=" * 65)
print("CELL 176: NEXT-TOKEN PREDICTION VERIFICATION")
print("=" * 65)

device = next(molecular_transformer.parameters()).device

molecular_transformer.eval()

# Select 32 training sequences
sequences = torch.tensor(
    X_train[:32],
    dtype=torch.long,
    device=device
)

# Shift inputs and targets
input_ids = sequences[:, :-1]
target_ids = sequences[:, 1:]

print("\n1. DATA ALIGNMENT")
print("Input shape:", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))

# Forward prediction
with torch.no_grad():

    logits = molecular_only_forward(
        molecular_transformer,
        input_ids
    )

print("\n2. MODEL OUTPUT")
print("Logits shape:", tuple(logits.shape))

# Calculate loss
loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    target_ids.reshape(-1),
    ignore_index=0
)

print("\n3. NEXT-TOKEN LOSS")
print("Independent loss:", round(loss.item(), 4))

# Calculate accuracy excluding padding
predictions = logits.argmax(dim=-1)

valid_positions = target_ids != 0

correct = (
    (predictions == target_ids) & valid_positions
).sum().item()

total = valid_positions.sum().item()

accuracy = correct / total if total > 0 else 0

print("\n4. TOKEN ACCURACY")
print("Correct predictions:", correct)
print("Valid target tokens:", total)
print("Accuracy:", round(accuracy * 100, 2), "%")

# EOS prediction audit
eos_targets = target_ids == 2

eos_total = eos_targets.sum().item()
eos_correct = (
    (predictions == 2) & eos_targets
).sum().item()

print("\n5. EOS PREDICTION")
print("EOS target positions:", eos_total)
print("Correct EOS predictions:", eos_correct)

if eos_total:
    print("EOS accuracy:",
          round(100 * eos_correct / eos_total, 2), "%")

print("\nCELL 176 COMPLETED")
print("=" * 65)

CELL 176: NEXT-TOKEN PREDICTION VERIFICATION

1. DATA ALIGNMENT
Input shape: (32, 142)
Target shape: (32, 142)

2. MODEL OUTPUT
Logits shape: (32, 142, 81)

3. NEXT-TOKEN LOSS
Independent loss: 1.2431

4. TOKEN ACCURACY
Correct predictions: 849
Valid target tokens: 1449
Accuracy: 58.59 %

5. EOS PREDICTION
EOS target positions: 32
Correct EOS predictions: 22
EOS accuracy: 68.75 %

CELL 176 COMPLETED


In [247]:
# ============================================================
# CELL 177: AUTOREGRESSIVE PROBABILITY AUDIT
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np

print("=" * 65)
print("CELL 177: AUTOREGRESSIVE PROBABILITY AUDIT")
print("=" * 65)

device = next(molecular_transformer.parameters()).device

molecular_transformer.eval()

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

# Select one real training sequence as a reference
reference = torch.tensor(
    X_train[:1],
    dtype=torch.long,
    device=device
)

# Start with BOS only
generated = torch.tensor(
    [[BOS_ID]],
    dtype=torch.long,
    device=device
)

print("\n1. REFERENCE SEQUENCE")
print("Reference shape:", tuple(reference.shape))

print("\n2. AUTOREGRESSIVE PREDICTIONS")

with torch.no_grad():

    for step in range(20):

        logits = molecular_only_forward(
            molecular_transformer,
            generated
        )

        # Prediction for the next token
        next_logits = logits[:, -1, :]

        probabilities = F.softmax(
            next_logits,
            dim=-1
        )

        # Top 5 predicted tokens
        top_probs, top_ids = torch.topk(
            probabilities,
            k=5,
            dim=-1
        )

        top_probs = top_probs[0].cpu().tolist()
        top_ids = top_ids[0].cpu().tolist()

        # Greedy prediction
        next_token = torch.argmax(
            probabilities,
            dim=-1,
            keepdim=True
        )

        print(f"\nStep {step + 1}")
        print("Generated length:", generated.shape[1])
        print("Top predictions:")

        for token_id, probability in zip(
            top_ids,
            top_probs
        ):
            print(
                f"  Token {token_id}: "
                f"Probability {probability:.4f}"
            )

        print("Selected token:", next_token.item())

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

        if next_token.item() == EOS_ID:
            print("\nEOS generated. Stopping.")
            break

print("\n3. FINAL TOKEN SEQUENCE")
print(generated[0].cpu().tolist())

print("\nCELL 177 COMPLETED")
print("=" * 65)

CELL 177: AUTOREGRESSIVE PROBABILITY AUDIT

1. REFERENCE SEQUENCE
Reference shape: (1, 143)

2. AUTOREGRESSIVE PREDICTIONS

Step 1
Generated length: 1
Top predictions:
  Token 22: Probability 0.7973
  Token 27: Probability 0.1167
  Token 26: Probability 0.0593
  Token 77: Probability 0.0086
  Token 23: Probability 0.0062
Selected token: 22

Step 2
Generated length: 2
Top predictions:
  Token 22: Probability 0.3728
  Token 27: Probability 0.2649
  Token 77: Probability 0.1665
  Token 26: Probability 0.0768
  Token 19: Probability 0.0353
Selected token: 22

Step 3
Generated length: 3
Top predictions:
  Token 6: Probability 0.4327
  Token 22: Probability 0.1887
  Token 27: Probability 0.1103
  Token 10: Probability 0.0985
  Token 77: Probability 0.0509
Selected token: 6

Step 4
Generated length: 4
Top predictions:
  Token 22: Probability 0.5437
  Token 19: Probability 0.3631
  Token 26: Probability 0.0305
  Token 27: Probability 0.0231
  Token 77: Probability 0.0169
Selected token: 22

St

In [248]:
# ============================================================
# CELL 178: GENERATED TOKEN DECODING AUDIT
# ============================================================

import json

print("=" * 65)
print("CELL 178: GENERATED TOKEN DECODING AUDIT")
print("=" * 65)

# Load tokenizer mapping
token_to_id = tokenizer["token_to_id"]

id_to_token = {
    int(token_id): token
    for token, token_id in token_to_id.items()
}

generated_ids = generated[0].cpu().tolist()

print("\n1. TOKEN-BY-TOKEN DECODING")

decoded_tokens = []

for position, token_id in enumerate(generated_ids):

    token = id_to_token.get(token_id, "<UNKNOWN>")

    decoded_tokens.append(token)

    print(
        f"Position {position:02d} | "
        f"ID {token_id:02d} | "
        f"Token: {token}"
    )

print("\n2. COMPLETE DECODED SEQUENCE")

decoded_sequence = "".join(decoded_tokens)

print(decoded_sequence)

# 3. Token frequency
print("\n3. TOKEN FREQUENCY")

from collections import Counter

frequency = Counter(generated_ids)

for token_id, count in frequency.most_common():

    print(
        f"ID {token_id:02d} | "
        f"{id_to_token.get(token_id, '<UNKNOWN>')} | "
        f"Count: {count}"
    )

# 4. RDKit validity
print("\n4. CHEMICAL VALIDITY")

from rdkit import Chem

mol = Chem.MolFromSmiles(decoded_sequence)

if mol is not None:

    print("RDKit validity: VALID")
    print("Canonical SMILES:",
          Chem.MolToSmiles(mol))

else:

    print("RDKit validity: INVALID")

print("\nCELL 178 COMPLETED")
print("=" * 65)

CELL 178: GENERATED TOKEN DECODING AUDIT

1. TOKEN-BY-TOKEN DECODING
Position 00 | ID 01 | Token: <BOS>
Position 01 | ID 22 | Token: C
Position 02 | ID 22 | Token: C
Position 03 | ID 06 | Token: (
Position 04 | ID 22 | Token: C
Position 05 | ID 07 | Token: )
Position 06 | ID 06 | Token: (
Position 07 | ID 22 | Token: C
Position 08 | ID 07 | Token: )
Position 09 | ID 22 | Token: C
Position 10 | ID 22 | Token: C
Position 11 | ID 06 | Token: (
Position 12 | ID 19 | Token: =
Position 13 | ID 27 | Token: O
Position 14 | ID 07 | Token: )
Position 15 | ID 26 | Token: N
Position 16 | ID 10 | Token: 1
Position 17 | ID 22 | Token: C
Position 18 | ID 22 | Token: C
Position 19 | ID 22 | Token: C
Position 20 | ID 22 | Token: C

2. COMPLETE DECODED SEQUENCE
<BOS>CC(C)(C)CC(=O)N1CCCC

3. TOKEN FREQUENCY
ID 22 | C | Count: 10
ID 06 | ( | Count: 3
ID 07 | ) | Count: 3
ID 01 | <BOS> | Count: 1
ID 19 | = | Count: 1
ID 27 | O | Count: 1
ID 26 | N | Count: 1
ID 10 | 1 | Count: 1

4. CHEMICAL VALIDITY
RDKit

In [249]:
# ============================================================
# CELL 179: STOCHASTIC GENERATION AND DIVERSITY AUDIT
# ============================================================

import torch
import torch.nn.functional as F
from rdkit import Chem
from collections import Counter

import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 179: STOCHASTIC GENERATION AUDIT")
print("=" * 65)

# Configuration
NUM_SAMPLES = 30
TEMPERATURE = 0.8
TOP_K = 20
MAX_LENGTH = 143

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

device = next(molecular_transformer.parameters()).device

molecular_transformer.eval()

id_to_token = {
    int(v): k
    for k, v in tokenizer["token_to_id"].items()
}

results = []

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("\nGenerating molecules...")

for sample_number in range(NUM_SAMPLES):

    generated_ids = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long,
        device=device
    )

    finished = False

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            logits = molecular_only_forward(
                molecular_transformer,
                generated_ids
            )

            next_logits = logits[:, -1, :].clone()

            # Temperature scaling
            next_logits = next_logits / TEMPERATURE

            # Prevent special tokens from being sampled
            next_logits[0, PAD_ID] = -float("inf")
            next_logits[0, BOS_ID] = -float("inf")
            next_logits[0, UNK_ID] = -float("inf")

            # Top-k sampling
            top_values, top_indices = torch.topk(
                next_logits,
                TOP_K,
                dim=-1
            )

            filtered_logits = torch.full_like(
                next_logits,
                -float("inf")
            )

            filtered_logits.scatter_(
                1,
                top_indices,
                top_values
            )

            probabilities = F.softmax(
                filtered_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            token_id = next_token.item()

            if token_id == EOS_ID:
                finished = True
                break

            generated_ids = torch.cat(
                [generated_ids, next_token],
                dim=1
            )

    # Remove BOS and convert molecular tokens to SMILES
    token_sequence = generated_ids[0].cpu().tolist()

    smiles = "".join(
        id_to_token[token_id]
        for token_id in token_sequence
        if token_id not in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]
    )

    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:

        canonical = Chem.MolToSmiles(mol)
        valid = True

    else:

        canonical = None
        valid = False

    results.append({
        "sample": sample_number + 1,
        "smiles": smiles,
        "canonical_smiles": canonical,
        "valid": valid,
        "eos_generated": finished,
        "length": len(smiles)
    })

    print(
        f"Sample {sample_number + 1:02d} | "
        f"Valid: {valid} | "
        f"EOS: {finished} | "
        f"Length: {len(smiles)}"
    )

# Results dataframe
generation_df = pd.DataFrame(results)

valid_df = generation_df[
    generation_df["valid"]
]

unique_molecules = valid_df[
    "canonical_smiles"
].nunique()

valid_count = len(valid_df)

uniqueness = (
    unique_molecules / valid_count * 100
    if valid_count else 0
)

validity = valid_count / NUM_SAMPLES * 100

print("\n" + "=" * 65)
print("GENERATION SUMMARY")
print("=" * 65)

print("Total generated:", NUM_SAMPLES)
print("Valid molecules:", valid_count)
print("Invalid molecules:", NUM_SAMPLES - valid_count)
print("Validity:", round(validity, 2), "%")
print("Unique valid molecules:", unique_molecules)
print("Uniqueness:", round(uniqueness, 2), "%")
print("EOS termination:",
      generation_df["eos_generated"].sum())

print("\nVALID MOLECULES")

for smiles in valid_df["canonical_smiles"].tolist():
    print(smiles)

# Save results
output_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell179_stochastic_audit.csv"
)

generation_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\nCELL 179 COMPLETED")
print("=" * 65)

CELL 179: STOCHASTIC GENERATION AUDIT

Generating molecules...
Sample 01 | Valid: False | EOS: True | Length: 29
Sample 02 | Valid: False | EOS: True | Length: 52
Sample 03 | Valid: False | EOS: True | Length: 38
Sample 04 | Valid: False | EOS: True | Length: 67
Sample 05 | Valid: False | EOS: True | Length: 33
Sample 06 | Valid: False | EOS: True | Length: 47
Sample 07 | Valid: False | EOS: True | Length: 39
Sample 08 | Valid: False | EOS: True | Length: 57
Sample 09 | Valid: False | EOS: True | Length: 42
Sample 10 | Valid: False | EOS: True | Length: 57
Sample 11 | Valid: False | EOS: True | Length: 23
Sample 12 | Valid: False | EOS: True | Length: 68
Sample 13 | Valid: False | EOS: True | Length: 32
Sample 14 | Valid: False | EOS: True | Length: 57
Sample 15 | Valid: False | EOS: True | Length: 37
Sample 16 | Valid: False | EOS: True | Length: 47
Sample 17 | Valid: False | EOS: True | Length: 29
Sample 18 | Valid: False | EOS: True | Length: 28
Sample 19 | Valid: False | EOS: True 

In [250]:
# ============================================================
# CELL 180: INVALID SMILES DIAGNOSTICS
# ============================================================

from collections import Counter
from rdkit import Chem
from rdkit import RDLogger
import pandas as pd

print("=" * 65)
print("CELL 180: INVALID SMILES DIAGNOSTICS")
print("=" * 65)

# Suppress RDKit error messages during batch inspection
RDLogger.DisableLog("rdApp.error")

def inspect_smiles(smiles):

    result = {}

    result["length"] = len(smiles)
    result["open_parentheses"] = smiles.count("(")
    result["close_parentheses"] = smiles.count(")")
    result["parentheses_balanced"] = (
        smiles.count("(") == smiles.count(")")
    )

    result["open_brackets"] = smiles.count("[")
    result["close_brackets"] = smiles.count("]")
    result["brackets_balanced"] = (
        smiles.count("[") == smiles.count("]")
    )

    # Ring closure digits
    ring_digits = Counter(
        char for char in smiles if char.isdigit()
    )

    result["ring_digits"] = dict(ring_digits)

    result["ring_digits_balanced"] = all(
        count % 2 == 0
        for count in ring_digits.values()
    )

    mol = Chem.MolFromSmiles(smiles)

    result["rdkit_valid"] = mol is not None

    return result


diagnostics = []

for _, row in generation_df.iterrows():

    smiles = row["smiles"]

    info = inspect_smiles(smiles)

    diagnostics.append({
        "sample": row["sample"],
        "smiles": smiles,
        **info
    })


diagnostic_df = pd.DataFrame(diagnostics)

# 1. Display first 10 invalid molecules
print("\n1. FIRST 10 GENERATED SMILES")

for _, row in diagnostic_df.head(10).iterrows():

    print(f"\nSample {row['sample']}")
    print("SMILES:", row["smiles"])
    print("Parentheses balanced:",
          row["parentheses_balanced"])
    print("Brackets balanced:",
          row["brackets_balanced"])
    print("Ring digits:", row["ring_digits"])
    print("Ring digits balanced:",
          row["ring_digits_balanced"])


# 2. Aggregate structural diagnostics
print("\n2. STRUCTURAL ERROR SUMMARY")

print(
    "Unbalanced parentheses:",
    (~diagnostic_df["parentheses_balanced"]).sum()
)

print(
    "Unbalanced brackets:",
    (~diagnostic_df["brackets_balanced"]).sum()
)

print(
    "Unbalanced ring digits:",
    (~diagnostic_df["ring_digits_balanced"]).sum()
)

print(
    "RDKit valid:",
    diagnostic_df["rdkit_valid"].sum()
)


# 3. Token frequency across generated molecules
print("\n3. GENERATED CHARACTER FREQUENCY")

all_characters = Counter(
    "".join(generation_df["smiles"].tolist())
)

for char, count in all_characters.most_common(20):

    print(repr(char), ":", count)


# 4. Save detailed diagnostics
diagnostic_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell180_invalid_smiles_diagnostics.csv"
)

diagnostic_df.to_csv(diagnostic_path, index=False)

print("\nSaved:", diagnostic_path)

print("\nCELL 180 COMPLETED")
print("=" * 65)

RDLogger.EnableLog("rdApp.error")

CELL 180: INVALID SMILES DIAGNOSTICS

1. FIRST 10 GENERATED SMILES

Sample 1
SMILES: CC(C)C(C)N1CCCCCC1=C(O)C2)CC1
Parentheses balanced: False
Brackets balanced: True
Ring digits: {'1': 3, '2': 1}
Ring digits balanced: False

Sample 2
SMILES: COc1cccc(C(=O)NCCC(=O)c2cccc3c(S(=O)=O)c(C)c3C#Nccc1
Parentheses balanced: False
Brackets balanced: True
Ring digits: {'1': 2, '2': 1, '3': 2}
Ring digits balanced: False

Sample 3
SMILES: COc1cc[nH]c2cc(NC(=O)C(c3cccc6)CC2)c1O
Parentheses balanced: True
Brackets balanced: True
Ring digits: {'1': 2, '2': 2, '3': 1, '6': 1}
Ring digits balanced: False

Sample 4
SMILES: COc1ccc(F)c1C(CC)(OCOCCC(=O)Nc1nc(C)ccc(N)c1ccc1C(N2ccccccccc(O)c1O
Parentheses balanced: False
Brackets balanced: True
Ring digits: {'1': 6, '2': 1}
Ring digits balanced: False

Sample 5
SMILES: COc1cc(C(=O)NC(=O)Nc2ccc1C(CO)cc1
Parentheses balanced: False
Brackets balanced: True
Ring digits: {'1': 3, '2': 1}
Ring digits balanced: False

Sample 6
SMILES: COc1cc(OC2=Nc3cc(=O)NCCCC(C3

In [251]:
# ============================================================
# CELL 181: TRAINING VS GENERATED SMILES COMPARISON
# ============================================================

from rdkit import Chem
from collections import Counter
import numpy as np

print("=" * 65)
print("CELL 181: TRAINING VS GENERATED SMILES AUDIT")
print("=" * 65)

# Reconstruct token mapping
id_to_token = {
    int(v): k
    for k, v in tokenizer["token_to_id"].items()
}

# Decode training sequences
def decode_sequence(token_ids):

    tokens = []

    for token_id in token_ids:

        token = id_to_token.get(int(token_id), "")

        if token in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]:
            continue

        tokens.append(token)

    return "".join(tokens)


training_smiles = [
    decode_sequence(sequence)
    for sequence in X_train[:1000]
]

generated_smiles = generation_df["smiles"].tolist()


# 1. Training validity
training_valid = sum(
    Chem.MolFromSmiles(s) is not None
    for s in training_smiles
)

# 2. Generated validity
generated_valid = sum(
    Chem.MolFromSmiles(s) is not None
    for s in generated_smiles
)

print("\n1. VALIDITY COMPARISON")

print("Training samples:", len(training_smiles))
print("Valid training:", training_valid)

print("Generated samples:", len(generated_smiles))
print("Valid generated:", generated_valid)


# 3. Sequence length comparison
train_lengths = [len(s) for s in training_smiles]
generated_lengths = [len(s) for s in generated_smiles]

print("\n2. SEQUENCE LENGTH COMPARISON")

print("Training mean length:",
      round(np.mean(train_lengths), 2))

print("Generated mean length:",
      round(np.mean(generated_lengths), 2))

print("Training median length:",
      np.median(train_lengths))

print("Generated median length:",
      np.median(generated_lengths))


# 4. Token distribution comparison
print("\n3. TOKEN DISTRIBUTION COMPARISON")

train_counter = Counter("".join(training_smiles))
generated_counter = Counter("".join(generated_smiles))

all_tokens = sorted(
    set(train_counter) | set(generated_counter)
)

print(f"{'Token':<10}{'Training':<15}{'Generated':<15}")

for token in all_tokens:

    print(
        f"{token:<10}"
        f"{train_counter[token]:<15}"
        f"{generated_counter[token]:<15}"
    )


# 5. Check training examples
print("\n4. FIRST 10 TRAINING EXAMPLES")

for i, smiles in enumerate(training_smiles[:10], 1):

    valid = Chem.MolFromSmiles(smiles) is not None

    print(f"{i}. {smiles} | Valid: {valid}")


# 6. Check generated sequences against training examples
print("\n5. EXACT TRAINING MATCHES")

training_set = set(training_smiles)

matches = sum(
    smiles in training_set
    for smiles in generated_smiles
)

print("Generated sequences:", len(generated_smiles))
print("Exact training matches:", matches)


print("\nCELL 181 COMPLETED")
print("=" * 65)

CELL 181: TRAINING VS GENERATED SMILES AUDIT

1. VALIDITY COMPARISON
Training samples: 1000
Valid training: 1000
Generated samples: 30
Valid generated: 0

2. SEQUENCE LENGTH COMPARISON
Training mean length: 54.32
Generated mean length: 46.53
Training median length: 50.0
Generated median length: 44.5

3. TOKEN DISTRIBUTION COMPARISON
Token     Training       Generated      
#         59             4              
(         5108           151            
)         5108           149            
+         63             0              
-         411            7              
/         218            4              
1         2903           88             
2         2384           55             
3         1362           28             
4         542            11             
5         128            0              
6         34             1              
7         12             0              
8         1              0              
=         1960           63             
@        

In [252]:
# ============================================================
# CELL 182: TEACHER-FORCED VS AUTOREGRESSIVE AUDIT
# ============================================================

import torch
import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 182: TEACHER-FORCED VS AUTOREGRESSIVE AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load model and tokenizer information
# ------------------------------------------------------------

molecular_transformer.eval()

device = next(molecular_transformer.parameters()).device

token_to_id = tokenizer["token_to_id"]

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]

print("Device:", device)
print("PAD:", PAD_ID, "BOS:", BOS_ID, "EOS:", EOS_ID)

# ------------------------------------------------------------
# 2. Select evaluation sequences
# ------------------------------------------------------------

NUM_SAMPLES = 32

eval_data = torch.tensor(
    X_train[:NUM_SAMPLES],
    dtype=torch.long,
    device=device
)

print("Evaluation shape:", eval_data.shape)

# ------------------------------------------------------------
# 3. Teacher-forced predictions
# ------------------------------------------------------------

input_ids = eval_data[:, :-1]
target_ids = eval_data[:, 1:]

with torch.no_grad():

    logits = molecular_only_forward(
        molecular_transformer,
        input_ids
    )

    teacher_predictions = logits.argmax(dim=-1)

valid_mask = target_ids != PAD_ID

correct = (
    (teacher_predictions == target_ids) & valid_mask
).sum().item()

total = valid_mask.sum().item()

teacher_accuracy = correct / total

print("\nTEACHER-FORCED RESULTS")
print("----------------------")
print("Correct predictions:", correct)
print("Total predictions:", total)
print("Accuracy:", round(teacher_accuracy * 100, 2), "%")

# ------------------------------------------------------------
# 4. Autoregressive generation
# ------------------------------------------------------------

generated = torch.full(
    (NUM_SAMPLES, 1),
    BOS_ID,
    dtype=torch.long,
    device=device
)

finished = torch.zeros(
    NUM_SAMPLES,
    dtype=torch.bool,
    device=device
)

MAX_STEPS = min(100, eval_data.shape[1] - 1)

with torch.no_grad():

    for step in range(MAX_STEPS):

        logits = molecular_only_forward(
            molecular_transformer,
            generated
        )

        next_logits = logits[:, -1, :]

        next_tokens = next_logits.argmax(dim=-1)

        next_tokens[finished] = PAD_ID

        generated = torch.cat(
            [generated, next_tokens.unsqueeze(1)],
            dim=1
        )

        finished = finished | (next_tokens == EOS_ID)

        if finished.all():
            break

print("\nAUTOREGRESSIVE RESULTS")
print("----------------------")
print("Generated shape:", tuple(generated.shape))
print("Sequences reaching EOS:", int(finished.sum().item()), "/", NUM_SAMPLES)

# ------------------------------------------------------------
# 5. Compare each generated sequence with its actual sequence
# ------------------------------------------------------------

first_error_positions = []
prefix_match_lengths = []

for i in range(NUM_SAMPLES):

    actual = eval_data[i].tolist()
    predicted = generated[i].tolist()

    # Find actual EOS
    if EOS_ID in actual:
        actual_end = actual.index(EOS_ID)
    else:
        actual_end = len(actual) - 1

    # Exclude BOS, include EOS
    actual_tokens = actual[1:actual_end + 1]
    predicted_tokens = predicted[1:len(actual_tokens) + 1]

    match_length = 0
    first_error = None

    for j, (a, p) in enumerate(
        zip(actual_tokens, predicted_tokens),
        start=1
    ):

        if a == p:
            match_length += 1
        else:
            first_error = j
            break

    if first_error is None and len(predicted_tokens) < len(actual_tokens):
        first_error = len(predicted_tokens) + 1

    first_error_positions.append(
        first_error if first_error is not None else -1
    )

    prefix_match_lengths.append(match_length)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

results = pd.DataFrame({
    "sample": range(1, NUM_SAMPLES + 1),
    "first_error_position": first_error_positions,
    "matching_prefix_tokens": prefix_match_lengths
})

print("\nPREFIX COMPARISON")
print("-----------------")
print(results.to_string(index=False))

print("\nSUMMARY")
print("-------")
print(
    "Average matching prefix:",
    round(np.mean(prefix_match_lengths), 2)
)

print(
    "Sequences matching completely:",
    sum(x == -1 for x in first_error_positions),
    "/",
    NUM_SAMPLES
)

print(
    "Average first error position:",
    round(
        np.mean([x for x in first_error_positions if x != -1]),
        2
    ) if any(x != -1 for x in first_error_positions)
    else "No errors"
)

print("\nCELL 182 COMPLETED")

CELL 182: TEACHER-FORCED VS AUTOREGRESSIVE AUDIT
Device: cpu
PAD: 0 BOS: 1 EOS: 2
Evaluation shape: torch.Size([32, 143])

TEACHER-FORCED RESULTS
----------------------
Correct predictions: 849
Total predictions: 1449
Accuracy: 58.59 %

AUTOREGRESSIVE RESULTS
----------------------
Generated shape: (32, 63)
Sequences reaching EOS: 32 / 32

PREFIX COMPARISON
-----------------
 sample  first_error_position  matching_prefix_tokens
      1                     1                       0
      2                     2                       1
      3                     2                       1
      4                     2                       1
      5                     2                       1
      6                     3                       2
      7                     2                       1
      8                     2                       1
      9                     2                       1
     10                     1                       0
     11                     

In [253]:
# ============================================================
# CELL 183: GREEDY GENERATION QUALITY AUDIT
# ============================================================

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

print("=" * 65)
print("CELL 183: GREEDY GENERATION QUALITY AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Prepare token decoder
# ------------------------------------------------------------

id_to_token = {
    int(v): k
    for k, v in tokenizer["token_to_id"].items()
}

PAD_ID = tokenizer["token_to_id"]["<PAD>"]
BOS_ID = tokenizer["token_to_id"]["<BOS>"]
EOS_ID = tokenizer["token_to_id"]["<EOS>"]
UNK_ID = tokenizer["token_to_id"]["<UNK>"]

SPECIAL_IDS = {PAD_ID, BOS_ID, EOS_ID, UNK_ID}

# ------------------------------------------------------------
# 2. Decode generated sequences
# ------------------------------------------------------------

def decode_sequence(sequence):

    tokens = []

    for token_id in sequence:

        token_id = int(token_id)

        if token_id == EOS_ID:
            break

        if token_id in SPECIAL_IDS:
            continue

        tokens.append(id_to_token[token_id])

    return "".join(tokens)


generated_smiles = [
    decode_sequence(sequence)
    for sequence in generated.cpu().tolist()
]

print("\nTotal generated:", len(generated_smiles))

# ------------------------------------------------------------
# 3. RDKit validity and descriptors
# ------------------------------------------------------------

records = []

for index, smiles in enumerate(generated_smiles, start=1):

    mol = Chem.MolFromSmiles(smiles)

    record = {
        "sample": index,
        "smiles": smiles,
        "valid": mol is not None
    }

    if mol is not None:

        record.update({
            "canonical_smiles": Chem.MolToSmiles(mol),
            "molecular_weight": Descriptors.MolWt(mol),
            "logP": Descriptors.MolLogP(mol),
            "TPSA": Descriptors.TPSA(mol),
            "HBD": Lipinski.NumHDonors(mol),
            "HBA": Lipinski.NumHAcceptors(mol),
            "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
            "heavy_atoms": Descriptors.HeavyAtomCount(mol),
            "rings": Lipinski.RingCount(mol)
        })

    records.append(record)


greedy_df = pd.DataFrame(records)

valid_df = greedy_df[greedy_df["valid"]].copy()

unique_count = (
    valid_df["canonical_smiles"].nunique()
    if len(valid_df) > 0 else 0
)

# ------------------------------------------------------------
# 4. Print results
# ------------------------------------------------------------

print("\nVALIDITY")
print("--------")
print("Total:", len(greedy_df))
print("Valid:", len(valid_df))
print("Invalid:", len(greedy_df) - len(valid_df))

print("\nUNIQUENESS")
print("----------")
print("Unique valid molecules:", unique_count)

if len(valid_df) > 0:

    print(
        "Uniqueness percentage:",
        round(unique_count / len(valid_df) * 100, 2)
    )

    print("\nDESCRIPTOR SUMMARY")
    print("------------------")

    descriptor_columns = [
        "molecular_weight",
        "logP",
        "TPSA",
        "HBD",
        "HBA",
        "rotatable_bonds",
        "heavy_atoms",
        "rings"
    ]

    print(valid_df[descriptor_columns].describe().round(2))

    print("\nFIRST 10 VALID MOLECULES")
    print("------------------------")

    print(
        valid_df[
            ["sample", "smiles", "molecular_weight", "logP"]
        ].head(10).to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

output_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell183_greedy_generation_audit.csv"
)

greedy_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\nCELL 183 COMPLETED")

CELL 183: GREEDY GENERATION QUALITY AUDIT

Total generated: 32

VALIDITY
--------
Total: 32
Valid: 32
Invalid: 0

UNIQUENESS
----------
Unique valid molecules: 1
Uniqueness percentage: 3.12

DESCRIPTOR SUMMARY
------------------
       molecular_weight   logP   TPSA   HBD   HBA  rotatable_bonds  \
count             32.00  32.00  32.00  32.0  32.0             32.0   
mean             730.35  17.65  20.31   0.0   1.0              1.0   
std                0.00   0.00   0.00   0.0   0.0              0.0   
min              730.35  17.65  20.31   0.0   1.0              1.0   
25%              730.35  17.65  20.31   0.0   1.0              1.0   
50%              730.35  17.65  20.31   0.0   1.0              1.0   
75%              730.35  17.65  20.31   0.0   1.0              1.0   
max              730.35  17.65  20.31   0.0   1.0              1.0   

       heavy_atoms  rings  
count         32.0   32.0  
mean          52.0    1.0  
std            0.0    0.0  
min           52.0    1.0  


In [255]:
import os
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

VAL_PATH = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "chembl_val_tokens.npy"
)

X_val = np.load(VAL_PATH)

print("Validation data loaded successfully")
print("Shape:", X_val.shape)
print("Data type:", X_val.dtype)
print("Contains NaN:", np.isnan(X_val).any())

Validation data loaded successfully
Shape: (5000, 143)
Data type: int32
Contains NaN: False


In [256]:
# ============================================================
# CELL 184: TRAIN VS VALIDATION PERFORMANCE AUDIT
# ============================================================

import torch
import numpy as np
import pandas as pd

print("=" * 65)
print("CELL 184: TRAIN VS VALIDATION PERFORMANCE AUDIT")
print("=" * 65)

molecular_transformer.eval()

device = next(molecular_transformer.parameters()).device

PAD_ID = tokenizer["token_to_id"]["<PAD>"]

# ------------------------------------------------------------
# 1. Evaluation function
# ------------------------------------------------------------

def evaluate_dataset(data, batch_size=32, max_samples=512):

    data = torch.as_tensor(data[:max_samples], dtype=torch.long)

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    batch_results = []

    with torch.no_grad():

        for start in range(0, len(data), batch_size):

            batch = data[start:start + batch_size].to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            logits = molecular_only_forward(
                molecular_transformer,
                inputs
            )

            loss_sum = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            valid_mask = targets != PAD_ID

            correct = (
                (predictions == targets) & valid_mask
            ).sum().item()

            token_count = valid_mask.sum().item()

            total_loss += loss_sum.item()
            total_correct += correct
            total_tokens += token_count

            batch_results.append({
                "batch_start": start,
                "loss": loss_sum.item() / max(token_count, 1),
                "accuracy": correct / max(token_count, 1)
            })

    return {
        "loss": total_loss / total_tokens,
        "accuracy": total_correct / total_tokens,
        "tokens": total_tokens,
        "batches": pd.DataFrame(batch_results)
    }


# ------------------------------------------------------------
# 2. Evaluate training and validation
# ------------------------------------------------------------

print("\nEvaluating training data...")

train_result = evaluate_dataset(
    X_train,
    max_samples=512
)

print("Evaluating validation data...")

val_result = evaluate_dataset(
    X_val,
    max_samples=512
)

# ------------------------------------------------------------
# 3. Display comparison
# ------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "dataset": "Training",
        "loss": train_result["loss"],
        "accuracy": train_result["accuracy"] * 100,
        "tokens": train_result["tokens"]
    },
    {
        "dataset": "Validation",
        "loss": val_result["loss"],
        "accuracy": val_result["accuracy"] * 100,
        "tokens": val_result["tokens"]
    }
])

print("\nPERFORMANCE COMPARISON")
print("----------------------")

print(comparison.round(4).to_string(index=False))

print("\nGENERALIZATION GAP")
print("------------------")

print(
    "Validation - Training loss:",
    round(val_result["loss"] - train_result["loss"], 4)
)

print(
    "Training - Validation accuracy:",
    round(
        (train_result["accuracy"] - val_result["accuracy"]) * 100,
        2
    ),
    "percentage points"
)

# ------------------------------------------------------------
# 4. Batch-level consistency
# ------------------------------------------------------------

print("\nTRAINING BATCH LOSS")
print("-------------------")
print(train_result["batches"].describe().round(4))

print("\nVALIDATION BATCH LOSS")
print("---------------------")
print(val_result["batches"].describe().round(4))

print("\nCELL 184 COMPLETED")

CELL 184: TRAIN VS VALIDATION PERFORMANCE AUDIT

Evaluating training data...
Evaluating validation data...

PERFORMANCE COMPARISON
----------------------
   dataset  loss  accuracy  tokens
  Training 1.319   56.8990   26301
Validation 1.324   56.9868   25820

GENERALIZATION GAP
------------------
Validation - Training loss: 0.0051
Training - Validation accuracy: -0.09 percentage points

TRAINING BATCH LOSS
-------------------
       batch_start     loss  accuracy
count      16.0000  16.0000   16.0000
mean      240.0000   1.3164    0.5696
std       152.3505   0.0542    0.0167
min         0.0000   1.2431    0.5435
25%       120.0000   1.2750    0.5540
50%       240.0000   1.3011    0.5724
75%       360.0000   1.3689    0.5828
max       480.0000   1.3850    0.6009

VALIDATION BATCH LOSS
---------------------
       batch_start     loss  accuracy
count      16.0000  16.0000   16.0000
mean      240.0000   1.3232    0.5702
std       152.3505   0.0424    0.0144
min         0.0000   1.2528    

In [257]:
# ============================================================
# CELL 185: TRAINING CONFIGURATION AUDIT
# ============================================================

import torch

print("=" * 65)
print("CELL 185: TRAINING CONFIGURATION AUDIT")
print("=" * 65)

checkpoint_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "checkpoints/transformer_chembl37_10k_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\n1. CHECKPOINT METADATA")
print("----------------------")

for key, value in checkpoint.items():

    if key not in [
        "model_state_dict",
        "optimizer_state_dict"
    ]:
        print(f"{key}: {value}")

print("\n2. OPTIMIZER INFORMATION")
print("------------------------")

optimizer_state = checkpoint.get(
    "optimizer_state_dict"
)

if optimizer_state:

    print("Optimizer:", optimizer_state.get("param_groups", [{}])[0].get("name", "Not recorded"))

    for i, group in enumerate(
        optimizer_state["param_groups"]
    ):

        print(f"\nParameter group {i + 1}")

        for key, value in group.items():

            if key != "params":
                print(f"{key}: {value}")

else:
    print("Optimizer state not available.")

print("\n3. MODEL INFORMATION")
print("--------------------")

print("Model parameters:", sum(
    p.numel() for p in molecular_transformer.parameters()
))

print("Model device:", next(
    molecular_transformer.parameters()
).device)

print("Decoder layers:", len(
    molecular_transformer.decoder.layers
))

print("Vocabulary size:", molecular_transformer.output_layer.out_features)

print("\nCELL 185 COMPLETED")

CELL 185: TRAINING CONFIGURATION AUDIT

1. CHECKPOINT METADATA
----------------------
epoch: 2
train_loss: 1.3710269425242854
validation_loss: 1.3104469297797816
training_molecules: 10000
training_type: chembl37_general_molecular_pretraining
pocket_cross_attention_used: False
vocabulary_size: 81

2. OPTIMIZER INFORMATION
------------------------
Optimizer: Not recorded

Parameter group 1
lr: 0.0001
betas: (0.9, 0.999)
eps: 1e-08
weight_decay: 1e-05
amsgrad: False
maximize: False
foreach: None
capturable: False
differentiable: False
fused: None
decoupled_weight_decay: True

3. MODEL INFORMATION
--------------------
Model parameters: 832849
Model device: cpu
Decoder layers: 3
Vocabulary size: 81

CELL 185 COMPLETED


In [258]:
# ============================================================
# CELL 186: EXACT TRAINING SUBSET VS VALIDATION AUDIT
# ============================================================

import os
import numpy as np
import torch
import pandas as pd

print("=" * 65)
print("CELL 186: EXACT TRAINING SUBSET VS VALIDATION AUDIT")
print("=" * 65)

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

# Load exact training indices
indices_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "transformer_selected_10k_indices.npy"
)

selected_indices = np.load(indices_path)

# Select actual training molecules
X_selected = X_train[selected_indices]

print("Selected training shape:", X_selected.shape)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def evaluate_exact(data, batch_size=32, max_samples=512):

    data = torch.as_tensor(
        data[:max_samples],
        dtype=torch.long
    )

    total_loss = 0
    total_correct = 0
    total_tokens = 0

    molecular_transformer.eval()

    with torch.no_grad():

        for start in range(0, len(data), batch_size):

            batch = data[start:start + batch_size].to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            logits = molecular_only_forward(
                molecular_transformer,
                inputs
            )

            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum"
            )

            mask = targets != PAD_ID

            correct = (
                (logits.argmax(dim=-1) == targets) & mask
            ).sum().item()

            total_loss += loss.item()
            total_correct += correct
            total_tokens += mask.sum().item()

    return {
        "loss": total_loss / total_tokens,
        "accuracy": total_correct / total_tokens,
        "tokens": total_tokens
    }


# ------------------------------------------------------------
# Evaluate both datasets
# ------------------------------------------------------------

train_result = evaluate_exact(X_selected)
val_result = evaluate_exact(X_val)

comparison = pd.DataFrame([
    {
        "Dataset": "Actual training subset",
        "Loss": train_result["loss"],
        "Accuracy (%)": train_result["accuracy"] * 100,
        "Tokens": train_result["tokens"]
    },
    {
        "Dataset": "Validation",
        "Loss": val_result["loss"],
        "Accuracy (%)": val_result["accuracy"] * 100,
        "Tokens": val_result["tokens"]
    }
])

print("\nRESULTS")
print("-------")
print(comparison.round(4).to_string(index=False))

print("\nGENERALIZATION GAP")
print("------------------")

print(
    "Loss difference:",
    round(val_result["loss"] - train_result["loss"], 4)
)

print(
    "Accuracy difference:",
    round(
        (train_result["accuracy"] - val_result["accuracy"]) * 100,
        2
    ),
    "percentage points"
)

print("\nCELL 186 COMPLETED")

CELL 186: EXACT TRAINING SUBSET VS VALIDATION AUDIT
Selected training shape: (10000, 143)
Validation shape: (5000, 143)

RESULTS
-------
               Dataset   Loss  Accuracy (%)  Tokens
Actual training subset 1.2788       57.9086   25055
            Validation 1.3240       56.9868   25820

GENERALIZATION GAP
------------------
Loss difference: 0.0452
Accuracy difference: 0.92 percentage points

CELL 186 COMPLETED


In [259]:
# ============================================================
# CELL 187: CONTROLLED TRANSFORMER FINE-TUNING
# ============================================================

import os
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

print("=" * 65)
print("CELL 187: CONTROLLED TRANSFORMER FINE-TUNING")
print("=" * 65)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR, "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_PATH = os.path.join(
    CHECKPOINT_DIR,
    "transformer_chembl37_improved_best.pt"
)

device = next(molecular_transformer.parameters()).device

PAD_ID = tokenizer["token_to_id"]["<PAD>"]

BATCH_SIZE = 32
MAX_EPOCHS = 5
LEARNING_RATE = 1e-4
PATIENCE = 2

print("Device:", device)
print("Training molecules:", len(X_selected))
print("Validation molecules:", len(X_val))

# ------------------------------------------------------------
# 2. Prepare datasets
# ------------------------------------------------------------

train_tensor = torch.tensor(
    X_selected,
    dtype=torch.long
)

val_tensor = torch.tensor(
    X_val,
    dtype=torch.long
)

train_loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(val_tensor),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ------------------------------------------------------------
# 3. Optimizer and scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    molecular_transformer.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

# ------------------------------------------------------------
# 4. Evaluation function
# ------------------------------------------------------------

def run_evaluation(loader):

    molecular_transformer.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for (batch,) in loader:

            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            logits = molecular_only_forward(
                molecular_transformer,
                inputs
            )

            loss = torch.nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum"
            )

            mask = targets != PAD_ID

            correct = (
                (logits.argmax(-1) == targets) & mask
            ).sum().item()

            total_loss += loss.item()
            total_correct += correct
            total_tokens += mask.sum().item()

    return (
        total_loss / total_tokens,
        total_correct / total_tokens
    )

# ------------------------------------------------------------
# 5. Training loop
# ------------------------------------------------------------

best_val_loss = float("inf")
epochs_without_improvement = 0

history = []

for epoch in range(MAX_EPOCHS):

    molecular_transformer.train()

    total_loss = 0.0
    total_tokens = 0

    for (batch,) in train_loader:

        batch = batch.to(device)

        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        optimizer.zero_grad(set_to_none=True)

        logits = molecular_only_forward(
            molecular_transformer,
            inputs
        )

        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=PAD_ID
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            molecular_transformer.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        valid_tokens = (targets != PAD_ID).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    train_loss = total_loss / total_tokens

    val_loss, val_accuracy = run_evaluation(val_loader)

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "validation_accuracy": val_accuracy,
        "learning_rate": current_lr
    })

    print(f"\nEpoch {epoch + 1}/{MAX_EPOCHS}")
    print("Training loss:", round(train_loss, 4))
    print("Validation loss:", round(val_loss, 4))
    print("Validation accuracy:", round(val_accuracy * 100, 2), "%")
    print("Learning rate:", current_lr)

    # Save best checkpoint
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save({
            "model_state_dict": molecular_transformer.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "validation_accuracy": val_accuracy,
            "training_molecules": len(X_selected),
            "training_type": "chembl37_general_molecular_finetuning",
            "pocket_cross_attention_used": False,
            "vocabulary_size": len(tokenizer["token_to_id"])
        }, BEST_PATH)

        print("Best checkpoint saved.")

    else:

        epochs_without_improvement += 1
        print("No validation improvement.")

    if epochs_without_improvement >= PATIENCE:

        print("\nEarly stopping triggered.")
        break

# ------------------------------------------------------------
# 6. Save training history
# ------------------------------------------------------------

history_df = pd.DataFrame(history)

history_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell187_training_history.csv"
)

history_df.to_csv(history_path, index=False)

print("\nFINAL TRAINING HISTORY")
print(history_df.to_string(index=False))

print("\nBest validation loss:", best_val_loss)
print("Best checkpoint:", BEST_PATH)
print("History:", history_path)

print("\nCELL 187 COMPLETED")

CELL 187: CONTROLLED TRANSFORMER FINE-TUNING
Device: cpu
Training molecules: 10000
Validation molecules: 5000

Epoch 1/5
Training loss: 1.3389
Validation loss: 1.2802
Validation accuracy: 57.88 %
Learning rate: 0.0001
Best checkpoint saved.

Epoch 2/5
Training loss: 1.3127
Validation loss: 1.2556
Validation accuracy: 58.8 %
Learning rate: 0.0001
Best checkpoint saved.

Epoch 3/5
Training loss: 1.2875
Validation loss: 1.231
Validation accuracy: 59.61 %
Learning rate: 0.0001
Best checkpoint saved.

Epoch 4/5
Training loss: 1.2665
Validation loss: 1.2183
Validation accuracy: 59.81 %
Learning rate: 0.0001
Best checkpoint saved.

Epoch 5/5
Training loss: 1.2469
Validation loss: 1.1923
Validation accuracy: 60.73 %
Learning rate: 0.0001
Best checkpoint saved.

FINAL TRAINING HISTORY
 epoch  train_loss  validation_loss  validation_accuracy  learning_rate
     1    1.338868         1.280150             0.578832         0.0001
     2    1.312710         1.255636             0.588032         0.00

In [260]:
# ============================================================
# CELL 188: IMPROVED MODEL GENERATION QUALITY EVALUATION
# ============================================================

import os
import json
import torch
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

print("=" * 65)
print("CELL 188: IMPROVED MODEL GENERATION QUALITY EVALUATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Load improved checkpoint
# ------------------------------------------------------------

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/Member2_Project/"
    "checkpoints/transformer_chembl37_improved_best.pt"
)

device = torch.device("cpu")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"]
)

molecular_transformer.to(device)
molecular_transformer.eval()

print("Improved checkpoint loaded.")
print("Training epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["validation_loss"])

# ------------------------------------------------------------
# 2. Load tokenizer vocabulary
# ------------------------------------------------------------

TOKENIZER_PATH = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/smiles_tokenizer.json"
)

with open(TOKENIZER_PATH, "r") as f:
    tokenizer_data = json.load(f)

# Find token-to-ID mapping in tokenizer JSON
def find_token_mapping(obj):
    if isinstance(obj, dict):

        if (
            len(obj) > 0
            and all(isinstance(v, int) for v in obj.values())
            and "<PAD>" in obj
            and "<BOS>" in obj
            and "<EOS>" in obj
        ):
            return obj

        for value in obj.values():
            result = find_token_mapping(value)
            if result is not None:
                return result

    elif isinstance(obj, list):
        for value in obj:
            result = find_token_mapping(value)
            if result is not None:
                return result

    return None


token_to_id = find_token_mapping(tokenizer_data)

if token_to_id is None:
    raise ValueError(
        "Could not automatically locate token-to-ID mapping. "
        "Inspect smiles_tokenizer.json before continuing."
    )

id_to_token = {
    value: key for key, value in token_to_id.items()
}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]

VOCAB_SIZE = len(token_to_id)

print("Vocabulary size:", VOCAB_SIZE)
print("PAD:", PAD_ID, "BOS:", BOS_ID, "EOS:", EOS_ID)

# ------------------------------------------------------------
# 3. SMILES decoding function
# ------------------------------------------------------------

def decode_tokens(token_ids):

    tokens = []

    for token_id in token_ids:

        token = id_to_token.get(int(token_id), "")

        if token in ["<PAD>", "<BOS>", "<EOS>"]:
            continue

        if token == "<UNK>":
            continue

        tokens.append(token)

    return "".join(tokens)


# ------------------------------------------------------------
# 4. Autoregressive generation
# ------------------------------------------------------------

MAX_LENGTH = 143

def generate_molecule(greedy=True, temperature=0.8, top_k=20):

    generated = [BOS_ID]

    for step in range(MAX_LENGTH - 1):

        input_tensor = torch.tensor(
            [generated],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():

            logits = molecular_only_forward(
                molecular_transformer,
                input_tensor
            )

        next_logits = logits[0, -1, :].clone()

        # Prevent generation of special starting tokens
        next_logits[PAD_ID] = -float("inf")
        next_logits[BOS_ID] = -float("inf")

        if greedy:

            next_token = torch.argmax(next_logits).item()

        else:

            next_logits = next_logits / temperature

            if top_k is not None:

                k = min(top_k, next_logits.numel())

                values, indices = torch.topk(next_logits, k)

                filtered_logits = torch.full_like(
                    next_logits,
                    -float("inf")
                )

                filtered_logits[indices] = values
                next_logits = filtered_logits

            probabilities = torch.softmax(next_logits, dim=-1)

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

        generated.append(next_token)

        if next_token == EOS_ID:
            break

    return decode_tokens(generated)


# ------------------------------------------------------------
# 5. Generate 30 greedy and 30 stochastic molecules
# ------------------------------------------------------------

print("\nGenerating greedy molecules...")

greedy_smiles = [
    generate_molecule(greedy=True)
    for _ in range(30)
]

print("Generating stochastic molecules...")

torch.manual_seed(42)
np.random.seed(42)

stochastic_smiles = [
    generate_molecule(
        greedy=False,
        temperature=0.8,
        top_k=20
    )
    for _ in range(30)
]

# ------------------------------------------------------------
# 6. Evaluate molecules
# ------------------------------------------------------------

def evaluate_molecules(smiles_list, generation_type):

    records = []

    for index, smiles in enumerate(smiles_list):

        mol = Chem.MolFromSmiles(smiles)

        record = {
            "generation_type": generation_type,
            "molecule_id": index + 1,
            "SMILES": smiles,
            "valid": mol is not None
        }

        if mol is not None:

            record.update({
                "MW": Descriptors.MolWt(mol),
                "LogP": Descriptors.MolLogP(mol),
                "TPSA": rdMolDescriptors.CalcTPSA(mol),
                "HBD": Lipinski.NumHDonors(mol),
                "HBA": Lipinski.NumHAcceptors(mol),
                "Rotatable_Bonds": Lipinski.NumRotatableBonds(mol),
                "Heavy_Atoms": mol.GetNumHeavyAtoms(),
                "Rings": rdMolDescriptors.CalcNumRings(mol)
            })

        records.append(record)

    return records


greedy_records = evaluate_molecules(
    greedy_smiles,
    "Greedy"
)

stochastic_records = evaluate_molecules(
    stochastic_smiles,
    "Stochastic"
)

all_records = greedy_records + stochastic_records

results_df = pd.DataFrame(all_records)

# ------------------------------------------------------------
# 7. Summary statistics
# ------------------------------------------------------------

def print_summary(records, name):

    valid_records = [
        r for r in records if r["valid"]
    ]

    valid_smiles = [
        r["SMILES"] for r in valid_records
    ]

    total = len(records)
    valid = len(valid_records)
    unique = len(set(valid_smiles))

    print("\n", "=" * 50)
    print(name, "GENERATION SUMMARY")
    print("=" * 50)

    print("Total generated:", total)
    print("Valid molecules:", valid)
    print("Validity:", round(valid / total * 100, 2), "%")
    print("Unique valid molecules:", unique)

    if valid > 0:

        print(
            "Uniqueness:",
            round(unique / valid * 100, 2),
            "%"
        )

        descriptor_df = pd.DataFrame(valid_records)

        print("\nAverage molecular descriptors:")

        columns = [
            "MW", "LogP", "TPSA",
            "HBD", "HBA",
            "Rotatable_Bonds",
            "Heavy_Atoms", "Rings"
        ]

        print(
            descriptor_df[columns].mean().round(2)
        )


print_summary(greedy_records, "GREEDY")

print_summary(stochastic_records, "STOCHASTIC")

# ------------------------------------------------------------
# 8. Save evaluation results
# ------------------------------------------------------------

OUTPUT_PATH = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell188_improved_generation_audit.csv"
)

results_df.to_csv(OUTPUT_PATH, index=False)

print("\nEvaluation saved to:")
print(OUTPUT_PATH)

print("\nCELL 188 COMPLETED")

CELL 188: IMPROVED MODEL GENERATION QUALITY EVALUATION
Improved checkpoint loaded.
Training epoch: 5
Validation loss: 1.1923144488281978
Vocabulary size: 81
PAD: 0 BOS: 1 EOS: 2

Generating greedy molecules...
Generating stochastic molecules...

GREEDY GENERATION SUMMARY
Total generated: 30
Valid molecules: 30
Validity: 100.0 %
Unique valid molecules: 1
Uniqueness: 3.33 %

Average molecular descriptors:
MW                 941.82
LogP                26.24
TPSA                 0.00
HBD                  0.00
HBA                  0.00
Rotatable_Bonds     63.00
Heavy_Atoms         67.00
Rings                0.00
dtype: float64

STOCHASTIC GENERATION SUMMARY
Total generated: 30
Valid molecules: 2
Validity: 6.67 %
Unique valid molecules: 2
Uniqueness: 100.0 %

Average molecular descriptors:
MW                 305.36
LogP                 2.69
TPSA                54.72
HBD                  1.00
HBA                  3.50
Rotatable_Bonds      5.00
Heavy_Atoms         22.50
Rings                3.

In [261]:
# ============================================================
# CELL 189: GENERATED MOLECULE AND TRAINING-DATA INVESTIGATION
# ============================================================

import os
import pandas as pd
import numpy as np
from collections import Counter

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

print("=" * 70)
print("CELL 189: GENERATION FAILURE DIAGNOSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load Cell 188 results
# ------------------------------------------------------------

BASE_PATH = "/content/drive/MyDrive/Member2_Project"

RESULT_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/cell188_improved_generation_audit.csv"
)

results = pd.read_csv(RESULT_PATH)

print("\nTotal generated records:", len(results))

# ------------------------------------------------------------
# 2. Separate greedy and stochastic results
# ------------------------------------------------------------

greedy_df = results[
    results["generation_type"] == "Greedy"
].copy()

stochastic_df = results[
    results["generation_type"] == "Stochastic"
].copy()

valid_greedy = greedy_df[greedy_df["valid"] == True]
valid_stochastic = stochastic_df[
    stochastic_df["valid"] == True
]

print("Valid greedy:", len(valid_greedy))
print("Valid stochastic:", len(valid_stochastic))

# ------------------------------------------------------------
# 3. Canonicalize generated molecules
# ------------------------------------------------------------

def canonicalize(smiles):

    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None

    return Chem.MolToSmiles(
        mol,
        canonical=True
    )


results["canonical_smiles"] = results["SMILES"].apply(
    canonicalize
)

# ------------------------------------------------------------
# 4. Display repeated greedy molecule
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GREEDY GENERATION: MODE COLLAPSE INVESTIGATION")
print("=" * 60)

greedy_unique = valid_greedy[
    "SMILES"
].drop_duplicates().tolist()

print("Number of unique greedy molecules:", len(greedy_unique))

for i, smiles in enumerate(greedy_unique, 1):

    print(f"\nGreedy molecule {i}:")
    print(smiles)

# ------------------------------------------------------------
# 5. Display valid stochastic molecules
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VALID STOCHASTIC MOLECULES")
print("=" * 60)

for _, row in valid_stochastic.iterrows():

    print("\nMolecule ID:", row["molecule_id"])
    print("SMILES:", row["SMILES"])
    print("MW:", round(row["MW"], 2))
    print("LogP:", round(row["LogP"], 2))
    print("TPSA:", round(row["TPSA"], 2))
    print("HBD:", row["HBD"])
    print("HBA:", row["HBA"])

# ------------------------------------------------------------
# 6. Load ChEMBL training molecules
# ------------------------------------------------------------

CHEMBL_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/chembl37_clean_smiles.txt"
)

train_canonical = set()
training_count = 0

with open(CHEMBL_PATH, "r") as f:

    for line in f:

        smiles = line.strip()

        if not smiles:
            continue

        training_count += 1

        canonical = canonicalize(smiles)

        if canonical is not None:
            train_canonical.add(canonical)

print("\n" + "=" * 60)
print("CHEMBL TRAINING DATA COMPARISON")
print("=" * 60)

print("Training records:", training_count)
print("Unique valid canonical molecules:", len(train_canonical))

# ------------------------------------------------------------
# 7. Check whether generated molecules exist in ChEMBL
# ------------------------------------------------------------

for generation_type, df in [
    ("Greedy", valid_greedy),
    ("Stochastic", valid_stochastic)
]:

    print(f"\n{generation_type} training-data comparison:")

    for _, row in df.iterrows():

        canonical = canonicalize(row["SMILES"])

        exists = canonical in train_canonical

        print(
            f"Molecule {row['molecule_id']}: "
            f"{'EXISTS in ChEMBL file' if exists else 'NOT FOUND in ChEMBL file'}"
        )

# ------------------------------------------------------------
# 8. Analyze molecular scaffolds
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MOLECULAR SCAFFOLD ANALYSIS")
print("=" * 60)

for generation_type, df in [
    ("Greedy", valid_greedy),
    ("Stochastic", valid_stochastic)
]:

    print(f"\n{generation_type} scaffolds:")

    for _, row in df.iterrows():

        mol = Chem.MolFromSmiles(row["SMILES"])

        if mol is None:
            continue

        scaffold = MurckoScaffold.GetScaffoldForMol(mol)

        scaffold_smiles = Chem.MolToSmiles(
            scaffold,
            canonical=True
        )

        print(
            f"Molecule {row['molecule_id']}: "
            f"{scaffold_smiles if scaffold_smiles else 'No ring scaffold'}"
        )

# ------------------------------------------------------------
# 9. Generated SMILES character frequency
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GENERATED SMILES CHARACTER FREQUENCY")
print("=" * 60)

for generation_type, df in [
    ("Greedy", greedy_df),
    ("Stochastic", stochastic_df)
]:

    text = "".join(
        df["SMILES"].dropna().astype(str).tolist()
    )

    frequencies = Counter(text)

    print(f"\n{generation_type} character frequencies:")

    for character, count in frequencies.most_common(20):

        print(repr(character), ":", count)

# ------------------------------------------------------------
# 10. Save canonicalized investigation results
# ------------------------------------------------------------

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/cell189_generation_diagnosis.csv"
)

results.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nDiagnosis saved to:")
print(OUTPUT_PATH)

print("\nCELL 189 COMPLETED")

CELL 189: GENERATION FAILURE DIAGNOSIS

Total generated records: 60
Valid greedy: 30
Valid stochastic: 2

GREEDY GENERATION: MODE COLLAPSE INVESTIGATION
Number of unique greedy molecules: 1

Greedy molecule 1:
CC(C)CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC

VALID STOCHASTIC MOLECULES

Molecule ID: 5
SMILES: COc1ccc(CN2CCOC(=O)Nc3ccccc3)cc21
MW: 298.34
LogP: 3.26
TPSA: 50.8
HBD: 1.0
HBA: 4.0

Molecule ID: 21
SMILES: COc1cc(NC(=O)CCC(=O)N2CCCc3ccccc3)c21
MW: 312.37
LogP: 2.11
TPSA: 58.64
HBD: 1.0
HBA: 3.0

CHEMBL TRAINING DATA COMPARISON
Training records: 50000
Unique valid canonical molecules: 50000

Greedy training-data comparison:
Molecule 1: NOT FOUND in ChEMBL file
Molecule 2: NOT FOUND in ChEMBL file
Molecule 3: NOT FOUND in ChEMBL file
Molecule 4: NOT FOUND in ChEMBL file
Molecule 5: NOT FOUND in ChEMBL file
Molecule 6: NOT FOUND in ChEMBL file
Molecule 7: NOT FOUND in ChEMBL file
Molecule 8: NOT FOUND in ChEMBL file
Molecule 9: NOT FOUND in ChEMBL file
Mole

In [262]:
# ============================================================
# CELL 190: GENERATED MOLECULE SIMILARITY AND NOVELTY AUDIT
# ============================================================

import os
import pandas as pd
import numpy as np

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

print("=" * 65)
print("CELL 190: MOLECULAR SIMILARITY AND NOVELTY ANALYSIS")
print("=" * 65)

BASE_PATH = "/content/drive/MyDrive/Member2_Project"

RESULT_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/cell188_improved_generation_audit.csv"
)

CHEMBL_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/chembl37_clean_smiles.txt"
)

OUTPUT_PATH = os.path.join(
    BASE_PATH,
    "member2_outputs/cell190_similarity_audit.csv"
)

# ------------------------------------------------------------
# 1. Load generated molecules
# ------------------------------------------------------------

results = pd.read_csv(RESULT_PATH)

valid_generated = results[
    results["valid"] == True
].copy()

valid_generated = valid_generated.drop_duplicates(
    subset=["SMILES"]
)

print("Unique valid generated molecules:", len(valid_generated))

# ------------------------------------------------------------
# 2. Load ChEMBL molecules
# ------------------------------------------------------------

training_molecules = {}

with open(CHEMBL_PATH, "r") as f:

    for line in f:

        smiles = line.strip()

        if not smiles:
            continue

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        canonical = Chem.MolToSmiles(
            mol,
            canonical=True
        )

        training_molecules[canonical] = mol

print("Unique valid ChEMBL molecules:", len(training_molecules))

# ------------------------------------------------------------
# 3. Create Morgan fingerprints
# ------------------------------------------------------------

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

training_fingerprints = []

for canonical, mol in training_molecules.items():

    fingerprint = fingerprint_generator.GetFingerprint(mol)

    training_fingerprints.append(
        (canonical, fingerprint)
    )

print("Training fingerprints created:", len(training_fingerprints))

# ------------------------------------------------------------
# 4. Compare generated molecules
# ------------------------------------------------------------

records = []

for _, row in valid_generated.iterrows():

    generated_smiles = row["SMILES"]

    generated_mol = Chem.MolFromSmiles(
        generated_smiles
    )

    if generated_mol is None:
        continue

    generated_canonical = Chem.MolToSmiles(
        generated_mol,
        canonical=True
    )

    generated_fp = fingerprint_generator.GetFingerprint(
        generated_mol
    )

    exact_match = (
        generated_canonical in training_molecules
    )

    best_similarity = -1
    nearest_smiles = None

    for train_smiles, train_fp in training_fingerprints:

        similarity = DataStructs.TanimotoSimilarity(
            generated_fp,
            train_fp
        )

        if similarity > best_similarity:

            best_similarity = similarity
            nearest_smiles = train_smiles

    records.append({

        "generation_type": row["generation_type"],

        "molecule_id": row["molecule_id"],

        "generated_smiles": generated_smiles,

        "canonical_smiles": generated_canonical,

        "exact_training_match": exact_match,

        "maximum_tanimoto_similarity": best_similarity,

        "nearest_training_smiles": nearest_smiles

    })

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

similarity_df = pd.DataFrame(records)

print("\n" + "=" * 65)
print("SIMILARITY RESULTS")
print("=" * 65)

if len(similarity_df) > 0:

    print(
        similarity_df[
            [
                "generation_type",
                "molecule_id",
                "exact_training_match",
                "maximum_tanimoto_similarity"
            ]
        ].to_string(index=False)
    )

    print("\nDetailed nearest training molecules:")

    for _, row in similarity_df.iterrows():

        print("\nGenerated molecule:", row["molecule_id"])

        print(
            "Generation type:",
            row["generation_type"]
        )

        print(
            "Exact training match:",
            row["exact_training_match"]
        )

        print(
            "Maximum Tanimoto:",
            round(
                row["maximum_tanimoto_similarity"],
                4
            )
        )

        print(
            "Nearest training SMILES:",
            row["nearest_training_smiles"]
        )

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

similarity_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:", OUTPUT_PATH)

print("\nCELL 190 COMPLETED")

CELL 190: MOLECULAR SIMILARITY AND NOVELTY ANALYSIS
Unique valid generated molecules: 3
Unique valid ChEMBL molecules: 50000
Training fingerprints created: 50000

SIMILARITY RESULTS
generation_type  molecule_id  exact_training_match  maximum_tanimoto_similarity
         Greedy            1                 False                     0.600000
     Stochastic            5                 False                     0.391892
     Stochastic           21                 False                     0.338710

Detailed nearest training molecules:

Generated molecule: 1
Generation type: Greedy
Exact training match: False
Maximum Tanimoto: 0.6
Nearest training SMILES: CCCCCCCCCC(C)O

Generated molecule: 5
Generation type: Stochastic
Exact training match: False
Maximum Tanimoto: 0.3919
Nearest training SMILES: O=C(Nc1ccccc1)OCCN1CC[C@H]2[C@H](C1)c1cc(F)ccc1N2c1ccc(F)cc1

Generated molecule: 21
Generation type: Stochastic
Exact training match: False
Maximum Tanimoto: 0.3387
Nearest training SMILES: COC

In [263]:
# ============================================================
# CELL 191: GREEDY GENERATION TOKEN-LEVEL INVESTIGATION
# ============================================================

import os
import json
import torch
import pandas as pd
from collections import Counter

print("=" * 65)
print("CELL 191: GREEDY TOKEN-LEVEL INVESTIGATION")
print("=" * 65)

device = torch.device("cpu")

BASE_PATH = "/content/drive/MyDrive/Member2_Project"

# ------------------------------------------------------------
# 1. Load improved checkpoint
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    BASE_PATH,
    "checkpoints/transformer_chembl37_improved_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"]
)

molecular_transformer.to(device)
molecular_transformer.eval()

# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    BASE_PATH,
    "member2_outputs/smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)


def find_token_mapping(obj):

    if isinstance(obj, dict):

        if (
            len(obj) > 0
            and all(isinstance(v, int) for v in obj.values())
            and "<PAD>" in obj
            and "<BOS>" in obj
            and "<EOS>" in obj
        ):
            return obj

        for value in obj.values():

            result = find_token_mapping(value)

            if result is not None:
                return result

    elif isinstance(obj, list):

        for value in obj:

            result = find_token_mapping(value)

            if result is not None:
                return result

    return None


token_to_id = find_token_mapping(tokenizer_data)

if token_to_id is None:
    raise ValueError("Tokenizer mapping not found.")

id_to_token = {
    value: key for key, value in token_to_id.items()
}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]

# ------------------------------------------------------------
# 3. Trace greedy generation
# ------------------------------------------------------------

generated = [BOS_ID]
trace_records = []

MAX_LENGTH = 143

for step in range(MAX_LENGTH - 1):

    input_tensor = torch.tensor(
        [generated],
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():

        logits = molecular_only_forward(
            molecular_transformer,
            input_tensor
        )

    next_logits = logits[0, -1, :].clone()

    next_logits[PAD_ID] = -float("inf")
    next_logits[BOS_ID] = -float("inf")

    probabilities = torch.softmax(
        next_logits,
        dim=-1
    )

    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=5
    )

    selected_id = top_indices[0].item()
    selected_token = id_to_token.get(
        selected_id,
        "<UNKNOWN>"
    )

    selected_probability = top_probabilities[0].item()

    entropy = -torch.sum(
        probabilities * torch.log(probabilities + 1e-12)
    ).item()

    alternatives = []

    for token_id, probability in zip(
        top_indices.tolist(),
        top_probabilities.tolist()
    ):

        alternatives.append({
            "token": id_to_token.get(token_id, "<UNKNOWN>"),
            "probability": round(probability, 5)
        })

    trace_records.append({

        "step": step + 1,

        "selected_token": selected_token,

        "selected_probability": selected_probability,

        "entropy": entropy,

        "top_5_alternatives": str(alternatives)

    })

    generated.append(selected_id)

    if selected_id == EOS_ID:
        break

# ------------------------------------------------------------
# 4. Display generation trace
# ------------------------------------------------------------

trace_df = pd.DataFrame(trace_records)

print("\nTotal generated tokens:", len(trace_records))

print("\nFIRST 30 GENERATION STEPS")

print(
    trace_df.head(30)[
        [
            "step",
            "selected_token",
            "selected_probability",
            "entropy"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 5. Token frequency
# ------------------------------------------------------------

selected_tokens = [
    record["selected_token"]
    for record in trace_records
]

print("\n" + "=" * 65)
print("GENERATED TOKEN FREQUENCY")
print("=" * 65)

for token, count in Counter(selected_tokens).most_common(20):

    print(repr(token), ":", count)

# ------------------------------------------------------------
# 6. Probability and entropy summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL CONFIDENCE SUMMARY")
print("=" * 65)

print(
    "Mean selected-token probability:",
    round(trace_df["selected_probability"].mean(), 4)
)

print(
    "Mean prediction entropy:",
    round(trace_df["entropy"].mean(), 4)
)

print(
    "Minimum selected-token probability:",
    round(trace_df["selected_probability"].min(), 4)
)

print(
    "Maximum selected-token probability:",
    round(trace_df["selected_probability"].max(), 4)
)

# ------------------------------------------------------------
# 7. Save detailed trace
# ------------------------------------------------------------

output_path = os.path.join(
    BASE_PATH,
    "member2_outputs/cell191_greedy_token_trace.csv"
)

trace_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nCELL 191 COMPLETED")

CELL 191: GREEDY TOKEN-LEVEL INVESTIGATION

Total generated tokens: 70

FIRST 30 GENERATION STEPS
 step selected_token  selected_probability  entropy
    1              C              0.698349 0.905091
    2              C              0.350393 1.736534
    3              (              0.380855 1.874631
    4              C              0.503792 1.138738
    5              )              0.944117 0.316740
    6              C              0.310954 1.938280
    7              C              0.388787 1.915800
    8              C              0.393486 1.887442
    9              C              0.405404 1.920448
   10              C              0.395899 1.854986
   11              C              0.455838 1.705286
   12              C              0.472245 1.717813
   13              C              0.496160 1.714882
   14              C              0.518919 1.640675
   15              C              0.475112 1.722180
   16              C              0.559955 1.540813
   17             

In [264]:
# ============================================================
# CELL 192: TRAINING DATA MOLECULAR DIVERSITY AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem import rdFingerprintGenerator

print("=" * 70)
print("CELL 192: TRAINING DATA DIVERSITY AUDIT")
print("=" * 70)

BASE = "/content/drive/MyDrive/Member2_Project"

# ------------------------------------------------------------
# 1. Load ChEMBL molecules
# ------------------------------------------------------------

chembl_path = os.path.join(
    BASE,
    "member2_outputs/chembl37_clean_smiles.txt"
)

smiles_list = []

with open(chembl_path, "r") as f:
    for line in f:
        smiles = line.strip()
        if smiles:
            smiles_list.append(smiles)

print("Total ChEMBL records:", len(smiles_list))

# ------------------------------------------------------------
# 2. Load exact training indices
# ------------------------------------------------------------

indices_path = os.path.join(
    BASE,
    "member2_outputs/transformer_selected_10k_indices.npy"
)

selected_indices = np.load(indices_path)

print("Selected training indices:", len(selected_indices))

selected_set = set(selected_indices.tolist())

# ------------------------------------------------------------
# 3. Build dataset groups
# ------------------------------------------------------------

groups = {
    "Full_50000": smiles_list,
    "Selected_10000": [
        smiles_list[i] for i in selected_indices
    ],
    "Remaining_40000": [
        smiles for i, smiles in enumerate(smiles_list)
        if i not in selected_set
    ]
}

# ------------------------------------------------------------
# 4. Calculate molecular descriptors
# ------------------------------------------------------------

def analyze_group(name, smiles_group):

    records = []

    for smiles in smiles_group:

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        records.append({

            "canonical": Chem.MolToSmiles(
                mol,
                canonical=True
            ),

            "MW": Descriptors.MolWt(mol),

            "LogP": Descriptors.MolLogP(mol),

            "TPSA": Descriptors.TPSA(mol),

            "HBD": Lipinski.NumHDonors(mol),

            "HBA": Lipinski.NumHAcceptors(mol),

            "HeavyAtoms": mol.GetNumHeavyAtoms(),

            "RotatableBonds": Lipinski.NumRotatableBonds(mol),

            "Rings": Lipinski.RingCount(mol)

        })

    df = pd.DataFrame(records)

    print("\n" + "-" * 60)
    print(name)
    print("-" * 60)

    print("Valid molecules:", len(df))
    print("Unique canonical molecules:", df["canonical"].nunique())

    descriptor_columns = [
        "MW", "LogP", "TPSA",
        "HBD", "HBA", "HeavyAtoms",
        "RotatableBonds", "Rings"
    ]

    print("\nDescriptor statistics:")

    print(
        df[descriptor_columns]
        .describe()
        .loc[["mean", "50%", "std", "min", "max"]]
        .round(2)
    )

    return df


group_results = {}

for name, group in groups.items():

    group_results[name] = analyze_group(
        name,
        group
    )

# ------------------------------------------------------------
# 5. Compare Morgan fingerprint diversity
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MORGAN FINGERPRINT DIVERSITY")
print("=" * 70)

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

def calculate_diversity(df, sample_size=500):

    molecules = [
        Chem.MolFromSmiles(s)
        for s in df["canonical"].drop_duplicates()
    ]

    molecules = [
        mol for mol in molecules if mol is not None
    ]

    if len(molecules) > sample_size:

        rng = np.random.default_rng(42)

        selected = rng.choice(
            len(molecules),
            sample_size,
            replace=False
        )

        molecules = [
            molecules[i] for i in selected
        ]

    fingerprints = [
        generator.GetFingerprint(mol)
        for mol in molecules
    ]

    similarities = []

    for i in range(len(fingerprints)):

        sims = DataStructs.BulkTanimotoSimilarity(
            fingerprints[i],
            fingerprints[i + 1:]
        )

        similarities.extend(sims)

    if not similarities:
        return None

    return {
        "sampled_molecules": len(molecules),
        "mean_similarity": np.mean(similarities),
        "median_similarity": np.median(similarities),
        "mean_diversity": 1 - np.mean(similarities)
    }


for name in [
    "Full_50000",
    "Selected_10000",
    "Remaining_40000"
]:

    result = calculate_diversity(
        group_results[name]
    )

    print("\n", name)
    print(result)

# ------------------------------------------------------------
# 6. Save comparison
# ------------------------------------------------------------

summary_records = []

for name, df in group_results.items():

    summary_records.append({

        "dataset": name,

        "valid_molecules": len(df),

        "unique_molecules": df["canonical"].nunique(),

        "mean_MW": df["MW"].mean(),

        "mean_LogP": df["LogP"].mean(),

        "mean_TPSA": df["TPSA"].mean(),

        "mean_HeavyAtoms": df["HeavyAtoms"].mean(),

        "mean_Rings": df["Rings"].mean()

    })

summary_df = pd.DataFrame(summary_records)

output_path = os.path.join(
    BASE,
    "member2_outputs/cell192_dataset_diversity_audit.csv"
)

summary_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nCELL 192 COMPLETED")

CELL 192: TRAINING DATA DIVERSITY AUDIT
Total ChEMBL records: 50000
Selected training indices: 10000

------------------------------------------------------------
Full_50000
------------------------------------------------------------
Valid molecules: 50000
Unique canonical molecules: 50000

Descriptor statistics:
          MW   LogP    TPSA    HBD    HBA  HeavyAtoms  RotatableBonds  Rings
mean  414.78   3.54   84.60   1.61   5.12       29.25            5.75   3.68
50%   400.20   3.50   78.67   1.00   5.00       28.00            5.00   4.00
std   124.85   1.84   41.47   1.47   2.26        8.88            3.57   1.47
min   101.11 -10.29    0.00   0.00   0.00        5.00            0.00   0.00
max   999.54  16.47  474.90  18.00  26.00       75.00           56.00  74.00

------------------------------------------------------------
Selected_10000
------------------------------------------------------------
Valid molecules: 10000
Unique canonical molecules: 10000

Descriptor statistics:
   

In [265]:
# ============================================================
# CELL 193: CORRECTED TRAINING DATASET DIVERSITY AUDIT
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem import rdFingerprintGenerator

print("=" * 70)
print("CELL 193: CORRECTED DATASET DIVERSITY AUDIT")
print("=" * 70)

BASE = "/content/drive/MyDrive/Member2_Project"
OUT = os.path.join(BASE, "member2_outputs")

# ------------------------------------------------------------
# 1. Load actual training and validation token arrays
# ------------------------------------------------------------

X_train = np.load(
    os.path.join(OUT, "chembl_train_tokens.npy")
)

X_val = np.load(
    os.path.join(OUT, "chembl_val_tokens.npy")
)

selected_indices = np.load(
    os.path.join(OUT, "transformer_selected_10k_indices.npy")
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Selected indices:", len(selected_indices))

assert len(X_train) == 45000
assert len(X_val) == 5000
assert len(selected_indices) == 10000

assert selected_indices.min() >= 0
assert selected_indices.max() < len(X_train)

# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

with open(
    os.path.join(OUT, "smiles_tokenizer.json"), "r"
) as f:
    tokenizer_data = json.load(f)


def find_mapping(obj):

    if isinstance(obj, dict):

        if (
            obj
            and all(isinstance(v, int) for v in obj.values())
            and "<PAD>" in obj
            and "<BOS>" in obj
            and "<EOS>" in obj
        ):
            return obj

        for value in obj.values():

            result = find_mapping(value)

            if result is not None:
                return result

    elif isinstance(obj, list):

        for value in obj:

            result = find_mapping(value)

            if result is not None:
                return result

    return None


token_to_id = find_mapping(tokenizer_data)

if token_to_id is None:
    raise ValueError("Tokenizer mapping not found.")

id_to_token = {
    value: key for key, value in token_to_id.items()
}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id.get("<UNK>", -1)

# ------------------------------------------------------------
# 3. Decode token arrays into SMILES
# ------------------------------------------------------------

def decode_sequence(sequence):

    tokens = []

    for token_id in sequence:

        token_id = int(token_id)

        if token_id == EOS_ID:
            break

        if token_id in [PAD_ID, BOS_ID, UNK_ID]:
            continue

        token = id_to_token.get(token_id)

        if token is None:
            continue

        tokens.append(token)

    return "".join(tokens)


def decode_dataset(token_array):

    return [
        decode_sequence(sequence)
        for sequence in token_array
    ]


train_smiles = decode_dataset(X_train)
val_smiles = decode_dataset(X_val)

selected_smiles = [
    train_smiles[i]
    for i in selected_indices
]

selected_set = set(selected_indices.tolist())

remaining_smiles = [
    smiles
    for i, smiles in enumerate(train_smiles)
    if i not in selected_set
]

groups = {
    "Training_45000": train_smiles,
    "Selected_10000": selected_smiles,
    "Remaining_35000": remaining_smiles,
    "Validation_5000": val_smiles
}

# ------------------------------------------------------------
# 4. Calculate molecular descriptors
# ------------------------------------------------------------

def analyze_group(name, smiles_group):

    records = []

    for smiles in smiles_group:

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        records.append({

            "canonical": Chem.MolToSmiles(mol),

            "MW": Descriptors.MolWt(mol),

            "LogP": Descriptors.MolLogP(mol),

            "TPSA": Descriptors.TPSA(mol),

            "HBD": Lipinski.NumHDonors(mol),

            "HBA": Lipinski.NumHAcceptors(mol),

            "HeavyAtoms": mol.GetNumHeavyAtoms(),

            "RotatableBonds": Lipinski.NumRotatableBonds(mol),

            "Rings": Lipinski.RingCount(mol)

        })

    df = pd.DataFrame(records)

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print("Total records:", len(smiles_group))
    print("Valid molecules:", len(df))
    print("Unique canonical molecules:", df["canonical"].nunique())

    columns = [
        "MW", "LogP", "TPSA", "HBD",
        "HBA", "HeavyAtoms", "RotatableBonds", "Rings"
    ]

    print(
        df[columns]
        .describe()
        .loc[["mean", "50%", "std"]]
        .round(3)
    )

    return df


group_results = {}

for name, smiles_group in groups.items():

    group_results[name] = analyze_group(
        name,
        smiles_group
    )

# ------------------------------------------------------------
# 5. Morgan fingerprint diversity
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MORGAN FINGERPRINT DIVERSITY")
print("=" * 70)

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)


def diversity_audit(df, sample_size=500):

    canonical_smiles = (
        df["canonical"].drop_duplicates().tolist()
    )

    if len(canonical_smiles) > sample_size:

        rng = np.random.default_rng(42)

        indices = rng.choice(
            len(canonical_smiles),
            sample_size,
            replace=False
        )

        canonical_smiles = [
            canonical_smiles[i] for i in indices
        ]

    molecules = [
        Chem.MolFromSmiles(s)
        for s in canonical_smiles
    ]

    fingerprints = [
        generator.GetFingerprint(mol)
        for mol in molecules
    ]

    similarities = []

    for i in range(len(fingerprints)):

        similarities.extend(
            DataStructs.BulkTanimotoSimilarity(
                fingerprints[i],
                fingerprints[i + 1:]
            )
        )

    if not similarities:
        return None

    mean_similarity = float(np.mean(similarities))

    return {
        "sampled_molecules": len(molecules),
        "mean_similarity": mean_similarity,
        "median_similarity": float(np.median(similarities)),
        "mean_diversity": 1 - mean_similarity
    }


diversity_results = {}

for name, df in group_results.items():

    result = diversity_audit(df)

    diversity_results[name] = result

    print("\n", name)
    print(result)

# ------------------------------------------------------------
# 6. Save corrected results
# ------------------------------------------------------------

summary = []

for name, df in group_results.items():

    summary.append({

        "dataset": name,

        "total_records": len(groups[name]),

        "valid_molecules": len(df),

        "unique_molecules": df["canonical"].nunique(),

        "mean_MW": df["MW"].mean(),

        "mean_LogP": df["LogP"].mean(),

        "mean_TPSA": df["TPSA"].mean(),

        "mean_HeavyAtoms": df["HeavyAtoms"].mean(),

        "mean_Rings": df["Rings"].mean(),

        "mean_fingerprint_similarity":
            diversity_results[name]["mean_similarity"],

        "mean_fingerprint_diversity":
            diversity_results[name]["mean_diversity"]

    })


summary_df = pd.DataFrame(summary)

output_path = os.path.join(
    OUT,
    "cell193_corrected_dataset_diversity.csv"
)

summary_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)
print("\nCELL 193 COMPLETED")

CELL 193: CORRECTED DATASET DIVERSITY AUDIT
Training shape: (45000, 143)
Validation shape: (5000, 143)
Selected indices: 10000

Training_45000
Total records: 45000
Valid molecules: 45000
Unique canonical molecules: 45000
           MW   LogP    TPSA    HBD    HBA  HeavyAtoms  RotatableBonds  Rings
mean  414.767  3.539  84.554  1.605  5.117      29.247           5.751  3.678
50%   399.573  3.501  78.600  1.000  5.000      28.000           5.000  4.000
std   124.868  1.840  41.495  1.464  2.267       8.882           3.558  1.476

Selected_10000
Total records: 10000
Valid molecules: 10000
Unique canonical molecules: 10000
           MW   LogP    TPSA    HBD    HBA  HeavyAtoms  RotatableBonds  Rings
mean  415.285  3.549  84.528  1.599  5.112      29.274           5.795  3.678
50%   402.417  3.494  78.845  1.000  5.000      28.000           5.000  4.000
std   123.441  1.867  41.071  1.451  2.253       8.770           3.520  1.435

Remaining_35000
Total records: 35000
Valid molecules: 35000


In [266]:
# ============================================================
# CELL 194: SMILES TOKEN DISTRIBUTION ANALYSIS
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from collections import Counter

print("=" * 70)
print("CELL 194: SMILES TOKEN DISTRIBUTION ANALYSIS")
print("=" * 70)

ROOT = "/content/drive/MyDrive/Member2_Project"

TRAIN_PATH = os.path.join(
    ROOT, "member2_outputs", "chembl_train_tokens.npy"
)

VAL_PATH = os.path.join(
    ROOT, "member2_outputs", "chembl_val_tokens.npy"
)

TOKENIZER_PATH = os.path.join(
    ROOT, "member2_outputs", "smiles_tokenizer.json"
)

OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. LOAD TOKENIZED DATA
# ------------------------------------------------------------

X_train = np.load(TRAIN_PATH)
X_val = np.load(VAL_PATH)

print("\nDataset shapes:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)

# ------------------------------------------------------------
# 2. LOAD TOKENIZER VOCABULARY
# ------------------------------------------------------------

with open(TOKENIZER_PATH, "r") as f:
    tokenizer_data = json.load(f)

def find_token_mapping(obj):
    """
    Find a dictionary mapping token strings to integer IDs.
    """
    if isinstance(obj, dict):

        if obj and all(
            isinstance(k, str) and isinstance(v, int)
            for k, v in obj.items()
        ):
            return obj

        for value in obj.values():
            result = find_token_mapping(value)

            if result is not None:
                return result

    elif isinstance(obj, list):

        for value in obj:
            result = find_token_mapping(value)

            if result is not None:
                return result

    return None


token_to_id = find_token_mapping(tokenizer_data)

if token_to_id is None:
    raise ValueError(
        "Could not automatically identify token-to-ID mapping. "
        "Inspect smiles_tokenizer.json before proceeding."
    )

id_to_token = {
    int(token_id): token
    for token, token_id in token_to_id.items()
}

print("\nVocabulary size:", len(token_to_id))

# ------------------------------------------------------------
# 3. IDENTIFY SPECIAL TOKENS
# ------------------------------------------------------------

print("\nSpecial tokens:")

for token, token_id in token_to_id.items():

    if token.upper() in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]:
        print(f"{token}: {token_id}")

PAD_ID = token_to_id.get("<PAD>", 0)
BOS_ID = token_to_id.get("<BOS>", 1)
EOS_ID = token_to_id.get("<EOS>", 2)
UNK_ID = token_to_id.get("<UNK>", 3)

# ------------------------------------------------------------
# 4. TOKEN FREQUENCY ANALYSIS
# ------------------------------------------------------------

def calculate_token_frequencies(data, dataset_name):

    counts = Counter(data.flatten().tolist())

    total_non_padding = sum(
        count for token_id, count in counts.items()
        if token_id != PAD_ID
    )

    rows = []

    for token_id, count in counts.items():

        token = id_to_token.get(
            int(token_id),
            f"UNKNOWN_ID_{token_id}"
        )

        rows.append({
            "dataset": dataset_name,
            "token_id": int(token_id),
            "token": token,
            "count": int(count),
            "frequency_percent": (
                100 * count / total_non_padding
                if total_non_padding else 0
            )
        })

    result = pd.DataFrame(rows)

    return result.sort_values(
        "count",
        ascending=False
    ).reset_index(drop=True)


train_freq = calculate_token_frequencies(
    X_train, "Training"
)

val_freq = calculate_token_frequencies(
    X_val, "Validation"
)

# ------------------------------------------------------------
# 5. DISPLAY MOST FREQUENT TOKENS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP 25 TRAINING TOKENS")
print("=" * 60)

print(
    train_freq.head(25).to_string(index=False)
)

print("\n" + "=" * 60)
print("TOP 25 VALIDATION TOKENS")
print("=" * 60)

print(
    val_freq.head(25).to_string(index=False)
)

# ------------------------------------------------------------
# 6. SEQUENCE LENGTH DISTRIBUTION
# ------------------------------------------------------------

def sequence_length_statistics(data, dataset_name):

    lengths = []

    for sequence in data:

        valid_tokens = sequence[
            sequence != PAD_ID
        ]

        special_count = sum(
            token_id in [BOS_ID, EOS_ID]
            for token_id in valid_tokens
        )

        molecular_length = max(
            0, len(valid_tokens) - special_count
        )

        lengths.append(molecular_length)

    lengths = np.array(lengths)

    print(f"\n{dataset_name} sequence lengths:")
    print("Mean:", round(float(np.mean(lengths)), 2))
    print("Median:", float(np.median(lengths)))
    print("Minimum:", int(np.min(lengths)))
    print("Maximum:", int(np.max(lengths)))
    print("95th percentile:", round(float(np.percentile(lengths, 95)), 2))

    return lengths


train_lengths = sequence_length_statistics(
    X_train, "Training"
)

val_lengths = sequence_length_statistics(
    X_val, "Validation"
)

# ------------------------------------------------------------
# 7. SPECIAL TOKEN COUNTS
# ------------------------------------------------------------

def special_token_counts(data, dataset_name):

    print(f"\n{dataset_name} special token counts:")

    print("PAD:", int(np.sum(data == PAD_ID)))
    print("BOS:", int(np.sum(data == BOS_ID)))
    print("EOS:", int(np.sum(data == EOS_ID)))
    print("UNK:", int(np.sum(data == UNK_ID)))


special_token_counts(X_train, "Training")
special_token_counts(X_val, "Validation")

# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

combined_freq = pd.concat(
    [train_freq, val_freq],
    ignore_index=True
)

output_path = os.path.join(
    OUTPUT_DIR,
    "cell194_token_frequency_audit.csv"
)

combined_freq.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\n" + "=" * 70)
print("CELL 194 COMPLETED")
print("=" * 70)

CELL 194: SMILES TOKEN DISTRIBUTION ANALYSIS

Dataset shapes:
Training: (45000, 143)
Validation: (5000, 143)

Vocabulary size: 81

Special tokens:
<PAD>: 0
<BOS>: 1
<EOS>: 2
<UNK>: 3

TOP 25 TRAINING TOKENS
 dataset  token_id  token   count  frequency_percent
Training         0  <PAD> 4123646         178.408240
Training        77      c  554727          24.000088
Training        22      C  377097          16.314982
Training         7      )  228827           9.900128
Training         6      (  228827           9.900128
Training        10      1  132658           5.739406
Training        27      O  132644           5.738801
Training        11      2  108104           4.677085
Training        19      =   86533           3.743823
Training        26      N   81258           3.515602
Training        78      n   64066           2.771795
Training        12      3   59398           2.569836
Training         1  <BOS>   45000           1.946911
Training         2  <EOS>   45000           1.94691

In [267]:
# ============================================================
# CELL 195: TRANSFORMER PREDICTION BIAS ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
from collections import Counter

print("=" * 70)
print("CELL 195: TRANSFORMER PREDICTION BIAS ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK REQUIRED OBJECTS
# ------------------------------------------------------------

required = [
    "molecular_transformer",
    "molecular_only_forward",
    "X_val",
    "id_to_token",
    "PAD_ID"
]

for name in required:
    if name not in globals():
        raise NameError(
            f"Required object '{name}' is missing. "
            "Run the corresponding model/data setup cells first."
        )

device = next(molecular_transformer.parameters()).device

print("Device:", device)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# 2. SET MODEL TO EVALUATION MODE
# ------------------------------------------------------------

molecular_transformer.eval()

batch_size = 32

actual_counts = Counter()
predicted_counts = Counter()

total_tokens = 0
correct_tokens = 0

# ------------------------------------------------------------
# 3. TEACHER-FORCED VALIDATION PREDICTIONS
# ------------------------------------------------------------

with torch.no_grad():

    for start in range(0, len(X_val), batch_size):

        batch = X_val[start:start + batch_size]

        inputs = torch.tensor(
            batch[:, :-1],
            dtype=torch.long,
            device=device
        )

        targets = torch.tensor(
            batch[:, 1:],
            dtype=torch.long,
            device=device
        )

        logits = molecular_only_forward(
            molecular_transformer,
            inputs
        )

        predictions = torch.argmax(
            logits,
            dim=-1
        )

        valid_mask = targets != PAD_ID

        actual_valid = targets[valid_mask]
        predicted_valid = predictions[valid_mask]

        actual_counts.update(
            actual_valid.cpu().numpy().tolist()
        )

        predicted_counts.update(
            predicted_valid.cpu().numpy().tolist()
        )

        correct_tokens += int(
            (predicted_valid == actual_valid).sum().item()
        )

        total_tokens += int(valid_mask.sum().item())

# ------------------------------------------------------------
# 4. OVERALL PREDICTION ACCURACY
# ------------------------------------------------------------

accuracy = correct_tokens / total_tokens

print("\n" + "=" * 60)
print("OVERALL VALIDATION PREDICTION")
print("=" * 60)

print("Total target tokens:", total_tokens)
print("Correct predictions:", correct_tokens)
print("Accuracy:", round(accuracy * 100, 3), "%")

# ------------------------------------------------------------
# 5. COMPARE ACTUAL AND PREDICTED TOKEN FREQUENCIES
# ------------------------------------------------------------

all_ids = sorted(
    set(actual_counts.keys()) |
    set(predicted_counts.keys())
)

rows = []

for token_id in all_ids:

    token = id_to_token.get(
        int(token_id),
        f"UNKNOWN_ID_{token_id}"
    )

    actual = actual_counts[token_id]
    predicted = predicted_counts[token_id]

    actual_percent = 100 * actual / total_tokens
    predicted_percent = 100 * predicted / total_tokens

    rows.append({
        "token_id": int(token_id),
        "token": token,
        "actual_count": actual,
        "predicted_count": predicted,
        "actual_percent": actual_percent,
        "predicted_percent": predicted_percent,
        "frequency_difference": (
            predicted_percent - actual_percent
        )
    })

comparison = pd.DataFrame(rows)

comparison = comparison.sort_values(
    "actual_count",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 60)
print("TOP 25 ACTUAL TOKENS VS PREDICTED TOKENS")
print("=" * 60)

print(
    comparison.head(25).round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 6. MOST FREQUENTLY PREDICTED TOKENS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP 15 MODEL-PREDICTED TOKENS")
print("=" * 60)

top_predicted = comparison.sort_values(
    "predicted_count",
    ascending=False
).head(15)

print(
    top_predicted[
        ["token", "actual_count", "predicted_count",
         "actual_percent", "predicted_percent"]
    ].round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 7. CHECK CARBON TOKEN PREDICTION BIAS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CARBON TOKEN ANALYSIS")
print("=" * 60)

for carbon_token in ["C", "c"]:

    row = comparison[
        comparison["token"] == carbon_token
    ]

    if len(row) == 0:
        print(carbon_token, "not found")
        continue

    row = row.iloc[0]

    print(f"\nToken: {carbon_token}")
    print("Actual frequency:",
          round(row["actual_percent"], 3), "%")
    print("Predicted frequency:",
          round(row["predicted_percent"], 3), "%")
    print("Difference:",
          round(row["frequency_difference"], 3), "percentage points")

# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    "/content/drive/MyDrive/Member2_Project",
    "member2_outputs",
    "cell195_prediction_bias_audit.csv"
)

comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\n" + "=" * 70)
print("CELL 195 COMPLETED")
print("=" * 70)

CELL 195: TRANSFORMER PREDICTION BIAS ANALYSIS
Device: cpu
Validation shape: (5000, 143)

OVERALL VALIDATION PREDICTION
Total target tokens: 252277
Correct predictions: 153219
Accuracy: 60.734 %

TOP 25 ACTUAL TOKENS VS PREDICTED TOKENS
 token_id  token  actual_count  predicted_count  actual_percent  predicted_percent  frequency_difference
       77      c         61378            75802         24.3296            30.0471                5.7175
       22      C         42157            58787         16.7106            23.3026                6.5920
        7      )         25499            33346         10.1075            13.2180                3.1105
        6      (         25499            21642         10.1075             8.5787               -1.5289
       10      1         14860            12851          5.8904             5.0940               -0.7963
       27      O         14725             9539          5.8368             3.7812               -2.0557
       11      2         120

In [268]:
# ============================================================
# CELL 196: TOKEN-LEVEL MODEL PERFORMANCE
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
from collections import defaultdict

print("=" * 70)
print("CELL 196: TOKEN-LEVEL MODEL PERFORMANCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. VERIFY REQUIRED OBJECTS
# ------------------------------------------------------------

required = [
    "molecular_transformer",
    "molecular_only_forward",
    "X_val",
    "id_to_token",
    "PAD_ID"
]

for name in required:
    if name not in globals():
        raise NameError(f"Missing required object: {name}")

device = next(
    molecular_transformer.parameters()
).device

molecular_transformer.eval()

print("Device:", device)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# 2. COLLECT VALIDATION PREDICTIONS
# ------------------------------------------------------------

actual_all = []
predicted_all = []
position_correct = defaultdict(int)
position_total = defaultdict(int)

batch_size = 32

with torch.no_grad():

    for start in range(0, len(X_val), batch_size):

        batch = X_val[start:start + batch_size]

        inputs = torch.tensor(
            batch[:, :-1],
            dtype=torch.long,
            device=device
        )

        targets = torch.tensor(
            batch[:, 1:],
            dtype=torch.long,
            device=device
        )

        logits = molecular_only_forward(
            molecular_transformer,
            inputs
        )

        predictions = logits.argmax(dim=-1)

        valid_mask = targets != PAD_ID

        actual = targets[valid_mask]
        predicted = predictions[valid_mask]

        actual_all.extend(
            actual.cpu().numpy().tolist()
        )

        predicted_all.extend(
            predicted.cpu().numpy().tolist()
        )

        # Accuracy by sequence position
        correct_matrix = (
            (predictions == targets) & valid_mask
        )

        for position in range(targets.shape[1]):

            position_correct[position] += int(
                correct_matrix[:, position].sum().item()
            )

            position_total[position] += int(
                valid_mask[:, position].sum().item()
            )

actual_all = np.array(actual_all)
predicted_all = np.array(predicted_all)

# ------------------------------------------------------------
# 3. CALCULATE TOKEN-LEVEL METRICS
# ------------------------------------------------------------

all_token_ids = sorted(
    set(actual_all.tolist()) |
    set(predicted_all.tolist())
)

rows = []

for token_id in all_token_ids:

    true_positive = int(
        np.sum(
            (actual_all == token_id) &
            (predicted_all == token_id)
        )
    )

    actual_count = int(
        np.sum(actual_all == token_id)
    )

    predicted_count = int(
        np.sum(predicted_all == token_id)
    )

    precision = (
        true_positive / predicted_count
        if predicted_count else 0.0
    )

    recall = (
        true_positive / actual_count
        if actual_count else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    rows.append({
        "token_id": token_id,
        "token": id_to_token.get(
            int(token_id),
            f"UNKNOWN_{token_id}"
        ),
        "actual_count": actual_count,
        "predicted_count": predicted_count,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

metrics = pd.DataFrame(rows)

metrics = metrics.sort_values(
    "actual_count",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. DISPLAY IMPORTANT TOKEN RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOKEN-LEVEL PRECISION, RECALL AND F1")
print("=" * 65)

print(
    metrics.head(30).round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 5. IDENTIFY WORST-PREDICTED COMMON TOKENS
# ------------------------------------------------------------

common_tokens = metrics[
    metrics["actual_count"] >= 100
].sort_values(
    "recall",
    ascending=True
)

print("\n" + "=" * 65)
print("15 MOST DIFFICULT COMMON TOKENS")
print("=" * 65)

print(
    common_tokens.head(15).round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 6. ACCURACY BY SEQUENCE POSITION
# ------------------------------------------------------------

position_rows = []

for position in sorted(position_total):

    total = position_total[position]

    if total == 0:
        continue

    correct = position_correct[position]

    position_rows.append({
        "position": position,
        "valid_targets": total,
        "correct_predictions": correct,
        "accuracy_percent": 100 * correct / total
    })

position_df = pd.DataFrame(position_rows)

print("\n" + "=" * 65)
print("ACCURACY BY SEQUENCE POSITION")
print("=" * 65)

print(position_df.head(30).round(3).to_string(index=False))

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_dir = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs"
)

os.makedirs(output_dir, exist_ok=True)

metrics_path = os.path.join(
    output_dir,
    "cell196_token_level_metrics.csv"
)

position_path = os.path.join(
    output_dir,
    "cell196_position_accuracy.csv"
)

metrics.to_csv(metrics_path, index=False)
position_df.to_csv(position_path, index=False)

print("\nSaved token metrics:", metrics_path)
print("Saved position metrics:", position_path)

print("\n" + "=" * 70)
print("CELL 196 COMPLETED")
print("=" * 70)

CELL 196: TOKEN-LEVEL MODEL PERFORMANCE
Device: cpu
Validation shape: (5000, 143)

TOKEN-LEVEL PRECISION, RECALL AND F1
 token_id  token  actual_count  predicted_count  precision  recall  f1_score
       77      c         61378            75802     0.6561  0.8103    0.7251
       22      C         42157            58787     0.4907  0.6843    0.5716
        7      )         25499            33346     0.7008  0.9165    0.7943
        6      (         25499            21642     0.5154  0.4374    0.4732
       10      1         14860            12851     0.6792  0.5873    0.6299
       27      O         14725             9539     0.8198  0.5311    0.6446
       11      2         12012             8162     0.6470  0.4396    0.5235
       19      =          9534             9544     0.7021  0.7029    0.7025
       26      N          9178             5295     0.4089  0.2359    0.2992
       78      n          7248               22     0.2273  0.0007    0.0014
       12      3          6676   

In [269]:
# ============================================================
# CELL 197: TOKEN PROBABILITY AND LOSS ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("CELL 197: TOKEN PROBABILITY AND LOSS ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. VERIFY OBJECTS
# ------------------------------------------------------------

required = [
    "molecular_transformer",
    "molecular_only_forward",
    "X_val",
    "id_to_token",
    "PAD_ID"
]

for name in required:
    if name not in globals():
        raise NameError(f"Missing required object: {name}")

device = next(
    molecular_transformer.parameters()
).device

molecular_transformer.eval()

print("Device:", device)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# 2. COLLECT TOKEN PROBABILITIES AND LOSSES
# ------------------------------------------------------------

token_loss_sum = {}
token_probability_sum = {}
token_correct = {}
token_count = {}

batch_size = 32

with torch.no_grad():

    for start in range(0, len(X_val), batch_size):

        batch = X_val[start:start + batch_size]

        inputs = torch.tensor(
            batch[:, :-1],
            dtype=torch.long,
            device=device
        )

        targets = torch.tensor(
            batch[:, 1:],
            dtype=torch.long,
            device=device
        )

        logits = molecular_only_forward(
            molecular_transformer,
            inputs
        )

        log_probs = torch.log_softmax(
            logits.float(),
            dim=-1
        )

        probs = torch.softmax(
            logits.float(),
            dim=-1
        )

        predictions = logits.argmax(dim=-1)

        valid_positions = torch.where(
            targets != PAD_ID
        )

        actual_ids = targets[valid_positions]

        correct_probs = probs[
            valid_positions[0],
            valid_positions[1],
            actual_ids
        ]

        correct_losses = -torch.log(
            correct_probs.clamp(min=1e-12)
        )

        correct_predictions = (
            predictions[valid_positions] == actual_ids
        )

        for token_id in torch.unique(actual_ids).tolist():

            mask = actual_ids == token_id

            count = int(mask.sum().item())

            token_loss_sum[token_id] = (
                token_loss_sum.get(token_id, 0.0)
                + correct_losses[mask].sum().item()
            )

            token_probability_sum[token_id] = (
                token_probability_sum.get(token_id, 0.0)
                + correct_probs[mask].sum().item()
            )

            token_correct[token_id] = (
                token_correct.get(token_id, 0)
                + correct_predictions[mask].sum().item()
            )

            token_count[token_id] = (
                token_count.get(token_id, 0) + count
            )

# ------------------------------------------------------------
# 3. BUILD TOKEN METRICS
# ------------------------------------------------------------

rows = []

for token_id, count in token_count.items():

    rows.append({
        "token_id": int(token_id),

        "token": id_to_token.get(
            int(token_id),
            f"UNKNOWN_{token_id}"
        ),

        "count": count,

        "mean_correct_probability": (
            token_probability_sum[token_id] / count
        ),

        "mean_nll": (
            token_loss_sum[token_id] / count
        ),

        "top1_accuracy": (
            token_correct[token_id] / count
        )
    })

results = pd.DataFrame(rows)

results = results.sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. DISPLAY IMPORTANT TOKEN RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TOP 30 TOKENS: PROBABILITY AND LOSS")
print("=" * 65)

print(
    results.head(30).round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 5. DISPLAY RARE TOKEN PERFORMANCE
# ------------------------------------------------------------

rare_tokens = results[
    results["count"] < 2000
].sort_values(
    "count",
    ascending=False
)

print("\n" + "=" * 65)
print("RARE TOKEN PERFORMANCE")
print("=" * 65)

print(
    rare_tokens.head(30).round(5).to_string(index=False)
)

# ------------------------------------------------------------
# 6. FREQUENCY GROUP ANALYSIS
# ------------------------------------------------------------

def frequency_group(count):

    if count < 500:
        return "Very rare (<500)"

    elif count < 2000:
        return "Rare (500-1999)"

    elif count < 10000:
        return "Medium (2000-9999)"

    else:
        return "Frequent (10000+)"

results["frequency_group"] = results[
    "count"
].apply(frequency_group)

group_summary = results.groupby(
    "frequency_group"
).apply(
    lambda group: pd.Series({
        "number_of_tokens": len(group),
        "total_occurrences": group["count"].sum(),
        "mean_top1_accuracy": np.average(
            group["top1_accuracy"],
            weights=group["count"]
        ),
        "mean_correct_probability": np.average(
            group["mean_correct_probability"],
            weights=group["count"]
        ),
        "mean_nll": np.average(
            group["mean_nll"],
            weights=group["count"]
        )
    }),
    include_groups=False
)

print("\n" + "=" * 65)
print("PERFORMANCE BY TOKEN FREQUENCY")
print("=" * 65)

print(group_summary.round(5).to_string())

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_dir = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs"
)

os.makedirs(output_dir, exist_ok=True)

token_path = os.path.join(
    output_dir,
    "cell197_token_probability_loss.csv"
)

group_path = os.path.join(
    output_dir,
    "cell197_frequency_group_summary.csv"
)

results.to_csv(token_path, index=False)
group_summary.to_csv(group_path)

print("\nSaved:", token_path)
print("Saved:", group_path)

print("\n" + "=" * 70)
print("CELL 197 COMPLETED")
print("=" * 70)

CELL 197: TOKEN PROBABILITY AND LOSS ANALYSIS
Device: cpu
Validation shape: (5000, 143)

TOP 30 TOKENS: PROBABILITY AND LOSS
 token_id  token  count  mean_correct_probability  mean_nll  top1_accuracy
       77      c  61378                   0.61379   0.66733        0.81027
       22      C  42157                   0.42744   1.03897        0.68432
        7      )  25499                   0.75947   0.39125        0.91647
        6      (  25499                   0.34484   1.33065        0.43743
       10      1  14860                   0.50809   1.03046        0.58735
       27      O  14725                   0.54837   1.14797        0.53107
       11      2  12012                   0.38132   1.47216        0.43964
       19      =   9534                   0.56790   1.05398        0.70285
       26      N   9178                   0.20125   2.03162        0.23589
       78      n   7248                   0.10514   2.59324        0.00069
       12      3   6676                   0.31077 

In [270]:
# ============================================================
# CELL 198: CONTROLLED FREQUENCY-WEIGHTED FINE-TUNING
# ============================================================

import os
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("=" * 70)
print("CELL 198: FREQUENCY-WEIGHTED FINE-TUNING")
print("=" * 70)

# ------------------------------------------------------------
# 1. VERIFY REQUIRED OBJECTS
# ------------------------------------------------------------

required = [
    "molecular_transformer",
    "molecular_only_forward",
    "X_train",
    "X_val",
    "PAD_ID"
]

for name in required:
    if name not in globals():
        raise NameError(f"Missing required object: {name}")

ROOT = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(
    ROOT, "member2_outputs"
)

CHECKPOINT_DIR = os.path.join(
    ROOT, "checkpoints"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = next(
    molecular_transformer.parameters()
).device

print("Device:", device)

# ------------------------------------------------------------
# 2. LOAD EXACT SELECTED TRAINING INDICES
# ------------------------------------------------------------

indices_path = os.path.join(
    OUTPUT_DIR,
    "transformer_selected_10k_indices.npy"
)

if os.path.exists(indices_path):

    selected_indices = np.load(indices_path)

elif "selected_indices" in globals():

    selected_indices = np.asarray(
        selected_indices
    )

else:

    raise FileNotFoundError(
        "Selected training indices are missing."
    )

if len(selected_indices) != 10000:
    raise ValueError(
        f"Expected 10000 selected indices, "
        f"found {len(selected_indices)}"
    )

if np.max(selected_indices) >= len(X_train):
    raise ValueError("Selected indices exceed training data.")

X_selected = X_train[selected_indices]

print("\nSelected training shape:", X_selected.shape)
print("Validation shape:", X_val.shape)

# ------------------------------------------------------------
# 3. CALCULATE TOKEN FREQUENCIES
# ------------------------------------------------------------

targets_np = X_selected[:, 1:]

valid_targets = targets_np[
    targets_np != PAD_ID
]

vocab_size = molecular_transformer.token_embedding.num_embeddings

token_counts = np.bincount(
    valid_targets.astype(np.int64),
    minlength=vocab_size
)

# ------------------------------------------------------------
# 4. CALCULATE MODERATE TOKEN WEIGHTS
# ------------------------------------------------------------

# Inverse square-root frequency:
# rare tokens receive higher weights.

frequencies = token_counts.astype(np.float64)

weights = np.ones(
    vocab_size,
    dtype=np.float64
)

present = frequencies > 0

weights[present] = (
    frequencies[present].mean()
    / frequencies[present]
) ** 0.5

# Avoid excessive weighting of extremely rare tokens.
weights = np.clip(weights, 0.5, 3.0)

# Normalize weights across observed tokens.
weights[present] /= weights[present].mean()

# PAD must contribute no training loss.
weights[PAD_ID] = 0.0

token_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=device
)

print("\nToken weight statistics:")
print("Minimum:", round(float(weights[present].min()), 4))
print("Maximum:", round(float(weights[present].max()), 4))
print("Mean:", round(float(weights[present].mean()), 4))

# ------------------------------------------------------------
# 5. CREATE DATA LOADERS
# ------------------------------------------------------------

train_inputs = torch.tensor(
    X_selected[:, :-1],
    dtype=torch.long
)

train_targets = torch.tensor(
    X_selected[:, 1:],
    dtype=torch.long
)

train_loader = DataLoader(
    TensorDataset(train_inputs, train_targets),
    batch_size=32,
    shuffle=True
)

# ------------------------------------------------------------
# 6. COPY EXISTING MODEL
# ------------------------------------------------------------

weighted_model = copy.deepcopy(
    molecular_transformer
)

weighted_model.to(device)
weighted_model.train()

optimizer = torch.optim.AdamW(
    weighted_model.parameters(),
    lr=2e-5,
    weight_decay=1e-5
)

print("\nExperimental model created.")
print("Original model remains unchanged.")

# ------------------------------------------------------------
# 7. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_model(model, data, batch_size=32):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.no_grad():

        for start in range(0, len(data), batch_size):

            batch = data[start:start + batch_size]

            inputs = torch.tensor(
                batch[:, :-1],
                dtype=torch.long,
                device=device
            )

            targets = torch.tensor(
                batch[:, 1:],
                dtype=torch.long,
                device=device
            )

            logits = molecular_only_forward(
                model,
                inputs
            )

            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1),
                ignore_index=PAD_ID,
                reduction="sum"
            )

            predictions = logits.argmax(dim=-1)

            valid = targets != PAD_ID

            total_loss += loss.item()

            total_correct += (
                (predictions == targets) & valid
            ).sum().item()

            total_tokens += valid.sum().item()

    return {
        "loss": total_loss / total_tokens,
        "accuracy": total_correct / total_tokens,
        "tokens": total_tokens
    }


# ------------------------------------------------------------
# 8. BASELINE VALIDATION
# ------------------------------------------------------------

baseline_metrics = evaluate_model(
    molecular_transformer,
    X_val
)

print("\nBaseline validation:")
print(baseline_metrics)

# ------------------------------------------------------------
# 9. WEIGHTED FINE-TUNING
# ------------------------------------------------------------

history = []

EPOCHS = 2

for epoch in range(EPOCHS):

    weighted_model.train()

    epoch_loss = 0.0
    epoch_tokens = 0

    for inputs, targets in train_loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = molecular_only_forward(
            weighted_model,
            inputs
        )

        per_token_loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            targets.reshape(-1),
            ignore_index=PAD_ID,
            reduction="none"
        ).reshape_as(targets)

        valid = targets != PAD_ID

        target_weights = token_weights[
            targets
        ]

        weighted_loss = (
            per_token_loss * target_weights * valid
        ).sum() / valid.sum().clamp(min=1)

        weighted_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            weighted_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        epoch_loss += (
            weighted_loss.item() * valid.sum().item()
        )

        epoch_tokens += valid.sum().item()

    # Ordinary validation metrics, not weighted.
    val_metrics = evaluate_model(
        weighted_model,
        X_val
    )

    record = {
        "epoch": epoch + 1,
        "weighted_train_loss": epoch_loss / epoch_tokens,
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"]
    }

    history.append(record)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(record)

# ------------------------------------------------------------
# 10. SAVE EXPERIMENTAL CHECKPOINT
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "transformer_frequency_weighted_experiment.pt"
)

torch.save(
    {
        "model_state_dict": weighted_model.state_dict(),
        "epochs": EPOCHS,
        "training_type": "frequency_weighted_molecular_finetuning",
        "token_weight_method": "inverse_sqrt_frequency_clipped_0.5_3.0",
        "baseline_validation": baseline_metrics,
        "history": history,
        "pocket_cross_attention_used": False
    },
    checkpoint_path
)

# ------------------------------------------------------------
# 11. SAVE TRAINING HISTORY
# ------------------------------------------------------------

history_path = os.path.join(
    OUTPUT_DIR,
    "cell198_weighted_training_history.csv"
)

pd.DataFrame(history).to_csv(
    history_path,
    index=False
)

# ------------------------------------------------------------
# 12. FINAL COMPARISON
# ------------------------------------------------------------

final_metrics = evaluate_model(
    weighted_model,
    X_val
)

print("\n" + "=" * 65)
print("BASELINE VS WEIGHTED MODEL")
print("=" * 65)

print("Baseline loss:",
      round(baseline_metrics["loss"], 5))

print("Weighted loss:",
      round(final_metrics["loss"], 5))

print("Baseline accuracy:",
      round(baseline_metrics["accuracy"] * 100, 3), "%")

print("Weighted accuracy:",
      round(final_metrics["accuracy"] * 100, 3), "%")

print("\nCheckpoint saved:", checkpoint_path)
print("History saved:", history_path)

print("\n" + "=" * 70)
print("CELL 198 COMPLETED")
print("=" * 70)

CELL 198: FREQUENCY-WEIGHTED FINE-TUNING
Device: cpu

Selected training shape: (10000, 143)
Validation shape: (5000, 143)

Token weight statistics:
Minimum: 0.2136
Maximum: 1.2818
Mean: 1.0

Experimental model created.
Original model remains unchanged.

Baseline validation:
{'loss': 1.1923144488281978, 'accuracy': 0.6073443080423503, 'tokens': 252277}

Epoch 1/2
{'epoch': 1, 'weighted_train_loss': 0.4153795076617523, 'validation_loss': 1.2455292443011086, 'validation_accuracy': 0.5997217344426959}

Epoch 2/2
{'epoch': 2, 'weighted_train_loss': 0.4116970300478706, 'validation_loss': 1.238500950868784, 'validation_accuracy': 0.6011011705387332}

BASELINE VS WEIGHTED MODEL
Baseline loss: 1.19231
Weighted loss: 1.2385
Baseline accuracy: 60.734 %
Weighted accuracy: 60.11 %

Checkpoint saved: /content/drive/MyDrive/Member2_Project/checkpoints/transformer_frequency_weighted_experiment.pt
History saved: /content/drive/MyDrive/Member2_Project/member2_outputs/cell198_weighted_training_history.cs

In [271]:
# ============================================================
# CELL 199: GENERATION PREREQUISITE CHECK
# ============================================================

import os
import json
import torch
import inspect

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

print("=" * 65)
print("CELL 199: GENERATION PREREQUISITE CHECK")
print("=" * 65)

# 1. Check models
required_variables = [
    "molecular_transformer",
    "weighted_model",
    "molecular_only_forward",
    "PAD_ID",
    "BOS_ID",
    "EOS_ID"
]

missing = [x for x in required_variables if x not in globals()]

if missing:
    raise RuntimeError(f"Missing required variables: {missing}")

print("\n1. MODEL CHECK")
print("Baseline model:", type(molecular_transformer).__name__)
print("Weighted model:", type(weighted_model).__name__)

print("Baseline parameters:",
      sum(p.numel() for p in molecular_transformer.parameters()))

print("Weighted parameters:",
      sum(p.numel() for p in weighted_model.parameters()))

# 2. Check model vocabulary
print("\n2. VOCABULARY CHECK")
print("Baseline vocabulary:",
      molecular_transformer.output_projection.out_features
      if hasattr(molecular_transformer, "output_projection")
      else "Inspecting output layer")

# 3. Check tokenizer file
tokenizer_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "smiles_tokenizer.json"
)

if not os.path.exists(tokenizer_path):
    raise FileNotFoundError(tokenizer_path)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

print("\n3. TOKENIZER CHECK")
print("Tokenizer file:", tokenizer_path)
print("Tokenizer type:", type(tokenizer_data).__name__)

if isinstance(tokenizer_data, dict):
    print("Tokenizer keys:", list(tokenizer_data.keys()))

    for key, value in tokenizer_data.items():
        if isinstance(value, dict):
            print(f"\n{key}:")
            print("  Entries:", len(value))
            print("  Sample:", list(value.items())[:8])
        elif isinstance(value, list):
            print(f"\n{key}:")
            print("  Length:", len(value))
            print("  Sample:", value[:8])

# 4. Check special tokens
print("\n4. SPECIAL TOKEN IDS")
print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

# 5. Check generation function
print("\n5. FORWARD FUNCTION")
print(inspect.signature(molecular_only_forward))

# 6. Check model output shape
molecular_transformer.eval()

test_input = torch.tensor(
    [[BOS_ID, 22, 22]],
    dtype=torch.long
)

with torch.no_grad():
    test_output = molecular_only_forward(
        molecular_transformer,
        test_input
    )

if isinstance(test_output, (tuple, list)):
    test_output = test_output[0]

print("Test input shape:", tuple(test_input.shape))
print("Test output shape:", tuple(test_output.shape))

# 7. Check checkpoint
checkpoint_path = os.path.join(
    PROJECT_DIR,
    "checkpoints",
    "transformer_frequency_weighted_experiment.pt"
)

print("\n6. CHECKPOINT")
print("Weighted checkpoint exists:", os.path.exists(checkpoint_path))

print("\n" + "=" * 65)
print("CELL 199 COMPLETED")
print("=" * 65)

CELL 199: GENERATION PREREQUISITE CHECK

1. MODEL CHECK
Baseline model: PocketConditionedTransformer
Weighted model: PocketConditionedTransformer
Baseline parameters: 832849
Weighted parameters: 832849

2. VOCABULARY CHECK
Baseline vocabulary: Inspecting output layer

3. TOKENIZER CHECK
Tokenizer file: /content/drive/MyDrive/Member2_Project/member2_outputs/smiles_tokenizer.json
Tokenizer type: dict
Tokenizer keys: ['vocabulary', 'token_to_id', 'max_length', 'special_tokens']

vocabulary:
  Length: 81
  Sample: ['<PAD>', '<BOS>', '<EOS>', '<UNK>', '#', '%10', '(', ')']

token_to_id:
  Entries: 81
  Sample: [('<PAD>', 0), ('<BOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('#', 4), ('%10', 5), ('(', 6), (')', 7)]

special_tokens:
  Length: 4
  Sample: ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

4. SPECIAL TOKEN IDS
PAD_ID: 0
BOS_ID: 1
EOS_ID: 2

5. FORWARD FUNCTION
(model, token_ids)
Test input shape: (1, 3)
Test output shape: (1, 3, 81)

6. CHECKPOINT
Weighted checkpoint exists: True

CELL 199 COMPLET

In [272]:
# ============================================================
# CELL 200: CONTROLLED GENERATION - BASELINE VS WEIGHTED
# ============================================================

import os
import json
import torch
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdFingerprintGenerator
from rdkit.DataStructs import TanimotoSimilarity

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "member2_outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_SAMPLES = 50
TEMPERATURE = 0.8
TOP_K = 20
RANDOM_SEED = 42
MAX_NEW_TOKENS = 142

device = torch.device("cpu")

# ------------------------------------------------------------
# 2. LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    OUTPUT_DIR, "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

vocabulary = tokenizer_data["vocabulary"]

print("=" * 70)
print("CELL 200: CONTROLLED MOLECULAR GENERATION")
print("=" * 70)

print("Samples per model:", NUM_SAMPLES)
print("Temperature:", TEMPERATURE)
print("Top-k:", TOP_K)
print("Random seed:", RANDOM_SEED)
print("Device:", device)

# ------------------------------------------------------------
# 3. GENERATION FUNCTION
# ------------------------------------------------------------

def generate_smiles(
    model,
    num_samples,
    seed,
    temperature=0.8,
    top_k=20,
    max_new_tokens=142
):

    model.eval()
    model.to(device)

    generator = torch.Generator(device=device)
    generator.manual_seed(seed)

    sequences = torch.full(
        (num_samples, 1),
        BOS_ID,
        dtype=torch.long,
        device=device
    )

    finished = torch.zeros(
        num_samples,
        dtype=torch.bool,
        device=device
    )

    with torch.no_grad():

        for step in range(max_new_tokens):

            logits = molecular_only_forward(
                model,
                sequences
            )

            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            next_logits = logits[:, -1, :].clone()

            # Prevent generation of special tokens
            next_logits[:, PAD_ID] = -float("inf")
            next_logits[:, BOS_ID] = -float("inf")
            next_logits[:, 3] = -float("inf")  # UNK

            # Temperature scaling
            next_logits = next_logits / temperature

            # Top-k filtering
            top_values, _ = torch.topk(
                next_logits,
                top_k,
                dim=-1
            )

            threshold = top_values[:, -1].unsqueeze(-1)

            next_logits[next_logits < threshold] = -float("inf")

            probabilities = torch.softmax(
                next_logits,
                dim=-1
            )

            next_tokens = torch.multinomial(
                probabilities,
                num_samples=1,
                generator=generator
            )

            # Already-finished sequences receive PAD
            next_tokens[finished] = PAD_ID

            sequences = torch.cat(
                [sequences, next_tokens],
                dim=1
            )

            finished = finished | (
                next_tokens.squeeze(1) == EOS_ID
            )

            if finished.all():
                break

    # Decode token IDs into SMILES
    generated = []

    for sequence in sequences.cpu().tolist():

        tokens = []

        for token_id in sequence:

            if token_id == BOS_ID:
                continue

            if token_id == EOS_ID:
                break

            if token_id == PAD_ID:
                break

            tokens.append(vocabulary[token_id])

        generated.append("".join(tokens))

    return generated


# ------------------------------------------------------------
# 4. GENERATE FROM BASELINE MODEL
# ------------------------------------------------------------

print("\nGenerating baseline molecules...")

baseline_smiles = generate_smiles(
    model=molecular_transformer,
    num_samples=NUM_SAMPLES,
    seed=RANDOM_SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    max_new_tokens=MAX_NEW_TOKENS
)

print("Baseline generation completed:", len(baseline_smiles))


# ------------------------------------------------------------
# 5. GENERATE FROM WEIGHTED MODEL
# ------------------------------------------------------------

print("\nGenerating weighted-model molecules...")

weighted_smiles = generate_smiles(
    model=weighted_model,
    num_samples=NUM_SAMPLES,
    seed=RANDOM_SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    max_new_tokens=MAX_NEW_TOKENS
)

print("Weighted generation completed:", len(weighted_smiles))


# ------------------------------------------------------------
# 6. MOLECULE VALIDATION AND DESCRIPTORS
# ------------------------------------------------------------

def audit_molecules(smiles_list, model_name):

    records = []

    for index, smiles in enumerate(smiles_list):

        mol = Chem.MolFromSmiles(smiles)

        record = {
            "model": model_name,
            "sample_id": index + 1,
            "smiles": smiles,
            "valid": mol is not None,
            "canonical_smiles": None,
            "MW": None,
            "LogP": None,
            "TPSA": None,
            "HBD": None,
            "HBA": None,
            "RotatableBonds": None,
            "HeavyAtoms": None,
            "Rings": None
        }

        if mol is not None:

            record["canonical_smiles"] = Chem.MolToSmiles(
                mol,
                canonical=True
            )

            record["MW"] = Descriptors.MolWt(mol)
            record["LogP"] = Descriptors.MolLogP(mol)
            record["TPSA"] = Descriptors.TPSA(mol)

            record["HBD"] = Lipinski.NumHDonors(mol)
            record["HBA"] = Lipinski.NumHAcceptors(mol)

            record["RotatableBonds"] = (
                Lipinski.NumRotatableBonds(mol)
            )

            record["HeavyAtoms"] = mol.GetNumHeavyAtoms()

            record["Rings"] = Lipinski.RingCount(mol)

        records.append(record)

    return records


baseline_records = audit_molecules(
    baseline_smiles,
    "Baseline"
)

weighted_records = audit_molecules(
    weighted_smiles,
    "Weighted"
)

all_records = baseline_records + weighted_records

generation_df = pd.DataFrame(all_records)


# ------------------------------------------------------------
# 7. GENERATION SUMMARY
# ------------------------------------------------------------

def generation_summary(records):

    df = pd.DataFrame(records)

    valid_df = df[df["valid"] == True]

    valid_count = len(valid_df)

    unique_count = valid_df[
        "canonical_smiles"
    ].nunique()

    return {
        "Total Generated": len(df),
        "Valid Molecules": valid_count,
        "Validity (%)": round(
            valid_count / len(df) * 100, 2
        ),
        "Unique Valid Molecules": unique_count,
        "Uniqueness (%)": round(
            unique_count / valid_count * 100, 2
        ) if valid_count else 0
    }


baseline_summary = generation_summary(baseline_records)
weighted_summary = generation_summary(weighted_records)

summary_df = pd.DataFrame(
    [
        {"Model": "Baseline", **baseline_summary},
        {"Model": "Weighted", **weighted_summary}
    ]
)


# ------------------------------------------------------------
# 8. MOLECULAR DIVERSITY
# ------------------------------------------------------------

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)


def calculate_diversity(records):

    valid_records = [
        r for r in records
        if r["valid"]
    ]

    unique_records = {}
    for record in valid_records:
        unique_records[record["canonical_smiles"]] = record

    molecules = [
        Chem.MolFromSmiles(smiles)
        for smiles in unique_records.keys()
    ]

    fingerprints = [
        fingerprint_generator.GetFingerprint(mol)
        for mol in molecules
    ]

    if len(fingerprints) < 2:
        return {
            "Unique Molecules": len(fingerprints),
            "Mean Pairwise Similarity": None,
            "Mean Pairwise Diversity": None
        }

    similarities = []

    for i in range(len(fingerprints)):
        for j in range(i + 1, len(fingerprints)):

            similarity = TanimotoSimilarity(
                fingerprints[i],
                fingerprints[j]
            )

            similarities.append(similarity)

    mean_similarity = float(np.mean(similarities))

    return {
        "Unique Molecules": len(fingerprints),
        "Mean Pairwise Similarity": round(
            mean_similarity, 4
        ),
        "Mean Pairwise Diversity": round(
            1 - mean_similarity, 4
        )
    }


baseline_diversity = calculate_diversity(baseline_records)
weighted_diversity = calculate_diversity(weighted_records)

diversity_df = pd.DataFrame([
    {"Model": "Baseline", **baseline_diversity},
    {"Model": "Weighted", **weighted_diversity}
])


# ------------------------------------------------------------
# 9. DESCRIPTOR SUMMARY
# ------------------------------------------------------------

descriptor_columns = [
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "HeavyAtoms",
    "Rings"
]

valid_generation_df = generation_df[
    generation_df["valid"] == True
]

descriptor_summary = (
    valid_generation_df
    .groupby("model")[descriptor_columns]
    .mean()
    .round(3)
)


# ------------------------------------------------------------
# 10. SAVE RESULTS
# ------------------------------------------------------------

generation_path = os.path.join(
    OUTPUT_DIR,
    "cell200_generation_comparison.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "cell200_generation_summary.csv"
)

diversity_path = os.path.join(
    OUTPUT_DIR,
    "cell200_diversity_comparison.csv"
)

descriptor_path = os.path.join(
    OUTPUT_DIR,
    "cell200_descriptor_comparison.csv"
)

generation_df.to_csv(generation_path, index=False)
summary_df.to_csv(summary_path, index=False)
diversity_df.to_csv(diversity_path, index=False)
descriptor_summary.to_csv(descriptor_path)


# ------------------------------------------------------------
# 11. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATION VALIDITY AND UNIQUENESS")
print("=" * 70)

print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("MOLECULAR DIVERSITY")
print("=" * 70)

print(diversity_df.to_string(index=False))

print("\n" + "=" * 70)
print("AVERAGE MOLECULAR DESCRIPTORS")
print("=" * 70)

print(descriptor_summary)

print("\n" + "=" * 70)
print("SAVED FILES")
print("=" * 70)

print(generation_path)
print(summary_path)
print(diversity_path)
print(descriptor_path)

print("\nCELL 200 COMPLETED")
print("=" * 70)

CELL 200: CONTROLLED MOLECULAR GENERATION
Samples per model: 50
Temperature: 0.8
Top-k: 20
Random seed: 42
Device: cpu

Generating baseline molecules...
Baseline generation completed: 50

Generating weighted-model molecules...
Weighted generation completed: 50

GENERATION VALIDITY AND UNIQUENESS
   Model  Total Generated  Valid Molecules  Validity (%)  Unique Valid Molecules  Uniqueness (%)
Baseline               50                2           4.0                       2           100.0
Weighted               50                2           4.0                       2           100.0

MOLECULAR DIVERSITY
   Model  Unique Molecules  Mean Pairwise Similarity  Mean Pairwise Diversity
Baseline                 2                    0.0746                   0.9254
Weighted                 2                    0.0755                   0.9245

AVERAGE MOLECULAR DESCRIPTORS
               MW   LogP    TPSA  HBD  HBA  RotatableBonds  HeavyAtoms  Rings
model                                           

In [273]:
# ============================================================
# CELL 201: INDIVIDUAL MOLECULE AND TOKEN DIAGNOSTICS
# ============================================================

import os
import json
import pandas as pd
from collections import Counter
from rdkit import Chem

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "member2_outputs"
)

INPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "cell200_generation_comparison.csv"
)

df = pd.read_csv(INPUT_PATH)

with open(
    os.path.join(OUTPUT_DIR, "smiles_tokenizer.json"), "r"
) as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]

# Longest tokens first for correct token matching
token_list = sorted(
    [
        token for token in token_to_id
        if token not in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
    ],
    key=len,
    reverse=True
)


# ------------------------------------------------------------
# 1. TOKENIZE GENERATED SMILES
# ------------------------------------------------------------

def tokenize_smiles(smiles):

    tokens = []
    i = 0

    while i < len(smiles):

        matched = None

        for token in token_list:

            if smiles.startswith(token, i):
                matched = token
                break

        if matched is None:
            tokens.append("<UNMATCHED>")
            i += 1
        else:
            tokens.append(matched)
            i += len(matched)

    return tokens


# ------------------------------------------------------------
# 2. INVALID MOLECULE DIAGNOSIS
# ------------------------------------------------------------

diagnostic_records = []

for _, row in df.iterrows():

    smiles = str(row["smiles"])

    mol = Chem.MolFromSmiles(smiles)

    tokens = tokenize_smiles(smiles)

    diagnostic_records.append({

        "model": row["model"],
        "sample_id": row["sample_id"],
        "smiles": smiles,
        "valid": mol is not None,
        "sequence_length": len(tokens),
        "token_count": len(tokens),
        "unmatched_tokens": tokens.count("<UNMATCHED>"),
        "contains_aromatic_n": "n" in tokens,
        "contains_stereochemistry": any(
            t in tokens
            for t in ["[C@H]", "[C@@H]", "[C@]", "[C@@]"]
        ),
        "contains_sulfur": any(
            t in tokens for t in ["S", "s"]
        )
    })


diagnostic_df = pd.DataFrame(diagnostic_records)


# ------------------------------------------------------------
# 3. DISPLAY VALID MOLECULES
# ------------------------------------------------------------

print("=" * 70)
print("CELL 201: VALID GENERATED MOLECULES")
print("=" * 70)

valid_df = df[df["valid"] == True]

for model_name in ["Baseline", "Weighted"]:

    print(f"\n{model_name} MODEL")

    model_valid = valid_df[
        valid_df["model"] == model_name
    ]

    if len(model_valid) == 0:
        print("No valid molecules.")
        continue

    for _, row in model_valid.iterrows():

        print("\nSample:", row["sample_id"])
        print("SMILES:", row["smiles"])
        print("Canonical:", row["canonical_smiles"])
        print("MW:", row["MW"])
        print("LogP:", row["LogP"])
        print("TPSA:", row["TPSA"])
        print("HBD:", row["HBD"])
        print("HBA:", row["HBA"])


# ------------------------------------------------------------
# 4. VALIDITY AND LENGTH SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATION DIAGNOSTICS")
print("=" * 70)

print(
    diagnostic_df.groupby("model").agg(
        total=("sample_id", "count"),
        valid=("valid", "sum"),
        mean_sequence_length=("sequence_length", "mean"),
        unmatched_token_count=("unmatched_tokens", "sum"),
        aromatic_n_molecules=("contains_aromatic_n", "sum"),
        stereochemistry_molecules=("contains_stereochemistry", "sum"),
        sulfur_molecules=("contains_sulfur", "sum")
    ).to_string()
)


# ------------------------------------------------------------
# 5. TOKEN FREQUENCY COMPARISON
# ------------------------------------------------------------

token_counts = {}

for model_name in ["Baseline", "Weighted"]:

    model_rows = df[
        df["model"] == model_name
    ]

    counts = Counter()

    for smiles in model_rows["smiles"]:

        counts.update(tokenize_smiles(str(smiles)))

    token_counts[model_name] = counts


all_tokens = set(
    token_counts["Baseline"].keys()
) | set(
    token_counts["Weighted"].keys()
)

token_frequency_df = pd.DataFrame([

    {
        "token": token,
        "baseline_count": token_counts["Baseline"][token],
        "weighted_count": token_counts["Weighted"][token]
    }

    for token in sorted(all_tokens)
])

token_frequency_df["difference"] = (
    token_frequency_df["weighted_count"]
    - token_frequency_df["baseline_count"]
)

print("\n" + "=" * 70)
print("GENERATED TOKEN FREQUENCIES")
print("=" * 70)

print(
    token_frequency_df.sort_values(
        "weighted_count",
        ascending=False
    ).head(30).to_string(index=False)
)


# ------------------------------------------------------------
# 6. SAVE DIAGNOSTICS
# ------------------------------------------------------------

diagnostic_path = os.path.join(
    OUTPUT_DIR,
    "cell201_generation_diagnostics.csv"
)

token_path = os.path.join(
    OUTPUT_DIR,
    "cell201_generated_token_frequency.csv"
)

diagnostic_df.to_csv(diagnostic_path, index=False)
token_frequency_df.to_csv(token_path, index=False)

print("\nSaved:")
print(diagnostic_path)
print(token_path)

print("\nCELL 201 COMPLETED")

CELL 201: VALID GENERATED MOLECULES

Baseline MODEL

Sample: 4
SMILES: COc1nc(N2C(=O)C2=O)ccc2ccccc2c1Cl
Canonical: COC1=NC(n2c(=O)c2=O)=CC=c2ccccc2=C1Cl
MW: 288.69000000000005
LogP: -0.2313
TPSA: 60.66
HBD: 0.0
HBA: 4.0

Sample: 12
SMILES: CCCC1CCC(C)C(=O)CC2CCCC2CC(C)c1
Canonical: CCCC1CCC(C)C(=O)CC2CCCC2CC(C)C1
MW: 278.48
LogP: 5.624400000000006
TPSA: 17.07
HBD: 0.0
HBA: 1.0

Weighted MODEL

Sample: 3
SMILES: COc1cc(C)n2c(C#N)c3[nH]c(-c3cc(Cl)nc3cc(Cl)c3nco2)cc1
Canonical: COC1=CC=C2NC3=C(C#N)N(OC=NC4=C(C=C4Cl)N=C(Cl)C=C23)C(C)=C1
MW: 414.25200000000007
LogP: 3.843480000000002
TPSA: 82.24
HBD: 1.0
HBA: 7.0

Sample: 13
SMILES: CCC[C@@](=O)c1c2c(cc(Cl)n3nnnc3SC2)nc(-c3ccccc3)n1
Canonical: CCCC(=O)c1nc(-c2ccccc2)nc2c1CSc1nnnn1C(Cl)=C2
MW: 398.8790000000001
LogP: 3.913000000000003
TPSA: 86.45
HBD: 0.0
HBA: 7.0

GENERATION DIAGNOSTICS
          total  valid  mean_sequence_length  unmatched_token_count  aromatic_n_molecules  stereochemistry_molecules  sulfur_molecules
model               

In [274]:
# ============================================================
# CELL 202: INVALID SMILES ROOT-CAUSE ANALYSIS
# ============================================================

import os
import re
import pandas as pd

from collections import Counter
from rdkit import Chem, rdBase

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "member2_outputs"
)

INPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "cell200_generation_comparison.csv"
)

df = pd.read_csv(INPUT_PATH)

print("=" * 70)
print("CELL 202: INVALID SMILES ROOT-CAUSE ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# 1. STRUCTURAL SYNTAX CHECKS
# ------------------------------------------------------------

def syntax_diagnostics(smiles):

    # Parentheses
    open_paren = smiles.count("(")
    close_paren = smiles.count(")")

    paren_balance = 0
    minimum_balance = 0

    for ch in smiles:
        if ch == "(":
            paren_balance += 1
        elif ch == ")":
            paren_balance -= 1

        minimum_balance = min(
            minimum_balance,
            paren_balance
        )

    # Square brackets
    open_bracket = smiles.count("[")
    close_bracket = smiles.count("]")

    # Ring identifiers
    ring_tokens = re.findall(
        r"%\d{2}|\d",
        smiles
    )

    ring_counts = Counter(ring_tokens)

    odd_ring_labels = [
        label
        for label, count in ring_counts.items()
        if count % 2 != 0
    ]

    # Directional bond count
    slash_count = smiles.count("/")
    backslash_count = smiles.count("\\")

    return {
        "open_parentheses": open_paren,
        "close_parentheses": close_paren,
        "parentheses_balanced": (
            paren_balance == 0
            and minimum_balance >= 0
        ),

        "open_brackets": open_bracket,
        "close_brackets": close_bracket,
        "brackets_balanced": (
            open_bracket == close_bracket
        ),

        "odd_ring_labels": ",".join(
            odd_ring_labels
        ),

        "ring_labels_balanced": (
            len(odd_ring_labels) == 0
        ),

        "slash_count": slash_count,
        "backslash_count": backslash_count
    }


# ------------------------------------------------------------
# 2. RDKit PARSING AND SANITIZATION
# ------------------------------------------------------------

def rdkit_diagnosis(smiles):

    result = {
        "parse_success": False,
        "sanitize_success": False,
        "failure_stage": None,
        "failure_reason": None
    }

    # Suppress RDKit diagnostic messages
    with rdBase.BlockLogs():

        try:

            mol = Chem.MolFromSmiles(
                smiles,
                sanitize=False
            )

            if mol is None:

                result["failure_stage"] = "Parsing"
                result["failure_reason"] = (
                    "RDKit could not construct molecular graph"
                )

                return result

            result["parse_success"] = True

            try:

                Chem.SanitizeMol(mol)

                result["sanitize_success"] = True

            except Exception as error:

                result["failure_stage"] = "Sanitization"
                result["failure_reason"] = str(error)

                return result

        except Exception as error:

            result["failure_stage"] = "Parsing"
            result["failure_reason"] = str(error)

            return result

    return result


# ------------------------------------------------------------
# 3. ANALYZE ALL GENERATED MOLECULES
# ------------------------------------------------------------

records = []

for _, row in df.iterrows():

    smiles = str(row["smiles"])

    syntax = syntax_diagnostics(smiles)

    rdkit_result = rdkit_diagnosis(smiles)

    records.append({

        "model": row["model"],
        "sample_id": row["sample_id"],
        "smiles": smiles,
        "previously_valid": row["valid"],

        **syntax,
        **rdkit_result

    })


diagnostic_df = pd.DataFrame(records)


# ------------------------------------------------------------
# 4. DISPLAY FAILURE SUMMARY
# ------------------------------------------------------------

print("\n1. PARSING AND SANITIZATION SUMMARY")

summary = diagnostic_df.groupby("model").agg(

    total=("sample_id", "count"),

    parse_success=("parse_success", "sum"),

    sanitize_success=("sanitize_success", "sum"),

    unbalanced_parentheses=(
        "parentheses_balanced",
        lambda x: (~x).sum()
    ),

    unbalanced_brackets=(
        "brackets_balanced",
        lambda x: (~x).sum()
    ),

    unbalanced_ring_labels=(
        "ring_labels_balanced",
        lambda x: (~x).sum()
    )

)

print(summary.to_string())


# ------------------------------------------------------------
# 5. FAILURE STAGE COUNTS
# ------------------------------------------------------------

print("\n2. FAILURE STAGES")

failure_summary = (

    diagnostic_df[
        diagnostic_df["sanitize_success"] == False
    ]

    .groupby(["model", "failure_stage"])

    .size()

    .unstack(fill_value=0)
)

print(failure_summary.to_string())


# ------------------------------------------------------------
# 6. INVALID MOLECULE EXAMPLES
# ------------------------------------------------------------

print("\n3. INVALID SMILES EXAMPLES")

invalid_df = diagnostic_df[
    diagnostic_df["sanitize_success"] == False
]

for model_name in ["Baseline", "Weighted"]:

    print(f"\n{model_name} MODEL")

    subset = invalid_df[
        invalid_df["model"] == model_name
    ].head(10)

    for _, row in subset.iterrows():

        print("\nSample:", row["sample_id"])
        print("SMILES:", row["smiles"])
        print("Failure stage:", row["failure_stage"])
        print("Reason:", row["failure_reason"])

        print(
            "Parentheses balanced:",
            row["parentheses_balanced"]
        )

        print(
            "Brackets balanced:",
            row["brackets_balanced"]
        )

        print(
            "Ring labels balanced:",
            row["ring_labels_balanced"]
        )


# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    OUTPUT_DIR,
    "cell202_invalid_smiles_diagnostics.csv"
)

diagnostic_df.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

print("\nCELL 202 COMPLETED")
print("=" * 70)

CELL 202: INVALID SMILES ROOT-CAUSE ANALYSIS

1. PARSING AND SANITIZATION SUMMARY
          total  parse_success  sanitize_success  unbalanced_parentheses  unbalanced_brackets  unbalanced_ring_labels
model                                                                                                                
Baseline     50             10                 2                      15                    0                      37
Weighted     50              5                 2                      16                    0                      43

2. FAILURE STAGES
failure_stage  Parsing  Sanitization
model                               
Baseline            40             8
Weighted            45             3

3. INVALID SMILES EXAMPLES

Baseline MODEL

Sample: 1
SMILES: CCOC(=O)C1=C(C)NC(=O)C2CCC(c3cnccc3)C(C)(C)CCCN)c1
Failure stage: Parsing
Reason: RDKit could not construct molecular graph
Parentheses balanced: False
Brackets balanced: True
Ring labels balanced: False

Sample: 2
S

In [275]:
# ============================================================
# CELL 203: EOS TERMINATION AND TRUNCATION ANALYSIS
# ============================================================

import os
import json
import torch
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

NUM_SAMPLES = 50
TEMPERATURE = 0.8
TOP_K = 20
RANDOM_SEED = 42
MAX_NEW_TOKENS = 142

device = torch.device("cpu")

with open(
    os.path.join(OUTPUT_DIR, "smiles_tokenizer.json"), "r"
) as f:
    tokenizer_data = json.load(f)

vocabulary = tokenizer_data["vocabulary"]


# ------------------------------------------------------------
# GENERATION WITH TERMINATION TRACKING
# ------------------------------------------------------------

def generate_with_termination(model, model_name):

    model.eval()

    generator = torch.Generator(device=device)
    generator.manual_seed(RANDOM_SEED)

    sequences = torch.full(
        (NUM_SAMPLES, 1),
        BOS_ID,
        dtype=torch.long,
        device=device
    )

    finished = torch.zeros(
        NUM_SAMPLES,
        dtype=torch.bool,
        device=device
    )

    eos_step = torch.full(
        (NUM_SAMPLES,),
        -1,
        dtype=torch.long
    )

    with torch.no_grad():

        for step in range(MAX_NEW_TOKENS):

            logits = molecular_only_forward(
                model,
                sequences
            )

            if isinstance(logits, (tuple, list)):
                logits = logits[0]

            next_logits = logits[:, -1, :].clone()

            next_logits[:, PAD_ID] = -float("inf")
            next_logits[:, BOS_ID] = -float("inf")
            next_logits[:, 3] = -float("inf")

            next_logits = next_logits / TEMPERATURE

            top_values, _ = torch.topk(
                next_logits,
                TOP_K,
                dim=-1
            )

            threshold = top_values[:, -1].unsqueeze(-1)

            next_logits[
                next_logits < threshold
            ] = -float("inf")

            probabilities = torch.softmax(
                next_logits,
                dim=-1
            )

            next_tokens = torch.multinomial(
                probabilities,
                1,
                generator=generator
            )

            next_tokens[finished] = PAD_ID

            newly_finished = (
                (~finished) &
                (next_tokens.squeeze(1) == EOS_ID)
            )

            eos_step[newly_finished] = step + 1

            sequences = torch.cat(
                [sequences, next_tokens],
                dim=1
            )

            finished = finished | newly_finished

            if finished.all():
                break

    records = []

    for i, sequence in enumerate(sequences.cpu().tolist()):

        tokens = []

        for token_id in sequence:

            if token_id == BOS_ID:
                continue

            if token_id == EOS_ID or token_id == PAD_ID:
                break

            tokens.append(vocabulary[token_id])

        records.append({

            "model": model_name,
            "sample_id": i + 1,
            "smiles": "".join(tokens),
            "reached_eos": eos_step[i].item() != -1,
            "eos_generation_step": (
                eos_step[i].item()
                if eos_step[i].item() != -1
                else None
            ),
            "generated_tokens": len(tokens),
            "maximum_tokens": MAX_NEW_TOKENS,
            "truncated_or_unfinished": (
                eos_step[i].item() == -1
            )

        })

    return records


# ------------------------------------------------------------
# RUN BOTH MODELS
# ------------------------------------------------------------

print("=" * 70)
print("CELL 203: EOS AND TRUNCATION ANALYSIS")
print("=" * 70)

baseline_records = generate_with_termination(
    molecular_transformer,
    "Baseline"
)

weighted_records = generate_with_termination(
    weighted_model,
    "Weighted"
)

termination_df = pd.DataFrame(
    baseline_records + weighted_records
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

summary = termination_df.groupby("model").agg(

    total_samples=("sample_id", "count"),

    reached_eos=("reached_eos", "sum"),

    unfinished=("truncated_or_unfinished", "sum"),

    mean_generated_tokens=("generated_tokens", "mean"),

    maximum_generated_tokens=("generated_tokens", "max")

)

summary["EOS_percentage"] = (
    summary["reached_eos"] /
    summary["total_samples"] * 100
).round(2)

print("\nTERMINATION SUMMARY")
print(summary.to_string())


# ------------------------------------------------------------
# SHOW UNFINISHED EXAMPLES
# ------------------------------------------------------------

print("\nUNFINISHED GENERATION EXAMPLES")

unfinished_df = termination_df[
    termination_df["truncated_or_unfinished"]
]

for _, row in unfinished_df.head(10).iterrows():

    print("\nModel:", row["model"])
    print("Sample:", row["sample_id"])
    print("Generated tokens:", row["generated_tokens"])
    print("SMILES:", row["smiles"])


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output_path = os.path.join(
    OUTPUT_DIR,
    "cell203_termination_analysis.csv"
)

termination_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\nCELL 203 COMPLETED")
print("=" * 70)

CELL 203: EOS AND TRUNCATION ANALYSIS

TERMINATION SUMMARY
          total_samples  reached_eos  unfinished  mean_generated_tokens  maximum_generated_tokens  EOS_percentage
model                                                                                                            
Baseline             50           50           0                  44.24                        72           100.0
Weighted             50           48           2                  54.04                       142            96.0

UNFINISHED GENERATION EXAMPLES

Model: Weighted
Sample: 9
Generated tokens: 142
SMILES: CC[C@H](N)[C@H]([C@@H](C)[C@@H](C#N)[C@H]1C[C@H]2)[C@H](C)C[C@@H]1[C@@H](C)[C@H]1[C@H](O)[C@@H]3[C@@H]1[C@@H](C)[C@H]1O[C@H](CC[C@]4[C@]3[C@@H]6C[C@]5C[C@]6[C@]6[C@]5[C@H](C[C@H]5(C[C@@]5(C[C@]6(O)[C@@]5(O)[C@]6[C@@H](O)[C@@H]4[C@H](O)(C)[C@H]4[C@@]5)[C@@H](=O)[C@H]5(C[C@]3[C@@H]4C(=O5)[C@H]3C[C@H]4[C@@]5OC(

Model: Weighted
Sample: 36
Generated tokens: 142
SMILES: CC(=O)NC[C@H]1C[C@@H](C)[C@H

In [276]:
# ============================================================
# CELL 204: CONSOLIDATED BASELINE MODEL AUDIT
# ============================================================

import os
import json
import torch
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import TanimotoSimilarity

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

print("=" * 70)
print("CELL 204: CONSOLIDATED BASELINE MODEL AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK FINAL BASELINE CHECKPOINT
# ------------------------------------------------------------

baseline_path = os.path.join(
    CHECKPOINT_DIR,
    "transformer_chembl37_improved_best.pt"
)

weighted_path = os.path.join(
    CHECKPOINT_DIR,
    "transformer_frequency_weighted_experiment.pt"
)

assert os.path.exists(baseline_path), "Baseline checkpoint missing!"

checkpoint = torch.load(
    baseline_path,
    map_location="cpu",
    weights_only=False
)

print("\n1. BASELINE CHECKPOINT")

print("Path:", baseline_path)

if isinstance(checkpoint, dict):
    for key in [
        "epoch",
        "train_loss",
        "validation_loss",
        "training_molecules",
        "training_type",
        "pocket_cross_attention_used"
    ]:
        if key in checkpoint:
            print(f"{key}: {checkpoint[key]}")

print("Weighted experiment checkpoint exists:",
      os.path.exists(weighted_path))


# ------------------------------------------------------------
# 2. VALIDATION PERFORMANCE
# ------------------------------------------------------------

print("\n2. VALIDATION PERFORMANCE")

baseline_validation = {
    "loss": 1.1923144488281978,
    "accuracy": 0.6073443080423503
}

weighted_validation = {
    "loss": 1.238500950868784,
    "accuracy": 0.6011011705387332
}

validation_df = pd.DataFrame([
    {
        "model": "Baseline",
        "validation_loss": baseline_validation["loss"],
        "validation_accuracy": baseline_validation["accuracy"] * 100
    },
    {
        "model": "Weighted",
        "validation_loss": weighted_validation["loss"],
        "validation_accuracy": weighted_validation["accuracy"] * 100
    }
])

print(validation_df.to_string(index=False))


# ------------------------------------------------------------
# 3. GENERATION PERFORMANCE
# ------------------------------------------------------------

generation_path = os.path.join(
    OUTPUT_DIR,
    "cell200_generation_comparison.csv"
)

generation_df = pd.read_csv(generation_path)

generation_summary = []

for model_name in ["Baseline", "Weighted"]:

    subset = generation_df[
        generation_df["model"] == model_name
    ]

    valid = subset[subset["valid"] == True]

    generation_summary.append({

        "model": model_name,

        "generated": len(subset),

        "valid": len(valid),

        "validity_percent": (
            len(valid) / len(subset) * 100
        ),

        "unique_valid": (
            valid["canonical_smiles"].nunique()
        )

    })

generation_summary_df = pd.DataFrame(
    generation_summary
)

print("\n3. GENERATION PERFORMANCE")

print(generation_summary_df.to_string(index=False))


# ------------------------------------------------------------
# 4. EOS TERMINATION
# ------------------------------------------------------------

termination_path = os.path.join(
    OUTPUT_DIR,
    "cell203_termination_analysis.csv"
)

termination_df = pd.read_csv(termination_path)

print("\n4. EOS TERMINATION")

termination_summary = termination_df.groupby("model").agg(
    total=("sample_id", "count"),
    reached_eos=("reached_eos", "sum"),
    unfinished=("truncated_or_unfinished", "sum")
)

termination_summary["EOS_percent"] = (
    termination_summary["reached_eos"] /
    termination_summary["total"] * 100
)

print(termination_summary.to_string())


# ------------------------------------------------------------
# 5. MOLECULAR DIVERSITY
# ------------------------------------------------------------

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

print("\n5. MOLECULAR DIVERSITY")

diversity_results = []

for model_name in ["Baseline", "Weighted"]:

    subset = generation_df[
        (generation_df["model"] == model_name) &
        (generation_df["valid"] == True)
    ]

    unique_smiles = subset[
        "canonical_smiles"
    ].dropna().unique()

    molecules = [
        Chem.MolFromSmiles(s)
        for s in unique_smiles
    ]

    fingerprints = [
        fingerprint_generator.GetFingerprint(m)
        for m in molecules
        if m is not None
    ]

    similarities = []

    for i in range(len(fingerprints)):
        for j in range(i + 1, len(fingerprints)):

            similarities.append(
                TanimotoSimilarity(
                    fingerprints[i],
                    fingerprints[j]
                )
            )

    diversity_results.append({

        "model": model_name,

        "unique_molecules": len(fingerprints),

        "mean_pairwise_similarity": (
            np.mean(similarities)
            if similarities else np.nan
        )

    })

diversity_df = pd.DataFrame(diversity_results)

print(diversity_df.to_string(index=False))


# ------------------------------------------------------------
# 6. POCKET CONDITIONING STATUS
# ------------------------------------------------------------

print("\n6. POCKET CONDITIONING STATUS")

print("Pocket encoder implemented: Yes")
print("Cross-attention architecture implemented: Yes")
print("Pocket encoder trained: No")
print("Pocket cross-attention used during ChEMBL training: No")
print("Actual PARP1 pocket conditioning demonstrated: No")


# ------------------------------------------------------------
# 7. FINAL DECISION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. CURRENT M2 STATUS")
print("=" * 70)

print("""
COMPLETED:
- ChEMBL-37 molecular dataset preparation.
- Molecular tokenizer and Transformer.
- General molecular pretraining.
- Molecular-only fine-tuning.
- Generation and chemical-validity diagnostics.

NOT YET COMPLETED:
- Reliable molecular generation.
- Trained pocket encoder.
- Pocket-conditioned training.
- PARP1-specific generation validation.
- Final molecular graph interface to M3.

PRIMARY CHECKPOINT:
transformer_chembl37_improved_best.pt

FREQUENCY-WEIGHTED MODEL:
Experimental only; not selected.
""")

# ------------------------------------------------------------
# 8. SAVE CONSOLIDATED REPORT
# ------------------------------------------------------------

report_path = os.path.join(
    OUTPUT_DIR,
    "cell204_consolidated_model_audit.txt"
)

with open(report_path, "w") as f:

    f.write("M2 CONSOLIDATED MODEL AUDIT\n")
    f.write("=" * 50 + "\n\n")

    f.write("VALIDATION PERFORMANCE\n")
    f.write(validation_df.to_string(index=False))

    f.write("\n\nGENERATION PERFORMANCE\n")
    f.write(generation_summary_df.to_string(index=False))

    f.write("\n\nEOS TERMINATION\n")
    f.write(termination_summary.to_string())

    f.write("\n\nMOLECULAR DIVERSITY\n")
    f.write(diversity_df.to_string(index=False))

    f.write("\n\nPOCKET CONDITIONING: NOT YET TRAINED\n")

print("\nReport saved:", report_path)

print("\nCELL 204 COMPLETED")
print("=" * 70)

CELL 204: CONSOLIDATED BASELINE MODEL AUDIT

1. BASELINE CHECKPOINT
Path: /content/drive/MyDrive/Member2_Project/checkpoints/transformer_chembl37_improved_best.pt
epoch: 5
train_loss: 1.2469141580714287
validation_loss: 1.1923144488281978
training_molecules: 10000
training_type: chembl37_general_molecular_finetuning
pocket_cross_attention_used: False
Weighted experiment checkpoint exists: True

2. VALIDATION PERFORMANCE
   model  validation_loss  validation_accuracy
Baseline         1.192314            60.734431
Weighted         1.238501            60.110117

3. GENERATION PERFORMANCE
   model  generated  valid  validity_percent  unique_valid
Baseline         50      2               4.0             2
Weighted         50      2               4.0             2

4. EOS TERMINATION
          total  reached_eos  unfinished  EOS_percent
model                                                
Baseline     50           50           0        100.0
Weighted     50           48           2         

In [277]:
# ============================================================
# CELL 205: TRAINING DATA SMILES GRAMMAR ANALYSIS
# ============================================================

import os
import re
import json
import numpy as np
import pandas as pd

from collections import Counter
from rdkit import Chem

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

SMILES_PATH = os.path.join(
    OUTPUT_DIR,
    "chembl37_clean_smiles.txt"
)

INDICES_PATH = os.path.join(
    OUTPUT_DIR,
    "transformer_selected_10k_indices.npy"
)

assert os.path.exists(SMILES_PATH), "SMILES dataset missing!"
assert os.path.exists(INDICES_PATH), "Selected indices missing!"

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

with open(SMILES_PATH, "r") as f:
    all_smiles = [line.strip() for line in f if line.strip()]

selected_indices = np.load(INDICES_PATH)

selected_smiles = [
    all_smiles[i] for i in selected_indices
]

print("=" * 65)
print("CELL 205: TRAINING DATA SMILES GRAMMAR ANALYSIS")
print("=" * 65)

print("\nDataset records:", len(all_smiles))
print("Selected training molecules:", len(selected_smiles))


# ------------------------------------------------------------
# 2. DEFINE SMILES TOKEN PATTERN
# ------------------------------------------------------------

token_pattern = re.compile(
    r'\[[^\]]+\]'
    r'|Br|Cl|Si|Se|Na|Li|Mg|Ca|Al'
    r'|[A-Z][a-z]?'
    r'|[bcnops]'
    r'|%\d{2}'
    r'|\d'
    r'|@@?'
    r'|>>'
    r'|[-=#$:~@/\\.+()*]'
)

# ------------------------------------------------------------
# 3. ANALYZE EACH MOLECULE
# ------------------------------------------------------------

records = []
global_token_counts = Counter()

for idx, smiles in enumerate(selected_smiles):

    tokens = token_pattern.findall(smiles)

    # Parentheses / branches
    open_brackets = smiles.count("(")
    close_brackets = smiles.count(")")

    # Maximum branch nesting depth
    depth = 0
    max_depth = 0

    for char in smiles:

        if char == "(":
            depth += 1
            max_depth = max(max_depth, depth)

        elif char == ")":
            depth -= 1

    balanced_parentheses = (
        open_brackets == close_brackets and depth == 0
    )

    # Ring closure labels
    ring_labels = re.findall(r'%\d{2}|\d', smiles)

    ring_counts = Counter(ring_labels)

    balanced_rings = all(
        count % 2 == 0
        for count in ring_counts.values()
    )

    # Bracket atoms
    bracket_atoms = re.findall(r'\[[^\]]+\]', smiles)

    # Aromatic atoms
    aromatic_atoms = sum(
        token in ["c", "n", "o", "s", "p"]
        for token in tokens
    )

    # Stereochemistry
    stereochemistry_count = (
        smiles.count("@")
    )

    # RDKit validation
    mol = Chem.MolFromSmiles(smiles)

    valid = mol is not None

    # Token coverage
    reconstructed = "".join(tokens)

    records.append({

        "molecule_id": idx,

        "smiles": smiles,

        "valid_rdkit": valid,

        "smiles_length": len(smiles),

        "token_count": len(tokens),

        "token_coverage": (
            len(reconstructed) / len(smiles)
            if smiles else 0
        ),

        "branch_count": open_brackets,

        "maximum_branch_depth": max_depth,

        "balanced_parentheses": balanced_parentheses,

        "ring_label_occurrences": len(ring_labels),

        "unique_ring_labels": len(ring_counts),

        "balanced_ring_labels": balanced_rings,

        "bracket_atom_count": len(bracket_atoms),

        "aromatic_atom_count": aromatic_atoms,

        "stereochemistry_symbols": stereochemistry_count

    })

    global_token_counts.update(tokens)


# ------------------------------------------------------------
# 4. CREATE DATAFRAME
# ------------------------------------------------------------

grammar_df = pd.DataFrame(records)

print("\n4. DATASET VALIDATION")

print("Total molecules:", len(grammar_df))

print(
    "RDKit valid:",
    grammar_df["valid_rdkit"].sum()
)

print(
    "Invalid:",
    (~grammar_df["valid_rdkit"]).sum()
)

print(
    "Parentheses balanced:",
    grammar_df["balanced_parentheses"].sum()
)

print(
    "Ring labels balanced:",
    grammar_df["balanced_ring_labels"].sum()
)

print(
    "Complete token coverage:",
    (grammar_df["token_coverage"] == 1).sum()
)


# ------------------------------------------------------------
# 5. SEQUENCE COMPLEXITY
# ------------------------------------------------------------

print("\n5. SMILES SEQUENCE COMPLEXITY")

complexity_columns = [
    "smiles_length",
    "token_count",
    "branch_count",
    "maximum_branch_depth",
    "ring_label_occurrences",
    "unique_ring_labels",
    "bracket_atom_count",
    "aromatic_atom_count",
    "stereochemistry_symbols"
]

print(
    grammar_df[complexity_columns].describe().round(2)
)


# ------------------------------------------------------------
# 6. TOKEN FREQUENCY
# ------------------------------------------------------------

print("\n6. MOST FREQUENT SMILES TOKENS")

token_df = pd.DataFrame(
    global_token_counts.most_common(30),
    columns=["token", "frequency"]
)

token_df["percentage"] = (
    token_df["frequency"] /
    sum(global_token_counts.values()) * 100
)

print(token_df.to_string(index=False))


# ------------------------------------------------------------
# 7. BRANCH COMPLEXITY DISTRIBUTION
# ------------------------------------------------------------

print("\n7. BRANCH COMPLEXITY DISTRIBUTION")

branch_distribution = (
    grammar_df["maximum_branch_depth"]
    .value_counts()
    .sort_index()
)

print(branch_distribution.to_string())


# ------------------------------------------------------------
# 8. RING COMPLEXITY DISTRIBUTION
# ------------------------------------------------------------

print("\n8. RING COMPLEXITY DISTRIBUTION")

ring_distribution = (
    grammar_df["unique_ring_labels"]
    .value_counts()
    .sort_index()
)

print(ring_distribution.to_string())


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

grammar_path = os.path.join(
    OUTPUT_DIR,
    "cell205_training_grammar_analysis.csv"
)

token_path = os.path.join(
    OUTPUT_DIR,
    "cell205_training_token_frequency.csv"
)

grammar_df.to_csv(grammar_path, index=False)
token_df.to_csv(token_path, index=False)

print("\n9. FILES SAVED")

print(grammar_path)
print(token_path)


# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 205 COMPLETED")
print("=" * 65)

print("Training molecules analyzed:", len(grammar_df))

print(
    "RDKit validity:",
    round(grammar_df["valid_rdkit"].mean() * 100, 2),
    "%"
)

print(
    "Mean SMILES length:",
    round(grammar_df["smiles_length"].mean(), 2)
)

print(
    "Mean branch depth:",
    round(grammar_df["maximum_branch_depth"].mean(), 2)
)

print(
    "Mean unique ring labels:",
    round(grammar_df["unique_ring_labels"].mean(), 2)
)

CELL 205: TRAINING DATA SMILES GRAMMAR ANALYSIS

Dataset records: 50000
Selected training molecules: 10000

4. DATASET VALIDATION
Total molecules: 10000
RDKit valid: 10000
Invalid: 0
Parentheses balanced: 10000
Ring labels balanced: 9995
Complete token coverage: 10000

5. SMILES SEQUENCE COMPLEXITY
       smiles_length  token_count  branch_count  maximum_branch_depth  \
count       10000.00     10000.00      10000.00              10000.00   
mean           54.29        48.53          5.15                  2.33   
std            21.37        16.15          2.78                  0.96   
min             8.00         7.00          0.00                  0.00   
25%            41.00        37.00          3.00                  2.00   
50%            50.00        47.00          5.00                  2.00   
75%            63.00        57.00          6.00                  3.00   
max           238.00       131.00         23.00                  8.00   

       ring_label_occurrences  unique_ring

In [279]:
# ============================================================
# CELL 206: CONTROLLED GENERATION STRATEGY EXPERIMENT
# ============================================================

import os
import json
import torch
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# 1. LOAD BASELINE MODEL
# ------------------------------------------------------------

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "transformer_chembl37_improved_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

assert "molecular_transformer" in globals(), (
    "molecular_transformer is not available. "
    "Run the model architecture cell first."
)

# Identify model weights in checkpoint
if "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
elif "model_state" in checkpoint:
    state_dict = checkpoint["model_state"]
else:
    raise KeyError(
        "Could not identify model weights. "
        f"Checkpoint keys: {list(checkpoint.keys())}"
    )

molecular_transformer.load_state_dict(state_dict)
molecular_transformer.to(DEVICE)
molecular_transformer.eval()

print("Baseline checkpoint loaded successfully.")


# ------------------------------------------------------------
# 2. LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    OUTPUT_DIR,
    "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]

id_to_token = {
    int(value): key
    for key, value in token_to_id.items()
}

# Special token IDs verified in Cell 199

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

VOCAB_SIZE = len(token_to_id)
MAX_LENGTH = tokenizer_data["max_length"]

print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)
print("UNK:", UNK_ID)
print("Vocabulary size:", VOCAB_SIZE)
print("Maximum length:", MAX_LENGTH)

VOCAB_SIZE = len(token_to_id)
MAX_LENGTH = tokenizer_data["max_length"]

print("Vocabulary size:", VOCAB_SIZE)
print("Maximum sequence length:", MAX_LENGTH)


# ------------------------------------------------------------
# 3. GENERATION CONFIGURATIONS
# ------------------------------------------------------------

configurations = [
    {"name": "A", "temperature": 0.7, "top_k": 10},
    {"name": "B", "temperature": 0.8, "top_k": 20},
    {"name": "C", "temperature": 1.0, "top_k": 40},
    {"name": "D", "temperature": 1.0, "top_k": 81}
]

SAMPLES_PER_CONFIG = 30

# ------------------------------------------------------------
# 4. GENERATION FUNCTION
# ------------------------------------------------------------

def generate_smiles(temperature, top_k, seed):

    torch.manual_seed(seed)
    np.random.seed(seed)

    input_ids = torch.tensor(
        [[BOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    generated_ids = []

    with torch.no_grad():

        for step in range(MAX_LENGTH - 1):

            logits = molecular_only_forward(
                molecular_transformer,
                input_ids
            )

            next_logits = logits[:, -1, :]

            # Avoid generating PAD or BOS
            next_logits[:, PAD_ID] = -float("inf")
            next_logits[:, BOS_ID] = -float("inf")

            next_logits = next_logits / temperature

            # Top-k filtering
            if top_k < VOCAB_SIZE:

                values, indices = torch.topk(
                    next_logits,
                    top_k,
                    dim=-1
                )

                filtered_logits = torch.full_like(
                    next_logits,
                    -float("inf")
                )

                filtered_logits.scatter_(
                    1,
                    indices,
                    values
                )

                next_logits = filtered_logits

            probabilities = torch.softmax(
                next_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            token_id = next_token.item()

            if token_id == EOS_ID:
                break

            generated_ids.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    smiles = "".join(
        id_to_token.get(token_id, "")
        for token_id in generated_ids
    )

    return smiles, len(generated_ids), (
        token_id == EOS_ID if generated_ids or token_id == EOS_ID
        else False
    )


# ------------------------------------------------------------
# 5. GENERATE MOLECULES
# ------------------------------------------------------------

results = []

print("\nStarting controlled generation...")

for config_index, config in enumerate(configurations):

    print(
        f"\nConfiguration {config['name']}: "
        f"Temperature={config['temperature']}, "
        f"Top-k={config['top_k']}"
    )

    for sample_id in range(SAMPLES_PER_CONFIG):

        smiles, token_length, reached_eos = generate_smiles(
            temperature=config["temperature"],
            top_k=config["top_k"],
            seed=1000 + config_index * 100 + sample_id
        )

        mol = Chem.MolFromSmiles(smiles)

        valid = mol is not None

        canonical_smiles = (
            Chem.MolToSmiles(mol)
            if valid else None
        )

        results.append({

            "configuration": config["name"],

            "temperature": config["temperature"],

            "top_k": config["top_k"],

            "sample_id": sample_id + 1,

            "smiles": smiles,

            "valid": valid,

            "canonical_smiles": canonical_smiles,

            "token_length": token_length,

            "reached_eos": reached_eos,

            "molecular_weight": (
                Descriptors.MolWt(mol) if valid else np.nan
            ),

            "logP": (
                Descriptors.MolLogP(mol) if valid else np.nan
            ),

            "TPSA": (
                rdMolDescriptors.CalcTPSA(mol) if valid else np.nan
            )

        })

    print("Completed:", config["name"])


# ------------------------------------------------------------
# 6. GENERATION RESULTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

summary = results_df.groupby("configuration").agg(

    generated=("sample_id", "count"),

    valid=("valid", "sum"),

    unique_valid=(
        "canonical_smiles",
        "nunique"
    ),

    EOS_count=("reached_eos", "sum"),

    mean_token_length=("token_length", "mean"),

    mean_MW=("molecular_weight", "mean"),

    mean_LogP=("logP", "mean"),

    mean_TPSA=("TPSA", "mean")

).reset_index()

summary["validity_percent"] = (
    summary["valid"] /
    summary["generated"] * 100
)

summary["EOS_percent"] = (
    summary["EOS_count"] /
    summary["generated"] * 100
)


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENERATION STRATEGY COMPARISON")
print("=" * 70)

print(summary.round(3).to_string(index=False))


# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

results_path = os.path.join(
    OUTPUT_DIR,
    "cell206_generation_strategy_results.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "cell206_generation_strategy_summary.csv"
)

results_df.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)

print("\nSaved:")
print(results_path)
print(summary_path)

print("\nCELL 206 COMPLETED")

Baseline checkpoint loaded successfully.
PAD: 0
BOS: 1
EOS: 2
UNK: 3
Vocabulary size: 81
Maximum length: 143
Vocabulary size: 81
Maximum sequence length: 143

Starting controlled generation...

Configuration A: Temperature=0.7, Top-k=10
Completed: A

Configuration B: Temperature=0.8, Top-k=20
Completed: B

Configuration C: Temperature=1.0, Top-k=40
Completed: C

Configuration D: Temperature=1.0, Top-k=81
Completed: D

GENERATION STRATEGY COMPARISON
configuration  generated  valid  unique_valid  EOS_count  mean_token_length  mean_MW  mean_LogP  mean_TPSA  validity_percent  EOS_percent
            A         30      1             1         30             47.167  290.366      3.800      41.13             3.333        100.0
            B         30      1             1         30             44.367  284.315      2.181      69.41             3.333        100.0
            C         30      1             1         30             50.100  327.336      0.604      91.93             3.333        1

In [280]:
# ============================================================
# CELL 207: GENERATION FAILURE ANALYSIS
# ============================================================

import os
import re
import pandas as pd
import numpy as np

from collections import Counter
from rdkit import Chem

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

INPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "cell206_generation_strategy_results.csv"
)

assert os.path.exists(INPUT_PATH), "Cell 206 results missing!"

df = pd.read_csv(INPUT_PATH)

# ------------------------------------------------------------
# 1. ANALYZE EACH GENERATED SMILES
# ------------------------------------------------------------

def analyze_smiles(smiles):

    smiles = str(smiles)

    # Parentheses
    open_count = smiles.count("(")
    close_count = smiles.count(")")

    depth = 0
    minimum_depth = 0

    for char in smiles:
        if char == "(":
            depth += 1
        elif char == ")":
            depth -= 1

        minimum_depth = min(minimum_depth, depth)

    balanced_parentheses = (
        open_count == close_count
        and depth == 0
        and minimum_depth >= 0
    )

    # Ring closures
    ring_labels = re.findall(r"%\d{2}|\d", smiles)
    ring_counts = Counter(ring_labels)

    balanced_rings = all(
        count % 2 == 0
        for count in ring_counts.values()
    )

    # Bracket balance
    balanced_brackets = (
        smiles.count("[") == smiles.count("]")
    )

    # RDKit parsing
    mol = Chem.MolFromSmiles(
        smiles,
        sanitize=False
    )

    parse_success = mol is not None

    sanitize_success = False

    if parse_success:
        try:
            Chem.SanitizeMol(mol)
            sanitize_success = True
        except Exception:
            sanitize_success = False

    if sanitize_success:
        failure_stage = "Valid"

    elif not parse_success:
        failure_stage = "Parsing failure"

    else:
        failure_stage = "Sanitization failure"

    return {

        "balanced_parentheses": balanced_parentheses,

        "balanced_rings": balanced_rings,

        "balanced_brackets": balanced_brackets,

        "parse_success": parse_success,

        "sanitize_success": sanitize_success,

        "failure_stage": failure_stage

    }


diagnostics = df["smiles"].apply(analyze_smiles)

diagnostics_df = pd.DataFrame(
    diagnostics.tolist()
)

df = pd.concat(
    [df.reset_index(drop=True),
     diagnostics_df.reset_index(drop=True)],
    axis=1
)


# ------------------------------------------------------------
# 2. FAILURE SUMMARY BY CONFIGURATION
# ------------------------------------------------------------

print("=" * 70)
print("CELL 207: GENERATION FAILURE ANALYSIS")
print("=" * 70)

summary = df.groupby("configuration").agg(

    total=("sample_id", "count"),

    valid=("sanitize_success", "sum"),

    parse_success=("parse_success", "sum"),

    balanced_parentheses=("balanced_parentheses", "sum"),

    balanced_rings=("balanced_rings", "sum"),

    balanced_brackets=("balanced_brackets", "sum"),

    EOS_count=("reached_eos", "sum")

).reset_index()

summary["parsing_failures"] = (
    summary["total"] - summary["parse_success"]
)

summary["sanitization_failures"] = (
    summary["parse_success"] - summary["valid"]
)

summary["validity_percent"] = (
    summary["valid"] / summary["total"] * 100
)

print("\n1. FAILURE SUMMARY")

print(summary.to_string(index=False))


# ------------------------------------------------------------
# 3. FAILURE STAGE DISTRIBUTION
# ------------------------------------------------------------

print("\n2. FAILURE STAGES")

failure_distribution = pd.crosstab(
    df["configuration"],
    df["failure_stage"]
)

print(failure_distribution.to_string())


# ------------------------------------------------------------
# 4. INVALID MOLECULE EXAMPLES
# ------------------------------------------------------------

print("\n3. INVALID MOLECULE EXAMPLES")

invalid_df = df[
    df["sanitize_success"] == False
]

for config in ["A", "B", "C", "D"]:

    subset = invalid_df[
        invalid_df["configuration"] == config
    ].head(3)

    print(f"\nConfiguration {config}")

    for _, row in subset.iterrows():

        print("\nSMILES:", row["smiles"])
        print("Failure:", row["failure_stage"])
        print(
            "Parentheses balanced:",
            row["balanced_parentheses"]
        )
        print(
            "Ring labels balanced:",
            row["balanced_rings"]
        )
        print(
            "Brackets balanced:",
            row["balanced_brackets"]
        )


# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

diagnostic_path = os.path.join(
    OUTPUT_DIR,
    "cell207_generation_failure_diagnostics.csv"
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "cell207_generation_failure_summary.csv"
)

df.to_csv(diagnostic_path, index=False)
summary.to_csv(summary_path, index=False)

print("\n4. FILES SAVED")

print(diagnostic_path)
print(summary_path)

print("\nCELL 207 COMPLETED")
print("=" * 70)

CELL 207: GENERATION FAILURE ANALYSIS

1. FAILURE SUMMARY
configuration  total  valid  parse_success  balanced_parentheses  balanced_rings  balanced_brackets  EOS_count  parsing_failures  sanitization_failures  validity_percent
            A     30      1              7                    17               9                 30         30                23                      6          3.333333
            B     30      1              7                    18               8                 30         30                23                      6          3.333333
            C     30      1              3                    18               6                 30         30                27                      2          3.333333
            D     30      1              2                    19               5                 30         30                28                      1          3.333333

2. FAILURE STAGES
failure_stage  Parsing failure  Sanitization failure  Valid
configuration

In [281]:
# ============================================================
# CELL 208: DETAILED SMILES SYNTAX DIAGNOSIS
# ============================================================

import os
import re
import pandas as pd
from collections import Counter
from rdkit import Chem

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

INPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "cell206_generation_strategy_results.csv"
)

df = pd.read_csv(INPUT_PATH)


def diagnose_smiles(smiles):

    smiles = str(smiles)

    # Remove bracket atoms before checking ring digits.
    # This prevents isotope/stereochemical bracket contents
    # from being mistaken for ring labels.
    without_brackets = re.sub(
        r'\[[^\]]*\]',
        '',
        smiles
    )

    # Parenthesis balance
    depth = 0
    min_depth = 0

    for char in without_brackets:
        if char == "(":
            depth += 1
        elif char == ")":
            depth -= 1

        min_depth = min(min_depth, depth)

    parentheses_valid = (
        depth == 0 and min_depth >= 0
    )

    # Ring closure analysis
    ring_labels = re.findall(
        r'%\d{2}|\d',
        without_brackets
    )

    ring_counts = Counter(ring_labels)

    unmatched_rings = [
        label for label, count in ring_counts.items()
        if count % 2 != 0
    ]

    rings_valid = len(unmatched_rings) == 0

    # Bracket balance
    brackets_valid = (
        smiles.count("[") == smiles.count("]")
        and re.search(r'\[[^\]]*\]', smiles) is not None
        if "[" in smiles or "]" in smiles
        else True
    )

    # Backslash and slash positions
    suspicious_backslash = bool(
        re.search(r'\\\s*[\(\)]', smiles)
    )

    # Unpaired slash/backslash indicators
    directional_bonds = (
        smiles.count("/") + smiles.count("\\")
    )

    # RDKit parsing
    mol = Chem.MolFromSmiles(
        smiles,
        sanitize=False
    )

    parse_success = mol is not None

    sanitize_success = False

    if parse_success:
        try:
            Chem.SanitizeMol(mol)
            sanitize_success = True
        except Exception:
            pass

    if sanitize_success:
        failure_reason = "Valid"
    elif not parse_success:
        failure_reason = "Parsing failure"
    else:
        failure_reason = "Sanitization failure"

    return {

        "parentheses_valid": parentheses_valid,

        "rings_valid": rings_valid,

        "unmatched_ring_labels": ",".join(unmatched_rings),

        "brackets_valid": brackets_valid,

        "suspicious_backslash": suspicious_backslash,

        "directional_bond_symbols": directional_bonds,

        "parse_success": parse_success,

        "sanitize_success": sanitize_success,

        "failure_reason": failure_reason

    }


# ------------------------------------------------------------
# APPLY DIAGNOSTICS
# ------------------------------------------------------------

diagnostics = pd.DataFrame(
    df["smiles"].apply(diagnose_smiles).tolist()
)

df = pd.concat(
    [df.reset_index(drop=True), diagnostics],
    axis=1
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("CELL 208: DETAILED SMILES SYNTAX DIAGNOSIS")
print("=" * 70)

print("\n1. OVERALL SYNTAX COUNTS")

columns = [
    "parentheses_valid",
    "rings_valid",
    "brackets_valid",
    "suspicious_backslash",
    "parse_success",
    "sanitize_success"
]

print(df[columns].sum().to_string())


# ------------------------------------------------------------
# FAILURE REASON BY CONFIGURATION
# ------------------------------------------------------------

print("\n2. FAILURE REASONS")

print(
    pd.crosstab(
        df["configuration"],
        df["failure_reason"]
    ).to_string()
)


# ------------------------------------------------------------
# SPECIFIC BACKSLASH EXAMPLES
# ------------------------------------------------------------

print("\n3. SUSPICIOUS BACKSLASH EXAMPLES")

backslash_df = df[
    df["suspicious_backslash"]
]

if len(backslash_df) == 0:
    print("No suspicious backslash placement found.")
else:
    for _, row in backslash_df.head(10).iterrows():
        print(
            f"\nConfiguration: {row['configuration']}"
        )
        print("SMILES:", row["smiles"])
        print("Failure:", row["failure_reason"])


# ------------------------------------------------------------
# UNMATCHED RING EXAMPLES
# ------------------------------------------------------------

print("\n4. UNMATCHED RING LABEL EXAMPLES")

ring_df = df[
    ~df["rings_valid"]
]

for _, row in ring_df.head(10).iterrows():

    print(
        f"\nConfiguration: {row['configuration']}"
    )

    print("SMILES:", row["smiles"])

    print(
        "Unmatched labels:",
        row["unmatched_ring_labels"]
    )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output_path = os.path.join(
    OUTPUT_DIR,
    "cell208_detailed_smiles_syntax_diagnosis.csv"
)

df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\nCELL 208 COMPLETED")
print("=" * 70)

CELL 208: DETAILED SMILES SYNTAX DIAGNOSIS

1. OVERALL SYNTAX COUNTS
parentheses_valid        72
rings_valid              28
brackets_valid          120
suspicious_backslash      0
parse_success            19
sanitize_success          4

2. FAILURE REASONS
failure_reason  Parsing failure  Sanitization failure  Valid
configuration                                               
A                            23                     6      1
B                            23                     6      1
C                            27                     2      1
D                            28                     1      1

3. SUSPICIOUS BACKSLASH EXAMPLES
No suspicious backslash placement found.

4. UNMATCHED RING LABEL EXAMPLES

Configuration: A
SMILES: O=C1C[C@H]2CN1CCN(C(O)c2cccc3ccccc2O)c1OC(=O)O
Unmatched labels: 1,2,3

Configuration: A
SMILES: Cc1cc(CC(=O)NC(=O)Cc2cc(C(=O)O)cc2OC)ccc2OCC(C)(=O)NC)NC1
Unmatched labels: 2

Configuration: A
SMILES: CC[C@H](C)N1C[C@@H]2C(CO)N1C(=O)CC(Cc3ccc

In [282]:
# ============================================================
# CELL 209: TOKENIZER VOCABULARY INTEGRITY AUDIT
# ============================================================

import os
import json
from rdkit import Chem
from rdkit.Chem import GetPeriodicTable

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

TOKENIZER_PATH = os.path.join(
    OUTPUT_DIR,
    "smiles_tokenizer.json"
)

with open(TOKENIZER_PATH, "r") as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]

tokens = list(token_to_id.keys())

periodic_table = GetPeriodicTable()

valid_elements = {
    periodic_table.GetElementSymbol(i)
    for i in range(1, 119)
}

# ------------------------------------------------------------
# 1. BASIC INFORMATION
# ------------------------------------------------------------

print("=" * 70)
print("CELL 209: TOKENIZER VOCABULARY INTEGRITY AUDIT")
print("=" * 70)

print("\nVocabulary size:", len(tokens))

print("\nSpecial tokens:")

for token, token_id in token_to_id.items():
    if token in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]:
        print(token, ":", token_id)


# ------------------------------------------------------------
# 2. IDENTIFY SUSPICIOUS TWO-CHARACTER TOKENS
# ------------------------------------------------------------

suspicious = []

for token in tokens:

    if len(token) == 2:

        if (
            token[0].isupper()
            and token[1].islower()
            and token not in valid_elements
        ):
            suspicious.append(token)

print("\n1. SUSPICIOUS TWO-CHARACTER TOKENS")

print("Count:", len(suspicious))

print(suspicious)


# ------------------------------------------------------------
# 3. IDENTIFY VALID ELEMENT TOKENS
# ------------------------------------------------------------

element_tokens = [
    token for token in tokens
    if token in valid_elements
]

print("\n2. VALID ELEMENT SYMBOL TOKENS")

print(element_tokens)


# ------------------------------------------------------------
# 4. IDENTIFY IMPORTANT SMILES TOKENS
# ------------------------------------------------------------

important_tokens = [
    "C", "c", "N", "n", "O", "o",
    "S", "s", "P", "p",
    "F", "Cl", "Br", "I",
    "(", ")", "[", "]",
    "1", "2", "3", "4",
    "=", "#", "-", "+",
    "/", "\\", "@", "@@"
]

print("\n3. IMPORTANT SMILES TOKEN COVERAGE")

for token in important_tokens:

    print(
        f"{token:5s}",
        "PRESENT" if token in token_to_id else "MISSING"
    )


# ------------------------------------------------------------
# 5. PRINT COMPLETE VOCABULARY
# ------------------------------------------------------------

print("\n4. COMPLETE VOCABULARY")

for token, token_id in sorted(
    token_to_id.items(),
    key=lambda item: item[1]
):

    print(token_id, repr(token))


# ------------------------------------------------------------
# 6. SAVE AUDIT
# ------------------------------------------------------------

audit_path = os.path.join(
    OUTPUT_DIR,
    "cell209_tokenizer_integrity_audit.txt"
)

with open(audit_path, "w") as f:

    f.write("TOKENIZER VOCABULARY INTEGRITY AUDIT\n")
    f.write("=" * 50 + "\n")

    f.write(f"\nVocabulary size: {len(tokens)}\n")

    f.write("\nSuspicious tokens:\n")
    f.write(str(suspicious))

    f.write("\n\nComplete vocabulary:\n")

    for token, token_id in sorted(
        token_to_id.items(),
        key=lambda item: item[1]
    ):

        f.write(f"{token_id}: {repr(token)}\n")

print("\nAudit saved:", audit_path)

print("\nCELL 209 COMPLETED")
print("=" * 70)

CELL 209: TOKENIZER VOCABULARY INTEGRITY AUDIT

Vocabulary size: 81

Special tokens:
<PAD> : 0
<BOS> : 1
<EOS> : 2
<UNK> : 3

1. SUSPICIOUS TWO-CHARACTER TOKENS
Count: 0
[]

2. VALID ELEMENT SYMBOL TOKENS
['B', 'Br', 'C', 'Cl', 'F', 'I', 'N', 'O', 'P', 'S']

3. IMPORTANT SMILES TOKEN COVERAGE
C     PRESENT
c     PRESENT
N     PRESENT
n     PRESENT
O     PRESENT
o     PRESENT
S     PRESENT
s     PRESENT
P     PRESENT
p     MISSING
F     PRESENT
Cl    PRESENT
Br    PRESENT
I     PRESENT
(     PRESENT
)     PRESENT
[     MISSING
]     MISSING
1     PRESENT
2     PRESENT
3     PRESENT
4     PRESENT
=     PRESENT
#     PRESENT
-     PRESENT
+     MISSING
/     PRESENT
\     PRESENT
@     MISSING
@@    MISSING

4. COMPLETE VOCABULARY
0 '<PAD>'
1 '<BOS>'
2 '<EOS>'
3 '<UNK>'
4 '#'
5 '%10'
6 '('
7 ')'
8 '-'
9 '/'
10 '1'
11 '2'
12 '3'
13 '4'
14 '5'
15 '6'
16 '7'
17 '8'
18 '9'
19 '='
20 'B'
21 'Br'
22 'C'
23 'Cl'
24 'F'
25 'I'
26 'N'
27 'O'
28 'P'
29 'S'
30 '[10B]'
31 '[11CH3]'
32 '[11C]'
33 '[12

In [283]:
# ============================================================
# CELL 210: ACTUAL TOKENIZER ROUND-TRIP VALIDATION
# ============================================================

import os
import json
import re
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "member2_outputs")

# ------------------------------------------------------------
# 1. LOAD TOKENIZER AND DATA
# ------------------------------------------------------------

with open(
    os.path.join(OUTPUT_DIR, "smiles_tokenizer.json"), "r"
) as f:
    tokenizer_data = json.load(f)

token_to_id = tokenizer_data["token_to_id"]
max_length = tokenizer_data["max_length"]

with open(
    os.path.join(OUTPUT_DIR, "chembl37_clean_smiles.txt"), "r"
) as f:
    all_smiles = [x.strip() for x in f if x.strip()]

selected_indices = np.load(
    os.path.join(
        OUTPUT_DIR,
        "transformer_selected_10k_indices.npy"
    )
)

selected_smiles = [
    all_smiles[i] for i in selected_indices
]

print("=" * 70)
print("CELL 210: TOKENIZER ROUND-TRIP VALIDATION")
print("=" * 70)

print("Selected molecules:", len(selected_smiles))
print("Vocabulary size:", len(token_to_id))
print("Maximum sequence length:", max_length)


# ------------------------------------------------------------
# 2. BUILD VOCABULARY MATCHER
# ------------------------------------------------------------

special = {"<PAD>", "<BOS>", "<EOS>", "<UNK>"}

chemical_tokens = [
    token for token in token_to_id
    if token not in special
]

# Longest-match-first prevents shorter tokens
# from consuming a longer valid token.

chemical_tokens.sort(key=len, reverse=True)

pattern = re.compile(
    "|".join(
        re.escape(token)
        for token in chemical_tokens
    )
)


# ------------------------------------------------------------
# 3. TOKENIZE AND RECONSTRUCT
# ------------------------------------------------------------

records = []

unknown_examples = []

for index, smiles in enumerate(selected_smiles):

    tokens = []
    position = 0
    unknown_count = 0

    while position < len(smiles):

        match = pattern.match(smiles, position)

        if match:

            token = match.group()

            tokens.append(token)

            position = match.end()

        else:

            unknown_count += 1

            if len(unknown_examples) < 20:
                unknown_examples.append({
                    "molecule_id": index,
                    "character": smiles[position],
                    "position": position,
                    "smiles": smiles
                })

            tokens.append("<UNK>")

            position += 1

    reconstructed = "".join(
        token if token != "<UNK>" else ""
        for token in tokens
    )

    exact_reconstruction = (
        reconstructed == smiles
        and unknown_count == 0
    )

    records.append({

        "molecule_id": index,

        "smiles": smiles,

        "token_count": len(tokens),

        "sequence_length_with_special": len(tokens) + 2,

        "unknown_count": unknown_count,

        "exact_reconstruction": exact_reconstruction,

        "within_max_length": (
            len(tokens) + 2 <= max_length
        )

    })


# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------

result_df = pd.DataFrame(records)

print("\n1. ROUND-TRIP RESULTS")

print("Total molecules:", len(result_df))

print(
    "Exact reconstruction:",
    result_df["exact_reconstruction"].sum()
)

print(
    "Unknown-token molecules:",
    (result_df["unknown_count"] > 0).sum()
)

print(
    "Total unknown occurrences:",
    result_df["unknown_count"].sum()
)

print(
    "Within maximum length:",
    result_df["within_max_length"].sum()
)

print("\n2. TOKEN LENGTH DISTRIBUTION")

print(
    result_df["token_count"].describe().round(2)
)


# ------------------------------------------------------------
# 5. UNKNOWN TOKEN EXAMPLES
# ------------------------------------------------------------

print("\n3. UNKNOWN TOKEN EXAMPLES")

if unknown_examples:

    for item in unknown_examples[:10]:
        print(item)

else:

    print("No unknown tokens found.")


# ------------------------------------------------------------
# 6. OVERLENGTH MOLECULES
# ------------------------------------------------------------

print("\n4. OVERLENGTH MOLECULES")

overlength = result_df[
    ~result_df["within_max_length"]
]

print("Count:", len(overlength))

if len(overlength):
    print(
        overlength[
            ["molecule_id", "token_count", "smiles"]
        ].head(10).to_string(index=False)
    )


# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    OUTPUT_DIR,
    "cell210_tokenizer_roundtrip_audit.csv"
)

result_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\nCELL 210 COMPLETED")
print("=" * 70)

CELL 210: TOKENIZER ROUND-TRIP VALIDATION
Selected molecules: 10000
Vocabulary size: 81
Maximum sequence length: 143

1. ROUND-TRIP RESULTS
Total molecules: 10000
Exact reconstruction: 10000
Unknown-token molecules: 0
Total unknown occurrences: 0
Within maximum length: 10000

2. TOKEN LENGTH DISTRIBUTION
count    10000.00
mean        49.65
std         16.26
min          7.00
25%         38.00
50%         48.00
75%         58.00
max        131.00
Name: token_count, dtype: float64

3. UNKNOWN TOKEN EXAMPLES
No unknown tokens found.

4. OVERLENGTH MOLECULES
Count: 0

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/cell210_tokenizer_roundtrip_audit.csv

CELL 210 COMPLETED


In [284]:
# ============================================================
# CELL 211: ORIGINAL TOKENIZER IMPLEMENTATION AUDIT
# ============================================================

import inspect
import json
import os

print("=" * 70)
print("CELL 211: ORIGINAL TOKENIZER IMPLEMENTATION AUDIT")
print("=" * 70)

# 1. Load tokenizer configuration
PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

TOKENIZER_PATH = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "smiles_tokenizer.json"
)

with open(TOKENIZER_PATH, "r") as f:
    tokenizer_config = json.load(f)

print("\n1. TOKENIZER CONFIGURATION")
print("Vocabulary size:", len(tokenizer_config["vocabulary"]))
print("Maximum length:", tokenizer_config["max_length"])
print("Special tokens:", tokenizer_config["special_tokens"])

# 2. Find tokenizer-related objects in the current runtime
print("\n2. TOKENIZER-RELATED RUNTIME OBJECTS")

found_objects = []

for name, obj in list(globals().items()):
    if any(key in name.lower() for key in [
        "tokenizer", "tokenize", "encode", "decode", "smiles"
    ]):
        if name.startswith("_"):
            continue

        obj_type = type(obj).__name__

        print(f"\nName: {name}")
        print("Type:", obj_type)

        if callable(obj):
            found_objects.append((name, obj))

            try:
                print("Signature:", inspect.signature(obj))
            except Exception:
                print("Signature: unavailable")

            try:
                source = inspect.getsource(obj)
                print("\nSource code:")
                print(source[:5000])
            except Exception:
                print("Source code unavailable.")

        else:
            print("Object is not directly callable.")

# 3. Check common tokenizer methods on discovered objects
print("\n3. ENCODE / DECODE METHOD CHECK")

for name, obj in list(globals().items()):
    if "tokenizer" not in name.lower():
        continue

    print(f"\nObject: {name}")

    for method_name in ["encode", "decode", "tokenize", "detokenize"]:
        method = getattr(obj, method_name, None)

        if callable(method):
            print(f"\nMethod: {method_name}")

            try:
                print("Signature:", inspect.signature(method))
            except Exception:
                print("Signature unavailable")

            try:
                print(inspect.getsource(method)[:3000])
            except Exception:
                print("Method source unavailable.")

print("\n" + "=" * 70)
print("CELL 211 COMPLETED")
print("=" * 70)

CELL 211: ORIGINAL TOKENIZER IMPLEMENTATION AUDIT

1. TOKENIZER CONFIGURATION
Vocabulary size: 81
Maximum length: 143
Special tokens: ['<PAD>', '<BOS>', '<EOS>', '<UNK>']

2. TOKENIZER-RELATED RUNTIME OBJECTS

Name: choose_smiles
Type: function
Signature: (record)

Source code:
def choose_smiles(record):
    descriptors = record.get(
        "pdbx_chem_comp_descriptor", []
    )

    candidates = []

    for item in descriptors:
        descriptor_type = str(item.get("type", "")).upper()
        program = str(item.get("program", "")).upper()
        smiles = item.get("descriptor")

        if not smiles:
            continue

        if "SMILES" not in descriptor_type:
            continue

        score = 0

        if "CANONICAL" in descriptor_type:
            score += 10

        if "STEREO" in descriptor_type:
            score += 5

        if "OPENEYE" in program:
            score += 3

        if "CACTVS" in program:
            score += 2

        candidates.append((score, sm

In [285]:
# ============================================================
# CELL 212: ORIGINAL TOKENIZER VS LONGEST-MATCH VALIDATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 212: ORIGINAL TOKENIZER VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load selected molecules
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/Member2_Project"

SMILES_PATH = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "chembl37_clean_smiles.txt"
)

INDICES_PATH = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "transformer_selected_10k_indices.npy"
)

with open(SMILES_PATH, "r") as f:
    all_smiles = [line.strip() for line in f if line.strip()]

selected_indices = np.load(INDICES_PATH)

selected_smiles = [
    all_smiles[int(i)] for i in selected_indices
]

print("\n1. DATA")
print("Selected molecules:", len(selected_smiles))
print("Original tokenizer available:", callable(tokenize_smiles))
print("Vocabulary size:", len(token_to_id))

# ------------------------------------------------------------
# 2. Compare original and longest-match tokenization
# ------------------------------------------------------------

vocab_tokens = [
    token for token in token_to_id.keys()
    if token not in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
]

vocab_tokens = sorted(
    vocab_tokens,
    key=len,
    reverse=True
)

original_unknown = 0
longest_unknown = 0
different_tokenizations = 0
exact_original_reconstruction = 0

examples = []

for smiles in selected_smiles:

    # Original implementation
    original_tokens = tokenize_smiles(smiles)

    original_reconstructed = "".join(original_tokens)

    if original_reconstructed == smiles:
        exact_original_reconstruction += 1

    if "<UNMATCHED>" in original_tokens:
        original_unknown += 1

    # Independent longest-match implementation
    longest_tokens = []

    i = 0

    while i < len(smiles):

        matched = None

        for token in vocab_tokens:
            if smiles.startswith(token, i):
                matched = token
                break

        if matched is None:
            longest_tokens.append("<UNK>")
            longest_unknown += 1
            i += 1
        else:
            longest_tokens.append(matched)
            i += len(matched)

    if original_tokens != longest_tokens:
        different_tokenizations += 1

        if len(examples) < 5:
            examples.append({
                "smiles": smiles,
                "original_tokens": original_tokens,
                "longest_tokens": longest_tokens
            })

# ------------------------------------------------------------
# 3. Check original encode/decode consistency
# ------------------------------------------------------------

encoding_success = 0
decoding_success = 0
encoding_errors = 0
length_errors = 0

for smiles in selected_smiles:

    try:
        encoded_sequence = encode_smiles(smiles)
        encoding_success += 1

        decoded_smiles = decode_sequence(encoded_sequence)

        if decoded_smiles == smiles:
            decoding_success += 1

        if len(encoded_sequence) != MAX_LEN:
            length_errors += 1

    except Exception:
        encoding_errors += 1

# ------------------------------------------------------------
# 4. Results
# ------------------------------------------------------------

print("\n2. ORIGINAL TOKENIZER RESULTS")
print("Exact reconstruction:", exact_original_reconstruction)
print("Unmatched molecules:", original_unknown)

print("\n3. LONGEST-MATCH RESULTS")
print("Unknown occurrences:", longest_unknown)

print("\n4. TOKENIZATION COMPARISON")
print("Different tokenizations:", different_tokenizations)

print("\n5. ORIGINAL ENCODE / DECODE")
print("Encoding success:", encoding_success)
print("Encoding errors:", encoding_errors)
print("Exact decoding:", decoding_success)
print("Sequence length errors:", length_errors)

print("\n6. EXAMPLE DIFFERENCES")

if examples:
    for index, item in enumerate(examples, 1):
        print(f"\nExample {index}")
        print("SMILES:", item["smiles"])
        print("Original:", item["original_tokens"])
        print("Longest:", item["longest_tokens"])
else:
    print("No tokenization differences found.")

# ------------------------------------------------------------
# 7. Save audit
# ------------------------------------------------------------

audit = pd.DataFrame([{
    "selected_molecules": len(selected_smiles),
    "original_exact_reconstruction": exact_original_reconstruction,
    "original_unmatched_molecules": original_unknown,
    "longest_match_unknown_occurrences": longest_unknown,
    "different_tokenizations": different_tokenizations,
    "encoding_success": encoding_success,
    "encoding_errors": encoding_errors,
    "exact_decoding": decoding_success,
    "sequence_length_errors": length_errors
}])

OUTPUT_PATH = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell212_original_tokenizer_audit.csv"
)

audit.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)

print("\n" + "=" * 70)
print("CELL 212 COMPLETED")
print("=" * 70)

CELL 212: ORIGINAL TOKENIZER VALIDATION

1. DATA
Selected molecules: 10000
Original tokenizer available: True
Vocabulary size: 81

2. ORIGINAL TOKENIZER RESULTS
Exact reconstruction: 10000
Unmatched molecules: 0

3. LONGEST-MATCH RESULTS
Unknown occurrences: 0

4. TOKENIZATION COMPARISON
Different tokenizations: 0

5. ORIGINAL ENCODE / DECODE
Encoding success: 10000
Encoding errors: 0
Exact decoding: 10000
Sequence length errors: 0

6. EXAMPLE DIFFERENCES
No tokenization differences found.

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/cell212_original_tokenizer_audit.csv

CELL 212 COMPLETED


In [286]:
# ============================================================
# CELL 213: TRAINING VS GENERATION FORWARD-PASS AUDIT
# ============================================================

import inspect
import torch
import torch.nn as nn

print("=" * 70)
print("CELL 213: TRAINING VS GENERATION FORWARD-PASS AUDIT")
print("=" * 70)

# 1. Inspect molecular-only forward function
print("\n1. MOLECULAR-ONLY FORWARD FUNCTION")

print(inspect.getsource(molecular_only_forward))

# 2. Inspect model forward function
print("\n2. MODEL FORWARD FUNCTION")

try:
    print(inspect.getsource(type(molecular_transformer).forward))
except Exception as e:
    print("Could not inspect model forward:", e)

# 3. Inspect Transformer decoder layer configuration
print("\n3. DECODER LAYER CONFIGURATION")

for i, layer in enumerate(molecular_transformer.decoder.layers):

    print(f"\nDecoder Layer {i + 1}")

    print("Self-attention:", type(layer.self_attn).__name__)
    print("Cross-attention:", type(layer.multihead_attn).__name__)
    print("Feed-forward 1:", type(layer.linear1).__name__)
    print("Feed-forward 2:", type(layer.linear2).__name__)
    print("norm_first:", layer.norm_first)
    print("dropout:", layer.dropout.p)

# 4. Verify model mode
print("\n4. MODEL MODE")

print("Training mode:", molecular_transformer.training)
print("Model device:", next(molecular_transformer.parameters()).device)

# 5. Compare output shapes
print("\n5. FORWARD-PASS OUTPUT SHAPES")

test_ids = torch.tensor(
    [[BOS_ID, token_to_id["C"], token_to_id["O"]]],
    dtype=torch.long,
    device=next(molecular_transformer.parameters()).device
)

molecular_transformer.eval()

with torch.no_grad():

    molecular_logits = molecular_only_forward(
        molecular_transformer,
        test_ids
    )

print("Input shape:", tuple(test_ids.shape))
print("Molecular-only logits:", tuple(molecular_logits.shape))
print("Expected vocabulary dimension:", VOCAB_SIZE)

# 6. Inspect checkpoint metadata
print("\n6. CHECKPOINT METADATA")

checkpoint_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "checkpoints/transformer_chembl37_improved_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

if isinstance(checkpoint, dict):

    for key, value in checkpoint.items():

        if key in ["model_state_dict", "state_dict"]:
            print(key, ": model weights present")

        elif isinstance(value, torch.Tensor):
            print(key, ": Tensor", tuple(value.shape))

        else:
            print(key, ":", value)

print("\n" + "=" * 70)
print("CELL 213 COMPLETED")
print("=" * 70)

CELL 213: TRAINING VS GENERATION FORWARD-PASS AUDIT

1. MOLECULAR-ONLY FORWARD FUNCTION
def molecular_only_forward(model, token_ids):

    batch_size, seq_length = token_ids.shape

    # Token and positional embeddings
    x = model.token_embedding(token_ids)
    x = model.position_encoding(x)

    # Padding mask
    padding_mask = token_ids == 0

    # Causal mask: prevent looking at future tokens
    causal_mask = torch.triu(
        torch.ones(
            seq_length,
            seq_length,
            device=token_ids.device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    # Apply molecular self-attention and feed-forward layers
    for layer in model.decoder.layers:

        if layer.norm_first:

            # Self-attention
            normalized = layer.norm1(x)

            attention_output = layer.self_attn(
                normalized,
                normalized,
                normalized,
                attn_mask=causal_mask,
                key_paddi

In [287]:
# ============================================================
# CELL 214: SPECIAL TOKEN GENERATION AUDIT
# ============================================================

import torch
import pandas as pd
from rdkit import Chem
from collections import Counter

print("=" * 70)
print("CELL 214: SPECIAL TOKEN GENERATION AUDIT")
print("=" * 70)

DEVICE = next(molecular_transformer.parameters()).device

molecular_transformer.eval()

PAD = token_to_id["<PAD>"]
BOS = token_to_id["<BOS>"]
EOS = token_to_id["<EOS>"]
UNK = token_to_id["<UNK>"]

NUM_SAMPLES = 100
TEMPERATURE = 0.8
TOP_K = 20

results = []
special_counts = Counter()

for sample_id in range(NUM_SAMPLES):

    torch.manual_seed(1000 + sample_id)

    input_ids = torch.tensor(
        [[BOS]],
        dtype=torch.long,
        device=DEVICE
    )

    generated_ids = []
    reached_eos = False

    with torch.no_grad():

        for step in range(MAX_LEN - 1):

            logits = molecular_only_forward(
                molecular_transformer,
                input_ids
            )

            next_logits = logits[:, -1, :].clone()

            next_logits[:, PAD] = -float("inf")
            next_logits[:, BOS] = -float("inf")

            next_logits = next_logits / TEMPERATURE

            k = min(TOP_K, next_logits.size(-1))

            values, indices = torch.topk(
                next_logits,
                k,
                dim=-1
            )

            probabilities = torch.softmax(values, dim=-1)

            selected = torch.multinomial(
                probabilities,
                1
            )

            next_token = indices.gather(1, selected)

            token_id = next_token.item()

            if token_id == EOS:
                reached_eos = True
                break

            generated_ids.append(token_id)

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    # Count generated special tokens
    special_counts["UNK"] += generated_ids.count(UNK)
    special_counts["PAD"] += generated_ids.count(PAD)
    special_counts["BOS"] += generated_ids.count(BOS)
    special_counts["EOS"] += int(reached_eos)

    # Raw decoding
    raw_smiles = "".join(
        id_to_token[token_id]
        for token_id in generated_ids
    )

    # Existing decoder behavior
    cleaned_smiles = decode_sequence(
        [BOS] + generated_ids + [EOS]
    )

    raw_mol = Chem.MolFromSmiles(raw_smiles)
    cleaned_mol = Chem.MolFromSmiles(cleaned_smiles)

    results.append({
        "sample_id": sample_id,
        "generated_tokens": len(generated_ids),
        "unk_count": generated_ids.count(UNK),
        "reached_eos": reached_eos,
        "raw_smiles": raw_smiles,
        "cleaned_smiles": cleaned_smiles,
        "raw_valid": raw_mol is not None,
        "cleaned_valid": cleaned_mol is not None
    })

df = pd.DataFrame(results)

print("\n1. SPECIAL TOKEN COUNTS")
for token, count in special_counts.items():
    print(f"{token}: {count}")

print("\n2. GENERATION SUMMARY")
print("Total samples:", len(df))
print("Samples containing UNK:", (df["unk_count"] > 0).sum())
print("Total UNK occurrences:", df["unk_count"].sum())
print("EOS completion:", df["reached_eos"].sum())
print("Raw valid SMILES:", df["raw_valid"].sum())
print("Cleaned valid SMILES:", df["cleaned_valid"].sum())

print("\n3. UNK EXAMPLES")

unk_examples = df[df["unk_count"] > 0]

if len(unk_examples):
    print(
        unk_examples[
            ["sample_id", "unk_count", "raw_smiles", "cleaned_smiles"]
        ].head(5).to_string(index=False)
    )
else:
    print("No UNK tokens generated.")

OUTPUT_PATH = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell214_special_token_audit.csv"
)

df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)

print("\n" + "=" * 70)
print("CELL 214 COMPLETED")
print("=" * 70)

CELL 214: SPECIAL TOKEN GENERATION AUDIT

1. SPECIAL TOKEN COUNTS
UNK: 0
PAD: 0
BOS: 0
EOS: 100

2. GENERATION SUMMARY
Total samples: 100
Samples containing UNK: 0
Total UNK occurrences: 0
EOS completion: 100
Raw valid SMILES: 5
Cleaned valid SMILES: 5

3. UNK EXAMPLES
No UNK tokens generated.

Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/cell214_special_token_audit.csv

CELL 214 COMPLETED


In [288]:
# ============================================================
# CELL 215: GENERATED SMILES FAILURE PATTERN AUDIT
# ============================================================

import os
import pandas as pd
from rdkit import Chem
from collections import Counter

print("=" * 70)
print("CELL 215: GENERATED SMILES FAILURE PATTERN AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD CELL 214 RESULTS
# ------------------------------------------------------------

audit_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

df = pd.read_csv(audit_path)

print("\n1. DATASET CHECK")
print("Total records:", len(df))
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. IDENTIFY SMILES AND VALIDITY COLUMNS
# ------------------------------------------------------------

print("\n2. AVAILABLE COLUMNS")

for col in df.columns:
    print(col, ":", df[col].dtype)

# ------------------------------------------------------------
# 3. DETECT SMILES COLUMN
# ------------------------------------------------------------

smiles_candidates = [
    col for col in df.columns
    if "smiles" in col.lower() and "clean" not in col.lower()
]

validity_candidates = [
    col for col in df.columns
    if "valid" in col.lower()
]

print("\nSMILES candidates:", smiles_candidates)
print("Validity candidates:", validity_candidates)

if not smiles_candidates:
    raise ValueError(
        "Could not identify the SMILES column. "
        "Check the printed column names."
    )

smiles_col = smiles_candidates[0]

# Prefer raw validity column if available
validity_col = None

for col in validity_candidates:
    if "raw" in col.lower():
        validity_col = col
        break

if validity_col is None and validity_candidates:
    validity_col = validity_candidates[0]

# ------------------------------------------------------------
# 4. CLASSIFY EACH GENERATED SMILES
# ------------------------------------------------------------

def classify_smiles(smiles):

    if not isinstance(smiles, str) or not smiles.strip():
        return "EMPTY_SEQUENCE", "Empty SMILES"

    # Parentheses
    if smiles.count("(") != smiles.count(")"):
        return "UNBALANCED_PARENTHESES", "Parentheses mismatch"

    # Square brackets
    if smiles.count("[") != smiles.count("]"):
        return "UNBALANCED_BRACKETS", "Bracket mismatch"

    # RDKit parsing
    mol = Chem.MolFromSmiles(smiles, sanitize=False)

    if mol is None:
        return "PARSING_FAILURE", "RDKit could not parse SMILES"

    # Sanitization
    try:
        Chem.SanitizeMol(mol)
    except Exception as e:
        return "SANITIZATION_FAILURE", str(e).split("\n")[0][:150]

    return "VALID", "Successfully parsed and sanitized"


results = []

for idx, row in df.iterrows():

    smiles = row[smiles_col]

    category, reason = classify_smiles(smiles)

    results.append({
        "sample_index": idx,
        "smiles": smiles,
        "failure_category": category,
        "failure_reason": reason
    })

result_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 5. FAILURE DISTRIBUTION
# ------------------------------------------------------------

print("\n3. FAILURE CATEGORY DISTRIBUTION")

distribution = (
    result_df["failure_category"]
    .value_counts()
    .reset_index()
)

distribution.columns = ["Failure Category", "Count"]
distribution["Percentage"] = (
    distribution["Count"] / len(result_df) * 100
).round(2)

print(distribution.to_string(index=False))

# ------------------------------------------------------------
# 6. DISPLAY FIVE VALID MOLECULES
# ------------------------------------------------------------

print("\n4. VALID GENERATED MOLECULES")

valid_df = result_df[
    result_df["failure_category"] == "VALID"
]

if len(valid_df) == 0:
    print("No valid molecules found.")
else:
    for _, row in valid_df.iterrows():
        print(
            f"Sample {row['sample_index']}: "
            f"{row['smiles']}"
        )

# ------------------------------------------------------------
# 7. DISPLAY INVALID EXAMPLES
# ------------------------------------------------------------

print("\n5. INVALID MOLECULE EXAMPLES")

invalid_df = result_df[
    result_df["failure_category"] != "VALID"
]

for category in invalid_df["failure_category"].unique():

    print(f"\n--- {category} ---")

    examples = invalid_df[
        invalid_df["failure_category"] == category
    ].head(3)

    for _, row in examples.iterrows():
        print("SMILES:", row["smiles"])
        print("Reason:", row["failure_reason"])

# ------------------------------------------------------------
# 8. SAVE AUDIT
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell215_generation_failure_analysis.csv"
)

result_df.to_csv(output_path, index=False)

distribution_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell215_failure_distribution.csv"
)

distribution.to_csv(distribution_path, index=False)

print("\n6. SAVED FILES")
print(output_path)
print(distribution_path)

print("\n" + "=" * 70)
print("CELL 215 COMPLETED")
print("=" * 70)

CELL 215: GENERATED SMILES FAILURE PATTERN AUDIT

1. DATASET CHECK
Total records: 100
Columns: ['sample_id', 'generated_tokens', 'unk_count', 'reached_eos', 'raw_smiles', 'cleaned_smiles', 'raw_valid', 'cleaned_valid']

2. AVAILABLE COLUMNS
sample_id : int64
generated_tokens : int64
unk_count : int64
reached_eos : bool
raw_smiles : object
cleaned_smiles : object
raw_valid : bool
cleaned_valid : bool

SMILES candidates: ['raw_smiles']
Validity candidates: ['raw_valid', 'cleaned_valid']

3. FAILURE CATEGORY DISTRIBUTION
      Failure Category  Count  Percentage
       PARSING_FAILURE     56        56.0
UNBALANCED_PARENTHESES     31        31.0
  SANITIZATION_FAILURE      8         8.0
                 VALID      5         5.0

4. VALID GENERATED MOLECULES
Sample 12: CC(=O)NC(=O)OCC1CCCC(C)=CCCC(=O)N1CCCCO
Sample 13: COc1ccc(SS(=O)(=O)N2CC2)nc1
Sample 16: CCOc1cc(NC(=O)C(=O)NCc2nc(C(F)F)c3c(cnc3)s2)c1
Sample 51: CCN(C(=O)N1CCCCC(=O)c2ccccc2)c1
Sample 55: CCOCCCCC(C)CC(=O)NC(=O)Nc1cccc(N)n

In [289]:
# ============================================================
# CELL 216: SMILES STRUCTURAL ERROR AUDIT
# ============================================================

import os
import re
import pandas as pd
from rdkit import Chem
from collections import Counter

print("=" * 70)
print("CELL 216: SMILES STRUCTURAL ERROR AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD GENERATED MOLECULES
# ------------------------------------------------------------

input_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

df = pd.read_csv(input_path)

print("\nTotal generated molecules:", len(df))

# ------------------------------------------------------------
# 2. STRUCTURAL CHECK FUNCTIONS
# ------------------------------------------------------------

def check_parentheses(smiles):
    depth = 0
    minimum_depth = 0

    for ch in smiles:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
            minimum_depth = min(minimum_depth, depth)

    return depth, minimum_depth


def check_brackets(smiles):
    return smiles.count("[") == smiles.count("]")


def check_ring_labels(smiles):
    """
    Count SMILES ring labels:
    0-9 and %nn.
    """

    labels = re.findall(r"%\d{2}|[0-9]", smiles)

    counts = Counter(labels)

    odd_labels = {
        label: count
        for label, count in counts.items()
        if count % 2 != 0
    }

    return len(odd_labels) == 0, odd_labels


def check_characters(smiles):

    allowed = set(
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "abcdefghijklmnopqrstuvwxyz"
        "0123456789"
        "()[]+-=#@\\/%."
    )

    unusual = sorted(set(smiles) - allowed)

    return unusual


# ------------------------------------------------------------
# 3. ANALYZE EVERY MOLECULE
# ------------------------------------------------------------

records = []

for _, row in df.iterrows():

    smiles = str(row["raw_smiles"])

    final_depth, minimum_depth = check_parentheses(smiles)

    brackets_valid = check_brackets(smiles)

    rings_valid, odd_ring_labels = check_ring_labels(smiles)

    unusual_characters = check_characters(smiles)

    mol = Chem.MolFromSmiles(
        smiles,
        sanitize=False
    )

    parse_success = mol is not None

    sanitize_success = False
    sanitize_error = ""

    if parse_success:

        try:
            Chem.SanitizeMol(mol)
            sanitize_success = True

        except Exception as e:
            sanitize_error = str(e).split("\n")[0][:200]

    records.append({

        "sample_id": row["sample_id"],
        "smiles": smiles,

        "parentheses_final_depth": final_depth,
        "parentheses_never_negative": minimum_depth >= 0,
        "parentheses_balanced": final_depth == 0,

        "brackets_balanced": brackets_valid,

        "ring_labels_balanced": rings_valid,
        "odd_ring_labels": str(odd_ring_labels),

        "unusual_characters": str(unusual_characters),

        "rdkit_parse_success": parse_success,
        "rdkit_sanitize_success": sanitize_success,

        "sanitize_error": sanitize_error

    })


diagnostics = pd.DataFrame(records)

# ------------------------------------------------------------
# 4. STRUCTURAL ERROR SUMMARY
# ------------------------------------------------------------

print("\n1. STRUCTURAL CHECK SUMMARY")

checks = [
    "parentheses_balanced",
    "parentheses_never_negative",
    "brackets_balanced",
    "ring_labels_balanced",
    "rdkit_parse_success",
    "rdkit_sanitize_success"
]

for check in checks:

    count = diagnostics[check].sum()

    print(
        f"{check}: {count}/{len(diagnostics)} "
        f"({count / len(diagnostics) * 100:.2f}%)"
    )


# ------------------------------------------------------------
# 5. ANALYZE PARSING FAILURES
# ------------------------------------------------------------

print("\n2. PARSING FAILURE STRUCTURAL PATTERNS")

parse_failures = diagnostics[
    diagnostics["rdkit_parse_success"] == False
]

print("Total parsing failures:", len(parse_failures))

print(
    "Unbalanced parentheses:",
    (~parse_failures["parentheses_balanced"]).sum()
)

print(
    "Unbalanced brackets:",
    (~parse_failures["brackets_balanced"]).sum()
)

print(
    "Unbalanced ring labels:",
    (~parse_failures["ring_labels_balanced"]).sum()
)


# ------------------------------------------------------------
# 6. SANITIZATION FAILURE DETAILS
# ------------------------------------------------------------

print("\n3. SANITIZATION FAILURE DETAILS")

sanitize_failures = diagnostics[
    (diagnostics["rdkit_parse_success"] == True) &
    (diagnostics["rdkit_sanitize_success"] == False)
]

print("Total:", len(sanitize_failures))

for _, row in sanitize_failures.head(10).iterrows():

    print("\nSample:", row["sample_id"])
    print("SMILES:", row["smiles"])
    print("Error:", row["sanitize_error"])


# ------------------------------------------------------------
# 7. SHOW RING LABEL PROBLEMS
# ------------------------------------------------------------

print("\n4. RING LABEL PROBLEMS")

ring_failures = diagnostics[
    diagnostics["ring_labels_balanced"] == False
]

print("Molecules with odd ring-label counts:", len(ring_failures))

for _, row in ring_failures.head(10).iterrows():

    print("\nSample:", row["sample_id"])
    print("SMILES:", row["smiles"])
    print("Odd labels:", row["odd_ring_labels"])


# ------------------------------------------------------------
# 8. SAVE REPORT
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell216_structural_error_audit.csv"
)

diagnostics.to_csv(output_path, index=False)

print("\n5. REPORT SAVED")
print(output_path)

print("\n" + "=" * 70)
print("CELL 216 COMPLETED")
print("=" * 70)

CELL 216: SMILES STRUCTURAL ERROR AUDIT

Total generated molecules: 100

1. STRUCTURAL CHECK SUMMARY
parentheses_balanced: 69/100 (69.00%)
parentheses_never_negative: 78/100 (78.00%)
brackets_balanced: 100/100 (100.00%)
ring_labels_balanced: 21/100 (21.00%)
rdkit_parse_success: 13/100 (13.00%)
rdkit_sanitize_success: 5/100 (5.00%)

2. PARSING FAILURE STRUCTURAL PATTERNS
Total parsing failures: 87
Unbalanced parentheses: 31
Unbalanced brackets: 0
Unbalanced ring labels: 79

3. SANITIZATION FAILURE DETAILS
Total: 8

Sample: 3
SMILES: COc1ccc(C(=O)N2CCCCCCN(c3cccccc3)C2)c1
Error: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 24

Sample: 28
SMILES: COc1cccc(C2=CC(F)F)cc2c(cccc2OC)cc2ccc1
Error: Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6 12 13 14 15 16 17 18 21 22 23 24 25

Sample: 36
SMILES: CN(c1ccc(Cl)c(C(=O)c2nc(-c3cnc(Cl)cc4CN(C)Cc3(C)C)C4)c2)n1
Error: Can't kekulize mol.  Unkekulized atoms: 7 10 28

Sample: 45
SMILES: NCCc1cccc1-c1c(=O)ccc(-c2cc(Cl)cc2)cc1
Error: Can't kekuli

In [290]:
# ============================================================
# CELL 217: ACCURATE RING AND BRANCH STRUCTURE AUDIT
# ============================================================

import os
import re
import pandas as pd
from collections import Counter

print("=" * 70)
print("CELL 217: ACCURATE RING AND BRANCH STRUCTURE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD GENERATED MOLECULES
# ------------------------------------------------------------

input_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

df = pd.read_csv(input_path)

# ------------------------------------------------------------
# 2. STRUCTURAL SCANNER
# ------------------------------------------------------------

def structural_scan(smiles):

    inside_bracket = False
    bracket_depth = 0

    branch_stack = []
    ring_counts = Counter()

    invalid_closings = 0
    unusual_ring_labels = []

    i = 0

    while i < len(smiles):

        ch = smiles[i]

        if ch == "[":
            if not inside_bracket:
                inside_bracket = True
                bracket_depth = 1
            else:
                bracket_depth += 1

        elif ch == "]":
            if inside_bracket:
                bracket_depth -= 1
                if bracket_depth == 0:
                    inside_bracket = False
            else:
                invalid_closings += 1

        elif not inside_bracket:

            if ch == "(":
                branch_stack.append(i)

            elif ch == ")":
                if branch_stack:
                    branch_stack.pop()
                else:
                    invalid_closings += 1

            elif ch == "%":
                label = smiles[i:i+3]

                if re.fullmatch(r"%\d{2}", label):
                    ring_counts[label] += 1
                    i += 2
                else:
                    unusual_ring_labels.append(label)

            elif ch.isdigit():
                ring_counts[ch] += 1

        i += 1

    odd_ring_labels = {
        label: count
        for label, count in ring_counts.items()
        if count % 2 != 0
    }

    return {
        "ring_counts": dict(ring_counts),
        "odd_ring_labels": odd_ring_labels,
        "ring_labels_balanced": len(odd_ring_labels) == 0,
        "unclosed_branches": len(branch_stack),
        "branch_closing_errors": invalid_closings,
        "brackets_closed": not inside_bracket,
        "unusual_ring_labels": str(unusual_ring_labels)
    }


# ------------------------------------------------------------
# 3. ANALYZE ALL MOLECULES
# ------------------------------------------------------------

records = []

for _, row in df.iterrows():

    smiles = str(row["raw_smiles"])

    result = structural_scan(smiles)

    records.append({
        "sample_id": row["sample_id"],
        "smiles": smiles,
        **result
    })

audit = pd.DataFrame(records)

audit["branch_structure_valid"] = (
    (audit["unclosed_branches"] == 0) &
    (audit["branch_closing_errors"] == 0)
)

# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------

print("\n1. ACCURATE STRUCTURAL SUMMARY")

checks = {
    "Balanced ring labels": audit["ring_labels_balanced"],
    "Valid branch structure": audit["branch_structure_valid"],
    "Closed square brackets": audit["brackets_closed"]
}

for name, values in checks.items():

    count = values.sum()

    print(
        f"{name}: {count}/{len(audit)} "
        f"({count / len(audit) * 100:.2f}%)"
    )

# ------------------------------------------------------------
# 5. OVERLAPPING STRUCTURAL ERRORS
# ------------------------------------------------------------

print("\n2. STRUCTURAL ERROR OVERLAP")

audit["ring_error"] = ~audit["ring_labels_balanced"]
audit["branch_error"] = ~audit["branch_structure_valid"]

overlap = pd.crosstab(
    audit["ring_error"],
    audit["branch_error"],
    rownames=["Ring error"],
    colnames=["Branch error"]
)

print(overlap)

print("\nInterpretation:")
print("Both errors:", (audit["ring_error"] & audit["branch_error"]).sum())
print("Only ring error:", (audit["ring_error"] & ~audit["branch_error"]).sum())
print("Only branch error:", (~audit["ring_error"] & audit["branch_error"]).sum())
print("Neither:", (~audit["ring_error"] & ~audit["branch_error"]).sum())

# ------------------------------------------------------------
# 6. DISPLAY EXAMPLES WITH RING ERRORS
# ------------------------------------------------------------

print("\n3. RING ERROR EXAMPLES")

ring_errors = audit[audit["ring_error"]]

for _, row in ring_errors.head(10).iterrows():

    print("\nSample:", row["sample_id"])
    print("SMILES:", row["smiles"])
    print("Ring counts:", row["ring_counts"])
    print("Odd labels:", row["odd_ring_labels"])

# ------------------------------------------------------------
# 7. DISPLAY BRANCH ERRORS
# ------------------------------------------------------------

print("\n4. BRANCH ERROR EXAMPLES")

branch_errors = audit[audit["branch_error"]]

for _, row in branch_errors.head(10).iterrows():

    print("\nSample:", row["sample_id"])
    print("SMILES:", row["smiles"])
    print("Unclosed branches:", row["unclosed_branches"])
    print("Unexpected closing symbols:", row["branch_closing_errors"])

# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell217_accurate_structure_audit.csv"
)

audit.to_csv(output_path, index=False)

print("\n5. SAVED:")
print(output_path)

print("\n" + "=" * 70)
print("CELL 217 COMPLETED")
print("=" * 70)

CELL 217: ACCURATE RING AND BRANCH STRUCTURE AUDIT

1. ACCURATE STRUCTURAL SUMMARY
Balanced ring labels: 21/100 (21.00%)
Valid branch structure: 66/100 (66.00%)
Closed square brackets: 100/100 (100.00%)

2. STRUCTURAL ERROR OVERLAP
Branch error  False  True 
Ring error                
False            14      7
True             52     27

Interpretation:
Both errors: 27
Only ring error: 52
Only branch error: 7
Neither: 14

3. RING ERROR EXAMPLES

Sample: 0
SMILES: COCC(=O)N1CCC(=O)[C@@H]1CC(=O)OC1C(C(=O)O)CCC=O)O
Ring counts: {'1': 3}
Odd labels: {'1': 3}

Sample: 2
SMILES: CCCCCC(=O)/C(=C(NC(c1ccccc1)cccc2)C(=O)c1
Ring counts: {'1': 3, '2': 1}
Odd labels: {'1': 3, '2': 1}

Sample: 4
SMILES: O=C(CCCCCCCCOc1cc(C(=O)OC2Cc2cccccc2)C(CN3CCCCCCCC4)n3CC2)cc1
Ring counts: {'1': 2, '2': 4, '3': 2, '4': 1}
Odd labels: {'4': 1}

Sample: 5
SMILES: O=C(c1ccc([N+](=O)NC2CNC3Nc3ccccc4c(ccc5)Cc4cccccc34)cc241
Ring counts: {'1': 2, '2': 2, '3': 3, '4': 4, '5': 1}
Odd labels: {'3': 3, '5': 1}

Sample: 

In [291]:
# ============================================================
# CELL 218: GENERATION LENGTH AND TRUNCATION AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 218: GENERATION LENGTH AND TRUNCATION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD GENERATED DATA
# ------------------------------------------------------------

generation_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

generation_df = pd.read_csv(generation_path)

print("\nGenerated samples:", len(generation_df))

# ------------------------------------------------------------
# 2. LOAD TRAINING TOKEN DATA
# ------------------------------------------------------------

train_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "chembl_train_tokens.npy"
)

train_tokens = np.load(train_path)

print("Training token array shape:", train_tokens.shape)

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2

# Count molecular tokens, excluding special tokens and padding
training_lengths = []

for sequence in train_tokens:

    tokens = [
        int(token)
        for token in sequence
        if int(token) not in [PAD_ID, BOS_ID, EOS_ID]
    ]

    training_lengths.append(len(tokens))

training_lengths = np.array(training_lengths)

# ------------------------------------------------------------
# 3. GENERATED SEQUENCE LENGTHS
# ------------------------------------------------------------

generated_lengths = generation_df["generated_tokens"].astype(int)

print("\n1. TRAINING LENGTH STATISTICS")

print("Molecules:", len(training_lengths))
print("Minimum:", training_lengths.min())
print("Maximum:", training_lengths.max())
print("Mean:", round(training_lengths.mean(), 2))
print("Median:", round(np.median(training_lengths), 2))
print("90th percentile:", np.percentile(training_lengths, 90))
print("95th percentile:", np.percentile(training_lengths, 95))

print("\n2. GENERATED LENGTH STATISTICS")

print("Molecules:", len(generated_lengths))
print("Minimum:", generated_lengths.min())
print("Maximum:", generated_lengths.max())
print("Mean:", round(generated_lengths.mean(), 2))
print("Median:", round(generated_lengths.median(), 2))
print("90th percentile:", generated_lengths.quantile(0.90))
print("95th percentile:", generated_lengths.quantile(0.95))

# ------------------------------------------------------------
# 4. COMPARE GENERATION LENGTH WITH TRAINING
# ------------------------------------------------------------

print("\n3. LENGTH COMPARISON")

percentiles = [10, 25, 50, 75, 90, 95, 99]

comparison = pd.DataFrame({
    "Percentile": percentiles,
    "Training": [
        np.percentile(training_lengths, p)
        for p in percentiles
    ],
    "Generated": [
        np.percentile(generated_lengths, p)
        for p in percentiles
    ]
})

print(comparison.to_string(index=False))

# ------------------------------------------------------------
# 5. CHECK GENERATION LENGTH AGAINST STRUCTURAL ERRORS
# ------------------------------------------------------------

failure_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell215_generation_failure_analysis.csv"
)

failure_df = pd.read_csv(failure_path)

merged = generation_df.merge(
    failure_df[["sample_index", "failure_category"]],
    left_on="sample_id",
    right_on="sample_index",
    how="left"
)

print("\n4. GENERATION LENGTH BY FAILURE CATEGORY")

length_summary = merged.groupby(
    "failure_category"
)["generated_tokens"].agg(
    ["count", "mean", "median", "min", "max"]
).round(2)

print(length_summary)

# ------------------------------------------------------------
# 6. CHECK EXTREMELY SHORT AND LONG GENERATIONS
# ------------------------------------------------------------

train_p95 = np.percentile(training_lengths, 95)

print("\n5. EXTREME GENERATED SEQUENCES")

print("Training 95th percentile:", round(train_p95, 2))

print(
    "Generated sequences shorter than 10 tokens:",
    (generated_lengths < 10).sum()
)

print(
    "Generated sequences longer than training 95th percentile:",
    (generated_lengths > train_p95).sum()
)

print("\nShortest generated samples:")

print(
    generation_df.nsmallest(
        5, "generated_tokens"
    )[["sample_id", "generated_tokens", "raw_smiles"]].to_string(index=False)
)

print("\nLongest generated samples:")

print(
    generation_df.nlargest(
        5, "generated_tokens"
    )[["sample_id", "generated_tokens", "raw_smiles"]].to_string(index=False)
)

# ------------------------------------------------------------
# 7. SAVE REPORT
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell218_generation_length_audit.csv"
)

comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)

print("\n" + "=" * 70)
print("CELL 218 COMPLETED")
print("=" * 70)

CELL 218: GENERATION LENGTH AND TRUNCATION AUDIT

Generated samples: 100
Training token array shape: (45000, 143)

1. TRAINING LENGTH STATISTICS
Molecules: 45000
Minimum: 7
Maximum: 141
Mean: 49.36
Median: 47.0
90th percentile: 69.0
95th percentile: 79.0

2. GENERATED LENGTH STATISTICS
Molecules: 100
Minimum: 24
Maximum: 80
Mean: 45.39
Median: 44.0
90th percentile: 64.10000000000001
95th percentile: 72.05

3. LENGTH COMPARISON
 Percentile  Training  Generated
         10      31.0      31.90
         25      38.0      35.75
         50      47.0      44.00
         75      58.0      51.25
         90      69.0      64.10
         95      79.0      72.05
         99     104.0      79.01

4. GENERATION LENGTH BY FAILURE CATEGORY
                        count   mean  median  min  max
failure_category                                      
PARSING_FAILURE            56  45.11    45.5   25   77
SANITIZATION_FAILURE        8  39.00    38.5   24   56
UNBALANCED_PARENTHESES     31  48.55    44.

In [292]:
# ============================================================
# CELL 219: TOKEN TRANSITION AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd
from collections import Counter

print("=" * 70)
print("CELL 219: TOKEN TRANSITION AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD TRAINING SMILES
# ------------------------------------------------------------

smiles_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "chembl37_clean_smiles.txt"
)

with open(smiles_path, "r") as f:
    training_smiles = [
        line.strip()
        for line in f
        if line.strip()
    ]

print("\nTraining molecules:", len(training_smiles))

# ------------------------------------------------------------
# 2. LOAD GENERATED SMILES
# ------------------------------------------------------------

generation_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

generation_df = pd.read_csv(generation_path)

generated_smiles = generation_df["raw_smiles"].astype(str).tolist()

print("Generated molecules:", len(generated_smiles))

# ------------------------------------------------------------
# 3. TOKENIZE BOTH DATASETS
# ------------------------------------------------------------

def safe_tokenize(smiles):

    tokens = tokenize_smiles(smiles)

    if isinstance(tokens, tuple):
        tokens = list(tokens)

    return list(tokens)


training_transitions = Counter()
generated_transitions = Counter()

training_unmatched = 0
generated_unmatched = 0

training_token_count = 0
generated_token_count = 0

# Training transitions
for smiles in training_smiles:

    tokens = safe_tokenize(smiles)

    if "<UNMATCHED>" in tokens:
        training_unmatched += 1
        continue

    training_token_count += len(tokens)

    for i in range(len(tokens) - 1):
        training_transitions[(tokens[i], tokens[i + 1])] += 1


# Generated transitions
for smiles in generated_smiles:

    tokens = safe_tokenize(smiles)

    if "<UNMATCHED>" in tokens:
        generated_unmatched += 1
        continue

    generated_token_count += len(tokens)

    for i in range(len(tokens) - 1):
        generated_transitions[(tokens[i], tokens[i + 1])] += 1


print("\n1. TOKENIZATION CHECK")
print("Training unmatched molecules:", training_unmatched)
print("Generated unmatched molecules:", generated_unmatched)

print("Training token count:", training_token_count)
print("Generated token count:", generated_token_count)

# ------------------------------------------------------------
# 4. TRANSITION STATISTICS
# ------------------------------------------------------------

training_unique = set(training_transitions.keys())
generated_unique = set(generated_transitions.keys())

shared = training_unique.intersection(generated_unique)

unseen = generated_unique - training_unique

print("\n2. TRANSITION SUMMARY")

print("Unique training transitions:", len(training_unique))
print("Unique generated transitions:", len(generated_unique))
print("Shared transitions:", len(shared))
print("Generated transitions unseen in training:", len(unseen))

if generated_unique:
    unseen_percentage = len(unseen) / len(generated_unique) * 100
else:
    unseen_percentage = 0

print("Unseen transition percentage:", round(unseen_percentage, 2))

# ------------------------------------------------------------
# 5. MOST FREQUENT UNSEEN TRANSITIONS
# ------------------------------------------------------------

print("\n3. MOST FREQUENT UNSEEN TRANSITIONS")

unseen_results = []

for transition in unseen:

    unseen_results.append({
        "previous_token": transition[0],
        "next_token": transition[1],
        "generated_count": generated_transitions[transition]
    })

unseen_df = pd.DataFrame(unseen_results)

if len(unseen_df):

    unseen_df = unseen_df.sort_values(
        "generated_count",
        ascending=False
    )

    print(unseen_df.head(25).to_string(index=False))

else:
    print("No unseen transitions found.")

# ------------------------------------------------------------
# 6. MOST FREQUENT GENERATED TRANSITIONS
# ------------------------------------------------------------

print("\n4. MOST FREQUENT GENERATED TRANSITIONS")

top_generated = generated_transitions.most_common(25)

for (previous, next_token), count in top_generated:

    train_count = training_transitions.get(
        (previous, next_token), 0
    )

    print(
        f"{previous!r} -> {next_token!r} | "
        f"Generated: {count} | Training: {train_count}"
    )

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell219_unseen_token_transitions.csv"
)

unseen_df.to_csv(output_path, index=False)

print("\n5. SAVED:")
print(output_path)

print("\n" + "=" * 70)
print("CELL 219 COMPLETED")
print("=" * 70)

CELL 219: TOKEN TRANSITION AUDIT

Training molecules: 50000
Generated molecules: 100

1. TOKENIZATION CHECK
Training unmatched molecules: 0
Generated unmatched molecules: 0
Training token count: 2468631
Generated token count: 4539

2. TRANSITION SUMMARY
Unique training transitions: 791
Unique generated transitions: 175
Shared transitions: 174
Generated transitions unseen in training: 1
Unseen transition percentage: 0.57

3. MOST FREQUENT UNSEEN TRANSITIONS
previous_token next_token  generated_count
             1          1                1

4. MOST FREQUENT GENERATED TRANSITIONS
'c' -> 'c' | Generated: 635 | Training: 269535
'C' -> 'C' | Generated: 338 | Training: 122999
'c' -> '1' | Generated: 195 | Training: 93027
'C' -> '(' | Generated: 187 | Training: 82745
'O' -> ')' | Generated: 174 | Training: 89439
'c' -> '2' | Generated: 166 | Training: 73904
'c' -> '(' | Generated: 165 | Training: 96559
')' -> 'c' | Generated: 164 | Training: 94503
'=' -> 'O' | Generated: 144 | Training: 730

In [293]:
# ============================================================
# CELL 220: THREE-TOKEN CONTEXT AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

print("=" * 70)
print("CELL 220: THREE-TOKEN CONTEXT AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

smiles_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "chembl37_clean_smiles.txt"
)

generation_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

with open(smiles_path, "r") as f:
    training_smiles = [
        line.strip() for line in f if line.strip()
    ]

generation_df = pd.read_csv(generation_path)

print("\nTraining molecules:", len(training_smiles))
print("Generated molecules:", len(generation_df))

# ------------------------------------------------------------
# 2. TOKENIZE AND COUNT TRIGRAMS
# ------------------------------------------------------------

training_trigrams = Counter()
generated_trigrams = Counter()

def get_tokens(smiles):
    tokens = tokenize_smiles(smiles)

    if isinstance(tokens, tuple):
        tokens = list(tokens)

    return list(tokens)


for smiles in training_smiles:

    tokens = get_tokens(smiles)

    if "<UNMATCHED>" in tokens:
        continue

    for i in range(len(tokens) - 2):
        trigram = tuple(tokens[i:i+3])
        training_trigrams[trigram] += 1


generated_token_sequences = {}

for _, row in generation_df.iterrows():

    tokens = get_tokens(str(row["raw_smiles"]))

    generated_token_sequences[row["sample_id"]] = tokens

    if "<UNMATCHED>" in tokens:
        continue

    for i in range(len(tokens) - 2):
        trigram = tuple(tokens[i:i+3])
        generated_trigrams[trigram] += 1


# ------------------------------------------------------------
# 3. TRIGRAM SUMMARY
# ------------------------------------------------------------

training_unique = set(training_trigrams)
generated_unique = set(generated_trigrams)

shared = training_unique & generated_unique
unseen = generated_unique - training_unique

print("\n1. TRIGRAM SUMMARY")

print("Unique training trigrams:", len(training_unique))
print("Unique generated trigrams:", len(generated_unique))
print("Shared trigrams:", len(shared))
print("Unseen generated trigrams:", len(unseen))

if generated_unique:
    print(
        "Unseen trigram percentage:",
        round(len(unseen) / len(generated_unique) * 100, 2)
    )

# ------------------------------------------------------------
# 4. MOST FREQUENT UNSEEN TRIGRAMS
# ------------------------------------------------------------

print("\n2. MOST FREQUENT UNSEEN TRIGRAMS")

unseen_records = []

for trigram in unseen:

    unseen_records.append({
        "token_1": trigram[0],
        "token_2": trigram[1],
        "token_3": trigram[2],
        "generated_count": generated_trigrams[trigram]
    })

unseen_df = pd.DataFrame(unseen_records)

if len(unseen_df):

    unseen_df = unseen_df.sort_values(
        "generated_count",
        ascending=False
    )

    print(unseen_df.head(30).to_string(index=False))

else:
    print("No unseen trigrams.")

# ------------------------------------------------------------
# 5. CONTEXT FREQUENCY AROUND RING AND BRANCH TOKENS
# ------------------------------------------------------------

print("\n3. STRUCTURAL TOKEN CONTEXT AUDIT")

structural_tokens = {"(", ")", "1", "2", "3", "4", "5"}

context_records = []

for sample_id, tokens in generated_token_sequences.items():

    for i, token in enumerate(tokens):

        if token not in structural_tokens:
            continue

        start = max(0, i - 2)
        end = min(len(tokens), i + 3)

        context = tuple(tokens[start:end])

        # Use the exact three-token window when available.
        if len(context) >= 3:
            context_key = tuple(context)

            # Check whether this complete context appears in training.
            found = False

            for j in range(len(context_key) - 2):
                tri = tuple(context_key[j:j+3])

                if tri in training_trigrams:
                    found = True
                else:
                    found = False
                    break

            context_records.append({
                "sample_id": sample_id,
                "structural_token": token,
                "context": " ".join(context),
                "context_seen_in_training": found
            })


context_df = pd.DataFrame(context_records)

if len(context_df):

    print("Structural token contexts:", len(context_df))

    print(
        "Contexts not fully observed in training:",
        (~context_df["context_seen_in_training"]).sum()
    )

    print("\nExamples:")

    print(
        context_df[
            context_df["context_seen_in_training"] == False
        ].head(20).to_string(index=False)
    )

# ------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------

output_dir = os.path.join(
    PROJECT_DIR,
    "member2_outputs"
)

trigram_path = os.path.join(
    output_dir,
    "cell220_unseen_trigrams.csv"
)

context_path = os.path.join(
    output_dir,
    "cell220_structural_contexts.csv"
)

unseen_df.to_csv(trigram_path, index=False)
context_df.to_csv(context_path, index=False)

print("\n4. SAVED FILES")
print(trigram_path)
print(context_path)

print("\n" + "=" * 70)
print("CELL 220 COMPLETED")
print("=" * 70)

CELL 220: THREE-TOKEN CONTEXT AUDIT

Training molecules: 50000
Generated molecules: 100

1. TRIGRAM SUMMARY
Unique training trigrams: 4163
Unique generated trigrams: 523
Shared trigrams: 509
Unseen generated trigrams: 14
Unseen trigram percentage: 2.68

2. MOST FREQUENT UNSEEN TRIGRAMS
token_1 token_2 token_3  generated_count
  [C@H]       (       =                2
      N       c       c                1
  [C@H]       2       =                1
      c       1       1                1
      2       3       S                1
      N       1       =                1
      N       3       n                1
      c       3       (                1
   [C@]       1       C                1
      #       C       (                1
      2       4       1                1
      )       F       c                1
      n       (      Cl                1
      O       c       c                1

3. STRUCTURAL TOKEN CONTEXT AUDIT
Structural token contexts: 1543
Contexts not fully observed in 

In [294]:
# ============================================================
# CELL 221: LONG-RANGE RING CLOSURE AUDIT
# ============================================================

import os
import re
import pandas as pd
from collections import defaultdict

print("=" * 70)
print("CELL 221: LONG-RANGE RING CLOSURE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD GENERATED MOLECULES
# ------------------------------------------------------------

input_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell214_special_token_audit.csv"
)

df = pd.read_csv(input_path)

# ------------------------------------------------------------
# 2. EXTRACT RING LABELS OUTSIDE BRACKETS
# ------------------------------------------------------------

def extract_ring_positions(smiles):

    positions = defaultdict(list)

    inside_bracket = False
    i = 0

    while i < len(smiles):

        ch = smiles[i]

        if ch == "[":
            inside_bracket = True

        elif ch == "]":
            inside_bracket = False

        elif not inside_bracket:

            if ch == "%":

                label = smiles[i:i+3]

                if re.fullmatch(r"%\d{2}", label):
                    positions[label].append(i)
                    i += 2

            elif ch.isdigit():
                positions[ch].append(i)

        i += 1

    return positions


# ------------------------------------------------------------
# 3. ANALYZE RING CLOSURES
# ------------------------------------------------------------

records = []

for _, row in df.iterrows():

    smiles = str(row["raw_smiles"])

    ring_positions = extract_ring_positions(smiles)

    total_occurrences = 0
    matched_pairs = 0
    unmatched_occurrences = 0
    maximum_span = 0

    ring_details = []

    for label, positions in ring_positions.items():

        count = len(positions)

        total_occurrences += count

        pairs = count // 2
        unmatched = count % 2

        matched_pairs += pairs
        unmatched_occurrences += unmatched

        spans = []

        for i in range(0, len(positions) - 1, 2):

            span = positions[i+1] - positions[i]

            spans.append(span)

            maximum_span = max(maximum_span, span)

        ring_details.append({
            "label": label,
            "occurrences": count,
            "matched_pairs": pairs,
            "unmatched": unmatched,
            "spans": spans
        })

    records.append({

        "sample_id": row["sample_id"],
        "smiles": smiles,

        "total_ring_occurrences": total_occurrences,
        "matched_ring_pairs": matched_pairs,
        "unmatched_ring_occurrences": unmatched_occurrences,

        "ring_labels_used": len(ring_positions),

        "maximum_ring_span": maximum_span,

        "ring_details": str(ring_details)

    })


audit = pd.DataFrame(records)

# ------------------------------------------------------------
# 4. SUMMARY
# ------------------------------------------------------------

print("\n1. RING CLOSURE SUMMARY")

print("Total molecules:", len(audit))

print(
    "Molecules with no unmatched ring occurrences:",
    (audit["unmatched_ring_occurrences"] == 0).sum()
)

print(
    "Molecules with unmatched ring occurrences:",
    (audit["unmatched_ring_occurrences"] > 0).sum()
)

print(
    "Total unmatched ring occurrences:",
    audit["unmatched_ring_occurrences"].sum()
)

print(
    "Total matched ring pairs:",
    audit["matched_ring_pairs"].sum()
)

# ------------------------------------------------------------
# 5. RING SPAN STATISTICS
# ------------------------------------------------------------

print("\n2. RING CLOSURE DISTANCE")

matched = audit[audit["matched_ring_pairs"] > 0]

if len(matched):

    print(
        "Mean maximum ring span:",
        round(matched["maximum_ring_span"].mean(), 2)
    )

    print(
        "Maximum observed ring span:",
        matched["maximum_ring_span"].max()
    )

    print(
        "Median maximum ring span:",
        matched["maximum_ring_span"].median()
    )

# ------------------------------------------------------------
# 6. DISPLAY PROBLEMATIC MOLECULES
# ------------------------------------------------------------

print("\n3. MOLECULES WITH UNMATCHED RING LABELS")

problematic = audit[
    audit["unmatched_ring_occurrences"] > 0
]

for _, row in problematic.head(15).iterrows():

    print("\nSample:", row["sample_id"])

    print("Unmatched occurrences:", row["unmatched_ring_occurrences"])

    print("Ring details:", row["ring_details"])

    print("SMILES:", row["smiles"])

# ------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------

output_path = os.path.join(
    PROJECT_DIR,
    "member2_outputs",
    "cell221_long_range_ring_audit.csv"
)

audit.to_csv(output_path, index=False)

print("\n4. SAVED:")
print(output_path)

print("\n" + "=" * 70)
print("CELL 221 COMPLETED")
print("=" * 70)

CELL 221: LONG-RANGE RING CLOSURE AUDIT

1. RING CLOSURE SUMMARY
Total molecules: 100
Molecules with no unmatched ring occurrences: 21
Molecules with unmatched ring occurrences: 79
Total unmatched ring occurrences: 122
Total matched ring pairs: 269

2. RING CLOSURE DISTANCE
Mean maximum ring span: 27.56
Maximum observed ring span: 75
Median maximum ring span: 25.0

3. MOLECULES WITH UNMATCHED RING LABELS

Sample: 0
Unmatched occurrences: 1
Ring details: [{'label': '1', 'occurrences': 3, 'matched_pairs': 1, 'unmatched': 1, 'spans': [14]}]
SMILES: COCC(=O)N1CCC(=O)[C@@H]1CC(=O)OC1C(C(=O)O)CCC=O)O

Sample: 2
Unmatched occurrences: 2
Ring details: [{'label': '1', 'occurrences': 3, 'matched_pairs': 1, 'unmatched': 1, 'spans': [6]}, {'label': '2', 'occurrences': 1, 'matched_pairs': 0, 'unmatched': 1, 'spans': []}]
SMILES: CCCCCC(=O)/C(=C(NC(c1ccccc1)cccc2)C(=O)c1

Sample: 4
Unmatched occurrences: 1
Ring details: [{'label': '1', 'occurrences': 2, 'matched_pairs': 1, 'unmatched': 0, 'spans': [

In [297]:
# ============================================================
# CELL 222: TRAINING vs GENERATED RING SPAN COMPARISON
# ============================================================

import numpy as np
import pandas as pd
from collections import defaultdict
from rdkit import Chem

print("=" * 65)
print("CELL 222: TRAINING vs GENERATED RING SPAN COMPARISON")
print("=" * 65)

# 1. Ring span extraction

def extract_ring_spans(smiles):

    positions = defaultdict(list)
    i = 0
    inside_bracket = False

    while i < len(smiles):

        ch = smiles[i]

        if ch == '[':
            inside_bracket = True
            i += 1
            continue

        if ch == ']':
            inside_bracket = False
            i += 1
            continue

        if not inside_bracket:

            if ch == '%' and i + 2 < len(smiles):
                label = smiles[i+1:i+3]

                if label.isdigit():
                    positions[label].append(i)
                    i += 3
                    continue

            elif ch.isdigit():
                positions[ch].append(i)

        i += 1

    spans = []

    for label, pos in positions.items():

        for j in range(0, len(pos)-1, 2):
            spans.append(pos[j+1] - pos[j])

    return spans


# 2. Load training molecules

train_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/chembl37_clean_smiles.txt"
)

with open(train_path, "r") as f:
    training_smiles = [x.strip() for x in f if x.strip()]


# 3. Load generated molecules

generated_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell214_special_token_audit.csv"
)

generated_df = pd.read_csv(generated_path)

generated_smiles = generated_df["raw_smiles"].dropna().tolist()


# 4. Extract spans only from RDKit-valid molecules

def collect_spans(smiles_list):

    all_spans = []
    valid_count = 0

    for smi in smiles_list:

        mol = Chem.MolFromSmiles(smi)

        if mol is None:
            continue

        valid_count += 1
        all_spans.extend(extract_ring_spans(smi))

    return all_spans, valid_count


training_spans, training_valid = collect_spans(training_smiles)

generated_spans, generated_valid = collect_spans(generated_smiles)


# 5. Display results

def report(name, spans):

    print(f"\n{name}")

    if not spans:
        print("No ring closures found.")
        return

    print("Total ring closures:", len(spans))
    print("Mean span:", round(np.mean(spans), 2))
    print("Median span:", round(np.median(spans), 2))
    print("90th percentile:", round(np.percentile(spans, 90), 2))
    print("Maximum span:", max(spans))


print("\nTRAINING")
print("Valid molecules:", training_valid)

print("\nGENERATED")
print("Valid molecules:", generated_valid)

report("TRAINING RING SPANS", training_spans)
report("GENERATED RING SPANS", generated_spans)

print("\nCELL 222 COMPLETED")

CELL 222: TRAINING vs GENERATED RING SPAN COMPARISON

TRAINING
Valid molecules: 50000

GENERATED
Valid molecules: 5

TRAINING RING SPANS
Total ring closures: 183101
Mean span: 21.93
Median span: 16.0
90th percentile: 47.0
Maximum span: 228

GENERATED RING SPANS
Total ring closures: 11
Mean span: 15.73
Median span: 18.0
90th percentile: 25.0
Maximum span: 41

CELL 222 COMPLETED


In [296]:
import pandas as pd

path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/cell214_special_token_audit.csv"
)

df = pd.read_csv(path)

print("Column names:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
display(df.head(3))

Column names:
['sample_id', 'generated_tokens', 'unk_count', 'reached_eos', 'raw_smiles', 'cleaned_smiles', 'raw_valid', 'cleaned_valid']

First 3 rows:


,sample_id,generated_tokens,unk_count,reached_eos,raw_smiles,cleaned_smiles,raw_valid,cleaned_valid
0,0,44,0,True,COCC(=O)N1CCC(=O)[C@@H]1CC(=O)OC1C(C(=O)O)CCC=O)O,COCC(=O)N1CCC(=O)[C@@H]1CC(=O)OC1C(C(=O)O)CCC=O)O,False,False
1,1,46,0,True,COc1cccc(N2CCCC2)c2[nH]c3c(ccc(NNCc3nc(CCC)no2)c1,COc1cccc(N2CCCC2)c2[nH]c3c(ccc(NNCc3nc(CCC)no2)c1,False,False
2,2,41,0,True,CCCCCC(=O)/C(=C(NC(c1ccccc1)cccc2)C(=O)c1,CCCCCC(=O)/C(=C(NC(c1ccccc1)cccc2)C(=O)c1,False,False


In [298]:
# CELL 1: FIND EXISTING DATASET AND SELECTION FILES

import os

ROOT = "/content/drive/MyDrive/Member2_Project"

keywords = [
    "selection", "selected", "manifest",
    "target", "parp1", "final"
]

matches = []

for root, dirs, files in os.walk(ROOT):

    # Avoid unnecessary checkpoint and cache directories
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".cache"]
    ]

    for filename in files:

        ext = os.path.splitext(filename)[1].lower()

        if ext not in [".csv", ".parquet", ".json", ".txt"]:
            continue

        if any(k in filename.lower() for k in keywords):

            path = os.path.join(root, filename)
            size_mb = os.path.getsize(path) / (1024 * 1024)

            matches.append(path)

            print(f"{filename}")
            print(f"  Path: {path}")
            print(f"  Size: {size_mb:.2f} MB")

print("\nTotal candidate files:", len(matches))

extraction_manifest.json
  Path: /content/drive/MyDrive/Member2_Project/crossdocked_subset/extraction_manifest.json
  Size: 0.02 MB
crossdocked_selected_500.csv
  Path: /content/drive/MyDrive/Member2_Project/checkpoints/crossdocked_selected_500.csv
  Size: 0.10 MB
selected_500_metadata.json
  Path: /content/drive/MyDrive/Member2_Project/checkpoints/selected_500_metadata.json
  Size: 0.00 MB
extraction_manifest.json
  Path: /content/drive/MyDrive/Member2_Project/final_selected_500/extraction_manifest.json
  Size: 0.20 MB
receptor_extraction_manifest.json
  Path: /content/drive/MyDrive/Member2_Project/final_selected_500/receptor_extraction_manifest.json
  Size: 0.13 MB
final_validated_500_mapping.csv
  Path: /content/drive/MyDrive/Member2_Project/member2_outputs/final_validated_500_mapping.csv
  Size: 0.03 MB
final_validated_495_mapping.csv
  Path: /content/drive/MyDrive/Member2_Project/member2_outputs/final_validated_495_mapping.csv
  Size: 0.03 MB
chembl_parp_target_verification.json
 

In [299]:
# CELL 2: INSPECT CANDIDATE DATASET FILES

import pandas as pd
import os

print("=" * 65)
print("DATASET FILE INSPECTION")
print("=" * 65)

for path in matches:

    try:
        ext = os.path.splitext(path)[1].lower()

        if ext == ".csv":
            df = pd.read_csv(path, nrows=1000)

        elif ext == ".parquet":
            df = pd.read_parquet(path)

        elif ext == ".json":
            continue

        else:
            continue

        print("\nFILE:", os.path.basename(path))
        print("PATH:", path)
        print("ROWS INSPECTED:", len(df))
        print("COLUMNS:", df.columns.tolist())

        if len(df) == 500:
            print("*** POSSIBLE 500-RECORD SELECTION ***")

        print("\nSAMPLE:")
        display(df.head(3))

    except Exception as e:
        print("\nCould not inspect:", path)
        print("Reason:", str(e)[:200])

DATASET FILE INSPECTION

FILE: crossdocked_selected_500.csv
PATH: /content/drive/MyDrive/Member2_Project/checkpoints/crossdocked_selected_500.csv
ROWS INSPECTED: 500
COLUMNS: ['pair_idx', 'complex_dir', 'receptor_pdb', 'ligand_sdf_gz', 'source_type', 'shard_idx', 'label', 'score1', 'score2', 'cdonly_fold0', 'cdonly_fold1', 'cdonly_fold2', 'it0_fold0', 'it0_fold1', 'it0_fold2', 'it2_fold0', 'it2_fold1', 'it2_fold2', 'it2_redocked_fold0', 'it2_redocked_fold1', 'it2_redocked_fold2', 'it2_10p20n_fold0', 'it2_10p20n_fold1', 'it2_10p20n_fold2', 'mod_it2_fold0', 'mod_it2_fold1', 'mod_it2_fold2', 'it2_completeset_fold0', 'it2_completeset_fold1', 'it2_completeset_fold2', 'it2_redocked_completeset_fold0', 'it2_redocked_completeset_fold1', 'it2_redocked_completeset_fold2', 'mod_it2_completeset_fold0', 'mod_it2_completeset_fold1', 'mod_it2_completeset_fold2', 'pose_type']
*** POSSIBLE 500-RECORD SELECTION ***

SAMPLE:


,pair_idx,complex_dir,receptor_pdb,ligand_sdf_gz,source_type,shard_idx,label,score1,score2,cdonly_fold0,...,it2_completeset_fold0,it2_completeset_fold1,it2_completeset_fold2,it2_redocked_completeset_fold0,it2_redocked_completeset_fold1,it2_redocked_completeset_fold2,mod_it2_completeset_fold0,mod_it2_completeset_fold1,mod_it2_completeset_fold2,pose_type
0,922691,FPPS_HUMAN_69_419_allosteric_0,2qis_A_rec.pdb,2qis_A_rec_3n1v_3n1_lig_tt_min.sdf.gz,cdonly,12,0,-3.301,5.6597,train,...,train,NaN,NaN,NaN,NaN,NaN,train,NaN,NaN,tt_min
1,158402,ARNB_SALTY_1_385_0,1mdo_A_rec.pdb,1mdo_A_rec_4oca_2qr_lig_tt_min.sdf.gz,cdonly,1,1,0.000,0.4744,train,...,train,NaN,NaN,NaN,NaN,NaN,train,NaN,NaN,tt_min
2,2279840,SYL_STRPJ_227_410_0,4k48_A_rec.pdb,4k48_A_rec_4k47_wmp_lig_tt_min.sdf.gz,cdonly,31,0,0.000,3.5824,train,...,train,NaN,NaN,NaN,NaN,NaN,train,NaN,NaN,tt_min



FILE: final_validated_500_mapping.csv
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/final_validated_500_mapping.csv
ROWS INSPECTED: 500
COLUMNS: ['pair_idx', 'ligand_file', 'receptor_pdb', 'complex_dir', 'ligand_exists', 'receptor_exists', 'pdb_has_atoms']
*** POSSIBLE 500-RECORD SELECTION ***

SAMPLE:


,pair_idx,ligand_file,receptor_pdb,complex_dir,ligand_exists,receptor_exists,pdb_has_atoms
0,158402,0158402.sdf.gz,1mdo_A_rec.pdb,ARNB_SALTY_1_385_0,True,True,True
1,159900,0159900.sdf.gz,1gg1_B_rec.pdb,AROG_ECOLI_1_350_0,True,True,True
2,84592,0084592.sdf.gz,5unf_B_rec.pdb,AGTR2_HUMAN_35_335_0,True,True,True



FILE: final_validated_495_mapping.csv
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/final_validated_495_mapping.csv
ROWS INSPECTED: 495
COLUMNS: ['pair_idx', 'ligand_file', 'receptor_pdb', 'complex_dir', 'ligand_exists', 'receptor_exists', 'pdb_has_atoms']

SAMPLE:


,pair_idx,ligand_file,receptor_pdb,complex_dir,ligand_exists,receptor_exists,pdb_has_atoms
0,158402,0158402.sdf.gz,1mdo_A_rec.pdb,ARNB_SALTY_1_385_0,True,True,True
1,159900,0159900.sdf.gz,1gg1_B_rec.pdb,AROG_ECOLI_1_350_0,True,True,True
2,84592,0084592.sdf.gz,5unf_B_rec.pdb,AGTR2_HUMAN_35_335_0,True,True,True


... (52 outputs left out)


FILE: final_composed_transform_validation.csv
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/final_composed_transform_validation.csv
ROWS INSPECTED: 464
COLUMNS: ['residue_number', 'residue_name', 'atom_name', 'receptor_x', 'receptor_y', 'receptor_z', 'm1_x', 'm1_y', 'm1_z', 'distance_A', 'is_pocket_residue', 'composed_transform_residual_A', 'within_2A']

SAMPLE:


,residue_number,residue_name,atom_name,receptor_x,receptor_y,receptor_z,m1_x,m1_y,m1_z,distance_A,is_pocket_residue,composed_transform_residual_A,within_2A
0,684,LYS,N,2.004942,31.994062,-4.943134,1.286,31.927,-5.033,0.727634,True,0.935108,True
1,684,LYS,CA,3.438969,31.765540,-5.031914,2.637,31.459,-5.312,0.903089,True,1.122713,True
2,684,LYS,C,4.417419,31.890643,-3.773755,3.529,31.633,-4.070,0.971303,True,1.188782,True



FILE: crossdocked_pose_quality_manifest.csv
PATH: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_pose_quality_manifest.csv
ROWS INSPECTED: 51
COLUMNS: ['pair_idx', 'ligand_identity', 'pose_index', 'atom_count', 'bond_count', 'geometry_error_A', 'reopened_validation', 'output_file', 'ligand_atoms', 'min_distance_A', 'atom_pairs_le_2A', 'atom_pairs_le_3A', 'atom_pairs_le_4A', 'atom_pairs_le_5A', 'ligand_atoms_le_3A', 'ligand_atoms_le_4A', 'ligand_atoms_le_5A', 'severe_overlap', 'has_receptor_contacts', 'pose_quality', 'usable_for_spatial_screening']

SAMPLE:


,pair_idx,ligand_identity,pose_index,atom_count,bond_count,geometry_error_A,reopened_validation,output_file,ligand_atoms,min_distance_A,...,atom_pairs_le_3A,atom_pairs_le_4A,atom_pairs_le_5A,ligand_atoms_le_3A,ligand_atoms_le_4A,ligand_atoms_le_5A,severe_overlap,has_receptor_contacts,pose_quality,usable_for_spatial_screening
0,1735676,1uk0_frm,0,28,31,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,28,2.811002,...,1,101,371,1,26,28,False,True,PASS,True
1,1735677,1uk1_frq,0,29,32,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,29,2.802332,...,1,99,372,1,27,29,False,True,PASS,True
2,1735678,1wok_cnq,0,20,22,1.065814e-14,True,/content/drive/MyDrive/Member2_Project/member2...,20,2.958304,...,2,61,262,2,15,20,False,True,PASS,True


In [300]:
import os
import json
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/Member2_Project"
OUT = os.path.join(ROOT, "member2_outputs")

manifest_path = os.path.join(
    OUT, "pocket_conditioned_training_manifest.csv"
)

ligand_path = os.path.join(
    OUT, "parp1_crossdocked_pose_chemical_identities.csv"
)

manifest = pd.read_csv(manifest_path)
ligands = pd.read_csv(ligand_path)

print("=" * 65)
print("CELL 223: POCKET-CONDITIONED DATASET AUDIT")
print("=" * 65)

print("\nMANIFEST")
print("Rows:", len(manifest))
print("Columns:", manifest.columns.tolist())
print("Unique ligand identities:", manifest["ligand_identity"].nunique())
print("Unique InChIKeys:", manifest["inchikey"].nunique())
print("Unique canonical SMILES:", manifest["canonical_smiles"].nunique())
print("Unique pair IDs:", manifest["pair_idx"].nunique())
print("Unique conformers:", manifest["conformer_id"].nunique())

print("\nSPLIT DISTRIBUTION")
print(manifest["split"].value_counts(dropna=False))

print("\nLIGAND POSE DATA")
print("Rows:", len(ligands))
print("Unique ligand identities:", ligands["ligand_identity"].nunique())
print("Unique InChIKeys:", ligands["inchikey"].nunique())

print("\nMISSING VALUES")
print(manifest.isnull().sum())

print("\nDUPLICATE CHECKS")
print("Duplicate rows:", manifest.duplicated().sum())
print(
    "Duplicate ligand-conformer pairs:",
    manifest.duplicated(
        subset=["ligand_identity", "conformer_id"]
    ).sum()
)

print("\nSAMPLE MANIFEST RECORDS")
display(manifest.head())

print("\nCELL 223 COMPLETED")

CELL 223: POCKET-CONDITIONED DATASET AUDIT

MANIFEST
Rows: 195
Columns: ['pair_idx', 'ligand_identity', 'inchikey', 'pose_index', 'conformer_id', 'canonical_smiles', 'ligand_atoms', 'ligand_bonds', 'split']
Unique ligand identities: 32
Unique InChIKeys: 32
Unique canonical SMILES: 32
Unique pair IDs: 32
Unique conformers: 5

SPLIT DISTRIBUTION
split
train         160
validation     35
Name: count, dtype: int64

LIGAND POSE DATA
Rows: 51
Unique ligand identities: 32
Unique InChIKeys: 32

MISSING VALUES
pair_idx            0
ligand_identity     0
inchikey            0
pose_index          0
conformer_id        0
canonical_smiles    0
ligand_atoms        0
ligand_bonds        0
split               0
dtype: int64

DUPLICATE CHECKS
Duplicate rows: 0
Duplicate ligand-conformer pairs: 35

SAMPLE MANIFEST RECORDS


,pair_idx,ligand_identity,inchikey,pose_index,conformer_id,canonical_smiles,ligand_atoms,ligand_bonds,split
0,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
1,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,2,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
2,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,3,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
3,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,4,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
4,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,5,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train



CELL 223 COMPLETED


In [301]:
print("=" * 65)
print("CELL 224: TRAIN/VALIDATION LEAKAGE AUDIT")
print("=" * 65)

train = manifest[
    manifest["split"].astype(str).str.lower() == "train"
].copy()

val = manifest[
    manifest["split"].astype(str).str.lower().isin(
        ["val", "valid", "validation"]
    )
].copy()

print("\nSPLIT SIZES")
print("Training rows:", len(train))
print("Validation rows:", len(val))

if len(train) == 0 or len(val) == 0:
    print("\nWARNING: Training or validation split is empty.")
else:
    checks = [
        "pair_idx",
        "ligand_identity",
        "inchikey",
        "canonical_smiles"
    ]

    for col in checks:
        train_values = set(train[col].dropna().astype(str))
        val_values = set(val[col].dropna().astype(str))

        overlap = train_values.intersection(val_values)

        print(f"\n{col}")
        print("Training unique:", len(train_values))
        print("Validation unique:", len(val_values))
        print("Overlapping values:", len(overlap))

        if overlap:
            print("WARNING: Leakage detected.")
        else:
            print("No overlap detected.")

print("\nCELL 224 COMPLETED")

CELL 224: TRAIN/VALIDATION LEAKAGE AUDIT

SPLIT SIZES
Training rows: 160
Validation rows: 35

pair_idx
Training unique: 25
Validation unique: 7
Overlapping values: 0
No overlap detected.

ligand_identity
Training unique: 25
Validation unique: 7
Overlapping values: 0
No overlap detected.

inchikey
Training unique: 25
Validation unique: 7
Overlapping values: 0
No overlap detected.

canonical_smiles
Training unique: 25
Validation unique: 7
Overlapping values: 0
No overlap detected.

CELL 224 COMPLETED


In [302]:
print("=" * 65)
print("CELL 225: REPEATED LIGAND-CONFORMER AUDIT")
print("=" * 65)

key_cols = ["ligand_identity", "conformer_id"]

duplicate_records = manifest[
    manifest.duplicated(subset=key_cols, keep=False)
].copy()

print("Rows involved in repeated combinations:", len(duplicate_records))
print(
    "Number of repeated ligand-conformer combinations:",
    duplicate_records[key_cols].drop_duplicates().shape[0]
)

if len(duplicate_records) > 0:
    print("\nRepeated combinations and their pose indices:")

    repeated_summary = (
        duplicate_records
        .groupby(key_cols)
        .agg(
            record_count=("pair_idx", "size"),
            pose_indices=("pose_index", lambda x: sorted(x.unique().tolist())),
            pair_ids=("pair_idx", lambda x: sorted(x.unique().tolist())),
            splits=("split", lambda x: sorted(x.unique().tolist()))
        )
        .reset_index()
    )

    display(repeated_summary.head(20))

    print("\nNumber of repeated combinations with multiple pose indices:",
          (repeated_summary["pose_indices"].apply(len) > 1).sum())
else:
    print("No repeated combinations found.")

print("\nCELL 225 COMPLETED")

CELL 225: REPEATED LIGAND-CONFORMER AUDIT
Rows involved in repeated combinations: 40
Number of repeated ligand-conformer combinations: 5

Repeated combinations and their pose indices:


,ligand_identity,conformer_id,record_count,pose_indices,pair_ids,splits
0,4hhy_15r,1,8,"[0, 1, 3, 4, 12, 14, 15, 17]",[1735686],[train]
1,4hhy_15r,2,8,"[0, 1, 3, 4, 12, 14, 15, 17]",[1735686],[train]
2,4hhy_15r,3,8,"[0, 1, 3, 4, 12, 14, 15, 17]",[1735686],[train]
3,4hhy_15r,4,8,"[0, 1, 3, 4, 12, 14, 15, 17]",[1735686],[train]
4,4hhy_15r,5,8,"[0, 1, 3, 4, 12, 14, 15, 17]",[1735686],[train]



Number of repeated combinations with multiple pose indices: 5

CELL 225 COMPLETED


In [303]:
print("=" * 65)
print("CELL 226: MANIFEST-TO-POSE DATA VALIDATION")
print("=" * 65)

pose_lookup = (
    ligands[
        ["pair_idx", "ligand_identity", "inchikey", "canonical_smiles"]
    ]
    .drop_duplicates()
)

comparison = manifest.merge(
    pose_lookup,
    on="pair_idx",
    how="left",
    suffixes=("_manifest", "_pose"),
    indicator=True
)

print("Manifest records:", len(manifest))
print("Matched records:", (comparison["_merge"] == "both").sum())
print("Unmatched records:", (comparison["_merge"] != "both").sum())

comparison["identity_match"] = (
    comparison["ligand_identity_manifest"]
    == comparison["ligand_identity_pose"]
)

comparison["inchikey_match"] = (
    comparison["inchikey_manifest"]
    == comparison["inchikey_pose"]
)

comparison["smiles_match"] = (
    comparison["canonical_smiles_manifest"]
    == comparison["canonical_smiles_pose"]
)

print("\nIDENTITY CHECKS")
print("Ligand identity matches:", comparison["identity_match"].sum())
print("InChIKey matches:", comparison["inchikey_match"].sum())
print("Canonical SMILES matches:", comparison["smiles_match"].sum())

failed = comparison[
    (comparison["_merge"] != "both")
    | (~comparison["identity_match"])
    | (~comparison["inchikey_match"])
    | (~comparison["smiles_match"])
]

print("\nFailed records:", len(failed))

if len(failed) > 0:
    display(failed.head(20))
else:
    print("All manifest records match their ligand pose metadata.")

print("\nCELL 226 COMPLETED")

CELL 226: MANIFEST-TO-POSE DATA VALIDATION
Manifest records: 195
Matched records: 195
Unmatched records: 0

IDENTITY CHECKS
Ligand identity matches: 195
InChIKey matches: 195
Canonical SMILES matches: 195

Failed records: 0
All manifest records match their ligand pose metadata.

CELL 226 COMPLETED


In [304]:
import os
import glob

print("=" * 65)
print("CELL 227: POSE AND POCKET FILE DISCOVERY")
print("=" * 65)

search_dirs = [
    os.path.join(ROOT, "member2_outputs"),
    os.path.join(ROOT, "member1_input")
]

extensions = ["*.sdf", "*.sdf.gz", "*.pdb", "*.npy", "*.json"]

for folder in search_dirs:
    print(f"\nDIRECTORY: {folder}")

    for ext in extensions:
        files = glob.glob(
            os.path.join(folder, "**", ext),
            recursive=True
        )

        print(f"\n{ext}: {len(files)} files")

        for path in files[:8]:
            print(" ", path)

print("\nCELL 227 COMPLETED")

CELL 227: POSE AND POCKET FILE DISCOVERY

DIRECTORY: /content/drive/MyDrive/Member2_Project/member2_outputs

*.sdf: 138 files
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_046.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_093.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_079.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_047.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_006.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_055.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_059.sdf
  /content/drive/MyDrive/Member2_Project/member2_outputs/ligand_3d_conformers/GRU_001.sdf

*.sdf.gz: 32 files
  /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_parp1_ligands_shard23/1735676.sdf.gz
  /content/drive/MyDrive/Mem

In [305]:
print("=" * 65)
print("CELL 228: SPATIAL VALIDATION FILE AUDIT")
print("=" * 65)

files_to_check = [
    "crossdocked_pose_quality_manifest.csv",
    "final_composed_transform_validation.csv",
    "pocket_conditioned_training_manifest.csv",
    "parp1_crossdocked_pose_chemical_identities.csv"
]

for filename in files_to_check:
    path = os.path.join(OUT, filename)

    print(f"\n{'-' * 55}")
    print("FILE:", filename)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path)

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))

print("\nCELL 228 COMPLETED")

CELL 228: SPATIAL VALIDATION FILE AUDIT

-------------------------------------------------------
FILE: crossdocked_pose_quality_manifest.csv
Exists: False

-------------------------------------------------------
FILE: final_composed_transform_validation.csv
Exists: False

-------------------------------------------------------
FILE: pocket_conditioned_training_manifest.csv
Exists: True
Shape: (195, 9)
Columns: ['pair_idx', 'ligand_identity', 'inchikey', 'pose_index', 'conformer_id', 'canonical_smiles', 'ligand_atoms', 'ligand_bonds', 'split']


,pair_idx,ligand_identity,inchikey,pose_index,conformer_id,canonical_smiles,ligand_atoms,ligand_bonds,split
0,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
1,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,2,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train
2,1735676,1uk0_frm,LOFDUAJQRUYHBR-UHFFFAOYSA-N,0,3,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,28,31,train



-------------------------------------------------------
FILE: parp1_crossdocked_pose_chemical_identities.csv
Exists: True
Shape: (51, 7)
Columns: ['pair_idx', 'ligand_identity', 'pose_number', 'canonical_smiles', 'inchikey', 'heavy_atoms', 'bonds']


,pair_idx,ligand_identity,pose_number,canonical_smiles,inchikey,heavy_atoms,bonds
0,1735676,1uk0_frm,1,Cc1cccc2c(=O)[nH]c(CCCN3CC=C(c4ccc(F)cc4)CC3)nc12,LOFDUAJQRUYHBR-UHFFFAOYSA-N,28,31
1,1735677,1uk1_frq,1,O=c1[nH]c(=O)n(CCCCN2CC=C(c3ccccc3)CC2)c2cccc(...,PNPFDRCIGCUCMN-UHFFFAOYSA-N,29,32
2,1735678,1wok_cnq,1,NC(=O)c1cccc2ncc(-c3ccc(Cl)cc3)nc12,FLYGLPYJEQPCFY-UHFFFAOYSA-N,20,22



CELL 228 COMPLETED


In [306]:
import os

print("=" * 65)
print("CELL 229: SEARCH EXISTING SPATIAL VALIDATION FILES")
print("=" * 65)

keywords = [
    "pose_quality",
    "composed_transform",
    "transform_validation",
    "receptor_validation",
    "spatial",
    "alignment"
]

for current_dir, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in [".ipynb_checkpoints", "__pycache__"]
    ]

    for filename in files:
        if filename.lower().endswith((".csv", ".json", ".npy")):
            if any(k in filename.lower() for k in keywords):
                print(os.path.join(current_dir, filename))

print("\nCELL 229 COMPLETED")

CELL 229: SEARCH EXISTING SPATIAL VALIDATION FILES
/content/drive/MyDrive/Member2_Project/final_selected_500/receptor_validation_report.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/experimental_two_chain_alignment_for_ligands.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/target_identification/experimental_chain_alignment_results.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis/spatially_compatible_feature_pairs.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis/ligand_spatial_pharmacophore_summary.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis/M1_conformer_coordinate_alignment_audit.csv
/content/drive/MyDrive/Member2_Project/member2_outputs/spatial_pharmacophore_analysis/M1_conformer_alignment_transformations.json
/content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/local_pocket_alignment_audit.csv
/content/dri

In [307]:
import gzip
import numpy as np
from rdkit import Chem
from Bio.PDB import PDBParser

print("=" * 65)
print("CELL 230: LIGAND AND M1 COORDINATE INSPECTION")
print("=" * 65)

# --------------------------------------------------
# 1. Inspect ligand SDF
# --------------------------------------------------

sdf_path = os.path.join(
    OUT,
    "crossdocked_parp1_ligands_shard23",
    "1735676.sdf.gz"
)

print("\nLIGAND FILE:", sdf_path)

if os.path.exists(sdf_path):
    with gzip.open(sdf_path, "rb") as f:
        supplier = Chem.ForwardSDMolSupplier(
            f,
            removeHs=False,
            sanitize=False
        )

        molecules = [
            mol for mol in supplier
            if mol is not None
        ]

    print("Number of valid readable SDF records:", len(molecules))

    for i, mol in enumerate(molecules[:5]):
        print(f"\nPose {i}")
        print("Atoms:", mol.GetNumAtoms())
        print("Conformers:", mol.GetNumConformers())

        if mol.GetNumConformers() > 0:
            coords = np.array(
                mol.GetConformer().GetPositions()
            )

            print("Coordinate minimum:", coords.min(axis=0))
            print("Coordinate maximum:", coords.max(axis=0))
            print("Coordinate centroid:", coords.mean(axis=0))
else:
    print("Ligand SDF file not found.")

# --------------------------------------------------
# 2. Inspect five M1 conformer coordinates
# --------------------------------------------------

parser = PDBParser(QUIET=True)

m1_dir = os.path.join(
    ROOT,
    "member1_input",
    "module1 parp1 files"
)

print("\n\nM1 CONFORMER COORDINATES")

for i in range(1, 6):
    pdb_path = os.path.join(
        m1_dir,
        f"conformer_{i:03d}.pdb"
    )

    if not os.path.exists(pdb_path):
        print(f"Missing: {pdb_path}")
        continue

    structure = parser.get_structure(
        f"conformer_{i}",
        pdb_path
    )

    coords = np.array([
        atom.coord
        for atom in structure.get_atoms()
        if atom.element not in ["H", "D"]
    ])

    print(f"\nConformer {i}")
    print("Heavy atoms:", len(coords))
    print("Coordinate minimum:", coords.min(axis=0))
    print("Coordinate maximum:", coords.max(axis=0))
    print("Coordinate centroid:", coords.mean(axis=0))

# --------------------------------------------------
# 3. Inspect M1 pharmacophore coordinates
# --------------------------------------------------

tensor_path = os.path.join(
    m1_dir,
    "pocket_1_pharmacophore(1).npy"
)

pocket_tensor = np.load(tensor_path)

print("\n\nM1 PHARMACOPHORE COORDINATES")
print("Tensor shape:", pocket_tensor.shape)

for i in range(pocket_tensor.shape[0]):
    xyz = pocket_tensor[i, :, :3]

    print(f"\nPocket conformer {i + 1}")
    print("Coordinate minimum:", xyz.min(axis=0))
    print("Coordinate maximum:", xyz.max(axis=0))
    print("Coordinate centroid:", xyz.mean(axis=0))

print("\nCELL 230 COMPLETED")

CELL 230: LIGAND AND M1 COORDINATE INSPECTION

LIGAND FILE: /content/drive/MyDrive/Member2_Project/member2_outputs/crossdocked_parp1_ligands_shard23/1735676.sdf.gz
Number of valid readable SDF records: 1

Pose 0
Atoms: 29
Conformers: 1
Coordinate minimum: [-35.617  -29.1531 -48.7129]
Coordinate maximum: [-27.7553 -18.3376 -37.6179]
Coordinate centroid: [-31.42145172 -22.88113793 -42.85994828]


M1 CONFORMER COORDINATES

Conformer 1
Heavy atoms: 5445
Coordinate minimum: [-19.443  -0.497 -19.677]
Coordinate maximum: [60.549 71.259 67.652]
Coordinate centroid: [20.33756  35.493855 21.386757]

Conformer 2
Heavy atoms: 5445
Coordinate minimum: [-19.876   0.205 -19.558]
Coordinate maximum: [59.254 71.197 67.898]
Coordinate centroid: [20.334486 35.497715 21.390053]

Conformer 3
Heavy atoms: 5445
Coordinate minimum: [-20.368   0.599 -19.016]
Coordinate maximum: [58.625 70.915 67.746]
Coordinate centroid: [20.339396 35.493427 21.385246]

Conformer 4
Heavy atoms: 5445
Coordinate minimum: [-19.86

In [308]:
import json
import os

print("=" * 65)
print("CELL 231: COMPOSED TRANSFORMATION INSPECTION")
print("=" * 65)

transform_path = os.path.join(
    OUT,
    "parp1_receptor_validation",
    "crossdocked_receptor_to_m1_composed_transform.json"
)

with open(transform_path, "r") as f:
    transform_data = json.load(f)

print("Transformation file:", transform_path)
print("\nJSON CONTENT:")

print(json.dumps(transform_data, indent=2)[:12000])

print("\nCELL 231 COMPLETED")

CELL 231: COMPOSED TRANSFORMATION INSPECTION
Transformation file: /content/drive/MyDrive/Member2_Project/member2_outputs/parp1_receptor_validation/crossdocked_receptor_to_m1_composed_transform.json

JSON CONTENT:
{
  "method": "Global receptor-to-M1 transform composed with local pocket correction",
  "rotation": [
    [
      -0.5024522402708655,
      0.8236847178334379,
      -0.26284069672415783
    ],
    [
      0.4713457154677987,
      0.006097337215188024,
      -0.8819274567610614
    ],
    [
      -0.7248275400099934,
      -0.5670152626574844,
      -0.39130388339055083
    ]
  ],
  "translation": [
    -24.81954352130499,
    44.5063919196756,
    -37.536795501855515
  ],
  "global_rmsd_A": null,
  "local_inlier_rmsd_A": 0.6847124644968873,
  "local_all_pocket_rmsd_A": 0.8328356210442559
}

CELL 231 COMPLETED


In [309]:
import pandas as pd
import os

print("=" * 65)
print("CELL 232: COMPOSED TRANSFORM VALIDATION AUDIT")
print("=" * 65)

validation_path = os.path.join(
    OUT,
    "parp1_receptor_validation",
    "final_composed_transform_validation.csv"
)

geometry_path = os.path.join(
    OUT,
    "parp1_receptor_validation",
    "composed_transform_ligand_geometry.csv"
)

quality_path = os.path.join(
    OUT,
    "parp1_receptor_validation",
    "crossdocked_pose_quality_manifest.csv"
)

for path in [validation_path, geometry_path, quality_path]:

    print("\n" + "-" * 60)
    print("FILE:", os.path.basename(path))
    print("Exists:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "within_2A" in df.columns:
        print("\nWithin 2 Å:")
        print(df["within_2A"].value_counts(dropna=False))

    if "pose_quality" in df.columns:
        print("\nPose quality:")
        print(df["pose_quality"].value_counts(dropna=False))

    if "usable_for_spatial_screening" in df.columns:
        print("\nUsable for spatial screening:")
        print(
            df["usable_for_spatial_screening"]
            .value_counts(dropna=False)
        )

    display(df.head(5))

print("\nCELL 232 COMPLETED")

CELL 232: COMPOSED TRANSFORM VALIDATION AUDIT

------------------------------------------------------------
FILE: final_composed_transform_validation.csv
Exists: True
Shape: (464, 13)
Columns: ['residue_number', 'residue_name', 'atom_name', 'receptor_x', 'receptor_y', 'receptor_z', 'm1_x', 'm1_y', 'm1_z', 'distance_A', 'is_pocket_residue', 'composed_transform_residual_A', 'within_2A']

Within 2 Å:
within_2A
True     453
False     11
Name: count, dtype: int64


,residue_number,residue_name,atom_name,receptor_x,receptor_y,receptor_z,m1_x,m1_y,m1_z,distance_A,is_pocket_residue,composed_transform_residual_A,within_2A
0,684,LYS,N,2.004942,31.994062,-4.943134,1.286,31.927,-5.033,0.727634,True,0.935108,True
1,684,LYS,CA,3.438969,31.765540,-5.031914,2.637,31.459,-5.312,0.903089,True,1.122713,True
2,684,LYS,C,4.417419,31.890643,-3.773755,3.529,31.633,-4.070,0.971303,True,1.188782,True
3,684,LYS,O,5.305559,31.071218,-3.501689,4.292,30.728,-3.756,1.099897,True,1.311702,True
4,684,LYS,CB,3.808320,32.703650,-6.162918,3.236,32.235,-6.510,0.817097,True,1.018979,True



------------------------------------------------------------
FILE: composed_transform_ligand_geometry.csv
Exists: True
Shape: (51, 9)
Columns: ['pair_idx', 'ligand_identity', 'pose_index', 'min_pocket_distance_A', 'pocket_pairs_le_4A', 'pocket_pairs_le_5A', 'ligand_atoms_le_5A', 'mean_shift_from_global_A', 'max_shift_from_global_A']


,pair_idx,ligand_identity,pose_index,min_pocket_distance_A,pocket_pairs_le_4A,pocket_pairs_le_5A,ligand_atoms_le_5A,mean_shift_from_global_A,max_shift_from_global_A
0,1735676,1uk0_frm,0,2.900583,75,250,19,0.228818,0.282216
1,1735677,1uk1_frq,0,2.774572,85,265,19,0.239882,0.279740
2,1735678,1wok_cnq,0,2.632791,79,251,20,0.228567,0.276471
3,1735679,2rcw_aai,0,2.629732,86,274,21,0.263104,0.334199
4,1735680,2rd6_78p,0,2.842654,78,252,18,0.244356,0.297091



------------------------------------------------------------
FILE: crossdocked_pose_quality_manifest.csv
Exists: True
Shape: (51, 21)
Columns: ['pair_idx', 'ligand_identity', 'pose_index', 'atom_count', 'bond_count', 'geometry_error_A', 'reopened_validation', 'output_file', 'ligand_atoms', 'min_distance_A', 'atom_pairs_le_2A', 'atom_pairs_le_3A', 'atom_pairs_le_4A', 'atom_pairs_le_5A', 'ligand_atoms_le_3A', 'ligand_atoms_le_4A', 'ligand_atoms_le_5A', 'severe_overlap', 'has_receptor_contacts', 'pose_quality', 'usable_for_spatial_screening']

Pose quality:
pose_quality
PASS              39
SEVERE_OVERLAP    12
Name: count, dtype: int64

Usable for spatial screening:
usable_for_spatial_screening
True     39
False    12
Name: count, dtype: int64


,pair_idx,ligand_identity,pose_index,atom_count,bond_count,geometry_error_A,reopened_validation,output_file,ligand_atoms,min_distance_A,...,atom_pairs_le_3A,atom_pairs_le_4A,atom_pairs_le_5A,ligand_atoms_le_3A,ligand_atoms_le_4A,ligand_atoms_le_5A,severe_overlap,has_receptor_contacts,pose_quality,usable_for_spatial_screening
0,1735676,1uk0_frm,0,28,31,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,28,2.811002,...,1,101,371,1,26,28,False,True,PASS,True
1,1735677,1uk1_frq,0,29,32,1.243450e-14,True,/content/drive/MyDrive/Member2_Project/member2...,29,2.802332,...,1,99,372,1,27,29,False,True,PASS,True
2,1735678,1wok_cnq,0,20,22,1.065814e-14,True,/content/drive/MyDrive/Member2_Project/member2...,20,2.958304,...,2,61,262,2,15,20,False,True,PASS,True
3,1735679,2rcw_aai,0,21,23,1.065814e-14,True,/content/drive/MyDrive/Member2_Project/member2...,21,2.686815,...,1,88,323,1,20,21,False,True,PASS,True
4,1735680,2rd6_78p,0,18,20,1.509903e-14,True,/content/drive/MyDrive/Member2_Project/member2...,18,2.899166,...,2,62,250,2,15,18,False,True,PASS,True



CELL 232 COMPLETED


In [310]:
import os
import pandas as pd

print("=" * 65)
print("CELL 233: QUALITY FILTERING OF POCKET-CONDITIONED DATA")
print("=" * 65)

quality_path = os.path.join(
    OUT,
    "parp1_receptor_validation",
    "crossdocked_pose_quality_manifest.csv"
)

quality = pd.read_csv(quality_path)

# Match each manifest record with its corresponding ligand pose.
merged = manifest.merge(
    quality[
        [
            "pair_idx",
            "pose_index",
            "pose_quality",
            "usable_for_spatial_screening",
            "severe_overlap",
            "min_distance_A"
        ]
    ],
    on=["pair_idx", "pose_index"],
    how="left",
    indicator=True
)

print("Total manifest rows:", len(merged))
print("Matched quality records:", (merged["_merge"] == "both").sum())
print("Unmatched records:", (merged["_merge"] != "both").sum())

print("\nQUALITY DISTRIBUTION")
print(merged["pose_quality"].value_counts(dropna=False))

print("\nUSABLE RECORDS BY SPLIT")
print(
    pd.crosstab(
        merged["split"],
        merged["usable_for_spatial_screening"],
        dropna=False
    )
)

print("\nUNIQUE LIGANDS BY QUALITY")
print(
    merged.groupby("pose_quality")["ligand_identity"]
    .nunique()
)

print("\nLIGANDS WITH SEVERE OVERLAP")
bad = merged[
    merged["pose_quality"] == "SEVERE_OVERLAP"
]

display(
    bad[
        [
            "pair_idx",
            "ligand_identity",
            "pose_index",
            "conformer_id",
            "split",
            "min_distance_A"
        ]
    ].drop_duplicates()
)

print("\nCELL 233 COMPLETED")

CELL 233: QUALITY FILTERING OF POCKET-CONDITIONED DATA
Total manifest rows: 195
Matched quality records: 195
Unmatched records: 0

QUALITY DISTRIBUTION
pose_quality
PASS    195
Name: count, dtype: int64

USABLE RECORDS BY SPLIT
usable_for_spatial_screening  True
split                             
train                          160
validation                      35

UNIQUE LIGANDS BY QUALITY
pose_quality
PASS    32
Name: ligand_identity, dtype: int64

LIGANDS WITH SEVERE OVERLAP


,pair_idx,ligand_identity,pose_index,conformer_id,split,min_distance_A



CELL 233 COMPLETED


In [311]:
import os
import pandas as pd

print("=" * 65)
print("CELL 234: FIVE-CONFORMER SPATIAL VALIDATION")
print("=" * 65)

folder = os.path.join(
    OUT,
    "parp1_receptor_validation"
)

files = [
    "spatial_pharmacophore_matches_5conformers.csv",
    "spatial_pharmacophore_pose_processing.csv",
    "spatial_pharmacophore_conformer_summary.csv"
]

for filename in files:

    path = os.path.join(folder, filename)

    print("\n" + "-" * 60)
    print("FILE:", filename)
    print("Exists:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    # Show useful categorical summaries.
    for col in df.columns:
        if any(word in col.lower() for word in [
            "status", "result", "pass", "conformer",
            "compatible", "quality"
        ]):
            if df[col].nunique() <= 20:
                print(f"\n{col} distribution:")
                print(df[col].value_counts(dropna=False))

    display(df.head(5))

print("\nCELL 234 COMPLETED")

CELL 234: FIVE-CONFORMER SPATIAL VALIDATION

------------------------------------------------------------
FILE: spatial_pharmacophore_matches_5conformers.csv
Exists: True
Shape: (2314, 12)
Columns: ['pair_idx', 'ligand_identity', 'pose_index', 'conformer', 'ligand_feature_index', 'ligand_feature_type', 'ligand_atom_ids', 'receptor_feature_index', 'receptor_feature_type', 'receptor_identity', 'distance_A', 'cutoff_A']

conformer distribution:
conformer
1    514
4    480
5    468
2    439
3    413
Name: count, dtype: int64


,pair_idx,ligand_identity,pose_index,conformer,ligand_feature_index,ligand_feature_type,ligand_atom_ids,receptor_feature_index,receptor_feature_type,receptor_identity,distance_A,cutoff_A
0,1735676,1uk0_frm,0,1,10,HYDROPHOBIC,3,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,4.171928,4.5
1,1735676,1uk0_frm,0,1,10,HYDROPHOBIC,3,25,HYDROPHOBIC,A|907||TYR|CE1,4.162290,4.5
2,1735676,1uk0_frm,0,1,10,HYDROPHOBIC,3,26,HYDROPHOBIC,A|907||TYR|CE2,3.756886,4.5
3,1735676,1uk0_frm,0,1,11,HYDROPHOBIC,5,23,HYDROPHOBIC,A|907||TYR|CD1;A|907||TYR|CD2;A|907||TYR|CE1;A...,3.967455,4.5
4,1735676,1uk0_frm,0,1,11,HYDROPHOBIC,5,24,HYDROPHOBIC,A|907||TYR|CD2,3.792064,4.5



------------------------------------------------------------
FILE: spatial_pharmacophore_pose_processing.csv
Exists: True
Shape: (39, 5)
Columns: ['pair_idx', 'ligand_identity', 'pose_index', 'ligand_feature_count', 'matches_across_five_conformers']


,pair_idx,ligand_identity,pose_index,ligand_feature_count,matches_across_five_conformers
0,1735676,1uk0_frm,0,18,82
1,1735677,1uk1_frq,0,17,40
2,1735678,1wok_cnq,0,12,58
3,1735679,2rcw_aai,0,14,49
4,1735680,2rd6_78p,0,12,46



------------------------------------------------------------
FILE: spatial_pharmacophore_conformer_summary.csv
Exists: True
Shape: (5, 6)
Columns: ['conformer', 'compatible_feature_matches', 'unique_ligands', 'unique_poses', 'mean_distance_A', 'median_distance_A']

conformer distribution:
conformer
1    1
2    1
3    1
4    1
5    1
Name: count, dtype: int64

compatible_feature_matches distribution:
compatible_feature_matches
514    1
439    1
413    1
480    1
468    1
Name: count, dtype: int64


,conformer,compatible_feature_matches,unique_ligands,unique_poses,mean_distance_A,median_distance_A
0,1,514,32,514,3.956031,3.977736
1,2,439,32,439,3.951412,4.000870
2,3,413,32,413,4.003876,4.030564
3,4,480,32,480,4.003383,4.017053
4,5,468,32,468,3.985159,4.006742



CELL 234 COMPLETED


In [312]:
print("=" * 65)
print("CELL 235: QUALITY-APPROVED POSE MATCHING AUDIT")
print("=" * 65)

quality = pd.read_csv(
    os.path.join(
        OUT,
        "parp1_receptor_validation",
        "crossdocked_pose_quality_manifest.csv"
    )
)

spatial = pd.read_csv(
    os.path.join(
        OUT,
        "parp1_receptor_validation",
        "spatial_pharmacophore_matches_5conformers.csv"
    )
)

processing = pd.read_csv(
    os.path.join(
        OUT,
        "parp1_receptor_validation",
        "spatial_pharmacophore_pose_processing.csv"
    )
)

# Unique pose identifiers
quality_poses = set(
    zip(quality["pair_idx"], quality["pose_index"])
)

passed_poses = set(
    zip(
        quality.loc[
            quality["usable_for_spatial_screening"].astype(str).str.lower()
            == "true",
            "pair_idx"
        ],
        quality.loc[
            quality["usable_for_spatial_screening"].astype(str).str.lower()
            == "true",
            "pose_index"
        ]
    )
)

spatial_poses = set(
    zip(spatial["pair_idx"], spatial["pose_index"])
)

processing_poses = set(
    zip(processing["pair_idx"], processing["pose_index"])
)

print("Total quality-checked poses:", len(quality_poses))
print("Quality-approved poses:", len(passed_poses))
print("Poses with spatial matches:", len(spatial_poses))
print("Poses in processing report:", len(processing_poses))

print("\nAPPROVED POSE COVERAGE")
print("Approved poses with spatial matches:",
      len(passed_poses & spatial_poses))

print("Approved poses missing spatial matches:",
      len(passed_poses - spatial_poses))

print("\nPROCESSING REPORT CONSISTENCY")
print("Approved poses present in processing report:",
      len(passed_poses & processing_poses))

print("Processing poses not in approved set:",
      len(processing_poses - passed_poses))

print("\nCELL 235 COMPLETED")

CELL 235: QUALITY-APPROVED POSE MATCHING AUDIT
Total quality-checked poses: 51
Quality-approved poses: 39
Poses with spatial matches: 37
Poses in processing report: 39

APPROVED POSE COVERAGE
Approved poses with spatial matches: 37
Approved poses missing spatial matches: 2

PROCESSING REPORT CONSISTENCY
Approved poses present in processing report: 39
Processing poses not in approved set: 0

CELL 235 COMPLETED


In [313]:
print("=" * 65)
print("CELL 236: MANIFEST-LEVEL SPATIAL MATCH COVERAGE")
print("=" * 65)

# Count feature matches for each pose and M1 conformer.
match_counts = (
    spatial.groupby(
        ["pair_idx", "pose_index", "conformer"]
    )
    .size()
    .reset_index(name="feature_match_count")
)

# Map M1 conformer IDs to spatial-analysis conformer IDs.
manifest_check = manifest.copy()

manifest_check["conformer"] = (
    manifest_check["conformer_id"].astype(int)
)

coverage = manifest_check.merge(
    match_counts,
    on=["pair_idx", "pose_index", "conformer"],
    how="left"
)

coverage["feature_match_count"] = (
    coverage["feature_match_count"].fillna(0).astype(int)
)

print("Manifest records:", len(coverage))
print("Records with spatial matches:",
      (coverage["feature_match_count"] > 0).sum())
print("Records without spatial matches:",
      (coverage["feature_match_count"] == 0).sum())

print("\nMATCH COUNTS BY M1 CONFORMER")
display(
    coverage.groupby("conformer")["feature_match_count"]
    .agg(["count", "sum", "mean", "min", "max"])
)

print("\nMATCH COUNTS BY DATA SPLIT")
display(
    coverage.groupby("split")["feature_match_count"]
    .agg(["count", "sum", "mean", "min", "max"])
)

print("\nRECORDS WITHOUT SPATIAL MATCHES")
display(
    coverage[
        coverage["feature_match_count"] == 0
    ][
        [
            "pair_idx",
            "ligand_identity",
            "pose_index",
            "conformer_id",
            "split"
        ]
    ].head(20)
)

print("\nCELL 236 COMPLETED")

CELL 236: MANIFEST-LEVEL SPATIAL MATCH COVERAGE
Manifest records: 195
Records with spatial matches: 184
Records without spatial matches: 11

MATCH COUNTS BY M1 CONFORMER


,count,sum,mean,min,max
conformer,,,,,
1,39,514,13.179487,0,28
2,39,439,11.256410,0,23
3,39,413,10.589744,0,21
4,39,480,12.307692,0,25
5,39,468,12.000000,0,24



MATCH COUNTS BY DATA SPLIT


,count,sum,mean,min,max
split,,,,,
train,160,1829,11.431250,0,26
validation,35,485,13.857143,6,28



RECORDS WITHOUT SPATIAL MATCHES


,pair_idx,ligand_identity,pose_index,conformer_id,split
70,1735686,4hhy_15r,12,1,train
71,1735686,4hhy_15r,12,2,train
72,1735686,4hhy_15r,12,3,train
73,1735686,4hhy_15r,12,4,train
74,1735686,4hhy_15r,12,5,train
75,1735686,4hhy_15r,14,1,train
76,1735686,4hhy_15r,14,2,train
77,1735686,4hhy_15r,14,3,train
78,1735686,4hhy_15r,14,4,train
79,1735686,4hhy_15r,14,5,train



CELL 236 COMPLETED


In [314]:
print("=" * 65)
print("CELL 237: INVESTIGATE UNMATCHED LIGAND POSES")
print("=" * 65)

target_pair = 1735686

print("\n1. MANIFEST RECORDS")

display(
    manifest[
        manifest["pair_idx"] == target_pair
    ].sort_values(["pose_index", "conformer_id"])
)

print("\n2. QUALITY MANIFEST")

display(
    quality[
        quality["pair_idx"] == target_pair
    ]
)

print("\n3. SPATIAL PROCESSING REPORT")

display(
    processing[
        processing["pair_idx"] == target_pair
    ]
)

print("\n4. SPATIAL MATCH COUNTS")

display(
    spatial[
        spatial["pair_idx"] == target_pair
    ]
    .groupby(["pose_index", "conformer"])
    .size()
    .reset_index(name="match_count")
)

print("\nCELL 237 COMPLETED")

CELL 237: INVESTIGATE UNMATCHED LIGAND POSES

1. MANIFEST RECORDS


,pair_idx,ligand_identity,inchikey,pose_index,conformer_id,canonical_smiles,ligand_atoms,ligand_bonds,split
50,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,0,1,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
51,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,0,2,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
52,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,0,3,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
53,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,0,4,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
54,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,0,5,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
55,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,1,1,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
56,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,1,2,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
57,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,1,3,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
58,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,1,4,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train
59,1735686,4hhy_15r,CSCBNPVHZXDTDS-NDEPHWFRSA-N,1,5,O=C1NC[C@H]2c3c(cccc31)CCN2C(=O)C1CCN(C(=O)c2c...,43,49,train



2. QUALITY MANIFEST


,pair_idx,ligand_identity,pose_index,atom_count,bond_count,geometry_error_A,reopened_validation,output_file,ligand_atoms,min_distance_A,...,atom_pairs_le_3A,atom_pairs_le_4A,atom_pairs_le_5A,ligand_atoms_le_3A,ligand_atoms_le_4A,ligand_atoms_le_5A,severe_overlap,has_receptor_contacts,pose_quality,usable_for_spatial_screening
10,1735686,4hhy_15r,0,43,49,1.598721e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,2.175414,...,9,138,521,8,39,43,False,True,PASS,True
11,1735686,4hhy_15r,1,43,49,1.598721e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,2.184660,...,16,131,410,10,36,43,False,True,PASS,True
12,1735686,4hhy_15r,2,43,49,2.309264e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.406274,...,30,158,418,17,36,40,True,True,SEVERE_OVERLAP,False
13,1735686,4hhy_15r,3,43,49,1.776357e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,2.859412,...,5,103,439,5,34,43,False,True,PASS,True
14,1735686,4hhy_15r,4,43,49,1.421085e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,2.263392,...,6,150,568,5,37,43,False,True,PASS,True
15,1735686,4hhy_15r,5,43,49,1.332268e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.959864,...,12,119,439,9,35,43,True,True,SEVERE_OVERLAP,False
16,1735686,4hhy_15r,6,43,49,1.776357e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.721758,...,16,72,216,10,27,41,True,True,SEVERE_OVERLAP,False
17,1735686,4hhy_15r,7,43,49,1.953993e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.245601,...,28,140,416,17,38,43,True,True,SEVERE_OVERLAP,False
18,1735686,4hhy_15r,8,43,49,1.776357e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.858311,...,17,127,409,12,39,43,True,True,SEVERE_OVERLAP,False
19,1735686,4hhy_15r,9,43,49,2.486900e-14,True,/content/drive/MyDrive/Member2_Project/member2...,43,1.759288,...,27,144,428,18,38,42,True,True,SEVERE_OVERLAP,False



3. SPATIAL PROCESSING REPORT


,pair_idx,ligand_identity,pose_index,ligand_feature_count,matches_across_five_conformers
10,1735686,4hhy_15r,0,27,110
11,1735686,4hhy_15r,1,27,92
12,1735686,4hhy_15r,3,27,113
13,1735686,4hhy_15r,4,27,77
14,1735686,4hhy_15r,12,27,0
15,1735686,4hhy_15r,14,27,0
16,1735686,4hhy_15r,15,27,54
17,1735686,4hhy_15r,17,27,16



4. SPATIAL MATCH COUNTS


,pose_index,conformer,match_count
0,0,1,26
1,0,2,22
2,0,3,19
3,0,4,22
4,0,5,21
5,1,1,19
6,1,2,19
7,1,3,18
8,1,4,18
9,1,5,18



CELL 237 COMPLETED


In [315]:
print("=" * 65)
print("CELL 238: POSE-INDEX CONSISTENCY AUDIT")
print("=" * 65)

sources = {
    "Manifest": manifest,
    "Quality": quality,
    "Spatial processing": processing,
    "Spatial matches": spatial
}

for name, df in sources.items():

    print("\n" + "-" * 55)
    print(name)

    subset = df[df["pair_idx"] == target_pair]

    print("Rows:", len(subset))

    if "pose_index" in subset.columns:
        print(
            "Pose indices:",
            sorted(subset["pose_index"].dropna().unique().tolist())
        )

    if "conformer_id" in subset.columns:
        print(
            "Conformer IDs:",
            sorted(subset["conformer_id"].dropna().unique().tolist())
        )

    if "conformer" in subset.columns:
        print(
            "Conformers:",
            sorted(subset["conformer"].dropna().unique().tolist())
        )

print("\nCELL 238 COMPLETED")

CELL 238: POSE-INDEX CONSISTENCY AUDIT

-------------------------------------------------------
Manifest
Rows: 40
Pose indices: [0, 1, 3, 4, 12, 14, 15, 17]
Conformer IDs: [1, 2, 3, 4, 5]

-------------------------------------------------------
Quality
Rows: 20
Pose indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

-------------------------------------------------------
Spatial processing
Rows: 8
Pose indices: [0, 1, 3, 4, 12, 14, 15, 17]

-------------------------------------------------------
Spatial matches
Rows: 462
Pose indices: [0, 1, 3, 4, 15, 17]
Conformers: [1, 2, 3, 4, 5]

CELL 238 COMPLETED


In [316]:
# ============================================================
# CELL 239: FINAL POSE SUPPORT SUMMARY
# ============================================================

import pandas as pd
import os

ROOT = "/content/drive/MyDrive/Member2_Project/member2_outputs"

VALIDATION_DIR = os.path.join(
    ROOT, "parp1_receptor_validation"
)

# Load existing reports
manifest = pd.read_csv(
    os.path.join(ROOT, "pocket_conditioned_training_manifest.csv")
)

quality = pd.read_csv(
    os.path.join(VALIDATION_DIR, "crossdocked_pose_quality_manifest.csv")
)

spatial = pd.read_csv(
    os.path.join(
        VALIDATION_DIR,
        "spatial_pharmacophore_matches_5conformers.csv"
    )
)

# Focus on ligand 4hhy_15r
pair_id = 1735686

selected = manifest[
    manifest["pair_idx"] == pair_id
][["pair_idx", "ligand_identity", "pose_index"]].drop_duplicates()

quality_selected = quality[
    quality["pair_idx"] == pair_id
][[
    "pair_idx", "pose_index",
    "pose_quality", "usable_for_spatial_screening",
    "min_distance_A", "severe_overlap"
]]

# Count spatial matches for each pose and conformer
match_counts = (
    spatial[spatial["pair_idx"] == pair_id]
    .groupby(["pair_idx", "pose_index", "conformer"])
    .size()
    .reset_index(name="match_count")
)

# Build complete pose-conformer combinations
conformers = pd.DataFrame({"conformer": [1, 2, 3, 4, 5]})

pose_conformers = selected.merge(conformers, how="cross")

pose_conformers = pose_conformers.merge(
    match_counts,
    on=["pair_idx", "pose_index", "conformer"],
    how="left"
)

pose_conformers["match_count"] = (
    pose_conformers["match_count"].fillna(0).astype(int)
)

# Aggregate across five conformers
support = (
    pose_conformers.groupby(
        ["pair_idx", "ligand_identity", "pose_index"]
    )
    .agg(
        total_matches=("match_count", "sum"),
        conformers_with_matches=(
            "match_count",
            lambda x: (x > 0).sum()
        ),
        minimum_matches=("match_count", "min"),
        maximum_matches=("match_count", "max")
    )
    .reset_index()
)

# Combine geometry and spatial evidence
summary = selected.merge(
    quality_selected,
    on=["pair_idx", "pose_index"],
    how="left"
).merge(
    support,
    on=["pair_idx", "ligand_identity", "pose_index"],
    how="left"
)

summary["spatial_support"] = summary[
    "conformers_with_matches"
].map(
    lambda x: (
        "ALL 5 CONFORMERS" if x == 5
        else "PARTIAL SUPPORT" if x > 0
        else "NO SUPPORT"
    )
)

print("=" * 65)
print("CELL 239: FINAL POSE SUPPORT SUMMARY")
print("=" * 65)

print(summary.to_string(index=False))

print("\nSUMMARY")
print("Total selected poses:", len(summary))
print(
    "All 5 conformers supported:",
    (summary["conformers_with_matches"] == 5).sum()
)
print(
    "Partially supported:",
    (
        (summary["conformers_with_matches"] > 0) &
        (summary["conformers_with_matches"] < 5)
    ).sum()
)
print(
    "No spatial support:",
    (summary["conformers_with_matches"] == 0).sum()
)

print("\nCELL 239 COMPLETED")

CELL 239: FINAL POSE SUPPORT SUMMARY
 pair_idx ligand_identity  pose_index pose_quality  usable_for_spatial_screening  min_distance_A  severe_overlap  total_matches  conformers_with_matches  minimum_matches  maximum_matches  spatial_support
  1735686        4hhy_15r           0         PASS                          True        2.175414           False            110                        5               19               26 ALL 5 CONFORMERS
  1735686        4hhy_15r           1         PASS                          True        2.184660           False             92                        5               18               19 ALL 5 CONFORMERS
  1735686        4hhy_15r           3         PASS                          True        2.859412           False            113                        5               20               25 ALL 5 CONFORMERS
  1735686        4hhy_15r           4         PASS                          True        2.263392           False             77                    

In [317]:
# ============================================================
# CELL 240: ALL-LIGAND POSE SUPPORT AUDIT
# ============================================================

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project/member2_outputs"
VAL_DIR = os.path.join(ROOT, "parp1_receptor_validation")

# Load existing files
manifest = pd.read_csv(
    os.path.join(ROOT, "pocket_conditioned_training_manifest.csv")
)

quality = pd.read_csv(
    os.path.join(VAL_DIR, "crossdocked_pose_quality_manifest.csv")
)

spatial = pd.read_csv(
    os.path.join(VAL_DIR, "spatial_pharmacophore_matches_5conformers.csv")
)

# Unique selected ligand-pose combinations
poses = manifest[
    ["pair_idx", "ligand_identity", "pose_index"]
].drop_duplicates()

# Count matches for every pose across each conformer
match_counts = (
    spatial.groupby(
        ["pair_idx", "pose_index", "conformer"]
    )
    .size()
    .reset_index(name="match_count")
)

# Generate all five conformer combinations
conformers = pd.DataFrame({
    "conformer": [1, 2, 3, 4, 5]
})

pose_conformers = poses.merge(conformers, how="cross")

pose_conformers = pose_conformers.merge(
    match_counts,
    on=["pair_idx", "pose_index", "conformer"],
    how="left"
)

pose_conformers["match_count"] = (
    pose_conformers["match_count"].fillna(0).astype(int)
)

# Aggregate spatial support
support = (
    pose_conformers.groupby(
        ["pair_idx", "ligand_identity", "pose_index"]
    )
    .agg(
        total_matches=("match_count", "sum"),
        conformers_supported=(
            "match_count",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

# Add geometry quality
quality_cols = [
    "pair_idx", "pose_index",
    "pose_quality", "usable_for_spatial_screening",
    "min_distance_A", "severe_overlap"
]

quality_subset = quality[quality_cols].drop_duplicates(
    subset=["pair_idx", "pose_index"]
)

audit = poses.merge(
    quality_subset,
    on=["pair_idx", "pose_index"],
    how="left"
).merge(
    support,
    on=["pair_idx", "ligand_identity", "pose_index"],
    how="left"
)

# Classify each pose
def classify(row):
    if row["pose_quality"] != "PASS":
        return "REJECT_GEOMETRY"

    if not row["usable_for_spatial_screening"]:
        return "REJECT_GEOMETRY"

    if row["conformers_supported"] == 5:
        return "SUPPORTED_ALL_5"

    if row["conformers_supported"] > 0:
        return "PARTIAL_SUPPORT"

    return "NO_SPATIAL_SUPPORT"

audit["support_category"] = audit.apply(classify, axis=1)

# Save audit
output_path = os.path.join(
    ROOT, "all_ligand_pose_support_audit.csv"
)

audit.to_csv(output_path, index=False)

# Summary
print("=" * 65)
print("CELL 240: ALL-LIGAND POSE SUPPORT AUDIT")
print("=" * 65)

print("Selected ligand-pose combinations:", len(audit))
print("Unique ligands:", audit["ligand_identity"].nunique())

print("\nPOSE SUPPORT CATEGORIES:")
print(audit["support_category"].value_counts(dropna=False))

print("\nUNIQUE LIGANDS PER CATEGORY:")
print(
    audit.groupby("support_category")["ligand_identity"]
    .nunique()
)

print("\nLIGAND-WISE SUMMARY:")
print(
    audit.groupby("ligand_identity")
    .agg(
        selected_poses=("pose_index", "count"),
        fully_supported=("support_category",
                         lambda x: (x == "SUPPORTED_ALL_5").sum()),
        partially_supported=("support_category",
                             lambda x: (x == "PARTIAL_SUPPORT").sum()),
        unsupported=("support_category",
                     lambda x: (x == "NO_SPATIAL_SUPPORT").sum()),
        geometry_rejected=("support_category",
                           lambda x: (x == "REJECT_GEOMETRY").sum())
    )
    .to_string()
)

print("\nSaved:", output_path)
print("\nCELL 240 COMPLETED")

CELL 240: ALL-LIGAND POSE SUPPORT AUDIT
Selected ligand-pose combinations: 39
Unique ligands: 32

POSE SUPPORT CATEGORIES:
support_category
SUPPORTED_ALL_5       36
NO_SPATIAL_SUPPORT     2
PARTIAL_SUPPORT        1
Name: count, dtype: int64

UNIQUE LIGANDS PER CATEGORY:
support_category
NO_SPATIAL_SUPPORT     1
PARTIAL_SUPPORT        1
SUPPORTED_ALL_5       32
Name: ligand_identity, dtype: int64

LIGAND-WISE SUMMARY:
                 selected_poses  fully_supported  partially_supported  unsupported  geometry_rejected
ligand_identity                                                                                      
1uk0_frm                      1                1                    0            0                  0
1uk1_frq                      1                1                    0            0                  0
1wok_cnq                      1                1                    0            0                  0
2rcw_aai                      1                1                    0

In [318]:
# ============================================================
# CELL 241: SELECT ONE BEST POSE PER LIGAND
# ============================================================

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project/member2_outputs"

# Load previous audit and original manifest
audit = pd.read_csv(
    os.path.join(ROOT, "all_ligand_pose_support_audit.csv")
)

manifest = pd.read_csv(
    os.path.join(ROOT, "pocket_conditioned_training_manifest.csv")
)

# Keep only poses supported across all five conformers
eligible = audit[
    (audit["support_category"] == "SUPPORTED_ALL_5")
].copy()

# Calculate average spatial matches per ligand feature per conformer.
# This normalizes the match count by the number of ligand features.
processing = pd.read_csv(
    os.path.join(
        ROOT,
        "parp1_receptor_validation",
        "spatial_pharmacophore_pose_processing.csv"
    )
)

eligible = eligible.merge(
    processing[
        ["pair_idx", "pose_index", "ligand_feature_count"]
    ],
    on=["pair_idx", "pose_index"],
    how="left"
)

eligible["normalized_support"] = (
    eligible["total_matches"] /
    (5 * eligible["ligand_feature_count"])
)

# Select the best-supported pose for each ligand
selected = (
    eligible.sort_values(
        ["ligand_identity", "normalized_support"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["ligand_identity"],
        keep="first"
    )
)

# Retrieve the five conformer records for each selected pose
final_manifest = manifest.merge(
    selected[
        ["pair_idx", "ligand_identity", "pose_index"]
    ],
    on=["pair_idx", "ligand_identity", "pose_index"],
    how="inner"
)

# Save separately
output_path = os.path.join(
    ROOT,
    "pocket_conditioned_training_manifest_selected.csv"
)

final_manifest.to_csv(output_path, index=False)

# Report
print("=" * 65)
print("CELL 241: ONE POSE PER LIGAND")
print("=" * 65)

print("Eligible fully supported poses:", len(eligible))
print("Selected unique ligands:", selected["ligand_identity"].nunique())
print("Selected poses:", len(selected))
print("Final manifest rows:", len(final_manifest))

print("\nTRAIN / VALIDATION SPLIT:")
print(final_manifest["split"].value_counts())

print("\nUNIQUE LIGANDS PER SPLIT:")
print(
    final_manifest.groupby("split")["ligand_identity"].nunique()
)

print("\nSELECTED POSES:")
print(
    selected[
        [
            "ligand_identity",
            "pose_index",
            "total_matches",
            "conformers_supported",
            "normalized_support"
        ]
    ].to_string(index=False)
)

print("\nSaved:", output_path)
print("\nCELL 241 COMPLETED")

CELL 241: ONE POSE PER LIGAND
Eligible fully supported poses: 36
Selected unique ligands: 32
Selected poses: 32
Final manifest rows: 160

TRAIN / VALIDATION SPLIT:
split
train         125
validation     35
Name: count, dtype: int64

UNIQUE LIGANDS PER SPLIT:
split
train         25
validation     7
Name: ligand_identity, dtype: int64

SELECTED POSES:
ligand_identity  pose_index  total_matches  conformers_supported  normalized_support
       1uk0_frm           0             82                     5            0.911111
       1uk1_frq           0             40                     5            0.470588
       1wok_cnq           0             58                     5            0.966667
       2rcw_aai           0             49                     5            0.700000
       2rd6_78p           0             46                     5            0.766667
       3gjw_gjw           0             63                     5            1.400000
       3gn7_3gn           0             50           

In [319]:
# ============================================================
# CELL 242: FINAL MANIFEST INTEGRITY VALIDATION
# ============================================================

import os
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project/member2_outputs"

path = os.path.join(
    ROOT,
    "pocket_conditioned_training_manifest_selected.csv"
)

df = pd.read_csv(path)

print("=" * 65)
print("CELL 242: FINAL MANIFEST INTEGRITY VALIDATION")
print("=" * 65)

# 1. Basic dimensions
print("\n1. DATASET DIMENSIONS")
print("Rows:", len(df))
print("Unique ligands:", df["ligand_identity"].nunique())
print("Unique pair IDs:", df["pair_idx"].nunique())
print("Unique SMILES:", df["canonical_smiles"].nunique())

# 2. Five conformers per ligand
conformer_counts = df.groupby("ligand_identity")["conformer_id"].nunique()

print("\n2. CONFORMER VALIDATION")
print("Conformers per ligand:")
print(conformer_counts.value_counts().to_dict())

print("Every ligand has 5 conformers:",
      (conformer_counts == 5).all())

# 3. One pose per ligand
pose_counts = df.groupby("ligand_identity")["pose_index"].nunique()

print("\n3. POSE VALIDATION")
print("Every ligand has one pose:",
      (pose_counts == 1).all())

# 4. Split integrity
split_counts = df.groupby("ligand_identity")["split"].nunique()

print("\n4. SPLIT LEAKAGE CHECK")
print("Every ligand belongs to one split:",
      (split_counts == 1).all())

print("Training ligands:",
      df[df["split"] == "train"]["ligand_identity"].nunique())

print("Validation ligands:",
      df[df["split"] == "validation"]["ligand_identity"].nunique())

# 5. Missing values
print("\n5. MISSING VALUE CHECK")
print("Total missing values:", df.isnull().sum().sum())

# 6. Duplicate records
print("\n6. DUPLICATE CHECK")
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Duplicate ligand-conformer combinations:",
    df.duplicated(
        subset=["ligand_identity", "conformer_id"]
    ).sum()
)

# 7. Molecular identity consistency
identity_counts = (
    df.groupby("ligand_identity")["canonical_smiles"].nunique()
)

print("\n7. MOLECULAR IDENTITY CHECK")
print("Each ligand has one canonical SMILES:",
      (identity_counts == 1).all())

# 8. Final decision
checks = {
    "32 unique ligands": df["ligand_identity"].nunique() == 32,
    "160 records": len(df) == 160,
    "5 conformers per ligand": (conformer_counts == 5).all(),
    "One pose per ligand": (pose_counts == 1).all(),
    "No split leakage": (split_counts == 1).all(),
    "No missing values": df.isnull().sum().sum() == 0,
    "No duplicate rows": df.duplicated().sum() == 0,
    "Consistent molecular identity": (identity_counts == 1).all()
}

print("\n8. FINAL VALIDATION")
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), "-", name)

print("\nFINAL STATUS:",
      "PASS" if all(checks.values()) else "NEEDS CORRECTION")

print("\nCELL 242 COMPLETED")

CELL 242: FINAL MANIFEST INTEGRITY VALIDATION

1. DATASET DIMENSIONS
Rows: 160
Unique ligands: 32
Unique pair IDs: 32
Unique SMILES: 32

2. CONFORMER VALIDATION
Conformers per ligand:
{5: 32}
Every ligand has 5 conformers: True

3. POSE VALIDATION
Every ligand has one pose: True

4. SPLIT LEAKAGE CHECK
Every ligand belongs to one split: True
Training ligands: 25
Validation ligands: 7

5. MISSING VALUE CHECK
Total missing values: 0

6. DUPLICATE CHECK
Exact duplicate rows: 0
Duplicate ligand-conformer combinations: 0

7. MOLECULAR IDENTITY CHECK
Each ligand has one canonical SMILES: True

8. FINAL VALIDATION
PASS - 32 unique ligands
PASS - 160 records
PASS - 5 conformers per ligand
PASS - One pose per ligand
PASS - No split leakage
PASS - No missing values
PASS - No duplicate rows
PASS - Consistent molecular identity

FINAL STATUS: PASS

CELL 242 COMPLETED


In [320]:
# ============================================================
# CELL 243: PREPARE POCKET-CONDITIONED TRAINING INPUTS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project"

OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")
INPUT_DIR = os.path.join(ROOT, "member1_input", "module1 parp1 files")

# ------------------------------------------------------------
# 1. LOAD SELECTED MANIFEST
# ------------------------------------------------------------

manifest_path = os.path.join(
    OUTPUT_DIR,
    "pocket_conditioned_training_manifest_selected.csv"
)

manifest = pd.read_csv(manifest_path)

# ------------------------------------------------------------
# 2. LOAD MEMBER-1 POCKET TENSORS
# ------------------------------------------------------------

pocket_path = os.path.join(
    INPUT_DIR,
    "pocket_1_pharmacophore(1).npy"
)

mask_path = os.path.join(
    INPUT_DIR,
    "pocket_1_mask(1).npy"
)

pocket_tensor = np.load(pocket_path)
pocket_mask = np.load(mask_path)

print("M1 POCKET SHAPES")
print("Pocket:", pocket_tensor.shape)
print("Mask:", pocket_mask.shape)

assert pocket_tensor.shape == (5, 95, 4)
assert pocket_mask.shape == (5, 95)

# ------------------------------------------------------------
# 3. LOAD EXISTING TOKENIZER
# ------------------------------------------------------------

tokenizer_path = os.path.join(
    OUTPUT_DIR,
    "smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
max_length = tokenizer["max_length"]
special_tokens = tokenizer["special_tokens"]

PAD_ID = special_tokens["PAD"]
BOS_ID = special_tokens["BOS"]
EOS_ID = special_tokens["EOS"]
UNK_ID = special_tokens["UNK"]

# Longest-match tokenization
tokens_sorted = sorted(
    [
        token for token in token_to_id
        if token not in ["PAD", "BOS", "EOS", "UNK"]
    ],
    key=len,
    reverse=True
)

def tokenize_smiles(smiles):

    result = []
    i = 0

    while i < len(smiles):

        matched = False

        for token in tokens_sorted:
            if smiles.startswith(token, i):
                result.append(token_to_id[token])
                i += len(token)
                matched = True
                break

        if not matched:
            return None

    return result

# ------------------------------------------------------------
# 4. TOKENIZE EACH LIGAND
# ------------------------------------------------------------

smiles_cache = {}

for _, row in manifest.iterrows():

    ligand = row["ligand_identity"]
    smiles = row["canonical_smiles"]

    if ligand not in smiles_cache:

        molecular_tokens = tokenize_smiles(smiles)

        if molecular_tokens is None:
            raise ValueError(
                f"Unknown SMILES token found: {ligand}"
            )

        sequence = [BOS_ID] + molecular_tokens + [EOS_ID]

        if len(sequence) > max_length:
            raise ValueError(
                f"Sequence exceeds maximum length: {ligand}"
            )

        sequence += [PAD_ID] * (
            max_length - len(sequence)
        )

        smiles_cache[ligand] = sequence

# ------------------------------------------------------------
# 5. BUILD POCKET-CONDITIONED EXAMPLES
# ------------------------------------------------------------

all_token_ids = []
all_pockets = []
all_masks = []
all_splits = []
all_ligands = []
all_conformers = []

for _, row in manifest.iterrows():

    ligand = row["ligand_identity"]
    conformer_id = int(row["conformer_id"])

    # Conformer IDs are 1-based; NumPy indexing is 0-based.
    conformer_index = conformer_id - 1

    all_token_ids.append(smiles_cache[ligand])

    all_pockets.append(
        pocket_tensor[conformer_index]
    )

    all_masks.append(
        pocket_mask[conformer_index]
    )

    all_splits.append(row["split"])
    all_ligands.append(ligand)
    all_conformers.append(conformer_id)

# Convert to NumPy arrays
all_token_ids = np.asarray(all_token_ids, dtype=np.int64)
all_pockets = np.asarray(all_pockets, dtype=np.float32)
all_masks = np.asarray(all_masks, dtype=np.float32)

all_splits = np.asarray(all_splits)
all_ligands = np.asarray(all_ligands)
all_conformers = np.asarray(all_conformers)

# ------------------------------------------------------------
# 6. SEPARATE TRAINING AND VALIDATION
# ------------------------------------------------------------

train_idx = np.where(all_splits == "train")[0]
val_idx = np.where(all_splits == "validation")[0]

train_data = {
    "token_ids": all_token_ids[train_idx],
    "pockets": all_pockets[train_idx],
    "masks": all_masks[train_idx],
    "ligands": all_ligands[train_idx],
    "conformers": all_conformers[train_idx]
}

val_data = {
    "token_ids": all_token_ids[val_idx],
    "pockets": all_pockets[val_idx],
    "masks": all_masks[val_idx],
    "ligands": all_ligands[val_idx],
    "conformers": all_conformers[val_idx]
}

# ------------------------------------------------------------
# 7. SAVE PREPARED DATA
# ------------------------------------------------------------

train_path = os.path.join(
    OUTPUT_DIR,
    "pocket_conditioned_train_inputs.npz"
)

val_path = os.path.join(
    OUTPUT_DIR,
    "pocket_conditioned_validation_inputs.npz"
)

np.savez_compressed(train_path, **train_data)
np.savez_compressed(val_path, **val_data)

# ------------------------------------------------------------
# 8. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CELL 243: POCKET-CONDITIONED INPUT PREPARATION")
print("=" * 65)

print("\nALL DATA")
print("Token IDs:", all_token_ids.shape)
print("Pocket tensors:", all_pockets.shape)
print("Pocket masks:", all_masks.shape)

print("\nTRAINING DATA")
for key, value in train_data.items():
    print(key, np.asarray(value).shape)

print("\nVALIDATION DATA")
for key, value in val_data.items():
    print(key, np.asarray(value).shape)

print("\nCHECKS")
print("All token IDs within vocabulary:",
      all_token_ids.max() < len(token_to_id))

print("No unknown tokens:",
      UNK_ID not in all_token_ids)

print("No NaN in pocket tensors:",
      np.isfinite(all_pockets).all())

print("No NaN in masks:",
      np.isfinite(all_masks).all())

print("All conformer IDs valid:",
      set(all_conformers).issubset({1, 2, 3, 4, 5}))

print("\nSAVED FILES")
print(train_path)
print(val_path)

print("\nCELL 243 COMPLETED")

M1 POCKET SHAPES
Pocket: (5, 95, 4)
Mask: (5, 95)


TypeError: list indices must be integers or slices, not str

In [321]:
import json
import os

tokenizer_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/smiles_tokenizer.json"
)

with open(tokenizer_path, "r") as f:
    tokenizer = json.load(f)

print("TOKENIZER KEYS:")
print(tokenizer.keys())

print("\nSPECIAL TOKENS:")
print(tokenizer["special_tokens"])

print("\nTOKEN TO ID SAMPLE:")
print(list(tokenizer["token_to_id"].items())[:15])

print("\nMAX LENGTH:")
print(tokenizer["max_length"])

print("\nCELL 243A COMPLETED")

TOKENIZER KEYS:
dict_keys(['vocabulary', 'token_to_id', 'max_length', 'special_tokens'])

SPECIAL TOKENS:
['<PAD>', '<BOS>', '<EOS>', '<UNK>']

TOKEN TO ID SAMPLE:
[('<PAD>', 0), ('<BOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('#', 4), ('%10', 5), ('(', 6), (')', 7), ('-', 8), ('/', 9), ('1', 10), ('2', 11), ('3', 12), ('4', 13), ('5', 14)]

MAX LENGTH:
143

CELL 243A COMPLETED


In [322]:
# Load tokenizer
with open(tokenizer_path, "r") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
vocab_size = len(token_to_id)
max_length = tokenizer["max_length"]

# Read special token IDs directly from vocabulary
PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

# Molecular tokens only
tokens_sorted = sorted(
    [
        token for token in token_to_id
        if token not in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
    ],
    key=len,
    reverse=True
)

print("TOKENIZER LOADED SUCCESSFULLY")
print("Vocabulary size:", vocab_size)
print("Maximum sequence length:", max_length)
print("PAD:", PAD_ID, "BOS:", BOS_ID,
      "EOS:", EOS_ID, "UNK:", UNK_ID)
print("Molecular tokens:", len(tokens_sorted))

TOKENIZER LOADED SUCCESSFULLY
Vocabulary size: 81
Maximum sequence length: 143
PAD: 0 BOS: 1 EOS: 2 UNK: 3
Molecular tokens: 77


In [323]:
import os
import numpy as np

ROOT = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")

train_path = os.path.join(
    OUTPUT_DIR, "pocket_conditioned_train_inputs.npz"
)

val_path = os.path.join(
    OUTPUT_DIR, "pocket_conditioned_validation_inputs.npz"
)

assert os.path.exists(train_path), "Training NPZ not found. Run Cell 243."
assert os.path.exists(val_path), "Validation NPZ not found. Run Cell 243."

def inspect_dataset(path, name):
    data = np.load(path, allow_pickle=True)

    print(f"\n========== {name} DATASET ==========")
    print("Keys:", data.files)

    for key in data.files:
        arr = data[key]
        print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

    tokens = data["token_ids"]
    pockets = data["pockets"]
    masks = data["masks"]
    ligands = data["ligands"]
    conformers = data["conformers"]

    assert tokens.ndim == 2
    assert pockets.shape[1:] == (95, 4)
    assert masks.shape[1:] == (95,)
    assert len(tokens) == len(pockets) == len(masks)
    assert len(tokens) == len(ligands) == len(conformers)

    assert np.isfinite(pockets).all()
    assert np.isfinite(masks).all()

    print("\nIntegrity checks: PASS")
    print("Molecules:", len(set(ligands)))
    print("Conformer IDs:", sorted(set(conformers.tolist())))
    print("UNK tokens:", int(np.sum(tokens == 3)))

    return data

train_data = inspect_dataset(train_path, "TRAINING")
val_data = inspect_dataset(val_path, "VALIDATION")

print("\n========== FINAL CHECK ==========")
print("Training examples:", len(train_data["token_ids"]))
print("Validation examples:", len(val_data["token_ids"]))
print("CELL 244 COMPLETED")

AssertionError: Training NPZ not found. Run Cell 243.

In [324]:
import os

ROOT = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")

print("OUTPUT DIRECTORY EXISTS:", os.path.exists(OUTPUT_DIR))

print("\nPOCKET-CONDITIONED FILES:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if "pocket_conditioned" in f:
        print(f)

print("\nEXPECTED FILES:")
for name in [
    "pocket_conditioned_training_manifest_selected.csv",
    "pocket_conditioned_train_inputs.npz",
    "pocket_conditioned_validation_inputs.npz"
]:
    path = os.path.join(OUTPUT_DIR, name)
    print(name, ":", os.path.exists(path))

OUTPUT DIRECTORY EXISTS: True

POCKET-CONDITIONED FILES:
pocket_conditioned_training_manifest.csv
pocket_conditioned_training_manifest_selected.csv
pocket_conditioned_training_records.jsonl

EXPECTED FILES:
pocket_conditioned_training_manifest_selected.csv : True
pocket_conditioned_train_inputs.npz : False
pocket_conditioned_validation_inputs.npz : False


In [325]:
import os
import json
import numpy as np
import pandas as pd

ROOT = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(ROOT, "member2_outputs")
INPUT_DIR = os.path.join(ROOT, "member1_input", "module1 parp1 files")

# --------------------------------------------------
# 1. LOAD INPUT FILES
# --------------------------------------------------

manifest_path = os.path.join(
    OUTPUT_DIR, "pocket_conditioned_training_manifest_selected.csv"
)

tokenizer_path = os.path.join(
    OUTPUT_DIR, "smiles_tokenizer.json"
)

manifest = pd.read_csv(manifest_path)

with open(tokenizer_path, "r") as f:
    tokenizer = json.load(f)

token_to_id = tokenizer["token_to_id"]
max_length = tokenizer["max_length"]

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

tokens_sorted = sorted(
    [
        t for t in token_to_id
        if t not in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
    ],
    key=len,
    reverse=True
)

print("Manifest rows:", len(manifest))

# --------------------------------------------------
# 2. LOAD M1 POCKET TENSOR AND MASK
# --------------------------------------------------

pocket_tensor = np.load(
    os.path.join(INPUT_DIR, "pocket_1_pharmacophore(1).npy")
)

pocket_mask = np.load(
    os.path.join(INPUT_DIR, "pocket_1_mask(1).npy")
)

assert pocket_tensor.shape == (5, 95, 4)
assert pocket_mask.shape == (5, 95)

print("M1 pocket:", pocket_tensor.shape)
print("M1 mask:", pocket_mask.shape)

# --------------------------------------------------
# 3. SMILES TOKENIZATION
# --------------------------------------------------

def tokenize_smiles(smiles):
    remaining = smiles
    ids = []

    while remaining:
        matched = False

        for token in tokens_sorted:
            if remaining.startswith(token):
                ids.append(token_to_id[token])
                remaining = remaining[len(token):]
                matched = True
                break

        if not matched:
            raise ValueError(
                f"Unknown SMILES fragment: {remaining[:30]}"
            )

    sequence = [BOS_ID] + ids + [EOS_ID]

    if len(sequence) > max_length:
        raise ValueError(f"SMILES exceeds max length: {smiles}")

    sequence += [PAD_ID] * (max_length - len(sequence))

    return sequence

# --------------------------------------------------
# 4. BUILD MOLECULE-POCKET PAIRS
# --------------------------------------------------

token_cache = {}

all_tokens = []
all_pockets = []
all_masks = []
all_ligands = []
all_conformers = []
all_splits = []

for _, row in manifest.iterrows():

    smiles = str(row["canonical_smiles"])
    ligand = str(row["ligand_identity"])
    conformer_id = int(row["conformer_id"])

    if ligand not in token_cache:
        token_cache[ligand] = tokenize_smiles(smiles)

    pocket_index = conformer_id - 1

    all_tokens.append(token_cache[ligand])
    all_pockets.append(pocket_tensor[pocket_index])
    all_masks.append(pocket_mask[pocket_index])

    all_ligands.append(ligand)
    all_conformers.append(conformer_id)
    all_splits.append(row["split"])

# --------------------------------------------------
# 5. CONVERT TO ARRAYS
# --------------------------------------------------

all_tokens = np.asarray(all_tokens, dtype=np.int64)
all_pockets = np.asarray(all_pockets, dtype=np.float32)
all_masks = np.asarray(all_masks, dtype=np.float32)

all_ligands = np.asarray(all_ligands)
all_conformers = np.asarray(all_conformers)
all_splits = np.asarray(all_splits)

train_idx = np.where(all_splits == "train")[0]
val_idx = np.where(all_splits == "validation")[0]

# --------------------------------------------------
# 6. SAVE DATASETS
# --------------------------------------------------

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "pocket_conditioned_train_inputs.npz"),
    token_ids=all_tokens[train_idx],
    pockets=all_pockets[train_idx],
    masks=all_masks[train_idx],
    ligands=all_ligands[train_idx],
    conformers=all_conformers[train_idx]
)

np.savez_compressed(
    os.path.join(OUTPUT_DIR, "pocket_conditioned_validation_inputs.npz"),
    token_ids=all_tokens[val_idx],
    pockets=all_pockets[val_idx],
    masks=all_masks[val_idx],
    ligands=all_ligands[val_idx],
    conformers=all_conformers[val_idx]
)

# --------------------------------------------------
# 7. FINAL SUMMARY
# --------------------------------------------------

print("\n========== DATASET GENERATION COMPLETE ==========")

print("Total examples:", len(all_tokens))
print("Training examples:", len(train_idx))
print("Validation examples:", len(val_idx))

print("Training unique ligands:",
      len(set(all_ligands[train_idx])))

print("Validation unique ligands:",
      len(set(all_ligands[val_idx])))

print("Training shape:", all_tokens[train_idx].shape)
print("Validation shape:", all_tokens[val_idx].shape)

print("\nSaved:")
print("pocket_conditioned_train_inputs.npz")
print("pocket_conditioned_validation_inputs.npz")

print("\nCELL 243 COMPLETED")

Manifest rows: 160
M1 pocket: (5, 95, 4)
M1 mask: (5, 95)

========== DATASET GENERATION COMPLETE ==========
Total examples: 160
Training examples: 125
Validation examples: 35
Training unique ligands: 25
Validation unique ligands: 7
Training shape: (125, 143)
Validation shape: (35, 143)

Saved:
pocket_conditioned_train_inputs.npz
pocket_conditioned_validation_inputs.npz

CELL 243 COMPLETED


In [326]:
import os
import numpy as np

OUTPUT_DIR = "/content/drive/MyDrive/Member2_Project/member2_outputs"

train_path = os.path.join(
    OUTPUT_DIR, "pocket_conditioned_train_inputs.npz"
)

val_path = os.path.join(
    OUTPUT_DIR, "pocket_conditioned_validation_inputs.npz"
)

def validate_dataset(path, name):

    data = np.load(path, allow_pickle=True)

    tokens = data["token_ids"]
    pockets = data["pockets"]
    masks = data["masks"]
    ligands = data["ligands"]
    conformers = data["conformers"]

    print(f"\n========== {name} ==========")
    print("Tokens:", tokens.shape)
    print("Pockets:", pockets.shape)
    print("Masks:", masks.shape)

    assert tokens.shape[1] == 143
    assert pockets.shape[1:] == (95, 4)
    assert masks.shape[1:] == (95,)
    assert len(tokens) == len(pockets) == len(masks)
    assert len(tokens) == len(ligands) == len(conformers)

    assert np.isfinite(pockets).all()
    assert np.isfinite(masks).all()
    assert set(conformers.tolist()) == {1, 2, 3, 4, 5}

    print("Unique ligands:", len(set(ligands)))
    print("Conformers:", sorted(set(conformers.tolist())))
    print("Unknown tokens:", np.sum(tokens == 3))
    print("Integrity: PASS")

validate_dataset(train_path, "TRAINING")
validate_dataset(val_path, "VALIDATION")

print("\nCELL 244 COMPLETED")


========== TRAINING ==========
Tokens: (125, 143)
Pockets: (125, 95, 4)
Masks: (125, 95)
Unique ligands: 25
Conformers: [1, 2, 3, 4, 5]
Unknown tokens: 0
Integrity: PASS

========== VALIDATION ==========
Tokens: (35, 143)
Pockets: (35, 95, 4)
Masks: (35, 95)
Unique ligands: 7
Conformers: [1, 2, 3, 4, 5]
Unknown tokens: 0
Integrity: PASS

CELL 244 COMPLETED


In [327]:
import os
import torch

ROOT = "/content/drive/MyDrive/Member2_Project"

checkpoint_path = os.path.join(
    ROOT,
    "checkpoints",
    "transformer_chembl37_improved_best.pt"
)

print("========== MODEL INSPECTION ==========")

# Check existing model classes
print("\nAvailable model classes:")

for name in [
    "DynamicPocketEncoder",
    "PocketConditionedTransformer"
]:
    print(name, ":", name in globals())

# Check existing model instances
print("\nExisting model variables:")

for name in [
    "molecular_transformer",
    "pocket_encoder",
    "model"
]:
    if name in globals():
        obj = globals()[name]
        print(name, ":", type(obj).__name__)

# Load checkpoint on CPU
assert os.path.exists(checkpoint_path), (
    "ChEMBL checkpoint not found!"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):

    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

    state = checkpoint.get(
        "model_state_dict",
        checkpoint.get("state_dict")
    )

    if state is None:
        print("\nNo standard state_dict key found.")
        print("Checkpoint structure needs inspection.")

    else:
        print("\nState dictionary entries:", len(state))

        print("\nFirst 15 parameter names:")
        for key in list(state.keys())[:15]:
            print(key, tuple(state[key].shape))

print("\nCELL 245 COMPLETED")

========== MODEL INSPECTION ==========

Available model classes:
DynamicPocketEncoder : True
PocketConditionedTransformer : True

Existing model variables:
molecular_transformer : PocketConditionedTransformer
pocket_encoder : DynamicPocketEncoder
model : Model

Checkpoint type: dict

Checkpoint keys:
['model_state_dict', 'optimizer_state_dict', 'epoch', 'train_loss', 'validation_loss', 'validation_accuracy', 'training_molecules', 'training_type', 'pocket_cross_attention_used', 'vocabulary_size']

State dictionary entries: 58

First 15 parameter names:
token_embedding.weight (81, 128)
position_encoding.position_embedding.weight (143, 128)
decoder.layers.0.self_attn.in_proj_weight (384, 128)
decoder.layers.0.self_attn.in_proj_bias (384,)
decoder.layers.0.self_attn.out_proj.weight (128, 128)
decoder.layers.0.self_attn.out_proj.bias (128,)
decoder.layers.0.multihead_attn.in_proj_weight (384, 128)
decoder.layers.0.multihead_attn.in_proj_bias (384,)
decoder.layers.0.multihead_attn.out_proj.w

In [328]:
import torch

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

checkpoint_state = checkpoint["model_state_dict"]
current_state = molecular_transformer.state_dict()

print("========== CHECKPOINT COMPATIBILITY ==========")

print("Checkpoint parameters:", len(checkpoint_state))
print("Current model parameters:", len(current_state))

missing = [
    key for key in current_state
    if key not in checkpoint_state
]

unexpected = [
    key for key in checkpoint_state
    if key not in current_state
]

shape_mismatches = []

for key in current_state:
    if key in checkpoint_state:
        if current_state[key].shape != checkpoint_state[key].shape:
            shape_mismatches.append(
                (key, tuple(current_state[key].shape),
                 tuple(checkpoint_state[key].shape))
            )

print("\nMissing parameters:", missing)
print("Unexpected parameters:", unexpected)
print("Shape mismatches:", shape_mismatches)

if not missing and not unexpected and not shape_mismatches:
    print("\nCHECKPOINT COMPATIBILITY: PASS")
else:
    print("\nCHECKPOINT COMPATIBILITY: FAIL")

print("\nTraining type:", checkpoint["training_type"])
print("Pocket cross-attention used:",
      checkpoint["pocket_cross_attention_used"])

print("\nCELL 246 COMPLETED")

========== CHECKPOINT COMPATIBILITY ==========
Checkpoint parameters: 58
Current model parameters: 58

Missing parameters: []
Unexpected parameters: []
Shape mismatches: []

CHECKPOINT COMPATIBILITY: PASS

Training type: chembl37_general_molecular_finetuning
Pocket cross-attention used: False

CELL 246 COMPLETED


In [329]:
import torch

# Load pretrained molecular Transformer weights
load_result = molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

print("========== PRETRAINED MODEL LOADED ==========")

print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)

# Verify exact weight restoration
loaded_state = molecular_transformer.state_dict()

all_match = all(
    torch.equal(
        loaded_state[key].cpu(),
        checkpoint["model_state_dict"][key].cpu()
    )
    for key in checkpoint["model_state_dict"]
)

print("All checkpoint weights restored:", all_match)

# Model parameter count
total_params = sum(
    p.numel() for p in molecular_transformer.parameters()
)

print("Transformer parameters:", total_params)

# Keep model on CPU for now
molecular_transformer = molecular_transformer.cpu()

print("Device: CPU")
print("CELL 247 COMPLETED")

========== PRETRAINED MODEL LOADED ==========
Missing keys: []
Unexpected keys: []
All checkpoint weights restored: True
Transformer parameters: 832849
Device: CPU
CELL 247 COMPLETED


In [330]:
import torch

print("========== POCKET ENCODER INSPECTION ==========")

# Move encoder to CPU
pocket_encoder = pocket_encoder.cpu()

# Inspect architecture
print("\nArchitecture:")
print(pocket_encoder)

# Count parameters
total_params = sum(
    p.numel() for p in pocket_encoder.parameters()
)

trainable_params = sum(
    p.numel()
    for p in pocket_encoder.parameters()
    if p.requires_grad
)

print("\nTotal parameters:", total_params)
print("Trainable parameters:", trainable_params)

# Check parameter initialization
print("\nParameter statistics:")

for name, param in pocket_encoder.named_parameters():
    print(
        name,
        "| Shape:", tuple(param.shape),
        "| Mean:", round(param.data.mean().item(), 5),
        "| Std:", round(param.data.std().item(), 5)
    )

print("\nDevice:", next(pocket_encoder.parameters()).device)

print("\nCELL 248 COMPLETED")

========== POCKET ENCODER INSPECTION ==========

Architecture:
DynamicPocketEncoder(
  (feature_embedding): Embedding(6, 128)
  (coordinate_projection): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (feature_fusion): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  )
)

Total parameters: 50944
Trainable parameters: 50944

Parameter statistics:
feature_embedding.weight | Shape: (6, 128) | Mean: -0.03288 | Std: 1.02069
coordinate_projection.0.weight | Shape: (128, 3) | Mean: 0.00956 | Std: 0.3349
coordinate_projection.0.bias | Shape: (128,) | Mean: -0.01018 | Std: 0.32959
coordinate_projection.2.weight | Shape: (128, 128) | Mean: 0.00035 | Std: 0.05091
coordinate_projection.2.bias | Shape: (128,) | Mean: 0.00446 | Std: 0.05497
fe

In [331]:
import inspect

print("========== POCKET ENCODER FORWARD ==========")
print(inspect.getsource(DynamicPocketEncoder.forward))

print("\n========== TRANSFORMER FORWARD ==========")
print(inspect.getsource(PocketConditionedTransformer.forward))

print("\n========== CURRENT FEATURE MAPPING ==========")

M1_TO_ENCODER = {
    0: 2,  # DONOR
    1: 0,  # ACCEPTOR
    2: 3,  # HYDROPHOBIC
    3: 1,  # AROMATIC
    4: 5,  # POSITIVE
    5: 4   # NEGATIVE
}

print(M1_TO_ENCODER)

print("\nCELL 249 COMPLETED")

========== POCKET ENCODER FORWARD ==========
    def forward(self, coordinates, feature_ids, mask):

        coordinate_embeddings = self.coordinate_projection(
            coordinates
        )

        feature_embeddings = self.feature_embedding(
            feature_ids
        )

        combined = torch.cat(
            [
                coordinate_embeddings,
                feature_embeddings
            ],
            dim=-1
        )

        pocket_embeddings = self.feature_fusion(combined)

        # Remove invalid positions
        pocket_embeddings = pocket_embeddings * mask.unsqueeze(-1)

        return pocket_embeddings


========== TRANSFORMER FORWARD ==========
    def forward(
        self,
        token_ids,
        pocket_memory,
        token_padding_mask,
        pocket_padding_mask
    ):

        # Molecular token representation
        token_embeddings = self.token_embedding(token_ids)

        token_embeddings = self.position_encoding(
            token_embeddi

In [332]:
import torch
import numpy as np

# Load one training example
data = np.load(
    "/content/drive/MyDrive/Member2_Project/member2_outputs/"
    "pocket_conditioned_train_inputs.npz",
    allow_pickle=True
)

# Convert to tensors
token_ids = torch.tensor(
    data["token_ids"][:2],
    dtype=torch.long
)

pockets = torch.tensor(
    data["pockets"][:2],
    dtype=torch.float32
)

masks = torch.tensor(
    data["masks"][:2],
    dtype=torch.float32
)

# Separate coordinates and M1 feature IDs
coordinates = pockets[:, :, :3]

m1_feature_ids = pockets[:, :, 3].long()

# Convert M1 IDs to encoder vocabulary IDs
mapping = torch.tensor(
    [2, 0, 3, 1, 5, 4],
    dtype=torch.long
)

feature_ids = mapping[m1_feature_ids]

# Convert masks
pocket_mask = masks.bool()

# Teacher-forcing input and target preparation
decoder_input = token_ids[:, :-1]

token_padding_mask = decoder_input.eq(0)

# Forward pass
pocket_encoder.eval()
molecular_transformer.eval()

with torch.no_grad():

    pocket_memory = pocket_encoder(
        coordinates,
        feature_ids,
        masks
    )

    logits = molecular_transformer(
        decoder_input,
        pocket_memory,
        token_padding_mask,
        ~pocket_mask
    )

targets = token_ids[:, 1:]

print("========== FORWARD PASS TEST ==========")

print("Decoder input:", decoder_input.shape)
print("Pocket memory:", pocket_memory.shape)
print("Logits:", logits.shape)
print("Targets:", targets.shape)

print("\nFeature IDs:", torch.unique(feature_ids).tolist())
print("Pocket memory finite:", torch.isfinite(pocket_memory).all().item())
print("Logits finite:", torch.isfinite(logits).all().item())

assert logits.shape == (
    2, 142, 81
)

assert targets.shape == (
    2, 142
)

assert torch.isfinite(logits).all()
assert torch.isfinite(pocket_memory).all()

print("\nFORWARD PASS: SUCCESS")
print("CELL 250 COMPLETED")

========== FORWARD PASS TEST ==========
Decoder input: torch.Size([2, 142])
Pocket memory: torch.Size([2, 95, 128])
Logits: torch.Size([2, 142, 81])
Targets: torch.Size([2, 142])

Feature IDs: [0, 1, 2, 3, 4, 5]
Pocket memory finite: True
Logits finite: True

FORWARD PASS: SUCCESS
CELL 250 COMPLETED


In [333]:
import torch

print("========== TRAINING PARAMETER AUDIT ==========")

print("\nTransformer modules:")

for name, module in molecular_transformer.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"{name}: {params:,} parameters")

print("\nPocket encoder parameters:")

print(
    f"{sum(p.numel() for p in pocket_encoder.parameters()):,}"
)

print("\nTotal model parameters:")

total = (
    sum(p.numel() for p in molecular_transformer.parameters())
    + sum(p.numel() for p in pocket_encoder.parameters())
)

print(f"{total:,}")

print("\nCELL 251 COMPLETED")

========== TRAINING PARAMETER AUDIT ==========

Transformer modules:
token_embedding: 10,368 parameters
position_encoding: 18,304 parameters
decoder: 793,728 parameters
output_layer: 10,449 parameters

Pocket encoder parameters:
50,944

Total model parameters:
883,793

CELL 251 COMPLETED


In [334]:
import torch

# 1. Freeze entire Transformer
for param in molecular_transformer.parameters():
    param.requires_grad = False

# 2. Unfreeze only cross-attention
for layer in molecular_transformer.decoder.layers:
    for param in layer.multihead_attn.parameters():
        param.requires_grad = True

# 3. Unfreeze pocket encoder
for param in pocket_encoder.parameters():
    param.requires_grad = True

# 4. Collect trainable parameters
trainable_parameters = (
    [p for p in molecular_transformer.parameters()
     if p.requires_grad]
    +
    [p for p in pocket_encoder.parameters()
     if p.requires_grad]
)

trainable_count = sum(
    p.numel() for p in trainable_parameters
)

total_count = (
    sum(p.numel() for p in molecular_transformer.parameters())
    +
    sum(p.numel() for p in pocket_encoder.parameters())
)

print("========== TRAINING CONFIGURATION ==========")

print("Total parameters:", total_count)
print("Trainable parameters:", trainable_count)
print("Frozen parameters:", total_count - trainable_count)

print("\nCross-attention trainable:")

for i, layer in enumerate(molecular_transformer.decoder.layers):
    count = sum(
        p.numel() for p in layer.multihead_attn.parameters()
        if p.requires_grad
    )
    print(f"Layer {i+1}: {count:,}")

print("\nPocket encoder trainable:",
      all(p.requires_grad for p in pocket_encoder.parameters()))

print("\nCELL 252 COMPLETED")

========== TRAINING CONFIGURATION ==========
Total parameters: 883793
Trainable parameters: 249088
Frozen parameters: 634705

Cross-attention trainable:
Layer 1: 66,048
Layer 2: 66,048
Layer 3: 66,048

Pocket encoder trainable: True

CELL 252 COMPLETED


In [335]:
import torch
import torch.nn as nn

# Separate parameter groups
cross_attention_params = [
    p
    for layer in molecular_transformer.decoder.layers
    for p in layer.multihead_attn.parameters()
    if p.requires_grad
]

pocket_encoder_params = [
    p for p in pocket_encoder.parameters()
    if p.requires_grad
]

# AdamW optimizer
optimizer = torch.optim.AdamW(
    [
        {
            "params": cross_attention_params,
            "lr": 1e-4
        },
        {
            "params": pocket_encoder_params,
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

# Ignore padding tokens during loss calculation
criterion = nn.CrossEntropyLoss(
    ignore_index=0
)

print("========== OPTIMIZER CONFIGURATION ==========")

print("Optimizer: AdamW")
print("Cross-attention learning rate:", 1e-4)
print("Pocket encoder learning rate:", 1e-4)
print("Weight decay:", 1e-4)
print("Loss: CrossEntropyLoss")
print("Ignored token ID:", 0)

print("\nOptimizer parameter groups:", len(optimizer.param_groups))

print("CELL 253 COMPLETED")

========== OPTIMIZER CONFIGURATION ==========
Optimizer: AdamW
Cross-attention learning rate: 0.0001
Pocket encoder learning rate: 0.0001
Weight decay: 0.0001
Loss: CrossEntropyLoss
Ignored token ID: 0

Optimizer parameter groups: 2
CELL 253 COMPLETED


In [336]:
import numpy as np
import torch

OUTPUT_DIR = "/content/drive/MyDrive/Member2_Project/member2_outputs"

# Load datasets
train_np = np.load(
    f"{OUTPUT_DIR}/pocket_conditioned_train_inputs.npz",
    allow_pickle=True
)

val_np = np.load(
    f"{OUTPUT_DIR}/pocket_conditioned_validation_inputs.npz",
    allow_pickle=True
)

# M1 feature ID -> encoder feature ID
FEATURE_MAPPING = torch.tensor(
    [2, 0, 3, 1, 5, 4],
    dtype=torch.long
)

def prepare_data(data):

    # Convert arrays into tensors
    tokens = torch.tensor(
        data["token_ids"], dtype=torch.long
    )

    pockets = torch.tensor(
        data["pockets"], dtype=torch.float32
    )

    masks = torch.tensor(
        data["masks"], dtype=torch.float32
    )

    # Teacher forcing
    decoder_input = tokens[:, :-1]
    targets = tokens[:, 1:]

    # Separate pocket coordinates and feature IDs
    coordinates = pockets[:, :, :3]

    m1_features = pockets[:, :, 3].long()

    feature_ids = FEATURE_MAPPING[m1_features]

    # Padding masks
    token_padding_mask = decoder_input.eq(0)

    pocket_padding_mask = masks.eq(0)

    return {
        "decoder_input": decoder_input,
        "targets": targets,
        "coordinates": coordinates,
        "feature_ids": feature_ids,
        "token_padding_mask": token_padding_mask,
        "pocket_padding_mask": pocket_padding_mask
    }

train_data = prepare_data(train_np)
val_data = prepare_data(val_np)

print("========== DATA PREPARATION ==========")

for name, value in train_data.items():
    print("TRAIN", name, value.shape)

print()

for name, value in val_data.items():
    print("VALIDATION", name, value.shape)

# Verify target shifting
assert torch.equal(
    train_data["decoder_input"][:, 1:],
    train_data["targets"][:, :-1]
)

assert train_data["decoder_input"].shape == (125, 142)
assert train_data["targets"].shape == (125, 142)

assert train_data["feature_ids"].max().item() <= 5
assert train_data["feature_ids"].min().item() >= 0

print("\nTeacher-forcing alignment: PASS")
print("Feature mapping: PASS")
print("CELL 254 COMPLETED")

========== DATA PREPARATION ==========
TRAIN decoder_input torch.Size([125, 142])
TRAIN targets torch.Size([125, 142])
TRAIN coordinates torch.Size([125, 95, 3])
TRAIN feature_ids torch.Size([125, 95])
TRAIN token_padding_mask torch.Size([125, 142])
TRAIN pocket_padding_mask torch.Size([125, 95])

VALIDATION decoder_input torch.Size([35, 142])
VALIDATION targets torch.Size([35, 142])
VALIDATION coordinates torch.Size([35, 95, 3])
VALIDATION feature_ids torch.Size([35, 95])
VALIDATION token_padding_mask torch.Size([35, 142])
VALIDATION pocket_padding_mask torch.Size([35, 95])

Teacher-forcing alignment: PASS
Feature mapping: PASS
CELL 254 COMPLETED


In [337]:
import torch
import numpy as np
import time
import os

DEVICE = torch.device("cpu")
BATCH_SIZE = 8

molecular_transformer.to(DEVICE)
pocket_encoder.to(DEVICE)

def run_epoch(data, training=True):

    if training:
        molecular_transformer.train()
        pocket_encoder.train()
    else:
        molecular_transformer.eval()
        pocket_encoder.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    n = data["decoder_input"].size(0)

    indices = torch.randperm(n) if training else torch.arange(n)

    context = torch.enable_grad() if training else torch.no_grad()

    with context:

        for start in range(0, n, BATCH_SIZE):

            idx = indices[start:start + BATCH_SIZE]

            decoder_input = data["decoder_input"][idx].to(DEVICE)
            targets = data["targets"][idx].to(DEVICE)

            coordinates = data["coordinates"][idx].to(DEVICE)
            feature_ids = data["feature_ids"][idx].to(DEVICE)

            token_padding_mask = data["token_padding_mask"][idx].to(DEVICE)
            pocket_padding_mask = data["pocket_padding_mask"][idx].to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

            pocket_memory = pocket_encoder(
                coordinates,
                feature_ids,
                (~pocket_padding_mask).float()
            )

            logits = molecular_transformer(
                decoder_input,
                pocket_memory,
                token_padding_mask,
                pocket_padding_mask
            )

            loss = criterion(
                logits.reshape(-1, 81),
                targets.reshape(-1)
            )

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    trainable_parameters,
                    max_norm=1.0
                )

                optimizer.step()

            valid = targets.ne(PAD_ID)

            correct = (
                logits.argmax(dim=-1).eq(targets) & valid
            ).sum().item()

            valid_count = valid.sum().item()

            total_loss += loss.item() * valid_count
            total_correct += correct
            total_tokens += valid_count

    return {
        "loss": total_loss / total_tokens,
        "accuracy": total_correct / total_tokens,
        "tokens": total_tokens
    }


# Run one training epoch
print("Starting one pocket-conditioned training epoch...")

start_time = time.time()

train_metrics = run_epoch(train_data, training=True)
val_metrics = run_epoch(val_data, training=False)

elapsed = time.time() - start_time

print("\n========== EPOCH 1 RESULTS ==========")

print("Training loss:", round(train_metrics["loss"], 5))
print("Training accuracy:", round(train_metrics["accuracy"] * 100, 2), "%")

print("\nValidation loss:", round(val_metrics["loss"], 5))
print("Validation accuracy:", round(val_metrics["accuracy"] * 100, 2), "%")

print("\nTraining tokens:", train_metrics["tokens"])
print("Validation tokens:", val_metrics["tokens"])

print("Elapsed time:", round(elapsed, 2), "seconds")

print("\nCELL 255 COMPLETED")

Starting one pocket-conditioned training epoch...

========== EPOCH 1 RESULTS ==========
Training loss: 1.25914
Training accuracy: 56.43 %

Validation loss: 1.18688
Validation accuracy: 59.52 %

Training tokens: 5455
Validation tokens: 1660
Elapsed time: 9.75 seconds

CELL 255 COMPLETED


In [338]:
import copy
import os
import time
import pandas as pd
import torch

EPOCHS = 20
PATIENCE = 5

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

best_transformer_state = None
best_encoder_state = None

history = []

checkpoint_dir = "/content/drive/MyDrive/Member2_Project/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print("========== POCKET-CONDITIONED TRAINING ==========")

training_start = time.time()

for epoch in range(1, EPOCHS + 1):

    train_metrics = run_epoch(train_data, training=True)
    val_metrics = run_epoch(val_data, training=False)

    train_loss = train_metrics["loss"]
    val_loss = val_metrics["loss"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_accuracy": train_metrics["accuracy"],
        "val_accuracy": val_metrics["accuracy"]
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.5f} | "
        f"Val Loss: {val_loss:.5f} | "
        f"Train Acc: {train_metrics['accuracy']*100:.2f}% | "
        f"Val Acc: {val_metrics['accuracy']*100:.2f}%"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0

        best_transformer_state = copy.deepcopy(
            molecular_transformer.state_dict()
        )

        best_encoder_state = copy.deepcopy(
            pocket_encoder.state_dict()
        )

        torch.save({
            "epoch": epoch,
            "transformer_state_dict": best_transformer_state,
            "pocket_encoder_state_dict": best_encoder_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "validation_loss": best_val_loss,
            "training_type": "experimental_pocket_conditioned_finetuning"
        }, os.path.join(
            checkpoint_dir,
            "member2_pocket_conditioned_best.pt"
        ))

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

elapsed = time.time() - training_start

history_df = pd.DataFrame(history)

history_path = os.path.join(
    "/content/drive/MyDrive/Member2_Project/member2_outputs",
    "pocket_conditioned_training_history.csv"
)

history_df.to_csv(history_path, index=False)

print("\n========== TRAINING SUMMARY ==========")
print("Completed epochs:", len(history))
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 5))
print("Elapsed time:", round(elapsed, 2), "seconds")
print("History saved:", history_path)

print("\nCELL 256 COMPLETED")

========== POCKET-CONDITIONED TRAINING ==========
Epoch 01/20 | Train Loss: 1.20027 | Val Loss: 1.15775 | Train Acc: 58.22% | Val Acc: 60.24%
Epoch 02/20 | Train Loss: 1.17932 | Val Loss: 1.14824 | Train Acc: 58.79% | Val Acc: 60.24%
Epoch 03/20 | Train Loss: 1.16693 | Val Loss: 1.14133 | Train Acc: 58.61% | Val Acc: 60.24%
Epoch 04/20 | Train Loss: 1.15427 | Val Loss: 1.13701 | Train Acc: 59.51% | Val Acc: 60.54%
Epoch 05/20 | Train Loss: 1.15805 | Val Loss: 1.13738 | Train Acc: 59.29% | Val Acc: 60.00%
Epoch 06/20 | Train Loss: 1.15163 | Val Loss: 1.13645 | Train Acc: 59.63% | Val Acc: 60.84%
Epoch 07/20 | Train Loss: 1.14389 | Val Loss: 1.12438 | Train Acc: 59.49% | Val Acc: 60.54%
Epoch 08/20 | Train Loss: 1.13791 | Val Loss: 1.12619 | Train Acc: 60.35% | Val Acc: 60.24%
Epoch 09/20 | Train Loss: 1.13522 | Val Loss: 1.11295 | Train Acc: 60.55% | Val Acc: 59.94%
Epoch 10/20 | Train Loss: 1.11068 | Val Loss: 1.11625 | Train Acc: 61.34% | Val Acc: 59.22%
Epoch 11/20 | Train Loss: 1.08

In [339]:
import torch
import torch.nn as nn
import os

checkpoint_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "checkpoints/member2_pocket_conditioned_best.pt"
)

# Load the best epoch checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["transformer_state_dict"]
)

pocket_encoder.load_state_dict(
    checkpoint["pocket_encoder_state_dict"]
)

molecular_transformer.eval()
pocket_encoder.eval()

print("========== BEST CHECKPOINT LOADED ==========")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["validation_loss"])

# Prepare validation tensors
decoder_input = val_data["decoder_input"]
targets = val_data["targets"]
coordinates = val_data["coordinates"]
feature_ids = val_data["feature_ids"]

token_padding_mask = val_data["token_padding_mask"]
pocket_padding_mask = val_data["pocket_padding_mask"]

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

with torch.no_grad():

    # Encode the correct pocket
    true_memory = pocket_encoder(
        coordinates,
        feature_ids,
        (~pocket_padding_mask).float()
    )

    # Create a zero-pocket condition
    zero_memory = torch.zeros_like(true_memory)

    # Shuffle pocket contexts between examples
    permutation = torch.randperm(
        true_memory.size(0)
    )

    shuffled_memory = true_memory[permutation]

    # Predictions using the correct pocket
    true_logits = molecular_transformer(
        decoder_input,
        true_memory,
        token_padding_mask,
        pocket_padding_mask
    )

    # Predictions with zero pocket information
    zero_logits = molecular_transformer(
        decoder_input,
        zero_memory,
        token_padding_mask,
        pocket_padding_mask
    )

    # Predictions using shuffled pocket information
    shuffled_logits = molecular_transformer(
        decoder_input,
        shuffled_memory,
        token_padding_mask,
        pocket_padding_mask
    )

    # Calculate losses
    true_loss = criterion(
        true_logits.reshape(-1, true_logits.size(-1)),
        targets.reshape(-1)
    )

    zero_loss = criterion(
        zero_logits.reshape(-1, zero_logits.size(-1)),
        targets.reshape(-1)
    )

    shuffled_loss = criterion(
        shuffled_logits.reshape(-1, shuffled_logits.size(-1)),
        targets.reshape(-1)
    )

    # Measure prediction changes
    zero_difference = (
        true_logits - zero_logits
    ).abs().mean().item()

    shuffled_difference = (
        true_logits - shuffled_logits
    ).abs().mean().item()


print("\n========== POCKET CONDITIONING TEST ==========")

print(f"True pocket loss:      {true_loss.item():.5f}")
print(f"Zero pocket loss:      {zero_loss.item():.5f}")
print(f"Shuffled pocket loss:  {shuffled_loss.item():.5f}")

print("\nPrediction differences:")
print(f"True vs zero:      {zero_difference:.6f}")
print(f"True vs shuffled:  {shuffled_difference:.6f}")

========== BEST CHECKPOINT LOADED ==========
Best epoch: 12
Best validation loss: 1.1077755056590919

========== POCKET CONDITIONING TEST ==========
True pocket loss:      1.10778
Zero pocket loss:      1.21975
Shuffled pocket loss:  1.10797

Prediction differences:
True vs zero:      0.535517
True vs shuffled:  0.004797


In [340]:
import torch
import torch.nn.functional as F

molecular_transformer.eval()
pocket_encoder.eval()

# Select the first validation molecule.
# Its five conformers are stored consecutively.
sample_coordinates = val_data["coordinates"][:5]
sample_features = val_data["feature_ids"][:5]
sample_masks = val_data["pocket_padding_mask"][:5]

with torch.no_grad():

    pocket_embeddings = pocket_encoder(
        sample_coordinates,
        sample_features,
        (~sample_masks).float()
    )

    # Mean pooling over valid pocket positions
    valid_mask = (~sample_masks).unsqueeze(-1).float()

    pooled_embeddings = (
        pocket_embeddings * valid_mask
    ).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)

print("========== CONFORMER REPRESENTATION ANALYSIS ==========")

print("Pocket embedding shape:", pocket_embeddings.shape)
print("Pooled representation shape:", pooled_embeddings.shape)

print("\nPairwise cosine similarities:")

for i in range(5):
    for j in range(i + 1, 5):

        similarity = F.cosine_similarity(
            pooled_embeddings[i:i+1],
            pooled_embeddings[j:j+1]
        ).item()

        print(f"Conformer {i+1} vs {j+1}: {similarity:.6f}")

========== CONFORMER REPRESENTATION ANALYSIS ==========
Pocket embedding shape: torch.Size([5, 95, 128])
Pooled representation shape: torch.Size([5, 128])

Pairwise cosine similarities:
Conformer 1 vs 2: 0.999929
Conformer 1 vs 3: 0.999954
Conformer 1 vs 4: 0.999969
Conformer 1 vs 5: 0.999988
Conformer 2 vs 3: 0.999970
Conformer 2 vs 4: 0.999843
Conformer 2 vs 5: 0.999961
Conformer 3 vs 4: 0.999928
Conformer 3 vs 5: 0.999989
Conformer 4 vs 5: 0.999960


In [341]:
import torch

molecular_transformer.eval()
pocket_encoder.eval()

coordinates = val_data["coordinates"][:5]
feature_ids = val_data["feature_ids"][:5]
masks = val_data["pocket_padding_mask"][:5]

with torch.no_grad():

    embeddings = pocket_encoder(
        coordinates,
        feature_ids,
        (~masks).float()
    )

print("========== EMBEDDING DIFFERENCE ANALYSIS ==========")

reference = embeddings[0]

for i in range(1, 5):

    difference = (
        embeddings[i] - reference
    ).abs()

    print(f"\nConformer 1 vs {i+1}")

    print("Mean absolute difference:",
          difference.mean().item())

    print("Maximum absolute difference:",
          difference.max().item())

    print("Embedding standard deviation:",
          embeddings[i].std().item())

print("\nReference embedding standard deviation:",
      reference.std().item())

========== EMBEDDING DIFFERENCE ANALYSIS ==========

Conformer 1 vs 2
Mean absolute difference: 0.009190881624817848
Maximum absolute difference: 0.11601084470748901
Embedding standard deviation: 1.001416802406311

Conformer 1 vs 3
Mean absolute difference: 0.006526657845824957
Maximum absolute difference: 0.05681806802749634
Embedding standard deviation: 1.001415491104126

Conformer 1 vs 4
Mean absolute difference: 0.005503654479980469
Maximum absolute difference: 0.1370692253112793
Embedding standard deviation: 1.0014134645462036

Conformer 1 vs 5
Mean absolute difference: 0.00326750660315156
Maximum absolute difference: 0.028493106365203857
Embedding standard deviation: 1.0014150142669678

Reference embedding standard deviation: 1.0014146566390991


In [342]:
import torch
import torch.nn.functional as F

molecular_transformer.eval()
pocket_encoder.eval()

# Validation data: 7 molecules × 5 conformers.
coordinates = val_data["coordinates"]
features = val_data["feature_ids"]
masks = val_data["pocket_padding_mask"]

with torch.no_grad():

    embeddings = pocket_encoder(
        coordinates,
        features,
        (~masks).float()
    )

    # One pooled representation per conformer
    valid_mask = (~masks).unsqueeze(-1).float()

    pooled = (
        embeddings * valid_mask
    ).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1)

    # Five conformers per molecule
    molecule_representations = pooled.reshape(7, 5, -1).mean(dim=1)

print("========== DIFFERENT MOLECULE POCKET COMPARISON ==========")

print("Molecule representations:",
      molecule_representations.shape)

print("\nPairwise cosine similarities:")

for i in range(7):
    for j in range(i + 1, 7):

        similarity = F.cosine_similarity(
            molecule_representations[i:i+1],
            molecule_representations[j:j+1]
        ).item()

        print(
            f"Molecule {i+1} vs Molecule {j+1}: "
            f"{similarity:.6f}"
        )

========== DIFFERENT MOLECULE POCKET COMPARISON ==========
Molecule representations: torch.Size([7, 128])

Pairwise cosine similarities:
Molecule 1 vs Molecule 2: 1.000000
Molecule 1 vs Molecule 3: 1.000000
Molecule 1 vs Molecule 4: 1.000000
Molecule 1 vs Molecule 5: 1.000000
Molecule 1 vs Molecule 6: 1.000000
Molecule 1 vs Molecule 7: 1.000000
Molecule 2 vs Molecule 3: 1.000000
Molecule 2 vs Molecule 4: 1.000000
Molecule 2 vs Molecule 5: 1.000000
Molecule 2 vs Molecule 6: 1.000000
Molecule 2 vs Molecule 7: 1.000000
Molecule 3 vs Molecule 4: 1.000000
Molecule 3 vs Molecule 5: 1.000000
Molecule 3 vs Molecule 6: 1.000000
Molecule 3 vs Molecule 7: 1.000000
Molecule 4 vs Molecule 5: 1.000000
Molecule 4 vs Molecule 6: 1.000000
Molecule 4 vs Molecule 7: 1.000000
Molecule 5 vs Molecule 6: 1.000000
Molecule 5 vs Molecule 7: 1.000000
Molecule 6 vs Molecule 7: 1.000000


In [343]:
import numpy as np

coords = val_data["coordinates"].numpy()
features = val_data["feature_ids"].numpy()
masks = val_data["pocket_padding_mask"].numpy()

print("========== POCKET REUSE VERIFICATION ==========")

reference_coords = coords[:5]
reference_features = features[:5]
reference_masks = masks[:5]

for molecule in range(1, 7):

    start = molecule * 5
    end = start + 5

    same_coords = np.allclose(
        reference_coords,
        coords[start:end]
    )

    same_features = np.array_equal(
        reference_features,
        features[start:end]
    )

    same_masks = np.array_equal(
        reference_masks,
        masks[start:end]
    )

    print(f"\nMolecule {molecule + 1}")

    print("Same coordinates:", same_coords)
    print("Same feature IDs:", same_features)
    print("Same masks:", same_masks)

print("\nVerification completed.")

========== POCKET REUSE VERIFICATION ==========

Molecule 2
Same coordinates: True
Same feature IDs: True
Same masks: True

Molecule 3
Same coordinates: True
Same feature IDs: True
Same masks: True

Molecule 4
Same coordinates: True
Same feature IDs: True
Same masks: True

Molecule 5
Same coordinates: True
Same feature IDs: True
Same masks: True

Molecule 6
Same coordinates: True
Same feature IDs: True
Same masks: True

Molecule 7
Same coordinates: True
Same feature IDs: True
Same masks: True

Verification completed.


In [344]:
import torch
import torch.nn as nn

molecular_transformer.eval()
pocket_encoder.eval()

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

decoder_input = val_data["decoder_input"]
targets = val_data["targets"]
coordinates = val_data["coordinates"]
feature_ids = val_data["feature_ids"]

token_padding_mask = val_data["token_padding_mask"]
pocket_padding_mask = val_data["pocket_padding_mask"]

num_molecules = 7
num_conformers = 5

conformer_losses = []
conformer_accuracies = []
conformer_logits = []

with torch.no_grad():

    for conformer_idx in range(num_conformers):

        # Select the same conformer across all 7 molecules
        indices = torch.arange(
            conformer_idx,
            num_molecules * num_conformers,
            num_conformers
        )

        current_coordinates = coordinates[indices]
        current_features = feature_ids[indices]
        current_masks = pocket_padding_mask[indices]

        current_decoder = decoder_input[indices]
        current_targets = targets[indices]
        current_token_mask = token_padding_mask[indices]

        # Encode the selected conformer
        pocket_memory = pocket_encoder(
            current_coordinates,
            current_features,
            (~current_masks).float()
        )

        # Predict molecular tokens
        logits = molecular_transformer(
            current_decoder,
            pocket_memory,
            current_token_mask,
            current_masks
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            current_targets.reshape(-1)
        )

        # Calculate accuracy excluding padding
        predictions = logits.argmax(dim=-1)
        valid_tokens = current_targets.ne(PAD_ID)

        correct = (
            (predictions == current_targets) & valid_tokens
        ).sum().item()

        total = valid_tokens.sum().item()

        accuracy = correct / total

        conformer_losses.append(loss.item())
        conformer_accuracies.append(accuracy)
        conformer_logits.append(logits.cpu())

        print(
            f"Conformer {conformer_idx + 1}: "
            f"Loss = {loss.item():.5f} | "
            f"Accuracy = {accuracy * 100:.2f}%"
        )

print("\n========== CONFORMER PREDICTION DIFFERENCES ==========")

reference_logits = conformer_logits[0]

for i in range(1, num_conformers):

    difference = (
        reference_logits - conformer_logits[i]
    ).abs().mean().item()

    print(
        f"Conformer 1 vs {i+1}: "
        f"Mean logit difference = {difference:.6f}"
    )

print("\n========== SUMMARY ==========")

print("Lowest-loss conformer:",
      conformer_losses.index(min(conformer_losses)) + 1)

print("Best validation loss:",
      min(conformer_losses))

Conformer 1: Loss = 1.10801 | Accuracy = 62.65%
Conformer 2: Loss = 1.10807 | Accuracy = 62.65%
Conformer 3: Loss = 1.10752 | Accuracy = 62.65%
Conformer 4: Loss = 1.10752 | Accuracy = 62.05%
Conformer 5: Loss = 1.10775 | Accuracy = 62.65%

========== CONFORMER PREDICTION DIFFERENCES ==========
Conformer 1 vs 2: Mean logit difference = 0.012006
Conformer 1 vs 3: Mean logit difference = 0.008547
Conformer 1 vs 4: Mean logit difference = 0.006269
Conformer 1 vs 5: Mean logit difference = 0.004280

========== SUMMARY ==========
Lowest-loss conformer: 4
Best validation loss: 1.107517957687378


In [345]:
import os
import torch

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "checkpoints",
    "member2_pocket_conditioned_best.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

molecular_transformer.load_state_dict(
    checkpoint["transformer_state_dict"]
)

pocket_encoder.load_state_dict(
    checkpoint["pocket_encoder_state_dict"]
)

molecular_transformer.eval()
pocket_encoder.eval()

print("Checkpoint loaded successfully.")
print("Best epoch:", checkpoint.get("epoch", "Not recorded"))
print("Validation loss:", checkpoint.get("best_val_loss", "Not recorded"))
print("Device: CPU")

Checkpoint loaded successfully.
Best epoch: 12
Validation loss: Not recorded
Device: CPU


In [346]:
# Tokenizer configuration
token_to_id = tokenizer["token_to_id"]
id_to_token = {v: k for k, v in token_to_id.items()}

PAD_ID = token_to_id["<PAD>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
UNK_ID = token_to_id["<UNK>"]

# Generation settings
MAX_LENGTH = 143
NUM_MOLECULES_PER_CONFORMER = 10
TEMPERATURE = 0.8
TOP_K = 10

# Verify required components
assert molecular_transformer is not None
assert pocket_encoder is not None
assert len(id_to_token) == len(token_to_id)

print("Tokenizer and generation settings ready.")
print("Vocabulary size:", len(token_to_id))
print("Special token IDs:", {
    "PAD": PAD_ID,
    "BOS": BOS_ID,
    "EOS": EOS_ID,
    "UNK": UNK_ID
})
print("Maximum sequence length:", MAX_LENGTH)
print("Molecules per conformer:", NUM_MOLECULES_PER_CONFORMER)
print("Total planned generations:", 5 * NUM_MOLECULES_PER_CONFORMER)

Tokenizer and generation settings ready.
Vocabulary size: 81
Special token IDs: {'PAD': 0, 'BOS': 1, 'EOS': 2, 'UNK': 3}
Maximum sequence length: 143
Molecules per conformer: 10
Total planned generations: 50


In [347]:
import torch
import torch.nn.functional as F

def generate_smiles(
    coordinates,
    feature_ids,
    pocket_mask,
    temperature=0.8,
    top_k=10,
    max_length=143
):
    molecular_transformer.eval()
    pocket_encoder.eval()

    with torch.no_grad():

        coordinates = coordinates.unsqueeze(0).to("cpu")
        feature_ids = feature_ids.unsqueeze(0).to("cpu")
        pocket_mask = pocket_mask.unsqueeze(0).to("cpu")

        # Encode the selected M1 pocket conformer
        pocket_memory = pocket_encoder(
            coordinates,
            feature_ids,
            (~pocket_mask).float()
        )

        # Start with BOS
        generated = torch.tensor(
            [[BOS_ID]],
            dtype=torch.long
        )

        for _ in range(max_length - 1):

            token_padding_mask = generated.eq(PAD_ID)

            logits = molecular_transformer(
                generated,
                pocket_memory,
                token_padding_mask,
                ~pocket_mask
            )

            next_logits = logits[0, -1, :].clone()

            # Prevent generation of special tokens
            next_logits[PAD_ID] = float("-inf")
            next_logits[BOS_ID] = float("-inf")
            next_logits[UNK_ID] = float("-inf")

            # Temperature scaling
            next_logits = next_logits / temperature

            # Top-k sampling
            k = min(top_k, next_logits.numel())
            values, indices = torch.topk(next_logits, k)

            filtered_logits = torch.full_like(
                next_logits,
                float("-inf")
            )
            filtered_logits[indices] = values

            probabilities = F.softmax(filtered_logits, dim=-1)

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            generated = torch.cat(
                [
                    generated,
                    torch.tensor([[next_token]], dtype=torch.long)
                ],
                dim=1
            )

            if next_token == EOS_ID:
                break

        # Convert generated IDs into SMILES
        token_ids = generated[0].tolist()

        smiles = "".join(
            id_to_token[token_id]
            for token_id in token_ids
            if token_id not in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]
        )

        return smiles


print("SMILES generation function defined successfully.")

SMILES generation function defined successfully.


In [348]:
import numpy as np
import pandas as pd
import torch

torch.manual_seed(42)

generated_records = []

for conformer_idx in range(5):

    print(f"\nGenerating for conformer {conformer_idx + 1}...")

    for sample_idx in range(NUM_MOLECULES_PER_CONFORMER):

        smiles = generate_smiles(
            coordinates=val_data["coordinates"][conformer_idx],
            feature_ids=val_data["feature_ids"][conformer_idx],
            pocket_mask=val_data["pocket_padding_mask"][conformer_idx].logical_not(),
            temperature=TEMPERATURE,
            top_k=TOP_K,
            max_length=MAX_LENGTH
        )

        generated_records.append({
            "conformer_id": conformer_idx + 1,
            "sample_id": sample_idx + 1,
            "smiles": smiles
        })

    print("Completed:", NUM_MOLECULES_PER_CONFORMER)

generated_df = pd.DataFrame(generated_records)

print("\nGeneration completed.")
print("Total generated:", len(generated_df))
print("\nFirst 10 generated SMILES:")
print(generated_df.head(10).to_string(index=False))


Generating for conformer 1...
Completed: 10

Generating for conformer 2...
Completed: 10

Generating for conformer 3...
Completed: 10

Generating for conformer 4...
Completed: 10

Generating for conformer 5...
Completed: 10

Generation completed.
Total generated: 50

First 10 generated SMILES:
 conformer_id  sample_id                                                              smiles
            1          1                                      CCCC(C)CCN1CCCN(Cc2ccc(O)c2)c1
            1          2                      O=C(Nc1ccc(-c2ccc(F)cc2)cnc2c(S(=O)=O)c1)cccc1
            1          3                                Cc1nc(-c2nc[nH]c3cc4ccc3OC2)c1CCCCC1
            1          4         COCN(CO)c1cccc(F)c1C(CC)c1cc(C(=O)N2CCC3(C)CCCCN)c1ccccncc2
            1          5                                   COc1ccc(CN2CCOC(=O)Nc3ccccc3)cc21
            1          6            O=C1CC(=O)Nc2ccc(Cl)c1C(C1)N1CCC(=O)NCC1=O)N1C1CCC(=O)C1
            1          7       COc1ccc(C)cc1NC(=O)c2c

In [349]:
from rdkit import Chem
import pandas as pd

validity_results = []

for _, row in generated_df.iterrows():

    smiles = row["smiles"]

    try:
        molecule = Chem.MolFromSmiles(smiles)

        if molecule is not None:
            canonical_smiles = Chem.MolToSmiles(molecule)

            validity_results.append({
                "conformer_id": row["conformer_id"],
                "sample_id": row["sample_id"],
                "original_smiles": smiles,
                "canonical_smiles": canonical_smiles,
                "valid": True
            })

        else:
            validity_results.append({
                "conformer_id": row["conformer_id"],
                "sample_id": row["sample_id"],
                "original_smiles": smiles,
                "canonical_smiles": None,
                "valid": False
            })

    except Exception:
        validity_results.append({
            "conformer_id": row["conformer_id"],
            "sample_id": row["sample_id"],
            "original_smiles": smiles,
            "canonical_smiles": None,
            "valid": False
        })

validation_df = pd.DataFrame(validity_results)

total = len(validation_df)
valid_count = int(validation_df["valid"].sum())
invalid_count = total - valid_count

print("========== SMILES VALIDATION ==========")
print("Total generated:", total)
print("Valid molecules:", valid_count)
print("Invalid molecules:", invalid_count)
print(f"Validity rate: {valid_count / total * 100:.2f}%")

print("\n========== VALIDATION BY CONFORMER ==========")

print(
    validation_df.groupby("conformer_id")["valid"]
    .agg(["count", "sum"])
)

print("\n========== INVALID SMILES ==========")

print(
    validation_df.loc[
        ~validation_df["valid"],
        ["conformer_id", "sample_id", "original_smiles"]
    ].to_string(index=False)
)

========== SMILES VALIDATION ==========
Total generated: 50
Valid molecules: 2
Invalid molecules: 48
Validity rate: 4.00%

========== VALIDATION BY CONFORMER ==========
              count  sum
conformer_id            
1                10    1
2                10    0
3                10    1
4                10    0
5                10    0

========== INVALID SMILES ==========
 conformer_id  sample_id                                                                                           original_smiles
            1          1                                                                            CCCC(C)CCN1CCCN(Cc2ccc(O)c2)c1
            1          2                                                            O=C(Nc1ccc(-c2ccc(F)cc2)cnc2c(S(=O)=O)c1)cccc1
            1          3                                                                      Cc1nc(-c2nc[nH]c3cc4ccc3OC2)c1CCCCC1
            1          4                                               COCN(CO)c1cccc(F)c1C(CC

In [350]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, QED

valid_df = validation_df[validation_df["valid"]].copy()

property_records = []

for _, row in valid_df.iterrows():

    mol = Chem.MolFromSmiles(row["canonical_smiles"])

    property_records.append({
        "conformer_id": row["conformer_id"],
        "sample_id": row["sample_id"],
        "canonical_smiles": row["canonical_smiles"],
        "molecular_weight": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Crippen.MolLogP(mol), 2),
        "TPSA": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "rings": rdMolDescriptors.CalcNumRings(mol),
        "QED": round(QED.qed(mol), 3)
    })

properties_df = pd.DataFrame(property_records)

print("========== VALID MOLECULE PROPERTIES ==========")

if len(properties_df) > 0:
    print(properties_df.to_string(index=False))
else:
    print("No valid molecules available for property calculation.")

========== VALID MOLECULE PROPERTIES ==========
 conformer_id  sample_id                     canonical_smiles  molecular_weight  LogP  TPSA  HBD  HBA  rotatable_bonds  rings   QED
            1          5    COc1ccc2cc1N(CCOC(=O)Nc1ccccc1)C2            298.34  3.26 50.80    1    4                5      3 0.920
            3          7 CCOC(=O)C(C)CCCN1CCN(c2ccccc2)C=C1OC            332.44  3.23 42.01    0    5                8      2 0.684


In [351]:
lipinski_results = []

for _, row in properties_df.iterrows():

    checks = {
        "MW <= 500": row["molecular_weight"] <= 500,
        "LogP <= 5": row["LogP"] <= 5,
        "HBD <= 5": row["HBD"] <= 5,
        "HBA <= 10": row["HBA"] <= 10
    }

    violations = sum(not result for result in checks.values())

    lipinski_results.append({
        "conformer_id": row["conformer_id"],
        "sample_id": row["sample_id"],
        **checks,
        "violations": violations,
        "passes_lipinski": violations == 0
    })

lipinski_df = pd.DataFrame(lipinski_results)

print("========== LIPINSKI ASSESSMENT ==========")
print(lipinski_df.to_string(index=False))

print("\nMolecules passing all four criteria:",
      lipinski_df["passes_lipinski"].sum(),
      "out of", len(lipinski_df))

========== LIPINSKI ASSESSMENT ==========
 conformer_id  sample_id  MW <= 500  LogP <= 5  HBD <= 5  HBA <= 10  violations  passes_lipinski
            1          5       True       True      True       True           0             True
            3          7       True       True      True       True           0             True

Molecules passing all four criteria: 2 out of 2


In [352]:
total_generated = len(generated_df)

valid_count = len(validation_df[validation_df["valid"]])

valid_canonical = validation_df.loc[
    validation_df["valid"], "canonical_smiles"
]

unique_valid = valid_canonical.nunique()

duplicate_count = valid_count - unique_valid

print("========== GENERATION QUALITY REPORT ==========")

print("Total generated:", total_generated)

print(
    f"Valid molecules: {valid_count}/{total_generated}"
)

print(
    f"Validity rate: {valid_count / total_generated * 100:.2f}%"
)

print("Unique valid molecules:", unique_valid)

print("Duplicate valid molecules:", duplicate_count)

print(
    f"Uniqueness among valid molecules: "
    f"{unique_valid / valid_count * 100:.2f}%"
    if valid_count else "Uniqueness: N/A"
)

print("\n========== CONFORMER-WISE VALIDITY ==========")

conformer_report = (
    validation_df.groupby("conformer_id")
    .agg(
        generated=("valid", "count"),
        valid=("valid", "sum")
    )
)

conformer_report["validity_percent"] = (
    conformer_report["valid"] /
    conformer_report["generated"] * 100
)

print(conformer_report)

========== GENERATION QUALITY REPORT ==========
Total generated: 50
Valid molecules: 2/50
Validity rate: 4.00%
Unique valid molecules: 2
Duplicate valid molecules: 0
Uniqueness among valid molecules: 100.00%

========== CONFORMER-WISE VALIDITY ==========
              generated  valid  validity_percent
conformer_id                                    
1                    10      1              10.0
2                    10      0               0.0
3                    10      1              10.0
4                    10      0               0.0
5                    10      0               0.0


In [353]:
print("========== CONFORMER GENERATION COMPARISON ==========")

for conformer_id in sorted(generated_df["conformer_id"].unique()):

    subset = generated_df[
        generated_df["conformer_id"] == conformer_id
    ]

    unique_smiles = subset["smiles"].nunique()

    valid_subset = validation_df[
        (validation_df["conformer_id"] == conformer_id) &
        (validation_df["valid"])
    ]

    print(f"\nConformer {conformer_id}")
    print("Generated:", len(subset))
    print("Unique raw SMILES:", unique_smiles)
    print("Valid molecules:", len(valid_subset))

    if len(valid_subset):
        print("Valid SMILES:")

        for smiles in valid_subset["canonical_smiles"]:
            print(" ", smiles)

print("\n========== OVERALL COMPARISON ==========")

print(
    "Unique raw SMILES across all conformers:",
    generated_df["smiles"].nunique()
)

print(
    "Unique chemically valid molecules:",
    validation_df.loc[
        validation_df["valid"], "canonical_smiles"
    ].nunique()
)

========== CONFORMER GENERATION COMPARISON ==========

Conformer 1
Generated: 10
Unique raw SMILES: 10
Valid molecules: 1
Valid SMILES:
  COc1ccc2cc1N(CCOC(=O)Nc1ccccc1)C2

Conformer 2
Generated: 10
Unique raw SMILES: 10
Valid molecules: 0

Conformer 3
Generated: 10
Unique raw SMILES: 10
Valid molecules: 1
Valid SMILES:
  CCOC(=O)C(C)CCCN1CCN(c2ccccc2)C=C1OC

Conformer 4
Generated: 10
Unique raw SMILES: 10
Valid molecules: 0

Conformer 5
Generated: 10
Unique raw SMILES: 10
Valid molecules: 0

========== OVERALL COMPARISON ==========
Unique raw SMILES across all conformers: 50
Unique chemically valid molecules: 2


In [354]:
import torch
import torch.nn.functional as F

molecular_transformer.eval()
pocket_encoder.eval()

# Use the first validation molecule as a common molecular context.
test_tokens = val_data["decoder_input"][:1].to("cpu")

# Compare predictions at the same sequence position.
test_position = min(10, test_tokens.shape[1] - 1)

conformer_logits = []

with torch.no_grad():

    for conformer_idx in range(5):

        coordinates = val_data["coordinates"][
            conformer_idx:conformer_idx+1
        ]

        feature_ids = val_data["feature_ids"][
            conformer_idx:conformer_idx+1
        ]

        pocket_mask = ~val_data["pocket_padding_mask"][
            conformer_idx:conformer_idx+1
        ]

        pocket_memory = pocket_encoder(
            coordinates,
            feature_ids,
            pocket_mask.float()
        )

        token_mask = test_tokens.eq(PAD_ID)

        logits = molecular_transformer(
            test_tokens,
            pocket_memory,
            token_mask,
            ~pocket_mask
        )

        conformer_logits.append(
            logits[0, test_position].cpu()
        )

print("========== CONFORMER PREDICTION DIFFERENCES ==========")

for i in range(1, 5):

    difference = torch.mean(
        torch.abs(conformer_logits[0] - conformer_logits[i])
    ).item()

    cosine = F.cosine_similarity(
        conformer_logits[0].unsqueeze(0),
        conformer_logits[i].unsqueeze(0)
    ).item()

    print(
        f"Conformer 1 vs {i+1}: "
        f"Mean absolute difference = {difference:.6f}, "
        f"Cosine similarity = {cosine:.6f}"
    )

========== CONFORMER PREDICTION DIFFERENCES ==========
Conformer 1 vs 2: Mean absolute difference = 0.011331, Cosine similarity = 0.999997
Conformer 1 vs 3: Mean absolute difference = 0.007556, Cosine similarity = 0.999999
Conformer 1 vs 4: Mean absolute difference = 0.007297, Cosine similarity = 0.999999
Conformer 1 vs 5: Mean absolute difference = 0.003786, Cosine similarity = 1.000000


In [355]:
import os

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "member2_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save all generated molecules and validation results
generation_report = generated_df.merge(
    validation_df[
        ["conformer_id", "sample_id",
         "canonical_smiles", "valid"]
    ],
    on=["conformer_id", "sample_id"],
    how="left"
)

generation_path = os.path.join(
    OUTPUT_DIR,
    "member2_generation_validation_report.csv"
)

generation_report.to_csv(generation_path, index=False)

# Save valid molecule properties separately
properties_path = os.path.join(
    OUTPUT_DIR,
    "member2_valid_molecule_properties.csv"
)

properties_df.to_csv(properties_path, index=False)

print("Generation reports saved successfully.")
print("Full report:", generation_path)
print("Valid molecule properties:", properties_path)
print("Total records saved:", len(generation_report))

Generation reports saved successfully.
Full report: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_generation_validation_report.csv
Valid molecule properties: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_valid_molecule_properties.csv
Total records saved: 50


In [356]:
import os
import pandas as pd

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "member2_outputs")

# Select only RDKit-valid generated molecules
handoff_df = properties_df.copy()

handoff_df.insert(
    0,
    "candidate_id",
    [f"M2_EXP_{i+1:03d}" for i in range(len(handoff_df))]
)

# Add validation and provenance information
handoff_df["validity_rate_overall"] = 0.04
handoff_df["pocket_conditioning_status"] = (
    "Meaningful pocket-specific conditioning not demonstrated"
)
handoff_df["candidate_status"] = "exploratory_unvalidated"
handoff_df["biological_activity_confirmed"] = False

handoff_path = os.path.join(
    OUTPUT_DIR,
    "member2_exploratory_M3_handoff.csv"
)

handoff_df.to_csv(handoff_path, index=False)

print("Exploratory M3 handoff created.")
print("File:", handoff_path)
print("Number of molecules:", len(handoff_df))
print("\nHandoff preview:")
display(handoff_df)

Exploratory M3 handoff created.
File: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_exploratory_M3_handoff.csv
Number of molecules: 2

Handoff preview:


,candidate_id,conformer_id,sample_id,canonical_smiles,molecular_weight,LogP,TPSA,HBD,HBA,rotatable_bonds,rings,QED,validity_rate_overall,pocket_conditioning_status,candidate_status,biological_activity_confirmed
0,M2_EXP_001,1,5,COc1ccc2cc1N(CCOC(=O)Nc1ccccc1)C2,298.34,3.26,50.80,1,4,5,3,0.920,0.04,Meaningful pocket-specific conditioning not de...,exploratory_unvalidated,False
1,M2_EXP_002,3,7,CCOC(=O)C(C)CCCN1CCN(c2ccccc2)C=C1OC,332.44,3.23,42.01,0,5,8,2,0.684,0.04,Meaningful pocket-specific conditioning not de...,exploratory_unvalidated,False


In [357]:
import os
from datetime import date

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "member2_outputs")

report = f"""
MEMBER-2 FINAL IMPLEMENTATION REPORT
====================================

Module:
Pocket-Conditioned 2D Molecular Topology Generation

1. INPUT
--------
Source: Member-1 dynamic PARP1 pocket representation
Pocket conformers: 5
Pocket tensor shape: (5, 95, 4)
Pocket mask shape: (5, 95)

2. TRAINING DATA
----------------
Training molecules: 25 unique
Validation molecules: 7 unique
Training examples: 125
Validation examples: 35

General molecular pretraining:
ChEMBL-37-derived dataset

Pocket-conditioned training:
CrossDocked2020-derived PARP1-associated ligand poses
paired with Member-1 pocket conformers.

3. MODEL
--------
Architecture:
- Molecular Transformer
- Dynamic Pocket Encoder
- Pocket cross-attention

Total parameters: 883,793
Trainable parameters during fine-tuning: 249,088

Training device: CPU
Best checkpoint epoch: 12
Best validation loss: 1.10778
Validation token accuracy: 62.53%

4. GENERATION RESULTS
---------------------
Total generated molecules: 50
RDKit-valid molecules: 2
Invalid molecules: 48
Validity rate: 4%
Unique valid molecules: 2

Both valid molecules passed the four Lipinski rules.

5. POCKET CONDITIONING EVALUATION
---------------------------------
Predictions changed only slightly across the five
pocket conformers.

Conformer comparisons produced cosine similarities
very close to 1.

Meaningful pocket-specific or conformer-specific
conditioning has NOT been demonstrated.

6. M3 HANDOFF
-------------
Exploratory molecules exported: 2

File:
member2_exploratory_M3_handoff.csv

Status:
exploratory_unvalidated

Biological activity confirmed: No

7. LIMITATIONS
--------------
- Very small pocket-conditioned training dataset.
- Only 2 out of 50 generated molecules were valid.
- Pocket-specific conditioning remains unproven.
- Lipinski compliance does not establish biological activity.
- The generated molecules are not confirmed PARP1 inhibitors.
- Further training, validation and experimental assessment
  are required.

8. OUTPUT FILES
---------------
member2_generation_validation_report.csv
member2_valid_molecule_properties.csv
member2_exploratory_M3_handoff.csv
pocket_conditioned_training_history.csv
../checkpoints/member2_pocket_conditioned_best.pt

FINAL STATUS
------------
The Member-2 implementation pipeline has been executed.
Exploratory molecular outputs have been generated and
exported for downstream evaluation.

The module is a preliminary research implementation,
not a validated pocket-conditioned molecular generator.

Report date: {date.today()}
"""

report_path = os.path.join(
    OUTPUT_DIR,
    "MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt"
)

with open(report_path, "w") as file:
    file.write(report)

print("Final Member-2 report saved.")
print("Location:", report_path)
print("\n", report)

Final Member-2 report saved.
Location: /content/drive/MyDrive/Member2_Project/member2_outputs/MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt

 
MEMBER-2 FINAL IMPLEMENTATION REPORT

Module:
Pocket-Conditioned 2D Molecular Topology Generation

1. INPUT
--------
Source: Member-1 dynamic PARP1 pocket representation
Pocket conformers: 5
Pocket tensor shape: (5, 95, 4)
Pocket mask shape: (5, 95)

2. TRAINING DATA
----------------
Training molecules: 25 unique
Validation molecules: 7 unique
Training examples: 125
Validation examples: 35

General molecular pretraining:
ChEMBL-37-derived dataset

Pocket-conditioned training:
CrossDocked2020-derived PARP1-associated ligand poses
paired with Member-1 pocket conformers.

3. MODEL
--------
Architecture:
- Molecular Transformer
- Dynamic Pocket Encoder
- Pocket cross-attention

Total parameters: 883,793
Trainable parameters during fine-tuning: 249,088

Training device: CPU
Best checkpoint epoch: 12
Best validation loss: 1.10778
Validation token accuracy: 

In [358]:
import os
import zipfile

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

ZIP_PATH = os.path.join(
    BASE_DIR,
    "Member2_Outputs_For_M3.zip"
)

files_to_include = [
    (OUTPUT_DIR, "member2_generation_validation_report.csv"),
    (OUTPUT_DIR, "member2_valid_molecule_properties.csv"),
    (OUTPUT_DIR, "member2_exploratory_M3_handoff.csv"),
    (OUTPUT_DIR, "pocket_conditioned_training_history.csv"),
    (OUTPUT_DIR, "MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt"),
    (OUTPUT_DIR, "smiles_tokenizer.json"),
    (CHECKPOINT_DIR, "member2_pocket_conditioned_best.pt"),
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:

    for folder, filename in files_to_include:
        file_path = os.path.join(folder, filename)

        if os.path.exists(file_path):
            zipf.write(file_path, arcname=filename)
            print("ADDED:", filename)
        else:
            print("NOT FOUND:", filename)

print("\nZIP created successfully:")
print(ZIP_PATH)

print("\nZIP size (MB):", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2))

ADDED: member2_generation_validation_report.csv
ADDED: member2_valid_molecule_properties.csv
ADDED: member2_exploratory_M3_handoff.csv
ADDED: pocket_conditioned_training_history.csv
ADDED: MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt
ADDED: smiles_tokenizer.json
ADDED: member2_pocket_conditioned_best.pt

ZIP created successfully:
/content/drive/MyDrive/Member2_Project/Member2_Outputs_For_M3.zip

ZIP size (MB): 4.87


In [361]:

import os
import zipfile

BASE_DIR = "/content/drive/MyDrive/Member2_Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "member2_outputs")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

ZIP_PATH = os.path.join(BASE_DIR, "Member2_Outputs_For_M3.zip")

files_to_include = [
    (OUTPUT_DIR, "member2_generation_validation_report.csv"),
    (OUTPUT_DIR, "member2_valid_molecule_properties.csv"),
    (OUTPUT_DIR, "member2_exploratory_M3_handoff.csv"),
    (OUTPUT_DIR, "pocket_conditioned_training_history.csv"),
    (OUTPUT_DIR, "MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt"),
    (OUTPUT_DIR, "smiles_tokenizer.json"),
    (CHECKPOINT_DIR, "member2_pocket_conditioned_best.pt"),
]

print("Creating Member-2 ZIP...\n")

added_files = []
missing_files = []

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder, filename in files_to_include:
        file_path = os.path.join(folder, filename)

        if os.path.exists(file_path):
            zipf.write(file_path, arcname=filename)
            added_files.append(filename)
            print("ADDED:", filename)
        else:
            missing_files.append(filename)
            print("MISSING:", filename)

print("\n========== ZIP SUMMARY ==========")
print("Files added:", len(added_files))
print("Files missing:", len(missing_files))

if os.path.exists(ZIP_PATH):
    print("ZIP created:", ZIP_PATH)
    print("ZIP size (MB):", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2))

    with zipfile.ZipFile(ZIP_PATH, "r") as zipf:
        print("\nZIP integrity:", zipf.testzip() or "PASS")
        print("\nFiles inside ZIP:")
        for name in zipf.namelist():
            print("-", name)


Creating Member-2 ZIP...

ADDED: member2_generation_validation_report.csv
ADDED: member2_valid_molecule_properties.csv
ADDED: member2_exploratory_M3_handoff.csv
ADDED: pocket_conditioned_training_history.csv
ADDED: MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt
ADDED: smiles_tokenizer.json
ADDED: member2_pocket_conditioned_best.pt

========== ZIP SUMMARY ==========
Files added: 7
Files missing: 0
ZIP created: /content/drive/MyDrive/Member2_Project/Member2_Outputs_For_M3.zip
ZIP size (MB): 4.87

ZIP integrity: PASS

Files inside ZIP:
- member2_generation_validation_report.csv
- member2_valid_molecule_properties.csv
- member2_exploratory_M3_handoff.csv
- pocket_conditioned_training_history.csv
- MEMBER2_FINAL_IMPLEMENTATION_REPORT.txt
- smiles_tokenizer.json
- member2_pocket_conditioned_best.pt


In [362]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Member2_Project/Member2_Outputs_For_M3.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [363]:
import torch

print("========== POCKET MASK DIAGNOSTIC ==========")

padding_mask = val_data["pocket_padding_mask"][0]

valid_mask = ~padding_mask

print("Total pocket positions:", len(padding_mask))
print("Valid positions:", valid_mask.sum().item())
print("Padding positions:", padding_mask.sum().item())

print("\nCorrect encoder mask:")
print("Valid positions should be 1.")

print("\nCurrent generation mask:")
print("Valid positions passed to encoder:",
      (~valid_mask).sum().item())

print("\nExpected valid positions:",
      valid_mask.sum().item())

========== POCKET MASK DIAGNOSTIC ==========
Total pocket positions: 95
Valid positions: 95
Padding positions: 0

Correct encoder mask:
Valid positions should be 1.

Current generation mask:
Valid positions passed to encoder: 0

Expected valid positions: 95


In [364]:
# ============================================================
# CELL 331: POCKET-CONDITIONED SMILES GENERATION
# CORRECTED MASK HANDLING
# ============================================================

import torch
import torch.nn.functional as F


def generate_smiles(
    coordinates,
    feature_ids,
    pocket_valid_mask,
    temperature=0.8,
    top_k=10,
    max_length=143
):

    # --------------------------------------------------------
    # 1. MODEL CONFIGURATION
    # --------------------------------------------------------

    molecular_transformer.eval()
    pocket_encoder.eval()

    device = torch.device("cpu")

    assert temperature > 0, "Temperature must be greater than zero."
    assert top_k > 0, "Top-k must be greater than zero."

    # --------------------------------------------------------
    # 2. PREPARE INPUT TENSORS
    # --------------------------------------------------------

    coordinates = coordinates.unsqueeze(0).to(
        device=device,
        dtype=torch.float32
    )

    feature_ids = feature_ids.unsqueeze(0).to(
        device=device,
        dtype=torch.long
    )

    pocket_valid_mask = pocket_valid_mask.unsqueeze(0).to(
        device=device,
        dtype=torch.bool
    )

    # --------------------------------------------------------
    # 3. VERIFY INPUT SHAPES AND MASK
    # --------------------------------------------------------

    assert coordinates.shape[:2] == feature_ids.shape, \
        "Coordinate and feature dimensions do not match."

    assert feature_ids.shape == pocket_valid_mask.shape, \
        "Feature IDs and pocket mask dimensions do not match."

    valid_count = pocket_valid_mask.sum().item()

    assert valid_count > 0, \
        "No valid pocket positions found."

    print("Valid pocket positions:", valid_count)

    # --------------------------------------------------------
    # 4. ENCODE POCKET
    # --------------------------------------------------------

    with torch.no_grad():

        # Encoder requires:
        # 1 = valid pocket position
        # 0 = padding position

        pocket_encoder_mask = pocket_valid_mask.float()

        pocket_memory = pocket_encoder(
            coordinates,
            feature_ids,
            pocket_encoder_mask
        )

        # Verify encoder output
        assert torch.isfinite(pocket_memory).all(), \
            "Pocket encoder produced NaN or Inf."

        assert pocket_memory.abs().sum().item() > 0, \
            "Pocket representation is completely zero."

        print(
            "Pocket memory shape:",
            tuple(pocket_memory.shape)
        )

        print(
            "Pocket memory absolute sum:",
            round(pocket_memory.abs().sum().item(), 4)
        )

        # ----------------------------------------------------
        # 5. START GENERATION WITH BOS
        # ----------------------------------------------------

        generated = torch.tensor(
            [[BOS_ID]],
            dtype=torch.long,
            device=device
        )

        # Transformer requires:
        # True = padding position
        # False = valid position

        pocket_padding_mask = ~pocket_valid_mask

        # ----------------------------------------------------
        # 6. AUTOREGRESSIVE GENERATION
        # ----------------------------------------------------

        for step in range(max_length - 1):

            token_padding_mask = generated.eq(PAD_ID)

            logits = molecular_transformer(
                generated,
                pocket_memory,
                token_padding_mask,
                pocket_padding_mask
            )

            next_logits = logits[0, -1, :].clone()

            # Prevent special tokens
            next_logits[PAD_ID] = float("-inf")
            next_logits[BOS_ID] = float("-inf")
            next_logits[UNK_ID] = float("-inf")

            # Temperature scaling
            next_logits = next_logits / temperature

            # Top-k filtering
            k = min(top_k, next_logits.numel())

            values, indices = torch.topk(
                next_logits,
                k
            )

            filtered_logits = torch.full_like(
                next_logits,
                float("-inf")
            )

            filtered_logits[indices] = values

            # Probability distribution
            probabilities = F.softmax(
                filtered_logits,
                dim=-1
            )

            # Sample next token
            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            # Append token
            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [[next_token]],
                        dtype=torch.long,
                        device=device
                    )
                ],
                dim=1
            )

            # Stop at EOS
            if next_token == EOS_ID:
                break

        # ----------------------------------------------------
        # 7. DECODE TOKEN IDs INTO SMILES
        # ----------------------------------------------------

        token_ids = generated[0].tolist()

        smiles_tokens = []

        for token_id in token_ids:

            if token_id in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]:
                continue

            smiles_tokens.append(
                id_to_token[token_id]
            )

        smiles = "".join(smiles_tokens)

        return smiles


print("=" * 60)
print("CELL 331 COMPLETED")
print("Corrected pocket mask handling.")
print("Encoder receives valid-position mask.")
print("Transformer receives padding-position mask.")
print("=" * 60)

CELL 331 COMPLETED
Corrected pocket mask handling.
Encoder receives valid-position mask.
Transformer receives padding-position mask.


In [365]:
# ============================================================
# CELL 332: GENERATE MOLECULES USING FIVE M1 CONFORMERS
# ============================================================

import numpy as np
import pandas as pd
import torch

torch.manual_seed(42)

generated_records = []

print("=" * 60)
print("POCKET-CONDITIONED MOLECULAR GENERATION")
print("=" * 60)

for conformer_idx in range(5):

    print(
        f"\nGenerating for conformer {conformer_idx + 1}..."
    )

    for sample_idx in range(NUM_MOLECULES_PER_CONFORMER):

        # Extract valid pocket mask.
        # val_data stores padding mask:
        # True = padding
        # False = valid
        #
        # Convert it into valid mask:
        # True = valid
        # False = padding

        pocket_valid_mask = (
            ~val_data["pocket_padding_mask"][conformer_idx]
        )

        smiles = generate_smiles(
            coordinates=val_data["coordinates"][conformer_idx],

            feature_ids=val_data["feature_ids"][conformer_idx],

            pocket_valid_mask=pocket_valid_mask,

            temperature=TEMPERATURE,

            top_k=TOP_K,

            max_length=MAX_LENGTH
        )

        generated_records.append({

            "conformer_id": conformer_idx + 1,

            "sample_id": sample_idx + 1,

            "smiles": smiles

        })

    print(
        "Completed:",
        NUM_MOLECULES_PER_CONFORMER,
        "molecules"
    )


# ------------------------------------------------------------
# SAVE GENERATION RESULTS IN DATAFRAME
# ------------------------------------------------------------

generated_df = pd.DataFrame(generated_records)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GENERATION COMPLETED")
print("=" * 60)

print("Total generated:", len(generated_df))

print("\nGeneration distribution:")

print(
    generated_df.groupby("conformer_id")
    .size()
    .to_string()
)

print("\nFirst 10 generated SMILES:")

print(
    generated_df.head(10).to_string(index=False)
)


# ------------------------------------------------------------
# SAVE OUTPUT
# ------------------------------------------------------------

generation_output_path = os.path.join(
    OUTPUT_DIR,
    "member2_mask_corrected_generation.csv"
)

generated_df.to_csv(
    generation_output_path,
    index=False
)

print("\nSaved generation results:")
print(generation_output_path)

POCKET-CONDITIONED MOLECULAR GENERATION

Generating for conformer 1...
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memory shape: (1, 95, 128)
Pocket memory absolute sum: 10126.957
Valid pocket positions: 95
Pocket memo

In [366]:
# ============================================================
# CELL 333: RDKit VALIDITY CHECK
# ============================================================

from rdkit import Chem
import pandas as pd

validation_records = []

for _, row in generated_df.iterrows():

    smiles = row["smiles"]

    mol = Chem.MolFromSmiles(smiles)

    is_valid = mol is not None

    canonical_smiles = (
        Chem.MolToSmiles(mol)
        if is_valid else None
    )

    validation_records.append({

        "conformer_id": row["conformer_id"],
        "sample_id": row["sample_id"],
        "original_smiles": smiles,
        "is_valid": is_valid,
        "canonical_smiles": canonical_smiles

    })


validation_df = pd.DataFrame(validation_records)

# ------------------------------------------------------------
# OVERALL METRICS
# ------------------------------------------------------------

total = len(validation_df)

valid_count = validation_df["is_valid"].sum()

invalid_count = total - valid_count

validity_rate = (
    valid_count / total * 100
    if total else 0
)

valid_molecules = validation_df[
    validation_df["is_valid"]
]

unique_valid = valid_molecules[
    "canonical_smiles"
].nunique()


print("=" * 55)
print("RDKit GENERATION VALIDATION")
print("=" * 55)

print("Total generated:", total)
print("Valid molecules:", valid_count)
print("Invalid molecules:", invalid_count)

print(f"Validity rate: {validity_rate:.2f}%")

print("Unique valid molecules:", unique_valid)


# ------------------------------------------------------------
# CONFORMER-WISE VALIDITY
# ------------------------------------------------------------

print("\nCONFORMER-WISE RESULTS")

conformer_results = (
    validation_df.groupby("conformer_id")
    .agg(
        total=("is_valid", "count"),
        valid=("is_valid", "sum")
    )
)

conformer_results["validity_percent"] = (
    conformer_results["valid"] /
    conformer_results["total"] * 100
)

print(conformer_results)


# ------------------------------------------------------------
# SHOW VALID MOLECULES
# ------------------------------------------------------------

print("\nVALID MOLECULES")

print(
    valid_molecules[
        ["conformer_id", "sample_id", "canonical_smiles"]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# SHOW INVALID EXAMPLES
# ------------------------------------------------------------

print("\nFIRST 10 INVALID SMILES")

print(
    validation_df[
        ~validation_df["is_valid"]
    ][["conformer_id", "sample_id", "original_smiles"]]
    .head(10)
    .to_string(index=False)
)


# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------

validation_path = os.path.join(
    OUTPUT_DIR,
    "member2_mask_corrected_validation.csv"
)

validation_df.to_csv(
    validation_path,
    index=False
)

print("\nSaved:", validation_path)

RDKit GENERATION VALIDATION
Total generated: 50
Valid molecules: 1
Invalid molecules: 49
Validity rate: 2.00%
Unique valid molecules: 1

CONFORMER-WISE RESULTS
              total  valid  validity_percent
conformer_id                                
1                10      0               0.0
2                10      0               0.0
3                10      1              10.0
4                10      0               0.0
5                10      0               0.0

VALID MOLECULES
 conformer_id  sample_id                                     canonical_smiles
            3          2 CCC1C(F)=CC(NC(=O)c2ccc(-c3ccc(F)cc3)nc2)=CC=C1N(C)C

FIRST 10 INVALID SMILES
 conformer_id  sample_id                                                                       original_smiles
            1          1                                                  CN(C)Cn1ccc(NCCCCN2C(=O)c3ccccc2)CC1
            1          2                   C[C@@]1(CO)[C@@H]2CC[C@@H]2CC[C@@H]3O1C[C@@H]3c(C(CO)CNC(=O)N)

In [367]:
from collections import Counter
import re

def diagnose_smiles(smiles):
    problems = []

    if smiles.count("(") != smiles.count(")"):
        problems.append("Unbalanced parentheses")

    if smiles.count("[") != smiles.count("]"):
        problems.append("Unbalanced square brackets")

    ring_digits = re.findall(r"\d", smiles)

    for digit in set(ring_digits):
        if ring_digits.count(digit) != 2:
            problems.append(f"Invalid ring closure: {digit}")

    if not smiles:
        problems.append("Empty SMILES")

    return problems if problems else ["Other chemical/syntax error"]


invalid_df = validation_df[~validation_df["is_valid"]].copy()

problem_counts = Counter()

for smiles in invalid_df["original_smiles"]:
    for problem in diagnose_smiles(smiles):
        problem_counts[problem] += 1

print("INVALID SMILES DIAGNOSTICS")
print("=" * 45)

for problem, count in problem_counts.most_common():
    print(f"{problem}: {count}")

print("\nExamples:")
for smiles in invalid_df["original_smiles"].head(5):
    print(smiles)
    print(diagnose_smiles(smiles))
    print()

INVALID SMILES DIAGNOSTICS
Invalid ring closure: 1: 30
Invalid ring closure: 2: 28
Invalid ring closure: 3: 22
Unbalanced parentheses: 18
Invalid ring closure: 4: 7
Other chemical/syntax error: 2

Examples:
CN(C)Cn1ccc(NCCCCN2C(=O)c3ccccc2)CC1
['Invalid ring closure: 3']

C[C@@]1(CO)[C@@H]2CC[C@@H]2CC[C@@H]3O1C[C@@H]3c(C(CO)CNC(=O)N)CN1C3
['Invalid ring closure: 1', 'Invalid ring closure: 3']

CN(C)CCOCCN1C(=O)cc2c(ccc3)ccc32CCN(CO)CN)C1
['Unbalanced parentheses']

O=C(c1ccc2cccc(F)(F)F)c1)c(CN1CCN1
['Invalid ring closure: 1', 'Invalid ring closure: 2']

O=C(CNC(=O)c1cc[nH]c2c(=O)c1)c(=O)c(Cl)c1OCC(=O)N3CN1C1(F)c1=O
['Invalid ring closure: 1', 'Invalid ring closure: 3', 'Invalid ring closure: 2']



In [368]:
from rdkit import Chem

test_smiles = [
    "CCO",
    "C1CCCCC1",
    "CC(=O)O",
    "c1ccccc1",
    "CCN(CC)CC",
    "COc1ccccc1"
]

print("TOKENIZER ROUND-TRIP TEST")
print("=" * 50)

for smiles in test_smiles:

    mol = Chem.MolFromSmiles(smiles)

    tokens = tokenize_smiles(smiles)

    token_ids = [
        token_to_id.get(token, UNK_ID)
        for token in tokens
    ]

    decoded = "".join(
        id_to_token[token_id]
        for token_id in token_ids
    )

    decoded_mol = Chem.MolFromSmiles(decoded)

    print("\nOriginal:", smiles)
    print("Tokens:", tokens)
    print("Decoded:", decoded)
    print("Original valid:", mol is not None)
    print("Decoded valid:", decoded_mol is not None)
    print("Exact match:", smiles == decoded)

TOKENIZER ROUND-TRIP TEST

Original: CCO
Tokens: [1, 22, 22, 27, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded: <UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK><UNK

In [369]:
from rdkit import Chem

test_smiles = [
    "CCO",
    "C1CCCCC1",
    "CC(=O)O",
    "c1ccccc1",
    "CCN(CC)CC",
    "COc1ccccc1"
]

print("CORRECTED TOKENIZER ROUND-TRIP TEST")
print("=" * 50)

for smiles in test_smiles:

    # Step 1: Original molecule validation
    original_valid = Chem.MolFromSmiles(smiles) is not None

    # Step 2: Tokenize directly into IDs
    token_ids = tokenize_smiles(smiles)

    # Step 3: Decode IDs, skipping BOS and stopping at EOS/PAD
    decoded_tokens = []

    for token_id in token_ids:
        token_id = int(token_id)

        if token_id == PAD_ID or token_id == EOS_ID:
            break

        if token_id == BOS_ID:
            continue

        decoded_tokens.append(id_to_token[token_id])

    decoded = "".join(decoded_tokens)

    # Step 4: Validate decoded molecule
    decoded_valid = Chem.MolFromSmiles(decoded) is not None

    print("\nOriginal:", smiles)
    print("Decoded :", decoded)
    print("Original valid:", original_valid)
    print("Decoded valid :", decoded_valid)
    print("Exact match   :", smiles == decoded)

CORRECTED TOKENIZER ROUND-TRIP TEST

Original: CCO
Decoded : CCO
Original valid: True
Decoded valid : True
Exact match   : True

Original: C1CCCCC1
Decoded : C1CCCCC1
Original valid: True
Decoded valid : True
Exact match   : True

Original: CC(=O)O
Decoded : CC(=O)O
Original valid: True
Decoded valid : True
Exact match   : True

Original: c1ccccc1
Decoded : c1ccccc1
Original valid: True
Decoded valid : True
Exact match   : True

Original: CCN(CC)CC
Decoded : CCN(CC)CC
Original valid: True
Decoded valid : True
Exact match   : True

Original: COc1ccccc1
Decoded : COc1ccccc1
Original valid: True
Decoded valid : True
Exact match   : True


In [370]:
import inspect

print(inspect.getsource(generate_smiles))

def generate_smiles(
    coordinates,
    feature_ids,
    pocket_valid_mask,
    temperature=0.8,
    top_k=10,
    max_length=143
):

    # --------------------------------------------------------
    # 1. MODEL CONFIGURATION
    # --------------------------------------------------------

    molecular_transformer.eval()
    pocket_encoder.eval()

    device = torch.device("cpu")

    assert temperature > 0, "Temperature must be greater than zero."
    assert top_k > 0, "Top-k must be greater than zero."

    # --------------------------------------------------------
    # 2. PREPARE INPUT TENSORS
    # --------------------------------------------------------

    coordinates = coordinates.unsqueeze(0).to(
        device=device,
        dtype=torch.float32
    )

    feature_ids = feature_ids.unsqueeze(0).to(
        device=device,
        dtype=torch.long
    )

    pocket_valid_mask = pocket_valid_mask.unsqueeze(0).to(
        device=device,
        dtype=torch.bool
    )

    # --

In [371]:
import torch
import torch.nn.functional as F
import pandas as pd

def generate_smiles(
    coordinates,
    feature_ids,
    pocket_valid_mask,
    use_pocket=True,
    temperature=0.8,
    top_k=10,
    max_length=143
):

    molecular_transformer.eval()
    pocket_encoder.eval()

    device = torch.device("cpu")

    coordinates = coordinates.unsqueeze(0).to(
        device=device, dtype=torch.float32
    )

    feature_ids = feature_ids.unsqueeze(0).to(
        device=device, dtype=torch.long
    )

    pocket_valid_mask = pocket_valid_mask.unsqueeze(0).to(
        device=device, dtype=torch.bool
    )

    with torch.no_grad():

        pocket_memory = pocket_encoder(
            coordinates,
            feature_ids,
            pocket_valid_mask.float()
        )

        # Experiment B: remove informative pocket representation
        if not use_pocket:
            pocket_memory = torch.zeros_like(pocket_memory)

        pocket_padding_mask = ~pocket_valid_mask

        generated = torch.tensor(
            [[BOS_ID]], dtype=torch.long, device=device
        )

        for step in range(max_length - 1):

            token_padding_mask = generated.eq(PAD_ID)

            logits = molecular_transformer(
                generated,
                pocket_memory,
                token_padding_mask,
                pocket_padding_mask
            )

            next_logits = logits[0, -1, :].clone()

            next_logits[PAD_ID] = float("-inf")
            next_logits[BOS_ID] = float("-inf")
            next_logits[UNK_ID] = float("-inf")

            next_logits = next_logits / temperature

            k = min(top_k, next_logits.numel())

            values, indices = torch.topk(next_logits, k)

            filtered_logits = torch.full_like(
                next_logits, float("-inf")
            )

            filtered_logits[indices] = values

            probabilities = F.softmax(
                filtered_logits, dim=-1
            )

            next_token = torch.multinomial(
                probabilities, 1
            ).item()

            generated = torch.cat(
                [
                    generated,
                    torch.tensor(
                        [[next_token]],
                        dtype=torch.long,
                        device=device
                    )
                ],
                dim=1
            )

            if next_token == EOS_ID:
                break

        token_ids = generated[0].tolist()

        smiles_tokens = []

        for token_id in token_ids:

            if token_id in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]:
                continue

            smiles_tokens.append(id_to_token[token_id])

        return "".join(smiles_tokens)


# --------------------------------------------------
# RUN CONTROLLED ABLATION
# --------------------------------------------------

results = []

NUM_SAMPLES = 10
TEMPERATURE = 0.8
TOP_K = 10

for condition, use_pocket in [
    ("actual_pocket", True),
    ("zero_pocket", False)
]:

    print("\nRunning:", condition)

    # Same random seed for controlled comparison
    torch.manual_seed(42)

    for conformer_idx in range(5):

        coordinates = val_data["coordinates"][conformer_idx]
        feature_ids = val_data["feature_ids"][conformer_idx]

        pocket_valid_mask = ~val_data[
            "pocket_padding_mask"
        ][conformer_idx]

        for sample_idx in range(NUM_SAMPLES):

            smiles = generate_smiles(
                coordinates=coordinates,
                feature_ids=feature_ids,
                pocket_valid_mask=pocket_valid_mask,
                use_pocket=use_pocket,
                temperature=TEMPERATURE,
                top_k=TOP_K
            )

            results.append({
                "condition": condition,
                "conformer_id": conformer_idx + 1,
                "sample_id": sample_idx + 1,
                "smiles": smiles
            })

    print("Completed:", condition)


ablation_df = pd.DataFrame(results)

save_path = (
    OUTPUT_DIR + "/member2_pocket_ablation_generation.csv"
)

ablation_df.to_csv(save_path, index=False)

print("\nABLATION GENERATION COMPLETED")
print("Total generated:", len(ablation_df))
print(ablation_df.groupby("condition").size())
print("Saved:", save_path)


Running: actual_pocket
Completed: actual_pocket

Running: zero_pocket
Completed: zero_pocket

ABLATION GENERATION COMPLETED
Total generated: 100
condition
actual_pocket    50
zero_pocket      50
dtype: int64
Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_pocket_ablation_generation.csv


In [372]:
from rdkit import Chem
import pandas as pd

validation_results = []

for _, row in ablation_df.iterrows():

    smiles = row["smiles"]
    mol = Chem.MolFromSmiles(smiles)

    valid = mol is not None

    canonical = (
        Chem.MolToSmiles(mol)
        if valid else None
    )

    validation_results.append({
        "condition": row["condition"],
        "conformer_id": row["conformer_id"],
        "sample_id": row["sample_id"],
        "smiles": smiles,
        "is_valid": valid,
        "canonical_smiles": canonical
    })


ablation_validation_df = pd.DataFrame(validation_results)


# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

summary = []

for condition, group in ablation_validation_df.groupby("condition"):

    total = len(group)
    valid_df = group[group["is_valid"]]

    valid_count = len(valid_df)

    summary.append({
        "condition": condition,
        "total": total,
        "valid": valid_count,
        "invalid": total - valid_count,
        "validity_percent": round(
            valid_count / total * 100, 2
        ),
        "unique_valid": valid_df[
            "canonical_smiles"
        ].nunique()
    })


summary_df = pd.DataFrame(summary)

print("=" * 65)
print("POCKET ABLATION COMPARISON")
print("=" * 65)

print(summary_df.to_string(index=False))


# --------------------------------------------------
# SAVE RESULTS
# --------------------------------------------------

validation_path = (
    OUTPUT_DIR + "/member2_pocket_ablation_validation.csv"
)

summary_path = (
    OUTPUT_DIR + "/member2_pocket_ablation_summary.csv"
)

ablation_validation_df.to_csv(
    validation_path, index=False
)

summary_df.to_csv(
    summary_path, index=False
)

print("\nSaved validation:", validation_path)
print("Saved summary:", summary_path)

POCKET ABLATION COMPARISON
    condition  total  valid  invalid  validity_percent  unique_valid
actual_pocket     50      1       49               2.0             1
  zero_pocket     50      2       48               4.0             2

Saved validation: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_pocket_ablation_validation.csv
Saved summary: /content/drive/MyDrive/Member2_Project/member2_outputs/member2_pocket_ablation_summary.csv


In [373]:
import torch
import os

checkpoint_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "checkpoints/transformer_chembl37_improved_best.pt"
)

print("Checkpoint exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print("-", key)

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\nPossible state dictionary: {key}")
            print("Number of entries:", len(value))
            print("First 5 entries:", list(value.keys())[:5])

Checkpoint exists: True

Checkpoint type: <class 'dict'>

Checkpoint keys:
- model_state_dict
- optimizer_state_dict
- epoch
- train_loss
- validation_loss
- validation_accuracy
- training_molecules
- training_type
- pocket_cross_attention_used
- vocabulary_size

Possible state dictionary: model_state_dict
Number of entries: 58
First 5 entries: ['token_embedding.weight', 'position_encoding.position_embedding.weight', 'decoder.layers.0.self_attn.in_proj_weight', 'decoder.layers.0.self_attn.in_proj_bias', 'decoder.layers.0.self_attn.out_proj.weight']

Possible state dictionary: optimizer_state_dict
Number of entries: 2
First 5 entries: ['state', 'param_groups']


In [374]:
baseline_state = checkpoint["model_state_dict"]
current_state = molecular_transformer.state_dict()

print("Training metadata:")
for key in [
    "epoch",
    "train_loss",
    "validation_loss",
    "validation_accuracy",
    "training_molecules",
    "training_type",
    "pocket_cross_attention_used"
]:
    print(f"{key}: {checkpoint.get(key)}")

print("\nModel compatibility:")
print("Checkpoint parameters:", len(baseline_state))
print("Current model parameters:", len(current_state))

missing = [k for k in baseline_state if k not in current_state]
mismatched = [
    k for k in baseline_state
    if k in current_state
    and baseline_state[k].shape != current_state[k].shape
]

print("Missing parameters:", missing)
print("Shape mismatches:", mismatched)

print("\nMolecular-only forward function exists:",
      "molecular_only_forward" in globals())

Training metadata:
epoch: 5
train_loss: 1.2469141580714287
validation_loss: 1.1923144488281978
validation_accuracy: 0.6073443080423503
training_molecules: 10000
training_type: chembl37_general_molecular_finetuning
pocket_cross_attention_used: False

Model compatibility:
Checkpoint parameters: 58
Current model parameters: 58
Missing parameters: []
Shape mismatches: []

Molecular-only forward function exists: True


In [375]:
import inspect

# Preserve the pocket-conditioned model for later restoration
pocket_conditioned_state = {
    key: value.detach().cpu().clone()
    for key, value in molecular_transformer.state_dict().items()
}

# Load original ChEMBL pretrained weights
molecular_transformer.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

molecular_transformer.eval()

print("Original ChEMBL checkpoint loaded successfully.")

# Inspect the existing molecular-only forward function
print("\nMolecular-only forward function:")
print(inspect.getsource(molecular_only_forward))

# Test a small input
test_input = torch.tensor([[BOS_ID, 4, 5]], dtype=torch.long)

with torch.no_grad():
    test_output = molecular_only_forward(
        molecular_transformer,
        test_input
    )

print("\nInput shape:", test_input.shape)
print("Output shape:", test_output.shape)
print("Output contains NaN:", torch.isnan(test_output).any().item())

Original ChEMBL checkpoint loaded successfully.

Molecular-only forward function:
def molecular_only_forward(model, token_ids):

    batch_size, seq_length = token_ids.shape

    # Token and positional embeddings
    x = model.token_embedding(token_ids)
    x = model.position_encoding(x)

    # Padding mask
    padding_mask = token_ids == 0

    # Causal mask: prevent looking at future tokens
    causal_mask = torch.triu(
        torch.ones(
            seq_length,
            seq_length,
            device=token_ids.device,
            dtype=torch.bool
        ),
        diagonal=1
    )

    # Apply molecular self-attention and feed-forward layers
    for layer in model.decoder.layers:

        if layer.norm_first:

            # Self-attention
            normalized = layer.norm1(x)

            attention_output = layer.self_attn(
                normalized,
                normalized,
                normalized,
                attn_mask=causal_mask,
                key_padding_mas

In [376]:
import torch
import torch.nn.functional as F
import pandas as pd
from rdkit import Chem
from collections import Counter

def generate_baseline_smiles(
    temperature=0.8,
    top_k=10,
    max_length=143
):
    molecular_transformer.eval()
    device = torch.device("cpu")

    generated = torch.tensor(
        [[BOS_ID]], dtype=torch.long, device=device
    )

    with torch.no_grad():
        for step in range(max_length - 1):

            logits = molecular_only_forward(
                molecular_transformer, generated
            )

            next_logits = logits[0, -1, :].clone()

            next_logits[PAD_ID] = float("-inf")
            next_logits[BOS_ID] = float("-inf")
            next_logits[UNK_ID] = float("-inf")

            next_logits = next_logits / temperature

            k = min(top_k, next_logits.numel())
            values, indices = torch.topk(next_logits, k)

            filtered = torch.full_like(
                next_logits, float("-inf")
            )
            filtered[indices] = values

            probabilities = F.softmax(filtered, dim=-1)

            next_token = torch.multinomial(
                probabilities, 1
            ).item()

            generated = torch.cat([
                generated,
                torch.tensor([[next_token]], dtype=torch.long)
            ], dim=1)

            if next_token == EOS_ID:
                break

    tokens = generated[0].tolist()

    smiles = "".join(
        id_to_token[token]
        for token in tokens
        if token not in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]
    )

    return smiles


# Generate 50 molecules
torch.manual_seed(42)

baseline_records = []

for i in range(50):
    smiles = generate_baseline_smiles()

    mol = Chem.MolFromSmiles(smiles)
    valid = mol is not None

    baseline_records.append({
        "sample_id": i + 1,
        "smiles": smiles,
        "valid": valid,
        "canonical_smiles": (
            Chem.MolToSmiles(mol) if valid else None
        )
    })

baseline_df = pd.DataFrame(baseline_records)

valid_df = baseline_df[baseline_df["valid"]]

print("Total generated:", len(baseline_df))
print("Valid:", len(valid_df))
print("Invalid:", len(baseline_df) - len(valid_df))
print("Validity:", round(len(valid_df) / 50 * 100, 2), "%")
print("Unique valid molecules:", valid_df["canonical_smiles"].nunique())

display(baseline_df.head(10))

output_path = (
    "/content/drive/MyDrive/Member2_Project/"
    "member2_outputs/chembl_baseline_generation.csv"
)

baseline_df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

Total generated: 50
Valid: 3
Invalid: 47
Validity: 6.0 %
Unique valid molecules: 3


,sample_id,smiles,valid,canonical_smiles
0,1,CCCC(C)CCN1CCCN(Cc2ccc(O)c2)c1,False,None
1,2,O=C(Nc1ccc(-c2ccc(F)cc2)cnc2c(S(=O)=O)c1)cccc1,False,None
2,3,Cc1nc(-c2nc[nH]c3cc4ccc3OC2)c1CCCCC1,False,None
3,4,COCN(CO)c1cccc(F)c1C(CC)c1cc(C(=O)N2CCC3(C)CCC...,False,None
4,5,COc1ccc(CN2CCOC(=O)Nc3ccccc3)cc21,True,COc1ccc2cc1N(CCOC(=O)Nc1ccccc1)C2
5,6,O=C1CC(=O)Nc2ccc(Cl)c1C(C1)N1CCC(=O)NCC1=O)N1C...,False,None
6,7,COc1ccc(C)cc1NC(=O)c2ccc(C(=O)NC(C)Nc3cc(OC(C)...,False,None
7,8,CC(=O)N[C@H](C(=O)Nc1ccccc1)N(C(=O)O)cc1OCCC(=...,False,None
8,9,CCC(=O)N1CC(C)c2ccc(S(=O)(=O)c3c(Cl)cc(O)ccc3)c21,False,None
9,10,Cc1ccc(CCc2ccccc2c1N(C(=O)C2Cc1cccccc1)ccc1,False,None



Saved: /content/drive/MyDrive/Member2_Project/member2_outputs/chembl_baseline_generation.csv


In [377]:
from rdkit import Chem
from collections import Counter

failure_counts = Counter()
failure_examples = {}

for _, row in baseline_df.iterrows():

    if row["valid"]:
        continue

    smiles = row["smiles"]

    # Check branch balance
    if smiles.count("(") != smiles.count(")"):
        reason = "Unbalanced parentheses"

    # Check ring-digit balance
    else:
        ring_counts = Counter(
            ch for ch in smiles if ch.isdigit()
        )

        if any(count % 2 != 0 for count in ring_counts.values()):
            reason = "Unclosed ring"

        else:
            mol = Chem.MolFromSmiles(smiles, sanitize=False)

            if mol is None:
                reason = "SMILES parsing failure"
            else:
                try:
                    Chem.SanitizeMol(mol)
                    reason = "Other validation failure"
                except Exception:
                    reason = "Chemical sanitization failure"

    failure_counts[reason] += 1
    failure_examples.setdefault(reason, smiles)

print("BASELINE FAILURE ANALYSIS")
print("-" * 40)

for reason, count in failure_counts.most_common():
    print(f"{reason}: {count}")
    print("Example:", failure_examples[reason])
    print()

BASELINE FAILURE ANALYSIS
----------------------------------------
Unclosed ring: 27
Example: O=C(Nc1ccc(-c2ccc(F)cc2)cnc2c(S(=O)=O)c1)cccc1

Unbalanced parentheses: 12
Example: O=C1CC(=O)Nc2ccc(Cl)c1C(C1)N1CCC(=O)NCC1=O)N1C1CCC(=O)C1

Chemical sanitization failure: 7
Example: CCCC(C)CCN1CCCN(Cc2ccc(O)c2)c1

SMILES parsing failure: 1
Example: O=C1Cc2ccc(-c3cc(F)cc(Cl)cc3)nc2cc1)N1CC(O)CC(=O)C(C1



In [378]:
def generate_baseline_audit(max_length=143, temperature=0.8, top_k=10):

    generated = torch.tensor([[BOS_ID]], dtype=torch.long)
    eos_reached = False

    with torch.no_grad():
        for step in range(max_length - 1):

            logits = molecular_only_forward(
                molecular_transformer, generated
            )

            next_logits = logits[0, -1, :].clone()

            next_logits[PAD_ID] = float("-inf")
            next_logits[BOS_ID] = float("-inf")
            next_logits[UNK_ID] = float("-inf")

            next_logits /= temperature

            values, indices = torch.topk(
                next_logits, min(top_k, next_logits.numel())
            )

            filtered = torch.full_like(next_logits, float("-inf"))
            filtered[indices] = values

            probabilities = F.softmax(filtered, dim=-1)

            next_token = torch.multinomial(probabilities, 1).item()

            generated = torch.cat([
                generated,
                torch.tensor([[next_token]], dtype=torch.long)
            ], dim=1)

            if next_token == EOS_ID:
                eos_reached = True
                break

    token_ids = generated[0].tolist()

    smiles = "".join(
        id_to_token[token]
        for token in token_ids
        if token not in [PAD_ID, BOS_ID, EOS_ID, UNK_ID]
    )

    return {
        "smiles": smiles,
        "eos_reached": eos_reached,
        "generated_tokens": len(token_ids) - 1,
        "valid": Chem.MolFromSmiles(smiles) is not None
    }


torch.manual_seed(42)

audit_results = [
    generate_baseline_audit()
    for _ in range(50)
]

audit_df = pd.DataFrame(audit_results)

print("EOS reached:", audit_df["eos_reached"].sum(), "/ 50")
print("Maximum-length generations:",
      (~audit_df["eos_reached"]).sum())
print("Average generated tokens:",
      round(audit_df["generated_tokens"].mean(), 2))

print("\nValidity by termination:")
print(audit_df.groupby("eos_reached")["valid"].agg(["count", "sum", "mean"]))

EOS reached: 50 / 50
Maximum-length generations: 0
Average generated tokens: 47.62

Validity by termination:
             count  sum  mean
eos_reached                  
True            50    3  0.06


In [379]:
import os

project_path = "/content/drive/MyDrive/Member2_Project"

print("Training data file:")
smiles_path = os.path.join(
    project_path,
    "member2_outputs",
    "chembl37_clean_smiles.txt"
)
print("Exists:", os.path.exists(smiles_path))

print("\nTokenizer variables:")
for name in [
    "token_to_id",
    "id_to_token",
    "tokenize_smiles",
    "encode_smiles",
    "smiles_tokenizer"
]:
    print(name, ":", name in globals())

print("\nTokenizer-related functions:")
for name, obj in globals().copy().items():
    if callable(obj) and any(
        word in name.lower()
        for word in ["token", "encode", "smiles"]
    ):
        print(name)

Training data file:
Exists: True

Tokenizer variables:
token_to_id : True
id_to_token : True
tokenize_smiles : True
encode_smiles : True
smiles_tokenizer : False

Tokenizer-related functions:
make_tokens_for_conformer
summarize_tokens
choose_smiles
tokenize_smiles
generate_one_smiles
DynamicPocketEncoder
pocket_encoder
encode_smiles
inspect_smiles
find_token_mapping
decode_tokens
calculate_token_frequencies
special_token_counts
generate_smiles
analyze_smiles
diagnose_smiles
classify_smiles
safe_tokenize
get_tokens
generate_baseline_smiles


In [380]:
import pandas as pd
import torch
import torch.nn.functional as F
from rdkit import Chem

# Load 100 valid ChEMBL molecules
with open(smiles_path, "r") as f:
    all_smiles = [line.strip() for line in f if line.strip()]

valid_smiles = []

for smi in all_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        valid_smiles.append(smi)

test_smiles = valid_smiles[:100]

print("Available valid molecules:", len(valid_smiles))
print("Testing molecules:", len(test_smiles))


def evaluate_teacher_forcing(smiles):
    token_ids = encode_smiles(smiles)

    # Add BOS and EOS if the encoder does not add them
    if token_ids[0] != BOS_ID:
        token_ids = [BOS_ID] + token_ids

    if token_ids[-1] != EOS_ID:
        token_ids = token_ids + [EOS_ID]

    input_ids = torch.tensor(
        [token_ids[:-1]], dtype=torch.long
    )

    target_ids = torch.tensor(
        [token_ids[1:]], dtype=torch.long
    )

    with torch.no_grad():
        logits = molecular_only_forward(
            molecular_transformer, input_ids
        )

        predictions = logits.argmax(dim=-1)

    correct = (predictions == target_ids).sum().item()
    total = target_ids.numel()

    return correct, total


total_correct = 0
total_tokens = 0
exact_matches = 0

for smi in test_smiles:
    correct, total = evaluate_teacher_forcing(smi)

    total_correct += correct
    total_tokens += total

    if correct == total:
        exact_matches += 1

accuracy = total_correct / total_tokens * 100

print("\n========== RECONSTRUCTION RESULTS ==========")
print("Molecules tested:", len(test_smiles))
print("Total tokens:", total_tokens)
print("Correct next-token predictions:", total_correct)
print("Token prediction accuracy:", round(accuracy, 2), "%")
print("Perfect sequence predictions:", exact_matches)
print("Exact sequence accuracy:",
      round(exact_matches / len(test_smiles) * 100, 2), "%")

Available valid molecules: 50000
Testing molecules: 100


KeyError: 1

In [381]:
import inspect

print("========== encode_smiles ==========")
print(inspect.getsource(encode_smiles))

print("\n========== tokenize_smiles ==========")
print(inspect.getsource(tokenize_smiles))

print("\n========== TOKENIZER TEST ==========")

sample = test_smiles[0]

print("Sample SMILES:", sample)
print("Tokenized output:", tokenize_smiles(sample))
print("Tokenized output type:", type(tokenize_smiles(sample)))

print("Tokenizer mapping sample:")
print(list(token_to_id.items())[:15])

========== encode_smiles ==========
def encode_smiles(smiles):

    tokens = tokenize_smiles(smiles)

    ids = [BOS] + [
        token_to_id[token] for token in tokens
    ] + [EOS]

    if len(ids) > MAX_LEN:
        raise ValueError(
            f"SMILES exceeds maximum length: {len(ids)}"
        )

    ids += [PAD] * (MAX_LEN - len(ids))

    return np.asarray(ids, dtype=np.int64)


========== tokenize_smiles ==========
def tokenize_smiles(smiles):
    remaining = smiles
    ids = []

    while remaining:
        matched = False

        for token in tokens_sorted:
            if remaining.startswith(token):
                ids.append(token_to_id[token])
                remaining = remaining[len(token):]
                matched = True
                break

        if not matched:
            raise ValueError(
                f"Unknown SMILES fragment: {remaining[:30]}"
            )

    sequence = [BOS_ID] + ids + [EOS_ID]

    if len(sequence) > max_length:
        raise ValueE

In [382]:
import torch

def evaluate_teacher_forcing(smiles):

    # Already contains BOS, EOS and PAD
    encoded = encode_smiles(smiles)

    # Remove trailing padding
    sequence = encoded[:]
    non_pad_length = int((sequence != PAD_ID).sum())

    sequence = sequence[:non_pad_length]

    input_ids = torch.tensor(
        [sequence[:-1]], dtype=torch.long
    )

    target_ids = torch.tensor(
        [sequence[1:]], dtype=torch.long
    )

    with torch.no_grad():
        logits = molecular_only_forward(
            molecular_transformer, input_ids
        )

        predictions = logits.argmax(dim=-1)

    correct = (predictions == target_ids).sum().item()
    total = target_ids.numel()

    return correct, total


total_correct = 0
total_tokens = 0
exact_matches = 0

for smi in test_smiles:

    correct, total = evaluate_teacher_forcing(smi)

    total_correct += correct
    total_tokens += total

    if correct == total:
        exact_matches += 1

print("========== RECONSTRUCTION RESULTS ==========")
print("Molecules tested:", len(test_smiles))
print("Total tokens:", total_tokens)
print("Correct predictions:", total_correct)
print("Token prediction accuracy:",
      round(total_correct / total_tokens * 100, 2), "%")
print("Perfect sequence predictions:", exact_matches)
print("Exact sequence accuracy:",
      round(exact_matches / len(test_smiles) * 100, 2), "%")

KeyError: 1

In [383]:
import torch

def evaluate_teacher_forcing(smiles):

    # tokenize_smiles already returns IDs, BOS, EOS and PAD
    encoded = tokenize_smiles(smiles)

    # Remove trailing padding
    non_pad_length = next(
        (i for i, token in enumerate(encoded) if token == PAD_ID),
        len(encoded)
    )

    sequence = encoded[:non_pad_length]

    input_ids = torch.tensor(
        [sequence[:-1]], dtype=torch.long
    )

    target_ids = torch.tensor(
        [sequence[1:]], dtype=torch.long
    )

    with torch.no_grad():
        logits = molecular_only_forward(
            molecular_transformer, input_ids
        )

        predictions = logits.argmax(dim=-1)

    correct = (predictions == target_ids).sum().item()
    total = target_ids.numel()

    return correct, total


total_correct = 0
total_tokens = 0
exact_matches = 0

for smi in test_smiles:

    correct, total = evaluate_teacher_forcing(smi)

    total_correct += correct
    total_tokens += total

    if correct == total:
        exact_matches += 1

print("========== RECONSTRUCTION RESULTS ==========")
print("Molecules tested:", len(test_smiles))
print("Total tokens:", total_tokens)
print("Correct predictions:", total_correct)
print("Token prediction accuracy:",
      round(total_correct / total_tokens * 100, 2), "%")
print("Perfect sequence predictions:", exact_matches)
print("Exact sequence accuracy:",
      round(exact_matches / len(test_smiles) * 100, 2), "%")

========== RECONSTRUCTION RESULTS ==========
Molecules tested: 100
Total tokens: 5206
Correct predictions: 3093
Token prediction accuracy: 59.41 %
Perfect sequence predictions: 0
Exact sequence accuracy: 0.0 %
